# Recruiting database pipeline on Cloudflare D1

End-to-end applicant-tracking backend: schema build → reference data (companies/positions scraped from LinkedIn, skill dictionaries) → raw submission ingestion → normalization → rule-based resume parsing → duplicate detection → Candidate / Person layer → education / employment / skill / project extraction → ML scoring → recruitment-stage pipeline.

**Configuration (Colab Secrets):** `CLOUDFLARE_KEY` (API token), `CLOUDFLARE_ACCOUNT_ID`, `D1_DATABASE_ID`.  
**Project folder:** `/content/drive/MyDrive/recruiting-pipeline` with `create.sql`, `drop.sql`, `code/`, `datasets/`.

> Inline code comments are in Chinese. Outputs that contained candidate data have been cleared; only aggregate outputs are kept.

# database build

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 独立的 D1 连接 cell —— 专门给 CV_Extraction 这部分用
# 不依赖任何其他 cell（不需要 RawSubmissionLoader 那个类，
# 不需要之前跑没跑过 cell 15）
# ============================================================

import requests
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')


class D1Client:
    """
    最基础的 D1 连接对象，只有一个 _execute() 方法。
    CV_Extraction 这条线上用到的几个函数
    （fetch_successful_submission_dataframe_from_db /
    build_resume_extraction_dataframes / ResumeExtractionInjector）
    都只依赖对象有一个 _execute(sql, params) 方法，不关心这个对象
    具体叫什么类名，所以这里单独定义一个最简版本就够用，
    不需要借用 RawSubmissionLoader 那个功能更多、但依赖更多的类。
    """

    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"
        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])
        return True, data.get("result", [])


loader = D1Client(ACCOUNT_ID, DATABASE_ID, API_TOKEN)

In [ ]:
# /content/drive/MyDrive/recruiting-pipeline/create.sql
# /content/drive/MyDrive/recruiting-pipeline/drop.sql

### delete table

In [ ]:
# Delete table
"""
reset_d1.py (Google Colab version)
Drop all tables in the D1 database using drop.sql stored on a shared Google Drive folder.

Usage (in a Colab cell, call directly, no command line args):
    resetter = D1Resetter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    resetter.run(DROP_SQL_PATH)
"""

import re
import time
import os
import requests
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
CREATE_SQL_PATH = f"{DRIVE_FOLDER}/create.sql"
DROP_SQL_PATH = f"{DRIVE_FOLDER}/drop.sql"


class D1Resetter:
    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    def _execute(self, sql: str, timeout: int = 30):
        payload = {"sql": sql}
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"

        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])

        return True, data.get("result", [])

    def _split_sql_statements(self, sql_text: str):
        lines = []
        for line in sql_text.splitlines():
            if line.strip().startswith("--"):
                continue
            lines.append(line)
        cleaned = "\n".join(lines)

        statements = []
        current = []
        in_single_quote = False
        in_double_quote = False

        for ch in cleaned:
            if ch == "'" and not in_double_quote:
                in_single_quote = not in_single_quote
            elif ch == '"' and not in_single_quote:
                in_double_quote = not in_double_quote

            current.append(ch)

            if ch == ";" and not in_single_quote and not in_double_quote:
                stmt = "".join(current).strip()
                if stmt and stmt != ";":
                    statements.append(stmt)
                current = []

        tail = "".join(current).strip()
        if tail:
            statements.append(tail)

        return statements

    def list_existing_tables(self):
        ok, result = self._execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;"
        )
        if not ok:
            print(f"Failed to list tables: {result}")
            return []

        rows = result[0].get("results", []) if result else []
        table_names = [r["name"] for r in rows]

        print(f"Database currently has {len(table_names)} tables:")
        for i, name in enumerate(table_names, 1):
            print(f"  {i}. {name}")

        return table_names

    def run(self, sql_path: str):
        if not os.path.exists(sql_path):
            print(f"File not found: {sql_path}")
            print("Check that Google Drive is mounted and the path/filename is correct.")
            return False, []

        with open(sql_path, "r", encoding="utf-8") as f:
            sql_text = f.read()

        statements = self._split_sql_statements(sql_text)
        print(f"Starting execution of [{sql_path}], {len(statements)} statements total\n")

        success_count = 0
        failed = []

        for i, stmt in enumerate(statements, start=1):
            summary = re.sub(r"\s+", " ", stmt)[:70]
            print(f"  [{i}/{len(statements)}] {summary}...", end=" ")

            ok, result = self._execute(stmt)
            if ok:
                print("OK")
                success_count += 1
            else:
                print("FAILED")
                print(f"      error detail: {result}")
                failed.append((i, stmt, result))

            time.sleep(0.2)

        print(f"\nExecution finished: {success_count} / {len(statements)} succeeded")

        if failed:
            fail_file = f"failed_{os.path.basename(sql_path)}"
            with open(fail_file, "w", encoding="utf-8") as f:
                for _, stmt, _ in failed:
                    f.write(stmt + "\n\n")
            print(f"    Failed statements saved to {fail_file}")

        print("\nVerifying remaining tables in the database:")
        self.list_existing_tables()

        return (len(failed) == 0), failed


# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

print(os.path.exists(DROP_SQL_PATH))
#
resetter = D1Resetter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
ok, failed = resetter.run(DROP_SQL_PATH)


### create table

In [ ]:
"""
import_schema.py (Google Colab version)
Import the full schema using create.sql stored on a shared Google Drive folder.

Usage (in a Colab cell, call directly, no command line args):
    importer = D1SchemaImporter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    importer.run(CREATE_SQL_PATH)
"""

import re
import time
import os
import requests
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
CREATE_SQL_PATH = f"{DRIVE_FOLDER}/create.sql"
DROP_SQL_PATH = f"{DRIVE_FOLDER}/drop.sql"


class D1SchemaImporter:
    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    def _execute(self, sql: str, timeout: int = 30):
        payload = {"sql": sql}
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"

        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])

        return True, data.get("result", [])

    def _split_sql_statements(self, sql_text: str):
        lines = []
        for line in sql_text.splitlines():
            if line.strip().startswith("--"):
                continue
            lines.append(line)
        cleaned = "\n".join(lines)

        statements = []
        current = []
        in_single_quote = False
        in_double_quote = False

        for ch in cleaned:
            if ch == "'" and not in_double_quote:
                in_single_quote = not in_single_quote
            elif ch == '"' and not in_single_quote:
                in_double_quote = not in_double_quote

            current.append(ch)

            if ch == ";" and not in_single_quote and not in_double_quote:
                stmt = "".join(current).strip()
                if stmt and stmt != ";":
                    statements.append(stmt)
                current = []

        tail = "".join(current).strip()
        if tail:
            statements.append(tail)

        return statements

    def list_existing_tables(self):
        ok, result = self._execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;"
        )
        if not ok:
            print(f"Failed to list tables: {result}")
            return []

        rows = result[0].get("results", []) if result else []
        table_names = [r["name"] for r in rows]

        print(f"Database now has {len(table_names)} tables:")
        for i, name in enumerate(table_names, 1):
            print(f"  {i}. {name}")

        return table_names

    def run(self, sql_path: str):
        if not os.path.exists(sql_path):
            print(f"File not found: {sql_path}")
            print("Check that Google Drive is mounted and the path/filename is correct.")
            return False, [], []

        with open(sql_path, "r", encoding="utf-8") as f:
            sql_text = f.read()

        statements = self._split_sql_statements(sql_text)
        print(f"Starting execution of [{sql_path}], {len(statements)} statements total\n")

        success_count = 0
        failed = []

        for i, stmt in enumerate(statements, start=1):
            summary = re.sub(r"\s+", " ", stmt)[:70]
            print(f"  [{i}/{len(statements)}] {summary}...", end=" ")

            ok, result = self._execute(stmt)
            if ok:
                print("OK")
                success_count += 1
            else:
                print("FAILED")
                print(f"      error detail: {result}")
                failed.append((i, stmt, result))

            time.sleep(0.2)

        print(f"\nExecution finished: {success_count} / {len(statements)} succeeded")

        if failed:
            fail_file = f"failed_{os.path.basename(sql_path)}"
            with open(fail_file, "w", encoding="utf-8") as f:
                for _, stmt, _ in failed:
                    f.write(stmt + "\n\n")
            print(f"    Failed statements saved to {fail_file}, fix and re-run them separately.")
        else:
            print("All statements executed successfully.")

        print("\nVerifying tables now present in the database:")
        table_names = self.list_existing_tables()

        return (len(failed) == 0), failed, table_names


# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

print(os.path.exists(CREATE_SQL_PATH))
#
importer = D1SchemaImporter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
ok, failed, tables = importer.run(CREATE_SQL_PATH)


# Reference table insert


### load company

In [ ]:
"""
linkedin_importer.py（Google Colab 版，适配重构后的新 schema）
把爬虫爬下来的 linkedin_jobs_english.csv 导入新版 D1 数据库的 Position 相关表。

和旧版的关键差异：
  Position 表不再直接存薪资字段（salary_min/salary_max/currency/salary_type
  已从 Position 移除），薪资拆到独立的 Position_salary_range 表，
  金额改成"分"为单位的整数（salary_min_cents/salary_max_cents），
  避免浮点误差——这是重构时按审查意见定下的规则。

其余设计原则不变（呼应之前几轮讨论的决定）：
1. 职位标题（Job Title）、职能分类（Job Function）均不做标准化匹配，
   Job Function 原始文本直接作为 Function.function_name（get_or_create），
   不做拆分/语义判断。
2. Company 按名称精确匹配/新建，不做模糊去重。
3. Seniority Level / Employment Type 按值查找/新建对应维度表的行。
4. Salary 通过正则从 Detailed JD 里尽力提取，提取不到就留空。
5. Location 尽力解析成 City/State/Country，解析不出来留空（location_id = NULL）。
6. Position.function_id 是 NOT NULL 约束，Job Function 为空时退回占位符
   "Unclassified"，只是为了满足约束，不是在做分类判断。

用法（Colab cell 里直接调用，不用命令行参数）：
    importer = LinkedInJobImporter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    importer.import_csv(CSV_PATH)
"""

import re
import csv
import time
import requests
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount('/content/drive')

# ============================================================
# 连接配置
# ============================================================
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline/datasets"
CSV_PATH = f"{DRIVE_FOLDER}/linkedin_jobs_english.csv"


# ============================================================
# 美国州名缩写对照表（Location 解析用）
# ============================================================
US_STATE_ABBR = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia',
}


class PositionJDBuilder:
    """
    position_jd 只存原始 Detailed JD。Job Function 不拼进文本里，
    走 get_or_create_function 单独处理成真实的 function_id。
    """

    def build(self, job_function: str, detailed_jd: str) -> str:
        return (detailed_jd or "").strip()

    def build_from_row(self, row: dict) -> str:
        return self.build(
            job_function=row.get("Job Function", ""),
            detailed_jd=row.get("Detailed JD", ""),
        )


class SalaryExtractor:
    """从 JD 自由文本里尽力提取薪资范围，提取不到返回 None。"""

    _salary_keywords = re.compile(
        r'salary|compensation|base pay|/hour|per hour|pay range|hourly rate',
        re.IGNORECASE
    )
    _range_pattern = re.compile(
        r'\$([\d,]+(?:\.\d{1,2})?)\s*(?:-|to|and)\s*\$([\d,]+(?:\.\d{1,2})?)\s*'
        r'(/\s*hour|per\s*hour|hourly|an?\s*hour|/\s*hr)?',
        re.IGNORECASE
    )

    def extract(self, jd_text: str):
        if not jd_text:
            return None

        matches = list(self._range_pattern.finditer(jd_text))
        if not matches:
            return None

        m = matches[0]
        min_raw, max_raw, hourly_flag = m.group(1), m.group(2), m.group(3)

        try:
            salary_min = float(min_raw.replace(",", ""))
            salary_max = float(max_raw.replace(",", ""))
        except ValueError:
            return None

        if salary_min >= salary_max:
            return None

        salary_type = "hourly" if hourly_flag else "annual"

        context_window = jd_text[max(0, m.start() - 80):m.end() + 80]
        confidence = "high" if self._salary_keywords.search(context_window) else "low"

        if salary_type == "annual" and salary_max < 1000:
            confidence = "low"

        if salary_type == "annual" and (salary_min > 2_000_000 or salary_max > 2_000_000):
            return None
        if salary_type == "hourly" and (salary_min > 500 or salary_max > 500):
            return None

        return {
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_type": salary_type,
            "currency": "USD",
            "confidence": confidence,
        }


class LocationParser:
    """
    尽力把 Location free text 解析成 (city_name, state_name, country_name)。
    解析不出来返回 None，不强行拆。
    """

    def parse(self, location_text: str):
        if not location_text:
            return None
        text = location_text.strip()

        if text.lower() == "united states":
            return {"city": None, "state": None, "country": "United States"}

        m = re.match(r'^(.+?),\s*([A-Z]{2})$', text)
        if m:
            city, abbr = m.group(1).strip(), m.group(2).strip()
            state = US_STATE_ABBR.get(abbr)
            if state:
                return {"city": city, "state": state, "country": "United States"}

        m = re.match(r'^(.+?),\s*United States$', text, re.IGNORECASE)
        if m:
            first_part = m.group(1).strip()
            if first_part in US_STATE_ABBR.values():
                return {"city": None, "state": first_part, "country": "United States"}
            return {"city": first_part, "state": None, "country": "United States"}

        if re.search(r'metropolitan area|-.*area$', text, re.IGNORECASE):
            return {"city": text, "state": None, "country": "United States"}

        return None


class LinkedInJobImporter:
    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

        self.jd_builder = PositionJDBuilder()
        self.salary_extractor = SalaryExtractor()
        self.location_parser = LocationParser()

        self._company_cache = {}
        self._seniority_cache = {}
        self._employment_type_cache = {}
        self._function_cache = {}
        self._country_cache = {}
        self._state_cache = {}
        self._city_cache = {}
        self._placeholder_function_id = None

        self.stats = {
            "total": 0,
            "imported": 0,
            "skipped_no_title": 0,
            "skipped_no_company": 0,
            "location_parsed": 0,
            "location_skipped": 0,
            "salary_extracted": 0,
            "errors": [],
        }

    # ------------------------------------------------------------
    # 底层 API 调用
    # ------------------------------------------------------------
    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params

        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"

        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])

        return True, data.get("result", [])

    @staticmethod
    def _now_iso():
        return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    def _get_or_create(self, table: str, name_column: str, name_value: str, cache: dict):
        """
        通用的"查找，找不到就新建"逻辑，返回 id。
        table/name_column 由调用方保证是可信的固定字符串（不是用户输入），
        不走参数化（因为 SQLite 不支持表名/列名参数化），name_value 走参数化。
        """
        key = name_value.strip().lower()
        if key in cache:
            return cache[key]

        ok, result = self._execute(
            f'SELECT "id" FROM "{table}" WHERE LOWER("{name_column}") = ?;',
            [key],
        )
        if not ok:
            raise RuntimeError(f"Query {table} failed: {result}")

        rows = result[0].get("results", []) if result else []
        if rows:
            cache[key] = rows[0]["id"]
            return cache[key]

        now = self._now_iso()
        ok, result = self._execute(
            f'INSERT INTO "{table}" ("{name_column}", "created_at", "updated_at") '
            f'VALUES (?, ?, ?) RETURNING "id";',
            [name_value.strip(), now, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into {table} failed: {result}")

        rows = result[0].get("results", []) if result else []
        new_id = rows[0]["id"]
        cache[key] = new_id
        return new_id

    # ------------------------------------------------------------
    # 各维度表的查找/新建
    # ------------------------------------------------------------
    def get_or_create_company(self, company_name: str):
        return self._get_or_create("Company", "company_name", company_name, self._company_cache)

    def get_or_create_seniority(self, seniority_name: str):
        seniority_name = (seniority_name or "").strip()
        if not seniority_name or seniority_name.lower() == "not applicable":
            return None
        return self._get_or_create("Seniority", "seniority_name", seniority_name, self._seniority_cache)

    def get_or_create_employment_type(self, employment_type_name: str):
        employment_type_name = (employment_type_name or "").strip()
        if not employment_type_name:
            return None
        return self._get_or_create(
            "Position_employment_type", "employment_type_name",
            employment_type_name, self._employment_type_cache
        )

    def get_placeholder_function_id(self):
        if self._placeholder_function_id is not None:
            return self._placeholder_function_id
        self._placeholder_function_id = self._get_or_create(
            "Function", "function_name", "Unclassified", {}
        )
        return self._placeholder_function_id

    def get_or_create_function(self, job_function_raw: str):
        job_function_raw = (job_function_raw or "").strip()
        if not job_function_raw:
            return self.get_placeholder_function_id()
        return self._get_or_create(
            "Function", "function_name", job_function_raw, self._function_cache
        )

    def resolve_location_id(self, location_text: str):
        parsed = self.location_parser.parse(location_text)
        if parsed is None:
            self.stats["location_skipped"] += 1
            return None

        country_id = self._get_or_create("Country", "country_name", parsed["country"], self._country_cache)

        state_id = None
        if parsed["state"]:
            state_id = self._get_or_create("State", "state_name", parsed["state"], self._state_cache)

        city_id = None
        if parsed["city"]:
            city_id = self._get_or_create("City", "city_name", parsed["city"], self._city_cache)

        now = self._now_iso()
        ok, result = self._execute(
            'INSERT INTO "Location" ("country_id", "state_id", "city_id", "created_at", "updated_at") '
            'VALUES (?, ?, ?, ?, ?) RETURNING "id";',
            [country_id, state_id, city_id, now, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into Location failed: {result}")

        rows = result[0].get("results", []) if result else []
        self.stats["location_parsed"] += 1
        return rows[0]["id"]

    def insert_position_salary_range(self, position_id: int, salary_info: dict):
        """
        新表：Position_salary_range（原来的薪资字段已从 Position 移除）。
        金额转成"分"为单位的整数（乘以100，四舍五入），避免浮点误差。
        """
        now = self._now_iso()
        salary_min_cents = round(salary_info["salary_min"] * 100)
        salary_max_cents = round(salary_info["salary_max"] * 100)

        sql = """
            INSERT INTO "Position_salary_range" (
              "position_id", "salary_min_cents", "salary_max_cents",
              "currency", "salary_type", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?);
        """
        params = [
            position_id, salary_min_cents, salary_max_cents,
            salary_info["currency"], salary_info["salary_type"],
            now, now,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Position_salary_range failed: {result}")

    # ------------------------------------------------------------
    # 单行导入
    # ------------------------------------------------------------
    def import_row(self, row: dict):
        job_title = (row.get("Job Title") or "").strip()
        company_name = (row.get("Company") or "").strip()

        if not job_title:
            self.stats["skipped_no_title"] += 1
            return None
        if not company_name:
            self.stats["skipped_no_company"] += 1
            return None

        company_id = self.get_or_create_company(company_name)
        function_id = self.get_or_create_function(row.get("Job Function"))
        seniority_id = self.get_or_create_seniority(row.get("Seniority Level"))
        employment_type_id = self.get_or_create_employment_type(row.get("Employment Type"))
        location_id = self.resolve_location_id(row.get("Location"))

        position_jd = self.jd_builder.build_from_row(row)

        salary_info = self.salary_extractor.extract(row.get("Detailed JD", ""))
        if salary_info:
            self.stats["salary_extracted"] += 1

        now = self._now_iso()

        # Position 表不再包含薪资字段（拆到 Position_salary_range）
        sql = """
            INSERT INTO "Position" (
              "company_id", "position_name", "position_jd",
              "employment_type_id", "function_id", "seniority_id",
              "location_id", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            RETURNING "id";
        """
        params = [
            company_id, job_title, position_jd,
            employment_type_id, function_id, seniority_id,
            location_id, now, now,
        ]

        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Position failed: {result}")

        rows = result[0].get("results", []) if result else []
        position_id = rows[0]["id"]

        if salary_info:
            self.insert_position_salary_range(position_id, salary_info)

        return position_id

    # ------------------------------------------------------------
    # 批量导入入口
    # ------------------------------------------------------------
    def import_csv(self, csv_path: str, limit: int = None, verbose: bool = True):
        with open(csv_path, encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        if limit:
            rows = rows[:limit]

        self.stats["total"] = len(rows)
        print(f"Starting import, {len(rows)} rows total")

        for i, row in enumerate(rows, start=1):
            try:
                new_id = self.import_row(row)
                if new_id is not None:
                    self.stats["imported"] += 1
                    if verbose and i % 50 == 0:
                        print(f"  processed {i}/{len(rows)} rows...")
            except Exception as e:
                self.stats["errors"].append((i, str(e)))
                print(f"  row {i} failed: {e}")

            time.sleep(0.05)

        self._print_summary()
        return self.stats

    def _print_summary(self):
        print("\n" + "=" * 60)
        print("Import summary")
        print("=" * 60)
        print(f"  total rows            : {self.stats['total']}")
        print(f"  imported              : {self.stats['imported']}")
        print(f"  skipped (no title)    : {self.stats['skipped_no_title']}")
        print(f"  skipped (no company)  : {self.stats['skipped_no_company']}")
        print(f"  location parsed       : {self.stats['location_parsed']}")
        print(f"  location skipped      : {self.stats['location_skipped']}")
        print(f"  salary extracted      : {self.stats['salary_extracted']}")
        print(f"  errored rows          : {len(self.stats['errors'])}")


# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

importer = LinkedInJobImporter(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
#
stats = importer.import_csv(CSV_PATH, limit=20)
#
stats = importer.import_csv(CSV_PATH)


In [ ]:
ok, result = importer._execute("SELECT id, company_name FROM Company LIMIT 10;")
print(result)

ok, result = importer._execute("SELECT id, company_id, position_name FROM Position LIMIT 10;")
print(result)

ok, result = importer._execute("SELECT COUNT(*) as total FROM Company;")
print(result)

ok, result = importer._execute("SELECT COUNT(*) as total FROM Position;")
print(result)

### Skill table

In [ ]:
"""
skill_reference_loader.py（Google Colab 版）
把三份技能参考数据（Language_Skills.csv / Technical_Skills.csv /
Function__Skills.csv）导入 Skill 表，统一用 skill_type 区分来源类别，
不单独建桥表。

设计要点：
1. 三份数据分别对应三个 skill_type：Language / Technology / Function
   （Skill_type 表里的 "Function" 只是一个分类标签字符串，
   和 Employment_group 里那张独立的 Function 表没有关系，
   两者是同名但完全不同的两个东西，不要混淆）。
2. Function__Skills.csv 的 Group 列（原本代表"这个技能常见于哪个职能"）
   直接忽略不导入，只取 Skill 这一列的技能名称。
3. 去重范围是"同一个技能名 + 同一个 skill_type"这个组合，不是单纯按
   名字去重——所以同一个名字如果出现在不同的 skill_type 下
   （比如 Technical 和 Function 两份数据里都有"ABAP"），会分别存成
   两条独立的记录，代表两个不同语境下的技能，不会被合并成一条。
   但同一份文件内部，同名的技能只会插入一次。

用法（Colab cell 里直接调用，不用命令行参数）：
    loader = SkillReferenceLoader(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    loader.load_all(LANGUAGE_CSV_PATH, TECHNICAL_CSV_PATH, FUNCTION_SKILLS_CSV_PATH)
"""

import csv
import time
import requests
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
LANGUAGE_CSV_PATH = f"{DRIVE_FOLDER}/datasets/Language_Skills.csv"
TECHNICAL_CSV_PATH = f"{DRIVE_FOLDER}/datasets/Technical_Skills.csv"
FUNCTION_SKILLS_CSV_PATH = f"{DRIVE_FOLDER}/datasets/Function_Skills.csv"


class SkillReferenceLoader:
    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

        # 缓存 skill_type_name -> skill_type_id，避免重复查询
        self._skill_type_cache = {}
        # 缓存 (skill_type_id, skill_name小写) -> skill_id，
        # key 里带上 skill_type_id，保证跨类型同名技能各自独立缓存
        self._skill_cache = {}

    # ------------------------------------------------------------
    # 底层 API 调用：不设固定请求间隔，只有报错时才重试，
    # 且只针对这一次报错的请求重试，成功的请求之间不等待
    # ------------------------------------------------------------
    def _execute(self, sql: str, params: list = None, timeout: int = 30,
                 max_retries: int = 3, retry_delay_seconds: float = 1.0):
        last_error = None

        for attempt in range(1, max_retries + 1):
            payload = {"sql": sql}
            if params is not None:
                payload["params"] = params

            try:
                resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
            except requests.exceptions.RequestException as e:
                # 网络层面的异常（超时、连接中断等），记下来，准备重试
                last_error = f"Network error: {e}"
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            try:
                data = resp.json()
            except ValueError:
                last_error = f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            if resp.status_code != 200 or not data.get("success", False):
                last_error = data.get("errors", [])
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            # 成功：直接返回，不额外等待，不管这是第几次尝试
            return True, data.get("result", [])

        return False, last_error

    @staticmethod
    def _now_iso():
        return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    # ------------------------------------------------------------
    # Skill_type：先确保分类本身存在
    # ------------------------------------------------------------
    def get_or_create_skill_type(self, skill_type_name: str):
        key = skill_type_name.strip().lower()
        if key in self._skill_type_cache:
            return self._skill_type_cache[key]

        ok, result = self._execute(
            'SELECT "id" FROM "Skill_type" WHERE LOWER("skill_type_name") = ?;',
            [key],
        )
        if not ok:
            raise RuntimeError(f"Query Skill_type failed: {result}")

        rows = result[0].get("results", []) if result else []
        if rows:
            self._skill_type_cache[key] = rows[0]["id"]
            return self._skill_type_cache[key]

        now = self._now_iso()
        ok, result = self._execute(
            'INSERT INTO "Skill_type" ("skill_type_name", "created_at", "updated_at") '
            'VALUES (?, ?, ?) RETURNING "id";',
            [skill_type_name.strip(), now, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into Skill_type failed: {result}")

        rows = result[0].get("results", []) if result else []
        new_id = rows[0]["id"]
        self._skill_type_cache[key] = new_id
        return new_id

    # ------------------------------------------------------------
    # Skill：同一 skill_type 下按名字去重，不同 skill_type 之间
    # 即使同名也各自独立存储
    # ------------------------------------------------------------
    def get_or_create_skill(self, skill_name: str, skill_type_id: int):
        cache_key = (skill_type_id, skill_name.strip().lower())
        if cache_key in self._skill_cache:
            return self._skill_cache[cache_key]

        # 查询时同时按 skill_name 和 skill_type_id 两个条件匹配，
        # 保证不会跨类型误判成"已存在"
        ok, result = self._execute(
            'SELECT "id" FROM "Skill" WHERE LOWER("skill_name") = ? AND "skill_type_id" = ?;',
            [skill_name.strip().lower(), skill_type_id],
        )
        if not ok:
            raise RuntimeError(f"Query Skill failed: {result}")

        rows = result[0].get("results", []) if result else []
        if rows:
            self._skill_cache[cache_key] = rows[0]["id"]
            return self._skill_cache[cache_key]

        now = self._now_iso()
        ok, result = self._execute(
            'INSERT INTO "Skill" ("skill_name", "skill_type_id", "created_at", "updated_at") '
            'VALUES (?, ?, ?, ?) RETURNING "id";',
            [skill_name.strip(), skill_type_id, now, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into Skill failed: {result}")

        rows = result[0].get("results", []) if result else []
        new_id = rows[0]["id"]
        self._skill_cache[cache_key] = new_id
        return new_id

    # ------------------------------------------------------------
    # 单份文件的导入逻辑
    # ------------------------------------------------------------
    def _load_single_column_csv(self, csv_path: str, name_column: str,
                                 skill_type_name: str, verbose: bool = True):
        with open(csv_path, encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        skill_type_id = self.get_or_create_skill_type(skill_type_name)

        inserted = 0
        reused = 0
        skipped_empty = 0
        errors = []

        for i, row in enumerate(rows, start=1):
            skill_name = (row.get(name_column) or "").strip()
            if not skill_name:
                skipped_empty += 1
                continue

            cache_key = (skill_type_id, skill_name.lower())
            already_cached = cache_key in self._skill_cache

            try:
                # get_or_create_skill 内部的 _execute 已经带了失败重试逻辑，
                # 这里只需要处理"重试次数用完、仍然失败"这种彻底失败的情况——
                # 记下这一行，继续处理下一行，不让整个批次因为一行数据崩掉
                self.get_or_create_skill(skill_name, skill_type_id)
            except Exception as e:
                errors.append((i, skill_name, str(e)))
                print(f"    row {i} ({skill_name!r}) failed after retries: {e}")
                continue

            if already_cached:
                reused += 1
            else:
                inserted += 1

            if verbose and i % 500 == 0:
                print(f"    processed {i}/{len(rows)} rows...")

        print(f"  [{skill_type_name}] total={len(rows)} inserted={inserted} "
              f"reused(within-file duplicates)={reused} skipped_empty={skipped_empty} "
              f"errored={len(errors)}")

        return {
            "total": len(rows), "inserted": inserted, "reused": reused,
            "skipped_empty": skipped_empty, "errors": errors,
        }

    def load_language_csv(self, csv_path: str):
        print("Loading Language_Skills.csv ...")
        return self._load_single_column_csv(csv_path, name_column="Language", skill_type_name="Language")

    def load_technical_csv(self, csv_path: str):
        print("Loading Technical_Skills.csv ...")
        return self._load_single_column_csv(csv_path, name_column="Technical", skill_type_name="Technology")

    def load_function_skills_csv(self, csv_path: str):
        # 注意：这里的 skill_type_name="Function" 只是一个分类标签，
        # Group 这一列本身完全不读取、不导入
        print("Loading Function__Skills.csv (Group column ignored) ...")
        return self._load_single_column_csv(csv_path, name_column="Skill", skill_type_name="Function")

    def load_all(self, language_csv_path: str, technical_csv_path: str, function_skills_csv_path: str):
        results = {}
        results["language"] = self.load_language_csv(language_csv_path)
        results["technical"] = self.load_technical_csv(technical_csv_path)
        results["function"] = self.load_function_skills_csv(function_skills_csv_path)

        print("\n" + "=" * 60)
        print("Load all summary")
        print("=" * 60)
        for name, stats in results.items():
            print(f"  {name}: {stats}")

        return results


# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

loader = SkillReferenceLoader(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
results = loader.load_all(LANGUAGE_CSV_PATH, TECHNICAL_CSV_PATH, FUNCTION_SKILLS_CSV_PATH)


# Raw submission

In [ ]:
"""
raw_submission_loader.py（Google Colab 版）

把原始投递数据（不管来自CSV批量导入还是表单实时提交）原样打包落地到
raw_submission 表，不做任何解析/清洗/去重过滤——这一层的职责就是
"如实收下"，解析和判重是后面独立的步骤（Submission_normalized /
Submission_dedup_run），不在这里做。

设计要点：
1. 每一行原始数据，按"字段名: 值"一行一个的方式拼成一段 UTF-8 纯文本，
   存进 raw_record_jsonb 这一列（字段名沿用原来的叫法，但存储内容
   已经不是 JSON 转义格式，而是能够原样保留空格和换行的utf-8文本，
   这样不会像 JSON 转义那样吞掉原始格式细节）。
2. raw_record_jsonb 主要用于：审计原始提交，追溯某次 ingestion，重新运行独立的处理任务，
   CSV 文件未来无法访问时保留原始内容，核对 normalization 的来源，根据原始哈希检查完全重复记录。

3. raw_record_jsonb_sha256 只是算出来存着，供后续查重步骤使用，
   这个 Raw submission section 里面并没有实现因为哈希重复就拒绝插入记录/行 —— CSV 里出现的65条完全重复的
   记录/行，也会照常插入65次。
4. 每次导入先新建一条 Ingestion_batch 记录，代表"这一批原始数据的导入记录"，
   每条 raw_submission 都归属于某一批原始数据的导入记录，方便后续追溯问题批次。
5. 所有表插入时，除了主键和 NOT NULL 约束的字段以外，
   凡是这次没有拿到实际值的字段，一律显式传 None（对应数据库里的 NULL），
   不会因为"漏传"而让数据库用它自己的默认值。

关于 id 主键：
   本项目所有表的 id 字段，建表时都定义成 INTEGER PRIMARY KEY AUTOINCREMENT，
   插入新记录时完全不需要我们自己指定 id 的值，D1（底层是 SQLite）
   会在插入那一刻自动分配一个新的、唯一递增的整数，我们只需要在
   INSERT 语句末尾加 RETURNING "id"，就能立刻拿到这个自动生成的值。
   唯一的例外是 Dataset_type 表：它的主键是 type_code（一个字符串），
   不是 id，这个值不会自动生成，必须由我们自己的代码算出来再插入
   （具体做法见下面的 _slugify_type_code 方法）。

用法（Colab cell 里直接调用，不用命令行参数）：
    loader = RawSubmissionLoader(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    loader.seed_dataset_types()          # 先确保三个 Dataset_type 都存在
    stats = loader.load_csv(CSV_PATH)    # 再导入 CSV
"""


import csv

import json

import time
import hashlib
import os
import re
import requests
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"


# 每次运行时都要求用户明确输入本批次实际使用的原始 CSV 文件名。
#
# 这样后续无论新批次包含 10 条、100 条还是更多记录，
# 都不需要重新修改 Cell 中的固定文件名。
raw_csv_file_name = input(
    "请输入本批次原始 CSV 文件名（例如 test_raw_submission_10.csv）："
).strip()

# 没有输入文件名时立即停止，防止意外读取旧批次文件。
if not raw_csv_file_name:
    raise ValueError(
        "本批次原始 CSV 文件名不能为空。"
    )

# 这里只接受文件名，不接受包含目录分隔符的路径，
# 防止输入内容跳出固定 datasets 文件夹。
if os.path.basename(raw_csv_file_name) != raw_csv_file_name:
    raise ValueError(
        "请输入单独的 CSV 文件名，不要输入完整路径。"
    )

# 本 Cell 只接受 CSV 文件。
if not raw_csv_file_name.lower().endswith(".csv"):
    raise ValueError(
        "本批次原始数据文件必须是 .csv 文件。"
    )

# 根据用户输入动态生成本批次原始 CSV 的完整路径。
CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/{raw_csv_file_name}"
)


class RawSubmissionLoader:
    def __init__(self, account_id: str, database_id: str, api_token: str):
        # 拼出 D1 REST API 的完整请求地址，后面每次发请求都用这一个地址
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        # 每次请求都要带的请求头：鉴权信息 + 声明请求体是 JSON 格式
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    # ------------------------------------------------------------
    # 底层 API 调用：所有 INSERT/SELECT/UPDATE 最终都走这一个方法
    # ------------------------------------------------------------
    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        # 把 SQL 语句打包成请求体；如果有参数化占位符 "?"，
        # 对应的值放在 params 这个列表里，不直接拼进 sql 字符串
        # （防止 SQL 注入，也是 D1 官方推荐的参数化查询方式）
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params

        # 真正发出 HTTP POST 请求，把 payload 作为请求体发给 D1
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)

        # 尝试把返回内容解析成 JSON；如果连 JSON 都不是（比如网络中间层
        # 返回了一个 HTML 错误页），直接判定失败，把原始文本截一段返回
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"

        # D1 返回的 JSON 里如果 success 字段不是 True，也判定失败，
        # 把 errors 字段（D1 给出的具体错误信息）返回给调用方
        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])

        # 成功的话，把 result 字段（真正的数据/受影响行数等信息）返回
        return True, data.get("result", [])

    @staticmethod
    def _now_iso():
        # 生成当前的 UTC 时间，格式化成 ISO8601 字符串
        # （例如 "2026-07-15T20:59:26Z"），所有 created_at/updated_at
        # 之类的时间字段统一用这个格式存储
        return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    # ------------------------------------------------------------
    # Dataset_type：type_code 是主键，不会自动生成，
    # 需要我们自己从 display_type_name 推导出来
    # ------------------------------------------------------------
    @staticmethod
    def _slugify_type_code(display_type_name: str) -> str:
        """
        把人类可读的 display_type_name（比如 "submission_w/_resume"）
        转换成适合当主键用的 type_code：
          1. 全部转小写
          2. 把所有"不是字母/数字/下划线"的字符（比如 "/"）替换成下划线
          3. 把连续多个下划线合并成一个，避免出现 "__" 这种难看的结果
          4. 去掉开头/结尾多余的下划线
        这样处理完，同一个 display_type_name 每次生成的 type_code
        都是完全一样的、可预测的，不需要额外的映射表去记"谁对应谁"。
        """
        # 第一步：转小写
        lowered = display_type_name.strip().lower()
        # 第二步：非字母数字下划线的字符，统一替换成下划线
        replaced = re.sub(r'[^a-z0-9_]+', '_', lowered)
        # 第三步：多个连续下划线合并成一个
        collapsed = re.sub(r'_+', '_', replaced)
        # 第四步：去掉首尾多余的下划线
        return collapsed.strip('_')

    def get_or_create_dataset_type(self, display_type_name: str,
                                    type_description: str = None,
                                    contains_pii=None):
        """
        确保某个 dataset_type 在 Dataset_type 维度表里真实存在。
        参数名和 Dataset_type 表里的字段名保持一致（display_type_name /
        type_description / contains_pii），方便一眼看出对应关系。

        contains_pii 现在按你的要求，测试阶段统一传 None（存成 NULL），
        不做 True/False 判断——注意这需要数据库那边已经跑过
        alter_dataset_type_contains_pii.sql，把这一列的 NOT NULL
        约束放宽掉，不然这里会插入失败。

        返回值：这个 dataset_type 对应的 type_code（不管是刚新建的，
        还是之前已经存在、这次只是查到的）。
        """
        # 先根据 display_type_name 算出确定性的 type_code
        type_code = self._slugify_type_code(display_type_name)

        # 查一下这个 type_code 是不是已经存在，存在就直接复用，不重复插入
        ok, result = self._execute(
            'SELECT "type_code" FROM "Dataset_type" WHERE "type_code" = ?;',
            [type_code],
        )
        if not ok:
            raise RuntimeError(f"Query Dataset_type failed: {result}")

        rows = result[0].get("results", []) if result else []
        if rows:
            # 已经存在，直接返回，不再往下插入新行
            return rows[0]["type_code"]

        # 不存在，准备插入一条新的 Dataset_type 记录
        created_at = self._now_iso()

        # contains_pii 在 SQLite 里没有原生的布尔类型，True/False 要
        # 分别转成整数 1/0 才能存；如果传进来的是 None，就原样存 None
        # （对应数据库里的 NULL），不强行转换成 0 或 1
        if contains_pii is None:
            contains_pii_value = None
        else:
            contains_pii_value = 1 if contains_pii else 0

        sql = """
            INSERT INTO "Dataset_type" (
              "type_code", "display_type_name", "type_description",
              "contains_pii", "is_active", "created_at"
            ) VALUES (?, ?, ?, ?, ?, ?)
            RETURNING "type_code";
        """
        # is_active 按你的要求，这批种子数据统一填 True（存成整数1）
        params = [
            type_code, display_type_name, type_description,
            contains_pii_value, 1, created_at,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Dataset_type failed: {result}")

        rows = result[0].get("results", []) if result else []
        return rows[0]["type_code"]

    def seed_dataset_types(self):
        """
        一次性确保当前阶段需要用到的三个 dataset_type 都已经存在：
          submission_w/_resume（本脚本导入的这份带简历文本的投递数据）
          job_posting（LinkedIn 爬虫的职位数据，供另一个导入脚本使用）
          company_info（未来的公司信息数据集，目前还没有对应的导入脚本）
        每一个都调用 get_or_create_dataset_type，已存在的会被跳过，
        不会重复插入。
        """
        display_type_names = ["submission_w/_resume", "job_posting", "company_info"]
        type_codes = {}
        for display_type_name in display_type_names:
            type_code = self.get_or_create_dataset_type(display_type_name)
            type_codes[display_type_name] = type_code
            print(f"  Dataset_type ready: display_type_name={display_type_name!r} -> type_code={type_code!r}")
        return type_codes

    # ------------------------------------------------------------
    # Ingestion_batch：记录"这一次批量导入"本身的执行过程
    # ------------------------------------------------------------
    def create_ingestion_batch(self, source_system: str, dataset_type: str,
                                source_schema_version: str, source_file_name: str,
                                source_object_key: str = None,
                                normalization_version: str = None,
                                total_row_count: int = None):
        """
        参数名对应 Ingestion_batch 表里的字段名：
          source_system            -> Ingestion_batch.source_system
          dataset_type              -> Ingestion_batch.dataset_type
                                        （存的是 Dataset_type.type_code 这个值）
          source_schema_version    -> Ingestion_batch.source_schema_version
          source_file_name          -> Ingestion_batch.source_file_name
          source_object_key         -> Ingestion_batch.source_object_key
          normalization_version     -> Ingestion_batch.normalization_version
          total_row_count            -> Ingestion_batch.total_row_count
        """
        imported_at = self._now_iso()
        created_at = imported_at

        sql = """
            INSERT INTO "Ingestion_batch" (
              "source_system", "dataset_type", "source_schema_version",
              "normalization_version", "source_file_name", "source_object_key",
              "imported_at", "import_status", "total_row_count", "created_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 'processing', ?, ?)
            RETURNING "id";
        """
        # 刚开始导入的时候，还不知道最后到底成功/失败几条，
        # 这些数字要等 finalize_ingestion_batch 跑完之后才能确定，
        # 这里先不传（数据库层面这几列本来也允许为空）
        params = [
            source_system, dataset_type, source_schema_version,
            normalization_version, source_file_name, source_object_key,
            imported_at, total_row_count, created_at,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Ingestion_batch failed: {result}")
        rows = result[0].get("results", []) if result else []
        return rows[0]["id"]

    def finalize_ingestion_batch(self, batch_id: int, successful_row_count: int,
                                  failed_row_count: int, skipped_row_count: int,
                                  file_sha256: str = None, errors: list = None):
        """
        导入循环全部跑完之后调用，把这一批执行的最终结果回填进
        Ingestion_batch 那一行，参数名同样对应表里的字段名。
        """
        # 只要有一条记录失败，状态就标成"completed_with_errors"，
        # 不是笼统地标 completed，方便一眼看出这一批是不是完全顺利
        import_status = "completed" if failed_row_count == 0 else "completed_with_errors"

        # error_summary 这个字段本身设计成存 JSON（不是 raw_record 那种
        # 需要保留原始格式的场景，这里只是记录"第几行、什么报错"这种
        # 结构化的排查信息，用 json.dumps 完全没问题）
        if errors:
            error_summary = json.dumps(
                [{"row": row_number, "error": str(error_message)}
                 for row_number, error_message in errors],
                ensure_ascii=False,
            )
        else:
            # 没有任何报错的话，显式存 None（对应NULL），不是空字符串
            error_summary = None

        sql = """
            UPDATE "Ingestion_batch"
            SET "import_status" = ?,
                "successful_row_count" = ?,
                "failed_row_count" = ?,
                "skipped_row_count" = ?,
                "file_sha256" = ?,
                "error_summary" = ?
            WHERE "id" = ?;
        """
        params = [
            import_status, successful_row_count, failed_row_count,
            skipped_row_count, file_sha256, error_summary, batch_id,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Failed to finalize Ingestion_batch {batch_id}: {result}")

    # ------------------------------------------------------------
    # raw_submission：把一整行原始数据原样落地，不做任何解析
    # ------------------------------------------------------------
    @staticmethod
    def _row_to_utf8_text(row: dict) -> str:
        """
        把一行原始数据（dict）转换成 UTF-8 纯文本块，格式是
        "字段名: 值" 一行一个，按字段名排序（保证同样的数据每次
        生成的文本完全一致，这样后面算哈希才有意义）。

        和用 json.dumps 相比，这里不对值做任何转义处理——原始值里
        本来是什么样的空格、换行、引号，就原样输出成什么样，
        不会被 JSON 的转义规则（比如把换行变成字面的 "\\n" 两个字符）
        改变掉，最大程度保留原始数据的样子。
        """
        lines = []
        # 按字段名排序，保证输出顺序稳定、可复现
        for field_name in sorted(row.keys()):
            field_value = row[field_name]
            # 值本身如果是 None（字段本来就是空的），显式写成空字符串，
            # 不是 Python 的 "None" 这几个字母
            if field_value is None:
                field_value = ""
            lines.append(f"{field_name}: {field_value}")
        # 用换行符把所有"字段名: 值"这些行拼成一整段文本
        return "\n".join(lines)

    def insert_raw_submission(self, ingestion_batch_id: int, source_record_number: int,
                               raw_row: dict, source_record_id: str = None):
        """
        参数名对应 raw_submission 表里的字段名：
          ingestion_batch_id    -> raw_submission.ingestion_batch_id
          source_record_number  -> raw_submission.source_record_number
          source_record_id      -> raw_submission.source_record_id
                                    （来源系统自带的记录ID，这份CSV没有，
                                     所以调用方不传的话默认就是None）
          raw_row（这行原始数据）  -> 加工后存进 raw_submission.raw_record_jsonb
        """
        # 把这一行原始数据转成 UTF-8 纯文本块（不是 JSON 转义格式）
        raw_record_text = self._row_to_utf8_text(raw_row)
        # 对这段文本算 sha256 哈希，用于后续查重步骤参考
        raw_record_jsonb_sha256 = hashlib.sha256(raw_record_text.encode("utf-8")).hexdigest()
        # 这条记录"落地"的时间，就是现在
        landed_at = self._now_iso()

        sql = """
            INSERT INTO "raw_submission" (
              "ingestion_batch_id", "source_record_number", "source_record_id",
              "raw_record_jsonb", "raw_record_jsonb_sha256",
              "processing_status", "processing_error", "landed_at"
            ) VALUES (?, ?, ?, ?, ?, 'pending', ?, ?)
            RETURNING "id";
        """
        # processing_error 这一列现在还没有任何处理发生（processing_status
        # 是刚插入的 'pending'），所以显式传 None，不是遗漏没处理这一列
        params = [
            ingestion_batch_id, source_record_number, source_record_id,
            raw_record_text, raw_record_jsonb_sha256,
            None, landed_at,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into raw_submission failed: {result}")

        rows = result[0].get("results", []) if result else []
        return rows[0]["id"]

    def update_processing_status(self, raw_submission_id: int, processing_status: str,
                                  processing_error: str = None):
        """
        更新某条 raw_submission 的处理状态，参数名对应表里的字段名。
        合法的 processing_status 值：pending / processing / normalized / failed
        （这个不是导入本身的状态，导入状态由 Ingestion_batch.import_status 追踪）

        processing_error：
          如果这次更新是因为处理失败（processing_status 传 'failed'），
          这里要传入具体的错误信息文本，会真正存进 processing_error 这一列；
          如果这次更新没有错误（比如成功变成 'normalized'），
          不传这个参数，默认 None，字段存 NULL，不会把无意义的空值
          或者占位字符串硬塞进去。

        真正调用这个方法的时机，是未来写"清洗/标准化"脚本的时候：
        每条 raw_submission 被解析、尝试生成对应的 Submission_normalized
        记录之后——
          解析成功：update_processing_status(id, "normalized")
          解析失败：update_processing_status(id, "failed", processing_error="具体报错内容")
        """
        valid_statuses = {"pending", "processing", "normalized", "failed"}
        if processing_status not in valid_statuses:
            raise ValueError(
                f"Invalid processing_status '{processing_status}', "
                f"must be one of {valid_statuses}"
            )

        sql = """
            UPDATE "raw_submission"
            SET "processing_status" = ?, "processing_error" = ?
            WHERE "id" = ?;
        """
        # processing_error 就是调用方传进来的值，可能是具体错误文本，
        # 也可能是默认的 None（对应存成 NULL），不做额外加工
        ok, result = self._execute(sql, [processing_status, processing_error, raw_submission_id])
        if not ok:
            raise RuntimeError(f"Failed to update raw_submission {raw_submission_id}: {result}")
        return True

    # ------------------------------------------------------------
    # 批量导入入口（历史CSV数据用）
    # ------------------------------------------------------------
    # resume 这一列（简历文件链接）经确认没有用，落地时排除掉，
    # 其余字段（包括已经拆好的 resume_text）原样保留，不做其他处理
    EXCLUDED_COLUMNS = {"resume"}

    def load_csv(self, csv_path: str, source_system: str = "manual_csv_upload",
                 display_type_name: str = "submission_w/_resume",
                 source_schema_version: str = "csv_v1",
                 limit: int = None, verbose: bool = True):
        # 先用二进制模式读一遍整份文件的原始字节，用来计算"整份文件"的
        # sha256 哈希（这个和后面每一行单独算的哈希是两回事，
        # 一个代表"这整份文件有没有变过"，一个代表"这一条记录长什么样"）。
        with open(csv_path, "rb") as f:
            raw_bytes = f.read()
        file_sha256 = hashlib.sha256(raw_bytes).hexdigest()

        # 再用文本模式重新打开一遍，交给 csv.DictReader 按表头解析成
        # 一行一个 dict 的列表，每个 dict 的 key 就是CSV表头的列名
        with open(csv_path, encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        # 如果调用方传了 limit（比如先跑10行测试），只取前面这些行
        if limit:
            rows = rows[:limit]

        # 把每一行里，属于 EXCLUDED_COLUMNS 的字段过滤掉（目前只有
        # "resume" 这一列，确认对我们没用的原始简历文件链接）
        rows = [
            {field_name: field_value for field_name, field_value in row.items()
             if field_name not in self.EXCLUDED_COLUMNS}
            for row in rows
        ]

        # 导入正式开始之前，先确保这次要用的 dataset_type 在
        # Dataset_type 维度表里已经存在，拿到对应的 type_code
        type_code = self.get_or_create_dataset_type(display_type_name)

        # 建一条 Ingestion_batch 记录，代表"这一次导入"，
        # source_object_key 直接存这次用的文件路径（本身就是这个"对象"
        # 在 Drive 里的位置，不需要额外再造一个 key）
        ingestion_batch_id = self.create_ingestion_batch(
            source_system=source_system,
            dataset_type=type_code,
            source_schema_version=source_schema_version,
            source_file_name=os.path.basename(csv_path),
            source_object_key=csv_path,
            total_row_count=len(rows),
        )
        print(f"Created Ingestion_batch id={ingestion_batch_id}")
        print(f"Starting load, {len(rows)} rows total\n")

        successful_row_count = 0
        errors = []

        # 逐行插入，source_record_number 就是这一行在（过滤后的）
        # 列表里排第几个，从1开始编号，方便以后按行号定位问题记录
        for source_record_number, row in enumerate(rows, start=1):
            try:
                self.insert_raw_submission(ingestion_batch_id, source_record_number, row)
                successful_row_count += 1
                if verbose and source_record_number % 100 == 0:
                    print(f"  processed {source_record_number}/{len(rows)} rows...")
            except Exception as e:
                # 某一行插入失败，记下"第几行、报的什么错"，
                # 不中断整体流程，继续处理剩下的行
                errors.append((source_record_number, str(e)))
                print(f"  row {source_record_number} failed: {e}")

            # 每插入一行短暂停顿，避免短时间内对 D1 发太密集的请求
            time.sleep(0.05)

        # 全部行都跑完之后，把最终结果回填进 Ingestion_batch 那一行
        self.finalize_ingestion_batch(
            ingestion_batch_id,
            successful_row_count=successful_row_count,
            failed_row_count=len(errors),
            skipped_row_count=0,
            file_sha256=file_sha256,
            errors=errors,
        )

        print("\n" + "=" * 60)
        print("Load summary")
        print("=" * 60)
        print(f"  ingestion_batch_id : {ingestion_batch_id}")
        print(f"  dataset_type       : {type_code}")
        print(f"  file_sha256        : {file_sha256}")
        print(f"  total rows         : {len(rows)}")
        print(f"  loaded             : {successful_row_count}")
        print(f"  errored rows       : {len(errors)}")

        return {
            "ingestion_batch_id": ingestion_batch_id,
            "dataset_type": type_code,
            "file_sha256": file_sha256,
            "total": len(rows),
            "loaded": successful_row_count,
            "errors": errors,
        }


In [ ]:
# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

loader = RawSubmissionLoader(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
#

loader.seed_dataset_types()
#
# # 第二步：先跑一小批确认没问题
#stats = loader.load_csv(CSV_PATH, limit=20)
#
# # 第三步：确认没问题后跑全量
stats = loader.load_csv(CSV_PATH)

  Dataset_type ready: display_type_name='submission_w/_resume' -> type_code='submission_w_resume'
  Dataset_type ready: display_type_name='job_posting' -> type_code='job_posting'
  Dataset_type ready: display_type_name='company_info' -> type_code='company_info'
Created Ingestion_batch id=8
Starting load, 20 rows total


Load summary
  ingestion_batch_id : 8
  dataset_type       : submission_w_resume
  file_sha256        : 37ffff2abc755e351f02d5c0b3e2515f84e1066db726b43667e1377e5d16af38
  total rows         : 20
  loaded             : 20
  errored rows       : 0


# Initial submission cleaning

1. 需要先运行一次 database build section 里面的第一个（实现数据库连接）和第二个cells（实现Google Drive连接） + Raw submission section 里面的第一个cell（执行写入一个叫RawSubmissionLoader 的 class，这个部分会引用到这个class里面的_execte() function，但切记不要运行这个部分的第二个cell执行导入数据的代码块）。

2. 这个section里的两个cells都可以重复无限次执行

In [ ]:
"""
Google Colab 新 cell：对已经导入 D1 的 raw submission 做初步清洗。

重要：
1. 本 cell 绝不调用 loader.load_csv()，因此不会重复导入 raw_submission。
2. 本 cell 只更新 raw_record_null_flags_utf_8。                                需要看一下raw_record_null_flags_utf_8这个字段的导入是不是用source record number来匹配的，如果是的话就没问题，如果不是的话就需要修改代码。
3. null_check_result 和 submission_status 只保存在 Colab 内存中。
4. 本 cell 不更新raw_submission表里面的 processing_error 或 processing_status。
5. 本 cell 不开始 Submission_normalized。
6. 本 cell 只在 Colab 内存中生成 successful_submission_dataframe，供下一个 Submission_normalized cell 使用。
7. 如果后续需要用到 submission_status 或 null_check_result， 可以在 Colab 内存中直接访问 initial_submission_cleaner.submission_status_dataframe 和 initial_submission_cleaner.null_check_result_dataframe。
"""

# pandas 用于重新读取 CSV 和处理 DataFrame。
import pandas as pd

# hashlib 用于计算当前整份 CSV 文件的 SHA-256。
import hashlib

# os 用于从完整路径中取得 CSV 文件名。
import os

# time 只用于 D1 请求失败后的重试等待。
import time


class InitialSubmissionCleaner:
    """对已经存在于 D1 的 raw_submission 做第一阶段空值检查。"""

    # 当前阶段不读取和处理原始简历文件链接。
    EXCLUDED_COLUMNS = ["resume"]

    def __init__(
        self,
        d1_client,
        csv_path,
        ingestion_batch_id=None,
    ):
        # 复用重新实例化后的 RawSubmissionLoader。
        #
        # 这里只会使用它的 _execute()。
        # 不会调用它的 load_csv()。
        self.d1_client = d1_client

        # 保存 Google Drive 中的原始 CSV 路径。
        self.csv_path = csv_path

        # 可以手动传入已经存在的 batch id。
        #
        # 如果传入 None，程序会从 D1 自动查找唯一匹配批次。
        self.ingestion_batch_id = (
            int(ingestion_batch_id)
            if ingestion_batch_id is not None
            else None
        )

        # 预先声明后面生成的数据对象。
        self.raw_submission_dataframe = None
        self.raw_record_null_flags_utf_8_dataframe = None
        self.raw_record_null_flags_utf_8_series = None
        self.null_check_result_series = None
        self.submission_status_series = None
        self.successful_submission_dataframe = None
        # 预先声明包含数据库id和submission status的DataFrame。
        self.submission_status_dataframe = None
        # 预先声明包含数据库id和null check result的DataFrame。
        self.null_check_result_dataframe = None

        # 保存从 D1 查询出来的 raw_submission_id 与 source_record_number 的对应关系。
        #
        # source_record_number 只在这个 class 内部用于匹配。
        # 不会作为 column 加进最后的 successful_submission_dataframe。
        self.raw_submission_id_by_source_record_number = None

    def _execute_with_retry(
        self,
        sql,     # 需要执行的 SQL 语句。
        params=None,     # SQL 中 ? 占位符对应的参数。
        max_retries=3,     # 最多尝试三次，包括第一次。
        retry_delay_seconds=1.0,      # 失败后等待一秒再尝试。
    ):
        """复用 RawSubmissionLoader class 里面的 _execute() function，只在请求失败时重试。"""

        # 用于保存最后一次失败信息。如果三次都失败，最终会把这个错误显示出来。
        last_error = None

        # 最多尝试 max_retries 次。
        for attempt in range(1, max_retries + 1):

            # 这里调用 RawSubmissionLoader class 里面的 _execute() function 返回 (ok, result)。
            ok, result = self.d1_client._execute(
                sql,
                params,
            )

            # 查询成功就立即返回。
            if ok:
                return result

            # 保存本次错误信息。
            last_error = result

            # 如果还有机会，就短暂等待后重试。
            if attempt < max_retries:
                time.sleep(retry_delay_seconds)

        # 三次都失败后才停止。
        raise RuntimeError(
            "D1 request failed after "
            f"{max_retries} attempts: {last_error}"
        )

    def _read_csv_with_pandas(self):
        """
        使用 pandas 重新读取 CSV，并排除 "resume" column。
        这里个function暂时不会因为某行 name 或 contact_email 为空而停止读取或跳过读取。
        """

        # 重新读取除 “resume” column 以外的完整csv。
        #
        # 后续可以在这里添加专属于
        # submission_w_resume 数据集的读取 schema（RAW_SUBMISSION_CSV_SCHEMA）。
        self.raw_submission_dataframe = pd.read_csv(
            self.csv_path,
            # usecols=lambda col: col not in self.EXCLUDED_COLUMNS
            #, dtype=RAW_SUBMISSION_CSV_SCHEMA
        )

        # 检查csv中是否存在 EXCLUDED_COLUMNS。
        excluded_columns_found = [
            column_name
            for column_name in self.EXCLUDED_COLUMNS
            if column_name
            in self.raw_submission_dataframe.columns
        ]

        # 从导入的完整 DataFrame 中删除 EXCLUDED_COLUMNS（“resume” column）。
        self.raw_submission_dataframe = (
            self.raw_submission_dataframe.drop(
                columns=excluded_columns_found,
                errors="ignore",
            )
        )

        # 检查csv中是否存在 “name” 这个column。如果不存在的话，会影响后面判断每一个record是否存在 name value。
        if (
            "name"
            not in self.raw_submission_dataframe.columns
        ):
            raise ValueError(
                "CSV schema 中缺少必要 column：name"
            )

        # 检查csv中是否存在 “contact_email” 这个column。如果不存在的话，会影响后面判断每一个record是否存在 contact_email value。
        if (
            "contact_email"
            not in self.raw_submission_dataframe.columns
        ):
            raise ValueError(
                "CSV schema 中缺少必要 column：contact_email"
            )

        # 空文件无法用于后续的原始数据初步清洗处理。
        if self.raw_submission_dataframe.empty:
            raise ValueError(
                "CSV 中没有任何 data record。"
            )

        # 显示读取结果。
        print("CSV successfully loaded.")
        print(
            "Total CSV records:",
            len(self.raw_submission_dataframe),
        )
        print(
            "Cleaned columns after removing EXCLUDED_COLUMNS:",
            list(self.raw_submission_dataframe.columns),
        )

        # 返回排除 resume 后的 DataFrame。
        return self.raw_submission_dataframe

    def _get_current_csv_identity(self):
        """
        取得当前重新读取的原始 CSV 文件名、文件 SHA-256 和 DataFrame record总数量。
        """

        # 用二进制模式读取原始包含 “resume” column 的完整 CSV。
        #
        # 这里仍包含 resume column，因为 raw_submission section 里面的 load_csv function 在写入数据导入批次的记录时，file_sha256 就是在排除 “resume” column 之前对包含 “resume” column 的原始完整 raw_submission.csv 文件计算的。
        with open(self.csv_path, "rb") as csv_file:
            csv_file_bytes = csv_file.read()

        # 使用与原 load_csv function 相同的方式计算整份文件的哈希值。
        file_sha256 = hashlib.sha256(
            csv_file_bytes
        ).hexdigest()

        # 从csv_path 中获取 CSV 文件名。
        source_file_name = os.path.basename(
            self.csv_path
        )

        # 取得从 raw_submission.csv 里面成功读取成 raw_submission_dataframe 的总记录数/总行数。
        df_record_count = len(
            self.raw_submission_dataframe
        )

        # 返回当前原始 CSV 的三个识别值。
        return (
            source_file_name,
            file_sha256,
            df_record_count,
        )

    def _resolve_existing_ingestion_batch_id(self):
        """
        从 D1 查找已经存在的原始导入批次，用于后续的raw_submission_id和ingestion_batch_id的回填。

        这个 function 只执行 SQL SELECT，不创建新的 batch，也不重新导入 raw_submission.csv。


        这个 function 只执行 SQL SELECT：
        1. 不会自动查找 ingestion_batch_id。
        2. 不会创建新的 ingestion batch。
        3. 不会重新导入 raw_submission.csv。
        """

        # 如果没有手动指定 ingestion_batch_id，就直接停止。
        #
        # 这个初步清洗 cell 不会自动猜测或查找目标数据导入批次，
        # 避免把清洗结果错误地写入其他 ingestion batch。
        if self.ingestion_batch_id is None:
            raise ValueError("ingestion_batch_id is None"

            )

        # 取得当前原始 CSV 的文件名、文件 SHA-256
        # 和 DataFrame record 总数量。
        (
            source_file_name,
            file_sha256,
            df_record_count,
        ) = self._get_current_csv_identity()

        # 检查手动指定的 ingestion_batch_id 是否属于当前原始 CSV，
        # 并且是否成功导入了全部 DataFrame records。
        #
        # SQL 查询条件：
        # "id"（ingestion_batch的唯一标识） = self.ingestion_batch_id（手动指定的 ingestion_batch_id）。筛选：满足手动指定的 ingestion_batch_id的所有批量导入记录（ingestion_batch）。
        # "source_file_name"（ingestion_batch使用的原始CSV文件名） = source_file_name（当前重新读取成raw_submission_dataframe的原始CSV文件名）。筛选：满足使用的原始CSV文件名与生成 raw_submission_dataframe 所使用的的原始csv文件名一样的所有批量导入记录（ingestion_batch）。
        # "file_sha256"（ingestion_batch使用的原始CSV文件） = file_sha256（当前重新读取成raw_submission_dataframe的原始CSV文件）。筛选：满足使用的原始CSV文件的file_sha256与生成 raw_submission_dataframe 所使用的的原始csv文件的file_sha256一样的所有批量导入记录（ingestion_batch）。
        # "successful_row_count"（ingestion_batch成功导入的raw_submission记录数） = df_record_count（当前重新读取成raw_submission_dataframe的原始CSV文件的总行数）。筛选：满足成功导入的raw_submission记录数与重新读取raw_submission.csv后生成的 raw_submission_dataframe 里面的总记录数/总行数一样的所有批量导入记录（ingestion_batch）。
        # COALESCE("failed_row_count", 0) = 0（ingestion_batch没有失败的raw submission record导入记录）。筛选：满足没有失败的raw submission record导入记录的所有批量导入记录（ingestion_batch）。
        sql = """
            SELECT
              "id",
              "imported_at",
              "import_status",
              "total_row_count",
              "successful_row_count",
              "failed_row_count"
            FROM "Ingestion_batch"
            WHERE "id" = ?
              AND "source_file_name" = ?
              AND "file_sha256" = ?
              AND "successful_row_count" = ?
              AND COALESCE("failed_row_count", 0) = 0;
        """

        # 这里只执行 SELECT 查询，不会执行任何 INSERT 或 UPDATE。
        result = self._execute_with_retry(
            sql,
            [
                self.ingestion_batch_id,
                source_file_name,
                file_sha256,
                df_record_count,
            ],
        )

        # 提取符合当前原始 CSV 的数据导入批次。
        matching_batches = (
            result[0].get("results", [])
            if result
            else []
        )

        # 因为 id 是主键，所以正常情况下查询结果只能是 0 条或 1 条。
        #
        # 如果不是恰好 1 条，说明手动指定的 ingestion_batch_id
        # 不存在，或者它与当前 CSV 文件不完全对应。
        if len(matching_batches) != 1:
            raise ValueError(
                "手动指定的 ingestion_batch_id "
                f"{self.ingestion_batch_id} 不存在，"
                "或者与当前 CSV 的文件名、SHA-256、"
                "成功导入记录数不一致，"
                "或者该批次存在导入失败记录。"
            )

        # 显示已经验证成功的数据导入批次 id。
        print(
            "Existing ingestion batch validated:",
            self.ingestion_batch_id,
        )

        # 返回经过验证的 ingestion_batch_id。
        return self.ingestion_batch_id

    def _resolve_existing_ingestion_batch_id(self):
        """
        从 D1 查找已经存在的原始导入批次，用于后续的raw_submission_id和ingestion_batch_id的回填。

        这个 function 只执行 SQL SELECT，不创建新的 batch，也不重新导入 raw_submission.csv。
        """

        # 取得当前原始 CSV 的文件名、文件 SHA-256 和 DataFrame record总数量。
        (
            source_file_name,
            file_sha256,
            df_record_count,
        ) = self._get_current_csv_identity()

        # 手动给定 ingestion batch id 时，仍然检查这个 ingestion batch id 是否属于当前这份原始 CSV 文件，并且是否成功导入了全部 DataFrame records。
        #
        # SQL 查询条件：
        # "id"（ingestion_batch的唯一标识） = self.ingestion_batch_id（手动指定的 ingestion_batch_id）。筛选：满足手动指定的 ingestion_batch_id的所有批量导入记录（ingestion_batch）。
        # "source_file_name"（ingestion_batch使用的原始CSV文件名） = source_file_name（当前重新读取成raw_submission_dataframe的原始CSV文件名）。筛选：满足使用的原始CSV文件名与生成 raw_submission_dataframe 所使用的的原始csv文件名一样的所有批量导入记录（ingestion_batch）。
        # "file_sha256"（ingestion_batch使用的原始CSV文件） = file_sha256（当前重新读取成raw_submission_dataframe的原始CSV文件）。筛选：满足使用的原始CSV文件的file_sha256与生成 raw_submission_dataframe 所使用的的原始csv文件的file_sha256一样的所有批量导入记录（ingestion_batch）。
        # "successful_row_count"（ingestion_batch成功导入的raw_submission记录数） = df_record_count（当前重新读取成raw_submission_dataframe的原始CSV文件的总行数）。筛选：满足成功导入的raw_submission记录数与重新读取raw_submission.csv后生成的 raw_submission_dataframe 里面的总记录数/总行数一样的所有批量导入记录（ingestion_batch）。
        # COALESCE("failed_row_count", 0) = 0（ingestion_batch没有失败的raw submission record导入记录）。筛选：满足没有失败的raw submission record导入记录的所有批量导入记录（ingestion_batch）。
        if self.ingestion_batch_id is not None:
            sql = """
                SELECT
                  "id",
                  "imported_at",
                  "import_status",
                  "total_row_count",
                  "successful_row_count",
                  "failed_row_count"
                FROM "Ingestion_batch"
                WHERE "id" = ?
                  AND "source_file_name" = ?
                  AND "file_sha256" = ?
                  AND "successful_row_count" = ?
                  AND COALESCE("failed_row_count", 0) = 0;
            """

            # 这里只执行查询，不会有任何 INSERT 的操作。
            result = self._execute_with_retry(
                sql,
                [
                    self.ingestion_batch_id,
                    source_file_name,
                    file_sha256,
                    df_record_count,
                ],
            )

            # 提取符合当前原始 CSV 的手动指定数据导入批次（同一份csv文件有可能被多次批量导入）。
            matching_batches = (
                result[0].get("results", [])
                if result
                else []
            )

            # 手动指定的数据导入批次不符合当前原始 CSV 时停止。
            if len(matching_batches) != 1:
                raise ValueError(
                    "手动指定的 ingestion_batch_id 与当前 CSV 文件名、SHA-256、"
                    "成功导入记录数不一致。"
                )

            print(
                "Using existing ingestion_batch_id:",
                self.ingestion_batch_id,
            )
            return self.ingestion_batch_id

        # 如果 EXISTING_INGESTION_BATCH_ID = None，就使用当前原始 CSV 的文件名、文件 SHA-256 和成功导入记录数自动查找唯一匹配的数据导入批次。
        sql = """
            SELECT
              "id",
              "imported_at",
              "import_status",
              "total_row_count",
              "successful_row_count",
              "failed_row_count"
            FROM "Ingestion_batch"
            WHERE "source_file_name" = ?
              AND "file_sha256" = ?
              AND "successful_row_count" = ?
              AND COALESCE("failed_row_count", 0) = 0
            ORDER BY "id";
        """

        # 这里只执行查询，不会有任何 INSERT 的操作。
        result = self._execute_with_retry(
            sql,
            [
                source_file_name,
                file_sha256,
                df_record_count,
            ],
        )

        # 提取所有匹配的数据导入批次（ingestion_batch）记录。
        matching_batches = (
            result[0].get("results", [])
            if result
            else []
        )

        # 没找到时停止，但绝不自动重新导入。
        if not matching_batches:
            raise ValueError(
                "D1 中没有找到与当前 CSV 文件名、SHA-256、"
                "成功导入记录数完全一致的已导入数据批次 (ingestion_batch)。"
            )

        # 找到多个时也停止，避免猜测目标批次。
        if len(matching_batches) > 1:
            matching_batch_ids = [
                int(matching_batch["id"])
                for matching_batch in matching_batches
            ]

            raise ValueError(
                "找到多个相同 CSV 的已导入数据批次 (ingestion_batch)："
                f"{matching_batch_ids}。"
                "请把正确 id 填入 "
                "EXISTING_INGESTION_BATCH_ID。"
            )

        # 唯一匹配时保存原有 batch id。
        self.ingestion_batch_id = int(
            matching_batches[0]["id"]
        )

        # 显示找到的已有批次。
        print(
            "Existing ingestion batch found:",
            self.ingestion_batch_id,
        )

        # 返回已有批次 id。
        return self.ingestion_batch_id

    def _validate_target_batch_and_load_raw_submission_ids(self):
        """
        写入前检查成功重新读取 CSV 后生成的 DataFrame 与已有 D1 batch 完全对应，并取得每一个record对应的raw_submission_id。

        这个function只会执行sql查询和检查，暂时还不会把匹配的raw_submission_id写入DataFrame（raw_submission_dataframe），因为后面还需要先处理submission record的空值检查和和封装，我们不希望把raw_submission_id也一起被封装。
        """

        # 查询目标 batch 的raw submission记录数量、source_record_number行号范围和每一个record对应的raw_submission_id。
        sql = """
            SELECT
              "id" AS raw_submission_id,
              "source_record_number"
            FROM "raw_submission"
            WHERE "ingestion_batch_id" = ?
            ORDER BY "source_record_number";
        """

        # 执行只读查询。
        result = self._execute_with_retry(
            sql,
            [self.ingestion_batch_id],
        )

        # 提取sql query output出来的d1里面每一行raw_submission记录对应的raw_submission_id + source_record_number。
        raw_submission_rows = (
            result[0].get("results", [])
            if result
            else []
        )

        # 取得d1里面特定batch的实际source_record_number。
        actual_source_record_numbers = [
            int(raw_submission_row["source_record_number"])
            for raw_submission_row in raw_submission_rows
        ]

        # 更改逻辑：因为 DataFrame 可能经过清洗（比如删除了缺失项），
        # 预期需要的 source_record_number 应该是当前 DataFrame 留下的真实 index + 1
        expected_source_record_numbers = (self.raw_submission_dataframe.index + 1).tolist()

        # 验证：检查 DataFrame 中目前保留的记录是否在数据库中都存在
        missing_records = set(expected_source_record_numbers) - set(actual_source_record_numbers)
        if missing_records:
            raise ValueError(
                f"目标 batch 缺失 DataFrame 中存在的记录 (source_record_number): {missing_records}"
            )

        # 将source_record_number与raw_submission_id的一对一对应关系保存成字典，供后续的raw_submission_id回填时使用。
        self.raw_submission_id_by_source_record_number = {
            int(raw_submission_row["source_record_number"]): int(
                raw_submission_row["raw_submission_id"]
            )
            for raw_submission_row in raw_submission_rows
        }

        # 显示验证成功。
        print("Existing batch row mapping validated with cleaned DataFrame.")

    @staticmethod
    def _null_flags_row_to_utf8_text(
        raw_record_null_flags_utf_8_row,      # 这里的raw_record_null_flags_utf_8_row是一个pandas Series对象，表示一个record/一行的空值标记。
    ):
        """
        将一行 1/0 空值标记封装成 UTF-8 纯文本。

        字段顺序沿用排除 resume 后的原始 CSV column 顺序，方便和原始 DataFrame 逐列对照。
        """

        # 用于保存所有字段的标记文本。
        raw_record_null_flag_lines = []

        # 按照 raw_submission_dataframe 的 column 顺序逐个处理。
        for field_name in (
            raw_record_null_flags_utf_8_row.index
        ):

            # 取得 1/0 标记。
            raw_record_null_flag = int(
                raw_record_null_flags_utf_8_row[
                    field_name
                ]
            )

            # 生成单个字段的文本。
            raw_record_null_flag_line = (
                f"{field_name}: "
                f"{raw_record_null_flag}"
            )

            # 保存单个字段的文本。
            raw_record_null_flag_lines.append(
                raw_record_null_flag_line
            )

        # 用换行符连接全部字段标记。
        return "\n".join(
            raw_record_null_flag_lines
        )

    def _build_processing_results(self):
        """
        生成空值文本封装、null_check_result 和 submission_status。

        后两个结果只保存在内存，不写入数据库的raw_submission表里。
        """

        # 对每个字段执行 pandas 空值判断。
        #
        # True 转成 1。
        # False 转成 0。
        self.raw_record_null_flags_utf_8_dataframe = (
            self.raw_submission_dataframe
            .isna()
            .astype("int8")
        )

        # 把每行 1/0 标记封装成 UTF-8 文本。
        self.raw_record_null_flags_utf_8_series = (
            self.raw_record_null_flags_utf_8_dataframe
            .apply(
                self._null_flags_row_to_utf8_text,
                axis=1,
            )
            .astype("string")
            .rename(
                "raw_record_null_flags_utf_8"
            )
        )

        # 保存每一个raw_submission的提交状态: failed_submission (submission record中缺失name或contact_email的值) / successful_submission（submission record中name和contact_email都存在）。
        submission_status_values = []

        # 保存每一个raw_submission的空值检查结果: all_attributes_missing (整行缺失所有值) / name_and_email_missing (name和contact_email都缺失) / name_missing (只有name缺失) / email_missing (只有contact_email缺失) / None (name和contact_email都存在)。
        null_check_result_values = []

        # 逐行执行name和contact_email的缺失判断，并生成submission_status和null_check_result。
        for (
            dataframe_index,
            null_flags_row,
        ) in (
            self.raw_record_null_flags_utf_8_dataframe
            .iterrows()
        ):

            # 判断 name 是否缺失。
            name_missing = (
                int(null_flags_row["name"]) == 1
            )

            # 判断 contact_email 是否缺失。
            email_missing = (
                int(null_flags_row["contact_email"]) == 1
            )

            # 判断排除 resume 后是否整行为空。
            all_attributes_missing = bool(
                null_flags_row.eq(1).all()
            )

            # 整行为空优先级最高。
            if all_attributes_missing:
                submission_status = (
                    "failed_submission"
                )
                null_check_result = (
                    "all_attributes_missing"
                )

            # name 和 email 都缺失。
            elif name_missing and email_missing:
                submission_status = (
                    "failed_submission"
                )
                null_check_result = (
                    "name_and_email_missing"
                )

            # 只有 name 缺失。
            elif name_missing:
                submission_status = (
                    "failed_submission"
                )
                null_check_result = "name_missing"

            # 只有 email 缺失。
            elif email_missing:
                submission_status = (
                    "failed_submission"
                )
                null_check_result = "email_missing"

            # name 和 email 都存在。
            else:
                submission_status = (
                    "successful_submission"
                )
                null_check_result = None

            # 保存当前raw_submission record的提交状态。
            submission_status_values.append(
                submission_status
            )

            # 保存当前raw_submission record的空值检查结果。
            null_check_result_values.append(
                null_check_result
            )

        # 生成内存 submission_status Series。
        self.submission_status_series = pd.Series(
            submission_status_values,
            index=self.raw_submission_dataframe.index,
            name="submission_status",
            dtype="string",
        )

        # 生成内存 null_check_result Series。
        self.null_check_result_series = pd.Series(
            null_check_result_values,
            index=self.raw_submission_dataframe.index,
            name="null_check_result",
            dtype="object",
        )

        # 找出 successful_submission，用于筛选能够进入下一个submission normalization阶段的raw_submission记录。
        successful_submission_mask = (
            self.submission_status_series
            == "successful_submission"
        )

        # 为下一个 Submission_normalized cell 建立成功提交的raw_submission记录 DataFrame。
        self.successful_submission_dataframe = (
            self.raw_submission_dataframe.loc[
                successful_submission_mask
            ].copy()
        )

        # 显示状态统计。
        print("\nSubmission status summary:")
        print(
            self.submission_status_series.value_counts(
                dropna=False
            )
        )

    def _attach_database_ids(self):
        """
        只将raw_submission_id和ingestion_batch_id回填到raw_submission_dataframe和successful_submission_dataframe。

        不会把source_record_number添加成新的DataFrame column。
        """

        # 确保已经从D1取得每一个source_record_number对应的raw_submission_id。
        if (
            self.raw_submission_id_by_source_record_number
            is None
        ):
            raise RuntimeError(
                "必须先验证目标batch并读取raw_submission_id。"
            )

        # 使用DataFrame（raw_submission_dataframe）原始index加1作为内部source_record_number（因为如果不出意外情况pandas能完全成功读取csv文件里面的每一行raw submission record），然后把每一行的raw_submission record通过raw_submission_id_series字典映射到对应的raw_submission_id。
        raw_submission_id_series = pd.Series(
            [
                self.raw_submission_id_by_source_record_number[
                    int(dataframe_index) + 1
                ]
                for dataframe_index
                in self.raw_submission_dataframe.index
            ],
            index=self.raw_submission_dataframe.index,
            name="raw_submission_id",
            dtype="int64",
        )

        # 将raw_submission_id添加到完整的raw_submission_dataframe。
        self.raw_submission_dataframe.insert(
            0,
            "raw_submission_id",
            raw_submission_id_series,
        )

        # 将ingestion_batch_id添加到完整raw_submission_dataframe。
        self.raw_submission_dataframe.insert(
            1,
            "ingestion_batch_id",
            self.ingestion_batch_id,
        )

        # 将对应的raw_submission_id添加到successful_submission_dataframe。
        self.successful_submission_dataframe.insert(
            0,
            "raw_submission_id",
            raw_submission_id_series.loc[
                self.successful_submission_dataframe.index       # successful_submission_dataframe的index是沿用的raw_submission_dataframe的index，所以可以直接用loc来匹配。
            ],
        )

        # 将ingestion_batch_id添加到successful_submission_dataframe。
        self.successful_submission_dataframe.insert(
            1,
            "ingestion_batch_id",
            self.ingestion_batch_id,
        )

        # 将raw_submission_id、ingestion_batch_id和submission_status
        # 合并成submission_status_dataframe。
        self.submission_status_dataframe = pd.DataFrame(
            {
                "raw_submission_id": (
                    self.raw_submission_dataframe[
                        "raw_submission_id"
                    ]
                ),
                "ingestion_batch_id": (
                    self.raw_submission_dataframe[
                        "ingestion_batch_id"
                    ]
                ),
                "submission_status": (
                    self.submission_status_series
                ),
            },
            index=self.raw_submission_dataframe.index,
        )

        # 将raw_submission_id、ingestion_batch_id和null_check_result
        # 合并成null_check_result_dataframe。
        self.null_check_result_dataframe = pd.DataFrame(
            {
                "raw_submission_id": (
                    self.raw_submission_dataframe[
                        "raw_submission_id"
                    ]
                ),
                "ingestion_batch_id": (
                    self.raw_submission_dataframe[
                        "ingestion_batch_id"
                    ]
                ),
                "null_check_result": (
                    self.null_check_result_series
                ),
            },
            index=self.raw_submission_dataframe.index,
        )

    def _write_null_flags_to_d1(self):
        """
        只更新 raw_record_null_flags_utf_8。

        不更新 processing_error 或 processing_status。
        """

        # UPDATE 只包含一个待更新字段。
        sql = """
            UPDATE "raw_submission"
            SET "raw_record_null_flags_utf_8" = ?
            WHERE "ingestion_batch_id" = ?
              AND "source_record_number" = ?;
        """

        # 保存已更新数量。
        updated_record_count = 0

        # 逐行更新已有 raw_submission。
        for dataframe_index, flags_text in (
            self.raw_record_null_flags_utf_8_series
            .items()
        ):

            # CSV 数据行号从 1 开始。
            #
            # source_record_number只作为这个function内部的临时匹配值，不会添加到最后的DataFrame中。
            source_record_number = (
                int(dataframe_index) + 1
            )

            # 只发送 null flags、已有 batch id 和行号。
            self._execute_with_retry(
                sql,
                [
                    str(flags_text),
                    self.ingestion_batch_id,
                    source_record_number,
                ],
            )

            # 增加更新计数。
            updated_record_count += 1

            # 每 100 条显示一次进度。
            if updated_record_count % 100 == 0:
                print(
                    "Updated:",
                    f"{updated_record_count}/"
                    f"{len(self.raw_submission_dataframe)}",
                )

        # 显示最终数量。
        print(
            "Null flags update completed:",
            updated_record_count,
        )

    def _verify_write(self):
        """验证目标 batch/所有raw_submission records 的 null flags 已全部写入。"""

        # 统计仍为 NULL 的 null flags。
        sql = """
            SELECT
              COUNT(*) AS total_record_count,
              SUM(
                CASE
                  WHEN "raw_record_null_flags_utf_8"
                       IS NULL
                  THEN 1
                  ELSE 0
                END
              ) AS missing_count
            FROM "raw_submission"
            WHERE "ingestion_batch_id" = ?;
        """

        # 执行验证。
        result = self._execute_with_retry(
            sql,
            [self.ingestion_batch_id],
        )

        # 提取验证结果。
        verification = result[0]["results"][0]

        # 显示验证结果。
        print("D1 verification:", verification)

        # 仍有 NULL 时停止。
        if int(verification["missing_count"] or 0) != 0:
            raise RuntimeError(
                "仍有记录没有 null flags。"
            )

    def run(self):
        """执行这个初步清洗 cell 的完整流程。"""

        # 第一步：重新读取 CSV，但不导入。这里我们不通过使用拆解后的raw_record_jsonb原始数据来进行原始数据的初步清洗和submission normalization，而是直接使用重新读取的 CSV 原始数据的方法来进行。
        self._read_csv_with_pandas()

        # 第二步：查找已经存在的导入批次。
        self._resolve_existing_ingestion_batch_id()

        # 第三步：验证目标batch的record数量和source_record_number，并取得每一个record对应的raw_submission_id。
        self._validate_target_batch_and_load_raw_submission_ids()

        # 第四步：生成内存处理结果：submission_status、null_check_result 和 raw_record_null_flags_utf_8。
        self._build_processing_results()

        # 第五步：将raw_submission_id和ingestion_batch_id回填到需要进入后续处理可能会用到的DataFrame（successful_submission_dataframe 和 raw_submission_dataframe ）。
        self._attach_database_ids()

        # 第六步：只更新 raw_submission 里面的null flags。
        self._write_null_flags_to_d1()

        # 第七步：验证更新。
        self._verify_write()

        # 第八步：返回成功提交的submission记录，供下一个 Submission_normalized cell 使用。
        return self.successful_submission_dataframe



In [ ]:
# ============================================================
# 运行本 cell
# ============================================================

# 重新连接 Colab 后，可以重新运行 RawSubmissionLoader 的：
#     import
#     ACCOUNT_ID / DATABASE_ID / API_TOKEN / CSV_PATH
#     class RawSubmissionLoader 定义
#
# 然后只重新创建连接对象：
loader = RawSubmissionLoader(
    ACCOUNT_ID,
    DATABASE_ID,
    API_TOKEN,
)

# 弹窗获取 EXISTING_INGESTION_BATCH_ID
user_input = input("请输入 EXISTING_INGESTION_BATCH_ID (直接回车或输入 'none' 代表自动选择): ").strip()

# 判断用户的输入逻辑
if user_input.lower() == 'none' or user_input == '':
    EXISTING_INGESTION_BATCH_ID = None
    print("-> 已选择自动查找当前 CSV 已经存在的唯一导入批次 (None)。")
else:
    try:
        EXISTING_INGESTION_BATCH_ID = int(user_input)
        print(f"-> 已手动设定 EXISTING_INGESTION_BATCH_ID = {EXISTING_INGESTION_BATCH_ID}")
    except ValueError:
        raise ValueError("输入无效！请填写纯数字，或者直接回车代表自动选择。")

# 创建初步清洗器。
initial_submission_cleaner = (
    InitialSubmissionCleaner(
        d1_client=loader,
        csv_path=CSV_PATH,
        ingestion_batch_id=(
            EXISTING_INGESTION_BATCH_ID
        ),
    )
)

# 执行初步清洗。
successful_submission_dataframe = (
    initial_submission_cleaner.run()
)

# 显示准备进入下一阶段的记录。
display(
    successful_submission_dataframe.head(10)
)

# 本 cell 到这里结束。
# 暂时不开始 Submission_normalized。

# Submission_normalized

1. 这个section里的cells之前，需要先运行 database build section 里面的第一个（实现数据库连接）和第二个cells（实现Google Drive连接） + Raw submission section 里面的第一个cell（执行写入一个叫RawSubmissionLoader 的 class，这个部分会引用到这个class里面的_execte function，但切记不要运行这个部分的第二个cell执行导入数据的代码块） + Initial submission cleaning section里面的两个cells（拿到原始数据初步清洗后的successful_submission_dataframe）

2.

In [ ]:
# 查看数据库中指定表的字段和字段约束

from google.colab import userdata
import requests


ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")

if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。请在 Colab 左侧 Secrets 中添加该密钥，"
        "并允许当前 notebook 访问。"
    )


def execute_d1_query(sql, params=None):
    """向 Cloudflare D1 执行参数化 SQL 查询。"""

    url = (
        f"https://api.cloudflare.com/client/v4/accounts/"
        f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
    )

    response = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {API_TOKEN}",
            "Content-Type": "application/json",
        },
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError(
            f"Cloudflare 返回了非 JSON 响应："
            f"HTTP {response.status_code}\n{response.text}"
        )

    if not response.ok or not payload.get("success"):
        raise RuntimeError(
            f"D1 查询失败：HTTP {response.status_code}\n"
            f"{payload.get('errors') or payload}"
        )

    return payload.get("result", [])


TABLE_NAMES = [
    "Company",
    "Position",
    "Position_application_cycle",
    "Application",
    "Application_document",
    "Candidate_snapshot",
    "Candidate_Skill",
    "Candidate_Position",
    "Candidate_Education",
    "Candidate_Certification",
    "Person",
    "Person_identifier_claim",
    "Person_Skill",
    "Person_Project",
    "Person_Position",
    "Person_Name",
    "Person_Education",
    "Person_Contact",
    "Person_Link",
    "Person_Certification",
    "Education",
    "Degree",
    "School",
    "Field_study",
    "Major",
]

for table_name in TABLE_NAMES:
    result = execute_d1_query(
        """
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table'
          AND name = ?;
        """,
        [table_name],
    )

    rows = result[0].get("results", []) if result else []

    print("\n" + "=" * 80)
    print(table_name)
    print("=" * 80)

    if rows and rows[0].get("sql"):
        print(rows[0]["sql"])
    else:
        print("Table not found.")

### Normalization

In [ ]:
"""
Google Colab 新 cell：标准化 successful submissions，并导入 Submission_normalized。

本 cell 使用：
1. successful_submission_dataframe：从 raw_submission.csv 初步清洗得到。
2. loader：已经连接 Cloudflare D1 的 client。

本 cell 会自己从 D1 读取 Company 和 Position，
并为每条 successful submission 生成 candidate_rank = 1 的 Position 候选。
因此不再依赖另一个 cell 预先生成 submission_position_candidate_dataframe。

本 cell 按已经确认的业务决定执行：
1. 每条 submission 只使用 candidate_rank = 1 的 Position 候选。
2. 候选 Position 所属的 candidate_company_id 直接写入 company_id。
3. 候选 position_id 直接写入 position_id。
4. 为每个被选中的 Position 查找或建立一条 cycle_name = "Unknown" 的 application cycle。
5. raw submission 真实填写的 company / position 分别写入：
   normalized_company_name / normalized_position_name。
6. normalized_application_cycle_name 写入 "Unknown"。
7. raw_submission.csv 没有 phone column，所有 phone 相关字段暂时写入 None，对应 SQL NULL。
8. application_cluster_id 和 open_window_id 暂时写入 None。
9. 本 cell 不拆分 resume_text，不从简历提取电话。
10. 重复运行时，已经存在 raw_submission_id 的 Submission_normalized 记录会按最新规则 UPDATE。
11. 姓名、email prefix/domain、work type/duration、company/position/cycle normalized字段统一小写。
12. email字符标准化完成后查询DNS；DNS结果只写入warnings，不会修改normalized email字段。
"""

# pandas 用于处理 DataFrame 和日期解析。
import pandas as pd

# json 用于将 normalization warnings 封装成可追溯的 JSON 文本。
import json

# re 用于检查和拆分 email address。
import re

# time 只用于 D1 请求失败后的重试等待。
import time

# sys和subprocess只用于Colab缺少dnspython时自动安装依赖。
import sys
import subprocess

# unicodedata 用于统一姓名和其他文本的 Unicode 形式。
import unicodedata

# datetime 用于生成 UTC 时间。
from datetime import datetime, timezone

# SequenceMatcher 用于计算原始 Position name 与 D1 Position name 的相似度。
from difflib import SequenceMatcher

# dnspython用于查询标准化domain的MX、A和AAAA records。
try:
    import dns.exception
    import dns.resolver
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "dnspython",
        ]
    )
    import dns.exception
    import dns.resolver


class RankOnePositionCandidateBuilder:
    """直接为每条 successful submission 生成一条 rank-1 Position 候选。"""

    def __init__(
        self,
        d1_client,
        successful_submission_dataframe,
        approximate_match_threshold=0.80,
        d1_page_size=500,
    ):
        # 复用初步清洗 cell 已经重新建立的 D1 client。
        self.d1_client = d1_client

        # 复制 successful submission DataFrame，避免修改上一个 cell 的对象。
        self.successful_submission_dataframe = (
            successful_submission_dataframe.copy()
        )

        # 保存判断近似匹配类型时使用的最低相似度。
        self.approximate_match_threshold = float(
            approximate_match_threshold
        )

        # 保存每次从 D1 读取 Position records 的分页大小。
        self.d1_page_size = int(
            d1_page_size
        )

        # 预先声明从 D1 读取的 Position DataFrame。
        self.database_position_dataframe = None

        # 预先声明最终生成的 rank-1 候选 DataFrame。
        self.submission_position_candidate_dataframe = None

    def _execute_with_retry(
        self,
        sql,
        params=None,
        max_retries=3,
        retry_delay_seconds=1.0,
    ):
        """复用 d1_client._execute()，只在 D1 请求失败时重试。"""

        # 保存最后一次 D1 请求返回的错误。
        last_error = None

        # 最多尝试 max_retries 次，其中包括第一次请求。
        for attempt in range(
            1,
            max_retries + 1,
        ):
            # 执行当前 D1 SQL 请求。
            ok, result = self.d1_client._execute(
                sql,
                params,
            )

            # 请求成功时立即返回结果。
            if ok:
                return result

            # 保存当前请求的错误结果。
            last_error = result

            # 仍有重试次数时等待后继续。
            if attempt < max_retries:
                time.sleep(
                    retry_delay_seconds
                )

        # 多次请求全部失败后停止，避免使用不完整的 Position 数据。
        raise RuntimeError(
            "D1 request failed after "
            f"{max_retries} attempts: {last_error}"
        )

    @staticmethod
    def _extract_d1_rows(result):
        """从 D1 REST API result 中提取 records。"""

        # D1 没有返回 statement result 时使用空列表。
        if not result:
            return []

        # 返回第一条 SQL statement 对应的 records。
        return result[0].get(
            "results",
            [],
        )

    @staticmethod
    def _normalize_comparison_text(value):
        """生成只用于 Position 和 Company 比较的文本。"""

        # pandas 空值无法参与字符串比较，统一转成空字符串。
        if pd.isna(value):
            return ""

        # 将输入值显式转换成 Python string。
        normalized_value = str(
            value
        )

        # 使用 NFKC 统一 Unicode 表示形式。
        normalized_value = unicodedata.normalize(
            "NFKC",
            normalized_value,
        )

        # 使用 casefold() 转成适合比较的小写形式。
        normalized_value = normalized_value.casefold()

        # 将 & 统一转换成单词 and。
        normalized_value = normalized_value.replace(
            "&",
            " and ",
        )

        # 将英文字母和数字以外的字符转换成空格。
        normalized_value = re.sub(
            r"[^a-z0-9]+",
            " ",
            normalized_value,
        )

        # 合并连续空格，并删除首尾空格。
        return " ".join(
            normalized_value.split()
        )

    @staticmethod
    def _calculate_title_similarity(
        normalized_submission_position_name,
        normalized_database_position_name,
    ):
        """计算两个已经标准化的 Position name 的相似度。"""

        # 任意一个 Position name 为空时，相似度直接使用 0。
        if (
            not normalized_submission_position_name
            or not normalized_database_position_name
        ):
            return 0.0

        # 返回 0 到 1 之间的 SequenceMatcher 相似度。
        return SequenceMatcher(
            None,
            normalized_submission_position_name,
            normalized_database_position_name,
        ).ratio()

    def _validate_successful_submission_dataframe(self):
        """检查初步清洗 cell 返回的 DataFrame 是否可以继续处理。"""

        # rank-1 Position 匹配必须使用的 columns。
        required_columns = {
            "raw_submission_id",
            "ingestion_batch_id",
            "company",
            "position",
        }

        # 找出缺失的必要 columns。
        missing_columns = sorted(
            required_columns
            - set(
                self.successful_submission_dataframe.columns
            )
        )

        # 缺少任何必要 column 时停止。
        if missing_columns:
            raise ValueError(
                "successful_submission_dataframe 缺少必要 columns："
                f"{missing_columns}"
            )

        # 空 DataFrame 没有可以处理的 submission。
        if self.successful_submission_dataframe.empty:
            raise ValueError(
                "successful_submission_dataframe 中没有任何 record。"
            )

        # raw_submission_id 必须一条 submission 对应一个 id。
        if self.successful_submission_dataframe[
            "raw_submission_id"
        ].duplicated().any():
            raise ValueError(
                "successful_submission_dataframe 中存在重复 raw_submission_id。"
            )

    def _load_database_positions(self):
        """从 D1 分页读取全部 Company + Position records。"""

        # 保存所有分页读取出来的 Position records。
        all_position_rows = []

        # 第一页从 offset 0 开始。
        offset = 0

        # 持续分页读取，直到最后一页。
        while True:
            # 读取生成 rank-1 候选需要的 Company 和 Position 字段。
            sql = """
                SELECT
                  p."id" AS position_id,
                  p."company_id" AS candidate_company_id,
                  c."company_name" AS candidate_company_name,
                  p."position_name" AS candidate_position_name
                FROM "Position" AS p
                INNER JOIN "Company" AS c
                  ON c."id" = p."company_id"
                ORDER BY p."id"
                LIMIT ?
                OFFSET ?;
            """

            # 执行当前分页的只读查询。
            result = self._execute_with_retry(
                sql,
                [
                    int(self.d1_page_size),
                    int(offset),
                ],
            )

            # 提取当前分页的 Position records。
            page_rows = self._extract_d1_rows(
                result
            )

            # 当前页为空时结束分页。
            if not page_rows:
                break

            # 将当前页加入完整 Position records。
            all_position_rows.extend(
                page_rows
            )

            # 当前页少于 page size 时说明已经读取到最后一页。
            if len(page_rows) < self.d1_page_size:
                break

            # 将 offset 移动到下一页。
            offset += self.d1_page_size

        # Position 表为空时无法满足 Submission_normalized 的 Position 外键。
        if not all_position_rows:
            raise ValueError(
                "D1 Position 表中没有任何可用 record。"
            )

        # 将 Position records 转换成 DataFrame。
        self.database_position_dataframe = pd.DataFrame(
            all_position_rows
        )

        # 将 position_id 显式转换成 pandas int64。
        self.database_position_dataframe[
            "position_id"
        ] = self.database_position_dataframe[
            "position_id"
        ].astype("int64")

        # 将 candidate_company_id 显式转换成 pandas int64。
        self.database_position_dataframe[
            "candidate_company_id"
        ] = self.database_position_dataframe[
            "candidate_company_id"
        ].astype("int64")

        # 生成只用于 Company name 比较的标准化文本。
        self.database_position_dataframe[
            "normalized_candidate_company_name"
        ] = self.database_position_dataframe[
            "candidate_company_name"
        ].map(
            self._normalize_comparison_text
        )

        # 生成只用于 Position name 比较的标准化文本。
        self.database_position_dataframe[
            "normalized_candidate_position_name"
        ] = self.database_position_dataframe[
            "candidate_position_name"
        ].map(
            self._normalize_comparison_text
        )

        # 显示成功读取的 D1 Position 数量。
        print(
            "D1 Position record count:",
            len(self.database_position_dataframe),
        )

    def _select_rank_one_for_submission(
        self,
        submission_row,
    ):
        """为一条 successful submission 选择唯一 rank-1 Position。"""

        # 标准化申请记录真实填写的 Company name。
        normalized_raw_company_name = (
            self._normalize_comparison_text(
                submission_row["company"]
            )
        )

        # 标准化申请记录真实填写的 Position name。
        normalized_raw_position_name = (
            self._normalize_comparison_text(
                submission_row["position"]
            )
        )

        # 复制数据库 Position DataFrame，用于添加当前申请的临时匹配分数。
        candidate_dataframe = (
            self.database_position_dataframe.copy()
        )

        # 判断候选 Company name 是否与原始 Company name 完全匹配。
        candidate_dataframe[
            "company_name_exact_match"
        ] = candidate_dataframe[
            "normalized_candidate_company_name"
        ].eq(
            normalized_raw_company_name
        )

        # 计算原始 Position name 与每一个候选 Position name 的相似度。
        candidate_dataframe[
            "position_title_similarity_score"
        ] = candidate_dataframe[
            "normalized_candidate_position_name"
        ].map(
            lambda candidate_name: (
                self._calculate_title_similarity(
                    normalized_raw_position_name,
                    candidate_name,
                )
            )
        )

        # 判断候选 Position name 是否与原始 Position name 完全匹配。
        candidate_dataframe[
            "position_title_exact_match"
        ] = candidate_dataframe[
            "normalized_candidate_position_name"
        ].eq(
            normalized_raw_position_name
        )

        # 取得所有 Position name 完全匹配的候选。
        exact_candidate_dataframe = (
            candidate_dataframe.loc[
                candidate_dataframe[
                    "position_title_exact_match"
                ]
            ].copy()
        )

        # 存在 Position name 完全匹配时，只在完全匹配集合中选择 rank 1。
        if not exact_candidate_dataframe.empty:
            # Company 也完全匹配的候选优先，然后使用最小 position_id 稳定选择。
            selected_row = (
                exact_candidate_dataframe
                .sort_values(
                    by=[
                        "company_name_exact_match",
                        "position_id",
                    ],
                    ascending=[
                        False,
                        True,
                    ],
                )
                .iloc[0]
            )

            # Company 和 Position 都一致时记录双重完全匹配。
            if bool(
                selected_row[
                    "company_name_exact_match"
                ]
            ):
                match_type = (
                    "exact_company_and_position"
                )

            # 只有 Position name 完全匹配时记录 exact_position_only。
            else:
                match_type = "exact_position_only"

        # 没有完全匹配 Position name 时，通过相似度选择 rank 1。
        else:
            # 相似度最高的候选优先，其次 Company 完全匹配，最后按 position_id。
            selected_row = (
                candidate_dataframe
                .sort_values(
                    by=[
                        "position_title_similarity_score",
                        "company_name_exact_match",
                        "position_id",
                    ],
                    ascending=[
                        False,
                        False,
                        True,
                    ],
                )
                .iloc[0]
            )

            # 达到近似匹配阈值时记录 approximate_position_candidate。
            if float(
                selected_row[
                    "position_title_similarity_score"
                ]
            ) >= self.approximate_match_threshold:
                match_type = (
                    "approximate_position_candidate"
                )

            # 没有达到阈值时仍保留相似度最高候选，但标记 no_position_match。
            else:
                match_type = "no_position_match"

        # 返回 normalization class 需要的 rank-1 候选字段。
        return {
            "raw_submission_id": int(
                submission_row["raw_submission_id"]
            ),
            "candidate_rank": 1,
            "match_type": match_type,
            "position_title_similarity_score": float(
                selected_row[
                    "position_title_similarity_score"
                ]
            ),
            "candidate_company_id": int(
                selected_row["candidate_company_id"]
            ),
            "candidate_company_name": str(
                selected_row["candidate_company_name"]
            ),
            "position_id": int(
                selected_row["position_id"]
            ),
            "candidate_position_name": str(
                selected_row["candidate_position_name"]
            ),
        }

    def _build_rank_one_candidate_dataframe(self):
        """为全部 successful submissions 生成 rank-1 候选。"""

        # 保存所有 submission 对应的 rank-1 candidate records。
        rank_one_candidate_rows = []

        # 逐条处理 successful submissions。
        for processed_count, (_, submission_row) in enumerate(
            self.successful_submission_dataframe.iterrows(),
            start=1,
        ):
            # 为当前 submission 选择唯一 rank-1 Position。
            rank_one_candidate_rows.append(
                self._select_rank_one_for_submission(
                    submission_row
                )
            )

            # 每处理 100 条显示一次进度。
            if processed_count % 100 == 0:
                print(
                    "Rank-1 Position candidates generated:",
                    f"{processed_count}/"
                    f"{len(self.successful_submission_dataframe)}",
                )

        # 将所有 rank-1 candidate records 转成 DataFrame。
        self.submission_position_candidate_dataframe = (
            pd.DataFrame(
                rank_one_candidate_rows
            )
        )

        # 检查生成数量是否与 successful submissions 完全一致。
        if len(
            self.submission_position_candidate_dataframe
        ) != len(
            self.successful_submission_dataframe
        ):
            raise RuntimeError(
                "rank-1 Position candidate 数量与 successful submission 数量不一致。"
            )

        # 检查每一个 raw_submission_id 是否只出现一次。
        if self.submission_position_candidate_dataframe[
            "raw_submission_id"
        ].duplicated().any():
            raise RuntimeError(
                "rank-1 Position candidate 中存在重复 raw_submission_id。"
            )

    def run(self):
        """执行只读 Position 加载和 rank-1 候选生成。"""

        # 第一步：检查 successful submission DataFrame。
        self._validate_successful_submission_dataframe()

        # 第二步：从 D1 读取全部 Position 和 Company。
        self._load_database_positions()

        # 第三步：生成每条 successful submission 的 rank-1 候选。
        self._build_rank_one_candidate_dataframe()

        # 显示生成的 rank-1 candidate 数量。
        print(
            "Rank-1 Position candidate count:",
            len(
                self.submission_position_candidate_dataframe
            ),
        )

        # 返回 normalization class 可以直接使用的候选 DataFrame。
        return self.submission_position_candidate_dataframe


class SubmissionNormalizedLoader:
    """生成并导入 Submission_normalized records。"""

    # 当前数据没有可信的 application cycle name。
    UNKNOWN_APPLICATION_CYCLE_NAME = "Unknown"

    # 删除prefix中不属于ASCII dot-atom允许集合的字符。
    EMAIL_PREFIX_ILLEGAL_CHARACTER_PATTERN = re.compile(
        r"[^a-z0-9!#$%&'*+\-/=?^_`{|}~.]"
    )

    # 删除domain中不属于ASCII domain允许集合的字符。
    EMAIL_DOMAIN_ILLEGAL_CHARACTER_PATTERN = re.compile(
        r"[^a-z0-9.-]"
    )

    def __init__(
        self,
        d1_client,
        successful_submission_dataframe,
        submission_position_candidate_dataframe,
    ):
        # 复用已经存在的 D1 client。
        self.d1_client = d1_client

        # 复制 successful submission DataFrame，避免修改上一个 cell 的对象。
        self.successful_submission_dataframe = (
            successful_submission_dataframe.copy()
        )

        # 复制 Position 候选 DataFrame。
        self.submission_position_candidate_dataframe = (
            submission_position_candidate_dataframe.copy()
        )

        # 预先声明每条 submission 的 rank-1 候选 DataFrame。
        self.rank_one_candidate_dataframe = None

        # 预先声明合并原始 submission 和 rank-1 外键后的 DataFrame。
        self.submission_with_candidate_dataframe = None

        # 预先声明完成 normalization 但尚未写入 D1 的 DataFrame。
        self.submission_normalized_dataframe = None

        # 预先声明写入后的 Submission_normalized id mapping DataFrame。
        self.submission_normalized_id_dataframe = None

        # 缓存 position_id -> application_cycle_id，避免同一 Position 重复查询。
        self._application_cycle_id_cache = {}

        # 缓存每一个唯一normalized domain的DNS验证结果。
        self._email_domain_dns_cache = {}

        # 保存本次执行的统计。
        self.stats = {
            "successful_submission_count": 0,
            "rank_one_candidate_count": 0,
            "cycles_inserted": 0,
            "cycles_reused": 0,
            "normalized_rows_built": 0,
            "submission_normalized_inserted": 0,
            "submission_normalized_updated": 0,
            "errors": [],
        }

    def _execute_with_retry(
        self,
        sql,
        params=None,
        max_retries=3,
        retry_delay_seconds=1.0,
    ):
        """复用 d1_client._execute()，只在 D1 请求失败时重试。"""

        # 保存最后一次请求失败的错误信息。
        last_error = None

        # 最多执行 max_retries 次，包括第一次请求。
        for attempt in range(1, max_retries + 1):
            # 调用已经存在的 D1 底层执行函数。
            ok, result = self.d1_client._execute(
                sql,
                params,
            )

            # 请求成功时立即返回。
            if ok:
                return result

            # 保存当前请求的错误信息。
            last_error = result

            # 如果还有重试机会，等待后再执行。
            if attempt < max_retries:
                time.sleep(
                    retry_delay_seconds
                )

        # 所有请求均失败后停止当前操作。
        raise RuntimeError(
            "D1 request failed after "
            f"{max_retries} attempts: {last_error}"
        )

    @staticmethod
    def _extract_d1_rows(result):
        """从 D1 REST API result 中提取真正的记录列表。"""

        # D1 没有返回 statement result 时使用空列表。
        if not result:
            return []

        # 提取第一条 SQL statement 的 results。
        return result[0].get(
            "results",
            [],
        )

    @staticmethod
    def _now_iso():
        """生成当前 UTC ISO 8601 时间。"""

        # 例如：2026-07-22T18:30:00Z。
        return datetime.now(timezone.utc).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        )

    @staticmethod
    def _clean_optional_text(value):
        """清理可选文本；缺失或空字符串统一返回 None。"""

        # pandas NaN / pd.NA 统一转成 Python None。
        if pd.isna(value):
            return None

        # 统一 Unicode 形式。
        cleaned_value = unicodedata.normalize(
            "NFKC",
            str(value),
        )

        # 合并连续空白，并删除首尾空白。
        cleaned_value = " ".join(
            cleaned_value.split()
        )

        # 空字符串也统一返回 None。
        return cleaned_value or None

    @classmethod
    def _normalize_lowercase_text(
        cls,
        value,
    ):
        """清理normalized文本，并使用casefold()统一成小写。"""

        # 先复用Unicode、空白和空值清理。
        cleaned_value = cls._clean_optional_text(
            value
        )

        # 空值继续保持None。
        if cleaned_value is None:
            return None

        # casefold()比lower()更适合Unicode大小写标准化。
        return cleaned_value.casefold()

    def _normalize_person_name(self, raw_person_name):
        """标准化姓名，并拆分 first / middle / last。"""

        # 保存姓名标准化过程中的 warnings。
        warnings = []

        # 先执行 Unicode、空格和空值清理。
        normalized_person_name = self._normalize_lowercase_text(
            raw_person_name
        )

        # 姓名缺失时无法拆分。
        if normalized_person_name is None:
            warnings.append(
                "person_name_missing"
            )
            return {
                "normalized_person_name": None,
                "normalized_first_name": None,
                "normalized_middle_name": None,
                "normalized_last_name": None,
                "warnings": warnings,
            }

        # 按空格拆分姓名 components。
        name_parts = normalized_person_name.split()

        # 只有一个 component 时，暂时存入 first_name。
        if len(name_parts) == 1:
            normalized_first_name = name_parts[0]
            normalized_middle_name = None
            normalized_last_name = None
            warnings.append(
                "single_component_person_name"
            )

        # 只有两个 components 时，按 first + last 处理。
        elif len(name_parts) == 2:
            normalized_first_name = name_parts[0]
            normalized_middle_name = None
            normalized_last_name = name_parts[1]

        # 三个或更多 components 时，中间部分统一当作 middle_name。
        else:
            normalized_first_name = name_parts[0]
            normalized_middle_name = " ".join(
                name_parts[1:-1]
            )
            normalized_last_name = name_parts[-1]

        # 返回完整姓名标准化结果。
        return {
            "normalized_person_name": normalized_person_name,
            "normalized_first_name": normalized_first_name,
            "normalized_middle_name": normalized_middle_name,
            "normalized_last_name": normalized_last_name,
            "warnings": warnings,
        }

    def _normalize_email(self, raw_person_email_address):
        """拆分、字符标准化email，并验证标准化domain的DNS状态。"""

        # 保存 email 标准化 warnings。
        warnings = []

        # 清理原始email首尾空白并统一Unicode，但保留原始显示内容。
        cleaned_email = self._clean_optional_text(
            raw_person_email_address
        )

        # Submission_normalized.raw_person_email_address 是 NOT NULL。
        if cleaned_email is None:
            raise ValueError(
                "raw_person_email_address is missing"
            )

        # 必须恰好包含一个@，否则无法唯一拆成prefix/domain。
        if cleaned_email.count("@") != 1:
            warnings.append(
                "email_cannot_be_split_into_one_prefix_and_one_domain"
            )
            return {
                "raw_person_email_address": cleaned_email,
                "normalized_email_prefix": None,
                "normalized_email_domain": None,
                "warnings": warnings,
            }

        # 在唯一@位置拆成原始prefix和domain。
        email_prefix, email_domain = cleaned_email.split(
            "@",
            1,
        )

        # prefix先执行NFKC，再小写化。
        prefix_before_removal = unicodedata.normalize(
            "NFKC",
            email_prefix,
        ).casefold()

        # 删除prefix允许集合之外的所有字符，包括“·”和空格。
        normalized_email_prefix = (
            self.EMAIL_PREFIX_ILLEGAL_CHARACTER_PATTERN.sub(
                "",
                prefix_before_removal,
            )
        )

        # 连续点号合并成一个，并删除prefix首尾点号。
        normalized_email_prefix = re.sub(
            r"\.{2,}",
            ".",
            normalized_email_prefix,
        ).strip(".")

        # 清理前后不同则记录warning。
        if normalized_email_prefix != prefix_before_removal:
            warnings.append(
                "email_prefix_illegal_characters_removed"
            )

        # 清理后为空或超过64 ASCII bytes时写入None。
        if (
            not normalized_email_prefix
            or len(
                normalized_email_prefix.encode("ascii")
            ) > 64
        ):
            normalized_email_prefix = None
            warnings.append(
                "normalized_email_prefix_invalid_after_cleaning"
            )

        # domain先执行NFKC，再小写化。
        domain_before_removal = unicodedata.normalize(
            "NFKC",
            email_domain,
        ).casefold()

        # 删除domain允许集合之外的所有字符。
        normalized_email_domain = (
            self.EMAIL_DOMAIN_ILLEGAL_CHARACTER_PATTERN.sub(
                "",
                domain_before_removal,
            )
        )

        # 连续点号合并成一个，并删除domain首尾点号。
        normalized_email_domain = re.sub(
            r"\.{2,}",
            ".",
            normalized_email_domain,
        ).strip(".")

        # 删除每个domain label首尾连字符。
        normalized_email_domain = ".".join(
            domain_label.strip("-")
            for domain_label
            in normalized_email_domain.split(".")
            if domain_label.strip("-")
        )

        # 清理前后不同则记录warning。
        if normalized_email_domain != domain_before_removal:
            warnings.append(
                "email_domain_illegal_characters_removed"
            )

        # 清理后为空、没有点号或超过253字符时写入None。
        if (
            not normalized_email_domain
            or "." not in normalized_email_domain
            or len(normalized_email_domain) > 253
        ):
            normalized_email_domain = None
            warnings.append(
                "normalized_email_domain_invalid_after_cleaning"
            )

        # 任意标准化component为None时记录统一warning。
        if (
            normalized_email_prefix is None
            or normalized_email_domain is None
        ):
            warnings.append(
                "normalized_email_component_missing"
            )

        # DNS验证只追加warning，不会修改normalized domain。
        if normalized_email_domain is not None:
            self._append_email_domain_dns_warning(
                normalized_email_domain,
                warnings,
            )

        # 返回原始email和标准化结果。
        return {
            "raw_person_email_address": cleaned_email,
            "normalized_email_prefix": normalized_email_prefix,
            "normalized_email_domain": normalized_email_domain,
            "warnings": warnings,
        }

    def _get_email_domain_dns_result(
        self,
        normalized_email_domain,
    ):
        """查询domain的MX/A/AAAA状态，并缓存结果。"""

        # 相同domain直接复用缓存。
        if (
            normalized_email_domain
            in self._email_domain_dns_cache
        ):
            return self._email_domain_dns_cache[
                normalized_email_domain
            ]

        # 创建有限超时时间的resolver。
        resolver = dns.resolver.Resolver()
        resolver.timeout = 3.0
        resolver.lifetime = 5.0

        # 预先声明DNS结果。
        dns_result = None

        try:
            # 优先查询MX。
            mx_answers = resolver.resolve(
                normalized_email_domain,
                "MX",
            )

            # 有MX时不生成warning。
            if len(mx_answers) > 0:
                dns_result = {
                    "status": "mx_exists",
                    "warning": None,
                }
            else:
                dns_result = {
                    "status": "no_relevant_dns_records",
                    "warning": (
                        "email_domain_dns_no_relevant_records"
                    ),
                }

        # NXDOMAIN明确表示domain不存在。
        except dns.resolver.NXDOMAIN:
            dns_result = {
                "status": "nxdomain",
                "warning": "email_domain_dns_nxdomain",
            }

        # 没有MX时继续检查A/AAAA。
        except dns.resolver.NoAnswer:
            address_record_exists = False
            address_check_indeterminate = False

            # 依次检查A和AAAA。
            for record_type in (
                "A",
                "AAAA",
            ):
                try:
                    address_answers = resolver.resolve(
                        normalized_email_domain,
                        record_type,
                    )

                    # 任一address record存在即可确认domain存在。
                    if len(address_answers) > 0:
                        address_record_exists = True

                # 当前record type没有answer时继续。
                except dns.resolver.NoAnswer:
                    continue

                # 明确NXDOMAIN时保存不存在状态。
                except dns.resolver.NXDOMAIN:
                    dns_result = {
                        "status": "nxdomain",
                        "warning": (
                            "email_domain_dns_nxdomain"
                        ),
                    }
                    break

                # DNS临时错误标记为无法确定。
                except (
                    dns.resolver.NoNameservers,
                    dns.exception.Timeout,
                ):
                    address_check_indeterminate = True

                # 其他DNS异常也只标记无法确定。
                except Exception:
                    address_check_indeterminate = True

            # 没有在循环中得到NXDOMAIN时，根据address结果分类。
            if dns_result is None:
                if address_record_exists:
                    dns_result = {
                        "status": (
                            "address_record_exists_without_mx"
                        ),
                        "warning": (
                            "email_domain_dns_exists_without_mx"
                        ),
                    }
                elif address_check_indeterminate:
                    dns_result = {
                        "status": "dns_check_indeterminate",
                        "warning": (
                            "email_domain_dns_check_indeterminate"
                        ),
                    }
                else:
                    dns_result = {
                        "status": "no_relevant_dns_records",
                        "warning": (
                            "email_domain_dns_no_relevant_records"
                        ),
                    }

        # timeout或nameserver错误只说明暂时无法确定。
        except (
            dns.resolver.NoNameservers,
            dns.exception.Timeout,
        ):
            dns_result = {
                "status": "dns_check_indeterminate",
                "warning": (
                    "email_domain_dns_check_indeterminate"
                ),
            }

        # 其他异常也只追加无法确定warning。
        except Exception:
            dns_result = {
                "status": "dns_check_indeterminate",
                "warning": (
                    "email_domain_dns_check_indeterminate"
                ),
            }

        # 缓存唯一domain结果。
        self._email_domain_dns_cache[
            normalized_email_domain
        ] = dns_result

        # 返回DNS结果。
        return dns_result

    def _append_email_domain_dns_warning(
        self,
        normalized_email_domain,
        warnings,
    ):
        """把DNS异常追加到warnings，但不改变normalized domain。"""

        # 查询或复用缓存结果。
        dns_result = self._get_email_domain_dns_result(
            normalized_email_domain
        )

        # 取得对应warning。
        dns_warning = dns_result.get(
            "warning"
        )

        # 有warning时追加。
        if dns_warning is not None:
            warnings.append(
                dns_warning
            )

    @staticmethod
    def _normalize_date(value, field_name):
        """尽力将日期转换成 YYYY-MM-DD；失败时返回 None 和 warning。"""

        # 空值不生成 warning，直接写入 SQL NULL。
        if pd.isna(value) or not str(value).strip():
            return None, None

        # pandas 尝试解析原始日期。
        parsed_date = pd.to_datetime(
            value,
            errors="coerce",
        )

        # 解析失败时返回 warning。
        if pd.isna(parsed_date):
            return (
                None,
                f"{field_name}_invalid_format",
            )

        # 统一输出 YYYY-MM-DD。
        return (
            parsed_date.strftime("%Y-%m-%d"),
            None,
        )

    @staticmethod
    def _normalize_submitted_at(value):
        """尽力将原始 created_at 转成 ISO 8601 submitted_at。"""

        # 原始提交时间缺失时返回 SQL NULL。
        if pd.isna(value) or not str(value).strip():
            return None, "submitted_at_missing"

        # 尝试解析原始提交时间。
        parsed_datetime = pd.to_datetime(
            value,
            errors="coerce",
            utc=True,
        )

        # 解析失败时返回 SQL NULL 和 warning。
        if pd.isna(parsed_datetime):
            return None, "submitted_at_invalid_format"

        # 统一输出 UTC ISO 8601 string。
        return (
            parsed_datetime.strftime(
                "%Y-%m-%dT%H:%M:%SZ"
            ),
            None,
        )

    def _validate_inputs_and_select_rank_one(self):
        """检查输入，并确保每条 successful submission 恰好有一个 rank-1 候选。"""

        # successful submission 必须包含的字段。
        required_submission_columns = {
            "raw_submission_id",
            "ingestion_batch_id",
            "name",
            "contact_email",
            "company",
            "position",
            "mode",
            "duration",
            "start_date",
            "end_date",
            "created_at",
        }

        # Position 候选明细必须包含的字段。
        required_candidate_columns = {
            "raw_submission_id",
            "candidate_rank",
            "match_type",
            "position_title_similarity_score",
            "candidate_company_id",
            "candidate_company_name",
            "position_id",
            "candidate_position_name",
        }

        # 找出 successful submission DataFrame 缺少的字段。
        missing_submission_columns = sorted(
            required_submission_columns
            - set(
                self.successful_submission_dataframe.columns
            )
        )

        # 找出 Position 候选 DataFrame 缺少的字段。
        missing_candidate_columns = sorted(
            required_candidate_columns
            - set(
                self.submission_position_candidate_dataframe.columns
            )
        )

        # 任意一份 DataFrame 缺少字段时停止。
        if missing_submission_columns:
            raise ValueError(
                "successful_submission_dataframe 缺少 columns："
                f"{missing_submission_columns}"
            )
        if missing_candidate_columns:
            raise ValueError(
                "submission_position_candidate_dataframe 缺少 columns："
                f"{missing_candidate_columns}"
            )

        # 保存 successful submission 总数。
        self.stats["successful_submission_count"] = len(
            self.successful_submission_dataframe
        )

        # 只保留 candidate_rank = 1 的候选记录。
        self.rank_one_candidate_dataframe = (
            self.submission_position_candidate_dataframe.loc[
                self.submission_position_candidate_dataframe[
                    "candidate_rank"
                ].astype("int64").eq(1)
            ].copy()
        )

        # 保存 rank-1 候选数量。
        self.stats["rank_one_candidate_count"] = len(
            self.rank_one_candidate_dataframe
        )

        # 检查 rank-1 DataFrame 中是否存在重复 raw_submission_id。
        if self.rank_one_candidate_dataframe[
            "raw_submission_id"
        ].duplicated().any():
            raise ValueError(
                "同一 raw_submission_id 存在多条 candidate_rank = 1 记录。"
            )

        # 检查 successful submission 和 rank-1 候选数量是否一致。
        if len(self.rank_one_candidate_dataframe) != len(
            self.successful_submission_dataframe
        ):
            raise ValueError(
                "candidate_rank = 1 的记录数与 successful submission 数不一致。"
            )

        # 只保留合并需要的候选字段。
        rank_one_columns = self.rank_one_candidate_dataframe[
            [
                "raw_submission_id",
                "match_type",
                "position_title_similarity_score",
                "candidate_company_id",
                "candidate_company_name",
                "position_id",
                "candidate_position_name",
            ]
        ].copy()

        # 通过 raw_submission_id 将原始 successful submission 与 rank-1 候选一对一合并。
        self.submission_with_candidate_dataframe = (
            self.successful_submission_dataframe.merge(
                rank_one_columns,
                on="raw_submission_id",
                how="left",
                validate="one_to_one",
            )
        )

        # 合并后任何 position_id 缺失都不能继续。
        if self.submission_with_candidate_dataframe[
            "position_id"
        ].isna().any():
            raise ValueError(
                "至少一条 successful submission 没有 rank-1 position_id。"
            )

        # 显示已选中的 rank-1 候选数量。
        print(
            "Rank-1 candidates selected:",
            len(self.rank_one_candidate_dataframe),
        )

    def _get_or_create_unknown_cycle(self, position_id):
        """为一个 Position 查询或建立唯一的 Unknown application cycle。"""

        # 将 position_id 统一转成 Python int。
        position_id = int(
            position_id
        )

        # 已经缓存的 Position 直接复用 cycle id。
        if position_id in self._application_cycle_id_cache:
            return (
                self._application_cycle_id_cache[position_id],
                "reused",
            )

        # 使用与唯一 index 相同的 LOWER(TRIM()) 逻辑查询 Unknown cycle。
        result = self._execute_with_retry(
            """
                SELECT "id"
                FROM "Position_application_cycle"
                WHERE "position_id" = ?
                  AND LOWER(TRIM("cycle_name")) = LOWER(TRIM(?))
                ORDER BY "id";
            """,
            [
                position_id,
                self.UNKNOWN_APPLICATION_CYCLE_NAME,
            ],
        )

        # 提取已存在的 cycle records。
        matching_rows = self._extract_d1_rows(
            result
        )

        # 找到唯一记录时直接复用。
        if len(matching_rows) == 1:
            application_cycle_id = int(
                matching_rows[0]["id"]
            )
            self._application_cycle_id_cache[
                position_id
            ] = application_cycle_id
            return application_cycle_id, "reused"

        # 找到多条时说明数据与唯一 index 的预期不一致。
        if len(matching_rows) > 1:
            raise RuntimeError(
                "同一 Position 存在多条 Unknown application cycles："
                f"position_id={position_id}"
            )

        # 第一次创建时 created_at 和 updated_at 使用同一 UTC 时间。
        now = self._now_iso()

        # 建立一条 Unknown application cycle。
        result = self._execute_with_retry(
            """
                INSERT INTO "Position_application_cycle" (
                  "position_id",
                  "cycle_name",
                  "cycle_start_month",
                  "cycle_end_month",
                  "duration_months",
                  "created_at",
                  "updated_at"
                ) VALUES (?, ?, ?, ?, ?, ?, ?)
                RETURNING "id";
            """,
            [
                position_id,
                self.UNKNOWN_APPLICATION_CYCLE_NAME,
                None,
                None,
                None,
                now,
                now,
            ],
        )

        # 提取 D1 自动生成的 application cycle id。
        inserted_rows = self._extract_d1_rows(
            result
        )

        # RETURNING 必须返回一条记录。
        if len(inserted_rows) != 1:
            raise RuntimeError(
                "创建 Unknown application cycle 后没有取得唯一 id。"
            )

        # 保存新生成的 application cycle id。
        application_cycle_id = int(
            inserted_rows[0]["id"]
        )
        self._application_cycle_id_cache[
            position_id
        ] = application_cycle_id

        # 返回 cycle id 和 inserted action。
        return application_cycle_id, "inserted"

    def _attach_application_cycle_ids(self):
        """为所有 rank-1 Position 建立或复用 Unknown cycle，并回填 id。"""

        # 取得所有不重复的 rank-1 position_id。
        unique_position_ids = sorted(
            self.submission_with_candidate_dataframe[
                "position_id"
            ].astype("int64").unique()
        )

        # 逐个处理被选中的 Position。
        for position_id in unique_position_ids:
            # 查询或建立当前 Position 的 Unknown cycle。
            application_cycle_id, action = (
                self._get_or_create_unknown_cycle(
                    position_id
                )
            )

            # 更新 inserted / reused 统计。
            self.stats[
                f"cycles_{action}"
            ] += 1

        # 通过 position_id 缓存将 application_cycle_id 回填到每条 submission。
        self.submission_with_candidate_dataframe[
            "application_cycle_id"
        ] = self.submission_with_candidate_dataframe[
            "position_id"
        ].astype("int64").map(
            self._application_cycle_id_cache
        )

        # application_cycle_id 是 Submission_normalized 的必需外键。
        if self.submission_with_candidate_dataframe[
            "application_cycle_id"
        ].isna().any():
            raise RuntimeError(
                "至少一条 submission 没有取得 application_cycle_id。"
            )

        # 将 application_cycle_id 转成整数。
        self.submission_with_candidate_dataframe[
            "application_cycle_id"
        ] = self.submission_with_candidate_dataframe[
            "application_cycle_id"
        ].astype("int64")

        # 显示本次使用的不重复 Position 数量。
        print(
            "Unique rank-1 positions with Unknown cycles:",
            len(unique_position_ids),
        )

    def _normalize_one_submission(self, row):
        """将一条 submission 转换成 Submission_normalized 字段。"""

        # 保存当前记录的所有 normalization warnings。
        warning_values = []

        # 标准化和拆分姓名。
        name_result = self._normalize_person_name(
            row["name"]
        )
        warning_values.extend(
            name_result["warnings"]
        )

        # 标准化和拆分 email。
        email_result = self._normalize_email(
            row["contact_email"]
        )
        warning_values.extend(
            email_result["warnings"]
        )

        # 标准化 start working date。
        start_working_date, start_date_warning = (
            self._normalize_date(
                row["start_date"],
                "start_working_date",
            )
        )
        if start_date_warning:
            warning_values.append(
                start_date_warning
            )

        # 标准化 end working date。
        end_working_date, end_date_warning = (
            self._normalize_date(
                row["end_date"],
                "end_working_date",
            )
        )
        if end_date_warning:
            warning_values.append(
                end_date_warning
            )

        # 开始日期晚于结束日期时保留 warning，但不中断整个批次。
        if (
            start_working_date is not None
            and end_working_date is not None
            and start_working_date > end_working_date
        ):
            warning_values.append(
                "start_working_date_after_end_working_date"
            )

        # 标准化 submission 原始 created_at。
        submitted_at, submitted_at_warning = (
            self._normalize_submitted_at(
                row["created_at"]
            )
        )
        if submitted_at_warning:
            warning_values.append(
                submitted_at_warning
            )

        # 把 rank-1 映射决定写入 warnings，便于后续追溯。
        mapping_warning = {
            "warning_type": "rank_one_position_candidate_used_as_submission_foreign_keys",
            "match_type": str(row["match_type"]),
            "position_title_similarity_score": float(
                row["position_title_similarity_score"]
            ),
            "raw_company_name": self._clean_optional_text(
                row["company"]
            ),
            "candidate_company_id": int(
                row["candidate_company_id"]
            ),
            "candidate_company_name": str(
                row["candidate_company_name"]
            ),
            "raw_position_name": self._clean_optional_text(
                row["position"]
            ),
            "candidate_position_id": int(
                row["position_id"]
            ),
            "candidate_position_name": str(
                row["candidate_position_name"]
            ),
        }

        # 只要 warnings 中有内容，就标记 normalized_with_warnings。
        #
        # 因为本次每条记录都使用 rank-1 候选外键，
        # 所以每条记录都会保留 mapping warning。
        normalization_status = "normalized_with_warnings"

        # 将普通 normalization warnings 和结构化 mapping warning 一起封装成 JSON。
        warnings_text = json.dumps(
            {
                "normalization_warnings": warning_values,
                "position_mapping_warning": mapping_warning,
            },
            ensure_ascii=False,
        )

        # 同一条 normalized record 的时间字段使用同一时刻。
        now = self._now_iso()

        # 返回与 Submission_normalized schema 对应的完整记录。
        return {
            "raw_submission_id": int(row["raw_submission_id"]),
            "application_cluster_id": None,
            "open_window_id": None,
            "company_id": int(row["candidate_company_id"]),
            "position_id": int(row["position_id"]),
            "application_cycle_id": int(row["application_cycle_id"]),
            "raw_person_name": self._clean_optional_text(row["name"]),
            "raw_person_email_address": email_result["raw_person_email_address"],
            "raw_person_phone": None,
            "person_name_aliases": None,
            "normalized_person_name": name_result["normalized_person_name"],
            "normalized_first_name": name_result["normalized_first_name"],
            "normalized_middle_name": name_result["normalized_middle_name"],
            "normalized_last_name": name_result["normalized_last_name"],
            "normalized_email_prefix": email_result["normalized_email_prefix"],
            "normalized_email_domain": email_result["normalized_email_domain"],
            "normalized_person_phone": None,
            "normalized_phone_country_code": None,
            "normalized_phone_region_code": None,
            "normalized_phone_number": None,
            "work_type": self._normalize_lowercase_text(row["mode"]),
            "work_duration": self._normalize_lowercase_text(row["duration"]),
            "start_working_date": start_working_date,
            "end_working_date": end_working_date,
            "normalization_status": normalization_status,
            "normalized_at": now,
            "warnings": warnings_text,
            "submitted_at": submitted_at,
            "created_at": now,
            "updated_at": now,
            "normalized_company_name": self._normalize_lowercase_text(row["company"]),
            "normalized_position_name": self._normalize_lowercase_text(row["position"]),
            "normalized_application_cycle_name": self._normalize_lowercase_text(
                self.UNKNOWN_APPLICATION_CYCLE_NAME
                ),
            "resume_text": row["resume_text"],

        }

    def _build_normalized_dataframe(self):
        """在内存中生成完整 Submission_normalized DataFrame。"""

        # 保存所有标准化记录。
        normalized_records = []

        # 逐条标准化 successful submissions。
        for _, row in self.submission_with_candidate_dataframe.iterrows():
            normalized_records.append(
                self._normalize_one_submission(
                    row
                )
            )

        # 转成可以在写入前查看的 DataFrame。
        self.submission_normalized_dataframe = pd.DataFrame(
            normalized_records
        )

        # 保存已生成的 normalized record 数量。
        self.stats["normalized_rows_built"] = len(
            self.submission_normalized_dataframe
        )

        # 确保生成数量与 successful submission 数一致。
        if len(self.submission_normalized_dataframe) != len(
            self.successful_submission_dataframe
        ):
            raise RuntimeError(
                "Submission_normalized DataFrame 记录数与 successful submission 数不一致。"
            )

        # 确保三个 NOT NULL 外键没有空值。
        required_foreign_key_columns = [
            "company_id",
            "position_id",
            "application_cycle_id",
        ]
        if self.submission_normalized_dataframe[
            required_foreign_key_columns
        ].isna().any().any():
            raise RuntimeError(
                "Submission_normalized DataFrame 的必需外键存在空值。"
            )

        # 显示写入 D1 前的前 10 条 normalized records。
        print("\nSubmission_normalized preview before D1 insert:")
        display(
            self.submission_normalized_dataframe.head(10)
        )

    def _get_existing_submission_normalized_id(self, raw_submission_id):
        """按 raw_submission_id 查询已经存在的 Submission_normalized record。"""

        # 执行只读查询。
        result = self._execute_with_retry(
            """
                SELECT "id"
                FROM "Submission_normalized"
                WHERE "raw_submission_id" = ?
                ORDER BY "id";
            """,
            [
                int(raw_submission_id),
            ],
        )

        # 提取已存在记录。
        matching_rows = self._extract_d1_rows(
            result
        )

        # 不存在时返回 None。
        if not matching_rows:
            return None

        # 同一 raw_submission_id 存在多条 normalized records 时停止，避免继续扩大重复数据。
        if len(matching_rows) > 1:
            raise RuntimeError(
                "同一 raw_submission_id 已经存在多条 Submission_normalized records："
                f"raw_submission_id={raw_submission_id}"
            )

        # 返回唯一的已存在 normalized id。
        return int(
            matching_rows[0]["id"]
        )

    def _insert_one_normalized_record(self, row):
        """插入一条 Submission_normalized record，并返回自动生成的 id。"""

        # INSERT columns 与 params 严格保持相同顺序。
        sql = """
            INSERT INTO "Submission_normalized" (
              "raw_submission_id",
              "application_cluster_id",
              "open_window_id",
              "company_id",
              "position_id",
              "application_cycle_id",
              "raw_person_name",
              "raw_person_email_address",
              "raw_person_phone",
              "person_name_aliases",
              "normalized_person_name",
              "normalized_first_name",
              "normalized_middle_name",
              "normalized_last_name",
              "normalized_email_prefix",
              "normalized_email_domain",
              "normalized_person_phone",
              "normalized_phone_country_code",
              "normalized_phone_region_code",
              "normalized_phone_number",
              "work_type",
              "work_duration",
              "start_working_date",
              "end_working_date",
              "normalization_status",
              "normalized_at",
              "warnings",
              "submitted_at",
              "created_at",
              "updated_at",
              "normalized_company_name",
              "normalized_position_name",
              "normalized_application_cycle_name"
            ) VALUES (
              ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
              ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
              ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            RETURNING "id";
        """

        # 按 INSERT columns 的相同顺序准备 parameters。
        params = [
            row["raw_submission_id"],
            row["application_cluster_id"],
            row["open_window_id"],
            row["company_id"],
            row["position_id"],
            row["application_cycle_id"],
            row["raw_person_name"],
            row["raw_person_email_address"],
            row["raw_person_phone"],
            row["person_name_aliases"],
            row["normalized_person_name"],
            row["normalized_first_name"],
            row["normalized_middle_name"],
            row["normalized_last_name"],
            row["normalized_email_prefix"],
            row["normalized_email_domain"],
            row["normalized_person_phone"],
            row["normalized_phone_country_code"],
            row["normalized_phone_region_code"],
            row["normalized_phone_number"],
            row["work_type"],
            row["work_duration"],
            row["start_working_date"],
            row["end_working_date"],
            row["normalization_status"],
            row["normalized_at"],
            row["warnings"],
            row["submitted_at"],
            row["created_at"],
            row["updated_at"],
            row["normalized_company_name"],
            row["normalized_position_name"],
            row["normalized_application_cycle_name"],
        ]

        # 执行 INSERT。
        result = self._execute_with_retry(
            sql,
            params,
        )

        # 提取新建记录的 id。
        inserted_rows = self._extract_d1_rows(
            result
        )

        # RETURNING 必须返回一条 id。
        if len(inserted_rows) != 1:
            raise RuntimeError(
                "插入 Submission_normalized 后没有取得唯一 id。"
            )

        # 返回 D1 生成的 normalized id。
        return int(
            inserted_rows[0]["id"]
        )

    def _update_existing_normalized_record(
        self,
        submission_normalized_id,
        row,
    ):
        """更新已经存在的 Submission_normalized 标准化结果。"""

        # UPDATE 只修改本 cell 负责重新计算的标准化字段。
        #
        # application_cluster_id 和 open_window_id 不在这里更新，
        # 避免未来已经完成分流后，重复运行本 cell 时清空它们。
        #
        # phone相关字段和person_name_aliases也不在这里更新，
        # 避免未来完成简历拆分和信息回填后，被本cell重新写成NULL。
        sql = """
            UPDATE "Submission_normalized"
            SET
              "company_id" = ?,
              "position_id" = ?,
              "application_cycle_id" = ?,
              "raw_person_name" = ?,
              "raw_person_email_address" = ?,
              "normalized_person_name" = ?,
              "normalized_first_name" = ?,
              "normalized_middle_name" = ?,
              "normalized_last_name" = ?,
              "normalized_email_prefix" = ?,
              "normalized_email_domain" = ?,
              "work_type" = ?,
              "work_duration" = ?,
              "start_working_date" = ?,
              "end_working_date" = ?,
              "normalization_status" = ?,
              "normalized_at" = ?,
              "warnings" = ?,
              "submitted_at" = ?,
              "updated_at" = ?,
              "normalized_company_name" = ?,
              "normalized_position_name" = ?,
              "normalized_application_cycle_name" = ?
            WHERE "id" = ?
              AND "raw_submission_id" = ?;
        """

        # 按 UPDATE columns 的相同顺序准备 parameters。
        params = [
            int(row["company_id"]),
            int(row["position_id"]),
            int(row["application_cycle_id"]),
            row["raw_person_name"],
            row["raw_person_email_address"],
            row["normalized_person_name"],
            row["normalized_first_name"],
            row["normalized_middle_name"],
            row["normalized_last_name"],
            row["normalized_email_prefix"],
            row["normalized_email_domain"],
            row["work_type"],
            row["work_duration"],
            row["start_working_date"],
            row["end_working_date"],
            row["normalization_status"],
            row["normalized_at"],
            row["warnings"],
            row["submitted_at"],
            row["updated_at"],
            row["normalized_company_name"],
            row["normalized_position_name"],
            row["normalized_application_cycle_name"],
            int(submission_normalized_id),
            int(row["raw_submission_id"]),
        ]

        # 执行 UPDATE。
        self._execute_with_retry(
            sql,
            params,
        )

    def _write_normalized_records(self, verbose=True):
        """逐条写入 Submission_normalized；已存在的记录会被更新。"""

        # 保存 raw_submission_id 与 submission_normalized_id 对应关系。
        id_mapping_rows = []

        # 逐条处理 normalized DataFrame。
        for processed_count, (_, row) in enumerate(
            self.submission_normalized_dataframe.iterrows(),
            start=1,
        ):
            # 取得当前 raw_submission_id。
            raw_submission_id = int(
                row["raw_submission_id"]
            )

            try:
                # 先检查当前 raw submission 是否已经标准化并写入。
                submission_normalized_id = (
                    self._get_existing_submission_normalized_id(
                        raw_submission_id
                    )
                )

                # 已经存在时更新本 cell 负责的标准化字段，不重复 INSERT。
                if submission_normalized_id is not None:
                    self._update_existing_normalized_record(
                        submission_normalized_id,
                        row,
                    )
                    action = "updated"

                # 不存在时才新建记录。
                else:
                    submission_normalized_id = (
                        self._insert_one_normalized_record(
                            row
                        )
                    )
                    action = "inserted"

                # 更新 inserted / updated 统计。
                self.stats[
                    f"submission_normalized_{action}"
                ] += 1

                # 保存 id mapping。
                id_mapping_rows.append(
                    {
                        "raw_submission_id": raw_submission_id,
                        "submission_normalized_id": submission_normalized_id,
                        "action": action,
                    }
                )

            except Exception as error:
                # 保存失败记录，继续处理后续 submission。
                self.stats["errors"].append(
                    {
                        "raw_submission_id": raw_submission_id,
                        "error": str(error),
                    }
                )
                print(
                    "Submission_normalized import failed:",
                    f"raw_submission_id={raw_submission_id},",
                    f"error={error}",
                )

            # 每处理 100 条记录显示一次进度。
            if verbose and processed_count % 100 == 0:
                print(
                    "Processed normalized records:",
                    f"{processed_count}/"
                    f"{len(self.submission_normalized_dataframe)}",
                )

        # 生成可供后续 cell 使用的 id mapping DataFrame。
        self.submission_normalized_id_dataframe = pd.DataFrame(
            id_mapping_rows
        )

    def _verify_database_result(self):
        """验证本批 successful submissions 在 D1 中都有且只有一条 normalized record。"""

        # 使用 ingestion_batch_id 查询这一批 raw submissions 对应的 normalized 数量。
        ingestion_batch_ids = sorted(
            self.successful_submission_dataframe[
                "ingestion_batch_id"
            ].astype("int64").unique()
        )

        # 当前 DataFrame 预期只属于一个 ingestion batch。
        if len(ingestion_batch_ids) != 1:
            raise ValueError(
                "successful_submission_dataframe 不是唯一 ingestion batch。"
            )

        # 取得唯一 ingestion_batch_id。
        # pandas / NumPy 从 int64 column 取出的值可能是 numpy.int64。
        #
        # requests 发送 D1 JSON payload 时不能直接序列化 numpy.int64，
        # 所以这里必须显式转成标准 Python int。
        ingestion_batch_id = int(
            ingestion_batch_ids[0]
        )

        # 统计这一批 successful raw submissions 对应的 normalized records。
        result = self._execute_with_retry(
            """
                SELECT
                  COUNT(sn."id") AS normalized_record_count,
                  COUNT(DISTINCT sn."raw_submission_id")
                    AS distinct_raw_submission_count
                FROM "Submission_normalized" AS sn
                INNER JOIN "raw_submission" AS rs
                  ON rs."id" = sn."raw_submission_id"
                WHERE rs."ingestion_batch_id" = ?;
            """,
            [
                ingestion_batch_id,
            ],
        )

        # 提取验证结果。
        verification_rows = self._extract_d1_rows(
            result
        )
        verification = verification_rows[0]

        # 显示 D1 验证结果。
        print(
            "D1 Submission_normalized verification:",
            verification,
        )

        # 有错误时不要将数量不完整误报为新的验证异常。
        if self.stats["errors"]:
            return

        # 没有错误时，数据库记录数应与 successful submission 数一致。
        expected_count = len(
            self.successful_submission_dataframe
        )
        if (
            int(verification["normalized_record_count"])
            != expected_count
            or int(verification["distinct_raw_submission_count"])
            != expected_count
        ):
            raise RuntimeError(
                "D1 Submission_normalized 验证数量与 successful submission 数不一致。"
            )

    def run(self, verbose=True):
        """执行完整 normalization、Unknown cycle 建立和 D1 导入。"""

        # 第一步：检查两个 DataFrame，并只选取 candidate_rank = 1。
        self._validate_inputs_and_select_rank_one()

        # 第二步：为所有 rank-1 Position 建立或复用 Unknown cycle。
        self._attach_application_cycle_ids()

        # 第三步：在内存中生成完整 Submission_normalized DataFrame。
        self._build_normalized_dataframe()

        # 第四步：逐条写入 D1，已存在的 raw_submission_id 会更新。
        self._write_normalized_records(
            verbose=verbose
        )

        # 第五步：验证数据库写入数量。
        self._verify_database_result()

        # 显示本次执行汇总。
        print("\n" + "=" * 60)
        print("Submission normalization import summary")
        print("=" * 60)
        for stat_name, stat_value in self.stats.items():
            if stat_name != "errors":
                print(
                    f"  {stat_name}: {stat_value}"
                )
        print(
            "  errors:",
            len(self.stats["errors"]),
        )

        # 返回标准化 DataFrame 和 D1 id mapping DataFrame。
        return (
            self.submission_normalized_dataframe,
            self.submission_normalized_id_dataframe,
        )


# ============================================================
# 运行本 cell
# ============================================================

# 运行前只需要已经存在：
#
# loader
# successful_submission_dataframe
#
# Submission_normalized 也必须已经添加：
# normalized_company_name
# normalized_position_name
# normalized_application_cycle_name

# 设置判断近似 Position name 候选时使用的最低相似度。
APPROXIMATE_MATCH_THRESHOLD = 0.80

# 创建内置的 rank-1 Position candidate builder。
rank_one_position_candidate_builder = (
    RankOnePositionCandidateBuilder(
        d1_client=loader,
        successful_submission_dataframe=(
            successful_submission_dataframe
        ),
        approximate_match_threshold=(
            APPROXIMATE_MATCH_THRESHOLD
        ),
    )
)

# 直接从 successful_submission_dataframe 和 D1 Position 生成 rank-1 候选。
#
# 这里不再要求提前运行另一个 Position candidate matching cell。
submission_position_candidate_dataframe = (
    rank_one_position_candidate_builder.run()
)

# 显示内置候选匹配生成的前 10 条 rank-1 records。
display(
    submission_position_candidate_dataframe.head(10)
)

# 创建 Submission_normalized loader。
submission_normalized_loader = (
    SubmissionNormalizedLoader(
        d1_client=loader,
        successful_submission_dataframe=(
            successful_submission_dataframe
        ),
        submission_position_candidate_dataframe=(
            submission_position_candidate_dataframe
        ),
    )
)

# 执行 normalization 和 D1 导入。
(
    submission_normalized_dataframe,
    submission_normalized_id_dataframe,
) = submission_normalized_loader.run()

# 显示成功写入或更新的 id mapping。
display(
    submission_normalized_id_dataframe.head(10)
)

# 如果需要查看错误记录，可以运行：
# display(
#     pd.DataFrame(
#         submission_normalized_loader.stats["errors"]
#     )
# )

# 本 cell 到这里结束。
# 暂时不回填 application_cluster_id。
# 暂时不回填 open_window_id。
# 暂时不从 resume_text 提取 phone。




In [ ]:
# ============================================================
# 存CSV，供 CV_Extraction 直接读取
# ============================================================

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
SUCCESSFUL_SUBMISSION_FOR_EXTRACTION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/successful_submission_for_extraction.csv"
)

successful_submission_for_extraction = successful_submission_dataframe[
    ["raw_submission_id", "resume_text"]
].merge(
    submission_normalized_id_dataframe[["raw_submission_id", "submission_normalized_id"]],
    on="raw_submission_id",
    how="inner",
)

successful_submission_for_extraction.to_csv(
    SUCCESSFUL_SUBMISSION_FOR_EXTRACTION_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved for Extraction: {SUCCESSFUL_SUBMISSION_FOR_EXTRACTION_CSV_PATH}")
print(f"  rows: {len(successful_submission_for_extraction)}")

# CV_Extraction

In [ ]:
# ============================================================
# 独立的 D1 连接 cell —— 专门给 CV_Extraction 这部分用
# 不依赖任何其他 cell（不需要 RawSubmissionLoader 那个类，
# 不需要之前跑没跑过 cell 15）
# ============================================================

import requests
from google.colab import drive, userdata

drive.mount('/content/drive')

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get('CLOUDFLARE_KEY')


class D1Client:
    """
    最基础的 D1 连接对象，只有一个 _execute() 方法。
    CV_Extraction 这条线上用到的几个函数
    （fetch_successful_submission_dataframe_from_db /
    build_resume_extraction_dataframes / ResumeExtractionInjector）
    都只依赖对象有一个 _execute(sql, params) 方法，不关心这个对象
    具体叫什么类名，所以这里单独定义一个最简版本就够用，
    不需要借用 RawSubmissionLoader 那个功能更多、但依赖更多的类。
    """

    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"
        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])
        return True, data.get("result", [])


loader = D1Client(ACCOUNT_ID, DATABASE_ID, API_TOKEN)

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/recruiting-pipeline/code")

from resume_text_parser import ResumeTextParser, ResumeParsingError
from resume_skill_matcher import ResumeSkillMatcher
# fetch_successful_submission_from_db 这次不需要了（改成读CSV），
# 这一行删掉，不导入

parser = ResumeTextParser()
matcher = ResumeSkillMatcher(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
matcher.load_skill_lookup()   # 这一步会加载全部Skill参考表，需要几秒到十几秒

In [ ]:
"""
build_resume_extraction_dataframes.py（Google Colab 版）

把 successful_submission_dataframe 里每一行的 resume_text，跑一遍
ResumeTextParser + ResumeSkillMatcher，产出结构对应 Resume_extraction
以及它下面几张子表的独立 DataFrame——不会往 successful_submission_dataframe
本身写任何东西，这两层数据完全分开。

已知的外键关系（已确认，不是待定）：
  Resume_extraction.submission_normalized_id 是 NOT NULL UNIQUE，
  外键指向 Submission_normalized.id（不是 attachment_id/
  Submission_attachment——resume_text 不是真实上传的附件文件，
  这条流程也不会真正生成 Submission_attachment 记录，硬套那条
  外键关系是错的，已经改成直接关联 Submission_normalized）。
  resume_extraction_df 里这一列的值，来自
  fetch_successful_submission_dataframe_from_db() 查询时顺便
  JOIN 出来的 Submission_normalized.id。

resume_emails / resume_phones 已经确认加进了 Resume_extraction 表
（多个值之间用分号拼接成一个字符串存储，不是单独建子表），
这两列现在是正式字段名，不再是待定的 _raw 后缀。

产出（一个 dict，5 个 DataFrame，彼此靠 raw_submission_id 关联）：
  resume_extraction_df           一行 = 一条 submission 的整体提取结果
  resume_education_df            一行 = 一条教育经历条目（一对多，展开成长表）
  resume_employment_df           一行 = 一条工作经历条目（一对多，展开成长表）
  resume_external_identifier_df  一行 = 一条 LinkedIn/GitHub 链接（一对多）
  resume_skill_matched_df        一行 = 一条匹配上 Skill 参考表的技能（一对多）

用法（Colab cell 里直接调用）：
    parser = ResumeTextParser()
    matcher = ResumeSkillMatcher(ACCOUNT_ID, DATABASE_ID, API_TOKEN)
    matcher.load_skill_lookup()

    dataframes = build_resume_extraction_dataframes(
        successful_submission_dataframe, parser, matcher
    )
    resume_extraction_df = dataframes["resume_extraction"]
    resume_education_df = dataframes["resume_education"]
    ...
"""

import hashlib
import os
from datetime import datetime, timezone

import pandas as pd
import requests

from resume_text_parser import ResumeTextParser, ResumeParsingError
from resume_skill_matcher import ResumeSkillMatcher

# 合并后的大CSV存放路径，跟项目里其他文件用同一个共享Drive文件夹
DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
CONSOLIDATED_CSV_PATH = f"{DRIVE_FOLDER}/datasets/resume_extraction_consolidated.csv"


def _now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


class ResumeExtractionInjector:
    """
    把 build_resume_extraction_dataframes() 产出的
    resume_extraction_df / resume_education_df / resume_employment_df
    这三个 DataFrame，真正写进 D1 数据库对应的三张表。

    只处理 submission 层次的三张表（Resume_extraction / Resume_education /
    Resume_employment），不处理 Resume_external_identifier、也不处理
    技能匹配结果——那些属于身份识别/application 层次的东西，按你的
    要求这次不碰。

    写入顺序固定：
      1. 先插入 Resume_extraction 这一行，拿到数据库自动生成的
         resume_extraction_id
      2. 再拿这个 resume_extraction_id，去插入这一条 submission
         对应的全部教育经历（Resume_education）和工作经历
         （Resume_employment）——这两张表的外键指向的是
         resume_extraction_id，不是 raw_submission_id，
         所以必须先有第1步的结果才能做第2步
    """

    def __init__(self, account_id: str, database_id: str, api_token: str):
        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params
        resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
        try:
            data = resp.json()
        except ValueError:
            return False, f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"
        if resp.status_code != 200 or not data.get("success", False):
            return False, data.get("errors", [])
        return True, data.get("result", [])

    @staticmethod
    def _bool_to_int(value):
        """
        is_current 这类字段在 Python 里是 True/False/None，
        SQLite 没有原生布尔类型，True/False 要转成整数 1/0 才能存，
        None 原样传 None（对应存 NULL），不强行转换。
        """
        if value is None:
            return None
        return 1 if value else 0

    def insert_resume_extraction_row(self, row: dict) -> int:
        """
        插入一条 Resume_extraction 记录，返回数据库自动生成的 id。

        row 需要包含的 key（对应表字段名）：
          submission_normalized_id, cleaned_resume_text, resume_text_sha256,
          text_length, extraction_status, extraction_error,
          resume_emails, resume_phones

        submission_normalized_id 是 NOT NULL UNIQUE 的必填字段，
        指向 Submission_normalized.id（不是 attachment_id/
        Submission_attachment——resume_text 不是真实上传的附件文件，
        这条外键关系已经改过来了）。这一行如果没有真实的
        submission_normalized_id，直接在这里报错拦下来，不会尝试
        往数据库塞 None 然后得到一个含糊的报错。
        """
        submission_normalized_id = row.get("submission_normalized_id")
        if submission_normalized_id is None or (
            isinstance(submission_normalized_id, float) and pd.isna(submission_normalized_id)
        ):
            raise ValueError(
                "submission_normalized_id 是 None，Resume_extraction.submission_normalized_id "
                "是 NOT NULL 且 UNIQUE 的必填字段，不能插入。"
                "请确认 fetch_successful_submission_dataframe_from_db() 返回的 DataFrame "
                "里包含了这一列。"
            )

        now = _now_iso()
        sql = """
            INSERT INTO "Resume_extraction" (
              "submission_normalized_id", "extracted_raw_resume_text", "cleaned_resume_text",
              "resume_text_sha256", "text_length", "extraction_status",
              "extraction_error", "resume_emails", "resume_phones",
              "extracted_at", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            RETURNING "id";
        """
        params = [
            int(submission_normalized_id),
            row.get("cleaned_resume_text"),   # extracted_raw_resume_text 和 cleaned_resume_text
            row.get("cleaned_resume_text"),   # 目前是同一份内容，没有额外做二次清洗
            row.get("resume_text_sha256"),
            row.get("text_length"),
            row.get("extraction_status"),
            row.get("extraction_error"),
            row.get("resume_emails"),
            row.get("resume_phones"),
            row.get("extracted_at") or now,
            now, now,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Resume_extraction failed: {result}")
        rows = result[0].get("results", []) if result else []
        return rows[0]["id"]

    def insert_resume_education_row(self, resume_extraction_id: int, row: dict) -> int:
        now = _now_iso()
        sql = """
            INSERT INTO "Resume_education" (
              "resume_extraction_id", "splitted_raw_education_text",
              "education_description", "normalized_school_name",
              "normalized_degree", "normalized_field_of_study",
              "is_current", "start_year", "start_month", "end_year", "end_month",
              "extraction_confidence", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            RETURNING "id";
        """
        params = [
            resume_extraction_id,
            row.get("splitted_raw_education_text"),
            row.get("education_description"),
            row.get("normalized_school_name"),
            row.get("normalized_degree"),
            row.get("normalized_field_of_study"),
            self._bool_to_int(row.get("is_current")),
            row.get("start_year"), row.get("start_month"),
            row.get("end_year"), row.get("end_month"),
            row.get("extraction_confidence"),
            now, now,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Resume_education failed: {result}")
        rows = result[0].get("results", []) if result else []
        return rows[0]["id"]

    def insert_resume_employment_row(self, resume_extraction_id: int, row: dict) -> int:
        now = _now_iso()
        sql = """
            INSERT INTO "Resume_employment" (
              "resume_extraction_id", "splitted_raw_employment_text",
              "employment_description", "normalized_company_name",
              "normalized_job_title", "is_current",
              "start_year", "start_month", "end_year", "end_month",
              "extraction_confidence", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            RETURNING "id";
        """
        params = [
            resume_extraction_id,
            row.get("splitted_raw_employment_text"),
            row.get("employment_description"),
            row.get("normalized_company_name"),
            row.get("normalized_job_title"),
            self._bool_to_int(row.get("is_current")),
            row.get("start_year"), row.get("start_month"),
            row.get("end_year"), row.get("end_month"),
            row.get("extraction_confidence"),
            now, now,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Resume_employment failed: {result}")
        rows = result[0].get("results", []) if result else []
        return rows[0]["id"]

    def inject_all(self, resume_extraction_df: pd.DataFrame,
                    resume_education_df: pd.DataFrame,
                    resume_employment_df: pd.DataFrame):
        """
        按 raw_submission_id 把三张表串起来，逐条 submission 写入：
          1. 插入这条 submission 的 Resume_extraction 行，拿到
             resume_extraction_id
          2. 用这个 resume_extraction_id，插入这条 submission
             名下全部的 Resume_education / Resume_employment 记录

        某一条 submission 写入失败（比如 submission_normalized_id 缺失），
        记下"哪个 raw_submission_id、什么错误"，继续处理下一条，
        不会让一条记录的失败中断整个批次。
        """
        inserted_extraction_count = 0
        inserted_education_count = 0
        inserted_employment_count = 0
        errors = []

        for _, extraction_row in resume_extraction_df.iterrows():
            raw_submission_id = extraction_row["raw_submission_id"]
            try:
                resume_extraction_id = self.insert_resume_extraction_row(extraction_row.to_dict())
                inserted_extraction_count += 1
            except Exception as e:
                errors.append((raw_submission_id, "Resume_extraction", str(e)))
                print(f"  raw_submission_id={raw_submission_id} Resume_extraction insert failed: {e}")
                continue   # 这一条 Resume_extraction 都没插进去，教育/工作经历也没地方挂，跳过

            matching_education_rows = resume_education_df[
                resume_education_df["raw_submission_id"] == raw_submission_id
            ]
            for _, edu_row in matching_education_rows.iterrows():
                try:
                    self.insert_resume_education_row(resume_extraction_id, edu_row.to_dict())
                    inserted_education_count += 1
                except Exception as e:
                    errors.append((raw_submission_id, "Resume_education", str(e)))
                    print(f"  raw_submission_id={raw_submission_id} Resume_education insert failed: {e}")

            matching_employment_rows = resume_employment_df[
                resume_employment_df["raw_submission_id"] == raw_submission_id
            ]
            for _, emp_row in matching_employment_rows.iterrows():
                try:
                    self.insert_resume_employment_row(resume_extraction_id, emp_row.to_dict())
                    inserted_employment_count += 1
                except Exception as e:
                    errors.append((raw_submission_id, "Resume_employment", str(e)))
                    print(f"  raw_submission_id={raw_submission_id} Resume_employment insert failed: {e}")

        print("\n" + "=" * 60)
        print("Injection summary")
        print("=" * 60)
        print(f"  Resume_extraction inserted : {inserted_extraction_count}")
        print(f"  Resume_education inserted  : {inserted_education_count}")
        print(f"  Resume_employment inserted : {inserted_employment_count}")
        print(f"  errors                     : {len(errors)}")

        return {
            "inserted_extraction_count": inserted_extraction_count,
            "inserted_education_count": inserted_education_count,
            "inserted_employment_count": inserted_employment_count,
            "errors": errors,
        }


def build_resume_extraction_dataframes(
    successful_submission_dataframe: pd.DataFrame,
    parser: ResumeTextParser,
    matcher: ResumeSkillMatcher,
) -> dict:
    """
    输入：successful_submission_dataframe（只读，不会被修改），
          必须包含 raw_submission_id / resume_text / submission_normalized_id 三列。
    输出：5 个 DataFrame 组成的 dict（见文件顶部说明）。
    """
    # 每一行对应 Resume_extraction 的一条记录（不管有没有解析成功，
    # 都会有一行——解析失败/resume_text为空的情况，extraction_status
    # 会标成对应的状态，不会因为失败就跳过不记录）
    extraction_rows = []

    # 下面四个是"一对多"的子表，用列表收集，最后统一转成 DataFrame
    education_rows = []
    employment_rows = []
    external_identifier_rows = []
    skill_matched_rows = []

    for _, submission_row in successful_submission_dataframe.iterrows():
        raw_submission_id = submission_row["raw_submission_id"]
        resume_text = submission_row.get("resume_text")

        extracted_at = _now_iso()
        text_length = len(str(resume_text)) if pd.notna(resume_text) else 0
        resume_text_sha256 = (
            hashlib.sha256(str(resume_text).encode("utf-8")).hexdigest()
            if text_length > 0 else None
        )

        extraction_status = None
        extraction_error = None
        parsed = None

        try:
            parsed = parser.parse(resume_text if pd.notna(resume_text) else None)
            if text_length == 0:
                # resume_text 本身是空的——按之前确认过的原则，直接忽略，
                # 不算"失败"，是"没有可处理的内容"，单独一个状态标出来
                extraction_status = "skipped_empty"
            else:
                extraction_status = "completed"
        except ResumeParsingError as e:
            extraction_status = "failed"
            extraction_error = str(e)
            parsed = {
                "resume_educations": [], "resume_employments": [],
                "resume_external_identifiers": [], "resume_skills_raw": [],
                "resume_emails": [], "resume_phones": [],
            }

        # ---- Resume_extraction 这一行 ----
        extraction_rows.append({
            "raw_submission_id": raw_submission_id,
            "submission_normalized_id": submission_row.get("submission_normalized_id"),
            "cleaned_resume_text": resume_text if text_length > 0 else None,
            "resume_text_sha256": resume_text_sha256,
            "text_length": text_length,
            "extraction_status": extraction_status,
            "extraction_error": extraction_error,
            "extracted_at": extracted_at,
            "resume_emails": ";".join(parsed["resume_emails"]) or None,
            "resume_phones": ";".join(parsed["resume_phones"]) or None,
        })

        # ---- 教育经历（一对多，展开） ----
        for edu in parsed["resume_educations"]:
            education_rows.append({"raw_submission_id": raw_submission_id, **edu})

        # ---- 工作经历（一对多，展开） ----
        for emp in parsed["resume_employments"]:
            employment_rows.append({"raw_submission_id": raw_submission_id, **emp})

        # ---- 外部身份链接（一对多，展开） ----
        for ext_id in parsed["resume_external_identifiers"]:
            external_identifier_rows.append({"raw_submission_id": raw_submission_id, **ext_id})

        # ---- 技能匹配（一对多，展开）----
        # 只有 extraction_status == "completed" 才有必要做技能匹配，
        # 空文本/解析失败的情况 resume_skills_raw 本来就是空列表，
        # match() 传空列表进去会正常返回空列表，不需要额外判断
        matched_skills = matcher.match(parsed["resume_skills_raw"])
        for skill in matched_skills:
            skill_matched_rows.append({"raw_submission_id": raw_submission_id, **skill})

    resume_extraction_df = pd.DataFrame(extraction_rows)
    resume_education_df = pd.DataFrame(education_rows)
    resume_employment_df = pd.DataFrame(employment_rows)
    resume_external_identifier_df = pd.DataFrame(external_identifier_rows)
    resume_skill_matched_df = pd.DataFrame(skill_matched_rows)

    print(f"resume_extraction_df: {len(resume_extraction_df)} 行")
    print(f"  其中 completed: {(resume_extraction_df['extraction_status'] == 'completed').sum()}")
    print(f"  其中 skipped_empty: {(resume_extraction_df['extraction_status'] == 'skipped_empty').sum()}")
    print(f"  其中 failed: {(resume_extraction_df['extraction_status'] == 'failed').sum()}")
    print(f"resume_education_df: {len(resume_education_df)} 行")
    print(f"resume_employment_df: {len(resume_employment_df)} 行")
    print(f"resume_external_identifier_df: {len(resume_external_identifier_df)} 行")
    print(f"resume_skill_matched_df: {len(resume_skill_matched_df)} 行")

    return {
        "resume_extraction": resume_extraction_df,
        "resume_education": resume_education_df,
        "resume_employment": resume_employment_df,
        "resume_external_identifier": resume_external_identifier_df,
        "resume_skill_matched": resume_skill_matched_df,
    }


# ============================================================
# 把上面5个 DataFrame 合并成一份大 CSV，存进 Google Drive
# ============================================================

def _summarize_education_entries(entries: pd.DataFrame) -> str:
    """
    把某一条 submission 名下的多条教育经历，拼成一段人类可读的
    摘要文本，用 " ;; " 分隔多条经历，每条经历内部用 " | " 分隔
    学校名/学位/专业/起止年月这几个字段。
    """
    if entries.empty:
        return None
    parts = []
    for _, row in entries.iterrows():
        start = f"{row['start_year'] or ''}-{row['start_month'] or ''}".strip("-")
        end = f"{row['end_year'] or ''}-{row['end_month'] or ''}".strip("-")
        date_range = f"{start} to {end}".strip()
        parts.append(
            f"{row['normalized_school_name'] or ''} | "
            f"{row['normalized_degree'] or ''} | "
            f"{row['normalized_field_of_study'] or ''} | "
            f"{date_range}"
        )
    return " ;; ".join(parts)


def _summarize_employment_entries(entries: pd.DataFrame) -> str:
    if entries.empty:
        return None
    parts = []
    for _, row in entries.iterrows():
        start = f"{row['start_year'] or ''}-{row['start_month'] or ''}".strip("-")
        end = f"{row['end_year'] or ''}-{row['end_month'] or ''}".strip("-")
        date_range = f"{start} to {end}".strip()
        parts.append(
            f"{row['normalized_company_name'] or ''} | "
            f"{row['normalized_job_title'] or ''} | "
            f"{date_range}"
        )
    return " ;; ".join(parts)


def _summarize_external_identifier_entries(entries: pd.DataFrame) -> str:
    if entries.empty:
        return None
    parts = [f"{row['identity_type']}: {row['normalized_url']}" for _, row in entries.iterrows()]
    return "; ".join(parts)


def _summarize_matched_skill_entries(entries: pd.DataFrame) -> str:
    if entries.empty:
        return None
    return "; ".join(entries["skill_name"].tolist())


def build_consolidated_resume_csv(dataframes: dict) -> pd.DataFrame:
    """
    把 build_resume_extraction_dataframes() 产出的5个 DataFrame，
    按 raw_submission_id 合并成一份"宽表"——每个 submission 一行，
    教育/工作经历/外部链接/技能这几类一对多的内容，拼成一段用分隔符
    隔开的摘要文本存进对应的一列（和 resume_emails/resume_phones
    分号拼接的做法保持同一种风格）。

    这份合并结果只是给人浏览/存档用的（比如整体看一眼提取质量怎么样），
    不是要往数据库写的东西，数据库那边该插入的三张表已经在
    ResumeExtractionInjector 里单独处理过了，这里不重复。
    """
    resume_extraction_df = dataframes["resume_extraction"]
    resume_education_df = dataframes["resume_education"]
    resume_employment_df = dataframes["resume_employment"]
    resume_external_identifier_df = dataframes["resume_external_identifier"]
    resume_skill_matched_df = dataframes["resume_skill_matched"]

    consolidated_rows = []
    for _, extraction_row in resume_extraction_df.iterrows():
        raw_submission_id = extraction_row["raw_submission_id"]

        education_summary = _summarize_education_entries(
            resume_education_df[resume_education_df["raw_submission_id"] == raw_submission_id]
        )
        employment_summary = _summarize_employment_entries(
            resume_employment_df[resume_employment_df["raw_submission_id"] == raw_submission_id]
        )
        external_identifier_summary = _summarize_external_identifier_entries(
            resume_external_identifier_df[
                resume_external_identifier_df["raw_submission_id"] == raw_submission_id]
        )
        matched_skill_summary = _summarize_matched_skill_entries(
            resume_skill_matched_df[resume_skill_matched_df["raw_submission_id"] == raw_submission_id]
        )

        consolidated_rows.append({
            "raw_submission_id": raw_submission_id,
            "extraction_status": extraction_row["extraction_status"],
            "extraction_error": extraction_row["extraction_error"],
            "text_length": extraction_row["text_length"],
            "resume_emails": extraction_row["resume_emails"],
            "resume_phones": extraction_row["resume_phones"],
            "resume_educations_summary": education_summary,
            "resume_employments_summary": employment_summary,
            "resume_external_identifiers_summary": external_identifier_summary,
            "resume_skills_matched_summary": matched_skill_summary,
        })

    return pd.DataFrame(consolidated_rows)


def save_consolidated_csv(consolidated_df: pd.DataFrame, output_path: str = CONSOLIDATED_CSV_PATH):
    """
    把合并好的大表存成 CSV，写到 Google Drive 里。
    用 utf-8-sig 编码（带 BOM），这样如果之后有人用 Excel 直接打开
    这份 CSV，中文/特殊字符不会显示成乱码。
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    consolidated_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Consolidated CSV saved to: {output_path}")
    print(f"  rows: {len(consolidated_df)}")
    return output_path


In [ ]:
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/recruiting-pipeline"
SUCCESSFUL_SUBMISSION_FOR_EXTRACTION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/successful_submission_for_extraction.csv"
)

# 直接读 Normalization 那一步存好的CSV，不再回数据库查
successful_submission_dataframe = pd.read_csv(
    SUCCESSFUL_SUBMISSION_FOR_EXTRACTION_CSV_PATH,
    encoding="utf-8-sig",
)
print(f"Loaded from CSV: {len(successful_submission_dataframe)} rows")

# 运行
dataframes = build_resume_extraction_dataframes(successful_submission_dataframe, parser, matcher)

# 合并成一份大CSV，存到Drive
consolidated_df = build_consolidated_resume_csv(dataframes)
save_consolidated_csv(consolidated_df)

Loaded from CSV: 20 rows
resume_extraction_df: 20 行
  其中 completed: 19
  其中 skipped_empty: 0
  其中 failed: 1
resume_education_df: 19 行
resume_employment_df: 18 行
resume_external_identifier_df: 2 行
resume_skill_matched_df: 18 行
Consolidated CSV saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/resume_extraction_consolidated.csv
  rows: 20


'/content/drive/MyDrive/recruiting-pipeline/datasets/resume_extraction_consolidated.csv'

# Submission duplicate check

In [ ]:
"""
Google Colab 独立 cell：生成岗位申请查重结果 CSV，暂时不写入 D1。

重要：
1. 本 cell 不依赖之前任何 Colab cell 中的 class、function 或内存变量。
2. 本 cell 只从 D1 读取 Submission_normalized 和 raw_submission 的关联数据。
3. CV 提取结果只从 Google Drive 中的 resume_extraction_consolidated.csv 读取。
4. 本 cell 不对 D1 执行 INSERT、UPDATE、DELETE、ALTER 或 DROP。
5. 同一个规则版本、相同数据库数据和相同 CV CSV 重复运行时，会生成相同结果。
6. 不同规则版本使用不同输出目录，可以同时保留多个版本。
7. 同一个规则版本重复运行时覆盖同一组 CSV，不会不断生成重复文件。

当前 application cluster 分流字段：
    normalized_company_name
    normalized_position_name
    start_working_date 中提取的 YYYY-MM

当前重复申请判断优先级：
1. 同一 cluster 内，完整标准化邮箱完全相同。
2. 邮箱不同时，标准化电话号码最后 10 位完全相同。
3. 邮箱和电话号码都不同时，LinkedIn 个人主页 URL 完全相同。
4. 前三层都不同时，GitHub URL 完全相同，并且两条申请标准化之后的
   last name 完全相同。

重复组使用连通关系生成：
    A 与 B 邮箱相同，
    B 与 C 电话相同，
    则 A、B、C 属于同一个重复组。

每一个重复组只保留 submitted_at 最新的一条作为 Application candidate。
如果 submitted_at 相同或缺失，则使用较大的 submission_normalized_id
作为稳定的最终排序条件。
"""

# pandas 用于读取 CSV、整理 DataFrame 和保存查重结果。
import pandas as pd

# requests 用于通过 Cloudflare D1 REST API 执行只读 SQL 查询。
import requests

# hashlib 用于生成 value_hash、cluster_code、duplicate_group_key 和 analysis_run_key。
import hashlib

# json 用于生成稳定、可读的 evidence_metadata 和规则配置文本。
import json

# os 用于建立 Google Drive 输出目录和处理文件路径。
import os

# re 用于电话号码、空格和外部身份链接的标准化处理。
import re

# combinations 用于在同一个 application cluster 内生成两两申请组合。
from itertools import combinations

# Google Drive 用于读取 CV extraction CSV 并保存全部查重结果 CSV。
from google.colab import drive

# userdata 用于从 Colab Secrets 安全读取 Cloudflare API Token。
from google.colab import userdata


# ============================================================
# 第一部分：独立运行配置
# ============================================================

# 挂载 Google Drive。
#
# Google Drive 已经挂载时，Colab 会直接复用现有挂载。
drive.mount(
    "/content/drive"
)

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 API Token 时立即停止，避免后面得到不清楚的授权错误。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请先在 Colab Secrets 中保存 Cloudflare API Token。"
    )

# ============================================================
# 每次运行时由用户明确输入 ingestion_batch_id
# ============================================================

# 本查重 Cell 是完全独立的，因此不会读取 initial submission cleaning
# Cell 内存中的 EXISTING_INGESTION_BATCH_ID。每次运行都必须重新输入。
user_input = input(
    "请输入本次查重使用的 INGESTION_BATCH_ID："
).strip()

# 查重结果会覆盖固定输出文件，因此这里不允许空输入或自动猜测批次。
if not user_input:
    raise ValueError(
        "INGESTION_BATCH_ID 不能为空。"
        "请重新运行本 Cell，并输入一个正整数 batch id。"
    )

# 将用户输入转换为标准 Python int。
try:
    INGESTION_BATCH_ID = int(
        user_input
    )
except ValueError as exc:
    raise ValueError(
        "INGESTION_BATCH_ID 必须是整数。"
        f"当前输入为：{user_input!r}"
    ) from exc

# D1 自增主键应为正整数。
if INGESTION_BATCH_ID <= 0:
    raise ValueError(
        "INGESTION_BATCH_ID 必须大于 0。"
        f"当前输入为：{INGESTION_BATCH_ID}"
    )

# 显示本次选择，供用户在读取和覆盖 CSV 前再次核对。
print(
    "本次查重使用的 INGESTION_BATCH_ID：",
    INGESTION_BATCH_ID,
)

# 指定当前查重规则版本。
#
# 第一版 application_dedup_v1 使用数据库中原有的
# normalized_application_cycle_name 进行 application cluster 分流。
#
# 当前第二版改为使用 Submission_normalized.start_working_date
# 中的年份和月份进行 application cluster 分流，
# 因此必须使用新的 rule version，避免覆盖已经生成的 v1 结果。
RULE_VERSION = "application_dedup_v2"

# Google Drive 项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# CV extraction 汇总 CSV 路径。
CV_EXTRACTION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/"
    "resume_extraction_consolidated.csv"
)

# 所有查重结果的根输出目录。
DEDUP_OUTPUT_ROOT_FOLDER = (
    f"{DRIVE_FOLDER}/datasets/dedup_results"
)

# 当前规则版本的输出目录。
#
# 同一个版本重复运行时会覆盖这个目录下相同文件名的 CSV。
RULE_VERSION_OUTPUT_FOLDER = (
    f"{DEDUP_OUTPUT_ROOT_FOLDER}/"
    f"{RULE_VERSION}"
)

# 当前规则配置。
#
# 这份配置也会参与 analysis_run_key 的计算。
# 修改规则时，应同时修改 RULE_VERSION。
RULE_CONFIG = {
    "cluster_fields": [
        "normalized_company_name",
        "normalized_position_name",
        "start_working_date_year_month",
    ],
    "email_rule": "normalized_full_email_exact",
    "phone_rule": "last_10_digits_exact",
    "linkedin_rule": "normalized_person_profile_url_exact",
    "github_rule": (
        "normalized_url_exact_and_both_submissions_"
        "have_same_normalized_last_name"
    ),
    "canonical_rule": (
        "submitted_at_desc_then_"
        "submission_normalized_id_desc"
    ),
}


# ============================================================
# 第二部分：独立 D1 只读连接
# ============================================================

class ReadOnlyD1Client:
    """
    本 cell 专用的最小 D1 只读连接对象。

    这个 class 只提供 query()。
    query() 会拒绝不是 SELECT 或 WITH 开头的 SQL，
    防止本 cell 意外修改数据库。
    """

    def __init__(
        self,
        account_id,
        database_id,
        api_token,
    ):
        # 生成 D1 REST API 查询地址。
        self.api_url = (
            "https://api.cloudflare.com/client/v4/"
            f"accounts/{account_id}/"
            f"d1/database/{database_id}/query"
        )

        # 保存每次请求都需要使用的授权与内容类型 header。
        self.headers = {
            "Authorization": (
                f"Bearer {api_token}"
            ),
            "Content-Type": "application/json",
        }

    def query(
        self,
        sql,
        params=None,
        timeout=60,
    ):
        """
        执行一条只读 SQL，并直接返回 D1 results rows。
        """

        # 去掉 SQL 开头的空格，并取得第一个 SQL keyword。
        first_sql_keyword = (
            str(sql)
            .lstrip()
            .split(None, 1)[0]
            .upper()
        )

        # 本 cell 只允许 SELECT 和 WITH 查询。
        if first_sql_keyword not in {
            "SELECT",
            "WITH",
        }:
            raise ValueError(
                "ReadOnlyD1Client 只允许执行 "
                "SELECT 或 WITH 查询。"
            )

        # 创建 D1 API 请求 body。
        payload = {
            "sql": sql,
        }

        # SQL 有参数时，把参数加入请求 body。
        if params is not None:
            payload["params"] = params

        # 向 D1 发送查询请求。
        response = requests.post(
            self.api_url,
            headers=self.headers,
            json=payload,
            timeout=timeout,
        )

        # 尝试解析 D1 JSON response。
        try:
            response_data = response.json()
        except ValueError:
            raise RuntimeError(
                "D1 response 不是有效 JSON："
                f"HTTP {response.status_code}，"
                f"{response.text[:500]}"
            )

        # D1 查询失败时显示完整错误。
        if (
            response.status_code != 200
            or not response_data.get(
                "success",
                False,
            )
        ):
            raise RuntimeError(
                "D1 只读查询失败："
                f"{response_data.get('errors', [])}"
            )

        # 取得 D1 result 数组。
        result = response_data.get(
            "result",
            [],
        )

        # 没有 result 时返回空列表。
        if not result:
            return []

        # 返回第一条 SQL statement 的 results rows。
        return result[0].get(
            "results",
            [],
        )


# 创建本 cell 独立使用的 D1 只读连接。
d1_reader = ReadOnlyD1Client(
    account_id=ACCOUNT_ID,
    database_id=DATABASE_ID,
    api_token=API_TOKEN,
)


def _validate_selected_ingestion_batch():
    """
    验证用户输入的 batch 存在，并且每条 raw submission 都已经生成
    Submission_normalized。这里只执行 SELECT，不修改数据库。
    """

    # 同时读取 batch metadata、raw 数量和 normalized 数量。
    rows = d1_reader.query(
        """
        SELECT
          ib."id" AS "ingestion_batch_id",
          ib."source_file_name",
          ib."import_status",
          ib."total_row_count",
          ib."successful_row_count",
          COUNT(DISTINCT rs."id") AS "raw_submission_count",
          COUNT(DISTINCT sn."id") AS "submission_normalized_count"
        FROM "Ingestion_batch" AS ib
        LEFT JOIN "raw_submission" AS rs
          ON rs."ingestion_batch_id" = ib."id"
        LEFT JOIN "Submission_normalized" AS sn
          ON sn."raw_submission_id" = rs."id"
        WHERE ib."id" = ?
        GROUP BY
          ib."id",
          ib."source_file_name",
          ib."import_status",
          ib."total_row_count",
          ib."successful_row_count";
        """,
        [
            int(
                INGESTION_BATCH_ID
            ),
        ],
    )

    # 没有返回行表示 batch id 不存在。
    if not rows:
        raise ValueError(
            "数据库中不存在 ingestion_batch_id="
            f"{INGESTION_BATCH_ID}。"
        )

    batch_validation = rows[0]
    raw_submission_count = int(
        batch_validation.get(
            "raw_submission_count"
        )
        or 0
    )
    submission_normalized_count = int(
        batch_validation.get(
            "submission_normalized_count"
        )
        or 0
    )

    # 空 batch 无法执行查重。
    if raw_submission_count == 0:
        raise ValueError(
            "目标 ingestion batch 中没有 raw_submission。"
        )

    # 当前流程要求完整 normalization 后才允许查重。
    if submission_normalized_count != raw_submission_count:
        raise ValueError(
            "目标 batch 尚未完成全部 Submission_normalized："
            f"raw_submission_count={raw_submission_count}, "
            "submission_normalized_count="
            f"{submission_normalized_count}。"
        )

    print(
        "\nIngestion batch validation:"
    )
    print(
        batch_validation
    )
    print(
        "目标 batch 验证通过，可以开始查重。\n"
    )

    return batch_validation


# 在读取 CV CSV 和覆盖任何查重结果文件以前完成 batch 验证。
SELECTED_BATCH_VALIDATION = (
    _validate_selected_ingestion_batch()
)


# ============================================================
# 第三部分：通用确定性标准化函数
# ============================================================

def _is_missing(value):
    """
    判断一个 pandas/Python value 是否为空。
    """

    # Python None 直接视为空值。
    if value is None:
        return True

    # pandas NaN / pd.NA 视为空值。
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    # 空字符串或只有空格的字符串视为空值。
    return str(value).strip() == ""


def _normalize_lower_text(value):
    """
    将普通 string 标准化成：
    1. 去掉首尾空格；
    2. 连续空白折叠成一个普通空格；
    3. 全部小写。
    """

    # 空值返回 None。
    if _is_missing(value):
        return None

    # 折叠全部连续空白并转换为小写。
    normalized_value = re.sub(
        r"\s+",
        " ",
        str(value).strip(),
    ).lower()

    # 标准化后为空时返回 None。
    return (
        normalized_value
        if normalized_value
        else None
    )


def _extract_year_month_from_start_working_date(
    start_working_date,
):
    """
    从 Submission_normalized.start_working_date 提取 YYYY-MM。

    这个 function 只在查重 cell 的内存 DataFrame 中生成 cluster 周期。
    不会更新 Submission_normalized，
    也不会创建或修改数据库 Position_application_cycle records。
    """

    # start_working_date 为空时不能生成查重周期。
    if _is_missing(
        start_working_date
    ):
        return None

    # 将数据库中的日期文本交给 pandas 解析。
    #
    # Submission_normalized normalization cell 已经把有效日期保存成 YYYY-MM-DD，
    # 这里仍使用 errors="coerce" 防止异常数据导致整个批次出现模糊错误。
    parsed_start_working_date = (
        pd.to_datetime(
            start_working_date,
            errors="coerce",
        )
    )

    # 无法解析时返回 None，
    # 后续资格检查会把这条申请放入 excluded_submission.csv。
    if pd.isna(
        parsed_start_working_date
    ):
        return None

    # 只保留年份和月份，作为查重阶段的 application cycle name。
    return parsed_start_working_date.strftime(
        "%Y-%m"
    )


def _sha256_text(value):
    """
    对确定性 UTF-8 文本生成 SHA-256。
    """

    # 空值不生成 hash。
    if _is_missing(value):
        return None

    # 对 UTF-8 bytes 计算 SHA-256。
    return hashlib.sha256(
        str(value).encode(
            "utf-8"
        )
    ).hexdigest()


def _stable_json(value):
    """
    将 Python object 转成 key 顺序稳定的 JSON。
    """

    # sort_keys=True 保证相同内容每次输出顺序一致。
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def _normalize_phone_last_10(raw_phone):
    """
    按当前项目规则把电话号码标准化成最后 10 位数字。

    当前项目暂时假设全部电话号码都是美国电话号码。
    少于 10 位数字时返回 None。
    """

    # 空电话号码返回 None。
    if _is_missing(raw_phone):
        return None

    # 删除全部非数字字符。
    phone_digits = re.sub(
        r"\D",
        "",
        str(raw_phone),
    )

    # 少于 10 位时不能作为可用电话号码。
    if len(phone_digits) < 10:
        return None

    # 返回最后 10 位数字。
    return phone_digits[-10:]


def _split_and_normalize_phones(
    resume_phones,
):
    """
    将 CSV 中分号连接的 resume_phones 拆成标准化号码集合。
    """

    # 空字段返回空集合。
    if _is_missing(resume_phones):
        return set()

    # 保存当前 submission 的全部可用电话号码。
    normalized_phones = set()

    # CV extraction 使用分号连接多个电话号码。
    for raw_phone in str(
        resume_phones
    ).split(";"):
        # 标准化当前电话号码。
        normalized_phone = (
            _normalize_phone_last_10(
                raw_phone
            )
        )

        # 标准化成功时加入集合，自动去重。
        if normalized_phone is not None:
            normalized_phones.add(
                normalized_phone
            )

    # 返回可用号码集合。
    return normalized_phones


def _parse_external_identifiers(
    external_identifier_summary,
):
    """
    从汇总 CSV 文本中提取标准化 LinkedIn 和 GitHub URL。

    输入示例：
        linkedin: https://linkedin.com/in/example;
        github: https://github.com/example
    """

    # 创建默认空集合。
    external_identifiers = {
        "linkedin": set(),
        "github": set(),
    }

    # 空字段直接返回空集合。
    if _is_missing(
        external_identifier_summary
    ):
        return external_identifiers

    # 使用 regex 全局提取 identity type 和 URL。
    #
    # 不能简单使用 split(":")，
    # 因为 https:// 本身也包含冒号。
    matches = re.finditer(
        (
            r"(?P<identity_type>"
            r"linkedin|github"
            r")\s*:\s*"
            r"(?P<normalized_url>"
            r"https?://[^\s;]+"
            r")"
        ),
        str(
            external_identifier_summary
        ),
        flags=re.IGNORECASE,
    )

    # 处理每一个外部身份链接。
    for match in matches:
        # identity type 转成小写。
        identity_type = (
            match.group(
                "identity_type"
            )
            .strip()
            .lower()
        )

        # URL 去掉首尾空格、常见结尾标点并小写化。
        normalized_url = (
            match.group(
                "normalized_url"
            )
            .strip()
            .rstrip(
                ".,;)"
            )
            .lower()
        )

        # LinkedIn 只接受个人主页 /in/。
        if (
            identity_type == "linkedin"
            and not re.match(
                (
                    r"^https?://"
                    r"(?:www\.)?"
                    r"linkedin\.com/in/"
                    r"[^/?#]+/?$"
                ),
                normalized_url,
                flags=re.IGNORECASE,
            )
        ):
            continue

        # GitHub 只接受账号根路径。
        if (
            identity_type == "github"
            and not re.match(
                (
                    r"^https?://"
                    r"(?:www\.)?"
                    r"github\.com/"
                    r"[^/?#]+/?$"
                ),
                normalized_url,
                flags=re.IGNORECASE,
            )
        ):
            continue

        # 删除 URL 最后的斜线，保证表示方式唯一。
        normalized_url = (
            normalized_url.rstrip(
                "/"
            )
        )

        # 保存到对应集合并自动去重。
        external_identifiers[
            identity_type
        ].add(
            normalized_url
        )

    # 返回 LinkedIn 和 GitHub 集合。
    return external_identifiers


# ============================================================
# 第四部分：读取 Submission_normalized 和 CV CSV
# ============================================================

def _load_submission_normalized_dataframe():
    """
    从 D1 读取指定 ingestion batch 的标准化 submission。

    只执行 SELECT，不修改数据库。
    """

    # 查询当前查重所需的 Submission_normalized 字段。
    sql = """
        SELECT
          sn."id" AS "submission_normalized_id",
          sn."raw_submission_id",
          rs."ingestion_batch_id",
          sn."normalized_company_name",
          sn."normalized_position_name",
          sn."normalized_application_cycle_name",
          sn."normalized_person_name",
          sn."normalized_first_name",
          sn."normalized_middle_name",
          sn."normalized_last_name",
          sn."normalized_email_prefix",
          sn."normalized_email_domain",
          sn."start_working_date",
          sn."submitted_at",
          sn."application_cycle_id"
        FROM "Submission_normalized" AS sn
        INNER JOIN "raw_submission" AS rs
          ON rs."id" = sn."raw_submission_id"
        WHERE rs."ingestion_batch_id" = ?
        ORDER BY sn."id";
    """

    # 执行只读查询。
    rows = d1_reader.query(
        sql,
        [
            int(
                INGESTION_BATCH_ID
            ),
        ],
    )

    # 没有查到记录时停止。
    if not rows:
        raise ValueError(
            "指定 ingestion batch 没有可用的 "
            "Submission_normalized 记录。"
        )

    # 转换成 DataFrame。
    submission_dataframe = pd.DataFrame(
        rows
    )

    # 数据库主键统一转换成标准 Python/pandas integer。
    submission_dataframe[
        "submission_normalized_id"
    ] = pd.to_numeric(
        submission_dataframe[
            "submission_normalized_id"
        ],
        errors="raise",
    ).astype(
        "int64"
    )

    # raw_submission_id 统一转换成 integer。
    submission_dataframe[
        "raw_submission_id"
    ] = pd.to_numeric(
        submission_dataframe[
            "raw_submission_id"
        ],
        errors="raise",
    ).astype(
        "int64"
    )

    # 检查 submission_normalized_id 唯一性。
    if submission_dataframe[
        "submission_normalized_id"
    ].duplicated().any():
        raise ValueError(
            "D1 查询结果中的 "
            "submission_normalized_id 不唯一。"
        )

    # 检查 raw_submission_id 一对一关系。
    if submission_dataframe[
        "raw_submission_id"
    ].duplicated().any():
        raise ValueError(
            "同一个 raw_submission_id 对应多个 "
            "Submission_normalized 记录。"
        )

    # 返回数据库 DataFrame。
    return submission_dataframe


def _load_cv_extraction_dataframe():
    """
    从 Google Drive 读取 CV extraction 汇总 CSV。
    """

    # 文件不存在时停止。
    if not os.path.exists(
        CV_EXTRACTION_CSV_PATH
    ):
        raise FileNotFoundError(
            "找不到 CV extraction CSV："
            f"{CV_EXTRACTION_CSV_PATH}"
        )

    # 使用 utf-8-sig 读取 CSV。
    cv_dataframe = pd.read_csv(
        CV_EXTRACTION_CSV_PATH,
        encoding="utf-8-sig",
    )

    # 当前查重需要的 CV CSV 字段。
    required_columns = {
        "raw_submission_id",
        "extraction_status",
        "resume_phones",
        "resume_external_identifiers_summary",
    }

    # 找出缺失字段。
    missing_columns = (
        required_columns
        - set(
            cv_dataframe.columns
        )
    )

    # 缺少必要字段时停止。
    if missing_columns:
        raise ValueError(
            "CV extraction CSV 缺少必要字段："
            f"{sorted(missing_columns)}"
        )

    # raw_submission_id 转成 integer。
    cv_dataframe[
        "raw_submission_id"
    ] = pd.to_numeric(
        cv_dataframe[
            "raw_submission_id"
        ],
        errors="raise",
    ).astype(
        "int64"
    )

    # 检查 raw_submission_id 唯一性。
    if cv_dataframe[
        "raw_submission_id"
    ].duplicated().any():
        duplicated_ids = (
            cv_dataframe.loc[
                cv_dataframe[
                    "raw_submission_id"
                ].duplicated(
                    keep=False
                ),
                "raw_submission_id",
            ]
            .astype(
                "int64"
            )
            .tolist()
        )

        raise ValueError(
            "CV extraction CSV 中存在重复 "
            "raw_submission_id："
            f"{duplicated_ids[:20]}"
        )

    # 只保留当前查重需要的 CV 字段。
    return cv_dataframe[
        [
            "raw_submission_id",
            "extraction_status",
            "resume_phones",
            "resume_external_identifiers_summary",
        ]
    ].copy()


def _build_dedup_source_dataframe():
    """
    合并数据库标准化结果和 CV extraction CSV，
    并完成查重前的资格筛选和身份特征标准化。
    """

    # 读取数据库标准化申请。
    submission_dataframe = (
        _load_submission_normalized_dataframe()
    )

    # 读取 CV extraction CSV。
    cv_dataframe = (
        _load_cv_extraction_dataframe()
    )

    # CV extraction 汇总文件应由当前 batch 的 successful submission
    # 文件覆盖生成，因此 raw_submission_id 集合必须与数据库当前 batch
    # 完全一致。extraction_status=failed/skipped_empty 的行仍然保留在 CSV，
    # 所以不应因为提取失败而缺少 raw_submission_id。
    submission_raw_ids = set(
        submission_dataframe[
            "raw_submission_id"
        ].astype(
            "int64"
        ).tolist()
    )
    cv_raw_ids = set(
        cv_dataframe[
            "raw_submission_id"
        ].astype(
            "int64"
        ).tolist()
    )

    missing_cv_raw_ids = sorted(
        submission_raw_ids
        - cv_raw_ids
    )
    unexpected_cv_raw_ids = sorted(
        cv_raw_ids
        - submission_raw_ids
    )

    # 任一方向存在差异都说明 CV CSV 与选中的 ingestion batch 不一致。
    if missing_cv_raw_ids or unexpected_cv_raw_ids:
        raise ValueError(
            "CV extraction CSV 与当前 ingestion batch 不一致。"
            f" INGESTION_BATCH_ID={INGESTION_BATCH_ID};"
            " database_raw_submission_count="
            f"{len(submission_raw_ids)};"
            f" cv_raw_submission_count={len(cv_raw_ids)};"
            " missing_in_cv_sample="
            f"{missing_cv_raw_ids[:20]};"
            " unexpected_in_cv_sample="
            f"{unexpected_cv_raw_ids[:20]}。"
        )

    print(
        "CV extraction batch mapping validated:",
        {
            "ingestion_batch_id": int(
                INGESTION_BATCH_ID
            ),
            "database_raw_submission_count": len(
                submission_raw_ids
            ),
            "cv_raw_submission_count": len(
                cv_raw_ids
            ),
        },
    )

    # 以 raw_submission_id 左连接 CV extraction。
    #
    # 左连接保证 CV 提取状态为 failed/skipped_empty 的 submission
    # 仍然可以使用表单邮箱参加第一层查重。
    merged_dataframe = (
        submission_dataframe.merge(
            cv_dataframe,
            on="raw_submission_id",
            how="left",
            validate="one_to_one",
        )
    )

    # 标准化 cluster 使用的公司名称。
    merged_dataframe[
        "dedup_company_name"
    ] = merged_dataframe[
        "normalized_company_name"
    ].apply(
        _normalize_lower_text
    )

    # 标准化 cluster 使用的岗位名称。
    merged_dataframe[
        "dedup_position_name"
    ] = merged_dataframe[
        "normalized_position_name"
    ].apply(
        _normalize_lower_text
    )

    # 从 Submission_normalized.start_working_date 提取 YYYY-MM，
    # 作为查重阶段 application cluster 使用的申请周期名称。
    #
    # 这里不再使用数据库中的 normalized_application_cycle_name，
    # 也不使用数据库 Position_application_cycle record。
    #
    # 这个值只存在于本次查重 DataFrame 和输出 CSV 中，
    # 不会回填或修改任何数据库字段。
    merged_dataframe[
        "dedup_application_cycle_name"
    ] = merged_dataframe[
        "start_working_date"
    ].apply(
        _extract_year_month_from_start_working_date
    )

    # 标准化邮箱前缀。
    merged_dataframe[
        "dedup_email_prefix"
    ] = merged_dataframe[
        "normalized_email_prefix"
    ].apply(
        _normalize_lower_text
    )

    # 标准化邮箱域名。
    merged_dataframe[
        "dedup_email_domain"
    ] = merged_dataframe[
        "normalized_email_domain"
    ].apply(
        _normalize_lower_text
    )

    # 判断邮箱 prefix 和 domain 是否都有效。
    merged_dataframe[
        "email_is_usable"
    ] = (
        merged_dataframe[
            "dedup_email_prefix"
        ].notna()
        & merged_dataframe[
            "dedup_email_domain"
        ].notna()
    )

    # 判断 cluster 三个字段是否都存在。
    merged_dataframe[
        "cluster_fields_are_usable"
    ] = (
        merged_dataframe[
            "dedup_company_name"
        ].notna()
        & merged_dataframe[
            "dedup_position_name"
        ].notna()
        & merged_dataframe[
            "dedup_application_cycle_name"
        ].notna()
    )

    # 保存排除原因。
    def build_exclusion_reason(
        row,
    ):
        # 邮箱不完整时优先记录邮箱排除原因。
        if not bool(
            row[
                "email_is_usable"
            ]
        ):
            return (
                "normalized_email_prefix_or_"
                "domain_missing"
            )

        # start_working_date 无法生成 YYYY-MM 时，
        # 保存专门的周期排除原因。
        if _is_missing(
            row[
                "dedup_application_cycle_name"
            ]
        ):
            return (
                "start_working_date_year_month_"
                "missing_or_invalid"
            )

        # 其余 cluster 字段不完整时记录 cluster 排除原因。
        if not bool(
            row[
                "cluster_fields_are_usable"
            ]
        ):
            return (
                "normalized_cluster_field_missing"
            )

        # 可进入查重时没有排除原因。
        return None

    # 逐行生成排除原因。
    merged_dataframe[
        "dedup_exclusion_reason"
    ] = merged_dataframe.apply(
        build_exclusion_reason,
        axis=1,
    )

    # 保存全部被排除记录。
    excluded_dataframe = (
        merged_dataframe.loc[
            merged_dataframe[
                "dedup_exclusion_reason"
            ].notna()
        ].copy()
    )

    # 只保留可进入 application cluster 和查重的记录。
    dedup_source_dataframe = (
        merged_dataframe.loc[
            merged_dataframe[
                "dedup_exclusion_reason"
            ].isna()
        ].copy()
    )

    # 生成完整标准化邮箱。
    dedup_source_dataframe[
        "normalized_email_address"
    ] = (
        dedup_source_dataframe[
            "dedup_email_prefix"
        ]
        + "@"
        + dedup_source_dataframe[
            "dedup_email_domain"
        ]
    )

    # 拆分并标准化电话号码。
    dedup_source_dataframe[
        "normalized_phone_values"
    ] = dedup_source_dataframe[
        "resume_phones"
    ].apply(
        _split_and_normalize_phones
    )

    # 解析 LinkedIn/GitHub。
    external_identifier_values = (
        dedup_source_dataframe[
            "resume_external_identifiers_summary"
        ].apply(
            _parse_external_identifiers
        )
    )

    # 保存 LinkedIn URL 集合。
    dedup_source_dataframe[
        "normalized_linkedin_values"
    ] = external_identifier_values.apply(
        lambda value: value[
            "linkedin"
        ]
    )

    # 保存 GitHub URL 集合。
    dedup_source_dataframe[
        "normalized_github_values"
    ] = external_identifier_values.apply(
        lambda value: value[
            "github"
        ]
    )

    # 对 last name 再执行一次确定性 string 标准化。
    #
    # 虽然这个字段来自 Submission_normalized，
    # 这里仍然统一执行去除首尾空格、折叠连续空白和小写化，
    # 保证 GitHub 辅助判断不会受到大小写或多余空格影响。
    dedup_source_dataframe[
        "dedup_last_name"
    ] = dedup_source_dataframe[
        "normalized_last_name"
    ].apply(
        _normalize_lower_text
    )

    # 生成稳定的 cluster identity text。
    dedup_source_dataframe[
        "cluster_identity_text"
    ] = dedup_source_dataframe.apply(
        lambda row: "||".join(
            [
                row[
                    "dedup_company_name"
                ],
                row[
                    "dedup_position_name"
                ],
                row[
                    "dedup_application_cycle_name"
                ],
            ]
        ),
        axis=1,
    )

    # 使用 cluster identity text 生成稳定 cluster_code。
    dedup_source_dataframe[
        "cluster_code"
    ] = dedup_source_dataframe[
        "cluster_identity_text"
    ].apply(
        lambda value: (
            "cluster_"
            + _sha256_text(
                value
            )[:24]
        )
    )

    # 按 submission_normalized_id 固定排序。
    dedup_source_dataframe = (
        dedup_source_dataframe.sort_values(
            [
                "submission_normalized_id",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 排除记录也固定排序。
    excluded_dataframe = (
        excluded_dataframe.sort_values(
            [
                "submission_normalized_id",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 返回查重源数据和排除数据。
    return (
        dedup_source_dataframe,
        excluded_dataframe,
    )


# ============================================================
# 第五部分：构建 Submission_identity_feature 草案
# ============================================================

def _build_identity_feature_dataframe(
    dedup_source_dataframe,
):
    """
    按 Submission_identity_feature 字段结构生成身份特征 CSV。

    本 function 只生成 DataFrame，不写数据库。
    """

    # 保存全部 identity feature rows。
    feature_rows = []

    # 按固定 submission 顺序处理。
    for _, row in (
        dedup_source_dataframe.iterrows()
    ):
        # 取得标准 Python int id。
        submission_normalized_id = int(
            row[
                "submission_normalized_id"
            ]
        )

        # 邮箱 feature。
        email_value = row[
            "normalized_email_address"
        ]

        # 保存完整标准化邮箱。
        feature_rows.append(
            {
                "id": None,
                "submission_normalized_id": (
                    submission_normalized_id
                ),
                "feature_type": "email",
                "normalized_value": email_value,
                "value_hash": _sha256_text(
                    email_value
                ),
                "feature_strength": "strong",
                "is_usable": 1,
                "created_at": None,
            }
        )

        # 保存全部标准化电话号码。
        for phone_value in sorted(
            row[
                "normalized_phone_values"
            ]
        ):
            feature_rows.append(
                {
                    "id": None,
                    "submission_normalized_id": (
                        submission_normalized_id
                    ),
                    "feature_type": "phone",
                    "normalized_value": phone_value,
                    "value_hash": _sha256_text(
                        phone_value
                    ),
                    "feature_strength": "strong",
                    "is_usable": 1,
                    "created_at": None,
                }
            )

        # 保存全部 LinkedIn URL。
        for linkedin_value in sorted(
            row[
                "normalized_linkedin_values"
            ]
        ):
            feature_rows.append(
                {
                    "id": None,
                    "submission_normalized_id": (
                        submission_normalized_id
                    ),
                    "feature_type": "linkedin",
                    "normalized_value": (
                        linkedin_value
                    ),
                    "value_hash": _sha256_text(
                        linkedin_value
                    ),
                    "feature_strength": "strong",
                    "is_usable": 1,
                    "created_at": None,
                }
            )

        # 保存全部 GitHub URL。
        for github_value in sorted(
            row[
                "normalized_github_values"
            ]
        ):
            feature_rows.append(
                {
                    "id": None,
                    "submission_normalized_id": (
                        submission_normalized_id
                    ),
                    "feature_type": "github",
                    "normalized_value": github_value,
                    "value_hash": _sha256_text(
                        github_value
                    ),
                    "feature_strength": "medium",
                    "is_usable": 1,
                    "created_at": None,
                }
            )

    # 创建 DataFrame。
    feature_dataframe = pd.DataFrame(
        feature_rows,
        columns=[
            "id",
            "submission_normalized_id",
            "feature_type",
            "normalized_value",
            "value_hash",
            "feature_strength",
            "is_usable",
            "created_at",
        ],
    )

    # 使用未来唯一索引对应字段去重。
    feature_dataframe = (
        feature_dataframe.drop_duplicates(
            subset=[
                "submission_normalized_id",
                "feature_type",
                "value_hash",
            ]
        )
    )

    # 固定输出排序。
    feature_dataframe = (
        feature_dataframe.sort_values(
            [
                "submission_normalized_id",
                "feature_type",
                "normalized_value",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 返回 identity feature 草案。
    return feature_dataframe


# ============================================================
# 第六部分：建立 application cluster 草案
# ============================================================

def _build_application_cluster_dataframe(
    dedup_source_dataframe,
):
    """
    按 Application_cluster 字段结构生成 cluster CSV 草案。

    当前数据库的 application_cycle_id 是 NOT NULL UNIQUE，
    但本次真实分流依据是标准化公司名、岗位名和周期名。
    因为暂时不写数据库，所以这里不伪造 application_cycle_id。
    """

    # 每个 cluster_code 只保留一行。
    cluster_base_dataframe = (
        dedup_source_dataframe[
            [
                "cluster_code",
                "cluster_identity_text",
                "dedup_company_name",
                "dedup_position_name",
                "dedup_application_cycle_name",
            ]
        ]
        .drop_duplicates(
            subset=[
                "cluster_code",
            ]
        )
        .copy()
    )

    # 计算每个 cluster 的 submission 数量。
    cluster_record_counts = (
        dedup_source_dataframe.groupby(
            "cluster_code",
            sort=True,
        )[
            "submission_normalized_id"
        ]
        .count()
        .rename(
            "submission_count"
        )
    )

    # 加入 submission 数量。
    cluster_base_dataframe = (
        cluster_base_dataframe.merge(
            cluster_record_counts,
            on="cluster_code",
            how="left",
            validate="one_to_one",
        )
    )

    # 保存每个 raw cluster 涉及的 mapped application_cycle_id。
    mapped_cycle_ids = (
        dedup_source_dataframe.groupby(
            "cluster_code",
            sort=True,
        )[
            "application_cycle_id"
        ]
        .apply(
            lambda values: _stable_json(
                sorted(
                    {
                        int(value)
                        for value in values
                        if not _is_missing(
                            value
                        )
                    }
                )
            )
        )
        .rename(
            "source_application_cycle_ids"
        )
    )

    # 合并 mapped cycle id 列表。
    cluster_base_dataframe = (
        cluster_base_dataframe.merge(
            mapped_cycle_ids,
            on="cluster_code",
            how="left",
            validate="one_to_one",
        )
    )

    # 保存每个查重 cluster 中原本已经存入 Submission_normalized 的
    # normalized_application_cycle_name 列表。
    #
    # 当前数据库值可能仍然是 unknown。
    # 这个字段只用于对照和审计，不参与 v2 cluster 分流。
    source_normalized_cycle_names = (
        dedup_source_dataframe.groupby(
            "cluster_code",
            sort=True,
        )[
            "normalized_application_cycle_name"
        ]
        .apply(
            lambda values: _stable_json(
                sorted(
                    {
                        str(
                            value
                        )
                        for value in values
                        if not _is_missing(
                            value
                        )
                    }
                )
            )
        )
        .rename(
            "source_normalized_application_cycle_names"
        )
    )

    # 把原数据库标准化周期名称列表加入 cluster 草案。
    cluster_base_dataframe = (
        cluster_base_dataframe.merge(
            source_normalized_cycle_names,
            on="cluster_code",
            how="left",
            validate="one_to_one",
        )
    )

    # 生成和数据库字段接近的 Application_cluster 草案。
    cluster_dataframe = pd.DataFrame(
        {
            "id": None,
            "application_cycle_id": None,
            "cluster_code": (
                cluster_base_dataframe[
                    "cluster_code"
                ]
            ),
            "display_name": (
                cluster_base_dataframe.apply(
                    lambda row: (
                        f"{row['dedup_company_name']}"
                        " | "
                        f"{row['dedup_position_name']}"
                        " | "
                        f"{row['dedup_application_cycle_name']}"
                    ),
                    axis=1,
                )
            ),
            "status": "analysis_only",
            "created_at": None,
            "normalized_company_name": (
                cluster_base_dataframe[
                    "dedup_company_name"
                ]
            ),
            "normalized_position_name": (
                cluster_base_dataframe[
                    "dedup_position_name"
                ]
            ),
            "normalized_application_cycle_name": (
                cluster_base_dataframe[
                    "dedup_application_cycle_name"
                ]
            ),
            "submission_count": (
                cluster_base_dataframe[
                    "submission_count"
                ]
            ),
            "source_application_cycle_ids": (
                cluster_base_dataframe[
                    "source_application_cycle_ids"
                ]
            ),
            "source_normalized_application_cycle_names": (
                cluster_base_dataframe[
                    "source_normalized_application_cycle_names"
                ]
            ),
        }
    )

    # 固定 cluster 排序。
    cluster_dataframe = (
        cluster_dataframe.sort_values(
            [
                "cluster_code",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 返回 cluster 草案。
    return cluster_dataframe


# ============================================================
# 第七部分：Union-Find 连通重复组
# ============================================================

class UnionFind:
    """
    用于将通过不同 evidence 连起来的 submission 合并为一个重复组。
    """

    def __init__(
        self,
        values,
    ):
        # 每个 submission 初始都以自己为 parent。
        self.parent = {
            int(value): int(value)
            for value in values
        }

    def find(
        self,
        value,
    ):
        # 转成标准 Python int。
        value = int(
            value
        )

        # 路径压缩。
        if self.parent[
            value
        ] != value:
            self.parent[
                value
            ] = self.find(
                self.parent[
                    value
                ]
            )

        # 返回根节点。
        return self.parent[
            value
        ]

    def union(
        self,
        first_value,
        second_value,
    ):
        # 找到第一个 submission 的根。
        first_root = self.find(
            first_value
        )

        # 找到第二个 submission 的根。
        second_root = self.find(
            second_value
        )

        # 已经属于同一组时不需要处理。
        if first_root == second_root:
            return

        # 始终让较小的根 id 成为 parent。
        #
        # 这不是业务排序，只是保证 union 结果确定。
        if first_root < second_root:
            self.parent[
                second_root
            ] = first_root
        else:
            self.parent[
                first_root
            ] = second_root


# ============================================================
# 第八部分：生成 pair evidence 和重复组
# ============================================================

def _build_pair_evidence_and_components(
    dedup_source_dataframe,
):
    """
    在每一个 cluster 内执行两两身份匹配并生成连通重复组。
    """

    # 创建 Union-Find。
    union_find = UnionFind(
        dedup_source_dataframe[
            "submission_normalized_id"
        ].tolist()
    )

    # 保存全部匹配 evidence。
    evidence_rows = []

    # 按 cluster_code 固定顺序分组。
    for cluster_code, cluster_rows in (
        dedup_source_dataframe.groupby(
            "cluster_code",
            sort=True,
        )
    ):
        # 将 cluster 内记录按 id 固定排序。
        cluster_records = (
            cluster_rows.sort_values(
                [
                    "submission_normalized_id",
                ],
                kind="mergesort",
            )
            .to_dict(
                orient="records"
            )
        )

        # 生成 cluster 内全部两两组合。
        for (
            first_record,
            second_record,
        ) in combinations(
            cluster_records,
            2,
        ):
            # 取得两个标准 Python int id。
            first_submission_id = int(
                first_record[
                    "submission_normalized_id"
                ]
            )
            second_submission_id = int(
                second_record[
                    "submission_normalized_id"
                ]
            )

            # 计算完整邮箱是否相同。
            email_match = (
                first_record[
                    "normalized_email_address"
                ]
                == second_record[
                    "normalized_email_address"
                ]
            )

            # 计算共同电话号码。
            shared_phone_values = sorted(
                first_record[
                    "normalized_phone_values"
                ]
                & second_record[
                    "normalized_phone_values"
                ]
            )

            # 计算共同 LinkedIn URL。
            shared_linkedin_values = sorted(
                first_record[
                    "normalized_linkedin_values"
                ]
                & second_record[
                    "normalized_linkedin_values"
                ]
            )

            # 计算共同 GitHub URL。
            shared_github_values = sorted(
                first_record[
                    "normalized_github_values"
                ]
                & second_record[
                    "normalized_github_values"
                ]
            )

            # 取得第一条申请标准化后的 last name。
            first_last_name = first_record[
                "dedup_last_name"
            ]

            # 取得第二条申请标准化后的 last name。
            second_last_name = second_record[
                "dedup_last_name"
            ]

            # 判断两条申请的标准化 last name 是否完全相同。
            #
            # 两个 last name 都必须不是 None，
            # 避免把两个缺失 last name 的申请错误判断为姓名相同。
            last_name_match = bool(
                first_last_name is not None
                and second_last_name is not None
                and first_last_name
                == second_last_name
            )

            # GitHub 规则必须同时满足：
            # 1. 两条申请至少有一个完全相同的标准化 GitHub URL；
            # 2. 两条申请标准化之后的 last name 完全相同。
            github_last_name_match = bool(
                shared_github_values
                and last_name_match
            )

            # 按规则优先级决定 pair 是否属于重复申请。
            if email_match:
                primary_match_rule = (
                    "email_exact_match"
                )
            elif shared_phone_values:
                primary_match_rule = (
                    "phone_last_10_exact_match"
                )
            elif shared_linkedin_values:
                primary_match_rule = (
                    "linkedin_exact_match"
                )
            elif github_last_name_match:
                primary_match_rule = (
                    "github_exact_match_with_"
                    "same_normalized_last_name"
                )
            else:
                primary_match_rule = None

            # 没有满足任何规则时不保存 match evidence。
            if primary_match_rule is None:
                continue

            # 满足任意规则时连接两个 submission。
            union_find.union(
                first_submission_id,
                second_submission_id,
            )

            # 定义保存 evidence 的内部 helper。
            def append_evidence(
                evidence_type,
                evidence_strength,
                matched_value,
                is_primary_rule,
            ):
                # 生成稳定 pair key。
                pair_key = (
                    f"{cluster_code}||"
                    f"{first_submission_id}||"
                    f"{second_submission_id}"
                )

                # 生成稳定 evidence key。
                evidence_key = (
                    "evidence_"
                    + _sha256_text(
                        (
                            f"{RULE_VERSION}||"
                            f"{pair_key}||"
                            f"{evidence_type}||"
                            f"{matched_value}"
                        )
                    )[:24]
                )

                # 保存一条与数据库字段接近的 evidence 草案。
                evidence_rows.append(
                    {
                        "id": None,
                        "dedup_run_id": None,
                        "matched_application_id": None,
                        "evidence_type": (
                            evidence_type
                        ),
                        "evidence_strength": (
                            evidence_strength
                        ),
                        "is_match": 1,
                        "score": 1.0,
                        "matched_person_identifier_claim_id": (
                            None
                        ),
                        "source_reference_type": (
                            "submission_normalized"
                        ),
                        "source_reference_id": (
                            first_submission_id
                        ),
                        "evidence_metadata": (
                            _stable_json(
                                {
                                    "matched_value_hash": (
                                        _sha256_text(
                                            matched_value
                                        )
                                    ),
                                    "primary_match_rule": (
                                        primary_match_rule
                                    ),
                                    "is_primary_rule": (
                                        bool(
                                            is_primary_rule
                                        )
                                    ),
                                    "rule_version": (
                                        RULE_VERSION
                                    ),
                                }
                            )
                        ),
                        "created_at": None,
                        "evidence_key": (
                            evidence_key
                        ),
                        "application_cluster_key": (
                            cluster_code
                        ),
                        "submission_normalized_id": (
                            first_submission_id
                        ),
                        "matched_submission_normalized_id": (
                            second_submission_id
                        ),
                        "matched_value_hash": (
                            _sha256_text(
                                matched_value
                            )
                        ),
                        "primary_match_rule": (
                            primary_match_rule
                        ),
                        # GitHub evidence 额外记录双方 last name
                        # 是否满足完全相同的辅助条件。
                        #
                        # 其他 evidence type 不使用这个条件，
                        # 因此保存 None，而不是错误地保存 0。
                        "github_last_name_match": (
                            1
                            if (
                                evidence_type
                                == (
                                    "github_exact_match_with_"
                                    "same_normalized_last_name"
                                )
                                and github_last_name_match
                            )
                            else None
                        ),
                        # 为了方便以后审计 GitHub 判断，
                        # 保存相同标准化 last name 的 SHA-256。
                        #
                        # 不在 evidence CSV 中额外暴露原始 last name。
                        "matched_normalized_last_name_hash": (
                            _sha256_text(
                                first_last_name
                            )
                            if (
                                evidence_type
                                == (
                                    "github_exact_match_with_"
                                    "same_normalized_last_name"
                                )
                                and github_last_name_match
                            )
                            else None
                        ),
                    }
                )

            # 邮箱相同时保存邮箱 evidence。
            if email_match:
                append_evidence(
                    evidence_type=(
                        "email_exact_match"
                    ),
                    evidence_strength="strong",
                    matched_value=(
                        first_record[
                            "normalized_email_address"
                        ]
                    ),
                    is_primary_rule=(
                        primary_match_rule
                        == "email_exact_match"
                    ),
                )

            # 保存全部共同电话号码 evidence。
            for shared_phone_value in (
                shared_phone_values
            ):
                append_evidence(
                    evidence_type=(
                        "phone_last_10_exact_match"
                    ),
                    evidence_strength="strong",
                    matched_value=(
                        shared_phone_value
                    ),
                    is_primary_rule=(
                        primary_match_rule
                        == "phone_last_10_exact_match"
                    ),
                )

            # 保存全部共同 LinkedIn evidence。
            for shared_linkedin_value in (
                shared_linkedin_values
            ):
                append_evidence(
                    evidence_type=(
                        "linkedin_exact_match"
                    ),
                    evidence_strength="strong",
                    matched_value=(
                        shared_linkedin_value
                    ),
                    is_primary_rule=(
                        primary_match_rule
                        == "linkedin_exact_match"
                    ),
                )

            # 只有 GitHub URL 相同并且双方标准化 last name 相同时，
            # 才保存 GitHub match evidence。
            if github_last_name_match:
                for shared_github_value in (
                    shared_github_values
                ):
                    append_evidence(
                        evidence_type=(
                            "github_exact_match_with_"
                            "same_normalized_last_name"
                        ),
                        evidence_strength="medium",
                        matched_value=(
                            shared_github_value
                        ),
                        is_primary_rule=(
                            primary_match_rule
                            == (
                                "github_exact_match_with_"
                                "same_normalized_last_name"
                            )
                        ),
                    )

    # 创建 evidence DataFrame。
    evidence_dataframe = pd.DataFrame(
        evidence_rows
    )

    # 没有 evidence 时创建固定 columns。
    if evidence_dataframe.empty:
        evidence_dataframe = pd.DataFrame(
            columns=[
                "id",
                "dedup_run_id",
                "matched_application_id",
                "evidence_type",
                "evidence_strength",
                "is_match",
                "score",
                "matched_person_identifier_claim_id",
                "source_reference_type",
                "source_reference_id",
                "evidence_metadata",
                "created_at",
                "evidence_key",
                "application_cluster_key",
                "submission_normalized_id",
                "matched_submission_normalized_id",
                "matched_value_hash",
                "primary_match_rule",
                "github_last_name_match",
                "matched_normalized_last_name_hash",
            ]
        )
    else:
        # 固定 evidence 排序。
        evidence_dataframe = (
            evidence_dataframe.sort_values(
                [
                    "application_cluster_key",
                    "submission_normalized_id",
                    "matched_submission_normalized_id",
                    "evidence_type",
                    "matched_value_hash",
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

    # 保存每个 submission 的最终 union root。
    component_root_by_submission_id = {
        int(submission_id): (
            union_find.find(
                submission_id
            )
        )
        for submission_id in (
            dedup_source_dataframe[
                "submission_normalized_id"
            ].tolist()
        )
    }

    # 返回 evidence 和 component root。
    return (
        evidence_dataframe,
        component_root_by_submission_id,
    )


# ============================================================
# 第九部分：选择每组最新 submission 并生成 dedup run 草案
# ============================================================

def _build_dedup_run_dataframe(
    dedup_source_dataframe,
    evidence_dataframe,
    component_root_by_submission_id,
):
    """
    为每一条进入查重的 submission 生成一条 Submission_dedup_run 草案。
    """

    # 复制源数据，避免修改调用方 DataFrame。
    working_dataframe = (
        dedup_source_dataframe.copy()
    )

    # 加入 union component root。
    working_dataframe[
        "component_root"
    ] = working_dataframe[
        "submission_normalized_id"
    ].map(
        component_root_by_submission_id
    )

    # duplicate group 不能跨 application cluster。
    working_dataframe[
        "component_identity_text"
    ] = working_dataframe.apply(
        lambda row: (
            f"{row['cluster_code']}||"
            f"{int(row['component_root'])}"
        ),
        axis=1,
    )

    # 先临时按 component 分组取得成员列表。
    component_members = (
        working_dataframe.groupby(
            "component_identity_text",
            sort=True,
        )[
            "submission_normalized_id"
        ]
        .apply(
            lambda values: sorted(
                int(value)
                for value in values
            )
        )
        .to_dict()
    )

    # 使用 cluster + 全部成员 id 生成稳定 duplicate_group_key。
    duplicate_group_key_by_component = {
        component_identity: (
            "duplicate_group_"
            + _sha256_text(
                (
                    f"{RULE_VERSION}||"
                    f"{component_identity.split('||')[0]}||"
                    + ",".join(
                        str(value)
                        for value in members
                    )
                )
            )[:24]
        )
        for (
            component_identity,
            members,
        ) in component_members.items()
    }

    # 回填 duplicate_group_key。
    working_dataframe[
        "duplicate_group_key"
    ] = working_dataframe[
        "component_identity_text"
    ].map(
        duplicate_group_key_by_component
    )

    # 将 submitted_at 解析成 UTC datetime，只用于排序。
    working_dataframe[
        "_submitted_at_sort"
    ] = pd.to_datetime(
        working_dataframe[
            "submitted_at"
        ],
        errors="coerce",
        utc=True,
    )

    # 保存每个 duplicate group 的 canonical submission id。
    canonical_submission_id_by_group = {}

    # 逐组选择最新 submission。
    for (
        duplicate_group_key,
        group_rows,
    ) in working_dataframe.groupby(
        "duplicate_group_key",
        sort=True,
    ):
        # submitted_at 缺失时排在更早位置。
        ordered_group_rows = (
            group_rows.sort_values(
                [
                    "_submitted_at_sort",
                    "submission_normalized_id",
                ],
                ascending=[
                    False,
                    False,
                ],
                na_position="last",
                kind="mergesort",
            )
        )

        # 第一行是稳定选出的最新 submission。
        canonical_submission_id = int(
            ordered_group_rows.iloc[0][
                "submission_normalized_id"
            ]
        )

        # 保存 canonical id。
        canonical_submission_id_by_group[
            duplicate_group_key
        ] = canonical_submission_id

    # 加入 canonical submission id。
    working_dataframe[
        "canonical_submission_normalized_id"
    ] = working_dataframe[
        "duplicate_group_key"
    ].map(
        canonical_submission_id_by_group
    )

    # 计算每个 duplicate group 的 submission 数量。
    duplicate_group_sizes = (
        working_dataframe.groupby(
            "duplicate_group_key",
            sort=True,
        )[
            "submission_normalized_id"
        ]
        .count()
        .to_dict()
    )

    # 构建每条 dedup run。
    dedup_run_rows = []

    # 按 submission id 固定顺序处理。
    for _, row in (
        working_dataframe.sort_values(
            [
                "submission_normalized_id",
            ],
            kind="mergesort",
        ).iterrows()
    ):
        # 取得标准 Python int id。
        submission_normalized_id = int(
            row[
                "submission_normalized_id"
            ]
        )

        # 取得 duplicate group key。
        duplicate_group_key = row[
            "duplicate_group_key"
        ]

        # 取得 canonical id。
        canonical_submission_id = int(
            row[
                "canonical_submission_normalized_id"
            ]
        )

        # 取得 group 大小。
        duplicate_group_size = int(
            duplicate_group_sizes[
                duplicate_group_key
            ]
        )

        # 判断当前 submission 是否为 canonical。
        is_canonical_submission = int(
            submission_normalized_id
            == canonical_submission_id
        )

        # 单条组和重复组 canonical 都准备进入 Application。
        if is_canonical_submission == 1:
            decision = (
                "accepted_as_application"
            )
            matched_submission_id = None
        else:
            decision = (
                "duplicate_suppressed"
            )
            matched_submission_id = (
                canonical_submission_id
            )

        # 检查当前 submission 是否参与 strong evidence。
        if evidence_dataframe.empty:
            strong_match_score = 0.0
            resume_match_score = 0.0
        else:
            # 找出包含当前 submission 的 evidence。
            submission_evidence = (
                evidence_dataframe.loc[
                    (
                        evidence_dataframe[
                            "submission_normalized_id"
                        ]
                        == submission_normalized_id
                    )
                    | (
                        evidence_dataframe[
                            "matched_submission_normalized_id"
                        ]
                        == submission_normalized_id
                    )
                ]
            )

            # 邮箱、电话、LinkedIn 属于 strong evidence。
            strong_match_score = float(
                (
                    submission_evidence[
                        "evidence_strength"
                    ]
                    == "strong"
                ).any()
            )

            # phone、LinkedIn、GitHub 属于 resume evidence。
            resume_match_score = float(
                submission_evidence[
                    "evidence_type"
                ].isin(
                    [
                        "phone_last_10_exact_match",
                        "linkedin_exact_match",
                        (
                            "github_exact_match_with_"
                            "same_normalized_last_name"
                        ),
                    ]
                ).any()
            )

        # duplicate group 大于一条时 final match score 为 1。
        final_match_score = float(
            duplicate_group_size > 1
        )

        # 生成稳定 dedup run key。
        dedup_run_key = (
            "dedup_run_"
            + _sha256_text(
                (
                    f"{RULE_VERSION}||"
                    f"{submission_normalized_id}||"
                    f"{row['cluster_code']}"
                )
            )[:24]
        )

        # 保存 dedup run 草案。
        dedup_run_rows.append(
            {
                "id": None,
                "submission_normalized_id": (
                    submission_normalized_id
                ),
                "application_cluster_id": None,
                "status": "completed",
                "decision": decision,
                "matched_application_id": None,
                "strong_match_score": (
                    strong_match_score
                ),
                "resume_match_score": (
                    resume_match_score
                ),
                "final_match_score": (
                    final_match_score
                ),
                "rule_version": RULE_VERSION,
                "started_at": None,
                "completed_at": None,
                "reviewed_by": None,
                "reviewed_at": None,
                "created_at": None,
                "dedup_run_key": (
                    dedup_run_key
                ),
                "application_cluster_key": (
                    row[
                        "cluster_code"
                    ]
                ),
                "duplicate_group_key": (
                    duplicate_group_key
                ),
                "duplicate_group_size": (
                    duplicate_group_size
                ),
                "is_canonical_submission": (
                    is_canonical_submission
                ),
                "canonical_submission_normalized_id": (
                    canonical_submission_id
                ),
                "matched_submission_normalized_id": (
                    matched_submission_id
                ),
                "submitted_at": row[
                    "submitted_at"
                ],
            }
        )

    # 创建 dedup run DataFrame。
    dedup_run_dataframe = pd.DataFrame(
        dedup_run_rows
    )

    # 固定 dedup run 排序。
    dedup_run_dataframe = (
        dedup_run_dataframe.sort_values(
            [
                "submission_normalized_id",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 建立 submission id -> dedup run key mapping。
    dedup_run_key_by_submission_id = (
        dedup_run_dataframe.set_index(
            "submission_normalized_id"
        )[
            "dedup_run_key"
        ].to_dict()
    )

    # 创建 evidence copy。
    #
    # 即使本批次没有任何 evidence，也仍然建立 dedup_run_key column，
    # 保证不同数据批次输出的 CSV schema 完全一致。
    evidence_dataframe = (
        evidence_dataframe.copy()
    )

    # evidence 属于 pair 中第一条 submission 的 dedup run。
    #
    # 加入可用于未来映射数据库 dedup_run_id 的稳定 key。
    evidence_dataframe[
        "dedup_run_key"
    ] = evidence_dataframe[
        "submission_normalized_id"
    ].map(
        dedup_run_key_by_submission_id
    )

    # 返回 dedup run、evidence 和包含 group 信息的工作表。
    return (
        dedup_run_dataframe,
        evidence_dataframe,
        working_dataframe,
    )


# ============================================================
# 第十部分：生成待创建 Application 草案
# ============================================================

def _build_application_candidate_dataframe(
    dedup_run_dataframe,
    working_dataframe,
):
    """
    只为每个重复组的 canonical submission 生成 Application candidate。

    当前不创建 Person、Hiring_pipeline 或 Application，
    所以对应数据库 id 保持空值。
    """

    # 只保留 accepted_as_application。
    accepted_runs = (
        dedup_run_dataframe.loc[
            dedup_run_dataframe[
                "decision"
            ]
            == "accepted_as_application"
        ].copy()
    )

    # 取得 application candidate 需要的源字段。
    candidate_source_columns = (
        working_dataframe[
            [
                "submission_normalized_id",
                "raw_submission_id",
                "cluster_code",
                "duplicate_group_key",
                "normalized_first_name",
                "normalized_last_name",
                "normalized_email_address",
                "submitted_at",
            ]
        ]
    )

    # 合并 accepted dedup run 和 submission 源字段。
    accepted_candidates = (
        accepted_runs.merge(
            candidate_source_columns,
            on=[
                "submission_normalized_id",
                "duplicate_group_key",
            ],
            how="left",
            validate="one_to_one",
            suffixes=(
                "",
                "_source",
            ),
        )
    )

    # 生成和 Application 表字段接近的草案。
    application_candidate_dataframe = (
        pd.DataFrame(
            {
                "id": None,
                "person_id": None,
                "application_cluster_id": None,
                "application_status": (
                    "dedup_accepted"
                ),
                "hiring_pipeline_id": None,
                "current_stage_id": None,
                "current_stage_entered_at": None,
                "applied_at": (
                    accepted_candidates[
                        "submitted_at_source"
                    ]
                ),
                "created_at": None,
                "updated_at": None,
                "submission_normalized_id": (
                    accepted_candidates[
                        "submission_normalized_id"
                    ]
                ),
                "raw_submission_id": (
                    accepted_candidates[
                        "raw_submission_id"
                    ]
                ),
                "application_cluster_key": (
                    accepted_candidates[
                        "cluster_code"
                    ]
                ),
                "duplicate_group_key": (
                    accepted_candidates[
                        "duplicate_group_key"
                    ]
                ),
                "dedup_run_key": (
                    accepted_candidates[
                        "dedup_run_key"
                    ]
                ),
                "normalized_first_name": (
                    accepted_candidates[
                        "normalized_first_name"
                    ]
                ),
                "normalized_last_name": (
                    accepted_candidates[
                        "normalized_last_name"
                    ]
                ),
                "normalized_email_address": (
                    accepted_candidates[
                        "normalized_email_address"
                    ]
                ),
                "rule_version": RULE_VERSION,
            }
        )
    )

    # 按 submission id 固定排序。
    application_candidate_dataframe = (
        application_candidate_dataframe.sort_values(
            [
                "submission_normalized_id",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # 返回 Application candidate 草案。
    return application_candidate_dataframe


# ============================================================
# 第十一部分：稳定输出与验证
# ============================================================

def _dataframe_for_csv(
    dataframe,
):
    """
    创建适合稳定 CSV 输出的 DataFrame copy。

    set/list/dict 会转换成稳定 JSON。
    """

    # 复制 DataFrame。
    output_dataframe = (
        dataframe.copy()
    )

    # 逐列处理 object values。
    for column_name in (
        output_dataframe.columns
    ):
        # 只处理 object dtype。
        if (
            output_dataframe[
                column_name
            ].dtype
            != "object"
        ):
            continue

        # set/list/dict 转换成稳定 JSON。
        output_dataframe[
            column_name
        ] = output_dataframe[
            column_name
        ].apply(
            lambda value: (
                _stable_json(
                    sorted(
                        value
                    )
                    if isinstance(
                        value,
                        set,
                    )
                    else value
                )
                if isinstance(
                    value,
                    (
                        set,
                        list,
                        dict,
                    ),
                )
                else value
            )
        )

    # 返回输出 copy。
    return output_dataframe


def _save_stable_csv(
    dataframe,
    file_name,
):
    """
    使用固定参数保存 CSV，并返回文件路径和 SHA-256。
    """

    # 确保规则版本目录存在。
    os.makedirs(
        RULE_VERSION_OUTPUT_FOLDER,
        exist_ok=True,
    )

    # 生成完整输出路径。
    output_path = os.path.join(
        RULE_VERSION_OUTPUT_FOLDER,
        file_name,
    )

    # 转换复杂 object values。
    output_dataframe = (
        _dataframe_for_csv(
            dataframe
        )
    )

    # 使用稳定列顺序、行顺序和换行符保存 CSV。
    output_dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n",
        na_rep="",
        float_format="%.6f",
    )

    # 读取最终文件 bytes。
    with open(
        output_path,
        "rb",
    ) as output_file:
        output_bytes = (
            output_file.read()
        )

    # 计算输出文件 hash。
    output_sha256 = hashlib.sha256(
        output_bytes
    ).hexdigest()

    # 返回路径和 hash。
    return (
        output_path,
        output_sha256,
    )


def _build_analysis_run_key(
    dedup_source_dataframe,
):
    """
    根据规则版本、规则配置、batch、CV CSV 和 submission ids
    生成确定性 analysis_run_key。
    """

    # 读取 CV CSV bytes 并计算 hash。
    with open(
        CV_EXTRACTION_CSV_PATH,
        "rb",
    ) as cv_file:
        cv_file_sha256 = (
            hashlib.sha256(
                cv_file.read()
            ).hexdigest()
        )

    # 固定排序的 submission ids。
    submission_ids = sorted(
        int(value)
        for value in (
            dedup_source_dataframe[
                "submission_normalized_id"
            ].tolist()
        )
    )

    # 生成 run identity object。
    run_identity = {
        "rule_version": RULE_VERSION,
        "rule_config": RULE_CONFIG,
        "ingestion_batch_id": int(
            INGESTION_BATCH_ID
        ),
        "cv_file_sha256": (
            cv_file_sha256
        ),
        "submission_normalized_ids": (
            submission_ids
        ),
    }

    # 生成完整 run identity JSON。
    run_identity_json = (
        _stable_json(
            run_identity
        )
    )

    # 生成稳定 run key。
    analysis_run_key = (
        "analysis_run_"
        + _sha256_text(
            run_identity_json
        )[:24]
    )

    # 返回 run key、CV hash 和 identity JSON。
    return (
        analysis_run_key,
        cv_file_sha256,
        run_identity_json,
    )


def _build_summary_dataframe(
    analysis_run_key,
    cv_file_sha256,
    run_identity_json,
    dedup_source_dataframe,
    excluded_dataframe,
    identity_feature_dataframe,
    cluster_dataframe,
    dedup_run_dataframe,
    evidence_dataframe,
    application_candidate_dataframe,
):
    """
    生成单行查重执行汇总。
    """

    # 统计 duplicate_suppressed。
    duplicate_suppressed_count = int(
        (
            dedup_run_dataframe[
                "decision"
            ]
            == "duplicate_suppressed"
        ).sum()
    )

    # 统计 accepted application candidates。
    accepted_application_count = int(
        (
            dedup_run_dataframe[
                "decision"
            ]
            == "accepted_as_application"
        ).sum()
    )

    # 统计大于一条记录的 duplicate groups。
    duplicate_group_count = int(
        dedup_run_dataframe.loc[
            dedup_run_dataframe[
                "duplicate_group_size"
            ]
            > 1,
            "duplicate_group_key",
        ].nunique()
    )

    # 创建单行汇总 DataFrame。
    return pd.DataFrame(
        [
            {
                "analysis_run_key": (
                    analysis_run_key
                ),
                "rule_version": RULE_VERSION,
                "ingestion_batch_id": int(
                    INGESTION_BATCH_ID
                ),
                "cv_file_sha256": (
                    cv_file_sha256
                ),
                "rule_config_json": (
                    _stable_json(
                        RULE_CONFIG
                    )
                ),
                "run_identity_json": (
                    run_identity_json
                ),
                "eligible_submission_count": (
                    len(
                        dedup_source_dataframe
                    )
                ),
                "excluded_submission_count": (
                    len(
                        excluded_dataframe
                    )
                ),
                "identity_feature_count": (
                    len(
                        identity_feature_dataframe
                    )
                ),
                "application_cluster_count": (
                    len(
                        cluster_dataframe
                    )
                ),
                "dedup_run_count": (
                    len(
                        dedup_run_dataframe
                    )
                ),
                "match_evidence_count": (
                    len(
                        evidence_dataframe
                    )
                ),
                "duplicate_group_count": (
                    duplicate_group_count
                ),
                "duplicate_suppressed_count": (
                    duplicate_suppressed_count
                ),
                "accepted_application_count": (
                    accepted_application_count
                ),
                "application_candidate_count": (
                    len(
                        application_candidate_dataframe
                    )
                ),
            }
        ]
    )


# ============================================================
# 第十二部分：运行完整独立查重流程
# ============================================================

# 读取、合并并筛选查重源数据。
(
    dedup_source_dataframe,
    excluded_submission_dataframe,
) = _build_dedup_source_dataframe()

# 生成 Submission_identity_feature 草案。
submission_identity_feature_dataframe = (
    _build_identity_feature_dataframe(
        dedup_source_dataframe
    )
)

# 生成 Application_cluster 草案。
application_cluster_dataframe = (
    _build_application_cluster_dataframe(
        dedup_source_dataframe
    )
)

# 生成 pair evidence 和连通重复组。
(
    submission_match_evidence_dataframe,
    component_root_by_submission_id,
) = _build_pair_evidence_and_components(
    dedup_source_dataframe
)

# 生成 Submission_dedup_run 草案并回填 evidence 的稳定 dedup_run_key。
(
    submission_dedup_run_dataframe,
    submission_match_evidence_dataframe,
    dedup_working_dataframe,
) = _build_dedup_run_dataframe(
    dedup_source_dataframe=(
        dedup_source_dataframe
    ),
    evidence_dataframe=(
        submission_match_evidence_dataframe
    ),
    component_root_by_submission_id=(
        component_root_by_submission_id
    ),
)

# 生成待创建 Application 草案。
application_candidate_dataframe = (
    _build_application_candidate_dataframe(
        dedup_run_dataframe=(
            submission_dedup_run_dataframe
        ),
        working_dataframe=(
            dedup_working_dataframe
        ),
    )
)

# 生成确定性 analysis run identity。
(
    ANALYSIS_RUN_KEY,
    CV_FILE_SHA256,
    RUN_IDENTITY_JSON,
) = _build_analysis_run_key(
    dedup_source_dataframe
)

# 为所有主要输出加入 analysis_run_key。
for output_dataframe in [
    submission_identity_feature_dataframe,
    application_cluster_dataframe,
    submission_dedup_run_dataframe,
    submission_match_evidence_dataframe,
    application_candidate_dataframe,
    excluded_submission_dataframe,
]:
    # 已经存在时先删除，保证列位置固定且不会重复。
    if (
        "analysis_run_key"
        in output_dataframe.columns
    ):
        output_dataframe.drop(
            columns=[
                "analysis_run_key",
            ],
            inplace=True,
        )

    # 将 analysis_run_key 插入第一列。
    output_dataframe.insert(
        0,
        "analysis_run_key",
        ANALYSIS_RUN_KEY,
    )

# 生成执行汇总。
dedup_analysis_summary_dataframe = (
    _build_summary_dataframe(
        analysis_run_key=(
            ANALYSIS_RUN_KEY
        ),
        cv_file_sha256=(
            CV_FILE_SHA256
        ),
        run_identity_json=(
            RUN_IDENTITY_JSON
        ),
        dedup_source_dataframe=(
            dedup_source_dataframe
        ),
        excluded_dataframe=(
            excluded_submission_dataframe
        ),
        identity_feature_dataframe=(
            submission_identity_feature_dataframe
        ),
        cluster_dataframe=(
            application_cluster_dataframe
        ),
        dedup_run_dataframe=(
            submission_dedup_run_dataframe
        ),
        evidence_dataframe=(
            submission_match_evidence_dataframe
        ),
        application_candidate_dataframe=(
            application_candidate_dataframe
        ),
    )
)

# 保存所有 CSV 的路径和 SHA-256。
saved_output_files = []

# 定义输出 DataFrame 和文件名。
output_file_definitions = [
    (
        submission_identity_feature_dataframe,
        "submission_identity_feature.csv",
    ),
    (
        application_cluster_dataframe,
        "application_cluster.csv",
    ),
    (
        submission_dedup_run_dataframe,
        "submission_dedup_run.csv",
    ),
    (
        submission_match_evidence_dataframe,
        "submission_match_evidence.csv",
    ),
    (
        application_candidate_dataframe,
        "application_candidate.csv",
    ),
    (
        excluded_submission_dataframe,
        "excluded_submission.csv",
    ),
    (
        dedup_analysis_summary_dataframe,
        "dedup_analysis_summary.csv",
    ),
]

# 逐个保存稳定 CSV。
for (
    output_dataframe,
    output_file_name,
) in output_file_definitions:
    # 保存当前 CSV。
    (
        output_path,
        output_sha256,
    ) = _save_stable_csv(
        dataframe=output_dataframe,
        file_name=output_file_name,
    )

    # 保存输出信息。
    saved_output_files.append(
        {
            "file_name": (
                output_file_name
            ),
            "output_path": (
                output_path
            ),
            "row_count": len(
                output_dataframe
            ),
            "sha256": (
                output_sha256
            ),
        }
    )

# 转换成输出文件清单 DataFrame。
saved_output_files_dataframe = (
    pd.DataFrame(
        saved_output_files
    )
)

# 显示规则版本和稳定 run key。
print(
    "\nRule version:",
    RULE_VERSION,
)
print(
    "Analysis run key:",
    ANALYSIS_RUN_KEY,
)
print(
    "Output folder:",
    RULE_VERSION_OUTPUT_FOLDER,
)

# 显示整体汇总。
print(
    "\nDedup analysis summary:"
)
display(
    dedup_analysis_summary_dataframe
)

# 显示各输出文件路径、记录数和文件 hash。
print(
    "\nSaved output files:"
)
display(
    saved_output_files_dataframe
)

# 显示 dedup run 前 10 条。
print(
    "\nSubmission dedup run preview:"
)
display(
    submission_dedup_run_dataframe.head(
        10
    )
)

# 显示 match evidence 前 10 条。
print(
    "\nSubmission match evidence preview:"
)
display(
    submission_match_evidence_dataframe.head(
        10
    )
)

# 本 cell 到这里结束。
#
# 当前只生成并保存查重结果 CSV。
# 不会向 D1 写入任何查重、cluster、Person 或 Application 数据。


# Application csv 生成存入 Google Drive

In [ ]:
"""
Google Colab 完全独立 cell：
使用已经生成的岗位申请查重 CSV、CV extraction CSV 和 D1 中的
Submission_normalized / raw_submission 数据，生成最终 Application CSV。

重要：
1. 本 cell 不依赖任何其他 Colab cell 中的 class、function 或 variable。
2. 本 cell 只读取 D1，不会向 D1 执行 INSERT、UPDATE、DELETE 或 ALTER。
3. 本 cell 每次都覆盖同一个 application.csv，不会无限增加 CSV 文件。
4. 相同输入、相同 D1 数据和相同 RULE_VERSION 会产生相同的结果与行顺序。
5. application.csv 一条记录代表一个被查重流程保留的 Application candidate。
6. matched_submission_normalized_id 使用 JSON array 保存零个或多个重复 submission id。
7. 一对多的 dedup run、match evidence 和 identity feature 使用稳定 JSON 保存，
   避免把一条 Application 展开成多行。
"""

# ============================================================
# 第一部分：安装和导入本 cell 自己需要的全部 dependency
# ============================================================

# subprocess 用于在当前 Colab runtime 中安装缺少的 Python package。
import subprocess

# sys 用于取得当前 Colab runtime 正在使用的 Python executable。
import sys

# 尝试导入 requests。
try:
    # requests 用于调用 Cloudflare D1 REST API。
    import requests

# requests 不存在时，只在当前 Colab runtime 中安装。
except ImportError:
    # 使用当前 Python executable 安装 requests。
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "requests",
        ]
    )

    # 安装完成后重新导入 requests。
    import requests

# hashlib 用于计算最终 application.csv 的 SHA-256。
import hashlib

# json 用于把一对多数据稳定封装成 JSON。
import json

# os 用于创建输出文件夹和处理文件路径。
import os

# time 只用于 D1 请求失败后的重试等待。
import time

# numpy 用于识别 pandas / NumPy 标量和空值。
import numpy as np

# pandas 用于读取、合并、验证和保存 CSV。
import pandas as pd

# Google Drive 用于读取已有 CSV 并保存最终 application.csv。
from google.colab import drive

# userdata 用于安全读取 Colab Secrets 中的 Cloudflare API Token。
from google.colab import userdata

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)


# ============================================================
# 第二部分：本 cell 的独立配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 API Token 时立即停止。
if not API_TOKEN:
    # 提示用户先配置 Colab Secret。
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请先在 Colab Secrets 中保存 Cloudflare API Token。"
    )

# 指定要读取的查重规则版本。
#
# 如果实际结果位于 application_dedup_v1 文件夹，
# 只需要把下面的值修改成 "application_dedup_v1"。
RULE_VERSION = "application_dedup_v2"

# Google Drive 项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 数据集文件夹。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 当前查重规则版本的结果文件夹。
DEDUP_RESULT_FOLDER = (
    f"{DATASET_FOLDER}/dedup_results/"
    f"{RULE_VERSION}"
)

# 初步清洗成功并准备用于 CV extraction 的 CSV。
SUCCESSFUL_SUBMISSION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "successful_submission_for_extraction.csv"
)

# CV extraction 汇总 CSV。
RESUME_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "resume_extraction_consolidated.csv"
)

# 查重阶段生成的 identity feature CSV。
SUBMISSION_IDENTITY_FEATURE_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "submission_identity_feature.csv"
)

# 查重阶段生成的 Application candidate CSV。
APPLICATION_CANDIDATE_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "application_candidate.csv"
)

# 查重阶段生成的 Application cluster CSV。
APPLICATION_CLUSTER_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "application_cluster.csv"
)

# 查重阶段生成的 dedup run CSV。
SUBMISSION_DEDUP_RUN_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "submission_dedup_run.csv"
)

# 查重阶段生成的 match evidence CSV。
SUBMISSION_MATCH_EVIDENCE_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "submission_match_evidence.csv"
)

# 最终 Application CSV 固定输出路径。
#
# 同一规则版本重复运行时会直接覆盖这个文件。
APPLICATION_OUTPUT_CSV_PATH = (
    f"{DEDUP_RESULT_FOLDER}/"
    "application.csv"
)

# D1 REST API endpoint。
D1_API_URL = (
    "https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# D1 REST API request headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# D1 每一页读取的记录数量。
D1_PAGE_SIZE = 500


# ============================================================
# 第三部分：定义稳定读取、JSON 和 D1 查询 helper functions
# ============================================================

def _require_file(
    file_path,
):
    """
    检查必要输入文件是否存在。
    """

    # 文件不存在时立即停止。
    if not os.path.isfile(
        file_path
    ):
        # 显示具体缺少的文件路径。
        raise FileNotFoundError(
            f"找不到必要输入文件：{file_path}"
        )


def _read_csv(
    file_path,
):
    """
    使用统一设置读取 CSV。
    """

    # 先确认文件存在。
    _require_file(
        file_path
    )

    # 读取 CSV。
    dataframe = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        low_memory=False,
    )

    # 返回读取完成的 DataFrame。
    return dataframe


def _require_columns(
    dataframe,
    required_columns,
    dataframe_name,
):
    """
    检查 DataFrame 是否包含必要 columns。
    """

    # 找出缺失的必要 columns。
    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in dataframe.columns
    ]

    # 存在缺失 column 时立即停止。
    if missing_columns:
        # 显示 DataFrame 名称和缺失 columns。
        raise ValueError(
            f"{dataframe_name} 缺少必要 columns："
            f"{missing_columns}"
        )


def _to_nullable_integer(
    series,
):
    """
    把数据库 id 或 integer flag 转换成 pandas 可空整数 Int64。
    """

    # 先把可解析值转换成数字。
    numeric_series = pd.to_numeric(
        series,
        errors="coerce",
    )

    # 检查所有非空数字是否真的是整数。
    non_null_numeric_series = (
        numeric_series.dropna()
    )

    # 任何值带有非零小数时立即停止。
    if not (
        non_null_numeric_series
        .mod(1)
        .eq(0)
        .all()
    ):
        # 防止把真正异常的小数 ID 静默截断。
        raise ValueError(
            f"column {series.name!r} 中出现了非整数 ID。"
        )

    # 转换成支持 pd.NA 的 Int64。
    return numeric_series.astype(
        "Int64"
    )


def _python_scalar_or_none(
    value,
):
    """
    把 pandas / NumPy value 转换成 JSON 可以稳定序列化的 Python value。
    """

    # list 需要递归处理每一个 value。
    if isinstance(
        value,
        list,
    ):
        # 返回完成递归转换的新 list。
        return [
            _python_scalar_or_none(
                item
            )
            for item in value
        ]

    # tuple 也统一转换成 JSON list。
    if isinstance(
        value,
        tuple,
    ):
        # 返回完成递归转换的新 list。
        return [
            _python_scalar_or_none(
                item
            )
            for item in value
        ]

    # dict 需要递归处理每一个 value。
    if isinstance(
        value,
        dict,
    ):
        # 按 key 排序，保证 JSON 顺序稳定。
        return {
            str(key): _python_scalar_or_none(
                value[key]
            )
            for key in sorted(
                value
            )
        }

    # pandas / NumPy 空值统一转换成 Python None。
    try:
        # pd.isna 对普通 scalar 返回 bool。
        if bool(
            pd.isna(
                value
            )
        ):
            # Python None 写入 JSON 后成为 null。
            return None

    # 某些复杂对象不支持直接执行 pd.isna。
    except (
        TypeError,
        ValueError,
    ):
        # 对这些对象继续执行下面的类型转换。
        pass

    # NumPy scalar 转换成对应的 Python scalar。
    if isinstance(
        value,
        np.generic,
    ):
        # .item() 会把 np.int64 等转换成 Python int。
        return value.item()

    # pandas Timestamp 转换成 ISO8601 string。
    if isinstance(
        value,
        pd.Timestamp,
    ):
        # 使用 ISO8601，避免 locale 影响结果。
        return value.isoformat()

    # 其他普通 Python value 原样返回。
    return value


def _stable_json(
    value,
):
    """
    将 value 转换成稳定 JSON string。
    """

    # 使用固定 JSON 设置生成可重复结果。
    return json.dumps(
        _python_scalar_or_none(
            value
        ),
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def _records_to_stable_json(
    dataframe,
):
    """
    把一个 DataFrame 中的全部 records 完整封装成稳定 JSON array。
    """

    # 没有 records 时返回空 JSON array。
    if dataframe.empty:
        # 保证空值也使用明确的 array schema。
        return "[]"

    # 按当前固定行顺序转换成 records。
    records = dataframe.to_dict(
        orient="records"
    )

    # 返回稳定 JSON。
    return _stable_json(
        records
    )


def _execute_d1_select(
    sql,
    params=None,
    max_retries=3,
    retry_delay_seconds=1.0,
):
    """
    执行只读 D1 SELECT，并在请求失败时重试。
    """

    # 保存最后一次失败信息。
    last_error = None

    # 最多尝试 max_retries 次。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        # 建立 D1 JSON request payload。
        payload = {
            "sql": sql,
        }

        # SQL 存在 params 时加入 payload。
        if params is not None:
            # 把 NumPy scalar 全部转换成标准 Python scalar。
            payload["params"] = [
                _python_scalar_or_none(
                    value
                )
                for value in params
            ]

        # 捕获网络层异常。
        try:
            # 调用 Cloudflare D1 REST API。
            response = requests.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json=payload,
                timeout=60,
            )

        # 网络超时或连接失败。
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = (
                f"Network error: {error}"
            )

            # 还有重试机会时等待。
            if attempt < max_retries:
                # 只在失败后等待。
                time.sleep(
                    retry_delay_seconds
                )

                # 进入下一次尝试。
                continue

            # 已经没有重试机会。
            break

        # 尝试解析 Cloudflare JSON response。
        try:
            # 取得 response JSON。
            response_data = (
                response.json()
            )

        # response 不是有效 JSON。
        except ValueError:
            # 保存响应错误。
            last_error = (
                f"HTTP {response.status_code}, "
                "response is not valid JSON: "
                f"{response.text[:500]}"
            )

            # 还有重试机会时等待。
            if attempt < max_retries:
                # 只在失败后等待。
                time.sleep(
                    retry_delay_seconds
                )

                # 进入下一次尝试。
                continue

            # 已经没有重试机会。
            break

        # Cloudflare 或 D1 返回失败状态。
        if (
            response.status_code != 200
            or not response_data.get(
                "success",
                False,
            )
        ):
            # 保存 D1 errors。
            last_error = response_data.get(
                "errors",
                response_data,
            )

            # 还有重试机会时等待。
            if attempt < max_retries:
                # 只在失败后等待。
                time.sleep(
                    retry_delay_seconds
                )

                # 进入下一次尝试。
                continue

            # 已经没有重试机会。
            break

        # 取得 D1 result list。
        result = response_data.get(
            "result",
            [],
        )

        # 提取第一条 statement 的 results。
        rows = (
            result[0].get(
                "results",
                [],
            )
            if result
            else []
        )

        # 返回查询 records。
        return rows

    # 所有重试都失败时停止。
    raise RuntimeError(
        "D1 SELECT failed after "
        f"{max_retries} attempts: {last_error}"
    )


def _read_submission_normalized_from_d1():
    """
    分页读取 D1 Submission_normalized，
    并同时取得 raw_submission.source_record_number。
    """

    # 保存所有分页 records。
    all_rows = []

    # 第一页 offset 从 0 开始。
    offset = 0

    # 持续读取，直到最后一页。
    while True:
        # 只执行 SELECT，不修改数据库。
        sql = """
            SELECT
              sn.*,
              rs."ingestion_batch_id"
                AS "raw_ingestion_batch_id",
              rs."source_record_number"
                AS "source_record_number"
            FROM "Submission_normalized" AS sn
            INNER JOIN "raw_submission" AS rs
              ON rs."id" = sn."raw_submission_id"
            ORDER BY sn."id"
            LIMIT ?
            OFFSET ?;
        """

        # 读取当前页。
        page_rows = _execute_d1_select(
            sql=sql,
            params=[
                int(
                    D1_PAGE_SIZE
                ),
                int(
                    offset
                ),
            ],
        )

        # 保存当前页 records。
        all_rows.extend(
            page_rows
        )

        # 当前页少于 page size 时已经到达最后一页。
        if len(
            page_rows
        ) < D1_PAGE_SIZE:
            # 结束分页读取。
            break

        # 移动到下一页。
        offset += D1_PAGE_SIZE

    # 转换成 DataFrame。
    dataframe = pd.DataFrame(
        all_rows
    )

    # 数据库没有任何 Submission_normalized 时停止。
    if dataframe.empty:
        # Application CSV 无法在没有 normalized submission 时生成。
        raise ValueError(
            "D1 Submission_normalized 表中没有任何数据。"
        )

    # 检查必要 columns。
    _require_columns(
        dataframe=dataframe,
        required_columns=[
            "id",
            "raw_submission_id",
            "source_record_number",
        ],
        dataframe_name=(
            "D1 Submission_normalized"
        ),
    )

    # 将 D1 主键转换成可空整数。
    dataframe["id"] = (
        _to_nullable_integer(
            dataframe[
                "id"
            ]
        )
    )

    # 将 raw_submission_id 转换成可空整数。
    dataframe["raw_submission_id"] = (
        _to_nullable_integer(
            dataframe[
                "raw_submission_id"
            ]
        )
    )

    # 将 source_record_number 转换成可空整数。
    dataframe["source_record_number"] = (
        _to_nullable_integer(
            dataframe[
                "source_record_number"
            ]
        )
    )

    # 返回完整 D1 DataFrame。
    return dataframe


def _prefix_non_key_columns(
    dataframe,
    prefix,
    key_columns,
):
    """
    给来源 DataFrame 的非关联键 columns 加前缀，避免同名字段覆盖。
    """

    # 建立 column rename mapping。
    rename_mapping = {
        column_name: (
            f"{prefix}{column_name}"
        )
        for column_name in dataframe.columns
        if column_name not in key_columns
    }

    # 返回改名后的 DataFrame copy。
    return dataframe.rename(
        columns=rename_mapping
    ).copy()


# ============================================================
# 第四部分：读取全部 CSV 和 D1 数据
# ============================================================

# 读取 Application candidate。
application_candidate_dataframe = (
    _read_csv(
        APPLICATION_CANDIDATE_CSV_PATH
    )
)

# 读取 Application cluster。
application_cluster_dataframe = (
    _read_csv(
        APPLICATION_CLUSTER_CSV_PATH
    )
)

# 读取 dedup run。
submission_dedup_run_dataframe = (
    _read_csv(
        SUBMISSION_DEDUP_RUN_CSV_PATH
    )
)

# 读取 match evidence。
submission_match_evidence_dataframe = (
    _read_csv(
        SUBMISSION_MATCH_EVIDENCE_CSV_PATH
    )
)

# 读取 identity feature。
submission_identity_feature_dataframe = (
    _read_csv(
        SUBMISSION_IDENTITY_FEATURE_CSV_PATH
    )
)

# 读取 successful submission 与 resume_text。
successful_submission_for_extraction_dataframe = (
    _read_csv(
        SUCCESSFUL_SUBMISSION_CSV_PATH
    )
)

# 读取 CV extraction 汇总结果。
resume_extraction_consolidated_dataframe = (
    _read_csv(
        RESUME_EXTRACTION_CSV_PATH
    )
)

# 从 D1 读取 Submission_normalized 和 source_record_number。
submission_normalized_dataframe = (
    _read_submission_normalized_from_d1()
)

# 显示各来源记录数。
print(
    "Application candidate records:",
    len(
        application_candidate_dataframe
    ),
)

# 显示 D1 Submission_normalized 记录数。
print(
    "D1 Submission_normalized records:",
    len(
        submission_normalized_dataframe
    ),
)


# ============================================================
# 第五部分：验证并统一所有关联 ID
# ============================================================

# 检查 Application candidate 必要 columns。
_require_columns(
    dataframe=application_candidate_dataframe,
    required_columns=[
        "submission_normalized_id",
        "raw_submission_id",
        "application_cluster_key",
        "dedup_run_key",
    ],
    dataframe_name="application_candidate.csv",
)

# 检查 cluster 必要 columns。
_require_columns(
    dataframe=application_cluster_dataframe,
    required_columns=[
        "cluster_code",
    ],
    dataframe_name="application_cluster.csv",
)

# 检查 dedup run 必要 columns。
_require_columns(
    dataframe=submission_dedup_run_dataframe,
    required_columns=[
        "submission_normalized_id",
        "canonical_submission_normalized_id",
        "dedup_run_key",
    ],
    dataframe_name="submission_dedup_run.csv",
)

# 检查 evidence 必要 columns。
_require_columns(
    dataframe=submission_match_evidence_dataframe,
    required_columns=[
        "submission_normalized_id",
        "matched_submission_normalized_id",
    ],
    dataframe_name="submission_match_evidence.csv",
)

# 检查 identity feature 必要 columns。
_require_columns(
    dataframe=submission_identity_feature_dataframe,
    required_columns=[
        "submission_normalized_id",
    ],
    dataframe_name="submission_identity_feature.csv",
)

# 检查 successful submission 必要 columns。
_require_columns(
    dataframe=successful_submission_for_extraction_dataframe,
    required_columns=[
        "raw_submission_id",
        "submission_normalized_id",
        "resume_text",
    ],
    dataframe_name=(
        "successful_submission_for_extraction.csv"
    ),
)

# 检查 resume extraction 必要关联键。
_require_columns(
    dataframe=resume_extraction_consolidated_dataframe,
    required_columns=[
        "raw_submission_id",
    ],
    dataframe_name=(
        "resume_extraction_consolidated.csv"
    ),
)

# 需要转换成可空整数的 DataFrame / column pairs。
integer_column_definitions = [
    (
        application_candidate_dataframe,
        "submission_normalized_id",
    ),
    (
        application_candidate_dataframe,
        "raw_submission_id",
    ),
    (
        submission_dedup_run_dataframe,
        "submission_normalized_id",
    ),
    (
        submission_dedup_run_dataframe,
        "canonical_submission_normalized_id",
    ),
    (
        submission_dedup_run_dataframe,
        "matched_submission_normalized_id",
    ),
    (
        submission_match_evidence_dataframe,
        "submission_normalized_id",
    ),
    (
        submission_match_evidence_dataframe,
        "matched_submission_normalized_id",
    ),
    (
        submission_identity_feature_dataframe,
        "submission_normalized_id",
    ),
    (
        successful_submission_for_extraction_dataframe,
        "submission_normalized_id",
    ),
    (
        successful_submission_for_extraction_dataframe,
        "raw_submission_id",
    ),
    (
        resume_extraction_consolidated_dataframe,
        "raw_submission_id",
    ),
]

# 逐个统一 ID dtype。
for (
    target_dataframe,
    column_name,
) in integer_column_definitions:
    # 某些 optional column 不存在时跳过。
    if column_name not in target_dataframe.columns:
        # 继续处理下一个定义。
        continue

    # 转换成 pandas Int64。
    target_dataframe[column_name] = (
        _to_nullable_integer(
            target_dataframe[
                column_name
            ]
        )
    )

# Application candidate 的 submission_normalized_id 必须唯一。
if application_candidate_dataframe[
    "submission_normalized_id"
].duplicated().any():
    # 一条最终 Application 不能对应多个 candidate rows。
    raise ValueError(
        "application_candidate.csv 中存在重复的 "
        "submission_normalized_id。"
    )

# Application candidate 的 raw_submission_id 必须唯一。
if application_candidate_dataframe[
    "raw_submission_id"
].duplicated().any():
    # 一条 raw submission 不能生成多个最终 Application。
    raise ValueError(
        "application_candidate.csv 中存在重复的 "
        "raw_submission_id。"
    )

# D1 Submission_normalized.id 必须唯一。
if submission_normalized_dataframe[
    "id"
].duplicated().any():
    # 数据库主键理论上不可能重复。
    raise ValueError(
        "D1 Submission_normalized.id 出现重复。"
    )

# successful submission 的 submission_normalized_id 必须唯一。
if successful_submission_for_extraction_dataframe[
    "submission_normalized_id"
].duplicated().any():
    # resume_text 必须可以一对一关联。
    raise ValueError(
        "successful_submission_for_extraction.csv 中存在重复的 "
        "submission_normalized_id。"
    )

# 取得所有 Application candidate 的 normalized submission ids。
application_candidate_submission_ids = {
    int(
        submission_id
    )
    for submission_id in application_candidate_dataframe[
        "submission_normalized_id"
    ].dropna()
}

# 取得 successful submission CSV 中的 normalized submission ids。
successful_submission_ids = {
    int(
        submission_id
    )
    for submission_id in successful_submission_for_extraction_dataframe[
        "submission_normalized_id"
    ].dropna()
}

# 找出完全没有对应 successful submission record 的 candidate。
missing_successful_submission_ids = sorted(
    application_candidate_submission_ids
    - successful_submission_ids
)

# 关联 record 不存在时停止。
if missing_successful_submission_ids:
    # 这与原始 resume_text 自身为空是不同的问题。
    raise ValueError(
        "以下 Application candidate 在 "
        "successful_submission_for_extraction.csv 中不存在："
        f"{missing_successful_submission_ids}"
    )

# resume extraction 每个 raw_submission_id 必须只有一条汇总 record。
if resume_extraction_consolidated_dataframe[
    "raw_submission_id"
].duplicated().any():
    # consolidated CSV 应该已经是一条 submission 一行。
    raise ValueError(
        "resume_extraction_consolidated.csv 中存在重复的 "
        "raw_submission_id。"
    )


# ============================================================
# 第六部分：建立 canonical submission 与全部 matched submission 的关系
# ============================================================

# 删除缺少 canonical id 的异常 dedup run。
dedup_group_mapping_dataframe = (
    submission_dedup_run_dataframe.dropna(
        subset=[
            "submission_normalized_id",
            "canonical_submission_normalized_id",
        ]
    ).copy()
)

# 固定 group mapping 行顺序。
dedup_group_mapping_dataframe = (
    dedup_group_mapping_dataframe.sort_values(
        [
            "canonical_submission_normalized_id",
            "submission_normalized_id",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

# 建立 submission id -> canonical submission id mapping。
canonical_id_by_submission_id = {
    int(
        row[
            "submission_normalized_id"
        ]
    ): int(
        row[
            "canonical_submission_normalized_id"
        ]
    )
    for _, row in (
        dedup_group_mapping_dataframe.iterrows()
    )
}

# 保存每一个 canonical submission 对应的全部 matched ids。
matched_ids_by_canonical_id = {}

# 按 canonical submission 分组。
for (
    canonical_submission_id,
    group_dataframe,
) in dedup_group_mapping_dataframe.groupby(
    "canonical_submission_normalized_id",
    sort=True,
):
    # 把 canonical id 转换成 Python int。
    canonical_submission_id = int(
        canonical_submission_id
    )

    # 取得当前重复组的全部 submission ids。
    group_submission_ids = sorted(
        {
            int(
                submission_id
            )
            for submission_id in group_dataframe[
                "submission_normalized_id"
            ].dropna()
        }
    )

    # matched ids 不包含 canonical submission 自己。
    matched_submission_ids = [
        submission_id
        for submission_id in group_submission_ids
        if submission_id
        != canonical_submission_id
    ]

    # 保存稳定排序后的 matched ids。
    matched_ids_by_canonical_id[
        canonical_submission_id
    ] = matched_submission_ids

# 为 Application candidate 生成 matched id JSON array。
application_candidate_dataframe[
    "matched_submission_normalized_id"
] = application_candidate_dataframe[
    "submission_normalized_id"
].map(
    lambda submission_id: _stable_json(
        matched_ids_by_canonical_id.get(
            int(
                submission_id
            ),
            [],
        )
    )
)


# ============================================================
# 第七部分：把一对多数据按最终 canonical Application 完整封装
# ============================================================

# 给每一个 dedup run record 标记对应的 canonical submission id。
submission_dedup_run_dataframe[
    "_canonical_submission_normalized_id"
] = submission_dedup_run_dataframe[
    "submission_normalized_id"
].map(
    canonical_id_by_submission_id
)

# 给每一个 identity feature 标记对应的 canonical submission id。
submission_identity_feature_dataframe[
    "_canonical_submission_normalized_id"
] = submission_identity_feature_dataframe[
    "submission_normalized_id"
].map(
    canonical_id_by_submission_id
)

# evidence 的第一条 submission id 用于查找 canonical group。
submission_match_evidence_dataframe[
    "_canonical_submission_normalized_id"
] = submission_match_evidence_dataframe[
    "submission_normalized_id"
].map(
    canonical_id_by_submission_id
)

# 保存每个 canonical id 对应的 dedup run JSON。
dedup_run_json_by_canonical_id = {}

# 按 canonical id 分组封装 dedup run。
for (
    canonical_submission_id,
    group_dataframe,
) in submission_dedup_run_dataframe.dropna(
    subset=[
        "_canonical_submission_normalized_id",
    ]
).groupby(
    "_canonical_submission_normalized_id",
    sort=True,
):
    # 删除只用于内部关联的 temporary column。
    json_dataframe = group_dataframe.drop(
        columns=[
            "_canonical_submission_normalized_id",
        ]
    ).copy()

    # 按 submission id 固定排序。
    json_dataframe = json_dataframe.sort_values(
        [
            "submission_normalized_id",
        ],
        kind="mergesort",
    )

    # 保存当前重复组全部 dedup run records。
    dedup_run_json_by_canonical_id[
        int(
            canonical_submission_id
        )
    ] = _records_to_stable_json(
        json_dataframe
    )

# 保存每个 canonical id 对应的 evidence JSON。
evidence_json_by_canonical_id = {}

# 按 canonical id 分组封装 evidence。
for (
    canonical_submission_id,
    group_dataframe,
) in submission_match_evidence_dataframe.dropna(
    subset=[
        "_canonical_submission_normalized_id",
    ]
).groupby(
    "_canonical_submission_normalized_id",
    sort=True,
):
    # 删除只用于内部关联的 temporary column。
    json_dataframe = group_dataframe.drop(
        columns=[
            "_canonical_submission_normalized_id",
        ]
    ).copy()

    # 优先使用 evidence_key 固定排序。
    evidence_sort_columns = (
        [
            "evidence_key",
        ]
        if "evidence_key"
        in json_dataframe.columns
        else [
            "submission_normalized_id",
            "matched_submission_normalized_id",
            "evidence_type",
        ]
    )

    # 固定 evidence record 顺序。
    json_dataframe = json_dataframe.sort_values(
        evidence_sort_columns,
        kind="mergesort",
    )

    # 保存当前重复组全部 evidence records。
    evidence_json_by_canonical_id[
        int(
            canonical_submission_id
        )
    ] = _records_to_stable_json(
        json_dataframe
    )

# 保存每个 canonical id 对应的 identity feature JSON。
identity_feature_json_by_canonical_id = {}

# 按 canonical id 分组封装 identity feature。
for (
    canonical_submission_id,
    group_dataframe,
) in submission_identity_feature_dataframe.dropna(
    subset=[
        "_canonical_submission_normalized_id",
    ]
).groupby(
    "_canonical_submission_normalized_id",
    sort=True,
):
    # 删除只用于内部关联的 temporary column。
    json_dataframe = group_dataframe.drop(
        columns=[
            "_canonical_submission_normalized_id",
        ]
    ).copy()

    # 固定 identity feature record 顺序。
    json_dataframe = json_dataframe.sort_values(
        [
            "submission_normalized_id",
            "feature_type",
            "value_hash",
        ],
        kind="mergesort",
    )

    # 保存当前重复组全部 identity feature records。
    identity_feature_json_by_canonical_id[
        int(
            canonical_submission_id
        )
    ] = _records_to_stable_json(
        json_dataframe
    )

# 给 Application candidate 加入全部 dedup run JSON。
application_candidate_dataframe[
    "submission_dedup_run_records_json"
] = application_candidate_dataframe[
    "submission_normalized_id"
].map(
    lambda submission_id: (
        dedup_run_json_by_canonical_id.get(
            int(
                submission_id
            ),
            "[]",
        )
    )
)

# 给 Application candidate 加入全部 match evidence JSON。
application_candidate_dataframe[
    "submission_match_evidence_records_json"
] = application_candidate_dataframe[
    "submission_normalized_id"
].map(
    lambda submission_id: (
        evidence_json_by_canonical_id.get(
            int(
                submission_id
            ),
            "[]",
        )
    )
)

# 给 Application candidate 加入全部 identity feature JSON。
application_candidate_dataframe[
    "submission_identity_feature_records_json"
] = application_candidate_dataframe[
    "submission_normalized_id"
].map(
    lambda submission_id: (
        identity_feature_json_by_canonical_id.get(
            int(
                submission_id
            ),
            "[]",
        )
    )
)


# ============================================================
# 第八部分：建立一对一来源 DataFrame 并合并
# ============================================================

# 复制 D1 Submission_normalized。
d1_normalized_for_merge = (
    submission_normalized_dataframe.copy()
)

# 把 D1 id 改成明确的关联键名称。
d1_normalized_for_merge = (
    d1_normalized_for_merge.rename(
        columns={
            "id": (
                "submission_normalized_id"
            ),
        }
    )
)

# source_record_number 和 raw_submission_id 需要成为最终 top-level columns，
# 其他 D1 columns 使用 submission_normalized__ 前缀。
d1_normalized_for_merge = (
    _prefix_non_key_columns(
        dataframe=d1_normalized_for_merge,
        prefix=(
            "submission_normalized__"
        ),
        key_columns=[
            "submission_normalized_id",
            "raw_submission_id",
            "source_record_number",
        ],
    )
)

# 复制 successful submission。
successful_submission_for_merge = (
    successful_submission_for_extraction_dataframe.copy()
)

# resume_text 保留为用户要求的 top-level column，
# 其他来源 columns 使用 successful_submission__ 前缀。
successful_submission_for_merge = (
    _prefix_non_key_columns(
        dataframe=(
            successful_submission_for_merge
        ),
        prefix=(
            "successful_submission__"
        ),
        key_columns=[
            "submission_normalized_id",
            "raw_submission_id",
            "resume_text",
        ],
    )
)

# 给 CV extraction 的非关联键 columns 加前缀。
resume_extraction_for_merge = (
    _prefix_non_key_columns(
        dataframe=(
            resume_extraction_consolidated_dataframe
        ),
        prefix=(
            "resume_extraction__"
        ),
        key_columns=[
            "raw_submission_id",
        ],
    )
)

# 给 cluster 的非关联键 columns 加前缀。
application_cluster_for_merge = (
    application_cluster_dataframe.rename(
        columns={
            "cluster_code": (
                "application_cluster_key"
            ),
        }
    )
)

# cluster 除关联键之外的所有字段都保留并加前缀。
application_cluster_for_merge = (
    _prefix_non_key_columns(
        dataframe=(
            application_cluster_for_merge
        ),
        prefix=(
            "application_cluster__"
        ),
        key_columns=[
            "application_cluster_key",
        ],
    )
)

# 为每个 canonical submission 只取得自己对应的一条 dedup run。
canonical_dedup_run_for_merge = (
    submission_dedup_run_dataframe.loc[
        submission_dedup_run_dataframe[
            "submission_normalized_id"
        ]
        == submission_dedup_run_dataframe[
            "canonical_submission_normalized_id"
        ]
    ].copy()
)

# 删除内部 temporary column。
canonical_dedup_run_for_merge = (
    canonical_dedup_run_for_merge.drop(
        columns=[
            "_canonical_submission_normalized_id",
        ],
        errors="ignore",
    )
)

# 给 canonical dedup run 的非关联键 columns 加前缀。
canonical_dedup_run_for_merge = (
    _prefix_non_key_columns(
        dataframe=(
            canonical_dedup_run_for_merge
        ),
        prefix=(
            "submission_dedup_run__"
        ),
        key_columns=[
            "submission_normalized_id",
        ],
    )
)

# 复制 Application candidate 作为最终合并基础。
application_dataframe = (
    application_candidate_dataframe.copy()
)

# 先保留 Application 表的十个原始字段。
application_table_columns = [
    "id",
    "person_id",
    "application_cluster_id",
    "application_status",
    "hiring_pipeline_id",
    "current_stage_id",
    "current_stage_entered_at",
    "applied_at",
    "created_at",
    "updated_at",
]

# 检查 Application candidate 是否包含 Application 的全部字段。
_require_columns(
    dataframe=application_dataframe,
    required_columns=(
        application_table_columns
    ),
    dataframe_name="application_candidate.csv",
)

# 保存 Application candidate 的所有字段副本。
application_candidate_all_columns = (
    application_candidate_dataframe.copy()
)

# 删除本 cell 后来生成的聚合 columns。
#
# 这些 columns 已经作为最终 top-level columns 保存，
# 不需要再生成一份 application_candidate__ 前缀副本。
application_candidate_all_columns = (
    application_candidate_all_columns.drop(
        columns=[
            "matched_submission_normalized_id",
            "submission_dedup_run_records_json",
            "submission_match_evidence_records_json",
            "submission_identity_feature_records_json",
        ],
        errors="ignore",
    )
)

# 给副本中除两个核心关联键外的 columns 加前缀。
application_candidate_all_columns = (
    _prefix_non_key_columns(
        dataframe=(
            application_candidate_all_columns
        ),
        prefix=(
            "application_candidate__"
        ),
        key_columns=[
            "submission_normalized_id",
            "raw_submission_id",
        ],
    )
)

# 从基础 DataFrame 中删除稍后通过带前缀副本重新加入的扩展 columns。
application_dataframe = (
    application_dataframe[
        application_table_columns
        + [
            "submission_normalized_id",
            "raw_submission_id",
            "matched_submission_normalized_id",
            "submission_dedup_run_records_json",
            "submission_match_evidence_records_json",
            "submission_identity_feature_records_json",
        ]
    ].copy()
)

# 合并 Application candidate 的全部原始 columns。
application_dataframe = (
    application_dataframe.merge(
        application_candidate_all_columns,
        on=[
            "submission_normalized_id",
            "raw_submission_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

# 合并 D1 Submission_normalized 和 source_record_number。
application_dataframe = (
    application_dataframe.merge(
        d1_normalized_for_merge,
        on=[
            "submission_normalized_id",
            "raw_submission_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

# 合并 successful submission 的 resume_text。
application_dataframe = (
    application_dataframe.merge(
        successful_submission_for_merge,
        on=[
            "submission_normalized_id",
            "raw_submission_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

# 合并 CV extraction 汇总数据。
application_dataframe = (
    application_dataframe.merge(
        resume_extraction_for_merge,
        on=[
            "raw_submission_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

# 合并 Application cluster 全部字段。
application_dataframe = (
    application_dataframe.merge(
        application_cluster_for_merge,
        left_on=(
            "application_candidate__"
            "application_cluster_key"
        ),
        right_on=(
            "application_cluster_key"
        ),
        how="left",
        validate="many_to_one",
    )
)

# 合并 canonical submission 自己的一条 dedup run 展开字段。
application_dataframe = (
    application_dataframe.merge(
        canonical_dedup_run_for_merge,
        on=[
            "submission_normalized_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 第九部分：最终验证、column 顺序和稳定 CSV 输出
# ============================================================

# 最终行数必须与 Application candidate 行数完全一致。
if len(
    application_dataframe
) != len(
    application_candidate_dataframe
):
    # 防止一对多 merge 意外增加 Application records。
    raise ValueError(
        "最终 Application 记录数与 "
        "application_candidate.csv 不一致。"
    )

# 每个 submission_normalized_id 必须唯一。
if application_dataframe[
    "submission_normalized_id"
].duplicated().any():
    # 一条 canonical submission 只能生成一条 Application。
    raise ValueError(
        "最终 application.csv 中存在重复的 "
        "submission_normalized_id。"
    )

# D1 对应 record 必须全部匹配成功。
if application_dataframe[
    "source_record_number"
].isna().any():
    # 缺失时列出没有匹配到的 normalized ids。
    missing_ids = (
        application_dataframe.loc[
            application_dataframe[
                "source_record_number"
            ].isna(),
            "submission_normalized_id",
        ]
        .astype(
            "Int64"
        )
        .tolist()
    )

    # 停止，避免输出缺少 D1 source mapping 的 Application。
    raise ValueError(
        "以下 submission_normalized_id 无法在 D1 中找到："
        f"{missing_ids}"
    )

# 统计原始数据中 resume_text 本身为空的记录。
missing_resume_text_count = int(
    application_dataframe[
        "resume_text"
    ].isna().sum()
)

# resume_text 为空不代表 CSV 关联失败。
#
# 初步清洗阶段只要求 name 和 contact_email 存在，
# 所以一部分 successful submission 的原始 resume_text 允许为空。
if missing_resume_text_count > 0:
    # 显示 warning，但保留这些 Application candidate。
    print(
        "Warning: Application candidates with empty "
        "resume_text:",
        missing_resume_text_count,
    )

# 所有数据库 ID / integer flag columns 的名称集合。
integer_column_names = {
    "id",
    "person_id",
    "application_cluster_id",
    "hiring_pipeline_id",
    "current_stage_id",
    "submission_normalized_id",
    "raw_submission_id",
    "source_record_number",
}

# 根据 column name pattern 补充 integer columns。
for column_name in application_dataframe.columns:
    # 数据库 id 和 *_id 都应该使用整数。
    if (
        column_name
        != "matched_submission_normalized_id"
        and (
            column_name.endswith(
                "_id"
            )
            or column_name.endswith(
                "_count"
            )
            or column_name.endswith(
                "_size"
            )
            or column_name.endswith(
                "__id"
            )
        )
    ):
        # 加入 integer column 集合。
        integer_column_names.add(
            column_name
        )

# 逐个转换实际存在的 integer columns。
for column_name in sorted(
    integer_column_names
):
    # 不存在于最终 DataFrame 时跳过。
    if column_name not in application_dataframe.columns:
        # 继续处理下一个 column。
        continue

    # 转换成 pandas 可空整数 Int64。
    application_dataframe[column_name] = (
        _to_nullable_integer(
            application_dataframe[
                column_name
            ]
        )
    )

# Application 表原始字段必须排在最前面。
front_columns = (
    application_table_columns
    + [
        "raw_submission_id",
        "submission_normalized_id",
        "matched_submission_normalized_id",
        "source_record_number",
        "resume_text",
    ]
)

# 取得其余扩展 columns。
remaining_columns = [
    column_name
    for column_name in application_dataframe.columns
    if column_name not in front_columns
]

# 按来源名称和 column name 稳定排序扩展 columns。
remaining_columns = sorted(
    remaining_columns
)

# 应用最终固定 column 顺序。
application_dataframe = (
    application_dataframe[
        front_columns
        + remaining_columns
    ]
)

# 按 submission_normalized_id 固定排序。
application_dataframe = (
    application_dataframe.sort_values(
        [
            "submission_normalized_id",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

# 确保输出文件夹存在。
os.makedirs(
    DEDUP_RESULT_FOLDER,
    exist_ok=True,
)

# 保存到固定路径。
#
# mode="w" 会覆盖同名文件，不会新增带时间戳的 CSV。
application_dataframe.to_csv(
    APPLICATION_OUTPUT_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
    na_rep="",
    float_format="%.6f",
    mode="w",
)

# 读取最终文件原始 bytes。
with open(
    APPLICATION_OUTPUT_CSV_PATH,
    "rb",
) as output_file:
    # 计算最终 CSV SHA-256。
    application_csv_sha256 = (
        hashlib.sha256(
            output_file.read()
        ).hexdigest()
    )

# 显示最终结果。
print(
    "\nApplication CSV successfully generated."
)

# 显示最终固定输出路径。
print(
    "Output path:",
    APPLICATION_OUTPUT_CSV_PATH,
)

# 显示最终 Application record 数量。
print(
    "Application record count:",
    len(
        application_dataframe
    ),
)

# 显示最终 column 数量。
print(
    "Application column count:",
    len(
        application_dataframe.columns
    ),
)

# 显示最终 CSV SHA-256。
print(
    "Application CSV SHA-256:",
    application_csv_sha256,
)

# 显示前十条最终 Application records。
display(
    application_dataframe.head(
        10
    )
)

# 本 cell 到这里结束。
# 没有向 D1 写入任何 Application 数据。


# Application load

In [ ]:
"""
application_importer.py（Google Colab 版，自包含）

把 application.csv 里真正有用的信息导入数据库的 Application 表，
生成真实的自增 id 之后，把结果覆盖写回一份新的 application.csv
（带上 Application.id 这一列），并把导入失败的行单独存成一份CSV。

自包含设计：
  这份文件运行/import 之后，只会往全局命名空间里放进
  ApplicationImporter 这一个名字，不会留下任何独立的全局变量
  （账号/数据库连接信息、路径、状态映射这些配置全部作为class属性，
  不是模块级别的全局变量）——不依赖之前哪个cell定义过的同名变量，
  也不会覆盖/影响之后其他cell自己定义的同名变量。

  drive.mount() 和读取 Colab 密钥这些有副作用的操作，放在真正创建
  ApplicationImporter 实例（__init__）的时候才触发，不是一
  import/运行这个cell就自动执行。

按已确认的四点处理原则：
  1. Application.id 由 D1 自动生成；插入前按现有 Application 可查询到的
     稳定业务组合查找，已经存在时复用 id，不重复 INSERT。跑完之后覆盖
     写回同一份 CSV，里面带真实的 application_id。
  2. hiring_pipeline_id / current_stage_id 统一用占位的
     "Unknown" Hiring_pipeline + "Unknown" Pipeline_stage，
     以后需要回填真实流程时再单独更新。
  3. person_id 按邮箱做"查找，不存在就新建"——同一个邮箱的多条记录
     会关联到同一个 Person，这是简化方案，不做姓名/电话相似度匹配。
  4. company_id / position_id / application_cycle_id 不信任 CSV 里
     摊平出来的列，改成拿 submission_normalized_id 现查数据库，
     保证三者来自同一条真实记录。

两处取值映射用的是合理默认，不是业务确认过的规则，需要复核：
  application_status："dedup_accepted" -> "active"
  Application_cluster.status："analysis_only" -> "active"

用法（Colab cell 里直接调用，不需要任何前置变量）：
    importer = ApplicationImporter()
    result_df = importer.import_csv()
"""

import os
import re
import time
from datetime import datetime, timezone

import pandas as pd
import requests


class ApplicationImporter:
    # ------------------------------------------------------------
    # class 级别常量：全部收在这里，不作为模块级别的全局变量，
    # 不会和其他 cell 里同名的变量互相冲突/覆盖
    # ------------------------------------------------------------

    DEFAULT_from google.colab import userdata
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
    DEFAULT_DATABASE_ID = userdata.get('D1_DATABASE_ID')
    DEFAULT_CLOUDFLARE_KEY_SECRET_NAME = "CLOUDFLARE_KEY"

    DEFAULT_APPLICATION_CSV_PATH = (
        "/content/drive/MyDrive/"
        "recruiting-pipeline/"
        "datasets/dedup_results/"
        "application_dedup_v2/"
        "application.csv"
    )

    # 取值映射，不是业务确认过的规则，是合理默认，需要复核
    APPLICATION_STATUS_MAPPING = {
        "dedup_accepted": "active",
    }
    APPLICATION_CLUSTER_STATUS_MAPPING = {
        "analysis_only": "active",
    }

    # hiring_pipeline / pipeline_stage 占位名
    UNKNOWN_HIRING_PIPELINE_NAME = "Unknown"
    UNKNOWN_PIPELINE_STAGE_NAME = "Unknown"

    # _save_failed_rows_csv 导出时挑选的关键列
    FAILED_ROW_KEY_COLUMNS = [
        "raw_submission_id",
        "submission_normalized_id",
        "submission_normalized__raw_person_name",
        "submission_normalized__raw_person_email_address",
        "submission_normalized__normalized_company_name",
        "submission_normalized__normalized_position_name",
        "application_cluster__display_name",
        "application_status",
    ]

    # ------------------------------------------------------------
    # 初始化：账号/数据库/token 不传的话，自动挂载 Drive、
    # 从 Colab 密钥里读取——这些有副作用的操作只在这里触发一次，
    # 不会因为 import 这个文件就自动执行
    # ------------------------------------------------------------
    def __init__(self, account_id: str = None, database_id: str = None, api_token: str = None):
        account_id = account_id or self.DEFAULT_ACCOUNT_ID
        database_id = database_id or self.DEFAULT_DATABASE_ID

        if api_token is None:
            from google.colab import drive, userdata
            drive.mount('/content/drive')
            api_token = userdata.get(self.DEFAULT_CLOUDFLARE_KEY_SECRET_NAME)

        self.api_url = (
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}"
            f"/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

        # 缓存，避免同一个邮箱/cycle_id/占位流程重复查询
        self._person_id_cache = {}
        self._application_cluster_id_cache = {}
        self._hiring_pipeline_id = None
        self._pipeline_stage_id = None

        self.stats = {
            "total_rows": 0,
            "inserted": 0,
            "reused": 0,
            "errors": [],
        }

    # ------------------------------------------------------------
    # 通用小工具（原来的模块级别函数，现在收成静态方法）
    # ------------------------------------------------------------
    @staticmethod
    def _now_iso() -> str:
        return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    @staticmethod
    def _slugify(text):
        lowered = (text or "").strip().lower()
        replaced = re.sub(r'[^a-z0-9_]+', '_', lowered)
        collapsed = re.sub(r'_+', '_', replaced)
        return collapsed.strip('_') or "unknown"

    @staticmethod
    def _extract_rows(result):
        return result[0].get("results", []) if result else []

    # ------------------------------------------------------------
    # 底层 D1 调用（带失败重试，只在报错时等待重试）
    # ------------------------------------------------------------
    def _execute(self, sql, params=None, timeout=30, max_retries=3, retry_delay_seconds=1.0):
        last_error = None
        for attempt in range(1, max_retries + 1):
            payload = {"sql": sql}
            if params is not None:
                payload["params"] = params
            try:
                resp = requests.post(self.api_url, headers=self.headers, json=payload, timeout=timeout)
            except requests.exceptions.RequestException as e:
                last_error = f"Network error: {e}"
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            try:
                data = resp.json()
            except ValueError:
                last_error = f"HTTP {resp.status_code}, response is not valid JSON: {resp.text[:300]}"
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            if resp.status_code != 200 or not data.get("success", False):
                last_error = data.get("errors", [])
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds)
                    continue
                return False, last_error

            return True, data.get("result", [])

        return False, last_error

    # ------------------------------------------------------------
    # 直接按 submission_normalized_id 查数据库里的真实记录，
    # 拿 company_id / position_id / application_cycle_id——
    # 不信任 CSV 里摊平出来的列（可能和数据库当前状态不一致），
    # 改成每次都现查数据库，保证用的是唯一、权威的数据源
    # ------------------------------------------------------------
    def get_submission_normalized_reference(self, submission_normalized_id):
        submission_normalized_id = int(submission_normalized_id)

        ok, result = self._execute(
            """
                SELECT "company_id", "position_id", "application_cycle_id"
                FROM "Submission_normalized"
                WHERE "id" = ?;
            """,
            [submission_normalized_id],
        )
        if not ok:
            raise RuntimeError(f"Query Submission_normalized failed: {result}")

        rows = self._extract_rows(result)
        if not rows:
            raise ValueError(
                f"submission_normalized_id={submission_normalized_id} "
                f"在数据库 Submission_normalized 表里查不到"
            )

        return rows[0]["company_id"], rows[0]["position_id"], rows[0]["application_cycle_id"]

    # ------------------------------------------------------------
    # Person：按邮箱 get-or-create
    # 局限性：同一个人如果用了两个不同邮箱投递，会被当成两个不同的人，
    # 不会合并——这是简化方案，不是最终版身份识别系统
    # ------------------------------------------------------------
    def get_or_create_person_by_email(self, email, first_name, last_name):
        key = (email or "").strip().lower()
        if not key:
            raise ValueError("email 为空，无法用邮箱作为身份识别依据")

        if key in self._person_id_cache:
            return self._person_id_cache[key]

        ok, result = self._execute(
            'SELECT "id" FROM "Person" WHERE LOWER("email") = ?;', [key]
        )
        if not ok:
            raise RuntimeError(f"Query Person failed: {result}")

        rows = self._extract_rows(result)
        if rows:
            self._person_id_cache[key] = rows[0]["id"]
            return self._person_id_cache[key]

        now = self._now_iso()
        ok, result = self._execute(
            """
                INSERT INTO "Person" (
                  "person_first_name", "person_last_name", "email",
                  "talent_pool_status", "created_at", "updated_at"
                ) VALUES (?, ?, ?, 'active', ?, ?)
                RETURNING "id";
            """,
            [first_name or "unknown", last_name or "unknown", key, now, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into Person failed: {result}")

        new_id = self._extract_rows(result)[0]["id"]
        self._person_id_cache[key] = new_id
        return new_id

    # ------------------------------------------------------------
    # Application_cluster：按 application_cycle_id get-or-create
    # （application_cycle_id 是这张表的 UNIQUE 字段）
    # ------------------------------------------------------------
    def get_or_create_application_cluster(self, application_cycle_id, display_name, raw_status):
        application_cycle_id = int(application_cycle_id)

        if application_cycle_id in self._application_cluster_id_cache:
            return self._application_cluster_id_cache[application_cycle_id]

        ok, result = self._execute(
            'SELECT "id" FROM "Application_cluster" WHERE "application_cycle_id" = ?;',
            [application_cycle_id],
        )
        if not ok:
            raise RuntimeError(f"Query Application_cluster failed: {result}")

        rows = self._extract_rows(result)
        if rows:
            self._application_cluster_id_cache[application_cycle_id] = rows[0]["id"]
            return self._application_cluster_id_cache[application_cycle_id]

        status = self.APPLICATION_CLUSTER_STATUS_MAPPING.get(raw_status, "active")
        cluster_code = self._slugify(display_name)
        now = self._now_iso()

        ok, result = self._execute(
            """
                INSERT INTO "Application_cluster" (
                  "application_cycle_id", "cluster_code", "display_name",
                  "status", "created_at"
                ) VALUES (?, ?, ?, ?, ?)
                RETURNING "id";
            """,
            [application_cycle_id, cluster_code, display_name or cluster_code, status, now],
        )
        if not ok:
            raise RuntimeError(f"Insert into Application_cluster failed: {result}")

        new_id = self._extract_rows(result)[0]["id"]
        self._application_cluster_id_cache[application_cycle_id] = new_id
        return new_id

    # ------------------------------------------------------------
    # Hiring_pipeline / Pipeline_stage：占位，全部 Application 共用同一个
    # ------------------------------------------------------------
    def get_or_create_unknown_pipeline(self):
        if self._hiring_pipeline_id is not None:
            return self._hiring_pipeline_id, self._pipeline_stage_id

        now = self._now_iso()

        ok, result = self._execute(
            'SELECT "id" FROM "Hiring_pipeline" WHERE "hiring_pipeline_name" = ?;',
            [self.UNKNOWN_HIRING_PIPELINE_NAME],
        )
        if not ok:
            raise RuntimeError(f"Query Hiring_pipeline failed: {result}")
        rows = self._extract_rows(result)

        if rows:
            hiring_pipeline_id = rows[0]["id"]
        else:
            ok, result = self._execute(
                """
                    INSERT INTO "Hiring_pipeline" (
                      "hiring_pipeline_name", "status", "created_at", "updated_at"
                    ) VALUES (?, 'active', ?, ?)
                    RETURNING "id";
                """,
                [self.UNKNOWN_HIRING_PIPELINE_NAME, now, now],
            )
            if not ok:
                raise RuntimeError(f"Insert into Hiring_pipeline failed: {result}")
            hiring_pipeline_id = self._extract_rows(result)[0]["id"]

        ok, result = self._execute(
            'SELECT "id" FROM "Pipeline_stage" WHERE "pipeline_id" = ? AND "pipeline_stage_name" = ?;',
            [hiring_pipeline_id, self.UNKNOWN_PIPELINE_STAGE_NAME],
        )
        if not ok:
            raise RuntimeError(f"Query Pipeline_stage failed: {result}")
        rows = self._extract_rows(result)

        if rows:
            pipeline_stage_id = rows[0]["id"]
        else:
            ok, result = self._execute(
                """
                    INSERT INTO "Pipeline_stage" (
                      "pipeline_id", "pipeline_stage_name", "stage_type",
                      "sequence_no", "is_initial", "is_terminal",
                      "created_at", "updated_at"
                    ) VALUES (?, ?, 'unknown', 1, 1, 0, ?, ?)
                    RETURNING "id";
                """,
                [hiring_pipeline_id, self.UNKNOWN_PIPELINE_STAGE_NAME, now, now],
            )
            if not ok:
                raise RuntimeError(f"Insert into Pipeline_stage failed: {result}")
            pipeline_stage_id = self._extract_rows(result)[0]["id"]

        self._hiring_pipeline_id = hiring_pipeline_id
        self._pipeline_stage_id = pipeline_stage_id
        return hiring_pipeline_id, pipeline_stage_id

    # ------------------------------------------------------------
    # Application 幂等查找
    # ------------------------------------------------------------
    def find_existing_application_id(
        self,
        person_id,
        application_cluster_id,
        company_id,
        position_id,
        applied_at,
    ):
        """
        使用 Application 表当前已经保存的稳定业务字段查找已有记录。

        Application 表没有 submission_normalized_id，因此不能直接使用
        submission_normalized_id 建立数据库级唯一键。当前在不修改 schema 的
        前提下，使用 person + cluster + company + position + applied_at 作为
        顺序重复运行的幂等判断组合。
        """

        # pandas NaN 不能作为 SQL NULL 正确比较，先转换成 None。
        applied_at_value = (
            None
            if pd.isna(applied_at)
            else str(applied_at)
        )

        ok, result = self._execute(
            """
                SELECT "id"
                FROM "Application"
                WHERE "person_id" = ?
                  AND "application_cluster_id" = ?
                  AND "company_id" = ?
                  AND "position_id" = ?
                  AND (
                    "applied_at" = ?
                    OR ("applied_at" IS NULL AND ? IS NULL)
                  )
                ORDER BY "id"
                LIMIT 2;
            """,
            [
                int(person_id),
                int(application_cluster_id),
                int(company_id),
                int(position_id),
                applied_at_value,
                applied_at_value,
            ],
        )
        if not ok:
            raise RuntimeError(
                f"Query existing Application failed: {result}"
            )

        rows = self._extract_rows(result)

        # 相同业务组合已有多行，说明数据库已经存在历史重复；
        # 此时不静默选择，避免继续掩盖数据问题。
        if len(rows) > 1:
            raise RuntimeError(
                "数据库中存在多个相同业务组合的 Application："
                f"person_id={person_id}, "
                f"application_cluster_id={application_cluster_id}, "
                f"company_id={company_id}, "
                f"position_id={position_id}, "
                f"applied_at={applied_at_value!r}"
            )

        # 找到一条时复用其真实 Application.id。
        if rows:
            return int(rows[0]["id"])

        # 没有找到时由调用方执行原来的 INSERT。
        return None

    # ------------------------------------------------------------
    # Application：不存在时执行原来的 INSERT，由 D1 自动生成 id
    # ------------------------------------------------------------
    def insert_application_row(self, person_id, application_cluster_id, company_id,
                                position_id, application_status, hiring_pipeline_id,
                                current_stage_id, applied_at):
        now = self._now_iso()
        sql = """
            INSERT INTO "Application" (
              "person_id", "application_cluster_id", "company_id", "position_id",
              "application_status", "hiring_pipeline_id", "current_stage_id",
              "current_stage_entered_at", "applied_at", "created_at", "updated_at"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            RETURNING "id";
        """
        params = [
            int(person_id), int(application_cluster_id), int(company_id), int(position_id),
            application_status, int(hiring_pipeline_id), current_stage_id,
            now, applied_at, now, now,
        ]
        ok, result = self._execute(sql, params)
        if not ok:
            raise RuntimeError(f"Insert into Application failed: {result}")
        return self._extract_rows(result)[0]["id"]

    # ------------------------------------------------------------
    # 失败行单独导出成CSV
    # ------------------------------------------------------------
    def _save_failed_rows_csv(self, df: pd.DataFrame, csv_path: str):
        """
        把这次导入失败的行（id 仍是空的那些），连同具体报错原因，
        单独存成一份CSV。只挑几个方便人工核对身份的关键列，
        不是把全部列都带过去。
        """
        if not self.stats["errors"]:
            print("No failed rows this run.")
            return None

        failed_row_indices = [i for i, _ in self.stats["errors"]]
        error_message_by_index = dict(self.stats["errors"])

        existing_key_columns = [c for c in self.FAILED_ROW_KEY_COLUMNS if c in df.columns]

        failed_df = df.loc[failed_row_indices, existing_key_columns].copy()
        failed_df["import_error"] = [
            error_message_by_index[i] for i in failed_row_indices
        ]

        failed_csv_path = csv_path.rsplit(".", 1)[0] + "_failed.csv"
        failed_df.to_csv(failed_csv_path, index=False, encoding="utf-8-sig")

        print(f"Saved {len(failed_df)} failed rows to: {failed_csv_path}")
        return failed_csv_path

    # ------------------------------------------------------------
    # 批量导入 + 覆盖写回CSV + 导出失败行CSV
    # ------------------------------------------------------------
    def import_csv(self, csv_path: str = None, limit: int = None, verbose: bool = True):
        csv_path = csv_path or self.DEFAULT_APPLICATION_CSV_PATH

        # 在读取前明确验证文件存在，避免 pandas 只返回较难定位的底层错误。
        if not os.path.isfile(csv_path):
            raise FileNotFoundError(
                "找不到准备导入 Application 表的 CSV："
                f"{csv_path}"
            )

        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        if limit:
            df = df.head(limit)

        # 每次调用 import_csv 都重置本次统计，避免同一个 importer 实例
        # 重复调用时累计上一轮的 inserted/reused/errors。
        self.stats = {
            "total_rows": len(df),
            "inserted": 0,
            "reused": 0,
            "errors": [],
        }

        # 只有第一次调用需要建立/查找占位流程，之后全部复用
        hiring_pipeline_id, pipeline_stage_id = self.get_or_create_unknown_pipeline()

        application_ids = []

        for i, row in df.iterrows():
            try:
                person_id = self.get_or_create_person_by_email(
                    row.get("submission_normalized__raw_person_email_address"),
                    row.get("submission_normalized__normalized_first_name"),
                    row.get("submission_normalized__normalized_last_name"),
                )

                # 不信任CSV里摊平的 submission_normalized__company_id /
                # submission_normalized__position_id / application_cluster__
                # application_cycle_id 这几列——改成拿 submission_normalized_id
                # 现查数据库，保证 company_id / position_id / application_cycle_id
                # 三者来自同一条真实记录，不会互相对不上
                company_id, position_id, application_cycle_id = (
                    self.get_submission_normalized_reference(
                        row.get("submission_normalized_id")
                    )
                )

                application_cluster_id = self.get_or_create_application_cluster(
                    application_cycle_id,
                    row.get("application_cluster__display_name"),
                    row.get("application_cluster__status"),
                )

                application_status = self.APPLICATION_STATUS_MAPPING.get(
                    row.get("application_status"), "active"
                )

                # 在 INSERT 前查找相同业务组合的 Application。
                # 已存在时直接复用 id，使同一 CSV 顺序重复运行保持幂等。
                existing_application_id = (
                    self.find_existing_application_id(
                        person_id=person_id,
                        application_cluster_id=application_cluster_id,
                        company_id=company_id,
                        position_id=position_id,
                        applied_at=row.get("applied_at"),
                    )
                )

                if existing_application_id is not None:
                    new_application_id = existing_application_id
                    self.stats["reused"] += 1
                else:
                    # 不存在时保持原来的 INSERT 逻辑。
                    new_application_id = self.insert_application_row(
                        person_id=person_id,
                        application_cluster_id=application_cluster_id,
                        company_id=company_id,
                        position_id=position_id,
                        application_status=application_status,
                        hiring_pipeline_id=hiring_pipeline_id,
                        current_stage_id=pipeline_stage_id,
                        applied_at=row.get("applied_at"),
                    )
                    self.stats["inserted"] += 1

                application_ids.append(new_application_id)

            except Exception as e:
                application_ids.append(None)
                self.stats["errors"].append((i, str(e)))
                print(f"  row {i} failed: {e}")

            if verbose and (i + 1) % 100 == 0:
                print(f"  processed {i + 1}/{len(df)} rows...")

        df["id"] = application_ids

        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        failed_csv_path = self._save_failed_rows_csv(df, csv_path)

        print("\n" + "=" * 60)
        print("Application import summary")
        print("=" * 60)
        print(f"  total rows : {self.stats['total_rows']}")
        print(f"  inserted   : {self.stats['inserted']}")
        print(f"  reused     : {self.stats['reused']}")
        print(f"  errors     : {len(self.stats['errors'])}")
        print(f"  Overwritten CSV with real ids: {csv_path}")
        if failed_csv_path:
            print(f"  Failed rows saved to: {failed_csv_path}")

        return df




In [ ]:
# ============================================================
# Test section (remove after confirming everything works)
# ============================================================

importer = ApplicationImporter()
# result_df = importer.import_csv(limit=20)
#
result_df = importer.import_csv()


# Candidate & Person 层导入代码

## Candidate & Person 层数据导入前对数据库做的一些小修改 （已经运行过一次了，就算是复制的Google Colab文件副本也坚决不能再重复运行！！！！！！）

In [ ]:
# ============================================================
# Google Colab 独立 Cell：
# 查询 Candidate / Person 数据切分相关表的当前记录数量。
#
# 重要：
# 1. 本 Cell 只执行 SELECT 查询。
# 2. 本 Cell 不执行 INSERT、UPDATE、DELETE 或 ALTER TABLE。
# 3. 本 Cell 不依赖之前其他 Cell 中的 loader 或 class。
# ============================================================


# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# pandas 用于把查询结果整理成 DataFrame。
import pandas as pd

# userdata 用于读取保存在 Google Colab Secrets 中的 API Token。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# 当前使用的 D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Google Colab Secrets 中读取 Cloudflare API Token。
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# 如果没有成功读取 API Token，就立即停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Google Colab 左侧 Secrets 中添加该密钥，"
        "并允许当前 notebook 访问。"
    )


# 拼接 Cloudflare D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# 创建 Cloudflare API 请求使用的 headers。
D1_HEADERS = {
    # 使用保存在 Colab Secrets 中的 Bearer Token。
    "Authorization": f"Bearer {API_TOKEN}",

    # 告诉 Cloudflare 请求正文使用 JSON 格式。
    "Content-Type": "application/json",
}


# ============================================================
# 需要查询记录数量的表
# ============================================================

# 这些表是后续 Application、Candidate 和 Person 数据切分会涉及的表。
TABLE_NAMES = [
    "Application",
    "Candidate_snapshot",
    "Person",
    "Person_Name",
    "Person_Contact",
    "Person_Link",
    "Person_identifier_claim",
    "Person_Skill",
    "Person_Position",
    "Person_Education",
    "Person_Project",
    "Candidate_Skill",
    "Candidate_Position",
    "Candidate_Education",
    "Education",
]


# ============================================================
# 逐张表执行只读 COUNT 查询
# ============================================================

# 用于保存每张表的查询结果。
table_count_results = []


# 逐个处理 TABLE_NAMES 中的表名。
for table_name in TABLE_NAMES:

    # 表名来自上面写死的可信列表，不来自外部用户输入。
    #
    # SQLite 不支持使用 ? 参数绑定表名，
    # 所以这里直接把可信表名放进 SQL。
    sql = (
        f'SELECT COUNT(*) AS "record_count" '
        f'FROM "{table_name}";'
    )

    # 构建发送给 Cloudflare D1 API 的 JSON。
    request_payload = {
        # 需要执行的只读 SQL。
        "sql": sql,

        # 当前 SQL 没有 ? 参数，所以使用空参数列表。
        "params": [],
    }

    try:
        # 向 Cloudflare D1 API 发送查询请求。
        response = requests.post(
            D1_API_URL,
            headers=D1_HEADERS,
            json=request_payload,
            timeout=30,
        )

    except requests.exceptions.RequestException as error:
        # 网络请求失败时，不中断其他表的查询。
        #
        # 把当前表的错误记录下来。
        table_count_results.append(
            {
                "table_name": table_name,
                "record_count": None,
                "query_status": "network_error",
                "query_error": str(error),
            }
        )

        # 继续查询下一张表。
        continue


    try:
        # 尝试把 Cloudflare 响应解析成 JSON。
        response_payload = response.json()

    except ValueError:
        # Cloudflare 返回的内容不是有效 JSON 时，
        # 保存 HTTP 状态码和部分原始响应。
        table_count_results.append(
            {
                "table_name": table_name,
                "record_count": None,
                "query_status": "invalid_json_response",
                "query_error": (
                    f"HTTP {response.status_code}: "
                    f"{response.text[:300]}"
                ),
            }
        )

        # 继续查询下一张表。
        continue


    # 判断 HTTP 请求和 D1 SQL 查询是否成功。
    if (
        response.status_code != 200
        or not response_payload.get("success", False)
    ):
        # 查询失败时保存 Cloudflare 返回的错误。
        table_count_results.append(
            {
                "table_name": table_name,
                "record_count": None,
                "query_status": "d1_query_failed",
                "query_error": str(
                    response_payload.get(
                        "errors",
                        response_payload,
                    )
                ),
            }
        )

        # 继续查询下一张表。
        continue


    # 取得 D1 查询的 result 数组。
    d1_result = response_payload.get(
        "result",
        [],
    )

    # 取得 SELECT 查询返回的数据行。
    result_rows = (
        d1_result[0].get("results", [])
        if d1_result
        else []
    )

    # 如果没有返回预期结果，就保存异常状态。
    if not result_rows:
        table_count_results.append(
            {
                "table_name": table_name,
                "record_count": None,
                "query_status": "no_result_returned",
                "query_error": (
                    "D1 查询成功，但没有返回 COUNT 结果。"
                ),
            }
        )

        # 继续查询下一张表。
        continue


    # 取得当前表的记录数量。
    #
    # 显式转换成标准 Python int，
    # 避免后续出现 NumPy int64 无法 JSON 序列化的问题。
    record_count = int(
        result_rows[0]["record_count"]
    )

    # 保存当前表的成功查询结果。
    table_count_results.append(
        {
            "table_name": table_name,
            "record_count": record_count,
            "query_status": "success",
            "query_error": None,
        }
    )


# ============================================================
# 生成并显示查询结果 DataFrame
# ============================================================

# 把所有表的查询结果转换成 pandas DataFrame。
table_count_dataframe = pd.DataFrame(
    table_count_results
)


# 按照最开始 TABLE_NAMES 中的顺序排列结果。
table_count_dataframe["table_name"] = pd.Categorical(
    table_count_dataframe["table_name"],
    categories=TABLE_NAMES,
    ordered=True,
)


# 根据指定的表顺序排序。
table_count_dataframe = (
    table_count_dataframe
    .sort_values("table_name")
    .reset_index(drop=True)
)


# 将 table_name 恢复为普通 string 类型。
table_count_dataframe["table_name"] = (
    table_count_dataframe["table_name"]
    .astype("string")
)


# 显示完整的记录数查询结果。
display(
    table_count_dataframe
)


# ============================================================
# 查询结果检查
# ============================================================

# 筛选查询失败的表。
failed_table_queries = (
    table_count_dataframe[
        table_count_dataframe["query_status"]
        != "success"
    ]
    .copy()
)


# 如果存在查询失败的表，就显示错误结果。
if not failed_table_queries.empty:
    print(
        "\n有表查询失败，请查看下面的错误："
    )

    display(
        failed_table_queries
    )

else:
    # 所有表都查询成功时显示确认信息。
    print(
        "\nAll table count queries completed successfully."
    )

                 table_name  record_count query_status query_error
0               Application          1049      success        None
1        Candidate_snapshot          1039      success        None
2                    Person          1030      success        None
3               Person_Name          2044      success        None
4            Person_Contact          1908      success        None
5               Person_Link           253      success        None
6   Person_identifier_claim             0      success        None
7              Person_Skill          3144      success        None
8           Person_Position          2042      success        None
9          Person_Education          1311      success        None
10           Person_Project           965      success        None
11          Candidate_Skill          3170      success        None
12       Candidate_Position          2042      success        None
13      Candidate_Education          1332      success        


All table count queries completed successfully.


In [ ]:
# ============================================================
# Google Colab 独立 Cell：
# 检查 Candidate_snapshot 旧字段的数据库 schema 依赖。
#
# 本 Cell 只执行 SELECT，不修改数据库。
# ============================================================


# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于显示查询结果。
import pandas as pd

# userdata 用于读取 Google Colab Secrets。
from google.colab import userdata


# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# 没有读取到 Token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
    )


# 生成 D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# 生成 D1 API headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 明确显示 Cell 已经开始执行。
print(
    "Starting Candidate_snapshot schema dependency check..."
)


# 查询所有在建表、索引、View 或 Trigger SQL 中
# 出现旧字段名称的数据库对象。
schema_dependency_sql = """
    SELECT
      "type",
      "name",
      "tbl_name",
      "sql"
    FROM "sqlite_master"
    WHERE LOWER(COALESCE("sql", ''))
          LIKE '%availability_date%'
       OR LOWER(COALESCE("sql", ''))
          LIKE '%employment_preference%'
    ORDER BY
      "type",
      "name";
"""


# 显示即将发送 D1 请求。
print(
    "Sending read-only query to D1..."
)


# 向 Cloudflare D1 发送只读查询。
response = requests.post(
    D1_API_URL,
    headers=D1_HEADERS,
    json={
        "sql": schema_dependency_sql,
        "params": [],
    },
    timeout=30,
)


# 显示 HTTP 状态码。
print(
    "Cloudflare HTTP status:",
    response.status_code,
)


# 尝试解析 Cloudflare JSON 响应。
try:
    payload = response.json()

except ValueError:
    raise RuntimeError(
        "Cloudflare 返回了非 JSON 响应："
        f"\nHTTP {response.status_code}"
        f"\n{response.text[:500]}"
    )


# 显示 Cloudflare D1 的 success 状态。
print(
    "D1 query success:",
    payload.get("success"),
)


# 如果 HTTP 请求或 D1 查询失败，就停止并显示错误。
if (
    response.status_code != 200
    or not payload.get("success", False)
):
    raise RuntimeError(
        "D1 查询失败："
        f"{payload.get('errors', payload)}"
    )


# 取得 Cloudflare D1 result 数组。
d1_results = payload.get(
    "result",
    [],
)


# 从第一个查询结果中取得数据行。
dependency_rows = (
    d1_results[0].get(
        "results",
        [],
    )
    if d1_results
    else []
)


# 明确显示查询返回的记录数量。
print(
    "Schema dependency record count:",
    len(dependency_rows),
)


# 如果没有返回任何 schema 对象。
if len(dependency_rows) == 0:
    print(
        "没有任何数据库 schema 对象引用 "
        "availability_date 或 employment_preference。"
    )

else:
    # 把返回结果转换成 DataFrame。
    schema_dependency_dataframe = pd.DataFrame(
        dependency_rows
    )

    # 显示查询结果。
    display(
        schema_dependency_dataframe
    )


# 明确显示 Cell 已经执行完成。
print(
    "Schema dependency check completed."
)

In [ ]:
# ============================================================
# Google Colab 独立 Cell：
# 给 Application 增加三个 nullable TEXT 字段。
#
# 本 Cell 会先检查字段是否已经存在，
# 因此可以安全地重复运行。
# ============================================================


# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于显示 Application 的字段结构。
import pandas as pd

# userdata 用于读取 Colab Secrets。
from google.colab import userdata


# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# 没有读取到 Token 时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# 生成 D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# 生成 D1 API headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 定义执行一条 D1 SQL 的 function。
def execute_d1_sql(sql, params=None):

    # 向 Cloudflare D1 发送 SQL。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 尝试解析 Cloudflare JSON 响应。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            "Cloudflare 返回了非 JSON 响应："
            f"HTTP {response.status_code} "
            f"{response.text[:500]}"
        )

    # SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            "D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result 数组。
    return payload.get("result", [])


# ============================================================
# 第一步：读取 Application 当前字段
# ============================================================

# 查询 Application 当前字段结构。
before_result = execute_d1_sql(
    'PRAGMA table_info("Application");'
)


# 取得字段记录。
before_rows = (
    before_result[0].get("results", [])
    if before_result
    else []
)


# 生成当前已存在的字段名称集合。
existing_columns = {
    row["name"]
    for row in before_rows
}


# 显示修改前字段。
print(
    "Application columns before migration:"
)

print(
    sorted(existing_columns)
)


# ============================================================
# 第二步：定义需要增加的字段
# ============================================================

# 每个 tuple 保存：
# 1. 新字段名称
# 2. 对应的 ALTER TABLE SQL
columns_to_add = [
    (
        "applied_company_name",
        'ALTER TABLE "Application" '
        'ADD COLUMN "applied_company_name" TEXT;',
    ),
    (
        "applied_position_name",
        'ALTER TABLE "Application" '
        'ADD COLUMN "applied_position_name" TEXT;',
    ),
    (
        "application_cluster_display_name",
        'ALTER TABLE "Application" '
        'ADD COLUMN "application_cluster_display_name" TEXT;',
    ),
]


# 用于保存本次新增字段。
added_columns = []


# 用于保存已经存在而被跳过的字段。
skipped_columns = []


# ============================================================
# 第三步：逐个增加缺失字段
# ============================================================

# 逐个处理需要增加的字段。
for column_name, alter_sql in columns_to_add:

    # 字段已经存在时跳过。
    if column_name in existing_columns:

        # 保存跳过字段。
        skipped_columns.append(
            column_name
        )

        # 显示跳过结果。
        print(
            f"Skipped existing column: {column_name}"
        )

        # 继续处理下一个字段。
        continue

    # 字段不存在时执行 ALTER TABLE。
    execute_d1_sql(
        alter_sql
    )

    # 保存新增字段。
    added_columns.append(
        column_name
    )

    # 显示新增结果。
    print(
        f"Added Application column: {column_name}"
    )


# ============================================================
# 第四步：验证修改后的字段结构
# ============================================================

# 重新读取 Application 字段。
after_result = execute_d1_sql(
    'PRAGMA table_info("Application");'
)


# 取得修改后的字段记录。
after_rows = (
    after_result[0].get("results", [])
    if after_result
    else []
)


# 生成修改后的字段名称集合。
after_column_names = {
    row["name"]
    for row in after_rows
}


# 定义三个必须存在的新字段。
required_columns = {
    "applied_company_name",
    "applied_position_name",
    "application_cluster_display_name",
}


# 计算修改后仍然缺少的字段。
missing_columns = (
    required_columns
    - after_column_names
)


# 如果仍有缺少字段，就停止。
if missing_columns:
    raise RuntimeError(
        "Application 修改后仍缺少字段："
        f"{sorted(missing_columns)}"
    )


# ============================================================
# 第五步：验证 Application 已有记录
# ============================================================

# 这条查询不会修改数据。
#
# 它只验证 Application 总记录数，
# 以及三个新字段当前为 NULL 的数量。
verification_sql = (
    'SELECT '
    'COUNT(*) AS "application_record_count", '
    'SUM(CASE WHEN "applied_company_name" IS NULL '
    'THEN 1 ELSE 0 END) '
    'AS "null_applied_company_name_count", '
    'SUM(CASE WHEN "applied_position_name" IS NULL '
    'THEN 1 ELSE 0 END) '
    'AS "null_applied_position_name_count", '
    'SUM(CASE WHEN "application_cluster_display_name" IS NULL '
    'THEN 1 ELSE 0 END) '
    'AS "null_application_cluster_display_name_count" '
    'FROM "Application";'
)


# 执行验证查询。
verification_result = execute_d1_sql(
    verification_sql
)


# 取得验证结果。
verification = (
    verification_result[0]["results"][0]
)


# ============================================================
# 第六步：显示最终结果
# ============================================================

# 显示本次新增字段。
print(
    "\nAdded columns:",
    added_columns,
)


# 显示本次跳过字段。
print(
    "Skipped existing columns:",
    skipped_columns,
)


# 将修改后的 Application schema 转成 DataFrame。
application_schema_dataframe = pd.DataFrame(
    after_rows
)


# 显示修改后的字段结构。
display(
    application_schema_dataframe
)


# 显示记录数验证结果。
print(
    "Application data verification:"
)

print(
    verification
)


# 显示执行完成。
print(
    "\nApplication schema migration completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 修改 Candidate_snapshot table。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell。
# 2. 本 Cell 会独立读取 Cloudflare API Token。
# 3. 本 Cell 会独立建立 D1 连接。
# 4. Candidate_snapshot 当前必须是空表。
# 5. 本 Cell 会先检查字段状态，支持重复运行。
# ============================================================


# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于显示修改后的字段结构。
import pandas as pd

# userdata 用于读取 Google Colab Secrets。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Google Colab Secrets 中读取 Cloudflare API Token。
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# 没有读取到 Token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Google Colab 左侧 Secrets 中添加密钥，"
        "并允许当前 notebook 访问。"
    )


# 生成 Cloudflare D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# 生成 D1 API 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# ============================================================
# 定义本 Cell 自己使用的 D1 SQL function
# ============================================================

def execute_candidate_schema_sql(
    sql,
    params=None,
):
    # 向 Cloudflare D1 发送 SQL。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 尝试解析 Cloudflare JSON 响应。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            "Cloudflare 返回了非 JSON 响应："
            f"HTTP {response.status_code} "
            f"{response.text[:500]}"
        )

    # HTTP 或 D1 SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            "D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result 数组。
    return payload.get(
        "result",
        [],
    )


# ============================================================
# 第一步：确认 Candidate_snapshot 是空表
# ============================================================

# 查询 Candidate_snapshot 当前记录数。
candidate_count_result = (
    execute_candidate_schema_sql(
        'SELECT COUNT(*) AS "record_count" '
        'FROM "Candidate_snapshot";'
    )
)


# 取得当前记录数。
candidate_record_count = int(
    candidate_count_result[0][
        "results"
    ][0]["record_count"]
)


# 当前不是空表时停止。
if candidate_record_count != 0:
    raise ValueError(
        "Candidate_snapshot 当前不是空表。"
        f"实际记录数：{candidate_record_count}。"
        "为了避免影响已有数据，本次 schema 修改已停止。"
    )


# 显示确认结果。
print(
    "Candidate_snapshot record count before migration:",
    candidate_record_count,
)


# ============================================================
# 第二步：读取修改前的 Candidate_snapshot schema
# ============================================================

# 查询 Candidate_snapshot 当前字段。
before_result = execute_candidate_schema_sql(
    'PRAGMA table_info("Candidate_snapshot");'
)


# 取得当前字段记录。
before_rows = (
    before_result[0].get(
        "results",
        [],
    )
    if before_result
    else []
)


# 生成当前字段名称集合。
candidate_columns = {
    row["name"]
    for row in before_rows
}


# 显示修改前字段。
print(
    "\nCandidate_snapshot columns before migration:"
)

print(
    sorted(candidate_columns)
)


# ============================================================
# 第三步：重命名 employment_preference
# ============================================================

# 旧字段存在、新字段不存在时执行重命名。
if (
    "employment_preference" in candidate_columns
    and "work_type" not in candidate_columns
):
    execute_candidate_schema_sql(
        'ALTER TABLE "Candidate_snapshot" '
        'RENAME COLUMN "employment_preference" '
        'TO "work_type";'
    )

    # 更新本 Cell 内存中的字段集合。
    candidate_columns.remove(
        "employment_preference"
    )

    candidate_columns.add(
        "work_type"
    )

    # 显示重命名结果。
    print(
        "Renamed employment_preference to work_type."
    )


# 新字段已经存在、旧字段不存在时跳过。
elif (
    "work_type" in candidate_columns
    and "employment_preference" not in candidate_columns
):
    print(
        "Skipped rename: work_type already exists."
    )


# 两个字段同时存在时停止。
elif (
    "employment_preference" in candidate_columns
    and "work_type" in candidate_columns
):
    raise RuntimeError(
        "Candidate_snapshot 同时存在 "
        "employment_preference 和 work_type。"
    )


# 两个字段都不存在时停止。
else:
    raise RuntimeError(
        "Candidate_snapshot 中既没有 "
        "employment_preference，也没有 work_type。"
    )


# ============================================================
# 第四步：重命名 availability_date
# ============================================================

# 旧字段存在、新字段不存在时执行重命名。
if (
    "availability_date" in candidate_columns
    and "available_start_date" not in candidate_columns
):
    execute_candidate_schema_sql(
        'ALTER TABLE "Candidate_snapshot" '
        'RENAME COLUMN "availability_date" '
        'TO "available_start_date";'
    )

    # 更新本 Cell 内存中的字段集合。
    candidate_columns.remove(
        "availability_date"
    )

    candidate_columns.add(
        "available_start_date"
    )

    # 显示重命名结果。
    print(
        "Renamed availability_date "
        "to available_start_date."
    )


# 新字段已经存在、旧字段不存在时跳过。
elif (
    "available_start_date" in candidate_columns
    and "availability_date" not in candidate_columns
):
    print(
        "Skipped rename: "
        "available_start_date already exists."
    )


# 两个字段同时存在时停止。
elif (
    "availability_date" in candidate_columns
    and "available_start_date" in candidate_columns
):
    raise RuntimeError(
        "Candidate_snapshot 同时存在 "
        "availability_date 和 available_start_date。"
    )


# 两个字段都不存在时停止。
else:
    raise RuntimeError(
        "Candidate_snapshot 中既没有 "
        "availability_date，也没有 available_start_date。"
    )


# ============================================================
# 第五步：定义需要增加的 nullable TEXT 字段
# ============================================================

# 这些字段全部允许存 SQL NULL。
candidate_columns_to_add = [
    "expected_end_date",
    "work_duration",
    "raw_person_name",
    "normalized_person_name",
    "normalized_first_name",
    "normalized_middle_name",
    "normalized_last_name",
    "raw_email_address",
    "normalized_email_address",
    "raw_phone",
    "normalized_phone",
    "raw_linkedin_url",
    "normalized_linkedin_url",
    "raw_github_url",
    "normalized_github_url",
    "raw_resume_text",
]


# 保存本次新增字段。
added_candidate_columns = []


# 保存已经存在而跳过的字段。
skipped_candidate_columns = []


# ============================================================
# 第六步：逐个增加缺失字段
# ============================================================

# 逐个处理字段。
for column_name in candidate_columns_to_add:

    # 字段已经存在时跳过。
    if column_name in candidate_columns:

        skipped_candidate_columns.append(
            column_name
        )

        print(
            f"Skipped existing column: {column_name}"
        )

        continue

    # 字段名来自上面写死的可信列表。
    alter_sql = (
        'ALTER TABLE "Candidate_snapshot" '
        f'ADD COLUMN "{column_name}" TEXT;'
    )

    # 执行字段新增。
    execute_candidate_schema_sql(
        alter_sql
    )

    # 更新内存字段集合。
    candidate_columns.add(
        column_name
    )

    # 保存新增字段名称。
    added_candidate_columns.append(
        column_name
    )

    # 显示新增成功信息。
    print(
        "Added Candidate_snapshot column:",
        column_name,
    )


# ============================================================
# 第七步：验证最终字段结构
# ============================================================

# 重新查询 Candidate_snapshot 字段。
after_result = execute_candidate_schema_sql(
    'PRAGMA table_info("Candidate_snapshot");'
)


# 取得修改后字段记录。
after_rows = (
    after_result[0].get(
        "results",
        [],
    )
    if after_result
    else []
)


# 生成修改后的字段名称集合。
after_column_names = {
    row["name"]
    for row in after_rows
}


# 定义必须存在的字段。
required_candidate_columns = (
    {
        "work_type",
        "available_start_date",
    }
    | set(candidate_columns_to_add)
)


# 计算仍然缺少的字段。
missing_candidate_columns = (
    required_candidate_columns
    - after_column_names
)


# 仍有缺少字段时停止。
if missing_candidate_columns:
    raise RuntimeError(
        "Candidate_snapshot 修改后仍缺少字段："
        f"{sorted(missing_candidate_columns)}"
    )


# 检查旧字段是否仍然存在。
old_columns_still_present = (
    {
        "employment_preference",
        "availability_date",
    }
    & after_column_names
)


# 旧字段仍然存在时停止。
if old_columns_still_present:
    raise RuntimeError(
        "Candidate_snapshot 修改后仍存在旧字段："
        f"{sorted(old_columns_still_present)}"
    )


# ============================================================
# 第八步：验证记录数量没有变化
# ============================================================

# 再次查询 Candidate_snapshot 记录数。
final_count_result = execute_candidate_schema_sql(
    'SELECT COUNT(*) AS "record_count" '
    'FROM "Candidate_snapshot";'
)


# 取得修改后的记录数。
final_candidate_record_count = int(
    final_count_result[0][
        "results"
    ][0]["record_count"]
)


# 记录数不是 0 时停止。
if final_candidate_record_count != 0:
    raise RuntimeError(
        "Candidate_snapshot schema 修改期间"
        "记录数发生变化。"
    )


# ============================================================
# 第九步：显示最终结果
# ============================================================

# 把最终 schema 转换成 DataFrame。
candidate_schema_dataframe = pd.DataFrame(
    after_rows
)


# 显示本次新增字段。
print(
    "\nAdded Candidate_snapshot columns:"
)

print(
    added_candidate_columns
)


# 显示本次跳过字段。
print(
    "\nSkipped existing Candidate_snapshot columns:"
)

print(
    skipped_candidate_columns
)


# 显示最终字段结构。
display(
    candidate_schema_dataframe
)


# 显示最终记录数。
print(
    "Candidate_snapshot record count:",
    final_candidate_record_count,
)


# 显示迁移完成。
print(
    "\nCandidate_snapshot schema migration "
    "completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 给 Person 子表增加 nullable source_candidate_id 外键字段。
#
# 每张表同时增加 source_candidate_id 普通查询索引。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell。
# 2. source_candidate_id 允许存 SQL NULL。
# 3. 已有记录不会被删除或修改。
# 4. 字段存在时跳过。
# 5. 索引使用 IF NOT EXISTS，可以重复运行。
# ============================================================


# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于显示最终验证结果。
import pandas as pd

# userdata 用于读取 Google Colab Secrets。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# 没有成功读取 Token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
    )


# 生成 D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# 生成 D1 API headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# ============================================================
# 定义本 Cell 自己使用的 D1 SQL function
# ============================================================

def execute_person_source_sql(
    sql,
    params=None,
):
    # 向 Cloudflare D1 发送 SQL。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 尝试解析 Cloudflare JSON 响应。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            "Cloudflare 返回了非 JSON 响应："
            f"HTTP {response.status_code} "
            f"{response.text[:500]}"
        )

    # HTTP 或 D1 SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            "D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result 数组。
    return payload.get(
        "result",
        [],
    )


# ============================================================
# 定义需要修改的 Person 子表
# ============================================================

# key 是可信的固定表名。
#
# value 是对应的索引名称。
person_child_tables = {
    "Person_Name": (
        "idx_person_name_source_candidate_id"
    ),
    "Person_Contact": (
        "idx_person_contact_source_candidate_id"
    ),
    "Person_Link": (
        "idx_person_link_source_candidate_id"
    ),
    "Person_identifier_claim": (
        "idx_person_identifier_claim_source_candidate_id"
    ),
    "Person_Skill": (
        "idx_person_skill_source_candidate_id"
    ),
    "Person_Position": (
        "idx_person_position_source_candidate_id"
    ),
    "Person_Education": (
        "idx_person_education_source_candidate_id"
    ),
    "Person_Project": (
        "idx_person_project_source_candidate_id"
    ),
}


# 用于保存每张表的处理和验证结果。
migration_results = []


# ============================================================
# 逐张表增加字段和索引
# ============================================================

# 逐张处理 Person 子表。
for (
    table_name,
    index_name,
) in person_child_tables.items():

    # --------------------------------------------------------
    # 读取当前记录数
    # --------------------------------------------------------

    # 查询当前表记录数量。
    count_result = execute_person_source_sql(
        f'SELECT COUNT(*) AS "record_count" '
        f'FROM "{table_name}";'
    )

    # 取得记录数量。
    record_count = int(
        count_result[0][
            "results"
        ][0]["record_count"]
    )

    # --------------------------------------------------------
    # 读取当前字段
    # --------------------------------------------------------

    # 查询当前表字段结构。
    table_info_result = execute_person_source_sql(
        f'PRAGMA table_info("{table_name}");'
    )

    # 取得字段记录。
    table_info_rows = (
        table_info_result[0].get(
            "results",
            [],
        )
        if table_info_result
        else []
    )

    # 生成当前字段名称集合。
    existing_columns = {
        row["name"]
        for row in table_info_rows
    }

    # --------------------------------------------------------
    # 增加 nullable source_candidate_id
    # --------------------------------------------------------

    # 字段不存在时才执行 ALTER TABLE。
    if "source_candidate_id" not in existing_columns:

        # 新字段允许为 NULL。
        #
        # 新字段引用 Candidate_snapshot.id。
        alter_sql = (
            f'ALTER TABLE "{table_name}" '
            f'ADD COLUMN "source_candidate_id" INTEGER '
            f'REFERENCES "Candidate_snapshot"("id");'
        )

        # 执行字段新增。
        execute_person_source_sql(
            alter_sql
        )

        # 保存字段处理状态。
        column_action = "added"

        # 显示执行结果。
        print(
            f"Added {table_name}.source_candidate_id"
        )

    else:
        # 字段已经存在时跳过。
        column_action = "skipped_existing"

        # 显示跳过结果。
        print(
            f"Skipped existing "
            f"{table_name}.source_candidate_id"
        )

    # --------------------------------------------------------
    # 增加 source_candidate_id 普通索引
    # --------------------------------------------------------

    # 使用 IF NOT EXISTS，
    # 因此相同索引已经存在时不会重复创建。
    create_index_sql = (
        f'CREATE INDEX IF NOT EXISTS "{index_name}" '
        f'ON "{table_name}" ("source_candidate_id");'
    )

    # 执行索引创建。
    execute_person_source_sql(
        create_index_sql
    )

    # --------------------------------------------------------
    # 验证字段
    # --------------------------------------------------------

    # 重新读取字段结构。
    verified_table_info_result = (
        execute_person_source_sql(
            f'PRAGMA table_info("{table_name}");'
        )
    )

    # 取得验证字段记录。
    verified_table_info_rows = (
        verified_table_info_result[0].get(
            "results",
            [],
        )
        if verified_table_info_result
        else []
    )

    # 找到 source_candidate_id 字段。
    source_candidate_column_rows = [
        row
        for row in verified_table_info_rows
        if row["name"] == "source_candidate_id"
    ]

    # 找不到字段时停止。
    if len(source_candidate_column_rows) != 1:
        raise RuntimeError(
            f"{table_name} 没有正确增加 "
            "source_candidate_id。"
        )

    # 取得 source_candidate_id 字段信息。
    source_candidate_column = (
        source_candidate_column_rows[0]
    )

    # 确认字段允许存 NULL。
    if int(
        source_candidate_column["notnull"]
    ) != 0:
        raise RuntimeError(
            f"{table_name}.source_candidate_id "
            "被错误设置为 NOT NULL。"
        )

    # --------------------------------------------------------
    # 验证外键
    # --------------------------------------------------------

    # 查询当前表的外键列表。
    foreign_key_result = (
        execute_person_source_sql(
            f'PRAGMA foreign_key_list("{table_name}");'
        )
    )

    # 取得外键记录。
    foreign_key_rows = (
        foreign_key_result[0].get(
            "results",
            [],
        )
        if foreign_key_result
        else []
    )

    # 查找 source_candidate_id 对应外键。
    source_candidate_foreign_keys = [
        row
        for row in foreign_key_rows
        if (
            row.get("from")
            == "source_candidate_id"
            and row.get("table")
            == "Candidate_snapshot"
            and row.get("to")
            == "id"
        )
    ]

    # 没找到正确外键时停止。
    if len(source_candidate_foreign_keys) != 1:
        raise RuntimeError(
            f"{table_name}.source_candidate_id "
            "没有正确引用 Candidate_snapshot.id。"
        )

    # --------------------------------------------------------
    # 验证索引
    # --------------------------------------------------------

    # 从 sqlite_master 查询目标索引。
    index_result = execute_person_source_sql(
        'SELECT "name", "sql" '
        'FROM "sqlite_master" '
        'WHERE "type" = ? '
        'AND "name" = ?;',
        [
            "index",
            index_name,
        ],
    )

    # 取得索引记录。
    index_rows = (
        index_result[0].get(
            "results",
            [],
        )
        if index_result
        else []
    )

    # 找不到索引时停止。
    if len(index_rows) != 1:
        raise RuntimeError(
            f"{table_name} 的 "
            f"{index_name} 没有正确创建。"
        )

    # --------------------------------------------------------
    # 保存当前表验证结果
    # --------------------------------------------------------

    # 保存当前表处理结果。
    migration_results.append(
        {
            "table_name": table_name,
            "record_count": record_count,
            "column_action": column_action,
            "source_candidate_nullable": True,
            "foreign_key_target": (
                "Candidate_snapshot.id"
            ),
            "index_name": index_name,
            "verification_status": "success",
        }
    )


# ============================================================
# 全部表完成后执行全数据库外键检查
# ============================================================

# 检查数据库是否存在外键违规。
foreign_key_check_result = (
    execute_person_source_sql(
        "PRAGMA foreign_key_check;"
    )
)


# 取得外键违规记录。
foreign_key_violation_rows = (
    foreign_key_check_result[0].get(
        "results",
        [],
    )
    if foreign_key_check_result
    else []
)


# 出现外键违规时停止。
if foreign_key_violation_rows:
    raise RuntimeError(
        "Person 子表修改完成后发现外键违规："
        f"{foreign_key_violation_rows}"
    )


# ============================================================
# 显示最终结果
# ============================================================

# 把验证结果转换成 DataFrame。
person_source_migration_dataframe = pd.DataFrame(
    migration_results
)


# 显示每张表的迁移结果。
display(
    person_source_migration_dataframe
)


# 显示外键检查结果。
print(
    "Foreign key violation count:",
    len(foreign_key_violation_rows),
)


# 显示本步骤完成。
print(
    "\nPerson child table source_candidate_id "
    "migration completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 给 Person_Position 增加简历工作经历文本字段。
#
# 本 Cell 不依赖任何其他 Cell，并且可以重复运行。
# ============================================================

# 导入 HTTP 请求工具。
import requests

# 导入 pandas。
import pandas as pd

# 导入 Colab Secrets。
from google.colab import userdata


# Cloudflare D1 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# Token 不存在时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# D1 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 本 Cell 自己使用的 D1 SQL function。
def execute_d1(sql, params=None):

    # 向 D1 发送请求。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 解析 Cloudflare JSON。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP {response.status_code} "
            f"{response.text[:300]}"
        )

    # SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            f"D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result。
    return payload.get("result", [])


# 查询 Person_Position 当前记录数。
count_result = execute_d1(
    'SELECT COUNT(*) AS "record_count" '
    'FROM "Person_Position";'
)


# 取得当前记录数。
record_count = int(
    count_result[0]["results"][0]["record_count"]
)


# 当前不是空表时停止。
if record_count != 0:
    raise ValueError(
        "Person_Position 当前不是空表。"
        f"实际记录数：{record_count}。"
    )


# 查询当前字段结构。
before_result = execute_d1(
    'PRAGMA table_info("Person_Position");'
)


# 取得字段记录。
before_rows = (
    before_result[0].get("results", [])
    if before_result
    else []
)


# 取得当前字段名称。
existing_columns = {
    row["name"]
    for row in before_rows
}


# 定义需要增加的 nullable TEXT 字段。
columns_to_add = [
    "raw_company_name",
    "normalized_company_name",
    "raw_position_name",
    "normalized_position_name",
    "position_description",
]


# 保存本次新增字段。
added_columns = []


# 保存已经存在而跳过的字段。
skipped_columns = []


# 逐个增加缺失字段。
for column_name in columns_to_add:

    # 字段已经存在时跳过。
    if column_name in existing_columns:
        skipped_columns.append(
            column_name
        )

        print(
            f"Skipped existing column: {column_name}"
        )

        continue

    # 增加 nullable TEXT 字段。
    execute_d1(
        'ALTER TABLE "Person_Position" '
        f'ADD COLUMN "{column_name}" TEXT;'
    )

    # 更新内存字段集合。
    existing_columns.add(
        column_name
    )

    # 保存新增结果。
    added_columns.append(
        column_name
    )

    # 显示新增字段。
    print(
        f"Added Person_Position column: "
        f"{column_name}"
    )


# 重新查询修改后的字段结构。
after_result = execute_d1(
    'PRAGMA table_info("Person_Position");'
)


# 取得修改后的字段记录。
after_rows = (
    after_result[0].get("results", [])
    if after_result
    else []
)


# 取得修改后的字段名称。
after_column_names = {
    row["name"]
    for row in after_rows
}


# 检查是否缺少任何目标字段。
missing_columns = (
    set(columns_to_add)
    - after_column_names
)


# 仍然缺少字段时停止。
if missing_columns:
    raise RuntimeError(
        "Person_Position 修改后仍缺少字段："
        f"{sorted(missing_columns)}"
    )


# 取得新增字段的字段定义。
new_column_rows = [
    row
    for row in after_rows
    if row["name"] in columns_to_add
]


# 检查所有新增字段都允许存 NULL。
not_null_columns = [
    row["name"]
    for row in new_column_rows
    if int(row["notnull"]) != 0
]


# 发现错误的 NOT NULL 时停止。
if not_null_columns:
    raise RuntimeError(
        "以下 Person_Position 新字段被错误设置为 NOT NULL："
        f"{not_null_columns}"
    )


# 执行全数据库外键检查。
foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)


# 取得外键违规记录。
foreign_key_violations = (
    foreign_key_result[0].get("results", [])
    if foreign_key_result
    else []
)


# 存在外键违规时停止。
if foreign_key_violations:
    raise RuntimeError(
        "数据库存在外键违规："
        f"{foreign_key_violations}"
    )


# 把修改后的字段结构转换成 DataFrame。
person_position_schema_dataframe = pd.DataFrame(
    after_rows
)


# 显示完整字段结构。
display(
    person_position_schema_dataframe
)


# 显示本次新增和跳过字段。
print(
    "Added columns:",
    added_columns,
)

print(
    "Skipped existing columns:",
    skipped_columns,
)


# 显示记录数和外键检查结果。
print(
    "Person_Position record count:",
    record_count,
)

print(
    "Foreign key violation count:",
    len(foreign_key_violations),
)


# 显示完成信息。
print(
    "Person_Position schema migration "
    "completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 给 Education 增加简历教育经历的原始文本字段。
#
# 本 Cell 不依赖任何其他 Cell，并且可以重复运行。
# ============================================================

# 导入 HTTP 请求工具。
import requests

# 导入 pandas。
import pandas as pd

# 导入 Colab Secrets。
from google.colab import userdata


# Cloudflare D1 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# Token 不存在时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# D1 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 本 Cell 自己使用的 D1 SQL function。
def execute_d1(sql, params=None):

    # 向 D1 发送请求。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 解析 Cloudflare JSON。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP {response.status_code} "
            f"{response.text[:300]}"
        )

    # SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            f"D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result。
    return payload.get("result", [])


# 查询 Education 当前记录数。
count_result = execute_d1(
    'SELECT COUNT(*) AS "record_count" '
    'FROM "Education";'
)


# 取得当前记录数。
record_count = int(
    count_result[0]["results"][0]["record_count"]
)


# Education 当前不是空表时停止。
if record_count != 0:
    raise ValueError(
        "Education 当前不是空表。"
        f"实际记录数：{record_count}。"
    )


# 查询 Education 当前字段。
before_result = execute_d1(
    'PRAGMA table_info("Education");'
)


# 取得当前字段记录。
before_rows = (
    before_result[0].get("results", [])
    if before_result
    else []
)


# 取得当前字段名称集合。
existing_columns = {
    row["name"]
    for row in before_rows
}


# 定义需要增加的 nullable TEXT 字段。
columns_to_add = [
    "raw_school_name",
    "raw_degree_name",
    "raw_field_study_name",
    "raw_major_name",
    "gpa",
    "education_description",
]


# 保存本次新增字段。
added_columns = []


# 保存已经存在而跳过的字段。
skipped_columns = []


# 逐个增加缺失字段。
for column_name in columns_to_add:

    # 字段已经存在时跳过。
    if column_name in existing_columns:

        skipped_columns.append(
            column_name
        )

        print(
            f"Skipped existing column: {column_name}"
        )

        continue

    # 增加 nullable TEXT 字段。
    execute_d1(
        'ALTER TABLE "Education" '
        f'ADD COLUMN "{column_name}" TEXT;'
    )

    # 更新内存字段集合。
    existing_columns.add(
        column_name
    )

    # 保存新增字段。
    added_columns.append(
        column_name
    )

    # 显示新增结果。
    print(
        f"Added Education column: {column_name}"
    )


# 重新读取修改后的字段结构。
after_result = execute_d1(
    'PRAGMA table_info("Education");'
)


# 取得修改后的字段记录。
after_rows = (
    after_result[0].get("results", [])
    if after_result
    else []
)


# 取得修改后的字段名称集合。
after_column_names = {
    row["name"]
    for row in after_rows
}


# 检查是否仍然缺少字段。
missing_columns = (
    set(columns_to_add)
    - after_column_names
)


# 仍然缺少字段时停止。
if missing_columns:
    raise RuntimeError(
        "Education 修改后仍缺少字段："
        f"{sorted(missing_columns)}"
    )


# 取得新字段的定义。
new_column_rows = [
    row
    for row in after_rows
    if row["name"] in columns_to_add
]


# 检查新增字段是否错误地设置了 NOT NULL。
not_null_columns = [
    row["name"]
    for row in new_column_rows
    if int(row["notnull"]) != 0
]


# 发现错误 NOT NULL 时停止。
if not_null_columns:
    raise RuntimeError(
        "以下 Education 新字段被错误设置为 NOT NULL："
        f"{not_null_columns}"
    )


# 执行全数据库外键检查。
foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)


# 取得外键违规记录。
foreign_key_violations = (
    foreign_key_result[0].get("results", [])
    if foreign_key_result
    else []
)


# 发现外键违规时停止。
if foreign_key_violations:
    raise RuntimeError(
        "数据库存在外键违规："
        f"{foreign_key_violations}"
    )


# 把修改后的字段结构转换成 DataFrame。
education_schema_dataframe = pd.DataFrame(
    after_rows
)


# 显示修改后的完整字段结构。
display(
    education_schema_dataframe
)


# 显示本次新增和跳过字段。
print(
    "Added columns:",
    added_columns,
)

print(
    "Skipped existing columns:",
    skipped_columns,
)


# 显示记录数和外键检查结果。
print(
    "Education record count:",
    record_count,
)

print(
    "Foreign key violation count:",
    len(foreign_key_violations),
)


# 显示完成信息。
print(
    "Education schema migration "
    "completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 创建 Candidate_Project 桥表和唯一索引。
#
# 本 Cell 不依赖任何其他 Cell，并且可以重复运行。
# ============================================================

# 导入 HTTP 请求工具。
import requests

# 导入 pandas。
import pandas as pd

# 导入 Colab Secrets。
from google.colab import userdata


# Cloudflare D1 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# Token 不存在时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# D1 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 本 Cell 自己使用的 D1 SQL function。
def execute_d1(sql, params=None):

    # 向 D1 发送请求。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 解析 Cloudflare JSON。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP {response.status_code} "
            f"{response.text[:300]}"
        )

    # SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            f"D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result。
    return payload.get("result", [])


# 查询 Candidate_Project 是否已经存在。
existing_table_result = execute_d1(
    'SELECT "name" '
    'FROM "sqlite_master" '
    'WHERE "type" = ? '
    'AND "name" = ?;',
    [
        "table",
        "Candidate_Project",
    ],
)


# 取得查询结果。
existing_table_rows = (
    existing_table_result[0].get("results", [])
    if existing_table_result
    else []
)


# Candidate_Project 不存在时创建。
if not existing_table_rows:

    # 生成 CREATE TABLE SQL。
    create_table_sql = (
        'CREATE TABLE "Candidate_Project" ('
        '"id" INTEGER PRIMARY KEY AUTOINCREMENT, '
        '"candidate_id" INTEGER NOT NULL, '
        '"person_project_id" INTEGER NOT NULL, '
        '"created_at" TEXT NOT NULL, '
        'FOREIGN KEY ("candidate_id") '
        'REFERENCES "Candidate_snapshot"("id"), '
        'FOREIGN KEY ("person_project_id") '
        'REFERENCES "Person_Project"("id")'
        ');'
    )

    # 创建 Candidate_Project。
    execute_d1(
        create_table_sql
    )

    # 保存创建状态。
    table_action = "created"

    # 显示创建结果。
    print(
        "Created Candidate_Project table."
    )

else:
    # 表已经存在时不重复创建。
    table_action = "skipped_existing"

    # 显示跳过结果。
    print(
        "Skipped existing Candidate_Project table."
    )


# 创建 candidate_id + person_project_id 唯一索引。
#
# IF NOT EXISTS 保证重复运行时不会重复创建。
execute_d1(
    'CREATE UNIQUE INDEX IF NOT EXISTS '
    '"idx_Candidate_Project_0_'
    'candidate_id_person_project_id" '
    'ON "Candidate_Project" ('
    '"candidate_id", '
    '"person_project_id"'
    ');'
)


# 查询 Candidate_Project 字段结构。
table_info_result = execute_d1(
    'PRAGMA table_info("Candidate_Project");'
)


# 取得字段记录。
table_info_rows = (
    table_info_result[0].get("results", [])
    if table_info_result
    else []
)


# 转换成以字段名为 key 的字典。
table_info_by_name = {
    row["name"]: row
    for row in table_info_rows
}


# 定义必须存在的四个字段。
required_columns = {
    "id",
    "candidate_id",
    "person_project_id",
    "created_at",
}


# 检查是否缺少字段。
missing_columns = (
    required_columns
    - set(table_info_by_name.keys())
)


# 缺少字段时停止。
if missing_columns:
    raise RuntimeError(
        "Candidate_Project 缺少字段："
        f"{sorted(missing_columns)}"
    )


# candidate_id 必须为 NOT NULL。
if int(
    table_info_by_name[
        "candidate_id"
    ]["notnull"]
) != 1:
    raise RuntimeError(
        "Candidate_Project.candidate_id "
        "没有正确设置 NOT NULL。"
    )


# person_project_id 必须为 NOT NULL。
if int(
    table_info_by_name[
        "person_project_id"
    ]["notnull"]
) != 1:
    raise RuntimeError(
        "Candidate_Project.person_project_id "
        "没有正确设置 NOT NULL。"
    )


# created_at 必须为 NOT NULL。
if int(
    table_info_by_name[
        "created_at"
    ]["notnull"]
) != 1:
    raise RuntimeError(
        "Candidate_Project.created_at "
        "没有正确设置 NOT NULL。"
    )


# 查询 Candidate_Project 外键。
foreign_key_result = execute_d1(
    'PRAGMA foreign_key_list("Candidate_Project");'
)


# 取得外键记录。
foreign_key_rows = (
    foreign_key_result[0].get("results", [])
    if foreign_key_result
    else []
)


# 验证 candidate_id 外键。
candidate_foreign_key_valid = any(
    row.get("from") == "candidate_id"
    and row.get("table") == "Candidate_snapshot"
    and row.get("to") == "id"
    for row in foreign_key_rows
)


# 验证 person_project_id 外键。
project_foreign_key_valid = any(
    row.get("from") == "person_project_id"
    and row.get("table") == "Person_Project"
    and row.get("to") == "id"
    for row in foreign_key_rows
)


# 任一外键不正确时停止。
if not candidate_foreign_key_valid:
    raise RuntimeError(
        "candidate_id 外键不正确。"
    )


if not project_foreign_key_valid:
    raise RuntimeError(
        "person_project_id 外键不正确。"
    )


# 查询唯一索引。
index_result = execute_d1(
    'SELECT "name", "sql" '
    'FROM "sqlite_master" '
    'WHERE "type" = ? '
    'AND "name" = ?;',
    [
        "index",
        (
            "idx_Candidate_Project_0_"
            "candidate_id_person_project_id"
        ),
    ],
)


# 取得索引结果。
index_rows = (
    index_result[0].get("results", [])
    if index_result
    else []
)


# 找不到唯一索引时停止。
if len(index_rows) != 1:
    raise RuntimeError(
        "Candidate_Project 唯一索引没有正确创建。"
    )


# 查询 Candidate_Project 记录数。
count_result = execute_d1(
    'SELECT COUNT(*) AS "record_count" '
    'FROM "Candidate_Project";'
)


# 取得记录数。
record_count = int(
    count_result[0]["results"][0]["record_count"]
)


# 执行全数据库外键检查。
foreign_key_check_result = execute_d1(
    "PRAGMA foreign_key_check;"
)


# 取得外键违规记录。
foreign_key_violations = (
    foreign_key_check_result[0].get("results", [])
    if foreign_key_check_result
    else []
)


# 存在外键违规时停止。
if foreign_key_violations:
    raise RuntimeError(
        "数据库存在外键违规："
        f"{foreign_key_violations}"
    )


# 将字段结构转换成 DataFrame。
candidate_project_schema_dataframe = pd.DataFrame(
    table_info_rows
)


# 显示字段结构。
display(
    candidate_project_schema_dataframe
)


# 显示外键结构。
display(
    pd.DataFrame(
        foreign_key_rows
    )
)


# 显示执行结果。
print(
    "Table action:",
    table_action,
)

print(
    "Candidate_Project record count:",
    record_count,
)

print(
    "Unique index:",
    index_rows[0]["name"],
)

print(
    "Foreign key violation count:",
    len(foreign_key_violations),
)


# 显示完成信息。
print(
    "Candidate_Project schema migration "
    "completed successfully."
)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 给 Person_Skill 增加 person_id + skill_id 唯一索引。
#
# 本 Cell 不依赖任何其他 Cell，并且可以重复运行。
# ============================================================

# 导入 HTTP 请求工具。
import requests

# 导入 pandas。
import pandas as pd

# 导入 Colab Secrets。
from google.colab import userdata


# Cloudflare D1 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# Token 不存在时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# D1 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 本 Cell 自己使用的 D1 SQL function。
def execute_d1(sql, params=None):

    # 向 D1 发送请求。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=30,
    )

    # 解析 Cloudflare JSON。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP {response.status_code} "
            f"{response.text[:300]}"
        )

    # SQL 执行失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            f"D1 SQL 执行失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result。
    return payload.get("result", [])


# 唯一索引名称。
INDEX_NAME = (
    "idx_Person_Skill_0_person_id_skill_id"
)


# 查询 Person_Skill 当前记录数。
count_result = execute_d1(
    'SELECT COUNT(*) AS "record_count" '
    'FROM "Person_Skill";'
)


# 取得当前记录数。
record_count = int(
    count_result[0]["results"][0]["record_count"]
)


# Person_Skill 当前不是空表时停止。
#
# 如果未来表内已有数据，
# 创建唯一索引前必须先检查重复组合。
if record_count != 0:
    raise ValueError(
        "Person_Skill 当前不是空表。"
        f"实际记录数：{record_count}。"
        "本次索引创建已停止。"
    )


# 创建 person_id + skill_id 唯一索引。
#
# IF NOT EXISTS 保证重复运行时不会重复创建。
execute_d1(
    f'CREATE UNIQUE INDEX IF NOT EXISTS "{INDEX_NAME}" '
    'ON "Person_Skill" ('
    '"person_id", '
    '"skill_id"'
    ');'
)


# 查询 Person_Skill 的索引列表。
index_list_result = execute_d1(
    'PRAGMA index_list("Person_Skill");'
)


# 取得索引记录。
index_list_rows = (
    index_list_result[0].get("results", [])
    if index_list_result
    else []
)


# 找到本次目标索引。
target_index_rows = [
    row
    for row in index_list_rows
    if row["name"] == INDEX_NAME
]


# 索引必须存在且只能出现一次。
if len(target_index_rows) != 1:
    raise RuntimeError(
        "Person_Skill 唯一索引没有正确创建。"
    )


# unique 必须等于 1。
if int(target_index_rows[0]["unique"]) != 1:
    raise RuntimeError(
        "Person_Skill 目标索引不是唯一索引。"
    )


# 查询索引中包含的字段及顺序。
index_info_result = execute_d1(
    f'PRAGMA index_info("{INDEX_NAME}");'
)


# 取得索引字段记录。
index_info_rows = (
    index_info_result[0].get("results", [])
    if index_info_result
    else []
)


# 按照索引字段顺序取得字段名。
indexed_columns = [
    row["name"]
    for row in sorted(
        index_info_rows,
        key=lambda row: int(row["seqno"]),
    )
]


# 索引字段必须严格等于 person_id、skill_id。
if indexed_columns != [
    "person_id",
    "skill_id",
]:
    raise RuntimeError(
        "Person_Skill 唯一索引字段不正确："
        f"{indexed_columns}"
    )


# 执行全数据库外键检查。
foreign_key_check_result = execute_d1(
    "PRAGMA foreign_key_check;"
)


# 取得外键违规记录。
foreign_key_violations = (
    foreign_key_check_result[0].get("results", [])
    if foreign_key_check_result
    else []
)


# 存在外键违规时停止。
if foreign_key_violations:
    raise RuntimeError(
        "数据库存在外键违规："
        f"{foreign_key_violations}"
    )


# 显示 Person_Skill 所有索引。
display(
    pd.DataFrame(
        index_list_rows
    )
)


# 显示目标索引字段。
display(
    pd.DataFrame(
        index_info_rows
    )
)


# 显示最终结果。
print(
    "Person_Skill record count:",
    record_count,
)

print(
    "Unique index:",
    INDEX_NAME,
)

print(
    "Indexed columns:",
    indexed_columns,
)

print(
    "Foreign key violation count:",
    len(foreign_key_violations),
)


# 显示完成信息。
print(
    "Person_Skill unique index migration "
    "completed successfully."
)

## 实际导入数据前的验证代码（没有任何作用，不生成任何东西，也不导入任何东西）

In [ ]:
"""
Google Colab 完全独立 Cell：
按照增量导入规则验证 application.csv 中已经导入的 Application ID。

当前 application.csv 可以只包含本次批次的 Application，
不要求包含 D1 Application 表中的历史记录。

本文件只执行 SELECT，不修改数据库。
"""

# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于读取和验证 application.csv。
import pandas as pd

# drive 用于挂载 Google Drive。
from google.colab import drive

# userdata 用于读取 Colab Secrets。
from google.colab import userdata


# 挂载 Google Drive；已经挂载时只会显示提示。
drive.mount("/content/drive")

# application.csv 固定路径。
APPLICATION_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline/"
    "datasets/dedup_results/application_dedup_v2"
)
APPLICATION_CSV_PATH = f"{APPLICATION_FOLDER}/application.csv"

# Cloudflare D1 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")

# D1 Query API 配置。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


def execute_read_only_d1(sql, params=None):
    """执行一条只读 D1 查询。"""

    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={"sql": sql, "params": params or []},
        timeout=30,
    )
    try:
        payload = response.json()
    except ValueError as error:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP {response.status_code} "
            f"{response.text[:300]}"
        ) from error
    if response.status_code != 200 or not payload.get("success", False):
        raise RuntimeError(f"D1 查询失败：{payload.get('errors', payload)}")
    return payload.get("result", [])


# 使用 utf-8-sig 读取 CSV，自动移除 BOM。
application_csv_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
)
application_csv_dataframe.columns = [
    str(column_name).strip()
    for column_name in application_csv_dataframe.columns
]

# 检查本次验证需要的字段。
required_csv_columns = [
    "id",
    "raw_submission_id",
    "submission_normalized_id",
    "source_record_number",
    "application_cluster__display_name",
]
missing_csv_columns = [
    column_name
    for column_name in required_csv_columns
    if column_name not in application_csv_dataframe.columns
]
if missing_csv_columns:
    raise ValueError(
        f"application.csv 缺少必要字段：{missing_csv_columns}"
    )

# 空 id 表示该行没有成功进入 D1 Application。
application_csv_dataframe["id"] = pd.to_numeric(
    application_csv_dataframe["id"],
    errors="coerce",
)
application_backfill_source_dataframe = application_csv_dataframe[
    application_csv_dataframe["id"].notna()
].copy()
application_not_imported_dataframe = application_csv_dataframe[
    application_csv_dataframe["id"].isna()
].copy()
application_backfill_source_dataframe["id"] = (
    application_backfill_source_dataframe["id"].astype("int64")
)

# 三个映射字段必须都是有效整数。
numeric_mapping_columns = [
    "raw_submission_id",
    "submission_normalized_id",
    "source_record_number",
]
for column_name in numeric_mapping_columns:
    application_backfill_source_dataframe[column_name] = pd.to_numeric(
        application_backfill_source_dataframe[column_name],
        errors="coerce",
    )
null_mapping_counts = application_backfill_source_dataframe[
    numeric_mapping_columns
].isna().sum()
if int(null_mapping_counts.sum()) != 0:
    raise ValueError(
        f"已导入 Application 的映射字段存在空值：\n{null_mapping_counts}"
    )
for column_name in numeric_mapping_columns:
    application_backfill_source_dataframe[column_name] = (
        application_backfill_source_dataframe[column_name].astype("int64")
    )

# 关键 ID 必须一对一。
if application_backfill_source_dataframe["id"].duplicated().any():
    raise ValueError("application.csv 中存在重复 Application.id。")
if application_backfill_source_dataframe["raw_submission_id"].duplicated().any():
    raise ValueError("已导入 Application 中存在重复 raw_submission_id。")

# cluster display name 不能为空。
cluster_display_missing_count = int(
    application_backfill_source_dataframe[
        "application_cluster__display_name"
    ].isna().sum()
)
if cluster_display_missing_count != 0:
    raise ValueError(
        "已导入 Application 中存在空的 application_cluster__display_name。"
    )

# 从 D1 读取全部 Application ID。
d1_result = execute_read_only_d1(
    'SELECT "id" FROM "Application" ORDER BY "id";'
)
d1_application_rows = (
    d1_result[0].get("results", []) if d1_result else []
)
d1_application_ids = {int(row["id"]) for row in d1_application_rows}
csv_application_ids = set(
    application_backfill_source_dataframe["id"].tolist()
)

# 计算两边集合差异。
ids_missing_from_csv = sorted(d1_application_ids - csv_application_ids)
ids_missing_from_d1 = sorted(csv_application_ids - d1_application_ids)

# 生成并显示验证汇总。
application_id_validation = {
    "application_csv_total_rows": int(len(application_csv_dataframe)),
    "csv_rows_with_application_id": int(
        len(application_backfill_source_dataframe)
    ),
    "csv_rows_without_application_id": int(
        len(application_not_imported_dataframe)
    ),
    "d1_application_record_count": int(len(d1_application_ids)),
    "unique_csv_application_id_count": int(len(csv_application_ids)),
    "ids_missing_from_csv_count": int(len(ids_missing_from_csv)),
    "ids_missing_from_d1_count": int(len(ids_missing_from_d1)),
}
print("Application ID validation:")
print(application_id_validation)
display(
    application_backfill_source_dataframe[required_csv_columns].head(10)
)

# ============================================================
# 按增量 CSV 规则执行最终验证
# ============================================================

# 当前 application.csv 只需要包含本次批次新增或处理的 Application。
# D1 中存在但当前 CSV 中不存在的 ID 属于历史批次，是正常情况。
#
# 只有 CSV 中存在、但 D1 Application 中不存在的 ID，
# 才表示本批次 Application 没有成功写入数据库。
if ids_missing_from_d1:
    raise ValueError(
        "application.csv 中存在尚未写入 D1 Application 的 ID。"
        f"\nCSV 有但 D1 没有：{ids_missing_from_d1[:20]}"
    )

# 显示 D1 中不属于当前增量 CSV 的历史 Application 数量。
print(
    "D1 中不属于当前增量 CSV 的历史 Application 数量：",
    len(ids_missing_from_csv),
)

# 显示最终验证成功结果。
print(
    "当前 application.csv 中的全部 Application ID "
    "均已存在于 D1 Application。"
)


In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 验证 Application 与 raw_submission、原始 CSV 的对应关系。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Cell。
# 2. 本 Cell只执行 SELECT，不更新数据库。
# 3. 本 Cell暂时不解析 company 和 position。
# ============================================================

# 导入系统文件工具。
import os

# 导入 CSV 读取工具。
import csv

# 导入 SHA-256 工具。
import hashlib

# 导入网络请求工具。
import requests

# 导入 pandas。
import pandas as pd

# 导入 Google Drive。
from google.colab import drive

# 导入 Colab Secrets。
from google.colab import userdata


# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)


# Google Drive 根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)


# 每次运行时都要求用户明确输入本批次实际使用的原始 CSV 文件名。
#
# 这样后续无论新批次包含 10 条、100 条还是更多记录，
# 都不需要重新修改 Cell 中的固定文件名。
raw_csv_file_name = input(
    "请输入本批次原始 CSV 文件名（例如 test_raw_submission_10.csv）："
).strip()

# 没有输入文件名时立即停止，防止意外读取旧批次文件。
if not raw_csv_file_name:
    raise ValueError(
        "本批次原始 CSV 文件名不能为空。"
    )

# 这里只接受文件名，不接受包含目录分隔符的路径，
# 防止输入内容跳出固定 datasets 文件夹。
if os.path.basename(raw_csv_file_name) != raw_csv_file_name:
    raise ValueError(
        "请输入单独的 CSV 文件名，不要输入完整路径。"
    )

# 本 Cell 只接受 CSV 文件。
if not raw_csv_file_name.lower().endswith(".csv"):
    raise ValueError(
        "本批次原始数据文件必须是 .csv 文件。"
    )

# 根据用户输入动态生成本批次原始 CSV 的完整路径。
RAW_SUBMISSION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/{raw_csv_file_name}"
)


# application.csv 路径。
APPLICATION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/dedup_results/"
    "application_dedup_v2/application.csv"
)


# Cloudflare 连接信息。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")


# Token 不存在时停止。
if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")


# 检查两个输入 CSV 是否存在。
for required_path in [
    RAW_SUBMISSION_CSV_PATH,
    APPLICATION_CSV_PATH,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            f"找不到必要文件：{required_path}"
        )


# D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)


# D1 请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# 本 Cell 自己使用的只读 D1 function。
def execute_read_only_d1(
    sql,
    params=None,
):
    # 向 D1 发送只读查询。
    response = requests.post(
        D1_API_URL,
        headers=D1_HEADERS,
        json={
            "sql": sql,
            "params": params or [],
        },
        timeout=60,
    )

    # 解析 JSON。
    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            f"D1 返回非 JSON：HTTP "
            f"{response.status_code} "
            f"{response.text[:300]}"
        )

    # 查询失败时停止。
    if (
        response.status_code != 200
        or not payload.get("success", False)
    ):
        raise RuntimeError(
            f"D1 查询失败："
            f"{payload.get('errors', payload)}"
        )

    # 返回 D1 result。
    return payload.get("result", [])


# 读取 application.csv。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
)


# 清理 column name。
application_dataframe.columns = [
    str(column_name).strip()
    for column_name in application_dataframe.columns
]


# 把 Application ID 转成数字。
application_dataframe["id"] = pd.to_numeric(
    application_dataframe["id"],
    errors="coerce",
)


# 只保留已经进入 D1 Application 的记录。
application_dataframe = (
    application_dataframe[
        application_dataframe["id"].notna()
    ]
    .copy()
)


# 转换三个必要 ID 字段。
for column_name in [
    "id",
    "raw_submission_id",
    "source_record_number",
]:
    application_dataframe[column_name] = (
        pd.to_numeric(
            application_dataframe[column_name],
            errors="raise",
        )
        .astype("int64")
    )


# 取得所有需要查询的 raw_submission_id。
raw_submission_ids = (
    application_dataframe[
        "raw_submission_id"
    ]
    .tolist()
)


# 用于保存 D1 raw_submission 查询结果。
d1_raw_submission_rows = []


# 每次查询 50 个 raw_submission，
# 避免一次 SQL 使用过多参数。
CHUNK_SIZE = 50


# 分批查询 raw_submission metadata。
for chunk_start in range(
    0,
    len(raw_submission_ids),
    CHUNK_SIZE,
):
    # 取得当前批次 ID。
    chunk_ids = raw_submission_ids[
        chunk_start:
        chunk_start + CHUNK_SIZE
    ]

    # 为当前批次生成 ? 占位符。
    placeholders = ",".join(
        ["?"] * len(chunk_ids)
    )

    # 只查询对应关系和 hash，
    # 暂时不读取很长的 raw_record_jsonb。
    sql = (
        'SELECT '
        '"id" AS "d1_raw_submission_id", '
        '"ingestion_batch_id", '
        '"source_record_number" '
        'AS "d1_source_record_number", '
        '"raw_record_jsonb_sha256" '
        'FROM "raw_submission" '
        f'WHERE "id" IN ({placeholders});'
    )

    # 执行当前批次查询。
    result = execute_read_only_d1(
        sql,
        [
            int(raw_submission_id)
            for raw_submission_id in chunk_ids
        ],
    )

    # 取得当前批次返回记录。
    rows = (
        result[0].get("results", [])
        if result
        else []
    )

    # 保存当前批次记录。
    d1_raw_submission_rows.extend(
        rows
    )


# 转成 DataFrame。
d1_raw_submission_dataframe = pd.DataFrame(
    d1_raw_submission_rows
)


# D1 必须返回与 Application 相同数量的 raw_submission。
if (
    len(d1_raw_submission_dataframe)
    != len(application_dataframe)
):
    raise ValueError(
        "D1 返回的 raw_submission 数量与"
        "已导入 Application 数量不一致。"
    )


# 转换 D1 数字字段。
for column_name in [
    "d1_raw_submission_id",
    "ingestion_batch_id",
    "d1_source_record_number",
]:
    d1_raw_submission_dataframe[column_name] = (
        pd.to_numeric(
            d1_raw_submission_dataframe[
                column_name
            ],
            errors="raise",
        )
        .astype("int64")
    )


# 把 Application mapping 与 D1 raw_submission 合并。
mapping_validation_dataframe = (
    application_dataframe[
        [
            "id",
            "raw_submission_id",
            "source_record_number",
            "application_cluster__display_name",
        ]
    ]
    .merge(
        d1_raw_submission_dataframe,
        left_on="raw_submission_id",
        right_on="d1_raw_submission_id",
        how="left",
        validate="one_to_one",
    )
)


# 验证 source_record_number 是否一致。
mapping_validation_dataframe[
    "source_record_number_matches"
] = (
    mapping_validation_dataframe[
        "source_record_number"
    ]
    == mapping_validation_dataframe[
        "d1_source_record_number"
    ]
)


# 所有记录必须属于唯一 ingestion batch。
ingestion_batch_ids = sorted(
    mapping_validation_dataframe[
        "ingestion_batch_id"
    ]
    .dropna()
    .astype("int64")
    .unique()
    .tolist()
)


# 不是唯一 batch 时停止。
if len(ingestion_batch_ids) != 1:
    raise ValueError(
        "Application 对应的 raw submissions "
        "不属于唯一 ingestion batch："
        f"{ingestion_batch_ids}"
    )


# 取得唯一 batch ID。
ingestion_batch_id = int(
    ingestion_batch_ids[0]
)


# 查询对应 Ingestion_batch。
batch_result = execute_read_only_d1(
    'SELECT '
    '"id", '
    '"source_file_name", '
    '"file_sha256", '
    '"total_row_count", '
    '"successful_row_count", '
    '"failed_row_count" '
    'FROM "Ingestion_batch" '
    'WHERE "id" = ?;',
    [
        ingestion_batch_id,
    ],
)


# 取得 batch 记录。
batch_rows = (
    batch_result[0].get("results", [])
    if batch_result
    else []
)


# 必须只找到一个 batch。
if len(batch_rows) != 1:
    raise ValueError(
        "没有找到唯一 Ingestion_batch。"
    )


# 取得 batch metadata。
batch_record = batch_rows[0]


# 读取当前原始 CSV 的完整 bytes。
with open(
    RAW_SUBMISSION_CSV_PATH,
    "rb",
) as raw_csv_file:
    raw_csv_bytes = raw_csv_file.read()


# 计算当前原始 CSV SHA-256。
current_csv_sha256 = hashlib.sha256(
    raw_csv_bytes
).hexdigest()


# 使用 csv.DictReader 统计原始 CSV 数据行数。
with open(
    RAW_SUBMISSION_CSV_PATH,
    encoding="utf-8-sig",
    newline="",
) as raw_csv_file:
    raw_csv_row_count = sum(
        1
        for _ in csv.DictReader(
            raw_csv_file
        )
    )


# 生成批次验证结果。
batch_validation = {
    "ingestion_batch_id": ingestion_batch_id,
    "d1_source_file_name": (
        batch_record["source_file_name"]
    ),
    "current_source_file_name": (
        os.path.basename(
            RAW_SUBMISSION_CSV_PATH
        )
    ),
    "d1_file_sha256": (
        batch_record["file_sha256"]
    ),
    "current_file_sha256": (
        current_csv_sha256
    ),
    "file_sha256_matches": (
        batch_record["file_sha256"]
        == current_csv_sha256
    ),
    "d1_total_row_count": int(
        batch_record["total_row_count"]
    ),
    "d1_successful_row_count": int(
        batch_record["successful_row_count"]
    ),
    "d1_failed_row_count": int(
        batch_record["failed_row_count"] or 0
    ),
    "current_csv_row_count": int(
        raw_csv_row_count
    ),
    "application_mapping_count": int(
        len(mapping_validation_dataframe)
    ),
    "source_record_number_mismatch_count": int(
        (
            ~mapping_validation_dataframe[
                "source_record_number_matches"
            ]
        ).sum()
    ),
}


# 显示批次验证结果。
print(
    "Raw submission mapping validation:"
)

print(
    batch_validation
)


# 显示前十条对应关系。
display(
    mapping_validation_dataframe.head(10)
)


# 文件 hash 必须一致。
if not batch_validation[
    "file_sha256_matches"
]:
    raise ValueError(
        "当前 raw_submission.csv 与导入 D1 时的"
        "原始 CSV SHA-256 不一致。"
    )


# CSV 行数必须一致。
if (
    batch_validation[
        "d1_total_row_count"
    ]
    != batch_validation[
        "current_csv_row_count"
    ]
):
    raise ValueError(
        "当前 raw_submission.csv 行数与"
        "Ingestion_batch.total_row_count 不一致。"
    )


# source_record_number 必须全部一致。
if (
    batch_validation[
        "source_record_number_mismatch_count"
    ]
    != 0
):
    raise ValueError(
        "Application CSV 与 D1 raw_submission 的"
        "source_record_number 存在不一致。"
    )


# 显示最终验证完成。
print(
    "Application, D1 raw_submission, and "
    "raw_submission.csv mappings are valid."
)



In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 从 D1 raw_submission.raw_record_jsonb 解析原始 company
# 和 position，并与 raw_submission.csv 逐条验证。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Cell。
# 2. 本 Cell 只执行 SELECT。
# 3. 本 Cell 不更新 Application。
# 4. 本 Cell 可以安全地重复运行。
# ============================================================


# os 用于检查文件路径。
import os

# time 用于网络失败后的短暂重试等待。
import time

# requests 用于调用 Cloudflare D1 API。
import requests

# pandas 用于处理 CSV 和验证结果。
import pandas as pd

# drive 用于挂载 Google Drive。
from google.colab import drive

# userdata 用于读取 Colab Secrets。
from google.colab import userdata


# ============================================================
# Google Drive 文件路径
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)


# Google Drive 项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)


# 每次运行时都要求用户明确输入本批次实际使用的原始 CSV 文件名。
#
# 这样后续无论新批次包含 10 条、100 条还是更多记录，
# 都不需要重新修改 Cell 中的固定文件名。
raw_csv_file_name = input(
    "请输入本批次原始 CSV 文件名（例如 test_raw_submission_10.csv）："
).strip()

# 没有输入文件名时立即停止，防止意外读取旧批次文件。
if not raw_csv_file_name:
    raise ValueError(
        "本批次原始 CSV 文件名不能为空。"
    )

# 这里只接受文件名，不接受包含目录分隔符的路径，
# 防止输入内容跳出固定 datasets 文件夹。
if os.path.basename(raw_csv_file_name) != raw_csv_file_name:
    raise ValueError(
        "请输入单独的 CSV 文件名，不要输入完整路径。"
    )

# 本 Cell 只接受 CSV 文件。
if not raw_csv_file_name.lower().endswith(".csv"):
    raise ValueError(
        "本批次原始数据文件必须是 .csv 文件。"
    )

# 根据用户输入动态生成本批次原始 CSV 的完整路径。
RAW_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/{raw_csv_file_name}"
)


# 已经用于导入 D1 Application 的 application.csv 路径。
APPLICATION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/dedup_results/"
    "application_dedup_v2/application.csv"
)


# 检查两个输入文件是否存在。
for required_file_path in [
    RAW_CSV_PATH,
    APPLICATION_CSV_PATH,
]:
    if not os.path.exists(
        required_file_path
    ):
        raise FileNotFoundError(
            f"找不到必要输入文件："
            f"{required_file_path}"
        )


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API Token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)


# 没有读取到 Token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
    )


# 生成 D1 Query API 地址。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/"
    f"{DATABASE_ID}/query"
)


# 生成 D1 API headers。
D1_HEADERS = {
    "Authorization": (
        f"Bearer {API_TOKEN}"
    ),
    "Content-Type": "application/json",
}


# ============================================================
# 定义本 Cell 自己使用的只读 D1 function
# ============================================================

def execute_read_only_d1(
    sql,
    params=None,
    max_retries=3,
):
    """
    执行一条只读 D1 SQL。

    网络失败时最多尝试三次。
    """

    # 保存最后一次错误。
    last_error = None

    # 最多尝试 max_retries 次。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 向 Cloudflare D1 发送请求。
            response = requests.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )

        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有重试机会时等待一秒。
            if attempt < max_retries:
                time.sleep(1.0)
                continue

            # 重试次数用完后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            )

        try:
            # 解析 Cloudflare JSON。
            payload = response.json()

        except ValueError:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            )

        # 查询成功时返回 D1 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有重试机会时等待一秒。
        if attempt < max_retries:
            time.sleep(1.0)

    # 三次都失败时停止。
    raise RuntimeError(
        "D1 查询在重试后仍然失败："
        f"{last_error}"
    )


# ============================================================
# 读取 application.csv
# ============================================================

# 读取 application.csv。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
)


# 清理 column name 两端空格。
application_dataframe.columns = [
    str(column_name).strip()
    for column_name
    in application_dataframe.columns
]


# 定义本 Cell 需要的 application.csv 字段。
required_application_columns = [
    "id",
    "raw_submission_id",
    "source_record_number",
    "application_cluster__display_name",
]


# 检查必要字段是否存在。
missing_application_columns = [
    column_name
    for column_name
    in required_application_columns
    if column_name
    not in application_dataframe.columns
]


# 缺少字段时停止。
if missing_application_columns:
    raise ValueError(
        "application.csv 缺少字段："
        f"{missing_application_columns}"
    )


# 将 Application ID 尽力转换成数字。
application_dataframe["id"] = (
    pd.to_numeric(
        application_dataframe["id"],
        errors="coerce",
    )
)


# 只保留已经成功进入 D1 Application 的记录。
application_dataframe = (
    application_dataframe[
        application_dataframe[
            "id"
        ].notna()
    ]
    .copy()
)


# 把 ID 和行号字段转换成整数。
for column_name in [
    "id",
    "raw_submission_id",
    "source_record_number",
]:
    application_dataframe[
        column_name
    ] = (
        pd.to_numeric(
            application_dataframe[
                column_name
            ],
            errors="raise",
        )
        .astype("int64")
    )


# 验证 Application ID 唯一。
if application_dataframe[
    "id"
].duplicated().any():
    raise ValueError(
        "application.csv 中存在重复 Application.id。"
    )


# 验证 raw_submission_id 唯一。
if application_dataframe[
    "raw_submission_id"
].duplicated().any():
    raise ValueError(
        "application.csv 中存在重复 raw_submission_id。"
    )


# ============================================================
# 读取原始 raw_submission.csv
# ============================================================

# 使用 string 类型读取原始 CSV。
#
# keep_default_na=False 可以保留原始空字符串。
raw_csv_dataframe = pd.read_csv(
    RAW_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)


# 检查 company 和 position 字段。
for required_column in [
    "company",
    "position",
]:
    if required_column not in (
        raw_csv_dataframe.columns
    ):
        raise ValueError(
            "raw_submission.csv 缺少字段："
            f"{required_column}"
        )


# ============================================================
# 从 D1 raw_record_jsonb 解析 company 和 position
# ============================================================

# 取得需要查询的 raw_submission_id。
raw_submission_ids = (
    application_dataframe[
        "raw_submission_id"
    ]
    .tolist()
)


# 保存 D1 解析结果。
parsed_d1_rows = []


# 每次查询 50 条记录。
CHUNK_SIZE = 50


# 分批查询 D1。
for chunk_start in range(
    0,
    len(raw_submission_ids),
    CHUNK_SIZE,
):
    # 取得当前批次 ID。
    chunk_ids = raw_submission_ids[
        chunk_start:
        chunk_start + CHUNK_SIZE
    ]

    # 生成 ? 参数占位符。
    placeholders = ",".join(
        ["?"] * len(chunk_ids)
    )

    # 在 D1 中定位固定的字段边界。
    #
    # company 是 raw_record_jsonb 的第一行。
    #
    # position 必须出现在 resume_text 字段之前。
    sql = (
        'WITH "selected" AS ('
        'SELECT '
        '"id", '
        '"ingestion_batch_id", '
        '"source_record_number", '
        '"raw_record_jsonb_sha256", '
        '"raw_record_jsonb", '
        'instr("raw_record_jsonb", char(10)) '
        'AS "first_newline", '
        'instr("raw_record_jsonb", '
        'char(10) || \'position: \') '
        'AS "position_marker", '
        'instr("raw_record_jsonb", '
        'char(10) || \'resume_text: \') '
        'AS "resume_marker" '
        'FROM "raw_submission" '
        f'WHERE "id" IN ({placeholders})'
        ') '
        'SELECT '
        '"id" AS "d1_raw_submission_id", '
        '"ingestion_batch_id", '
        '"source_record_number" '
        'AS "d1_source_record_number", '
        '"raw_record_jsonb_sha256", '
        'CASE '
        'WHEN substr('
        '"raw_record_jsonb", 1, 9'
        ') = \'company: \' '
        'AND "first_newline" > 9 '
        'THEN substr('
        '"raw_record_jsonb", '
        '10, '
        '"first_newline" - 10'
        ') '
        'ELSE NULL '
        'END AS "d1_company", '
        'CASE '
        'WHEN "position_marker" > 0 '
        'AND "resume_marker" '
        '> "position_marker" '
        'AND instr('
        'substr('
        '"raw_record_jsonb", '
        '"position_marker" + 11'
        '), '
        'char(10)'
        ') > 0 '
        'THEN substr('
        '"raw_record_jsonb", '
        '"position_marker" + 11, '
        'instr('
        'substr('
        '"raw_record_jsonb", '
        '"position_marker" + 11'
        '), '
        'char(10)'
        ') - 1'
        ') '
        'ELSE NULL '
        'END AS "d1_position", '
        'CASE '
        'WHEN substr('
        '"raw_record_jsonb", 1, 9'
        ') = \'company: \' '
        'AND "first_newline" > 9 '
        'AND "position_marker" > 0 '
        'AND "resume_marker" '
        '> "position_marker" '
        'THEN 1 '
        'ELSE 0 '
        'END AS "parse_structure_valid" '
        'FROM "selected";'
    )

    # 执行当前批次只读查询。
    result = execute_read_only_d1(
        sql,
        [
            int(raw_submission_id)
            for raw_submission_id
            in chunk_ids
        ],
    )

    # 取得当前批次记录。
    rows = (
        result[0].get(
            "results",
            [],
        )
        if result
        else []
    )

    # 保存当前批次记录。
    parsed_d1_rows.extend(
        rows
    )

    # 计算已处理数量。
    processed_count = min(
        chunk_start + CHUNK_SIZE,
        len(raw_submission_ids),
    )

    # 每 200 条或最后一批显示进度。
    if (
        processed_count % 200 == 0
        or processed_count
        == len(raw_submission_ids)
    ):
        print(
            "Parsed D1 raw submissions:",
            f"{processed_count}/"
            f"{len(raw_submission_ids)}",
        )


# 将 D1 解析结果转换成 DataFrame。
parsed_d1_dataframe = pd.DataFrame(
    parsed_d1_rows
)


# 解析结果数量必须与 Application 数量一致。
if (
    len(parsed_d1_dataframe)
    != len(application_dataframe)
):
    raise ValueError(
        "D1 解析结果数量与 Application 数量不一致。"
    )


# 转换 D1 数字字段。
for column_name in [
    "d1_raw_submission_id",
    "ingestion_batch_id",
    "d1_source_record_number",
    "parse_structure_valid",
]:
    parsed_d1_dataframe[
        column_name
    ] = (
        pd.to_numeric(
            parsed_d1_dataframe[
                column_name
            ],
            errors="raise",
        )
        .astype("int64")
    )


# ============================================================
# 合并 Application 与 D1 解析结果
# ============================================================

# 按 raw_submission_id 一对一合并。
validation_dataframe = (
    application_dataframe[
        [
            "id",
            "raw_submission_id",
            "source_record_number",
            "application_cluster__display_name",
        ]
    ]
    .merge(
        parsed_d1_dataframe,
        left_on="raw_submission_id",
        right_on="d1_raw_submission_id",
        how="left",
        validate="one_to_one",
    )
)


# 验证 source_record_number 一致。
source_record_number_matches = (
    validation_dataframe[
        "source_record_number"
    ]
    ==
    validation_dataframe[
        "d1_source_record_number"
    ]
)


# 发现不一致时停止。
if not source_record_number_matches.all():
    raise ValueError(
        "Application 与 D1 raw_submission 的 "
        "source_record_number 不一致。"
    )


# ============================================================
# 从原始 CSV 取得 company 和 position
# ============================================================

def get_raw_csv_value(
    source_record_number,
    column_name,
):
    """
    根据从 1 开始的 source_record_number，
    取得原始 CSV 对应字段。
    """

    # pandas index 从 0 开始。
    dataframe_index = (
        int(source_record_number) - 1
    )

    # 行号超出范围时返回 None。
    if (
        dataframe_index < 0
        or dataframe_index
        >= len(raw_csv_dataframe)
    ):
        return None

    # 返回原始 CSV 字段值。
    return raw_csv_dataframe.iloc[
        dataframe_index
    ][column_name]


# 取得 CSV company。
validation_dataframe[
    "csv_company"
] = [
    get_raw_csv_value(
        source_record_number,
        "company",
    )
    for source_record_number
    in validation_dataframe[
        "source_record_number"
    ]
]


# 取得 CSV position。
validation_dataframe[
    "csv_position"
] = [
    get_raw_csv_value(
        source_record_number,
        "position",
    )
    for source_record_number
    in validation_dataframe[
        "source_record_number"
    ]
]


# ============================================================
# 比较 D1 与原始 CSV
# ============================================================

# 比较 company 原始值是否完全相同。
validation_dataframe[
    "company_exact_match"
] = (
    validation_dataframe[
        "d1_company"
    ]
    ==
    validation_dataframe[
        "csv_company"
    ]
)


# 比较去除首尾空格后的 company。
validation_dataframe[
    "company_trimmed_match"
] = (
    validation_dataframe[
        "d1_company"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    ==
    validation_dataframe[
        "csv_company"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


# 比较 position 原始值是否完全相同。
validation_dataframe[
    "position_exact_match"
] = (
    validation_dataframe[
        "d1_position"
    ]
    ==
    validation_dataframe[
        "csv_position"
    ]
)


# 比较去除首尾空格后的 position。
validation_dataframe[
    "position_trimmed_match"
] = (
    validation_dataframe[
        "d1_position"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    ==
    validation_dataframe[
        "csv_position"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


# 生成准备写入 Application 的 company。
#
# 保留原始大小写，只去除首尾空格。
validation_dataframe[
    "applied_company_name"
] = (
    validation_dataframe[
        "d1_company"
    ]
    .astype("string")
    .str.strip()
)


# 生成准备写入 Application 的 position。
validation_dataframe[
    "applied_position_name"
] = (
    validation_dataframe[
        "d1_position"
    ]
    .astype("string")
    .str.strip()
)


# 取得准备写入 Application 的 cluster display name。
validation_dataframe[
    "application_cluster_display_name"
] = (
    validation_dataframe[
        "application_cluster__display_name"
    ]
    .astype("string")
    .str.strip()
)


# ============================================================
# 生成每条记录的验证状态
# ============================================================

# 默认所有记录通过验证。
validation_dataframe[
    "validation_status"
] = "validated"


# 原始封装结构不正确时标记失败。
validation_dataframe.loc[
    validation_dataframe[
        "parse_structure_valid"
    ] != 1,
    "validation_status",
] = "invalid_raw_record_structure"


# company 去除首尾空格后不一致时标记失败。
validation_dataframe.loc[
    ~validation_dataframe[
        "company_trimmed_match"
    ],
    "validation_status",
] = "company_mismatch"


# position 去除首尾空格后不一致时标记失败。
validation_dataframe.loc[
    ~validation_dataframe[
        "position_trimmed_match"
    ],
    "validation_status",
] = "position_mismatch"


# cluster display name 为空时标记失败。
cluster_name_missing = (
    validation_dataframe[
        "application_cluster_display_name"
    ].isna()
    |
    validation_dataframe[
        "application_cluster_display_name"
    ].eq("")
)


# 保存 cluster 缺失状态。
validation_dataframe.loc[
    cluster_name_missing,
    "validation_status",
] = "cluster_display_name_missing"


# ============================================================
# 生成验证汇总
# ============================================================

# 这里先对布尔 Series 执行 sum，
# 再把结果转换成 Python int。
validation_summary = {
    "total_records": int(
        len(validation_dataframe)
    ),

    "valid_structure_count": int(
        validation_dataframe[
            "parse_structure_valid"
        ]
        .eq(1)
        .sum()
    ),

    "company_exact_match_count": int(
        validation_dataframe[
            "company_exact_match"
        ]
        .sum()
    ),

    "company_trimmed_match_count": int(
        validation_dataframe[
            "company_trimmed_match"
        ]
        .sum()
    ),

    "position_exact_match_count": int(
        validation_dataframe[
            "position_exact_match"
        ]
        .sum()
    ),

    "position_trimmed_match_count": int(
        validation_dataframe[
            "position_trimmed_match"
        ]
        .sum()
    ),

    "validated_count": int(
        validation_dataframe[
            "validation_status"
        ]
        .eq("validated")
        .sum()
    ),

    "failed_validation_count": int(
        (
            ~validation_dataframe[
                "validation_status"
            ]
            .eq("validated")
        )
        .sum()
    ),
}


# ============================================================
# 显示验证结果
# ============================================================

# 显示汇总。
print(
    "Company and position validation summary:"
)

print(
    validation_summary
)


# 显示前十条验证记录。
display_columns = [
    "id",
    "raw_submission_id",
    "source_record_number",
    "d1_company",
    "csv_company",
    "company_exact_match",
    "d1_position",
    "csv_position",
    "position_exact_match",
    "application_cluster_display_name",
    "validation_status",
]


display(
    validation_dataframe[
        display_columns
    ]
    .head(10)
)


# 取得验证失败记录。
failed_validation_dataframe = (
    validation_dataframe[
        validation_dataframe[
            "validation_status"
        ]
        != "validated"
    ]
    .copy()
)


# 存在失败记录时显示并停止。
if not failed_validation_dataframe.empty:
    display(
        failed_validation_dataframe[
            display_columns
        ]
    )

    raise ValueError(
        "存在 company、position 或 cluster "
        "验证失败记录，暂时不能回填 Application。"
    )


# 显示全部验证通过。
print(
    "All Application company, position, and "
    "cluster display values are validated."
)

## 真正的 Candidate & Person 层数据导入代码

### Application 表新增三个字段的数据导入

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 验证并回填 Application 的三个快照字段：
#
# 1. applied_company_name
# 2. applied_position_name
# 3. application_cluster_display_name
#
# 数据来源：
# - company / position：
#   D1 raw_submission.raw_record_jsonb
# - 双重核对：
#   Google Drive raw_submission.csv
# - cluster display name：
#   Google Drive application.csv
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Cell。
# 2. 写入前会重新执行全部关键验证。
# 3. 只更新与计划值不同的 Application。
# 4. 重复运行不会创建新记录，也不会改变已经一致的记录。
# 5. 网络中断后可以安全地重新运行。
# ============================================================

import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from google.colab import drive, userdata


# ============================================================
# Google Drive 文件路径
# ============================================================

drive.mount("/content/drive")

DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 每次运行时都要求用户明确输入本批次实际使用的原始 CSV 文件名。
#
# 这样后续无论新批次包含 10 条、100 条还是更多记录，
# 都不需要重新修改 Cell 中的固定文件名。
raw_csv_file_name = input(
    "请输入本批次原始 CSV 文件名（例如 test_raw_submission_10.csv）："
).strip()

# 没有输入文件名时立即停止，防止意外读取旧批次文件。
if not raw_csv_file_name:
    raise ValueError(
        "本批次原始 CSV 文件名不能为空。"
    )

# 这里只接受文件名，不接受包含目录分隔符的路径，
# 防止输入内容跳出固定 datasets 文件夹。
if os.path.basename(raw_csv_file_name) != raw_csv_file_name:
    raise ValueError(
        "请输入单独的 CSV 文件名，不要输入完整路径。"
    )

# 本 Cell 只接受 CSV 文件。
if not raw_csv_file_name.lower().endswith(".csv"):
    raise ValueError(
        "本批次原始数据文件必须是 .csv 文件。"
    )

# 根据用户输入动态生成本批次原始 CSV 的完整路径。
RAW_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/{raw_csv_file_name}"
)

APPLICATION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/dedup_results/"
    "application_dedup_v2/application.csv"
)

for required_path in [
    RAW_CSV_PATH,
    APPLICATION_CSV_PATH,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            f"找不到必要输入文件：{required_path}"
        )


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")

if not API_TOKEN:
    raise ValueError("未找到 CLOUDFLARE_KEY。")

D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用同一个 HTTP session，减少大量请求时的连接开销。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行一条 D1 SQL；网络或临时请求失败时有限重试。"""

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            last_error = error
            if attempt < max_retries:
                time.sleep(1.0)
                continue
            raise RuntimeError(
                f"无法连接 Cloudflare D1：{last_error}"
            ) from error

        try:
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        if (
            response.status_code == 200
            and payload.get("success", False)
        ):
            return payload.get("result", [])

        last_error = payload.get("errors", payload)
        if attempt < max_retries:
            time.sleep(1.0)

    raise RuntimeError(
        f"D1 SQL 在重试后仍然失败：{last_error}"
    )


# ============================================================
# 读取并筛选 application.csv
# ============================================================

application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
)

application_dataframe.columns = [
    str(column_name).strip()
    for column_name in application_dataframe.columns
]

required_application_columns = [
    "id",
    "raw_submission_id",
    "source_record_number",
    "application_cluster__display_name",
]

missing_application_columns = [
    column_name
    for column_name in required_application_columns
    if column_name not in application_dataframe.columns
]

if missing_application_columns:
    raise ValueError(
        "application.csv 缺少必要字段："
        f"{missing_application_columns}"
    )

application_dataframe["id"] = pd.to_numeric(
    application_dataframe["id"],
    errors="coerce",
)

# 只处理已经拥有 D1 Application.id 的记录。
application_dataframe = (
    application_dataframe[
        application_dataframe["id"].notna()
    ]
    .copy()
)

# 当前增量 CSV 中必须至少存在一条已经成功写入 D1 Application、
# 因而拥有 Application.id 的记录。
if application_dataframe.empty:
    raise ValueError(
        "当前 application.csv 中没有可处理的 Application.id。"
    )

for column_name in [
    "id",
    "raw_submission_id",
    "source_record_number",
]:
    application_dataframe[column_name] = (
        pd.to_numeric(
            application_dataframe[column_name],
            errors="raise",
        )
        .astype("int64")
    )

if application_dataframe["id"].duplicated().any():
    raise ValueError(
        "application.csv 中存在重复 Application.id。"
    )

if application_dataframe["raw_submission_id"].duplicated().any():
    raise ValueError(
        "application.csv 中存在重复 raw_submission_id。"
    )


# ============================================================
# 读取原始 raw_submission.csv
# ============================================================

raw_csv_dataframe = pd.read_csv(
    RAW_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)

for required_column in [
    "company",
    "position",
]:
    if required_column not in raw_csv_dataframe.columns:
        raise ValueError(
            "raw_submission.csv 缺少必要字段："
            f"{required_column}"
        )


# ============================================================
# 从 D1 UTF-8 文本中解析 company 和 position
# ============================================================

raw_submission_ids = (
    application_dataframe["raw_submission_id"].tolist()
)

parsed_d1_rows = []
chunk_size = 50

for chunk_start in range(
    0,
    len(raw_submission_ids),
    chunk_size,
):
    chunk_ids = raw_submission_ids[
        chunk_start:
        chunk_start + chunk_size
    ]
    placeholders = ",".join(["?"] * len(chunk_ids))

    # company 是稳定封装文本的第一行。
    # position 位于 resume_text 之前，避免误取简历正文。
    sql = (
        'WITH "selected" AS ('
        'SELECT '
        '"id", '
        '"source_record_number", '
        '"raw_record_jsonb", '
        'instr("raw_record_jsonb", char(10)) AS "first_newline", '
        'instr("raw_record_jsonb", char(10) || \'position: \') '
        'AS "position_marker", '
        'instr("raw_record_jsonb", char(10) || \'resume_text: \') '
        'AS "resume_marker" '
        'FROM "raw_submission" '
        f'WHERE "id" IN ({placeholders})'
        ') '
        'SELECT '
        '"id" AS "d1_raw_submission_id", '
        '"source_record_number" AS "d1_source_record_number", '
        'CASE '
        'WHEN substr("raw_record_jsonb", 1, 9) = \'company: \' '
        'AND "first_newline" > 9 '
        'THEN substr("raw_record_jsonb", 10, "first_newline" - 10) '
        'ELSE NULL END AS "d1_company", '
        'CASE '
        'WHEN "position_marker" > 0 '
        'AND "resume_marker" > "position_marker" '
        'AND instr(substr("raw_record_jsonb", '
        '"position_marker" + 11), char(10)) > 0 '
        'THEN substr('
        '"raw_record_jsonb", '
        '"position_marker" + 11, '
        'instr(substr("raw_record_jsonb", '
        '"position_marker" + 11), char(10)) - 1'
        ') '
        'ELSE NULL END AS "d1_position", '
        'CASE '
        'WHEN substr("raw_record_jsonb", 1, 9) = \'company: \' '
        'AND "first_newline" > 9 '
        'AND "position_marker" > 0 '
        'AND "resume_marker" > "position_marker" '
        'THEN 1 ELSE 0 END AS "parse_structure_valid" '
        'FROM "selected";'
    )

    result = execute_d1(
        sql,
        [int(value) for value in chunk_ids],
    )
    rows = (
        result[0].get("results", [])
        if result
        else []
    )
    parsed_d1_rows.extend(rows)

    processed_count = min(
        chunk_start + chunk_size,
        len(raw_submission_ids),
    )
    if (
        processed_count % 200 == 0
        or processed_count == len(raw_submission_ids)
    ):
        print(
            "Validated D1 raw submissions:",
            f"{processed_count}/{len(raw_submission_ids)}",
        )

parsed_d1_dataframe = pd.DataFrame(parsed_d1_rows)

if len(parsed_d1_dataframe) != len(application_dataframe):
    raise ValueError(
        "D1 raw_submission 数量与 Application 数量不一致。"
    )

for column_name in [
    "d1_raw_submission_id",
    "d1_source_record_number",
    "parse_structure_valid",
]:
    parsed_d1_dataframe[column_name] = (
        pd.to_numeric(
            parsed_d1_dataframe[column_name],
            errors="raise",
        )
        .astype("int64")
    )


# ============================================================
# 建立并验证回填计划
# ============================================================

backfill_dataframe = (
    application_dataframe[
        [
            "id",
            "raw_submission_id",
            "source_record_number",
            "application_cluster__display_name",
        ]
    ]
    .merge(
        parsed_d1_dataframe,
        left_on="raw_submission_id",
        right_on="d1_raw_submission_id",
        how="left",
        validate="one_to_one",
    )
)

backfill_dataframe["csv_company"] = [
    raw_csv_dataframe.iloc[int(record_number) - 1]["company"]
    for record_number
    in backfill_dataframe["source_record_number"]
]

backfill_dataframe["csv_position"] = [
    raw_csv_dataframe.iloc[int(record_number) - 1]["position"]
    for record_number
    in backfill_dataframe["source_record_number"]
]

backfill_dataframe["applied_company_name"] = (
    backfill_dataframe["d1_company"]
    .astype("string")
    .str.strip()
)

backfill_dataframe["applied_position_name"] = (
    backfill_dataframe["d1_position"]
    .astype("string")
    .str.strip()
)

backfill_dataframe["application_cluster_display_name"] = (
    backfill_dataframe[
        "application_cluster__display_name"
    ]
    .astype("string")
    .str.strip()
)

source_number_valid = (
    backfill_dataframe["source_record_number"]
    == backfill_dataframe["d1_source_record_number"]
)

company_valid = (
    backfill_dataframe["d1_company"].fillna("").astype(str)
    == backfill_dataframe["csv_company"].fillna("").astype(str)
)

position_valid = (
    backfill_dataframe["d1_position"].fillna("").astype(str)
    == backfill_dataframe["csv_position"].fillna("").astype(str)
)

cluster_valid = (
    backfill_dataframe["application_cluster_display_name"]
    .notna()
    & backfill_dataframe["application_cluster_display_name"].ne("")
)

all_rows_valid = (
    backfill_dataframe["parse_structure_valid"].eq(1)
    & source_number_valid
    & company_valid
    & position_valid
    & cluster_valid
)

if not all_rows_valid.all():
    failed_rows = backfill_dataframe.loc[
        ~all_rows_valid
    ].copy()
    display(failed_rows)
    raise ValueError(
        "回填前验证失败，Application 尚未更新。"
    )


# ============================================================
# 验证 Application ID 集合并找出真正需要更新的记录
# ============================================================

# 取得当前增量 application.csv 中的全部 Application ID。
current_batch_application_ids = (
    backfill_dataframe["id"]
    .astype("int64")
    .tolist()
)

# 每次最多查询 50 个 Application ID，避免未来批次较大时
# 单条 SQL 的参数数量或请求内容过大。
APPLICATION_QUERY_CHUNK_SIZE = 50


def load_current_application_fields(application_ids):
    """分块读取指定 Application 的三个待回填字段。"""

    # 保存所有分块查询返回的记录。
    selected_application_rows = []

    # 按固定大小逐块读取 Application。
    for chunk_start in range(
        0,
        len(application_ids),
        APPLICATION_QUERY_CHUNK_SIZE,
    ):
        # 取得当前分块的 Application ID。
        chunk_application_ids = application_ids[
            chunk_start:
            chunk_start + APPLICATION_QUERY_CHUNK_SIZE
        ]

        # 根据当前分块数量动态生成参数占位符。
        placeholders = ",".join(
            ["?"] * len(chunk_application_ids)
        )

        # 只读取当前增量批次对应的 Application，
        # 不扫描并返回以前批次的全部 Application 记录。
        selected_result = execute_d1(
            'SELECT '
            '"id", '
            '"applied_company_name", '
            '"applied_position_name", '
            '"application_cluster_display_name" '
            'FROM "Application" '
            f'WHERE "id" IN ({placeholders}) '
            'ORDER BY "id";',
            [
                int(application_id)
                for application_id
                in chunk_application_ids
            ],
        )

        # 提取当前分块返回的 D1 records。
        selected_rows = (
            selected_result[0].get("results", [])
            if selected_result
            else []
        )

        # 合并当前分块结果。
        selected_application_rows.extend(
            selected_rows
        )

    # 返回当前增量批次的全部查询结果。
    return selected_application_rows


# 只读取当前增量 CSV 中的 Application。
current_rows = load_current_application_fields(
    current_batch_application_ids
)

current_application_dataframe = pd.DataFrame(current_rows)
current_application_dataframe["id"] = (
    pd.to_numeric(
        current_application_dataframe["id"],
        errors="raise",
    )
    .astype("int64")
)

if set(current_application_dataframe["id"]) != set(
    backfill_dataframe["id"]
):
    raise ValueError(
        "当前增量 application.csv 中存在尚未写入 "
        "D1 Application 的 ID。"
    )

backfill_dataframe = backfill_dataframe.merge(
    current_application_dataframe,
    on="id",
    how="left",
    validate="one_to_one",
    suffixes=("_planned", "_current"),
)

different_mask = (
    backfill_dataframe[
        "applied_company_name_current"
    ].fillna("").astype(str)
    != backfill_dataframe[
        "applied_company_name_planned"
    ].fillna("").astype(str)
) | (
    backfill_dataframe[
        "applied_position_name_current"
    ].fillna("").astype(str)
    != backfill_dataframe[
        "applied_position_name_planned"
    ].fillna("").astype(str)
) | (
    backfill_dataframe[
        "application_cluster_display_name_current"
    ].fillna("").astype(str)
    != backfill_dataframe[
        "application_cluster_display_name_planned"
    ].fillna("").astype(str)
)

records_to_update = (
    backfill_dataframe.loc[different_mask].copy()
)

print(
    "Validated Application records:",
    len(backfill_dataframe),
)
print(
    "Application records requiring update:",
    len(records_to_update),
)


# ============================================================
# 更新不同的 Application 记录
# ============================================================

updated_at = datetime.now(timezone.utc).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

update_sql = (
    'UPDATE "Application" '
    'SET "applied_company_name" = ?, '
    '"applied_position_name" = ?, '
    '"application_cluster_display_name" = ?, '
    '"updated_at" = ? '
    'WHERE "id" = ?;'
)

for update_number, row in enumerate(
    records_to_update.itertuples(index=False),
    start=1,
):
    execute_d1(
        update_sql,
        [
            str(row.applied_company_name_planned),
            str(row.applied_position_name_planned),
            str(row.application_cluster_display_name_planned),
            updated_at,
            int(row.id),
        ],
    )

    if (
        update_number % 100 == 0
        or update_number == len(records_to_update)
    ):
        print(
            "Updated Application records:",
            f"{update_number}/{len(records_to_update)}",
        )


# ============================================================
# 更新后重新从 D1 验证当前增量批次结果
# ============================================================

# 更新完成后，只重新读取当前增量 CSV 对应的 Application。
verification_rows = load_current_application_fields(
    current_batch_application_ids
)
verification_dataframe = pd.DataFrame(verification_rows)
verification_dataframe["id"] = (
    pd.to_numeric(
        verification_dataframe["id"],
        errors="raise",
    )
    .astype("int64")
)

planned_values = backfill_dataframe[
    [
        "id",
        "applied_company_name_planned",
        "applied_position_name_planned",
        "application_cluster_display_name_planned",
    ]
].copy()

final_check = planned_values.merge(
    verification_dataframe,
    on="id",
    how="left",
    validate="one_to_one",
)

final_values_match = (
    final_check["applied_company_name_planned"]
    .fillna("").astype(str)
    == final_check["applied_company_name"]
    .fillna("").astype(str)
) & (
    final_check["applied_position_name_planned"]
    .fillna("").astype(str)
    == final_check["applied_position_name"]
    .fillna("").astype(str)
) & (
    final_check["application_cluster_display_name_planned"]
    .fillna("").astype(str)
    == final_check["application_cluster_display_name"]
    .fillna("").astype(str)
)

if not final_values_match.all():
    display(final_check.loc[~final_values_match])
    raise RuntimeError(
        "Application 回填后的数据库验证失败。"
    )

foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)
foreign_key_violations = (
    foreign_key_result[0].get("results", [])
    if foreign_key_result
    else []
)

if foreign_key_violations:
    raise RuntimeError(
        f"数据库存在外键违规：{foreign_key_violations}"
    )

print(
    "Application backfill summary:",
    {
        "validated_records": int(len(backfill_dataframe)),
        "updated_records": int(len(records_to_update)),
        "verified_records": int(final_values_match.sum()),
        "foreign_key_violation_count": int(
            len(foreign_key_violations)
        ),
    },
)

display(
    verification_dataframe.head(10)
)

print(
    "Application field backfill completed successfully."
)


### Candidate snapshot 表数据导入

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 从 Google Drive application.csv 生成并写入 Candidate_snapshot。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell 中的变量、function 或 class。
# 2. 本 Cell 不修改 Application，也不创建新的 Person。
# 3. Candidate_snapshot.application_id 已有 UNIQUE 约束。
# 4. 不存在的 Candidate_snapshot 才执行 INSERT。
# 5. 已存在但内容不同的 Candidate_snapshot 才执行 UPDATE。
# 6. 已存在且内容完全相同的记录直接跳过。
# 7. 因此本 Cell 可以重复运行，不会产生重复 Candidate_snapshot。
# 8. 网络中断后可以安全重新运行，已经完成的记录会被复用。
# 9. 姓名和邮箱使用此前 Submission_normalized 的标准化结果，
#    不会在本 Cell 中重新执行姓名或邮箱 normalization。
# 10. 电话、LinkedIn 和 GitHub 使用查重阶段生成的标准化身份特征。
# ============================================================

# os 用于检查 Google Drive 文件是否存在。
import os

# json 用于解析 application.csv 中的身份特征 JSON。
import json

# math 用于验证 CSV 中的数字是否为有限整数。
import math

# re 用于识别邮箱、电话号码格式和 GitHub handle。
import re

# time 只用于 D1 请求失败后的短暂重试等待。
import time

# datetime 用于生成 UTC created_at 和 updated_at。
from datetime import datetime, timezone

# pandas 用于读取 CSV 和建立预览 DataFrame。
import pandas as pd

# requests 用于调用 Cloudflare D1 API。
import requests

# drive 用于挂载 Drive；userdata 用于读取 Colab Secret。
from google.colab import drive, userdata


# ============================================================
# Google Drive 文件路径
# ============================================================

# 挂载 Google Drive。
# 已挂载时只会显示 already mounted，不会修改文件。
drive.mount(
    "/content/drive"
)

# 保存 Google Drive 项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 本 Cell 唯一读取的业务输入文件。
APPLICATION_CSV_PATH = (
    f"{DRIVE_FOLDER}/datasets/dedup_results/"
    "application_dedup_v2/application.csv"
)

# 输入文件不存在时停止。
if not os.path.exists(
    APPLICATION_CSV_PATH
):
    raise FileNotFoundError(
        "找不到必要输入文件："
        f"{APPLICATION_CSV_PATH}"
    )


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 Cloudflare D1 query endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session，减少逐条写入时的连接开销。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在 max_retries 次尝试中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 SQL 和标准 Python 参数。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部网络请求失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            # 非 JSON response 无法可靠继续。
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 D1 query 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 SQL 或 API 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败后停止，避免静默漏写。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def utc_now_text():
    """返回统一的 UTC ISO timestamp。"""

    # 生成不包含微秒的 UTC 时间文本。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_text_value(
    value,
):
    """把 pandas 空值转成 None，其余 scalar 转成去除首尾空格的文本。"""

    # Python None 直接对应 SQL NULL。
    if value is None:
        return None

    # pandas NA 和 NumPy NaN 也转换成 SQL NULL。
    try:
        if pd.isna(
            value
        ):
            return None
    except (TypeError, ValueError):
        # list/dict 不适合执行单值 pd.isna 判断。
        pass

    # 转成标准 Python string。
    cleaned_value = str(
        value
    ).strip()

    # 空 string 写成 SQL NULL。
    if not cleaned_value:
        return None

    # 防止文本化的 pandas 空值进入数据库。
    if cleaned_value.lower() in {
        "nan",
        "none",
        "<na>",
    }:
        return None

    # 不改变业务文本的大小写。
    return cleaned_value


def normalize_integer(
    value,
    field_name,
):
    """把 CSV integer 安全转换成标准 Python int。"""

    # 先转换空值。
    cleaned_value = clean_text_value(
        value
    )

    # 必需 id 不允许为空。
    if cleaned_value is None:
        raise ValueError(
            f"{field_name} 不能为空。"
        )

    try:
        # float 中转可以兼容 CSV 中的 1.0。
        numeric_value = float(
            cleaned_value
        )
    except ValueError as error:
        raise ValueError(
            f"{field_name} 不是有效数字：{value}"
        ) from error

    # NaN 和 infinity 不能成为数据库 id。
    if not math.isfinite(
        numeric_value
    ):
        raise ValueError(
            f"{field_name} 不是有限数字：{value}"
        )

    # 不允许把 1.5 静默截断成 1。
    if not numeric_value.is_integer():
        raise ValueError(
            f"{field_name} 不是整数：{value}"
        )

    # 显式变成 Python int，避免 numpy.int64 JSON serialization error。
    return int(
        numeric_value
    )


def build_normalized_email_address(
    email_prefix,
    email_domain,
):
    """使用已经标准化的 prefix/domain 组成完整邮箱。"""

    # 读取此前 normalization cell 生成的 prefix。
    normalized_prefix = clean_text_value(
        email_prefix
    )

    # 读取此前 normalization cell 生成的 domain。
    normalized_domain = clean_text_value(
        email_domain
    )

    # 任意部分为空时，完整邮箱保持 NULL。
    if (
        normalized_prefix is None
        or normalized_domain is None
    ):
        return None

    # 不重新标准化，只连接已经标准化的两部分。
    return (
        f"{normalized_prefix}@"
        f"{normalized_domain}"
    )


def parse_identity_feature_records(
    json_text,
    submission_normalized_id,
):
    """
    只提取属于当前 submission_normalized_id 的可用身份特征。

    同一种 feature_type 的多个不同值都会按照 JSON 原始稳定顺序保留，
    后续再由电话或 GitHub 的专用规则选择可靠的单值。
    """

    # 空 JSON 代表没有查重身份特征。
    cleaned_json_text = clean_text_value(
        json_text
    )

    # 返回空 dictionary。
    if cleaned_json_text is None:
        return {}

    try:
        # JSON array 转成 Python list。
        feature_records = json.loads(
            cleaned_json_text
        )
    except json.JSONDecodeError as error:
        raise ValueError(
            "submission_identity_feature_records_json "
            "不是有效 JSON。"
        ) from error

    # 顶层必须是 list。
    if not isinstance(
        feature_records,
        list,
    ):
        raise ValueError(
            "submission_identity_feature_records_json "
            "顶层结构不是 list。"
        )

    # 保存当前 submission 的特征 list。
    current_submission_features = {}

    # 逐条检查 feature record。
    for feature_record in feature_records:
        # 非 dictionary 元素不能使用。
        if not isinstance(
            feature_record,
            dict,
        ):
            continue

        # 读取 feature 所属 submission id。
        feature_submission_id_value = feature_record.get(
            "submission_normalized_id"
        )

        # 没有所属 id 时跳过。
        if feature_submission_id_value is None:
            continue

        try:
            # 转成标准 Python int。
            feature_submission_id = int(
                feature_submission_id_value
            )
        except (TypeError, ValueError):
            continue

        # JSON 可能包含 matched submissions。
        # 这里只保留当前 application 自己的 submission。
        if (
            feature_submission_id
            != submission_normalized_id
        ):
            continue

        try:
            # 只有 is_usable=1 的特征可以使用。
            is_usable = int(
                feature_record.get(
                    "is_usable",
                    0,
                )
            )
        except (TypeError, ValueError):
            is_usable = 0

        # 跳过不可用特征。
        if is_usable != 1:
            continue

        # 读取 feature type。
        feature_type = clean_text_value(
            feature_record.get(
                "feature_type"
            )
        )

        # 读取 normalized value。
        normalized_value = clean_text_value(
            feature_record.get(
                "normalized_value"
            )
        )

        # type 或 value 为空时跳过。
        if (
            feature_type is None
            or normalized_value is None
        ):
            continue

        # type 转成小写 key。
        normalized_feature_type = (
            feature_type.lower()
        )

        # 为当前 feature type 建立 list。
        feature_values = (
            current_submission_features.setdefault(
                normalized_feature_type,
                [],
            )
        )

        # 相同值只保存一次。
        if normalized_value not in feature_values:
            feature_values.append(
                normalized_value
            )

    # 返回每种 type 对应的唯一值 list。
    return current_submission_features


def normalize_phone_last_ten_digits(
    phone_text,
):
    """按照当前项目规则，把可靠的美国电话号码标准化成最后10位。"""

    # 空电话无法标准化。
    cleaned_phone_text = clean_text_value(
        phone_text
    )

    # 空值返回 None。
    if cleaned_phone_text is None:
        return None

    # 只保留数字。
    phone_digits = re.sub(
        r"\D",
        "",
        cleaned_phone_text,
    )

    # 标准美国本地号码必须是10位。
    if len(
        phone_digits
    ) == 10:
        return phone_digits

    # 带美国国家码1时允许11位，并去掉开头1。
    if (
        len(
            phone_digits
        ) == 11
        and phone_digits.startswith(
            "1"
        )
    ):
        return phone_digits[-10:]

    # 其他长度不按美国电话规则强行截断。
    return None


def select_reliable_phone(
    raw_phone_text,
    resume_text,
    raw_email_address,
    normalized_phone_candidates,
):
    """
    从简历电话候选中选择可靠的主要美国电话号码。

    优先级：
    1. 明确带 +1；
    2. 带括号的区号；
    3. 使用空格或连字符分隔的10位号码；
    4. 单独出现的10位纯数字。

    出现在邮箱地址中的数字候选会被排除。
    最高分存在并列不同号码时返回 NULL，不自动猜测。
    """

    # 清理简历提取的原始电话集合文本。
    cleaned_raw_phone_text = clean_text_value(
        raw_phone_text
    )

    # 没有原始电话候选时无法可靠选择。
    if cleaned_raw_phone_text is None:
        return None, None, "no_raw_phone_candidate"

    # 建立当前 submission 的可用标准化电话 set。
    normalized_candidate_set = {
        str(
            candidate
        ).strip()
        for candidate
        in normalized_phone_candidates
        if clean_text_value(
            candidate
        )
        is not None
    }

    # 收集表单邮箱和简历正文中的邮箱。
    email_texts = []

    # 加入原始表单邮箱。
    cleaned_raw_email = clean_text_value(
        raw_email_address
    )

    # 原始邮箱存在时保存。
    if cleaned_raw_email is not None:
        email_texts.append(
            cleaned_raw_email.lower()
        )

    # 读取简历正文。
    cleaned_resume_text = clean_text_value(
        resume_text
    ) or ""

    # 提取简历正文中的邮箱地址。
    email_texts.extend(
        email_match.lower()
        for email_match
        in re.findall(
            r"[A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,}",
            cleaned_resume_text,
            flags=re.IGNORECASE,
        )
    )

    # 保存全部带评分的可靠候选。
    scored_phone_candidates = []

    # CV extraction 当前使用分号连接多个原始电话候选。
    raw_phone_candidates = [
        phone_candidate.strip()
        for phone_candidate
        in cleaned_raw_phone_text.split(
            ";"
        )
        if phone_candidate.strip()
    ]

    # 逐个检查电话候选。
    for candidate_index, phone_candidate in enumerate(
        raw_phone_candidates
    ):
        # 根据美国电话规则标准化。
        normalized_phone = (
            normalize_phone_last_ten_digits(
                phone_candidate
            )
        )

        # 非10位美国号码候选不使用。
        if normalized_phone is None:
            continue

        # 如果查重身份特征存在电话，
        # 当前候选必须也存在于该特征集合中。
        if (
            normalized_candidate_set
            and normalized_phone
            not in normalized_candidate_set
        ):
            continue

        # 候选数字如果属于某个邮箱文本，按误识别排除。
        if any(
            normalized_phone
            in re.sub(
                r"\D",
                "",
                email_text,
            )
            for email_text
            in email_texts
        ):
            continue

        # 默认给独立10位纯数字最低可靠分。
        candidate_score = 40

        # 明确的美国国家码 +1 可靠性最高。
        if re.search(
            r"(?<!\d)\+1(?:\D|$)",
            phone_candidate,
        ):
            candidate_score = 100

        # 带括号区号通常是明确的电话号码格式。
        elif re.search(
            r"\(\s*\d{3}\s*\)",
            phone_candidate,
        ):
            candidate_score = 90

        # 10位数字之间有空格、连字符或点号时也具有电话格式。
        elif re.search(
            r"\d{3}[\s.\-]+\d{3}[\s.\-]+\d{4}",
            phone_candidate,
        ):
            candidate_score = 80

        # 保存分数、原顺序、原始值和标准化值。
        scored_phone_candidates.append(
            {
                "score": candidate_score,
                "candidate_index": candidate_index,
                "raw_phone": phone_candidate,
                "normalized_phone": normalized_phone,
            }
        )

    # 没有通过过滤的候选时保持 NULL。
    if not scored_phone_candidates:
        return None, None, "no_reliable_phone_candidate"

    # 找出最高评分。
    highest_score = max(
        candidate[
            "score"
        ]
        for candidate
        in scored_phone_candidates
    )

    # 取得所有最高分候选。
    highest_scored_candidates = [
        candidate
        for candidate
        in scored_phone_candidates
        if candidate[
            "score"
        ]
        == highest_score
    ]

    # 最高分对应多个不同号码时保持 NULL。
    highest_normalized_values = {
        candidate[
            "normalized_phone"
        ]
        for candidate
        in highest_scored_candidates
    }

    # 不自动猜测并列候选。
    if len(
        highest_normalized_values
    ) != 1:
        return None, None, "ambiguous_phone_candidates"

    # 相同号码重复出现时按照原始顺序选择第一个。
    selected_candidate = sorted(
        highest_scored_candidates,
        key=lambda candidate: candidate[
            "candidate_index"
        ],
    )[0]

    # 返回原始电话、标准化电话和选择状态。
    return (
        selected_candidate[
            "raw_phone"
        ],
        selected_candidate[
            "normalized_phone"
        ],
        "selected",
    )


def normalize_name_token(
    value,
):
    """把姓名或邮箱 prefix 转成只包含小写字母和数字的比较 token。"""

    # 空值转换成空 token。
    cleaned_value = clean_text_value(
        value
    )

    # 空值返回空 string。
    if cleaned_value is None:
        return ""

    # 删除空格、标点和其他非字母数字字符。
    return re.sub(
        r"[^a-z0-9]",
        "",
        cleaned_value.lower(),
    )


def github_handle_from_url(
    github_url,
):
    """从标准化 GitHub URL 中取得第一个 path segment 作为 handle。"""

    # 清理 URL。
    cleaned_url = clean_text_value(
        github_url
    )

    # 空 URL 没有 handle。
    if cleaned_url is None:
        return None

    # 匹配 github.com 后面的第一个 path segment。
    handle_match = re.search(
        r"(?:https?://)?(?:www\.)?github\.com/([^/?#]+)",
        cleaned_url,
        flags=re.IGNORECASE,
    )

    # 不符合 GitHub URL 结构时返回 None。
    if handle_match is None:
        return None

    # 返回标准化小写 handle。
    return normalize_name_token(
        handle_match.group(
            1
        )
    )


def select_reliable_github_url(
    github_candidates,
    normalized_email_prefix,
    normalized_first_name,
    normalized_last_name,
):
    """
    选择可靠的个人 GitHub URL。

    单一候选直接使用。
    多个候选时，优先选择 handle 与邮箱 prefix 或标准化姓名相符的 URL。
    无法得到唯一最高分时返回 NULL。
    """

    # 按原始顺序去除重复 URL。
    unique_candidates = []

    # 逐个处理候选。
    for github_candidate in github_candidates:
        # 清理 URL。
        cleaned_candidate = clean_text_value(
            github_candidate
        )

        # 跳过空值和完全重复值。
        if (
            cleaned_candidate is not None
            and cleaned_candidate
            not in unique_candidates
        ):
            unique_candidates.append(
                cleaned_candidate
            )

    # 没有候选时保持 NULL。
    if not unique_candidates:
        return None, "no_github_candidate"

    # 唯一候选不需要在多个 URL 中判断。
    if len(
        unique_candidates
    ) == 1:
        return unique_candidates[0], "single_candidate"

    # 生成比较 tokens。
    email_prefix_token = normalize_name_token(
        normalized_email_prefix
    )

    # 生成 first name token。
    first_name_token = normalize_name_token(
        normalized_first_name
    )

    # 生成 last name token。
    last_name_token = normalize_name_token(
        normalized_last_name
    )

    # 生成 first+last 和 last+first 组合。
    first_last_token = (
        first_name_token
        + last_name_token
    )

    # 生成 last+first 组合。
    last_first_token = (
        last_name_token
        + first_name_token
    )

    # 保存带评分 GitHub 候选。
    scored_candidates = []

    # 逐个计算个人身份相关性。
    for candidate_index, github_candidate in enumerate(
        unique_candidates
    ):
        # 取得 GitHub handle。
        github_handle = github_handle_from_url(
            github_candidate
        )

        # 无有效 handle 时得分为0。
        candidate_score = 0

        # handle 与邮箱 prefix 完全相同，可靠性最高。
        if (
            github_handle
            and email_prefix_token
            and github_handle
            == email_prefix_token
        ):
            candidate_score = 120

        # 长度至少4时，handle 与邮箱 prefix 互相包含。
        elif (
            github_handle
            and email_prefix_token
            and min(
                len(
                    github_handle
                ),
                len(
                    email_prefix_token
                ),
            )
            >= 4
            and (
                github_handle
                in email_prefix_token
                or email_prefix_token
                in github_handle
            )
        ):
            candidate_score = 110

        # handle 包含完整 first+last 或 last+first。
        elif (
            github_handle
            and (
                (
                    first_last_token
                    and first_last_token
                    in github_handle
                )
                or (
                    last_first_token
                    and last_first_token
                    in github_handle
                )
            )
        ):
            candidate_score = 100

        # first name 和 last name 都分别出现在 handle 中。
        elif (
            github_handle
            and first_name_token
            and last_name_token
            and first_name_token
            in github_handle
            and last_name_token
            in github_handle
        ):
            candidate_score = 90

        # last name 加 first initial 也能提供较强个人关联。
        elif (
            github_handle
            and last_name_token
            and first_name_token
            and last_name_token
            in github_handle
            and first_name_token[0]
            in github_handle
        ):
            candidate_score = 70

        # 保存候选分数。
        scored_candidates.append(
            {
                "score": candidate_score,
                "candidate_index": candidate_index,
                "github_url": github_candidate,
            }
        )

    # 取得最高分。
    highest_score = max(
        candidate[
            "score"
        ]
        for candidate
        in scored_candidates
    )

    # 多个候选都无法与身份信息关联时保持 NULL。
    if highest_score <= 0:
        return None, "no_personal_github_evidence"

    # 取得全部最高分候选。
    highest_scored_candidates = [
        candidate
        for candidate
        in scored_candidates
        if candidate[
            "score"
        ]
        == highest_score
    ]

    # 多个不同 URL 同分时保持 NULL。
    if len(
        highest_scored_candidates
    ) != 1:
        return None, "ambiguous_github_candidates"

    # 返回唯一最高分 URL。
    return (
        highest_scored_candidates[0][
            "github_url"
        ],
        "selected_by_identity_evidence",
    )


def values_equal(
    database_value,
    planned_value,
):
    """比较数据库现值与计划写入值。"""

    # 两边都为 NULL 时相同。
    if (
        database_value is None
        and planned_value is None
    ):
        return True

    # 只有一边为 NULL 时不同。
    if (
        database_value is None
        or planned_value is None
    ):
        return False

    # 其余字段按文本值比较。
    return str(
        database_value
    ) == str(
        planned_value
    )


# ============================================================
# 读取并验证 application.csv
# ============================================================

# 使用 utf-8-sig 兼容带 BOM 的 CSV。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 清理 column name 首尾空格。
application_dataframe.columns = [
    str(
        column_name
    ).strip()
    for column_name
    in application_dataframe.columns
]

# 定义导入需要的 CSV columns。
required_application_columns = [
    "id",
    "submission_normalized_id",
    "applied_at",
    "resume_text",
    "submission_normalized__raw_person_name",
    "submission_normalized__normalized_person_name",
    "submission_normalized__normalized_first_name",
    "submission_normalized__normalized_middle_name",
    "submission_normalized__normalized_last_name",
    "submission_normalized__raw_person_email_address",
    "submission_normalized__normalized_email_prefix",
    "submission_normalized__normalized_email_domain",
    "submission_normalized__work_type",
    "submission_normalized__work_duration",
    "submission_normalized__start_working_date",
    "submission_normalized__end_working_date",
    "resume_extraction__resume_phones",
    "submission_identity_feature_records_json",
]

# 查找缺失 columns。
missing_application_columns = [
    column_name
    for column_name
    in required_application_columns
    if column_name
    not in application_dataframe.columns
]

# 缺少必要 column 时停止。
if missing_application_columns:
    raise ValueError(
        "application.csv 缺少必要字段："
        f"{missing_application_columns}"
    )

# 只处理已经拥有 D1 Application.id 的记录。
application_dataframe = (
    application_dataframe[
        application_dataframe["id"].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

# 空数据集无法继续。
if application_dataframe.empty:
    raise ValueError(
        "application.csv 中没有包含 Application.id 的记录。"
    )

# 转换 Application.id。
application_dataframe["id"] = (
    application_dataframe["id"].map(
        lambda value: normalize_integer(
            value,
            "Application.id",
        )
    )
)

# 转换 Submission_normalized.id。
application_dataframe[
    "submission_normalized_id"
] = application_dataframe[
    "submission_normalized_id"
].map(
    lambda value: normalize_integer(
        value,
        "submission_normalized_id",
    )
)

# 同一个 Application 只能创建一个 Candidate_snapshot。
if application_dataframe[
    "id"
].duplicated().any():
    raise ValueError(
        "application.csv 中存在重复 Application.id。"
    )


# ============================================================
# 验证 D1 Application 和 Candidate_snapshot schema
# ============================================================

# 查询所有 D1 Application.id。
application_result = execute_d1(
    """
    SELECT "id"
    FROM "Application"
    ORDER BY "id";
    """
)

# 提取 Application rows。
application_rows = (
    application_result[0].get(
        "results",
        [],
    )
    if application_result
    else []
)

# 生成 D1 Application.id set。
d1_application_ids = {
    int(
        application_row["id"]
    )
    for application_row
    in application_rows
}

# 生成 CSV Application.id set。
csv_application_ids = {
    int(
        application_id
    )
    for application_id
    in application_dataframe[
        "id"
    ].tolist()
}

# 检查 CSV id 是否全部存在于 D1。
application_ids_missing_from_d1 = sorted(
    csv_application_ids
    - d1_application_ids
)

# 显示缺失 id 并停止。
if application_ids_missing_from_d1:
    raise ValueError(
        "以下 Application.id 不存在于 D1："
        f"{application_ids_missing_from_d1[:20]}"
    )

# 查询 Candidate_snapshot columns。
candidate_schema_result = execute_d1(
    """
    PRAGMA table_info("Candidate_snapshot");
    """
)

# 提取 schema rows。
candidate_schema_rows = (
    candidate_schema_result[0].get(
        "results",
        [],
    )
    if candidate_schema_result
    else []
)

# 生成当前 column set。
candidate_schema_columns = {
    str(
        schema_row["name"]
    )
    for schema_row
    in candidate_schema_rows
}

# 本 Cell 预期的目标 columns。
required_candidate_columns = {
    "id",
    "application_id",
    "application_date",
    "salary_expectation_min_cents",
    "salary_expectation_max_cents",
    "salary_expectation_currency",
    "work_authorization_type",
    "work_type",
    "available_start_date",
    "notice_period_days",
    "created_at",
    "updated_at",
    "expected_end_date",
    "work_duration",
    "raw_person_name",
    "normalized_person_name",
    "normalized_first_name",
    "normalized_middle_name",
    "normalized_last_name",
    "raw_email_address",
    "normalized_email_address",
    "raw_phone",
    "normalized_phone",
    "raw_linkedin_url",
    "normalized_linkedin_url",
    "raw_github_url",
    "normalized_github_url",
    "raw_resume_text",
}

# 查找目标表缺少的 columns。
missing_candidate_columns = sorted(
    required_candidate_columns
    - candidate_schema_columns
)

# schema 不符合预期时停止。
if missing_candidate_columns:
    raise ValueError(
        "D1 Candidate_snapshot 缺少必要字段："
        f"{missing_candidate_columns}"
    )


# ============================================================
# 生成 Candidate_snapshot 计划数据
# ============================================================

# 保存全部计划 records。
planned_candidate_records = []

# 统计电话选择结果。
phone_selection_status_counts = {}

# 统计 GitHub 选择结果。
github_selection_status_counts = {}

# 逐条处理 application.csv。
for dataframe_index, application_row in (
    application_dataframe.iterrows()
):
    # 取得 Application.id。
    application_id = int(
        application_row["id"]
    )

    # 取得当前 Submission_normalized.id。
    submission_normalized_id = int(
        application_row[
            "submission_normalized_id"
        ]
    )

    # 只解析当前 submission 自己的身份特征。
    identity_features = parse_identity_feature_records(
        application_row[
            "submission_identity_feature_records_json"
        ],
        submission_normalized_id,
    )

    # 从已经标准化的 prefix/domain 生成完整邮箱。
    normalized_email_address = (
        build_normalized_email_address(
            application_row[
                "submission_normalized__normalized_email_prefix"
            ],
            application_row[
                "submission_normalized__normalized_email_domain"
            ],
        )
    )

    # 读取 identity feature 中的当前 submission email。
    identity_email_candidates = identity_features.get(
        "email",
        [],
    )

    # prefix/domain 邮箱存在时，身份特征必须包含该值。
    if (
        normalized_email_address is not None
        and identity_email_candidates
        and normalized_email_address
        not in identity_email_candidates
    ):
        raise ValueError(
            "标准化邮箱来源不一致："
            f"Application.id={application_id}, "
            f"prefix/domain={normalized_email_address}, "
            f"identity_feature={identity_email_candidates}"
        )

    # prefix/domain 为空且 identity email 只有一个时使用后者。
    if (
        normalized_email_address is None
        and len(
            identity_email_candidates
        )
        == 1
    ):
        normalized_email_address = (
            identity_email_candidates[0]
        )

    # 使用格式、邮箱排除和美国号码规则选择主要电话。
    (
        raw_phone,
        normalized_phone,
        phone_selection_status,
    ) = select_reliable_phone(
        application_row[
            "resume_extraction__resume_phones"
        ],
        application_row[
            "resume_text"
        ],
        application_row[
            "submission_normalized__raw_person_email_address"
        ],
        identity_features.get(
            "phone",
            [],
        ),
    )

    # 累加电话选择状态。
    phone_selection_status_counts[
        phone_selection_status
    ] = (
        phone_selection_status_counts.get(
            phone_selection_status,
            0,
        )
        + 1
    )

    # 读取标准化 LinkedIn URL。
    linkedin_candidates = identity_features.get(
        "linkedin",
        [],
    )

    # LinkedIn 只有一个候选时写入；多个候选时不自动猜测。
    normalized_linkedin_url = (
        linkedin_candidates[0]
        if len(
            linkedin_candidates
        )
        == 1
        else None
    )

    # 使用邮箱 prefix 和标准化姓名选择个人 GitHub。
    (
        normalized_github_url,
        github_selection_status,
    ) = select_reliable_github_url(
        identity_features.get(
            "github",
            [],
        ),
        application_row[
            "submission_normalized__normalized_email_prefix"
        ],
        application_row[
            "submission_normalized__normalized_first_name"
        ],
        application_row[
            "submission_normalized__normalized_last_name"
        ],
    )

    # 累加 GitHub 选择状态。
    github_selection_status_counts[
        github_selection_status
    ] = (
        github_selection_status_counts.get(
            github_selection_status,
            0,
        )
        + 1
    )

    # 生成计划写入 record。
    planned_candidate_record = {
        "application_id": application_id,
        "application_date": clean_text_value(
            application_row[
                "applied_at"
            ]
        ),
        "salary_expectation_min_cents": None,
        "salary_expectation_max_cents": None,
        "salary_expectation_currency": None,
        "work_authorization_type": None,
        "work_type": clean_text_value(
            application_row[
                "submission_normalized__work_type"
            ]
        ),
        "available_start_date": clean_text_value(
            application_row[
                "submission_normalized__start_working_date"
            ]
        ),
        "notice_period_days": None,
        "expected_end_date": clean_text_value(
            application_row[
                "submission_normalized__end_working_date"
            ]
        ),
        "work_duration": clean_text_value(
            application_row[
                "submission_normalized__work_duration"
            ]
        ),
        "raw_person_name": clean_text_value(
            application_row[
                "submission_normalized__raw_person_name"
            ]
        ),
        "normalized_person_name": clean_text_value(
            application_row[
                "submission_normalized__normalized_person_name"
            ]
        ),
        "normalized_first_name": clean_text_value(
            application_row[
                "submission_normalized__normalized_first_name"
            ]
        ),
        "normalized_middle_name": clean_text_value(
            application_row[
                "submission_normalized__normalized_middle_name"
            ]
        ),
        "normalized_last_name": clean_text_value(
            application_row[
                "submission_normalized__normalized_last_name"
            ]
        ),
        "raw_email_address": clean_text_value(
            application_row[
                "submission_normalized__raw_person_email_address"
            ]
        ),
        "normalized_email_address": (
            normalized_email_address
        ),
        "raw_phone": raw_phone,
        "normalized_phone": normalized_phone,
        # 当前 application.csv 没有保留可靠的原始 LinkedIn URL column。
        # 不把标准化 URL 冒充成原始值，所以 raw 字段暂时写入 NULL。
        "raw_linkedin_url": None,
        "normalized_linkedin_url": (
            normalized_linkedin_url
        ),
        # 当前 application.csv 没有保留可靠的原始 GitHub URL column。
        # 不把标准化 URL 冒充成原始值，所以 raw 字段暂时写入 NULL。
        "raw_github_url": None,
        "normalized_github_url": (
            normalized_github_url
        ),
        "raw_resume_text": clean_text_value(
            application_row[
                "resume_text"
            ]
        ),
    }

    # application_date 是 NOT NULL。
    if (
        planned_candidate_record[
            "application_date"
        ]
        is None
    ):
        raise ValueError(
            "applied_at 为空，无法生成 application_date："
            f"Application.id={application_id}"
        )

    # 保存计划 record。
    planned_candidate_records.append(
        planned_candidate_record
    )

    # 计算进度。
    processed_count = (
        dataframe_index + 1
    )

    # 每200条以及最后一条显示构建进度。
    if (
        processed_count % 200 == 0
        or processed_count
        == len(
            application_dataframe
        )
    ):
        print(
            "Prepared Candidate_snapshot records:",
            f"{processed_count}/"
            f"{len(application_dataframe)}",
        )

# 建立计划 DataFrame。
candidate_snapshot_plan_dataframe = (
    pd.DataFrame(
        planned_candidate_records
    )
)

# 验证 application_id 唯一。
if candidate_snapshot_plan_dataframe[
    "application_id"
].duplicated().any():
    raise ValueError(
        "计划数据中存在重复 application_id。"
    )

# 显示计划统计。
print(
    "Candidate_snapshot plan summary:",
    {
        "planned_record_count": int(
            len(
                candidate_snapshot_plan_dataframe
            )
        ),
        "raw_resume_text_present_count": int(
            candidate_snapshot_plan_dataframe[
                "raw_resume_text"
            ].notna().sum()
        ),
        "normalized_email_present_count": int(
            candidate_snapshot_plan_dataframe[
                "normalized_email_address"
            ].notna().sum()
        ),
        "normalized_phone_present_count": int(
            candidate_snapshot_plan_dataframe[
                "normalized_phone"
            ].notna().sum()
        ),
        "normalized_linkedin_present_count": int(
            candidate_snapshot_plan_dataframe[
                "normalized_linkedin_url"
            ].notna().sum()
        ),
        "normalized_github_present_count": int(
            candidate_snapshot_plan_dataframe[
                "normalized_github_url"
            ].notna().sum()
        ),
        "phone_selection_status_counts": (
            phone_selection_status_counts
        ),
        "github_selection_status_counts": (
            github_selection_status_counts
        ),
    },
)

# 显示前10条写入计划。
display(
    candidate_snapshot_plan_dataframe.head(
        10
    )
)


# ============================================================
# 读取已有 Candidate_snapshot
# ============================================================

# 定义需要比较和写入的业务字段顺序。
candidate_business_columns = [
    "application_date",
    "salary_expectation_min_cents",
    "salary_expectation_max_cents",
    "salary_expectation_currency",
    "work_authorization_type",
    "work_type",
    "available_start_date",
    "notice_period_days",
    "expected_end_date",
    "work_duration",
    "raw_person_name",
    "normalized_person_name",
    "normalized_first_name",
    "normalized_middle_name",
    "normalized_last_name",
    "raw_email_address",
    "normalized_email_address",
    "raw_phone",
    "normalized_phone",
    "raw_linkedin_url",
    "normalized_linkedin_url",
    "raw_github_url",
    "normalized_github_url",
    "raw_resume_text",
]

# 查询全部已有 Candidate_snapshot 的轻量 id mapping。
# 这里不一次性读取 raw_resume_text，避免第二次运行时
# 1039份简历长文本让单个 D1 API response 过大。
existing_candidate_result = execute_d1(
    """
    SELECT
      "id",
      "application_id"
    FROM "Candidate_snapshot"
    ORDER BY "application_id";
    """
)

# 提取已有 rows。
existing_candidate_rows = (
    existing_candidate_result[0].get(
        "results",
        [],
    )
    if existing_candidate_result
    else []
)

# 按 application_id 建立 mapping。
existing_candidate_by_application_id = {}

# 逐条建立已有记录 mapping。
for existing_candidate_row in (
    existing_candidate_rows
):
    # 转换成 Python int。
    existing_application_id = int(
        existing_candidate_row[
            "application_id"
        ]
    )

    # application_id UNIQUE，不应该重复。
    if (
        existing_application_id
        in existing_candidate_by_application_id
    ):
        raise ValueError(
            "D1 Candidate_snapshot 存在重复 application_id："
            f"{existing_application_id}"
        )

    # 保存已有记录。
    existing_candidate_by_application_id[
        existing_application_id
    ] = existing_candidate_row


# ============================================================
# 参数化 INSERT 和 UPDATE SQL
# ============================================================

# 已有记录逐条读取完整内容。
# 逐条查询能够避免把全部 raw_resume_text 放在一个 response 中。
select_existing_candidate_sql = """
    SELECT
      "id",
      "application_id",
      "application_date",
      "salary_expectation_min_cents",
      "salary_expectation_max_cents",
      "salary_expectation_currency",
      "work_authorization_type",
      "work_type",
      "available_start_date",
      "notice_period_days",
      "expected_end_date",
      "work_duration",
      "raw_person_name",
      "normalized_person_name",
      "normalized_first_name",
      "normalized_middle_name",
      "normalized_last_name",
      "raw_email_address",
      "normalized_email_address",
      "raw_phone",
      "normalized_phone",
      "raw_linkedin_url",
      "normalized_linkedin_url",
      "raw_github_url",
      "normalized_github_url",
      "raw_resume_text",
      "created_at",
      "updated_at"
    FROM "Candidate_snapshot"
    WHERE "application_id" = ?;
"""

# INSERT 新 Candidate_snapshot。
insert_sql = """
    INSERT INTO "Candidate_snapshot" (
      "application_id",
      "application_date",
      "salary_expectation_min_cents",
      "salary_expectation_max_cents",
      "salary_expectation_currency",
      "work_authorization_type",
      "work_type",
      "available_start_date",
      "notice_period_days",
      "expected_end_date",
      "work_duration",
      "raw_person_name",
      "normalized_person_name",
      "normalized_first_name",
      "normalized_middle_name",
      "normalized_last_name",
      "raw_email_address",
      "normalized_email_address",
      "raw_phone",
      "normalized_phone",
      "raw_linkedin_url",
      "normalized_linkedin_url",
      "raw_github_url",
      "normalized_github_url",
      "raw_resume_text",
      "created_at",
      "updated_at"
    )
    VALUES (
      ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
      ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
      ?, ?, ?, ?, ?
    )
    RETURNING "id";
"""

# UPDATE 只修改发生变化的已有 Candidate_snapshot 内容。
update_sql = """
    UPDATE "Candidate_snapshot"
    SET
      "application_date" = ?,
      "salary_expectation_min_cents" = ?,
      "salary_expectation_max_cents" = ?,
      "salary_expectation_currency" = ?,
      "work_authorization_type" = ?,
      "work_type" = ?,
      "available_start_date" = ?,
      "notice_period_days" = ?,
      "expected_end_date" = ?,
      "work_duration" = ?,
      "raw_person_name" = ?,
      "normalized_person_name" = ?,
      "normalized_first_name" = ?,
      "normalized_middle_name" = ?,
      "normalized_last_name" = ?,
      "raw_email_address" = ?,
      "normalized_email_address" = ?,
      "raw_phone" = ?,
      "normalized_phone" = ?,
      "raw_linkedin_url" = ?,
      "normalized_linkedin_url" = ?,
      "raw_github_url" = ?,
      "normalized_github_url" = ?,
      "raw_resume_text" = ?,
      "updated_at" = ?
    WHERE "application_id" = ?;
"""

# 初始化统计。
inserted_record_count = 0
updated_record_count = 0
reused_record_count = 0

# 保存 id mapping。
candidate_snapshot_id_records = []

# 逐条执行幂等写入。
for record_index, planned_record in enumerate(
    planned_candidate_records,
    start=1,
):
    # 读取 Application.id。
    application_id = int(
        planned_record[
            "application_id"
        ]
    )

    # 查找已有 Candidate_snapshot。
    existing_record = (
        existing_candidate_by_application_id.get(
            application_id
        )
    )

    # 不存在时 INSERT。
    if existing_record is None:
        # 新记录 created_at=updated_at。
        current_timestamp = utc_now_text()

        # 按 SQL 顺序生成 parameters。
        insert_params = [
            application_id,
        ] + [
            planned_record[
                column_name
            ]
            for column_name
            in candidate_business_columns
        ] + [
            current_timestamp,
            current_timestamp,
        ]

        # 执行 INSERT。
        insert_result = execute_d1(
            insert_sql,
            insert_params,
        )

        # 提取 RETURNING row。
        inserted_rows = (
            insert_result[0].get(
                "results",
                [],
            )
            if insert_result
            else []
        )

        # 必须取得新 id。
        if not inserted_rows:
            raise RuntimeError(
                "INSERT 没有返回 Candidate_snapshot.id："
                f"Application.id={application_id}"
            )

        # 转成 Python int。
        candidate_snapshot_id = int(
            inserted_rows[0]["id"]
        )

        # 增加 INSERT 统计。
        inserted_record_count += 1

        # 保存 action。
        record_action = "inserted"

    else:
        # 已存在时逐条读取完整 Candidate_snapshot 内容。
        existing_detail_result = execute_d1(
            select_existing_candidate_sql,
            [
                application_id,
            ],
        )

        # 提取完整已有 record。
        existing_detail_rows = (
            existing_detail_result[0].get(
                "results",
                [],
            )
            if existing_detail_result
            else []
        )

        # id mapping 存在时必须且只能查到一条完整记录。
        if len(
            existing_detail_rows
        ) != 1:
            raise RuntimeError(
                "无法唯一读取已有 Candidate_snapshot："
                f"Application.id={application_id}, "
                f"record_count={len(existing_detail_rows)}"
            )

        # 使用完整 record 进行字段比较。
        existing_record = (
            existing_detail_rows[0]
        )

        # 找出与计划值不同的业务字段。
        differing_columns = [
            column_name
            for column_name
            in candidate_business_columns
            if not values_equal(
                existing_record.get(
                    column_name
                ),
                planned_record[
                    column_name
                ],
            )
        ]

        # 有差异时 UPDATE。
        if differing_columns:
            # 实际 UPDATE 时才刷新 updated_at。
            current_timestamp = utc_now_text()

            # 按 SQL 顺序生成 parameters。
            update_params = [
                planned_record[
                    column_name
                ]
                for column_name
                in candidate_business_columns
            ] + [
                current_timestamp,
                application_id,
            ]

            # 执行 UPDATE。
            execute_d1(
                update_sql,
                update_params,
            )

            # 复用已有 Candidate_snapshot.id。
            candidate_snapshot_id = int(
                existing_record["id"]
            )

            # 增加 UPDATE 统计。
            updated_record_count += 1

            # 保存 action。
            record_action = "updated"

        else:
            # 完全一致时不发送写请求。
            candidate_snapshot_id = int(
                existing_record["id"]
            )

            # 增加复用统计。
            reused_record_count += 1

            # 保存 action。
            record_action = "reused"

    # 保存 id mapping。
    candidate_snapshot_id_records.append(
        {
            "application_id": application_id,
            "candidate_snapshot_id": (
                candidate_snapshot_id
            ),
            "write_action": record_action,
        }
    )

    # 每100条以及最后一条显示进度。
    if (
        record_index % 100 == 0
        or record_index
        == len(
            planned_candidate_records
        )
    ):
        print(
            "Processed Candidate_snapshot records:",
            f"{record_index}/"
            f"{len(planned_candidate_records)}",
        )


# ============================================================
# 写入后验证
# ============================================================

# 当前 application.csv 可以只包含本次增量批次，
# 因此写入后只验证当前 CSV 对应的 Candidate_snapshot，
# 不再把数据库中的历史 Candidate_snapshot 总数
# 与本次计划记录数量进行比较。

# 保存当前增量批次需要验证的 Application ID。
verification_application_ids = sorted(
    int(application_id)
    for application_id
    in csv_application_ids
)

# 每次最多验证 50 个 Application ID，
# 避免未来批次较大时单条 SQL 参数过多。
VERIFICATION_QUERY_CHUNK_SIZE = 50

# 初始化当前增量批次的验证汇总。
verification_summary = {
    "candidate_snapshot_count": 0,
    "distinct_application_count": 0,
    "raw_resume_text_present_count": 0,
    "normalized_email_present_count": 0,
}

# 分块验证当前增量批次。
for verification_chunk_start in range(
    0,
    len(verification_application_ids),
    VERIFICATION_QUERY_CHUNK_SIZE,
):
    # 取得当前分块的 Application ID。
    verification_chunk_ids = (
        verification_application_ids[
            verification_chunk_start:
            verification_chunk_start
            + VERIFICATION_QUERY_CHUNK_SIZE
        ]
    )

    # 按当前分块数量生成参数化 IN 占位符。
    verification_placeholders = ",".join(
        ["?"] * len(verification_chunk_ids)
    )

    # 只统计当前分块对应的 Candidate_snapshot。
    verification_result = execute_d1(
        f"""
        SELECT
          COUNT(cs."id") AS candidate_snapshot_count,
          COUNT(DISTINCT cs."application_id")
            AS distinct_application_count,
          SUM(
            CASE
              WHEN cs."raw_resume_text" IS NOT NULL
              THEN 1 ELSE 0
            END
          ) AS raw_resume_text_present_count,
          SUM(
            CASE
              WHEN cs."normalized_email_address" IS NOT NULL
              THEN 1 ELSE 0
            END
          ) AS normalized_email_present_count
        FROM "Candidate_snapshot" AS cs
        INNER JOIN "Application" AS app
          ON app."id" = cs."application_id"
        WHERE cs."application_id" IN (
          {verification_placeholders}
        );
        """,
        [
            int(application_id)
            for application_id
            in verification_chunk_ids
        ],
    )

    # 提取当前分块的汇总记录。
    verification_chunk_summary = (
        verification_result[0]["results"][0]
    )

    # 将当前分块计数累加到本次增量批次汇总。
    for verification_field_name in (
        "candidate_snapshot_count",
        "distinct_application_count",
        "raw_resume_text_present_count",
        "normalized_email_present_count",
    ):
        verification_summary[
            verification_field_name
        ] += int(
            verification_chunk_summary.get(
                verification_field_name
            )
            or 0
        )

# 总记录数必须与计划数完全一致。
if int(
    verification_summary[
        "candidate_snapshot_count"
    ]
) != len(
    planned_candidate_records
):
    raise RuntimeError(
        "Candidate_snapshot 总数与计划数不一致："
        f"{verification_summary}"
    )

# distinct application 数量必须完全一致。
if int(
    verification_summary[
        "distinct_application_count"
    ]
) != len(
    planned_candidate_records
):
    raise RuntimeError(
        "Candidate_snapshot application_id 唯一性验证失败："
        f"{verification_summary}"
    )

# 检查全部外键。
foreign_key_result = execute_d1(
    """
    PRAGMA foreign_key_check;
    """
)

# 提取外键违规 rows。
foreign_key_rows = (
    foreign_key_result[0].get(
        "results",
        [],
    )
    if foreign_key_result
    else []
)

# 发现外键违规时停止。
if foreign_key_rows:
    raise RuntimeError(
        "写入后发现外键违规："
        f"{foreign_key_rows[:20]}"
    )

# 建立最终 id mapping DataFrame。
candidate_snapshot_id_dataframe = (
    pd.DataFrame(
        candidate_snapshot_id_records
    )
)

# 显示最终统计。
print(
    "Candidate_snapshot import summary:",
    {
        "planned_records": int(
            len(
                planned_candidate_records
            )
        ),
        "inserted_records": int(
            inserted_record_count
        ),
        "updated_records": int(
            updated_record_count
        ),
        "reused_records": int(
            reused_record_count
        ),
        "verified_records": int(
            verification_summary[
                "candidate_snapshot_count"
            ]
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_rows
            )
        ),
    },
)

# 显示前10条 id mapping。
display(
    candidate_snapshot_id_dataframe.head(
        10
    )
)

# 显示完成信息。
print(
    "Candidate_snapshot import completed successfully."
)


### Person 表里的lasted_candidate_id字段数据回填（索引回每一个person对应的最新Candidate_snapshot，可重复运行）

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 使用 D1 Application 和 Candidate_snapshot
# 核对并补全已经存在的 Person 主档。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell。
# 2. 本 Cell 不读取 application.csv，也不依赖 CSV 中为空的 person_id。
# 3. Person 映射直接来自 D1 Application.person_id。
# 4. 本 Cell不会创建、删除或合并 Person。
# 5. 同一个标准化邮箱如果对应多个 Person，本 Cell 会在写入前停止。
# 6. 每个 Person 使用 application_date 最新的 Candidate_snapshot
#    作为 latest_candidate_id。
# 7. 姓名和邮箱使用最新 Candidate_snapshot 的标准化值。
# 8. 电话使用该 Person 最近一条非空的可靠标准化电话。
# 9. 只有字段发生变化时才执行 UPDATE。
# 10. 因此本 Cell 可以重复运行，结果不会重复增加。
# ============================================================

# time 只用于 D1 请求临时失败后的重试等待。
import time

# datetime 用于生成 UTC updated_at。
from datetime import datetime, timezone

# pandas 只用于生成计划和结果预览。
import pandas as pd

# requests 用于调用 Cloudflare D1 API。
import requests

# userdata 用于读取 Colab Secrets。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 Cloudflare D1 API endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在重试次数中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 SQL。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部网络尝试失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 与 query 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 API 或 SQL 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def utc_now_text():
    """生成不包含微秒的 UTC timestamp。"""

    # 返回统一 D1 timestamp 格式。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_text_value(
    value,
):
    """把 SQL/CSV 空值转换成 None，其余值转换成去除首尾空格的文本。"""

    # SQL NULL 直接保持 None。
    if value is None:
        return None

    # 转成标准 Python string。
    cleaned_value = str(
        value
    ).strip()

    # 空 string 转成 None。
    if not cleaned_value:
        return None

    # 返回不改变大小写的业务文本。
    return cleaned_value


def values_equal(
    existing_value,
    planned_value,
):
    """比较数据库现值和计划值。"""

    # 两边都为空时相同。
    if (
        existing_value is None
        and planned_value is None
    ):
        return True

    # 只有一边为空时不同。
    if (
        existing_value is None
        or planned_value is None
    ):
        return False

    # 其余值按文本比较。
    return str(
        existing_value
    ) == str(
        planned_value
    )


# ============================================================
# 验证 Person schema
# ============================================================

# 查询 Person columns。
person_schema_result = execute_d1(
    """
    PRAGMA table_info("Person");
    """
)

# 提取 schema rows。
person_schema_rows = (
    person_schema_result[0].get(
        "results",
        [],
    )
    if person_schema_result
    else []
)

# 生成现有 column set。
person_schema_columns = {
    str(
        schema_row["name"]
    )
    for schema_row
    in person_schema_rows
}

# 本 Cell 需要的 Person columns。
required_person_columns = {
    "id",
    "person_first_name",
    "person_last_name",
    "email",
    "phone",
    "latest_candidate_id",
    "updated_at",
}

# 查找缺失 columns。
missing_person_columns = sorted(
    required_person_columns
    - person_schema_columns
)

# schema 不符合预期时停止。
if missing_person_columns:
    raise ValueError(
        "D1 Person 缺少必要字段："
        f"{missing_person_columns}"
    )


# ============================================================
# 读取 Person、Application 与 Candidate_snapshot 对应关系
# ============================================================

# 查询每条 Candidate_snapshot 对应的 Person。
# 不读取 raw_resume_text，避免无关的大型 response。
mapping_result = execute_d1(
    """
    SELECT
      p."id" AS person_id,
      p."person_first_name"
        AS existing_person_first_name,
      p."person_last_name"
        AS existing_person_last_name,
      p."email" AS existing_email,
      p."phone" AS existing_phone,
      p."latest_candidate_id"
        AS existing_latest_candidate_id,
      p."updated_at" AS existing_updated_at,
      app."id" AS application_id,
      cs."id" AS candidate_snapshot_id,
      cs."application_date",
      cs."normalized_first_name",
      cs."normalized_last_name",
      cs."normalized_email_address",
      cs."normalized_phone"
    FROM "Application" AS app
    INNER JOIN "Person" AS p
      ON p."id" = app."person_id"
    INNER JOIN "Candidate_snapshot" AS cs
      ON cs."application_id" = app."id"
    ORDER BY
      p."id",
      cs."application_date",
      cs."id";
    """
)

# 提取 mapping rows。
mapping_rows = (
    mapping_result[0].get(
        "results",
        [],
    )
    if mapping_result
    else []
)

# 没有 mapping 时无法继续。
if not mapping_rows:
    raise ValueError(
        "D1 中没有找到 Application、Person 与 "
        "Candidate_snapshot 的对应记录。"
    )

# 查询独立表数量，用于完整性验证。
count_result = execute_d1(
    """
    SELECT
      (SELECT COUNT(*) FROM "Application")
        AS application_count,
      (SELECT COUNT(*) FROM "Candidate_snapshot")
        AS candidate_snapshot_count,
      (SELECT COUNT(*) FROM "Person")
        AS person_count;
    """
)

# 提取数量 summary。
count_summary = (
    count_result[0]["results"][0]
)

# 每条 Application 都应该已经拥有 Candidate_snapshot。
if int(
    count_summary[
        "application_count"
    ]
) != int(
    count_summary[
        "candidate_snapshot_count"
    ]
):
    raise ValueError(
        "Application 与 Candidate_snapshot 数量不一致："
        f"{count_summary}"
    )

# JOIN 数量也必须等于 Application 数量。
if len(
    mapping_rows
) != int(
    count_summary[
        "application_count"
    ]
):
    raise ValueError(
        "Person/Application/Candidate_snapshot JOIN 数量不完整："
        f"join_count={len(mapping_rows)}, "
        f"counts={count_summary}"
    )


# ============================================================
# 写入前检查 normalized email 是否跨 Person 冲突
# ============================================================

# 保存 normalized email 对应的 Person ids。
person_ids_by_normalized_email = {}

# 逐条建立 email mapping。
for mapping_row in mapping_rows:
    # 清理 normalized email。
    normalized_email_address = (
        clean_text_value(
            mapping_row[
                "normalized_email_address"
            ]
        )
    )

    # 空邮箱不参与冲突检查。
    if normalized_email_address is None:
        continue

    # 邮箱统一小写作为比较 key。
    normalized_email_key = (
        normalized_email_address.lower()
    )

    # 加入当前 Person.id。
    person_ids_by_normalized_email.setdefault(
        normalized_email_key,
        set(),
    ).add(
        int(
            mapping_row[
                "person_id"
            ]
        )
    )

# 只保留同一个邮箱对应多个 Person 的冲突。
email_person_conflicts = {
    normalized_email: sorted(
        person_ids
    )
    for normalized_email, person_ids
    in person_ids_by_normalized_email.items()
    if len(
        person_ids
    )
    > 1
}

# 发现冲突时显示并停止，不自动合并 Person。
if email_person_conflicts:
    # 建立冲突预览 DataFrame。
    email_conflict_dataframe = pd.DataFrame(
        [
            {
                "normalized_email_address": (
                    normalized_email
                ),
                "person_ids": ",".join(
                    str(
                        person_id
                    )
                    for person_id
                    in person_ids
                ),
                "person_count": len(
                    person_ids
                ),
            }
            for normalized_email, person_ids
            in sorted(
                email_person_conflicts.items()
            )
        ]
    )

    # 显示冲突。
    display(
        email_conflict_dataframe
    )

    # 写入前停止。
    raise ValueError(
        "发现同一个 normalized email 对应多个 Person。"
        "本 Cell 没有修改数据库，也不会自动合并 Person。"
    )


# ============================================================
# 按 Person 建立最新 Candidate 计划
# ============================================================

# 保存每个 Person 的全部 Candidate rows。
candidate_rows_by_person_id = {}

# 逐条分组。
for mapping_row in mapping_rows:
    # 读取 Person.id。
    person_id = int(
        mapping_row[
            "person_id"
        ]
    )

    # 保存到当前 Person 的 records list。
    candidate_rows_by_person_id.setdefault(
        person_id,
        [],
    ).append(
        mapping_row
    )

# 保存全部 Person 更新计划。
person_update_plans = []

# 按 Person.id 稳定处理。
for person_id in sorted(
    candidate_rows_by_person_id
):
    # 读取当前 Person 的全部 Candidate rows。
    person_candidate_rows = (
        candidate_rows_by_person_id[
            person_id
        ]
    )

    # application_date 使用 ISO 文本；
    # Candidate_snapshot.id 作为相同时间下的稳定 tie-breaker。
    sorted_candidate_rows = sorted(
        person_candidate_rows,
        key=lambda candidate_row: (
            clean_text_value(
                candidate_row[
                    "application_date"
                ]
            )
            or "",
            int(
                candidate_row[
                    "candidate_snapshot_id"
                ]
            ),
        ),
    )

    # 最后一条是最新 Candidate。
    latest_candidate_row = (
        sorted_candidate_rows[-1]
    )

    # 姓名使用最新 Candidate 的标准化结果。
    planned_first_name = clean_text_value(
        latest_candidate_row[
            "normalized_first_name"
        ]
    )

    # last name 使用最新 Candidate。
    planned_last_name = clean_text_value(
        latest_candidate_row[
            "normalized_last_name"
        ]
    )

    # Person 姓名字段是 NOT NULL。
    # 最新 Candidate 缺失时保留 Person 现有值。
    if planned_first_name is None:
        planned_first_name = clean_text_value(
            latest_candidate_row[
                "existing_person_first_name"
            ]
        )

    # last name 缺失时也保留现有值。
    if planned_last_name is None:
        planned_last_name = clean_text_value(
            latest_candidate_row[
                "existing_person_last_name"
            ]
        )

    # 最终仍为空时停止。
    if (
        planned_first_name is None
        or planned_last_name is None
    ):
        raise ValueError(
            "Person 必需姓名无法生成："
            f"Person.id={person_id}"
        )

    # 邮箱使用最新 Candidate 的标准化完整邮箱。
    planned_email = clean_text_value(
        latest_candidate_row[
            "normalized_email_address"
        ]
    )

    # 最新 Candidate 邮箱为空时保留现有 Person.email。
    if planned_email is None:
        planned_email = clean_text_value(
            latest_candidate_row[
                "existing_email"
            ]
        )

    # 电话使用按时间从新到旧的第一条非空可靠电话。
    planned_phone = None

    # 反向遍历，优先最新非空号码。
    for candidate_row in reversed(
        sorted_candidate_rows
    ):
        # 清理当前 Candidate 电话。
        candidate_phone = clean_text_value(
            candidate_row[
                "normalized_phone"
            ]
        )

        # 找到第一个非空电话后停止。
        if candidate_phone is not None:
            planned_phone = candidate_phone
            break

    # 所有 Candidate 都没有电话时保留现有 Person.phone。
    if planned_phone is None:
        planned_phone = clean_text_value(
            latest_candidate_row[
                "existing_phone"
            ]
        )

    # 建立 Person 更新计划。
    person_update_plans.append(
        {
            "person_id": person_id,
            "person_first_name": planned_first_name,
            "person_last_name": planned_last_name,
            "email": planned_email,
            "phone": planned_phone,
            "latest_candidate_id": int(
                latest_candidate_row[
                    "candidate_snapshot_id"
                ]
            ),
            "existing_person_first_name": (
                latest_candidate_row[
                    "existing_person_first_name"
                ]
            ),
            "existing_person_last_name": (
                latest_candidate_row[
                    "existing_person_last_name"
                ]
            ),
            "existing_email": latest_candidate_row[
                "existing_email"
            ],
            "existing_phone": latest_candidate_row[
                "existing_phone"
            ],
            "existing_latest_candidate_id": (
                latest_candidate_row[
                    "existing_latest_candidate_id"
                ]
            ),
            "candidate_count": len(
                person_candidate_rows
            ),
        }
    )

# 建立计划 DataFrame。
person_profile_plan_dataframe = (
    pd.DataFrame(
        person_update_plans
    )
)

# 计划 Person 数量必须与数据库 Person 数量一致。
if len(
    person_update_plans
) != int(
    count_summary[
        "person_count"
    ]
):
    raise ValueError(
        "并非所有 Person 都能通过 Application 找到 Candidate："
        f"planned_person_count={len(person_update_plans)}, "
        f"person_count={count_summary['person_count']}"
    )

# 显示计划统计。
print(
    "Person profile backfill plan summary:",
    {
        "person_count": int(
            len(
                person_update_plans
            )
        ),
        "candidate_snapshot_count": int(
            len(
                mapping_rows
            )
        ),
        "persons_with_multiple_candidates": int(
            (
                person_profile_plan_dataframe[
                    "candidate_count"
                ]
                > 1
            ).sum()
        ),
        "planned_email_present_count": int(
            person_profile_plan_dataframe[
                "email"
            ].notna().sum()
        ),
        "planned_phone_present_count": int(
            person_profile_plan_dataframe[
                "phone"
            ].notna().sum()
        ),
        "normalized_email_conflict_count": int(
            len(
                email_person_conflicts
            )
        ),
    },
)

# 显示前10条计划。
display(
    person_profile_plan_dataframe[
        [
            "person_id",
            "person_first_name",
            "person_last_name",
            "email",
            "phone",
            "latest_candidate_id",
            "candidate_count",
        ]
    ].head(
        10
    )
)


# ============================================================
# 幂等更新 Person
# ============================================================

# 参数化 UPDATE SQL。
update_person_sql = """
    UPDATE "Person"
    SET
      "person_first_name" = ?,
      "person_last_name" = ?,
      "email" = ?,
      "phone" = ?,
      "latest_candidate_id" = ?,
      "updated_at" = ?
    WHERE "id" = ?;
"""

# 初始化统计。
updated_person_count = 0
reused_person_count = 0

# 保存每条处理 action。
person_backfill_result_records = []

# 逐条处理 Person。
for plan_index, person_plan in enumerate(
    person_update_plans,
    start=1,
):
    # 定义计划业务值。
    planned_values = {
        "person_first_name": person_plan[
            "person_first_name"
        ],
        "person_last_name": person_plan[
            "person_last_name"
        ],
        "email": person_plan[
            "email"
        ],
        "phone": person_plan[
            "phone"
        ],
        "latest_candidate_id": person_plan[
            "latest_candidate_id"
        ],
    }

    # 定义数据库现有业务值。
    existing_values = {
        "person_first_name": person_plan[
            "existing_person_first_name"
        ],
        "person_last_name": person_plan[
            "existing_person_last_name"
        ],
        "email": person_plan[
            "existing_email"
        ],
        "phone": person_plan[
            "existing_phone"
        ],
        "latest_candidate_id": person_plan[
            "existing_latest_candidate_id"
        ],
    }

    # 找出发生变化的字段。
    differing_columns = [
        column_name
        for column_name
        in planned_values
        if not values_equal(
            existing_values[
                column_name
            ],
            planned_values[
                column_name
            ],
        )
    ]

    # 有变化时才 UPDATE。
    if differing_columns:
        # 执行参数化 UPDATE。
        execute_d1(
            update_person_sql,
            [
                planned_values[
                    "person_first_name"
                ],
                planned_values[
                    "person_last_name"
                ],
                planned_values[
                    "email"
                ],
                planned_values[
                    "phone"
                ],
                int(
                    planned_values[
                        "latest_candidate_id"
                    ]
                ),
                utc_now_text(),
                int(
                    person_plan[
                        "person_id"
                    ]
                ),
            ],
        )

        # 增加更新统计。
        updated_person_count += 1

        # 保存 action。
        write_action = "updated"

    else:
        # 完全一致时不发送 UPDATE。
        reused_person_count += 1

        # 保存 action。
        write_action = "reused"

    # 保存结果。
    person_backfill_result_records.append(
        {
            "person_id": int(
                person_plan[
                    "person_id"
                ]
            ),
            "latest_candidate_id": int(
                person_plan[
                    "latest_candidate_id"
                ]
            ),
            "write_action": write_action,
            "changed_columns": ",".join(
                differing_columns
            ),
        }
    )

    # 每100条以及最后一条显示进度。
    if (
        plan_index % 100 == 0
        or plan_index
        == len(
            person_update_plans
        )
    ):
        print(
            "Processed Person profiles:",
            f"{plan_index}/"
            f"{len(person_update_plans)}",
        )


# ============================================================
# 写入后验证
# ============================================================

# 查询 Person 补全结果。
verification_result = execute_d1(
    """
    SELECT
      COUNT(*) AS person_count,
      SUM(
        CASE
          WHEN "latest_candidate_id" IS NOT NULL
          THEN 1 ELSE 0
        END
      ) AS latest_candidate_present_count,
      SUM(
        CASE
          WHEN "email" IS NOT NULL
          THEN 1 ELSE 0
        END
      ) AS email_present_count,
      SUM(
        CASE
          WHEN "phone" IS NOT NULL
          THEN 1 ELSE 0
        END
      ) AS phone_present_count
    FROM "Person";
    """
)

# 提取验证 summary。
verification_summary = (
    verification_result[0]["results"][0]
)

# 每个 Person 都应该拥有 latest_candidate_id。
if int(
    verification_summary[
        "latest_candidate_present_count"
    ]
) != int(
    verification_summary[
        "person_count"
    ]
):
    raise RuntimeError(
        "部分 Person 没有 latest_candidate_id："
        f"{verification_summary}"
    )

# 执行全数据库外键检查。
foreign_key_result = execute_d1(
    """
    PRAGMA foreign_key_check;
    """
)

# 提取外键违规 rows。
foreign_key_rows = (
    foreign_key_result[0].get(
        "results",
        [],
    )
    if foreign_key_result
    else []
)

# 任何外键违规都停止。
if foreign_key_rows:
    raise RuntimeError(
        "Person 回填后发现外键违规："
        f"{foreign_key_rows[:20]}"
    )

# 建立最终结果 DataFrame。
person_backfill_result_dataframe = (
    pd.DataFrame(
        person_backfill_result_records
    )
)

# 显示最终统计。
print(
    "Person profile backfill summary:",
    {
        "planned_person_records": int(
            len(
                person_update_plans
            )
        ),
        "updated_person_records": int(
            updated_person_count
        ),
        "reused_person_records": int(
            reused_person_count
        ),
        "verified_person_records": int(
            verification_summary[
                "person_count"
            ]
        ),
        "email_present_count": int(
            verification_summary[
                "email_present_count"
            ]
            or 0
        ),
        "phone_present_count": int(
            verification_summary[
                "phone_present_count"
            ]
            or 0
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_rows
            )
        ),
    },
)

# 显示前10条处理结果。
display(
    person_backfill_result_dataframe.head(
        10
    )
)

# 显示完成信息。
print(
    "Person profile backfill completed successfully."
)


### Person_name, Person_contact, Person_link, Contact_type 数据导入

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 从 Candidate_snapshot 幂等生成 Person 的身份历史子表：
#
# 1. Person_Name
# 2. Person_Contact
# 3. Person_Link
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell 或 CSV。
# 2. 数据直接来自 D1 Application、Person 和 Candidate_snapshot。
# 3. 同一个 Person 的相同身份值只保存一条，不按重复申请重复保存。
# 4. source_candidate_id 保存该身份值第一次出现的 Candidate_snapshot.id。
# 5. Person_Name 的 first_seen_at/last_seen_at 保存首次和最近出现时间。
# 6. Person_Contact 和 Person_Link 的 created_at/updated_at
#    分别保存首次和最近出现时间。
# 7. source_submission_id 统一写入 SQL NULL。
# 8. 已存在且一致的记录直接复用。
# 9. 已存在但 primary/time 信息不同的记录才更新。
# 10. 新记录使用多行批量 INSERT，减少 D1 网络请求。
# ============================================================

# time 用于 D1 请求失败后的重试等待。
import time

# datetime 用于缺少 application_date 时生成稳定的 UTC fallback。
from datetime import datetime, timezone

# pandas 用于建立计划与结果预览。
import pandas as pd

# requests 用于调用 Cloudflare D1 API。
import requests

# userdata 用于读取 Colab Secrets。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session。
HTTP_SESSION = requests.Session()

# 每个批量 INSERT 最多包含10条记录。
# 每条记录最多7个参数，因此单次请求不超过70个参数，
# 同时显著少于逐条发送约3000次请求。
INSERT_BATCH_SIZE = 10


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次调用也包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部网络请求失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # 请求和 SQL 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def utc_now_text():
    """生成统一 UTC timestamp。"""

    # 返回不包含微秒的 ISO 时间。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_text_value(
    value,
):
    """把 SQL 空值转换成 None，其余值转成去除首尾空格的文本。"""

    # SQL NULL 保持 None。
    if value is None:
        return None

    # 转换成标准 Python string。
    cleaned_value = str(
        value
    ).strip()

    # 空 string 转成 None。
    if not cleaned_value:
        return None

    # 返回不改变大小写的文本。
    return cleaned_value


def normalized_key_text(
    value,
):
    """生成身份值查重使用的小写 key。"""

    # 先清理空值。
    cleaned_value = clean_text_value(
        value
    )

    # 空值没有 key。
    if cleaned_value is None:
        return None

    # 邮箱和 URL 使用小写 key。
    return cleaned_value.lower()


def values_equal(
    existing_value,
    planned_value,
):
    """比较数据库现值与计划值。"""

    # 两边都为空时相同。
    if (
        existing_value is None
        and planned_value is None
    ):
        return True

    # 只有一边为空时不同。
    if (
        existing_value is None
        or planned_value is None
    ):
        return False

    # 其余按文本比较。
    return str(
        existing_value
    ) == str(
        planned_value
    )


def require_application_date(
    identity_row,
):
    """读取 Candidate application_date；意外为空时停止。"""

    # 清理 application_date。
    application_date = clean_text_value(
        identity_row[
            "application_date"
        ]
    )

    # 时间字段是身份历史 first/last seen 的正式来源。
    if application_date is None:
        raise ValueError(
            "Candidate_snapshot.application_date 为空："
            f"candidate_snapshot_id="
            f"{identity_row['candidate_snapshot_id']}"
        )

    # 返回稳定的已有业务时间，不使用当前运行时间。
    return application_date


def chunk_records(
    records,
    batch_size,
):
    """把 records 按固定大小拆成多个批次。"""

    # 从0开始按 batch_size 前进。
    for start_index in range(
        0,
        len(
            records
        ),
        batch_size,
    ):
        # 返回当前 records slice。
        yield records[
            start_index:
            start_index + batch_size
        ]


def batch_insert_records(
    table_name,
    column_names,
    records,
):
    """使用参数化多行 VALUES 批量插入 records。"""

    # 没有新记录时不发送 SQL。
    if not records:
        return 0

    # 表名和字段名都由本文件内部固定值调用，不接受用户输入。
    quoted_columns = ", ".join(
        f'"{column_name}"'
        for column_name
        in column_names
    )

    # 保存已插入数量。
    inserted_count = 0

    # 分批插入。
    for record_batch in chunk_records(
        records,
        INSERT_BATCH_SIZE,
    ):
        # 单行 placeholder 数量等于 column 数量。
        single_row_placeholders = (
            "("
            + ", ".join(
                "?"
                for _ in column_names
            )
            + ")"
        )

        # 为当前批次生成多行 placeholders。
        all_row_placeholders = ", ".join(
            single_row_placeholders
            for _ in record_batch
        )

        # 生成 INSERT SQL。
        insert_sql = (
            f'INSERT INTO "{table_name}" '
            f"({quoted_columns}) VALUES "
            f"{all_row_placeholders};"
        )

        # 按行、按字段顺序展开参数。
        insert_params = [
            record[
                column_name
            ]
            for record in record_batch
            for column_name
            in column_names
        ]

        # 执行当前批次 INSERT。
        execute_d1(
            insert_sql,
            insert_params,
        )

        # 增加统计。
        inserted_count += len(
            record_batch
        )

        # 显示批量插入进度。
        print(
            f"Inserted {table_name}:",
            f"{inserted_count}/{len(records)}",
        )

    # 返回插入数量。
    return inserted_count


# ============================================================
# 验证三张目标表的 schema
# ============================================================

# 定义每张表必须存在的 columns。
required_columns_by_table = {
    "Person_Name": {
        "id",
        "person_id",
        "name_value",
        "is_primary",
        "source_submission_id",
        "first_seen_at",
        "last_seen_at",
        "source_candidate_id",
    },
    "Person_Contact": {
        "id",
        "person_id",
        "contact_type_id",
        "contact_value",
        "is_primary",
        "created_at",
        "updated_at",
        "source_candidate_id",
    },
    "Person_Link": {
        "id",
        "person_id",
        "link_type",
        "url",
        "created_at",
        "updated_at",
        "source_candidate_id",
    },
}

# 逐张表验证 columns。
for table_name, required_columns in (
    required_columns_by_table.items()
):
    # 查询 table info。
    table_info_result = execute_d1(
        f'PRAGMA table_info("{table_name}");'
    )

    # 提取 rows。
    table_info_rows = (
        table_info_result[0].get(
            "results",
            [],
        )
        if table_info_result
        else []
    )

    # 生成 column set。
    existing_columns = {
        str(
            table_info_row[
                "name"
            ]
        )
        for table_info_row
        in table_info_rows
    }

    # 查找缺失 columns。
    missing_columns = sorted(
        required_columns
        - existing_columns
    )

    # schema 不符合预期时停止。
    if missing_columns:
        raise ValueError(
            f"{table_name} 缺少必要字段："
            f"{missing_columns}"
        )


# ============================================================
# 解析或创建 Contact_type：Email 和 Phone
# ============================================================

# 查询 Contact_type。
contact_type_result = execute_d1(
    """
    SELECT
      "id",
      "contact_type_name"
    FROM "Contact_type"
    ORDER BY "id";
    """
)

# 提取 rows。
contact_type_rows = (
    contact_type_result[0].get(
        "results",
        [],
    )
    if contact_type_result
    else []
)

# Email 的允许名称别名。
email_contact_type_aliases = {
    "email",
    "e-mail",
    "email address",
}

# Phone 的允许名称别名。
phone_contact_type_aliases = {
    "phone",
    "telephone",
    "mobile",
    "phone number",
}


def find_contact_type_ids(
    aliases,
):
    """按标准化名称别名查找 Contact_type ids。"""

    # 保存匹配 ids。
    matching_ids = []

    # 逐条匹配。
    for contact_type_row in (
        contact_type_rows
    ):
        # 标准化名称。
        normalized_name = normalized_key_text(
            contact_type_row[
                "contact_type_name"
            ]
        )

        # 别名匹配时保存 id。
        if normalized_name in aliases:
            matching_ids.append(
                int(
                    contact_type_row[
                        "id"
                    ]
                )
            )

    # 返回匹配 ids。
    return matching_ids


def resolve_or_create_contact_type(
    canonical_name,
    aliases,
):
    """唯一查找 Contact_type；不存在时创建 canonical record。"""

    # 查找当前匹配 ids。
    matching_ids = find_contact_type_ids(
        aliases
    )

    # 多个别名 records 会造成语义不唯一，必须停止。
    if len(
        matching_ids
    ) > 1:
        raise ValueError(
            f"Contact_type 中存在多个 {canonical_name} 类型："
            f"{matching_ids}"
        )

    # 唯一存在时直接复用。
    if len(
        matching_ids
    ) == 1:
        return matching_ids[0]

    # 不存在时生成 timestamps。
    current_timestamp = utc_now_text()

    # 创建必要 Contact_type。
    insert_result = execute_d1(
        """
        INSERT INTO "Contact_type" (
          "contact_type_name",
          "created_at",
          "updated_at"
        )
        VALUES (?, ?, ?)
        RETURNING "id";
        """,
        [
            canonical_name,
            current_timestamp,
            current_timestamp,
        ],
    )

    # 提取新 id。
    inserted_rows = (
        insert_result[0].get(
            "results",
            [],
        )
        if insert_result
        else []
    )

    # 没有返回 id 时停止。
    if not inserted_rows:
        raise RuntimeError(
            f"创建 Contact_type {canonical_name} 时没有返回 id。"
        )

    # 保存到内存 rows，供第二种类型解析使用。
    new_contact_type_id = int(
        inserted_rows[0]["id"]
    )

    # 添加新 row。
    contact_type_rows.append(
        {
            "id": new_contact_type_id,
            "contact_type_name": canonical_name,
        }
    )

    # 返回新 id。
    return new_contact_type_id


# 解析或创建 Email contact type。
email_contact_type_id = (
    resolve_or_create_contact_type(
        "Email",
        email_contact_type_aliases,
    )
)

# 解析或创建 Phone contact type。
phone_contact_type_id = (
    resolve_or_create_contact_type(
        "Phone",
        phone_contact_type_aliases,
    )
)

# 显示最终 contact type ids。
print(
    "Resolved Contact_type ids:",
    {
        "email_contact_type_id": (
            email_contact_type_id
        ),
        "phone_contact_type_id": (
            phone_contact_type_id
        ),
    },
)


# ============================================================
# 读取 Candidate identity snapshots
# ============================================================

# 查询 Candidate 与 Person 映射及身份快照。
identity_result = execute_d1(
    """
    SELECT
      p."id" AS person_id,
      p."email" AS primary_person_email,
      p."phone" AS primary_person_phone,
      p."latest_candidate_id",
      app."id" AS application_id,
      cs."id" AS candidate_snapshot_id,
      cs."application_date",
      cs."raw_person_name",
      cs."normalized_person_name",
      cs."normalized_first_name",
      cs."normalized_middle_name",
      cs."normalized_last_name",
      cs."raw_email_address",
      cs."normalized_email_address",
      cs."raw_phone",
      cs."normalized_phone",
      cs."normalized_linkedin_url",
      cs."normalized_github_url"
    FROM "Application" AS app
    INNER JOIN "Person" AS p
      ON p."id" = app."person_id"
    INNER JOIN "Candidate_snapshot" AS cs
      ON cs."application_id" = app."id"
    ORDER BY
      p."id",
      cs."application_date",
      cs."id";
    """
)

# 提取 identity rows。
identity_rows = (
    identity_result[0].get(
        "results",
        [],
    )
    if identity_result
    else []
)

# 没有数据时停止。
if not identity_rows:
    raise ValueError(
        "没有找到 Candidate identity snapshot records。"
    )

# ============================================================
# 生成 Person_Name 计划
# ============================================================

# 使用 (person_id, name_value) 保存 distinct name。
# 与现有 Person_Name 唯一索引语义一致。
planned_name_by_key = {}

# 逐条收集 raw 和 normalized name。
for identity_row in identity_rows:
    # 读取 Person.id。
    person_id = int(
        identity_row[
            "person_id"
        ]
    )

    # 读取 Candidate.id。
    candidate_snapshot_id = int(
        identity_row[
            "candidate_snapshot_id"
        ]
    )

    # application date 作为 seen 时间。
    seen_at = require_application_date(
        identity_row
    )

    # 当前 Candidate 的 normalized name 是 primary 候选。
    normalized_person_name = clean_text_value(
        identity_row[
            "normalized_person_name"
        ]
    )

    # 同时保存 raw name 和 normalized name。
    candidate_name_values = [
        clean_text_value(
            identity_row[
                "raw_person_name"
            ]
        ),
        normalized_person_name,
    ]

    # 逐个处理非空姓名。
    for name_value in candidate_name_values:
        # 跳过空值。
        if name_value is None:
            continue

        # Person_Name 现有唯一索引是区分大小写的文本值。
        name_key = (
            person_id,
            name_value,
        )

        # 判断是否为最新 Candidate 的 normalized name。
        is_primary = int(
            candidate_snapshot_id
            == int(
                identity_row[
                    "latest_candidate_id"
                ]
            )
            and name_value
            == normalized_person_name
        )

        # 第一次出现时建立 record。
        if name_key not in planned_name_by_key:
            planned_name_by_key[
                name_key
            ] = {
                "person_id": person_id,
                "name_value": name_value,
                "is_primary": is_primary,
                "source_submission_id": None,
                "first_seen_at": seen_at,
                "last_seen_at": seen_at,
                "source_candidate_id": (
                    candidate_snapshot_id
                ),
            }
        else:
            # 取得已有计划。
            planned_name_record = (
                planned_name_by_key[
                    name_key
                ]
            )

            # 更新时间范围。
            planned_name_record[
                "first_seen_at"
            ] = min(
                planned_name_record[
                    "first_seen_at"
                ],
                seen_at,
            )

            # 更新时间范围。
            planned_name_record[
                "last_seen_at"
            ] = max(
                planned_name_record[
                    "last_seen_at"
                ],
                seen_at,
            )

            # 任意最新 Candidate 匹配都会标记 primary。
            planned_name_record[
                "is_primary"
            ] = max(
                int(
                    planned_name_record[
                        "is_primary"
                    ]
                ),
                is_primary,
            )

# 转换成稳定顺序 list。
planned_name_records = sorted(
    planned_name_by_key.values(),
    key=lambda record: (
        record[
            "person_id"
        ],
        record[
            "name_value"
        ],
    ),
)


# ============================================================
# 生成 Person_Contact 计划
# ============================================================

# 使用 Person + Contact type + normalized value 去重。
planned_contact_by_key = {}

# 逐条收集邮箱和电话。
for identity_row in identity_rows:
    # 读取 Person.id。
    person_id = int(
        identity_row[
            "person_id"
        ]
    )

    # 读取 Candidate.id。
    candidate_snapshot_id = int(
        identity_row[
            "candidate_snapshot_id"
        ]
    )

    # 读取 seen time。
    seen_at = require_application_date(
        identity_row
    )

    # 定义标准化 email 和 phone 候选。
    contact_candidates = [
        (
            email_contact_type_id,
            clean_text_value(
                identity_row[
                    "normalized_email_address"
                ]
            ),
            clean_text_value(
                identity_row[
                    "primary_person_email"
                ]
            ),
        ),
        (
            phone_contact_type_id,
            clean_text_value(
                identity_row[
                    "normalized_phone"
                ]
            ),
            clean_text_value(
                identity_row[
                    "primary_person_phone"
                ]
            ),
        ),
    ]

    # 逐个处理 contact。
    for (
        contact_type_id,
        contact_value,
        primary_value,
    ) in contact_candidates:
        # 跳过空值。
        if contact_value is None:
            continue

        # 使用小写 normalized key。
        contact_key = (
            person_id,
            int(
                contact_type_id
            ),
            normalized_key_text(
                contact_value
            ),
        )

        # Person 当前 canonical 值相同则标记 primary。
        is_primary = int(
            normalized_key_text(
                contact_value
            )
            == normalized_key_text(
                primary_value
            )
        )

        # 第一次出现时建立计划。
        if contact_key not in planned_contact_by_key:
            planned_contact_by_key[
                contact_key
            ] = {
                "person_id": person_id,
                "contact_type_id": int(
                    contact_type_id
                ),
                "contact_value": contact_value,
                "is_primary": is_primary,
                "created_at": seen_at,
                "updated_at": seen_at,
                "source_candidate_id": (
                    candidate_snapshot_id
                ),
            }
        else:
            # 取得已有计划。
            planned_contact_record = (
                planned_contact_by_key[
                    contact_key
                ]
            )

            # 保存首次出现时间。
            planned_contact_record[
                "created_at"
            ] = min(
                planned_contact_record[
                    "created_at"
                ],
                seen_at,
            )

            # 保存最近出现时间。
            planned_contact_record[
                "updated_at"
            ] = max(
                planned_contact_record[
                    "updated_at"
                ],
                seen_at,
            )

            # 更新 primary。
            planned_contact_record[
                "is_primary"
            ] = max(
                int(
                    planned_contact_record[
                        "is_primary"
                    ]
                ),
                is_primary,
            )

# 转换成稳定顺序 list。
planned_contact_records = sorted(
    planned_contact_by_key.values(),
    key=lambda record: (
        record[
            "person_id"
        ],
        record[
            "contact_type_id"
        ],
        normalized_key_text(
            record[
                "contact_value"
            ]
        ),
    ),
)


# ============================================================
# 生成 Person_Link 计划
# ============================================================

# 使用 Person + link_type + normalized URL 去重。
planned_link_by_key = {}

# 逐条收集 LinkedIn 和 GitHub。
for identity_row in identity_rows:
    # 读取 Person.id。
    person_id = int(
        identity_row[
            "person_id"
        ]
    )

    # 读取 Candidate.id。
    candidate_snapshot_id = int(
        identity_row[
            "candidate_snapshot_id"
        ]
    )

    # 读取 seen time。
    seen_at = require_application_date(
        identity_row
    )

    # 定义链接候选。
    link_candidates = [
        (
            "linkedin",
            clean_text_value(
                identity_row[
                    "normalized_linkedin_url"
                ]
            ),
        ),
        (
            "github",
            clean_text_value(
                identity_row[
                    "normalized_github_url"
                ]
            ),
        ),
    ]

    # 逐个处理链接。
    for link_type, url in link_candidates:
        # 跳过空 URL。
        if url is None:
            continue

        # 建立 normalized key。
        link_key = (
            person_id,
            link_type,
            normalized_key_text(
                url
            ),
        )

        # 第一次出现时建立计划。
        if link_key not in planned_link_by_key:
            planned_link_by_key[
                link_key
            ] = {
                "person_id": person_id,
                "link_type": link_type,
                "url": url,
                "created_at": seen_at,
                "updated_at": seen_at,
                "source_candidate_id": (
                    candidate_snapshot_id
                ),
            }
        else:
            # 取得已有计划。
            planned_link_record = (
                planned_link_by_key[
                    link_key
                ]
            )

            # 保存首次出现时间。
            planned_link_record[
                "created_at"
            ] = min(
                planned_link_record[
                    "created_at"
                ],
                seen_at,
            )

            # 保存最近出现时间。
            planned_link_record[
                "updated_at"
            ] = max(
                planned_link_record[
                    "updated_at"
                ],
                seen_at,
            )

# 转换成稳定顺序 list。
planned_link_records = sorted(
    planned_link_by_key.values(),
    key=lambda record: (
        record[
            "person_id"
        ],
        record[
            "link_type"
        ],
        normalized_key_text(
            record[
                "url"
            ]
        ),
    ),
)

# 显示计划统计。
print(
    "Person identity history plan summary:",
    {
        "person_name_records": int(
            len(
                planned_name_records
            )
        ),
        "person_contact_records": int(
            len(
                planned_contact_records
            )
        ),
        "person_link_records": int(
            len(
                planned_link_records
            )
        ),
    },
)

# 显示各表计划预览。
display(
    pd.DataFrame(
        planned_name_records
    ).head(
        10
    )
)

# 显示 contact 预览。
display(
    pd.DataFrame(
        planned_contact_records
    ).head(
        10
    )
)

# 显示 link 预览。
display(
    pd.DataFrame(
        planned_link_records
    ).head(
        10
    )
)


# ============================================================
# 读取三张表已有记录并建立 idempotency mapping
# ============================================================

# 查询已有 Person_Name。
existing_name_result = execute_d1(
    """
    SELECT
      "id",
      "person_id",
      "name_value",
      "is_primary",
      "source_submission_id",
      "first_seen_at",
      "last_seen_at",
      "source_candidate_id"
    FROM "Person_Name";
    """
)

# 提取 rows。
existing_name_rows = (
    existing_name_result[0].get(
        "results",
        [],
    )
    if existing_name_result
    else []
)

# 建立 name key mapping。
existing_name_by_key = {
    (
        int(
            row[
                "person_id"
            ]
        ),
        str(
            row[
                "name_value"
            ]
        ),
    ): row
    for row in existing_name_rows
}

# 查询已有 Person_Contact。
existing_contact_result = execute_d1(
    """
    SELECT
      "id",
      "person_id",
      "contact_type_id",
      "contact_value",
      "is_primary",
      "created_at",
      "updated_at",
      "source_candidate_id"
    FROM "Person_Contact";
    """
)

# 提取 rows。
existing_contact_rows = (
    existing_contact_result[0].get(
        "results",
        [],
    )
    if existing_contact_result
    else []
)

# 建立 contact key mapping。
existing_contact_by_key = {
    (
        int(
            row[
                "person_id"
            ]
        ),
        int(
            row[
                "contact_type_id"
            ]
        ),
        normalized_key_text(
            row[
                "contact_value"
            ]
        ),
    ): row
    for row in existing_contact_rows
}

# 查询已有 Person_Link。
existing_link_result = execute_d1(
    """
    SELECT
      "id",
      "person_id",
      "link_type",
      "url",
      "created_at",
      "updated_at",
      "source_candidate_id"
    FROM "Person_Link";
    """
)

# 提取 rows。
existing_link_rows = (
    existing_link_result[0].get(
        "results",
        [],
    )
    if existing_link_result
    else []
)

# 建立 link key mapping。
existing_link_by_key = {
    (
        int(
            row[
                "person_id"
            ]
        ),
        normalized_key_text(
            row[
                "link_type"
            ]
        ),
        normalized_key_text(
            row[
                "url"
            ]
        ),
    ): row
    for row in existing_link_rows
}


# ============================================================
# 分离需要 INSERT、UPDATE 和 REUSE 的 records
# ============================================================

# 保存各表新记录。
new_name_records = []
new_contact_records = []
new_link_records = []

# 保存各表更新计划。
name_update_pairs = []
contact_update_pairs = []
link_update_pairs = []

# 保存复用统计。
reused_name_count = 0
reused_contact_count = 0
reused_link_count = 0

# 分类 Person_Name。
for planned_record in planned_name_records:
    # 建立 key。
    record_key = (
        planned_record[
            "person_id"
        ],
        planned_record[
            "name_value"
        ],
    )

    # 查找已有记录。
    existing_record = existing_name_by_key.get(
        record_key
    )

    # 不存在时 INSERT。
    if existing_record is None:
        new_name_records.append(
            planned_record
        )
        continue

    # 需要保持首次 source_candidate_id，不覆盖来源。
    comparable_columns = [
        "is_primary",
        "first_seen_at",
        "last_seen_at",
    ]

    # 判断是否有变化。
    has_difference = any(
        not values_equal(
            existing_record.get(
                column_name
            ),
            planned_record[
                column_name
            ],
        )
        for column_name
        in comparable_columns
    )

    # 保存 UPDATE 或 REUSE。
    if has_difference:
        name_update_pairs.append(
            (
                existing_record,
                planned_record,
            )
        )
    else:
        reused_name_count += 1

# 分类 Person_Contact。
for planned_record in planned_contact_records:
    # 建立 key。
    record_key = (
        planned_record[
            "person_id"
        ],
        planned_record[
            "contact_type_id"
        ],
        normalized_key_text(
            planned_record[
                "contact_value"
            ]
        ),
    )

    # 查找已有记录。
    existing_record = (
        existing_contact_by_key.get(
            record_key
        )
    )

    # 不存在时 INSERT。
    if existing_record is None:
        new_contact_records.append(
            planned_record
        )
        continue

    # 判断 primary 和时间是否变化。
    comparable_columns = [
        "is_primary",
        "created_at",
        "updated_at",
    ]

    # 检查差异。
    has_difference = any(
        not values_equal(
            existing_record.get(
                column_name
            ),
            planned_record[
                column_name
            ],
        )
        for column_name
        in comparable_columns
    )

    # 保存 UPDATE 或 REUSE。
    if has_difference:
        contact_update_pairs.append(
            (
                existing_record,
                planned_record,
            )
        )
    else:
        reused_contact_count += 1

# 分类 Person_Link。
for planned_record in planned_link_records:
    # 建立 key。
    record_key = (
        planned_record[
            "person_id"
        ],
        planned_record[
            "link_type"
        ],
        normalized_key_text(
            planned_record[
                "url"
            ]
        ),
    )

    # 查找已有记录。
    existing_record = existing_link_by_key.get(
        record_key
    )

    # 不存在时 INSERT。
    if existing_record is None:
        new_link_records.append(
            planned_record
        )
        continue

    # 判断时间是否变化。
    comparable_columns = [
        "created_at",
        "updated_at",
    ]

    # 检查差异。
    has_difference = any(
        not values_equal(
            existing_record.get(
                column_name
            ),
            planned_record[
                column_name
            ],
        )
        for column_name
        in comparable_columns
    )

    # 保存 UPDATE 或 REUSE。
    if has_difference:
        link_update_pairs.append(
            (
                existing_record,
                planned_record,
            )
        )
    else:
        reused_link_count += 1


# ============================================================
# 批量 INSERT 新记录
# ============================================================

# 批量插入 Person_Name。
inserted_name_count = batch_insert_records(
    "Person_Name",
    [
        "person_id",
        "name_value",
        "is_primary",
        "source_submission_id",
        "first_seen_at",
        "last_seen_at",
        "source_candidate_id",
    ],
    new_name_records,
)

# 批量插入 Person_Contact。
inserted_contact_count = (
    batch_insert_records(
        "Person_Contact",
        [
            "person_id",
            "contact_type_id",
            "contact_value",
            "is_primary",
            "created_at",
            "updated_at",
            "source_candidate_id",
        ],
        new_contact_records,
    )
)

# 批量插入 Person_Link。
inserted_link_count = batch_insert_records(
    "Person_Link",
    [
        "person_id",
        "link_type",
        "url",
        "created_at",
        "updated_at",
        "source_candidate_id",
    ],
    new_link_records,
)


# ============================================================
# UPDATE 发生变化的已有记录
# ============================================================

# 更新 Person_Name。
for existing_record, planned_record in (
    name_update_pairs
):
    # 执行 UPDATE。
    execute_d1(
        """
        UPDATE "Person_Name"
        SET
          "is_primary" = ?,
          "first_seen_at" = ?,
          "last_seen_at" = ?
        WHERE "id" = ?;
        """,
        [
            int(
                planned_record[
                    "is_primary"
                ]
            ),
            planned_record[
                "first_seen_at"
            ],
            planned_record[
                "last_seen_at"
            ],
            int(
                existing_record[
                    "id"
                ]
            ),
        ],
    )

# 更新 Person_Contact。
for existing_record, planned_record in (
    contact_update_pairs
):
    # 执行 UPDATE。
    execute_d1(
        """
        UPDATE "Person_Contact"
        SET
          "is_primary" = ?,
          "created_at" = ?,
          "updated_at" = ?
        WHERE "id" = ?;
        """,
        [
            int(
                planned_record[
                    "is_primary"
                ]
            ),
            planned_record[
                "created_at"
            ],
            planned_record[
                "updated_at"
            ],
            int(
                existing_record[
                    "id"
                ]
            ),
        ],
    )

# 更新 Person_Link。
for existing_record, planned_record in (
    link_update_pairs
):
    # 执行 UPDATE。
    execute_d1(
        """
        UPDATE "Person_Link"
        SET
          "created_at" = ?,
          "updated_at" = ?
        WHERE "id" = ?;
        """,
        [
            planned_record[
                "created_at"
            ],
            planned_record[
                "updated_at"
            ],
            int(
                existing_record[
                    "id"
                ]
            ),
        ],
    )


# ============================================================
# 写入后验证
# ============================================================

# 查询目标表数量。
verification_result = execute_d1(
    """
    SELECT
      (SELECT COUNT(*) FROM "Person_Name")
        AS person_name_count,
      (SELECT COUNT(*) FROM "Person_Contact")
        AS person_contact_count,
      (SELECT COUNT(*) FROM "Person_Link")
        AS person_link_count;
    """
)

# 提取 summary。
verification_summary = (
    verification_result[0]["results"][0]
)

# 三张表的总数必须等于计划 distinct records 数量。
expected_counts = {
    "person_name_count": len(
        planned_name_records
    ),
    "person_contact_count": len(
        planned_contact_records
    ),
    "person_link_count": len(
        planned_link_records
    ),
}

# 逐项验证。
for count_name, expected_count in (
    expected_counts.items()
):
    # 数量不一致时停止。
    if int(
        verification_summary[
            count_name
        ]
    ) != int(
        expected_count
    ):
        raise RuntimeError(
            "Person identity history 数量验证失败："
            f"{count_name}, expected={expected_count}, "
            f"actual={verification_summary[count_name]}"
        )

# 检查外键。
foreign_key_result = execute_d1(
    """
    PRAGMA foreign_key_check;
    """
)

# 提取违规 rows。
foreign_key_rows = (
    foreign_key_result[0].get(
        "results",
        [],
    )
    if foreign_key_result
    else []
)

# 任何违规都停止。
if foreign_key_rows:
    raise RuntimeError(
        "Person identity history 写入后发现外键违规："
        f"{foreign_key_rows[:20]}"
    )

# 建立结果 DataFrame。
identity_import_result_dataframe = (
    pd.DataFrame(
        [
            {
                "table_name": "Person_Name",
                "planned_count": len(
                    planned_name_records
                ),
                "inserted_count": (
                    inserted_name_count
                ),
                "updated_count": len(
                    name_update_pairs
                ),
                "reused_count": (
                    reused_name_count
                ),
            },
            {
                "table_name": "Person_Contact",
                "planned_count": len(
                    planned_contact_records
                ),
                "inserted_count": (
                    inserted_contact_count
                ),
                "updated_count": len(
                    contact_update_pairs
                ),
                "reused_count": (
                    reused_contact_count
                ),
            },
            {
                "table_name": "Person_Link",
                "planned_count": len(
                    planned_link_records
                ),
                "inserted_count": (
                    inserted_link_count
                ),
                "updated_count": len(
                    link_update_pairs
                ),
                "reused_count": (
                    reused_link_count
                ),
            },
        ]
    )
)

# 显示结果。
display(
    identity_import_result_dataframe
)

# 显示完成 summary。
print(
    "Person identity history import summary:",
    {
        "person_name_count": int(
            verification_summary[
                "person_name_count"
            ]
        ),
        "person_contact_count": int(
            verification_summary[
                "person_contact_count"
            ]
        ),
        "person_link_count": int(
            verification_summary[
                "person_link_count"
            ]
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_rows
            )
        ),
    },
)

# 显示完成信息。
print(
    "Person identity history import completed successfully."
)


### 教育经历和学位提取 + 生成csv存入Google drive (可以重复运行)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 使用确定性规则从 Candidate 简历中提取全部教育经历，
# 判断每份 Candidate 简历中出现过的最高学历，
# 并覆盖写入 Google Drive CSV。
#
# 重要：
# 1. 本 Cell 不使用任何语言模型或外部 AI API。
# 2. 本 Cell 不写入 D1 任何业务表。
# 3. 本 Cell 不依赖其他 Colab Cell 中的变量、function 或 class。
# 4. Candidate/Application/Person mapping 从 D1 独立读取。
# 5. resume_text 从固定路径 application.csv 读取。
# 6. raw resume 优先；现有 education summary 只做保守 fallback。
# 7. 相同输入和相同 RULE_VERSION 会生成相同结果。
# 8. 输出 CSV 使用固定文件名覆盖，不会无限增加版本文件。
# 9. 不区分学位是否已经完成；只看简历中出现的最高层级。
# 10. 最终输出：candidate_education_rule_extraction.csv 和 candidate_education_rule_extraction_status.csv
# ============================================================

# os 用于检查 Google Drive 输入文件。
import os

# re 用于确定性文本和学位规则匹配。
import re

# time 只用于 D1 网络请求失败后的重试等待。
import time

# pandas 用于读取和写入 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定规则版本与文件路径
# ============================================================

# 每次修改教育提取规则时应显式修改这个版本。
RULE_VERSION = "candidate_education_rule_v4"

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# application.csv 提供 Application.id、raw_submission_id 和 resume_text。
APPLICATION_CSV_PATH = (
    f"{DATASET_FOLDER}/dedup_results/"
    "application_dedup_v2/application.csv"
)

# 现有 CV extraction summary 只作为 raw parser 无结果时的 fallback。
RESUME_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "resume_extraction_consolidated.csv"
)

# 结构化教育记录固定覆盖输出。
EDUCATION_OUTPUT_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_education_rule_extraction.csv"
)

# 每个 Candidate 的提取状态固定覆盖输出。
EDUCATION_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_education_rule_extraction_status.csv"
)

# 检查必要输入。
for required_path in [
    APPLICATION_CSV_PATH,
    RESUME_EXTRACTION_CSV_PATH,
]:
    # 输入不存在时停止。
    if not os.path.exists(
        required_path
    ):
        raise FileNotFoundError(
            "找不到必要输入文件："
            f"{required_path}"
        )


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# token 不存在时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存请求 headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行一条参数化 D1 SQL，并对临时失败有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在重试次数中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送请求。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 JSON。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # 成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败时停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


# ============================================================
# 通用清理 functions
# ============================================================

def clean_text_value(
    value,
):
    """把 pandas 空值转成 None，其余值转成清理后的文本。"""

    # Python None 保持 None。
    if value is None:
        return None

    # pandas NA/NaN 转成 None。
    try:
        if pd.isna(
            value
        ):
            return None
    except (TypeError, ValueError):
        pass

    # 转成标准 string。
    cleaned_value = str(
        value
    ).strip()

    # 空 string 和文本化空值转成 None。
    if (
        not cleaned_value
        or cleaned_value.lower()
        in {
            "nan",
            "none",
            "<na>",
        }
    ):
        return None

    # 返回文本。
    return cleaned_value


def normalize_space(
    value,
):
    """合并文本中的连续空白，但保留原始大小写。"""

    # 清理空值。
    cleaned_value = clean_text_value(
        value
    )

    # 空值返回 None。
    if cleaned_value is None:
        return None

    # 合并连续空白。
    return re.sub(
        r"\s+",
        " ",
        cleaned_value,
    ).strip()


def normalized_lower_text(
    value,
):
    """生成 lower-case normalized text。"""

    # 合并空白。
    normalized_value = normalize_space(
        value
    )

    # 空值返回 None。
    if normalized_value is None:
        return None

    # 标准化为小写。
    return normalized_value.lower()


# ============================================================
# Education section 和 institution 规则
# ============================================================

# Education section 标题。
EDUCATION_HEADER_PATTERN = re.compile(
    r"^\s*(?:education|education background|educational background|"
    r"academic background|academic history|学历|教育|教育背景)"
    r"\s*[:：]?\s*$",
    re.IGNORECASE,
)

# 遇到这些标题时 Education section 结束。
SECTION_END_PATTERN = re.compile(
    r"^\s*(?:work experiences?|professional experiences?|employment|"
    r"employment history|career history|experience|internship experience|"
    r"work\s*(?:&|and)\s*leadership experience|"
    r"professional\s*(?:&|and)\s*leadership experience|"
    r"academic experience|liberal arts experience|"
    r"writing\s*(?:&|and)\s*art experience|community engagement experience|"
    r"relevant experience|teaching experiences?|"
    r"research\s*(?:&|and)\s*development achievements?|"
    r"project experiences?|selected projects?|projects?|research experiences?|"
    r"skills?|technical skills?|leadership|leadership experience|"
    r"leadership\s*(?:&|and)\s*extracurricular activities|"
    r"activities|academic involvement|extracurricular activities|"
    r"extracurriculars?\s*(?:&|and)\s*internships?|"
    r"volunteer\s*(?:&|and)\s*extracurricular activities|"
    r"soft skills?|conference presentations?|"
    r"certifications?|awards?|publications?|additional information|"
    r"other information|工作经历|项目经历|技能)\s*[:：]?\s*$",
    re.IGNORECASE,
)

# 学校/教育机构关键字。
INSTITUTION_KEYWORD_PATTERN = re.compile(
    r"\b(?:university|college|school|institute|academy|polytechnic|"
    r"conservatory|seminary|faculty)\b",
    re.IGNORECASE,
)

# 常见教育机构缩写。
INSTITUTION_ABBREVIATION_PATTERN = re.compile(
    r"(?:^|[\s,(])(?:MIT|NYU|UCLA|USC|UCSD|UCSB|UCI|UCD|UCR|"
    r"UCF|UBC|LSE|ETH|EPFL|CALTECH|JUilliard)(?:$|[\s,)])",
    re.IGNORECASE,
)

# 明显不是学校名称的内容。
NON_INSTITUTION_PATTERN = re.compile(
    r"^\s*(?:expected|expected graduation|class|gpa|awards?|honors?|"
    r"member|coursework|courses?|projects?|skills?|"
    r"invited|developed|conducted|analyzed|created|built|led|managed|"
    r"assisted|supported|reported|committed|independently|participated|"
    r"under|academics?|interests?|partial coursework|"
    r"activities?|supervisor|designed|received|featured|co-founded|"
    r"certificates?|cumulative gpa|within|delivered|"
    r"graduate research assistant|"
    r"teaching assistant|research assistant|student assistant|"
    r"silver prize|bronze prize|gold prize)\b|"
    r"\b(?:intern|internship|project|team|database|forecasting|"
    r"proficient|performance|responsibilities|school-level|"
    r"club|conference|competition|league|award|scholarship|dean['’]s list|"
    r"research assistant|student assistant)\b",
    re.IGNORECASE,
)

# 简历 bullet 前缀。
BULLET_PREFIX_PATTERN = re.compile(
    r"^\s*[•●▪■◆◇►▶\-–—]+\s*"
)

# 常见地点后缀；只用于学校行末尾保守清理。
LOCATION_SUFFIX_PATTERN = re.compile(
    r"\s{2,}(?:[A-Z][A-Za-z .'-]+,\s*)?"
    r"(?:[A-Z]{2}|USA|United States|China|Canada|UK)\s*$"
)


def is_institution_line(
    line,
):
    """判断一行是否可以作为一段教育经历的学校锚点。"""

    # 清理行。
    cleaned_line = normalize_space(
        line
    )

    # 空行不是机构。
    if cleaned_line is None:
        return False

    # 太长的自然语言句子通常不是学校名称行。
    if len(
        cleaned_line
    ) > 160:
        return False

    # 超过18个单词的行通常是描述句，而不是学校名称。
    if len(
        cleaned_line.split()
    ) > 18:
        return False

    # 排除明显错误内容。
    if NON_INSTITUTION_PATTERN.search(
        cleaned_line
    ):
        return False

    # 查找机构关键字。
    institution_keyword_match = (
        INSTITUTION_KEYWORD_PATTERN.search(
            cleaned_line
        )
    )

    # 包含机构关键字时继续检查它在行中的位置。
    if institution_keyword_match is not None:
        # 机构关键字以前如果已经有很长的自然语言，
        # 这一行通常是在描述中提到某所大学，不是学校名称。
        prefix_text = cleaned_line[
            :institution_keyword_match.start()
        ].strip()

        # Study Abroad Program: School Name 是允许的特殊前缀。
        study_abroad_prefix = bool(
            re.fullmatch(
                r"(?:one-year\s+)?study abroad(?:\s+program)?\s*[:：]?\s*",
                prefix_text,
                flags=re.IGNORECASE,
            )
        )

        # 一般学校名称在 institution keyword 前不会超过八个单词。
        if (
            len(
                prefix_text.split()
            )
            > 8
            and not study_abroad_prefix
        ):
            return False

        # “University Academic Achievement Award” 不是学校名称。
        suffix_text = cleaned_line[
            institution_keyword_match.end():
        ].strip()

        # 排除机构关键字后直接出现的奖项/活动描述。
        if re.match(
            r"^(?:academic achievement award|award|honor|"
            r"student assistant|research assistant|teaching assistant)\b",
            suffix_text,
            flags=re.IGNORECASE,
        ):
            return False

        # 通过检查。
        return True

    # 命中明确学校缩写时接受。
    abbreviation_match = (
        INSTITUTION_ABBREVIATION_PATTERN.search(
            cleaned_line
        )
    )

    # 学校缩写也不能埋在很长的自然语言后面。
    if (
        abbreviation_match is not None
        and len(
            cleaned_line[
                :abbreviation_match.start()
            ].split()
        )
        <= 8
    ):
        return True

    # 其余内容不作为学校锚点。
    return False


def extract_education_section_lines(
    resume_text,
):
    """从完整 resume_text 中提取 Education section lines。"""

    # 清理 resume。
    cleaned_resume_text = clean_text_value(
        resume_text
    )

    # 空简历没有 section。
    if cleaned_resume_text is None:
        return [], "no_resume_text"

    # 统一换行。
    normalized_resume_text = (
        cleaned_resume_text.replace(
            "\r\n",
            "\n",
        ).replace(
            "\r",
            "\n",
        )
    )

    # 保存原始行。
    all_lines = normalized_resume_text.split(
        "\n"
    )

    # 保存 Education section lines。
    education_lines = []

    # 当前是否位于 Education section。
    in_education_section = False

    # 是否找到标题。
    found_education_header = False

    # 逐行处理。
    for line in all_lines:
        # 用去除首尾空格的行做标题判断。
        stripped_line = str(
            line
        ).strip()

        # 找到 Education 标题。
        if EDUCATION_HEADER_PATTERN.match(
            stripped_line
        ):
            # 开始收集。
            in_education_section = True
            found_education_header = True
            continue

        # Education 开始以后遇到下一 section 标题。
        if (
            in_education_section
            and SECTION_END_PATTERN.match(
                stripped_line
            )
        ):
            break

        # 只收集 Education section 内部行。
        if in_education_section:
            education_lines.append(
                line
            )

    # 找到标题时返回 section。
    if found_education_header:
        return (
            education_lines,
            "explicit_education_section",
        )

    # 没找到标题时不扫描整份简历，
    # 避免把工作经历中的 University 雇主误判为教育经历。
    return [], "no_explicit_education_section"


def split_education_entries(
    education_lines,
):
    """以 institution line 为锚点拆分教育经历。"""

    # 保存 entries。
    entries = []

    # 保存当前 entry lines。
    current_entry_lines = []

    # 保存第一个学校锚点以前的少量行。
    pre_anchor_lines = []

    # 逐行处理。
    for line in education_lines:
        # 保留原始行但清理两端空格。
        stripped_line = str(
            line
        ).strip()

        # 跳过空行。
        if not stripped_line:
            continue

        # 判断是否是学校锚点。
        if is_institution_line(
            stripped_line
        ):
            # 已有当前 entry 时先保存。
            if current_entry_lines:
                entries.append(
                    current_entry_lines
                )

            # 学位行偶尔出现在学校行前面；
            # 只把最多前两行附到第一个 entry。
            current_entry_lines = (
                pre_anchor_lines[-2:]
                + [
                    stripped_line,
                ]
            )

            # 锚点找到后清空前置行。
            pre_anchor_lines = []
        else:
            # 已经找到学校时归入当前 entry。
            if current_entry_lines:
                current_entry_lines.append(
                    stripped_line
                )
            else:
                # 尚未找到学校时暂存。
                pre_anchor_lines.append(
                    stripped_line
                )

    # 保存最后一个 entry。
    if current_entry_lines:
        entries.append(
            current_entry_lines
        )

    # 返回锚点 entries。
    return entries


# ============================================================
# Degree alias 和层级规则
# ============================================================

# Pattern 顺序从最高学历到最低学历。
# 每个类别内部优先匹配较具体的写法。
DEGREE_CATEGORY_RULES = [
    (
        "Doctorate",
        5,
        re.compile(
            r"\b(?:Ph\.?\s*D\.?|DPhil|Ed\.?\s*D\.?|DBA|"
            r"J\.\s*D\.?|M\.\s*D\.?|doctor(?:ate|al)?|"
            r"doctor of philosophy|doctor of education|"
            r"doctor of medicine|juris doctor)"
            r"(?=$|[^A-Za-z0-9])",
            re.IGNORECASE,
        ),
    ),
    (
        "Master",
        4,
        re.compile(
            r"\b(?:M\.?\s*B\.?\s*A\.?|M\.?\s*S\.?|MSc|M\.?\s*A\.?|"
            r"MSE|MEd|MPhil|MFA|MPS|MTech|"
            r"MEng|M\.?\s*Eng\.?|MPH|MFin|MAcc|LLM|M\.?\s*P\.?\s*A\.?|"
            r"M\.?\s*P\.?\s*P\.?|MEM|master(?:'s)?|"
            r"master of [A-Za-z& ]+)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Bachelor",
        3,
        re.compile(
            r"\b(?:B\.?\s*S\.?|BSc|B\.?\s*A\.?|BBA|B\.?\s*B\.?\s*A\.?|"
            r"BEng|B\.?\s*Eng\.?|BTech|BASc|BFA|BCom|LLB|"
            r"bachelor(?:'s)?|bachelor of [A-Za-z& ]+)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Associate",
        2,
        re.compile(
            r"\b(?:A\.\s*A\.\s*S\.?|AAS|A\.\s*A\.?|A\.\s*S\.?|"
            r"associate(?:'s)?|associate of (?:arts|science))\b",
            re.IGNORECASE,
        ),
    ),
    (
        "High School",
        1,
        re.compile(
            r"\b(?:high school|high school diploma|secondary school diploma|"
            r"general educational development|GED)\b",
            re.IGNORECASE,
        ),
    ),
]


def degree_match_is_reliable(
    degree_match,
    degree_category,
    source_line,
):
    """检查容易歧义的短缩写是否真的出现在学历语境中。"""

    # 保存匹配文本。
    matched_text = degree_match.group(
        0
    )

    # 保存统一后的匹配文本。
    compact_match = re.sub(
        r"[^a-z]",
        "",
        matched_text.lower(),
    )

    # 保存统一后的当前行。
    normalized_line = normalize_space(
        source_line
    ) or ""

    # PhD 出现在 courses/coursework 描述中时不是申请人的学位。
    if (
        compact_match == "phd"
        and re.search(
            r"\b(?:course|courses|coursework)\b",
            normalized_line,
            flags=re.IGNORECASE,
        )
    ):
        return False

    # PhD 必须描述当前申请人的学位，
    # 不能只是导师姓名后的 Ph.D.、研究团队里的 PhD student，
    # 或研究对象中的 PhD graduate。
    if compact_match == "phd":
        # Advisor/Researcher for NAME, Ph.D. 明确描述的是别人。
        if re.search(
            r"\b(?:advisor|researcher\s+for|supervisor|professor|prof\.)\b",
            normalized_line,
            flags=re.IGNORECASE,
        ):
            return False

        # “PhD graduate + 人名”描述的是研究对象或第三方。
        if re.search(
            r"\bPh\.?\s*D\.?\s+graduate\s+[A-Z][A-Za-z'-]+",
            normalized_line,
        ):
            return False

        # 只接受明确的申请人学位表达。
        if not re.search(
            r"(?:^|[:;,|])\s*Ph\.?\s*D\.?\b|"
            r"\bPh\.?\s*D\.?\s+(?:in|of|candidate|program|student|"
            r"biosciences?|engineering|science|economics|physics|"
            r"astronomy|psychology|education|computer)\b",
            normalized_line,
            flags=re.IGNORECASE,
        ):
            return False

    # 未带点的 MA/MS 容易与 Massachusetts/Mississippi 州缩写混淆。
    #
    # 例如 “Groton, MA” 不能被识别为 Master of Arts。
    if (
        compact_match
        in {
            "ma",
            "ms",
        }
        and "." not in matched_text
        and not re.search(
            r"(?:^|[:;,|])\s*M[AS]\s+(?:in|of|degree|program)|"
            r"^\s*M[AS]\s*[,;-]\s*[A-Za-z]|"
            r"\b(?:degree|candidate|student|program)\b",
            normalized_line,
            flags=re.IGNORECASE,
        )
    ):
        return False

    # 未带点的 MD/JD 容易分别与 Maryland 地址和普通文本混淆。
    #
    # 当前主规则只直接接受 M.D. / J.D.；
    # 如果未来需要接受无点写法，必须在明确 degree/candidate/program
    # 语境中单独增加规则，不能恢复为无条件匹配。
    if compact_match in {
        "md",
        "jd",
    }:
        return bool(
            re.search(
                r"\b(?:degree|candidate|student|program|"
                r"doctor of medicine|juris doctor)\b",
                normalized_line,
                flags=re.IGNORECASE,
            )
        )

    # High School 只在当前行本身像学校或明确 diploma 时接受。
    #
    # “high school students”“high school contest”等描述不能代表
    # 当前申请人的最高学历。
    if degree_category == "High School":
        if re.search(
            r"\b(?:students?|contest|speaker|community|club|champion)\b",
            normalized_line,
            flags=re.IGNORECASE,
        ):
            return False

        if (
            "diploma" not in normalized_line.lower()
            and (
                len(
                    normalized_line.split()
                )
                > 12
                or not re.search(
                    r"\b(?:high school|secondary school)\b",
                    normalized_line,
                    flags=re.IGNORECASE,
                )
            )
        ):
            return False

    # 其余匹配接受。
    return True


def extract_degree(
    entry_text,
    entry_lines=None,
):
    """从学历条目的学校/学位区域识别学位和标准层级。"""

    # 清理 entry。
    cleaned_entry_text = clean_text_value(
        entry_text
    )

    # 空值归为 Other。
    if cleaned_entry_text is None:
        return None, "Other", 0

    # raw resume 有逐行结构时，只检查学校行和其后的前五行。
    #
    # 这样可以避免 entry 后半部分的课程说明中出现 PhD/MBA，
    # 或下一个 section 没有被识别时把工作内容误认为学位。
    if entry_lines is not None:
        degree_search_lines = [
            normalize_space(
                line
            )
            for line in entry_lines[:6]
            if normalize_space(
                line
            )
        ]
    else:
        # summary fallback 没有逐行结构，只检查传入的 degree 字段。
        degree_search_lines = [
            cleaned_entry_text
        ]

    # 按文本行顺序检查。
    for source_line in degree_search_lines:
        # 同一行存在多个学位时仍按最高层级优先。
        for (
            degree_category,
            degree_level_rank,
            degree_pattern,
        ) in DEGREE_CATEGORY_RULES:
            # 搜索学位。
            degree_match = degree_pattern.search(
                source_line
            )

            # 没有匹配时检查下一个规则。
            if degree_match is None:
                continue

            # 容易歧义的缩写必须通过额外语境检查。
            if not degree_match_is_reliable(
                degree_match,
                degree_category,
                source_line,
            ):
                continue

            # 返回原始匹配文本和标准层级。
            return (
                normalize_space(
                    degree_match.group(
                        0
                    )
                ),
                degree_category,
                degree_level_rank,
            )

    # 无法识别层级时保留 Other。
    return None, "Other", 0


# ============================================================
# School、field、GPA 和日期提取规则
# ============================================================

# 日期范围和单个年月。
YEAR_MONTH_PATTERN = re.compile(
    r"(?<![\d./])"
    r"(?P<year>(?:19|20)\d{2})"
    r"(?:[./-](?P<month>1[0-2]|0?[1-9]))?"
)

# 数字 month/year，例如 08/2019、5-2023。
#
# 这个 pattern 与 YEAR_MONTH_PATTERN 分开，
# 防止 08/2019 - 12/2022 中的 “2019 - 12”
# 被错误理解成 YYYY-MM。
MONTH_YEAR_PATTERN = re.compile(
    r"(?<!\d)"
    r"(?P<month>1[0-2]|0?[1-9])"
    r"[./-]"
    r"(?P<year>(?:19|20)\d{2})"
    r"\b"
)

# 月份名称。
MONTH_NAME_TO_NUMBER = {
    "jan": 1,
    "january": 1,
    "feb": 2,
    "february": 2,
    "mar": 3,
    "march": 3,
    "apr": 4,
    "april": 4,
    "may": 5,
    "jun": 6,
    "june": 6,
    "jul": 7,
    "july": 7,
    "aug": 8,
    "august": 8,
    "sep": 9,
    "sept": 9,
    "september": 9,
    "oct": 10,
    "october": 10,
    "nov": 11,
    "november": 11,
    "dec": 12,
    "december": 12,
}

# GPA。
GPA_PATTERN = re.compile(
    r"\bGPA\s*[:：]?\s*"
    r"(?P<gpa>\d+(?:\.\d+)?(?:\s*/\s*\d+(?:\.\d+)?)?)",
    re.IGNORECASE,
)

# Major/field 标识。
FIELD_PATTERN = re.compile(
    r"\b(?:major(?:ing)?|field of study|concentration)\s*"
    r"(?:in|:|：)?\s*"
    r"(?P<field>[A-Za-z][A-Za-z0-9 &/,+()'\-]{1,100})",
    re.IGNORECASE,
)


def extract_school_from_entry_lines(
    entry_lines,
):
    """从 entry 中找到学校锚点并生成 raw/normalized school。"""

    # 找到第一条 institution line。
    institution_lines = [
        line
        for line in entry_lines
        if is_institution_line(
            line
        )
    ]

    # 没有学校锚点时返回空。
    if not institution_lines:
        return None, None

    # 第一条机构行为原始学校候选。
    raw_school_name = normalize_space(
        institution_lines[0]
    )

    # 删除 bullet。
    raw_school_name = BULLET_PREFIX_PATTERN.sub(
        "",
        raw_school_name,
    )

    # 删除明显日期。
    raw_school_name = YEAR_MONTH_PATTERN.sub(
        "",
        raw_school_name,
    )

    # “High School Diploma, Hong Kong International School”
    # 这类行把学历标签写在真实学校名前面。
    #
    # 删除前置学历标签，保留后面的真实学校名称。
    raw_school_name = re.sub(
        r"^(?:high school diploma|high school degree|high school)"
        r"\s*[:：,;|–—-]\s*"
        r"(?=.+\b(?:school|academy|university|college|institute)\b)",
        "",
        raw_school_name,
        flags=re.IGNORECASE,
    )

    # 找到当前行中的学位位置。
    degree_positions = []

    # 检查全部 degree patterns。
    for (
        degree_category,
        _,
        degree_pattern,
    ) in DEGREE_CATEGORY_RULES:
        # High School 可能本身就是学校正式名称的一部分，
        # 例如 University High School。
        #
        # 不能因为它同时也是学历类别就把学校名截断成 University。
        if degree_category == "High School":
            # 真实学校名中的 High School 应保留，
            # 例如 University High School、Shanghai High School。
            #
            # 但 “High School Diploma / High School Degree”
            # 只是学位名称，不是 School.school_name。
            generic_high_school_label = bool(
                re.match(
                    r"^high school\b",
                    raw_school_name,
                    flags=re.IGNORECASE,
                )
                or re.fullmatch(
                    r"secondary school diploma[\s.,;:()A-Za-z0-9/&!-]*",
                    raw_school_name,
                    flags=re.IGNORECASE,
                )
            )

            # 真实学校名不截断。
            if not generic_high_school_label:
                continue

        # 搜索 degree。
        degree_match = degree_pattern.search(
            raw_school_name
        )

        # 保存最早位置。
        if degree_match is not None:
            degree_positions.append(
                degree_match.start()
            )

    # 学位与学校同一行时，只保留学位以前部分。
    if degree_positions:
        raw_school_name = (
            raw_school_name[
                :min(
                    degree_positions
                )
            ]
        )

    # 清理地点后缀和标点。
    raw_school_name = LOCATION_SUFFIX_PATTERN.sub(
        "",
        raw_school_name,
    ).strip(
        " ,;|–—-"
    )

    # 重新验证清理后仍然像学校。
    if not is_institution_line(
        raw_school_name
    ):
        return None, None

    # normalized school 使用小写和合并空格。
    normalized_school_name = (
        normalized_lower_text(
            raw_school_name
        )
    )

    # 返回原始和标准化学校名称。
    return (
        raw_school_name,
        normalized_school_name,
    )


def extract_field_of_study(
    entry_text,
    raw_degree_name,
):
    """提取 field/major；无法可靠提取时返回 NULL。"""

    # 清理文本。
    cleaned_entry_text = clean_text_value(
        entry_text
    )

    # 空 entry 没有 field。
    if cleaned_entry_text is None:
        return None, None

    # 优先匹配明确 major/field/concentration。
    field_match = FIELD_PATTERN.search(
        cleaned_entry_text
    )

    # 命中时清理到换行、GPA 或日期以前。
    if field_match is not None:
        raw_field = field_match.group(
            "field"
        )

        # 截断常见后续内容。
        raw_field = re.split(
            r"\b(?:GPA|Expected|Graduation|Coursework)\b|[\n;|]",
            raw_field,
            maxsplit=1,
            flags=re.IGNORECASE,
        )[0]

        # 清理。
        raw_field = normalize_space(
            raw_field
        )

        # 有效时返回。
        if raw_field:
            return (
                raw_field,
                normalized_lower_text(
                    raw_field
                ),
            )

    # 学位写成 degree in field 时尝试提取 in 后内容。
    if raw_degree_name is not None:
        # 转义原始学位文本。
        degree_in_pattern = re.compile(
            re.escape(
                raw_degree_name
            )
            + r"\s+in\s+"
            + r"(?P<field>[A-Za-z][A-Za-z0-9 &/,+()'\-]{1,100})",
            re.IGNORECASE,
        )

        # 搜索。
        degree_in_match = degree_in_pattern.search(
            cleaned_entry_text
        )

        # 命中时清理。
        if degree_in_match is not None:
            raw_field = re.split(
                r"\b(?:GPA|Expected|Graduation|Coursework)\b|[\n;|]",
                degree_in_match.group(
                    "field"
                ),
                maxsplit=1,
                flags=re.IGNORECASE,
            )[0]

            # 合并空白。
            raw_field = normalize_space(
                raw_field
            )

            # 返回。
            if raw_field:
                return (
                    raw_field,
                    normalized_lower_text(
                        raw_field
                    ),
                )

    # 无可靠 field。
    return None, None


def extract_gpa(
    entry_text,
):
    """提取原始 GPA 文本。"""

    # 清理文本。
    cleaned_entry_text = clean_text_value(
        entry_text
    )

    # 空值没有 GPA。
    if cleaned_entry_text is None:
        return None

    # 搜索 GPA。
    gpa_match = GPA_PATTERN.search(
        cleaned_entry_text
    )

    # 没有命中。
    if gpa_match is None:
        return None

    # 保留数值/分母格式。
    return normalize_space(
        gpa_match.group(
            "gpa"
        )
    )


def format_year_month(
    year,
    month,
):
    """把 year/month 生成 YYYY 或 YYYY-MM。"""

    # 年份为空时返回 None。
    if year is None:
        return None

    # 没有月份时只保存年份。
    if month is None:
        return str(
            int(
                year
            )
        )

    # 有月份时保存 YYYY-MM。
    return (
        f"{int(year):04d}-"
        f"{int(month):02d}"
    )


def extract_dates_and_current(
    entry_text,
    entry_lines=None,
):
    """提取开始/结束年月，并判断是否在读或 Expected。"""

    # 清理文本。
    cleaned_entry_text = clean_text_value(
        entry_text
    )

    # 空 entry。
    if cleaned_entry_text is None:
        return None, None, 0

    # 判断 current/expected/present。
    is_current = int(
        re.search(
            r"\b(?:present|current|expected|candidate|pursuing|ongoing)\b",
            cleaned_entry_text,
            flags=re.IGNORECASE,
        )
        is not None
    )

    # raw resume 有逐行结构时，只在学校/学位附近查找日期。
    #
    # 遇到 GPA、Honors、Awards、Coursework、Project 等描述行后停止，
    # 避免把荣誉年份、课程年份或项目年份当成教育起止日期。
    if entry_lines is not None:
        date_scope_lines = []

        # 最多检查前六行。
        for entry_line in entry_lines[:6]:
            # 清理当前行。
            cleaned_line = normalize_space(
                entry_line
            )

            # 空行跳过。
            if cleaned_line is None:
                continue

            # 描述性内容开始后停止收集日期。
            if re.match(
                r"^(?:GPA|Honors?|Awards?|Relevant Coursework|"
                r"Coursework|Courses?|Projects?|Activities)\b",
                cleaned_line,
                flags=re.IGNORECASE,
            ):
                break

            # 保存学校/学位附近的行。
            date_scope_lines.append(
                cleaned_line
            )

        # 合并日期范围。
        date_search_text = "\n".join(
            date_scope_lines
        )
    else:
        # summary fallback 使用原始传入内容。
        date_search_text = cleaned_entry_text

    # Expected/Class of/Graduation 后只有一个年份时，
    # 这个年份只能表示预计结束年份，不能作为开始年份。
    expected_date_match = re.search(
        r"\b(?:expected(?:\s+(?:graduation|year of graduation))?|"
        r"class of|graduation(?:\s+date)?)\s*[:：]?\s*"
        r"(?:(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)"
        r"[A-Za-z]*\.?\s+)?"
        r"(?P<year>(?:19|20)\d{2})\b",
        date_search_text,
        flags=re.IGNORECASE,
    )

    # Month YYYY pattern。
    #
    # 允许 Sept. 2019、Dec.2020 和 May 2025。
    month_name_pattern = re.compile(
        r"\b("
        + "|".join(
            sorted(
                MONTH_NAME_TO_NUMBER,
                key=len,
                reverse=True,
            )
        )
        + r")\.?\s*((?:19|20)\d{2})\b",
        re.IGNORECASE,
    )

    # 预先保存 Month YYYY matches，
    # 后面避免再把其中的年份单独提取一次。
    month_name_matches = list(
        month_name_pattern.finditer(
            date_search_text
        )
    )

    # 保存所有 year-month matches。
    date_values = []

    # 先保存全部数字 month/year matches，
    # 后面用来避免把其中的 year 再提取一次。
    numeric_month_year_matches = list(
        MONTH_YEAR_PATTERN.finditer(
            date_search_text
        )
    )

    # 处理数字年月。
    for date_match in YEAR_MONTH_PATTERN.finditer(
        date_search_text
    ):
        # 如果当前 year 位于 08/2019 或 Sept. 2019 这类 match 内，
        # 就交给对应的完整日期 pattern 处理，避免重复。
        if any(
            month_year_match.start()
            <= date_match.start()
            < month_year_match.end()
            for month_year_match
            in numeric_month_year_matches
        ) or any(
            month_name_match.start()
            <= date_match.start()
            < month_name_match.end()
            for month_name_match
            in month_name_matches
        ):
            continue

        # 读取 year。
        year = int(
            date_match.group(
                "year"
            )
        )

        # 读取可选 month。
        month_text = date_match.group(
            "month"
        )

        # 转 month。
        month = (
            int(
                month_text
            )
            if month_text
            else None
        )

        # 保存。
        date_values.append(
            (
                date_match.start(),
                year,
                month,
            )
        )

    # 处理数字 month/year。
    for date_match in numeric_month_year_matches:
        # 保存文本位置、年份和月份。
        date_values.append(
            (
                date_match.start(),
                int(
                    date_match.group(
                        "year"
                    )
                ),
                int(
                    date_match.group(
                        "month"
                    )
                ),
            )
        )

    # 逐个保存 Month YYYY。
    for date_match in month_name_matches:
        # 月份。
        month = MONTH_NAME_TO_NUMBER[
            date_match.group(
                1
            ).lower()
        ]

        # 年份。
        year = int(
            date_match.group(
                2
            )
        )

        # 保存。
        date_values.append(
            (
                date_match.start(),
                year,
                month,
            )
        )

    # 按文本位置排序并去重。
    unique_date_values = []

    # 逐个去重。
    for date_value in sorted(
        date_values,
        key=lambda item: item[0],
    ):
        # year/month 组合。
        year_month = (
            date_value[1],
            date_value[2],
        )

        # 避免同一日期被两个 pattern 重复提取。
        if year_month not in [
            (
                existing_value[1],
                existing_value[2],
            )
            for existing_value
            in unique_date_values
        ]:
            unique_date_values.append(
                date_value
            )

    # 没有日期。
    if not unique_date_values:
        return None, None, is_current

    # Expected/Class of/Graduation 的年份优先作为唯一结束日期。
    #
    # 如果同一小段中还有明确日期范围，后面的通用逻辑仍会处理范围；
    # 只有没有可靠范围时才使用 expected year。
    has_explicit_date_range = bool(
        re.search(
            r"(?:19|20)\d{2}"
            r"(?:[./\-\s]+(?:1[0-2]|0?[1-9]))?"
            r"\s*(?:-|–|—|to)\s*"
            r"(?:present|current|(?:19|20)\d{2})",
            date_search_text,
            flags=re.IGNORECASE,
        )
        or re.search(
            r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|"
            r"Oct|Nov|Dec)[A-Za-z]*\.?\s+(?:19|20)\d{2}"
            r"\s*(?:-|–|—|to)\s*"
            r"(?:present|current|"
            r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|"
            r"Oct|Nov|Dec)[A-Za-z]*\.?\s+(?:19|20)\d{2})",
            date_search_text,
            flags=re.IGNORECASE,
        )
    )

    # 没有明确范围但有 Expected year 时只返回结束年份。
    if (
        expected_date_match is not None
        and not has_explicit_date_range
    ):
        return (
            None,
            format_year_month(
                int(
                    expected_date_match.group(
                        "year"
                    )
                ),
                None,
            ),
            is_current,
        )

    # 一个日期通常代表结束/Expected 时间。
    if len(
        unique_date_values
    ) == 1:
        only_date = unique_date_values[0]
        return (
            None,
            format_year_month(
                only_date[1],
                only_date[2],
            ),
            is_current,
        )

    # 两个以上日期：第一个作为开始，最后一个作为结束。
    first_date = unique_date_values[0]
    last_date = unique_date_values[-1]

    # 生成结果。
    start_date = format_year_month(
        first_date[1],
        first_date[2],
    )

    # 生成结束日期。
    end_date = format_year_month(
        last_date[1],
        last_date[2],
    )

    # 防止输出开始日期晚于结束日期。
    #
    # 这种情况说明文本中混入了非教育日期；
    # 保留第一个最靠近学校/学位的日期作为结束日期，
    # 不猜测开始日期。
    if (
        end_date_sort_key(
            start_date
        )
        > end_date_sort_key(
            end_date
        )
    ):
        return (
            None,
            start_date,
            is_current,
        )

    # 返回日期。
    return (
        start_date,
        end_date,
        is_current,
    )


def end_date_sort_key(
    end_date,
):
    """把 YYYY/ YYYY-MM 转成最高学历 tie-breaker 数字。"""

    # 空日期最低。
    cleaned_end_date = clean_text_value(
        end_date
    )

    # 空值。
    if cleaned_end_date is None:
        return 0

    # 读取 year/month。
    date_match = re.fullmatch(
        r"(?P<year>\d{4})(?:-(?P<month>\d{2}))?",
        cleaned_end_date,
    )

    # 非标准格式最低。
    if date_match is None:
        return 0

    # 转成 YYYYMM。
    return (
        int(
            date_match.group(
                "year"
            )
        )
        * 100
        + int(
            date_match.group(
                "month"
            )
            or 0
        )
    )


# ============================================================
# raw resume entry 解析
# ============================================================

def build_entry_from_raw_lines(
    entry_lines,
    extraction_order,
):
    """把一个 institution-anchored entry 转成结构化 record。"""

    # 保存完整原始 entry。
    raw_education_text = "\n".join(
        entry_lines
    ).strip()

    # 提取学校。
    (
        raw_school_name,
        normalized_school_name,
    ) = extract_school_from_entry_lines(
        entry_lines
    )

    # 学校是 Education.school_id 的必要来源。
    if normalized_school_name is None:
        return None

    # 提取学位及层级。
    (
        raw_degree_name,
        degree_category,
        degree_level_rank,
    ) = extract_degree(
        raw_education_text,
        entry_lines=entry_lines,
    )

    # 提取 field。
    (
        raw_field_study_name,
        normalized_field_study_name,
    ) = extract_field_of_study(
        raw_education_text,
        raw_degree_name,
    )

    # 提取 GPA。
    gpa = extract_gpa(
        raw_education_text
    )

    # 提取日期和 current。
    (
        education_start_date,
        education_end_date,
        is_current,
    ) = extract_dates_and_current(
        raw_education_text,
        entry_lines=entry_lines,
    )

    # 无明确学位、field 或 GPA 的 Other entry，
    # 如果正文明确描述 research/project/club/award 等活动，
    # 就不是一段学历记录。
    #
    # 这条规则只删除明显活动记录；
    # 单纯的 School + date 仍会保留，避免漏掉未写学位名称的教育经历。
    if (
        degree_category == "Other"
        and normalized_field_study_name is None
        and gpa is None
        and re.search(
            r"\b(?:research|project|club|award|scholarship|conference|"
            r"assistant|intern|skills?|coursework|course|competition|"
            r"volunteer|website)\b",
            raw_education_text,
            flags=re.IGNORECASE,
        )
    ):
        return None

    # 计算确定性 confidence。
    confidence_score = 0.55

    # 有明确学位时加分。
    if raw_degree_name is not None:
        confidence_score += 0.15

    # 有日期时加分。
    if (
        education_start_date is not None
        or education_end_date is not None
    ):
        confidence_score += 0.15

    # 有 field 时加分。
    if normalized_field_study_name is not None:
        confidence_score += 0.10

    # 最大1.0。
    confidence_score = min(
        confidence_score,
        1.0,
    )

    # 返回 entry。
    return {
        "extraction_source": "raw_resume_rule",
        "education_entry_order": int(
            extraction_order
        ),
        "raw_education_text": (
            raw_education_text
        ),
        "raw_school_name": raw_school_name,
        "normalized_school_name": (
            normalized_school_name
        ),
        "raw_degree_name": raw_degree_name,
        "normalized_degree_name": (
            normalized_lower_text(
                raw_degree_name
            )
        ),
        "degree_category": degree_category,
        "degree_level_rank": int(
            degree_level_rank
        ),
        "raw_field_study_name": (
            raw_field_study_name
        ),
        "normalized_field_study_name": (
            normalized_field_study_name
        ),
        # 当前来源无法可靠区分 field 与 major。
        "raw_major_name": None,
        "normalized_major_name": None,
        "gpa": gpa,
        "education_description": (
            raw_education_text
        ),
        "education_start_date": (
            education_start_date
        ),
        "education_end_date": (
            education_end_date
        ),
        "is_current": int(
            is_current
        ),
        "extraction_confidence": round(
            confidence_score,
            2,
        ),
    }


# ============================================================
# summary fallback 解析
# ============================================================

def parse_summary_date_component(
    component,
):
    """把 summary 的 2023.0-9.0 转成 YYYY-MM。"""

    # 清理。
    cleaned_component = clean_text_value(
        component
    )

    # 空/nan。
    if cleaned_component is None:
        return None

    # 匹配 float-like year/month。
    component_match = re.fullmatch(
        r"(?P<year>(?:19|20)\d{2})(?:\.0)?"
        r"(?:-(?P<month>\d{1,2})(?:\.0)?)?",
        cleaned_component,
    )

    # 无法解析。
    if component_match is None:
        return None

    # 返回 YYYY/ YYYY-MM。
    return format_year_month(
        int(
            component_match.group(
                "year"
            )
        ),
        (
            int(
                component_match.group(
                    "month"
                )
            )
            if component_match.group(
                "month"
            )
            else None
        ),
    )


def parse_summary_entries(
    summary_text,
):
    """保守解析现有 human-readable education summary。"""

    # 清理摘要。
    cleaned_summary = clean_text_value(
        summary_text
    )

    # 空摘要。
    if cleaned_summary is None:
        return []

    # 保存 fallback entries。
    fallback_entries = []

    # 逐条拆分。
    for summary_order, summary_entry in enumerate(
        cleaned_summary.split(
            " ;; "
        ),
        start=1,
    ):
        # 每条预期4个字段。
        summary_parts = summary_entry.split(
            " | "
        )

        # 格式不正确时跳过。
        if len(
            summary_parts
        ) != 4:
            continue

        # 读取字段。
        (
            school_name,
            degree_name,
            field_name,
            date_range,
        ) = [
            clean_text_value(
                summary_part
            )
            for summary_part
            in summary_parts
        ]

        # 学校必须明确像教育机构。
        if not is_institution_line(
            school_name
        ):
            continue

        # 提取 degree。
        (
            detected_degree_name,
            degree_category,
            degree_level_rank,
        ) = extract_degree(
            degree_name
        )

        # 优先保留摘要 degree 字段。
        raw_degree_name = (
            degree_name
            or detected_degree_name
        )

        # 解析 date range。
        start_date = None
        end_date = None

        # 按 to 拆分。
        if date_range is not None:
            date_parts = re.split(
                r"\s+to\s+",
                date_range,
                maxsplit=1,
                flags=re.IGNORECASE,
            )

            # 解析 start。
            if date_parts:
                start_date = (
                    parse_summary_date_component(
                        date_parts[0].replace(
                            "nan-nan",
                            "",
                        ).strip()
                    )
                )

            # 解析 end。
            if len(
                date_parts
            ) > 1:
                end_date = (
                    parse_summary_date_component(
                        date_parts[1].replace(
                            "nan-nan",
                            "",
                        ).strip()
                    )
                )

        # fallback 只保存摘要真实提供的 normalized 信息，
        # 不把 normalized value 冒充成 raw value。
        fallback_entries.append(
            {
                "extraction_source": (
                    "education_summary_fallback"
                ),
                "education_entry_order": int(
                    summary_order
                ),
                "raw_education_text": None,
                "raw_school_name": None,
                "normalized_school_name": (
                    normalized_lower_text(
                        school_name
                    )
                ),
                "raw_degree_name": None,
                "normalized_degree_name": (
                    normalized_lower_text(
                        raw_degree_name
                    )
                ),
                "degree_category": (
                    degree_category
                ),
                "degree_level_rank": int(
                    degree_level_rank
                ),
                "raw_field_study_name": None,
                "normalized_field_study_name": (
                    normalized_lower_text(
                        field_name
                    )
                ),
                "raw_major_name": None,
                "normalized_major_name": None,
                "gpa": None,
                "education_description": None,
                "education_start_date": (
                    start_date
                ),
                "education_end_date": (
                    end_date
                ),
                "is_current": int(
                    re.search(
                        r"\b(?:present|expected|current)\b",
                        summary_entry,
                        flags=re.IGNORECASE,
                    )
                    is not None
                ),
                "extraction_confidence": 0.55,
            }
        )

    # 返回 fallback。
    return fallback_entries


# ============================================================
# 读取 application.csv 与 extraction summary
# ============================================================

# 读取 application.csv。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 必要 columns。
required_application_columns = [
    "id",
    "person_id",
    "raw_submission_id",
    "resume_text",
]

# person_id 在 application.csv 中可能为空，因此真正 person mapping 来自 D1。
missing_application_columns = [
    column_name
    for column_name
    in required_application_columns
    if column_name
    not in application_dataframe.columns
]

# 缺失时停止。
if missing_application_columns:
    raise ValueError(
        "application.csv 缺少必要字段："
        f"{missing_application_columns}"
    )

# 只处理有 Application.id 的 rows。
application_dataframe = (
    application_dataframe[
        application_dataframe["id"].notna()
    ]
    .copy()
)

# 转换 ids。
application_dataframe["id"] = (
    pd.to_numeric(
        application_dataframe["id"],
        errors="raise",
    ).astype(
        "int64"
    )
)

# 转换 raw_submission_id。
application_dataframe[
    "raw_submission_id"
] = pd.to_numeric(
    application_dataframe[
        "raw_submission_id"
    ],
    errors="raise",
).astype(
    "int64"
)

# 验证 Application.id 唯一。
if application_dataframe[
    "id"
].duplicated().any():
    raise ValueError(
        "application.csv 存在重复 Application.id。"
    )

# 读取 extraction summary。
resume_extraction_dataframe = pd.read_csv(
    RESUME_EXTRACTION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 验证 summary columns。
if (
    "raw_submission_id"
    not in resume_extraction_dataframe.columns
    or "resume_educations_summary"
    not in resume_extraction_dataframe.columns
):
    raise ValueError(
        "resume_extraction_consolidated.csv "
        "缺少 raw_submission_id 或 resume_educations_summary。"
    )

# 转换 summary raw id。
resume_extraction_dataframe[
    "raw_submission_id"
] = pd.to_numeric(
    resume_extraction_dataframe[
        "raw_submission_id"
    ],
    errors="raise",
).astype(
    "int64"
)

# raw_submission_id 应唯一。
if resume_extraction_dataframe[
    "raw_submission_id"
].duplicated().any():
    raise ValueError(
        "resume_extraction_consolidated.csv "
        "存在重复 raw_submission_id。"
    )

# 建立 summary mapping。
summary_by_raw_submission_id = {
    int(
        summary_row[
            "raw_submission_id"
        ]
    ): summary_row[
        "resume_educations_summary"
    ]
    for _, summary_row
    in resume_extraction_dataframe.iterrows()
}


# ============================================================
# 从 D1 取得真实 Application/Person/Candidate mapping
# ============================================================

# 查询轻量 mapping。
mapping_result = execute_d1(
    """
    SELECT
      app."id" AS application_id,
      app."person_id",
      cs."id" AS candidate_snapshot_id
    FROM "Application" AS app
    INNER JOIN "Candidate_snapshot" AS cs
      ON cs."application_id" = app."id"
    ORDER BY app."id";
    """
)

# 提取 rows。
mapping_rows = (
    mapping_result[0].get(
        "results",
        [],
    )
    if mapping_result
    else []
)

# 建立 mapping。
d1_mapping_by_application_id = {
    int(
        mapping_row[
            "application_id"
        ]
    ): {
        "person_id": int(
            mapping_row[
                "person_id"
            ]
        ),
        "candidate_snapshot_id": int(
            mapping_row[
                "candidate_snapshot_id"
            ]
        ),
    }
    for mapping_row
    in mapping_rows
}

# 生成当前增量 application.csv 中的 Application ID 集合。
csv_application_ids = {
    int(
        application_id
    )
    for application_id
    in application_dataframe[
        "id"
    ].tolist()
}

# 生成 D1 中全部已有 Application/Candidate mapping 的 ID 集合。
d1_application_ids = set(
    d1_mapping_by_application_id
)

# 当前 application.csv 可以只包含本次新增批次，
# 因此只要求 CSV 中的每个 Application.id 都能在 D1 中找到。
# D1 中存在但当前 CSV 中不存在的 ID 属于历史批次，是正常情况。
application_ids_missing_from_d1 = sorted(
    csv_application_ids
    - d1_application_ids
)

# 当前 CSV 中存在无法映射的 Application.id 时停止。
if application_ids_missing_from_d1:
    raise ValueError(
        "当前 application.csv 中存在无法映射到 "
        "D1 Application/Candidate_snapshot 的记录："
        f"{application_ids_missing_from_d1[:20]}"
    )

# 显示当前增量批次和历史批次数量，供人工核对。
print(
    "Education extraction Application mapping validation:",
    {
        "current_csv_application_count": int(
            len(csv_application_ids)
        ),
        "d1_application_count": int(
            len(d1_application_ids)
        ),
        "historical_application_not_in_current_csv_count": int(
            len(
                d1_application_ids
                - csv_application_ids
            )
        ),
        "current_csv_ids_missing_from_d1_count": int(
            len(application_ids_missing_from_d1)
        ),
    },
)


# ============================================================
# 对每份 Candidate 执行确定性教育提取
# ============================================================

# 保存结构化教育 records。
education_records = []

# 保存 Candidate extraction status。
education_status_records = []

# 按 Application.id 稳定处理。
application_dataframe = (
    application_dataframe.sort_values(
        "id"
    ).reset_index(
        drop=True
    )
)

# 逐条处理。
for dataframe_index, application_row in (
    application_dataframe.iterrows()
):
    # 读取 ids。
    application_id = int(
        application_row[
            "id"
        ]
    )

    # 读取 D1 mapping。
    d1_mapping = (
        d1_mapping_by_application_id[
            application_id
        ]
    )

    # 读取 raw id。
    raw_submission_id = int(
        application_row[
            "raw_submission_id"
        ]
    )

    # 读取 resume。
    resume_text = clean_text_value(
        application_row[
            "resume_text"
        ]
    )

    # 提取 Education section。
    (
        education_section_lines,
        section_status,
    ) = extract_education_section_lines(
        resume_text
    )

    # 按 institution anchor 拆分。
    raw_entry_lines_list = (
        split_education_entries(
            education_section_lines
        )
    )

    # 保存当前 Candidate entries。
    candidate_entries = []

    # 逐条解析 raw entries。
    for entry_order, entry_lines in enumerate(
        raw_entry_lines_list,
        start=1,
    ):
        # 解析。
        structured_entry = (
            build_entry_from_raw_lines(
                entry_lines,
                entry_order,
            )
        )

        # 只保存带学校的有效 entry。
        if structured_entry is not None:
            candidate_entries.append(
                structured_entry
            )

    # raw parser 无有效结果时使用 summary fallback。
    used_fallback = False

    # fallback。
    if not candidate_entries:
        # 读取 summary。
        summary_text = (
            summary_by_raw_submission_id.get(
                raw_submission_id
            )
        )

        # 解析 fallback。
        candidate_entries = (
            parse_summary_entries(
                summary_text
            )
        )

        # 标记 fallback。
        used_fallback = bool(
            candidate_entries
        )

    # 判断最高学历 entry。
    highest_entry_order = None

    # 有 entries 时排序。
    if candidate_entries:
        # max key：rank最高、结束日期最新、原顺序最前。
        highest_entry = max(
            candidate_entries,
            key=lambda entry: (
                int(
                    entry[
                        "degree_level_rank"
                    ]
                ),
                end_date_sort_key(
                    entry[
                        "education_end_date"
                    ]
                ),
                -int(
                    entry[
                        "education_entry_order"
                    ]
                ),
            ),
        )

        # 保存最高 entry order。
        highest_entry_order = int(
            highest_entry[
                "education_entry_order"
            ]
        )

    # 保存每条结构化 entry。
    for structured_entry in candidate_entries:
        # 合并固定标识字段。
        output_record = {
            "rule_version": RULE_VERSION,
            "application_id": application_id,
            "person_id": int(
                d1_mapping[
                    "person_id"
                ]
            ),
            "candidate_snapshot_id": int(
                d1_mapping[
                    "candidate_snapshot_id"
                ]
            ),
            "raw_submission_id": (
                raw_submission_id
            ),
            **structured_entry,
            "is_highest_degree": int(
                int(
                    structured_entry[
                        "education_entry_order"
                    ]
                )
                == highest_entry_order
            ),
        }

        # 保存。
        education_records.append(
            output_record
        )

    # 保存 Candidate status。
    education_status_records.append(
        {
            "rule_version": RULE_VERSION,
            "application_id": application_id,
            "person_id": int(
                d1_mapping[
                    "person_id"
                ]
            ),
            "candidate_snapshot_id": int(
                d1_mapping[
                    "candidate_snapshot_id"
                ]
            ),
            "raw_submission_id": (
                raw_submission_id
            ),
            "resume_text_present": int(
                resume_text is not None
            ),
            "education_section_status": (
                section_status
            ),
            "used_summary_fallback": int(
                used_fallback
            ),
            "education_record_count": int(
                len(
                    candidate_entries
                )
            ),
            "highest_degree_category": (
                highest_entry[
                    "degree_category"
                ]
                if candidate_entries
                else None
            ),
            "highest_degree_level_rank": (
                int(
                    highest_entry[
                        "degree_level_rank"
                    ]
                )
                if candidate_entries
                else None
            ),
            "extraction_status": (
                "extracted"
                if candidate_entries
                else "no_reliable_education_found"
            ),
        }
    )

    # 每100条显示进度。
    processed_count = (
        dataframe_index + 1
    )

    # 同时显示最后一条。
    if (
        processed_count % 100 == 0
        or processed_count
        == len(
            application_dataframe
        )
    ):
        print(
            "Processed Candidate education extraction:",
            f"{processed_count}/"
            f"{len(application_dataframe)}",
        )


# ============================================================
# 建立、验证并覆盖写入 CSV
# ============================================================

# 定义 education columns，保证空结果时也有稳定 schema。
education_output_columns = [
    "rule_version",
    "application_id",
    "person_id",
    "candidate_snapshot_id",
    "raw_submission_id",
    "extraction_source",
    "education_entry_order",
    "raw_education_text",
    "raw_school_name",
    "normalized_school_name",
    "raw_degree_name",
    "normalized_degree_name",
    "degree_category",
    "degree_level_rank",
    "raw_field_study_name",
    "normalized_field_study_name",
    "raw_major_name",
    "normalized_major_name",
    "gpa",
    "education_description",
    "education_start_date",
    "education_end_date",
    "is_current",
    "extraction_confidence",
    "is_highest_degree",
]

# 建立 Education DataFrame。
education_output_dataframe = pd.DataFrame(
    education_records,
    columns=education_output_columns,
)

# 建立 status DataFrame。
education_status_dataframe = pd.DataFrame(
    education_status_records
)

# 每个 Candidate 最多只能有一个 highest entry。
if not education_output_dataframe.empty:
    # 按 Candidate 统计 highest 数量。
    highest_count_by_candidate = (
        education_output_dataframe.groupby(
            "candidate_snapshot_id"
        )[
            "is_highest_degree"
        ].sum()
    )

    # 每个拥有教育记录的 Candidate 必须恰好一个 highest。
    invalid_highest_counts = (
        highest_count_by_candidate[
            highest_count_by_candidate
            != 1
        ]
    )

    # 发现错误时停止。
    if not invalid_highest_counts.empty:
        raise RuntimeError(
            "Candidate highest education 标记数量错误："
            f"{invalid_highest_counts.head(20).to_dict()}"
        )

# 使用固定排序保证文件内容稳定。
education_output_dataframe = (
    education_output_dataframe.sort_values(
        [
            "application_id",
            "education_entry_order",
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# status 也稳定排序。
education_status_dataframe = (
    education_status_dataframe.sort_values(
        "application_id",
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# 覆盖写入结构化教育 CSV。
education_output_dataframe.to_csv(
    EDUCATION_OUTPUT_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

# 覆盖写入 status CSV。
education_status_dataframe.to_csv(
    EDUCATION_STATUS_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

# 显示 summary。
print(
    "Candidate education extraction summary:",
    {
        "candidate_count": int(
            len(
                education_status_dataframe
            )
        ),
        "education_record_count": int(
            len(
                education_output_dataframe
            )
        ),
        "candidates_with_education": int(
            (
                education_status_dataframe[
                    "education_record_count"
                ]
                > 0
            ).sum()
        ),
        "candidates_without_reliable_education": int(
            (
                education_status_dataframe[
                    "education_record_count"
                ]
                == 0
            ).sum()
        ),
        "summary_fallback_candidate_count": int(
            education_status_dataframe[
                "used_summary_fallback"
            ].sum()
        ),
        "doctorate_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "Doctorate"
            ).sum()
        ),
        "master_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "Master"
            ).sum()
        ),
        "bachelor_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "Bachelor"
            ).sum()
        ),
        "associate_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "Associate"
            ).sum()
        ),
        "high_school_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "High School"
            ).sum()
        ),
        "other_highest_count": int(
            (
                education_status_dataframe[
                    "highest_degree_category"
                ]
                == "Other"
            ).sum()
        ),
    },
)

# 显示输出路径。
print(
    "Education records CSV:",
    EDUCATION_OUTPUT_CSV_PATH,
)

# 显示 status 路径。
print(
    "Education status CSV:",
    EDUCATION_STATUS_CSV_PATH,
)

# 显示预览。
display(
    education_output_dataframe.head(
        20
    )
)

# 显示没有可靠教育经历的 Candidate 预览。
display(
    education_status_dataframe[
        education_status_dataframe[
            "education_record_count"
        ]
        == 0
    ].head(
        20
    )
)

# 显示完成。
print(
    "Candidate education rule extraction completed successfully."
)


### 全库最高学历审计与修复

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 一次性审计、修复并验证 D1 中全部 Candidate / Person 的最高学历。
#
# 本 Cell 修复两个层级：
# 1. Candidate_Education.is_highest_degree
#    每个已经拥有 Candidate_Education 的 Candidate 必须且只能有一条 1。
# 2. Person.highest_education_id
#    每个已经拥有 Person_Education 的 Person 必须指向其唯一最高学历。
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell、CSV、variable、function 或 class。
# 2. 本 Cell 不 INSERT、不 DELETE Education 相关记录。
# 3. 本 Cell 只修复最高学历标记或外键值不正确的 Candidate / Person。
# 4. 已经正确的记录不会执行 UPDATE。
# 5. 相同数据库状态重复运行时不会继续修改数据。
# 6. 网络中断后可以安全重新运行。
# 7. 建议现在运行一次；未来作为定期全库审计按需运行，
#    不需要在每个增量批次后都运行。
# ============================================================

# time 只用于 D1 临时请求失败后的重试等待。
import time

# datetime 用于更新 Person.updated_at。
from datetime import datetime, timezone

# pandas 用于生成修复结果预览。
import pandas as pd

# requests 用于调用 Cloudflare D1 API。
import requests

# userdata 用于读取 Google Colab Secret。
from google.colab import userdata


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 query endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用同一个 HTTP session。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在 max_retries 次尝试中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 SQL 和标准 Python 参数。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(1.0)
                continue

            # 全部网络请求失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            # 非 JSON response 无法可靠继续。
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 D1 query 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get("success", False)
        ):
            return payload.get("result", [])

        # 保存 SQL 或 API 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(1.0)

    # 全部重试失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(result):
    """从 D1 result 中稳定取得第一条 statement 的 rows。"""

    # 空 result 返回空 list。
    if not result:
        return []

    # 返回 results rows。
    return result[0].get(
        "results",
        [],
    )


def utc_now_text():
    """生成不包含微秒的 UTC timestamp。"""

    # 返回与数据库现有时间格式一致的文本。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


# ============================================================
# 验证必要 schema
# ============================================================

# 定义本 Cell 使用的必要字段。
required_schema_columns = {
    "Candidate_Education": {
        "id",
        "candidate_id",
        "person_education_id",
        "is_highest_degree",
    },
    "Person_Education": {
        "id",
        "person_id",
        "education_id",
    },
    "Education": {
        "id",
        "degree_id",
        "education_end_date",
    },
    "Degree": {
        "id",
        "degree_level_rank",
    },
    "Person": {
        "id",
        "highest_education_id",
        "updated_at",
    },
}

# 逐表检查必要字段。
for table_name, required_columns in (
    required_schema_columns.items()
):
    # 读取当前表结构。
    schema_result = execute_d1(
        f'PRAGMA table_info("{table_name}");'
    )

    # 生成现有字段集合。
    existing_columns = {
        str(row["name"])
        for row in result_rows(schema_result)
    }

    # 找出缺少字段。
    missing_columns = sorted(
        required_columns
        - existing_columns
    )

    # 缺少字段时停止，不执行任何修复。
    if missing_columns:
        raise ValueError(
            f"D1 {table_name} 缺少必要字段："
            f"{missing_columns}"
        )


# ============================================================
# 读取并规划全部 Candidate 最高学历
# ============================================================

# SQL 已按确定性规则排序：
# 1. 学位等级高者优先；
# 2. 结束日期较晚者优先；
# 3. Person_Education.id 较小者作为稳定 tie-breaker；
# 4. Candidate_Education.id 较小者作为最后 tie-breaker。
candidate_rows_result = execute_d1(
    """
    SELECT
      ce."id" AS candidate_education_id,
      ce."candidate_id",
      ce."person_education_id",
      ce."is_highest_degree",
      COALESCE(d."degree_level_rank", 0)
        AS degree_level_rank,
      COALESCE(e."education_end_date", '')
        AS education_end_date
    FROM "Candidate_Education" AS ce
    INNER JOIN "Person_Education" AS pe
      ON pe."id" = ce."person_education_id"
    INNER JOIN "Education" AS e
      ON e."id" = pe."education_id"
    LEFT JOIN "Degree" AS d
      ON d."id" = e."degree_id"
    ORDER BY
      ce."candidate_id" ASC,
      COALESCE(d."degree_level_rank", 0) DESC,
      COALESCE(e."education_end_date", '') DESC,
      pe."id" ASC,
      ce."id" ASC;
    """
)

# 提取全部 Candidate education links。
candidate_rows = result_rows(
    candidate_rows_result
)

# 按 Candidate.id 保存已经排序的 links。
candidate_links_by_id = {}

# 逐条建立 mapping。
for candidate_row in candidate_rows:
    # 读取 Candidate.id。
    candidate_id = int(
        candidate_row["candidate_id"]
    )

    # 保存到当前 Candidate 的有序 list。
    candidate_links_by_id.setdefault(
        candidate_id,
        [],
    ).append(
        candidate_row
    )

# 保存 Candidate 修复计划。
candidate_repair_plans = []

# 按 Candidate.id 稳定处理。
for candidate_id in sorted(
    candidate_links_by_id
):
    # 当前 links 已由 SQL 按最高学历规则排序。
    candidate_links = (
        candidate_links_by_id[candidate_id]
    )

    # 第一条就是唯一应保留 highest=1 的 link。
    selected_link_id = int(
        candidate_links[0][
            "candidate_education_id"
        ]
    )

    # 统计当前 highest=1 数量。
    current_highest_count = sum(
        int(link.get("is_highest_degree") or 0)
        == 1
        for link in candidate_links
    )

    # 判断当前被标记的 link 是否正是规则选中的 link。
    selected_link_is_current_highest = (
        int(
            candidate_links[0].get(
                "is_highest_degree"
            )
            or 0
        )
        == 1
    )

    # 只有数量不是1或选中的不是正确记录时才需要 UPDATE。
    requires_update = (
        current_highest_count != 1
        or not selected_link_is_current_highest
    )

    # 保存计划和审计字段。
    candidate_repair_plans.append(
        {
            "candidate_id": candidate_id,
            "candidate_education_count": int(
                len(candidate_links)
            ),
            "current_highest_count": int(
                current_highest_count
            ),
            "selected_candidate_education_id": (
                selected_link_id
            ),
            "selected_degree_level_rank": int(
                candidate_links[0].get(
                    "degree_level_rank"
                )
                or 0
            ),
            "selected_education_end_date": (
                candidate_links[0].get(
                    "education_end_date"
                )
            ),
            "requires_update": bool(
                requires_update
            ),
        }
    )

# 建立 Candidate 计划 DataFrame。
candidate_repair_dataframe = pd.DataFrame(
    candidate_repair_plans
)

# 取得需要修复的 Candidate。
candidate_updates = [
    plan
    for plan in candidate_repair_plans
    if plan["requires_update"]
]

# 显示写入前计划。
print(
    "Candidate highest education audit summary:",
    {
        "candidate_with_education_count": int(
            len(candidate_repair_plans)
        ),
        "candidate_requiring_repair_count": int(
            len(candidate_updates)
        ),
        "candidate_already_valid_count": int(
            len(candidate_repair_plans)
            - len(candidate_updates)
        ),
    },
)

# 只显示需要修复的 Candidate。
if not candidate_repair_dataframe.empty:
    display(
        candidate_repair_dataframe.loc[
            candidate_repair_dataframe[
                "requires_update"
            ]
        ].head(50)
    )


# ============================================================
# 幂等修复 Candidate_Education highest 标记
# ============================================================

# 逐个修复异常 Candidate。
for repair_number, repair_plan in enumerate(
    candidate_updates,
    start=1,
):
    # 一条 UPDATE 同时把唯一选中记录设为1、其余记录设为0。
    execute_d1(
        """
        UPDATE "Candidate_Education"
        SET "is_highest_degree" =
          CASE
            WHEN "id" = ? THEN 1
            ELSE 0
          END
        WHERE "candidate_id" = ?;
        """,
        [
            int(
                repair_plan[
                    "selected_candidate_education_id"
                ]
            ),
            int(
                repair_plan["candidate_id"]
            ),
        ],
    )

    # 每100个显示一次进度，同时显示最后一个。
    if (
        repair_number % 100 == 0
        or repair_number
        == len(candidate_updates)
    ):
        print(
            "Repaired Candidate highest education:",
            f"{repair_number}/{len(candidate_updates)}",
        )


# ============================================================
# 读取并规划全部 Person 最高学历
# ============================================================

# 读取每个 Person 的全部 Person_Education，排序规则与 Candidate 一致。
person_rows_result = execute_d1(
    """
    SELECT
      p."id" AS person_id,
      p."highest_education_id"
        AS current_highest_education_id,
      pe."id" AS person_education_id,
      COALESCE(d."degree_level_rank", 0)
        AS degree_level_rank,
      COALESCE(e."education_end_date", '')
        AS education_end_date
    FROM "Person" AS p
    INNER JOIN "Person_Education" AS pe
      ON pe."person_id" = p."id"
    INNER JOIN "Education" AS e
      ON e."id" = pe."education_id"
    LEFT JOIN "Degree" AS d
      ON d."id" = e."degree_id"
    ORDER BY
      p."id" ASC,
      COALESCE(d."degree_level_rank", 0) DESC,
      COALESCE(e."education_end_date", '') DESC,
      pe."id" ASC;
    """
)

# 提取 Person education rows。
person_rows = result_rows(
    person_rows_result
)

# 按 Person.id 保存有序学历。
person_education_by_id = {}

# 建立 mapping。
for person_row in person_rows:
    # 读取 Person.id。
    person_id = int(
        person_row["person_id"]
    )

    # 保存有序 row。
    person_education_by_id.setdefault(
        person_id,
        [],
    ).append(
        person_row
    )

# 保存 Person 更新计划。
person_repair_plans = []

# 按 Person.id 稳定处理。
for person_id in sorted(
    person_education_by_id
):
    # 当前 rows 已按最高学历规则排序。
    person_education_rows = (
        person_education_by_id[person_id]
    )

    # 第一条是应写入 Person.highest_education_id 的记录。
    selected_person_education_id = int(
        person_education_rows[0][
            "person_education_id"
        ]
    )

    # 读取 Person 当前值。
    current_value = person_education_rows[0].get(
        "current_highest_education_id"
    )

    # 将非空当前值转成 int。
    current_person_education_id = (
        int(current_value)
        if current_value is not None
        else None
    )

    # 保存计划。
    person_repair_plans.append(
        {
            "person_id": person_id,
            "current_highest_education_id": (
                current_person_education_id
            ),
            "selected_highest_education_id": (
                selected_person_education_id
            ),
            "selected_degree_level_rank": int(
                person_education_rows[0].get(
                    "degree_level_rank"
                )
                or 0
            ),
            "selected_education_end_date": (
                person_education_rows[0].get(
                    "education_end_date"
                )
            ),
            "requires_update": (
                current_person_education_id
                != selected_person_education_id
            ),
        }
    )

# 取得需要更新的 Person。
person_updates = [
    plan
    for plan in person_repair_plans
    if plan["requires_update"]
]

# 显示 Person 审计汇总。
print(
    "Person highest education audit summary:",
    {
        "person_with_education_count": int(
            len(person_repair_plans)
        ),
        "person_requiring_repair_count": int(
            len(person_updates)
        ),
        "person_already_valid_count": int(
            len(person_repair_plans)
            - len(person_updates)
        ),
    },
)


# ============================================================
# 幂等修复 Person.highest_education_id
# ============================================================

# 本次所有实际 Person UPDATE 使用同一个 timestamp。
person_updated_at = utc_now_text()

# 逐个更新真正不一致的 Person。
for person_update_number, person_plan in enumerate(
    person_updates,
    start=1,
):
    # 只在值不一致时更新。
    execute_d1(
        """
        UPDATE "Person"
        SET
          "highest_education_id" = ?,
          "updated_at" = ?
        WHERE "id" = ?
          AND "highest_education_id" IS NOT ?;
        """,
        [
            int(
                person_plan[
                    "selected_highest_education_id"
                ]
            ),
            person_updated_at,
            int(person_plan["person_id"]),
            int(
                person_plan[
                    "selected_highest_education_id"
                ]
            ),
        ],
    )

    # 每100个显示一次进度，同时显示最后一个。
    if (
        person_update_number % 100 == 0
        or person_update_number
        == len(person_updates)
    ):
        print(
            "Repaired Person highest education:",
            f"{person_update_number}/{len(person_updates)}",
        )


# ============================================================
# 全库写入后验证
# ============================================================

# 验证每个有学历 link 的 Candidate 恰好一个 highest=1。
candidate_verification_result = execute_d1(
    """
    SELECT
      ce."candidate_id",
      SUM(
        CASE
          WHEN ce."is_highest_degree" = 1
          THEN 1 ELSE 0
        END
      ) AS highest_count
    FROM "Candidate_Education" AS ce
    GROUP BY ce."candidate_id"
    HAVING highest_count != 1
    ORDER BY ce."candidate_id";
    """
)

# 提取 Candidate 异常。
invalid_candidate_rows = result_rows(
    candidate_verification_result
)

# 仍有异常时停止。
if invalid_candidate_rows:
    raise RuntimeError(
        "全库 Candidate_Education 最高学历修复后仍有异常："
        f"{invalid_candidate_rows[:20]}"
    )

# 重新读取 Person 教育排序和当前 highest 值。
person_verification_result = execute_d1(
    """
    SELECT
      p."id" AS person_id,
      p."highest_education_id",
      pe."id" AS person_education_id,
      COALESCE(d."degree_level_rank", 0)
        AS degree_level_rank,
      COALESCE(e."education_end_date", '')
        AS education_end_date
    FROM "Person" AS p
    INNER JOIN "Person_Education" AS pe
      ON pe."person_id" = p."id"
    INNER JOIN "Education" AS e
      ON e."id" = pe."education_id"
    LEFT JOIN "Degree" AS d
      ON d."id" = e."degree_id"
    ORDER BY
      p."id" ASC,
      COALESCE(d."degree_level_rank", 0) DESC,
      COALESCE(e."education_end_date", '') DESC,
      pe."id" ASC;
    """
)

# 保存 Person 验证结果。
person_verification_by_id = {}

# 每个 Person 的第一条 row 是预期最高学历。
for verification_row in result_rows(
    person_verification_result
):
    # 读取 Person.id。
    person_id = int(
        verification_row["person_id"]
    )

    # 只保存排序后的第一条。
    if person_id not in person_verification_by_id:
        person_verification_by_id[person_id] = (
            verification_row
        )

# 保存 Person 异常。
invalid_person_rows = []

# 验证当前值等于规则选择值。
for person_id, verification_row in (
    person_verification_by_id.items()
):
    # 读取数据库当前 highest_education_id。
    current_highest_id = verification_row.get(
        "highest_education_id"
    )

    # 读取规则选择的 Person_Education.id。
    expected_highest_id = int(
        verification_row[
            "person_education_id"
        ]
    )

    # 不一致时保存异常。
    if (
        current_highest_id is None
        or int(current_highest_id)
        != expected_highest_id
    ):
        invalid_person_rows.append(
            {
                "person_id": int(person_id),
                "current_highest_education_id": (
                    current_highest_id
                ),
                "expected_highest_education_id": (
                    expected_highest_id
                ),
            }
        )

# 仍有 Person 异常时停止。
if invalid_person_rows:
    raise RuntimeError(
        "全库 Person.highest_education_id 修复后仍有异常："
        f"{invalid_person_rows[:20]}"
    )

# 执行全库外键检查。
foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)

# 提取外键异常。
foreign_key_rows = result_rows(
    foreign_key_result
)

# 有外键异常时停止。
if foreign_key_rows:
    raise RuntimeError(
        "全库最高学历修复后发现外键违规："
        f"{foreign_key_rows[:20]}"
    )

# 显示最终汇总。
print(
    "Global highest education repair summary:",
    {
        "candidate_with_education_count": int(
            len(candidate_repair_plans)
        ),
        "candidate_repaired_count": int(
            len(candidate_updates)
        ),
        "candidate_verified_count": int(
            len(candidate_repair_plans)
        ),
        "person_with_education_count": int(
            len(person_repair_plans)
        ),
        "person_repaired_count": int(
            len(person_updates)
        ),
        "person_verified_count": int(
            len(person_repair_plans)
        ),
        "foreign_key_violation_count": int(
            len(foreign_key_rows)
        ),
    },
)

# 显示完成信息。
print(
    "Global Candidate and Person highest education "
    "repair completed successfully."
)


### Candidate_education, Person_education, Education, Field_study, Degree, School 数据导入 (可重复运行)

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 把固定的 Candidate education CSV 结果经过准入检查后幂等导入 D1。
#
# 写入或更新：
# 1. School
# 2. Degree
# 3. Field_study
# 4. Education
# 5. Person_Education
# 6. Candidate_Education
# 7. Person.highest_education_id
#
# 重要：
# 1. 本 Cell 不依赖任何其他 Colab Cell 中的变量、function 或 class。
# 2. 本 Cell 不使用 rule_version 决定是否允许导入。
# 3. 本 Cell 直接使用 CSV.degree_category 作为标准化学位名称。
# 4. 本 Cell 不按学校、学位或日期删除任何学历记录。
# 5. 相同输入可以重复运行；已有记录会被复用，不会重复 INSERT。
# 6. 只在通过准入的学历中重新计算 Candidate 的最高学历。
# 7. Person.highest_education_id 在所有已导入 Person_Education 中重新计算。
# 8. Major 暂时没有可靠来源，因此 major_id 和 raw_major_name 保持 NULL。
# 9. 本 Cell 直接覆盖更新原来的 extraction CSV 和 status CSV。
# 10. 数据库状态只使用 success、rejected 和 failed。
# ============================================================

# os 用于检查 Google Drive 输入文件是否存在。
import os

# time 只用于 D1 网络请求失败后的等待重试。
import time

# datetime 用于生成统一 UTC timestamp。
from datetime import datetime, timezone

# pandas 用于读取学历 CSV、执行准入检查和保存结果 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定 Google Drive 路径
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 结构化学历 CSV。
EDUCATION_INPUT_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_education_rule_extraction.csv"
)

# Candidate 级学历提取状态 CSV。
EDUCATION_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_education_rule_extraction_status.csv"
)

# 输入不存在时停止。
if not os.path.exists(
    EDUCATION_INPUT_CSV_PATH
):
    raise FileNotFoundError(
        "找不到学历结果 CSV："
        f"{EDUCATION_INPUT_CSV_PATH}"
    )

# status CSV 不存在时停止。
if not os.path.exists(
    EDUCATION_STATUS_CSV_PATH
):
    raise FileNotFoundError(
        "找不到 Candidate education status CSV："
        f"{EDUCATION_STATUS_CSV_PATH}"
    )


# ============================================================
# Cloudflare D1 连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session。
HTTP_SESSION = requests.Session()

# 缓存已经查询或新建的维度 id。
#
# key 包含 table、name column 和标准化名称，
# 避免同一个学校、学位或 field 在循环中重复请求 D1。
NAMED_DIMENSION_ID_CACHE = {}


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时失败有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在重试次数中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # 请求和 SQL 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败时停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    result,
):
    """从 D1 result 中稳定取得 results rows。"""

    # 空 result 返回空 list。
    if not result:
        return []

    # 返回第一条 statement 的 rows。
    return result[0].get(
        "results",
        [],
    )


def utc_now_text():
    """生成不包含微秒的 UTC ISO timestamp。"""

    # 返回统一格式。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_text_value(
    value,
):
    """把 pandas 空值转成 None，其余值转成清理后的 string。"""

    # Python None 保持 None。
    if value is None:
        return None

    # pandas 空值转成 None。
    try:
        if pd.isna(
            value
        ):
            return None
    except (TypeError, ValueError):
        pass

    # 转成 string 并清除首尾空格。
    cleaned_value = str(
        value
    ).strip()

    # 文本化空值转成 None。
    if (
        not cleaned_value
        or cleaned_value.lower()
        in {
            "nan",
            "none",
            "<na>",
        }
    ):
        return None

    # 返回原始大小写文本。
    return cleaned_value


def python_int(
    value,
):
    """把 pandas/NumPy integer 显式转成标准 Python int。"""

    # 空值保持 None。
    if clean_text_value(
        value
    ) is None:
        return None

    # 先按 numeric 读取，再转 Python int。
    return int(
        float(
            value
        )
    )


def get_or_create_named_dimension(
    table_name,
    name_column,
    raw_name,
    normalized_name,
    now_text,
    extra_insert_columns=None,
    extra_insert_values=None,
):
    """按 LOWER(TRIM(name)) 查找或新建一个维度记录。"""

    # normalized name 是稳定匹配 key。
    normalized_key = clean_text_value(
        normalized_name
    )

    # 没有 key 时不创建维度记录。
    if normalized_key is None:
        return None, "null"

    # 统一使用小写 key 查询。
    normalized_key = normalized_key.lower()

    # 生成不会跨表或跨 column 混淆的缓存 key。
    cache_key = (
        table_name,
        name_column,
        normalized_key,
    )

    # 当前 Cell 已经解析过相同维度时直接复用。
    if cache_key in NAMED_DIMENSION_ID_CACHE:
        return (
            NAMED_DIMENSION_ID_CACHE[
                cache_key
            ],
            "reused",
        )

    # 查询所有匹配记录。
    existing_result = execute_d1(
        f"""
        SELECT "id"
        FROM "{table_name}"
        WHERE LOWER(TRIM("{name_column}")) = ?
        ORDER BY "id";
        """,
        [
            normalized_key,
        ],
    )

    # 提取 rows。
    existing_rows = result_rows(
        existing_result
    )

    # 同一 key 存在多条时停止，避免猜测。
    if len(
        existing_rows
    ) > 1:
        raise ValueError(
            f'{table_name}.{name_column} 存在多个相同 normalized key：'
            f"{normalized_key}"
        )

    # 已存在时复用。
    if existing_rows:
        # 保存数据库 id 到本次运行缓存。
        NAMED_DIMENSION_ID_CACHE[
            cache_key
        ] = int(
            existing_rows[0][
                "id"
            ]
        )

        # 返回数据库 id。
        return (
            NAMED_DIMENSION_ID_CACHE[
                cache_key
            ],
            "reused",
        )

    # 优先保存原始可读名称。
    display_name = (
        clean_text_value(
            raw_name
        )
        or normalized_key
    )

    # 保存额外 columns。
    extra_insert_columns = (
        extra_insert_columns
        or []
    )

    # 保存额外 values。
    extra_insert_values = (
        extra_insert_values
        or []
    )

    # 生成 INSERT columns。
    insert_columns = [
        name_column,
        *extra_insert_columns,
        "created_at",
        "updated_at",
    ]

    # 生成占位符。
    placeholders = ", ".join(
        [
            "?"
            for _ in insert_columns
        ]
    )

    # 生成带引号的 column names。
    quoted_columns = ", ".join(
        [
            f'"{column_name}"'
            for column_name
            in insert_columns
        ]
    )

    # 执行 INSERT。
    insert_result = execute_d1(
        f"""
        INSERT INTO "{table_name}" (
          {quoted_columns}
        )
        VALUES ({placeholders})
        RETURNING "id";
        """,
        [
            display_name,
            *extra_insert_values,
            now_text,
            now_text,
        ],
    )

    # 提取新 id。
    inserted_rows = result_rows(
        insert_result
    )

    # 没返回 id 时停止。
    if len(
        inserted_rows
    ) != 1:
        raise RuntimeError(
            f"新建 {table_name} 后没有取得唯一 id。"
        )

    # 保存新 id 到本次运行缓存。
    NAMED_DIMENSION_ID_CACHE[
        cache_key
    ] = int(
        inserted_rows[0][
            "id"
        ]
    )

    # 返回新 id。
    return (
        NAMED_DIMENSION_ID_CACHE[
            cache_key
        ],
        "inserted",
    )


# ============================================================
# 读取 CSV 并执行学历准入检查
# ============================================================

# 读取全部字段为 pandas string。
all_education_dataframe = pd.read_csv(
    EDUCATION_INPUT_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 读取 Candidate 级 status CSV。
education_status_dataframe = pd.read_csv(
    EDUCATION_STATUS_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 为每条 extraction record 建立本次运行内部稳定 key。
#
# 这个 key 只用于把 D1 ids 回填到原来的 CSV row，
# 保存 CSV 前会删除。
all_education_dataframe[
    "_import_row_key"
] = [
    f"education_row_{row_number}"
    for row_number
    in range(
        len(
            all_education_dataframe
        )
    )
]

# 保存必要 columns。
required_columns = [
    "application_id",
    "person_id",
    "candidate_snapshot_id",
    "raw_submission_id",
    "education_entry_order",
    "raw_education_text",
    "raw_school_name",
    "normalized_school_name",
    "raw_degree_name",
    "degree_category",
    "degree_level_rank",
    "raw_field_study_name",
    "normalized_field_study_name",
    "raw_major_name",
    "normalized_major_name",
    "gpa",
    "education_start_date",
    "education_end_date",
    "is_current",
]

# 找出缺失 columns。
missing_columns = [
    column_name
    for column_name
    in required_columns
    if column_name
    not in all_education_dataframe.columns
]

# 缺失时停止。
if missing_columns:
    raise ValueError(
        "学历 CSV 缺少必要字段："
        f"{missing_columns}"
    )

# 空文件不能继续。
if all_education_dataframe.empty:
    raise ValueError(
        "学历 CSV 没有任何记录。"
    )

# 标准化学位类别与固定等级的唯一对应关系。
DEGREE_CATEGORY_TO_RANK = {
    "High School": 1,
    "Associate": 2,
    "Bachelor": 3,
    "Master": 4,
    "Doctorate": 5,
}

# 保存准入状态。
eligibility_status_values = []

# 保存拒绝原因。
rejection_reason_values = []

# 逐条执行保守准入检查。
for _, education_row in (
    all_education_dataframe.iterrows()
):
    # 保存当前记录的全部拒绝原因。
    rejection_reasons = []

    # 读取原始学历段落。
    raw_education_text = clean_text_value(
        education_row[
            "raw_education_text"
        ]
    )

    # 读取原始学校。
    raw_school_name = clean_text_value(
        education_row[
            "raw_school_name"
        ]
    )

    # 读取原始学位。
    raw_degree_name = clean_text_value(
        education_row[
            "raw_degree_name"
        ]
    )

    # 直接读取 CSV 已经标准化的学位类别。
    degree_category = clean_text_value(
        education_row[
            "degree_category"
        ]
    )

    # 读取 CSV 学位等级。
    degree_level_rank = python_int(
        education_row[
            "degree_level_rank"
        ]
    )

    # 原始学历段落不能为空。
    if raw_education_text is None:
        rejection_reasons.append(
            "missing_raw_education_text"
        )

    # 原始学校不能为空。
    if raw_school_name is None:
        rejection_reasons.append(
            "missing_raw_school_name"
        )

    # 原始学位不能为空。
    if raw_degree_name is None:
        rejection_reasons.append(
            "missing_raw_degree_name"
        )

    # 取得当前标准类别应有的 rank。
    expected_degree_level_rank = (
        DEGREE_CATEGORY_TO_RANK.get(
            degree_category
        )
    )

    # Other 或其他未知类别不允许导入。
    if expected_degree_level_rank is None:
        rejection_reasons.append(
            "degree_category_other_or_invalid"
        )

    # rank 必须与 category 一致。
    if (
        expected_degree_level_rank
        is not None
        and degree_level_rank
        != expected_degree_level_rank
    ):
        rejection_reasons.append(
            "degree_rank_mismatch"
        )

    # 原始学位必须能够在原始学历段落中找到。
    if (
        raw_degree_name is not None
        and raw_education_text is not None
        and raw_degree_name.lower()
        not in raw_education_text.lower()
    ):
        rejection_reasons.append(
            "degree_evidence_not_found"
        )

    # 没有拒绝原因时接受。
    if not rejection_reasons:
        eligibility_status_values.append(
            "accepted"
        )
        rejection_reason_values.append(
            None
        )
    else:
        # 有任一拒绝原因时拒绝。
        eligibility_status_values.append(
            "rejected"
        )
        rejection_reason_values.append(
            "|".join(
                rejection_reasons
            )
        )

# 把准入状态加入完整结果。
all_education_dataframe[
    "import_eligibility_status"
] = eligibility_status_values

# 把拒绝原因加入完整结果。
all_education_dataframe[
    "rejection_reason"
] = rejection_reason_values

# 生成全部拒绝记录。
rejected_education_dataframe = (
    all_education_dataframe.loc[
        all_education_dataframe[
            "import_eligibility_status"
        ]
        == "rejected"
    ].copy()
)

# 只保留通过准入的学历。
education_dataframe = (
    all_education_dataframe.loc[
        all_education_dataframe[
            "import_eligibility_status"
        ]
        == "accepted"
    ].copy()
)

# 没有 accepted record 时停止。
if education_dataframe.empty:
    raise ValueError(
        "没有任何学历记录通过数据库导入准入检查。"
    )

# 将 ids 和排序字段转换成整数。
for integer_column_name in [
    "application_id",
    "person_id",
    "candidate_snapshot_id",
    "education_entry_order",
    "degree_level_rank",
]:
    education_dataframe[
        integer_column_name
    ] = pd.to_numeric(
        education_dataframe[
            integer_column_name
        ],
        errors="raise",
    ).astype(
        "int64"
    )

# 建立最高学历选择使用的稳定结束日期排序值。
education_dataframe[
    "_highest_end_date_sort"
] = education_dataframe[
    "education_end_date"
].fillna(
    ""
)

# 按学位等级、结束日期和原始顺序选择最高学历。
education_dataframe = (
    education_dataframe.sort_values(
        [
            "candidate_snapshot_id",
            "degree_level_rank",
            "_highest_end_date_sort",
            "education_entry_order",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# 所有 accepted records 先标记为非最高学历。
education_dataframe[
    "final_is_highest_degree"
] = 0

# 每个 Candidate 排序后的第一条是唯一最高学历。
highest_row_indexes = (
    education_dataframe.groupby(
        "candidate_snapshot_id",
        sort=False,
    ).head(
        1
    ).index
)

# 写入唯一最高学历标记。
education_dataframe.loc[
    highest_row_indexes,
    "final_is_highest_degree",
] = 1

# 在完整 extraction 结果中初始化最终最高学历标记。
all_education_dataframe[
    "final_is_highest_degree"
] = 0

# 把 accepted records 的最终最高学历标记回填到原始 rows。
final_highest_by_row_key = dict(
    zip(
        education_dataframe[
            "_import_row_key"
        ],
        education_dataframe[
            "final_is_highest_degree"
        ],
    )
)

# 根据内部 row key 回填。
all_education_dataframe[
    "final_is_highest_degree"
] = (
    all_education_dataframe[
        "_import_row_key"
    ].map(
        final_highest_by_row_key
    ).fillna(
        0
    ).astype(
        "int64"
    )
)

# 删除临时排序字段。
education_dataframe = (
    education_dataframe.drop(
        columns=[
            "_highest_end_date_sort",
        ]
    )
)

# 恢复最终稳定导入顺序。
education_dataframe = (
    education_dataframe.sort_values(
        [
            "application_id",
            "education_entry_order",
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# 显示准入统计。
print(
    "Candidate education eligibility summary:",
    {
        "input_record_count": int(
            len(
                all_education_dataframe
            )
        ),
        "accepted_record_count": int(
            len(
                education_dataframe
            )
        ),
        "rejected_record_count": int(
            len(
                rejected_education_dataframe
            )
        ),
        "accepted_candidate_count": int(
            education_dataframe[
                "candidate_snapshot_id"
            ].nunique()
        ),
    },
)


# ============================================================
# 验证 D1 schema 和 Candidate mapping
# ============================================================

# 需要的当前 schema columns。
required_table_columns = {
    "Education": {
        "id",
        "degree_id",
        "school_id",
        "field_study_id",
        "major_id",
        "is_current",
        "education_start_date",
        "education_end_date",
        "raw_school_name",
        "raw_degree_name",
        "raw_field_study_name",
        "raw_major_name",
        "gpa",
        "education_description",
        "created_at",
        "updated_at",
    },
    "Person_Education": {
        "id",
        "person_id",
        "education_id",
        "recorded_at",
        "is_current",
        "source_candidate_id",
        "created_at",
        "updated_at",
    },
    "Candidate_Education": {
        "id",
        "candidate_id",
        "person_education_id",
        "is_highest_degree",
        "created_at",
    },
    "Person": {
        "id",
        "highest_education_id",
    },
}

# 逐表验证 columns。
for table_name, expected_columns in (
    required_table_columns.items()
):
    # 查询 table info。
    table_info_result = execute_d1(
        f'PRAGMA table_info("{table_name}");'
    )

    # 取得实际 columns。
    actual_columns = {
        table_info_row[
            "name"
        ]
        for table_info_row
        in result_rows(
            table_info_result
        )
    }

    # 找出缺失项。
    missing_table_columns = (
        expected_columns
        - actual_columns
    )

    # 缺失时停止。
    if missing_table_columns:
        raise ValueError(
            f"{table_name} 缺少必要 columns："
            f"{sorted(missing_table_columns)}"
        )

# 保存本 Cell 依赖的两个唯一索引。
required_unique_indexes = {
    "Candidate_Education": (
        "idx_candidate_education_candidate_person_education"
    ),
    "Person_Education": (
        "idx_person_education_person_education"
    ),
}

# 验证两个索引都已经存在且确实是 UNIQUE。
for table_name, required_index_name in (
    required_unique_indexes.items()
):
    # 查询当前表的 indexes。
    index_list_result = execute_d1(
        f'PRAGMA index_list("{table_name}");'
    )

    # 查找指定 index。
    matching_index_rows = [
        index_row
        for index_row
        in result_rows(
            index_list_result
        )
        if index_row.get(
            "name"
        )
        == required_index_name
    ]

    # 索引必须唯一存在并且 unique = 1。
    if (
        len(
            matching_index_rows
        )
        != 1
        or int(
            matching_index_rows[0].get(
                "unique",
                0,
            )
        )
        != 1
    ):
        raise ValueError(
            f"{table_name} 缺少必要唯一索引："
            f"{required_index_name}"
        )

# 查询 D1 的真实 mapping。
mapping_result = execute_d1(
    """
    SELECT
      app."id" AS application_id,
      app."person_id",
      cs."id" AS candidate_snapshot_id,
      cs."application_date"
    FROM "Application" AS app
    INNER JOIN "Candidate_snapshot" AS cs
      ON cs."application_id" = app."id"
    ORDER BY app."id";
    """
)

# 建立 Candidate mapping。
mapping_by_candidate_id = {
    int(
        mapping_row[
            "candidate_snapshot_id"
        ]
    ): {
        "application_id": int(
            mapping_row[
                "application_id"
            ]
        ),
        "person_id": int(
            mapping_row[
                "person_id"
            ]
        ),
        "application_date": clean_text_value(
            mapping_row[
                "application_date"
            ]
        ),
    }
    for mapping_row
    in result_rows(
        mapping_result
    )
}

# 逐条验证 CSV mapping。
for _, education_row in (
    education_dataframe.iterrows()
):
    # 读取 Candidate id。
    candidate_id = python_int(
        education_row[
            "candidate_snapshot_id"
        ]
    )

    # Candidate 必须存在。
    if candidate_id not in (
        mapping_by_candidate_id
    ):
        raise ValueError(
            "D1 中不存在 Candidate_snapshot.id："
            f"{candidate_id}"
        )

    # 读取 D1 mapping。
    d1_mapping = (
        mapping_by_candidate_id[
            candidate_id
        ]
    )

    # Application 必须一致。
    if (
        python_int(
            education_row[
                "application_id"
            ]
        )
        != d1_mapping[
            "application_id"
        ]
    ):
        raise ValueError(
            f"Candidate {candidate_id} 的 application_id 不一致。"
        )

    # Person 必须一致。
    if (
        python_int(
            education_row[
                "person_id"
            ]
        )
        != d1_mapping[
            "person_id"
        ]
    ):
        raise ValueError(
            f"Candidate {candidate_id} 的 person_id 不一致。"
        )


# ============================================================
# 幂等写入全部通过准入的学历记录
# ============================================================

# 保存本次统一 timestamp。
now_text = utc_now_text()

# 保存 mapping output。
mapping_records = []

# 保存失败 output。
failed_records = []

# 保存需要重新计算最高学历的 Person ids。
affected_person_ids = set()

# 保存需要重新计算 Candidate 层唯一最高学历的 Candidate ids。
#
# 本集合只包含当前增量 CSV 中成功写入或复用学历的 Candidate，
# 不会在每次增量导入时扫描并修改全部历史 Candidate。
affected_candidate_ids = set()

# 逐条处理通过准入的结果。
for dataframe_index, education_row in (
    education_dataframe.iterrows()
):
    try:
        # 读取 ids。
        application_id = python_int(
            education_row[
                "application_id"
            ]
        )

        # 读取 Person id。
        person_id = python_int(
            education_row[
                "person_id"
            ]
        )

        # 读取 Candidate id。
        candidate_id = python_int(
            education_row[
                "candidate_snapshot_id"
            ]
        )

        # 读取 entry order。
        education_entry_order = python_int(
            education_row[
                "education_entry_order"
            ]
        )

        # 取得统一 mapping。
        d1_mapping = (
            mapping_by_candidate_id[
                candidate_id
            ]
        )

        # 取得 school id。
        (
            school_id,
            school_action,
        ) = get_or_create_named_dimension(
            table_name="School",
            name_column="school_name",
            raw_name=education_row[
                "raw_school_name"
            ],
            normalized_name=education_row[
                "normalized_school_name"
            ],
            now_text=now_text,
        )

        # Education.school_id NOT NULL，因此必须有 school。
        if school_id is None:
            raise ValueError(
                "normalized_school_name 为空，无法生成 school_id。"
            )

        # 读取 degree category。
        degree_category = clean_text_value(
            education_row[
                "degree_category"
            ]
        )

        # 读取 degree rank。
        degree_level_rank = python_int(
            education_row[
                "degree_level_rank"
            ]
        )

        # Other 没有明确学位，不创建假的 Degree。
        if (
            degree_category is None
            or degree_category == "Other"
        ):
            degree_id = None
            degree_action = "null"
        else:
            # 使用 canonical category 建立 Degree。
            (
                degree_id,
                degree_action,
            ) = get_or_create_named_dimension(
                table_name="Degree",
                name_column="degree_name",
                raw_name=degree_category,
                normalized_name=degree_category.lower(),
                now_text=now_text,
                extra_insert_columns=[
                    "degree_level_rank",
                ],
                extra_insert_values=[
                    degree_level_rank,
                ],
            )

        # 读取 field。
        normalized_field_name = clean_text_value(
            education_row[
                "normalized_field_study_name"
            ]
        )

        # 有 field 时查找或新建。
        if normalized_field_name is not None:
            (
                field_study_id,
                field_action,
            ) = get_or_create_named_dimension(
                table_name="Field_study",
                name_column="field_study_name",
                raw_name=education_row[
                    "raw_field_study_name"
                ],
                normalized_name=normalized_field_name,
                now_text=now_text,
            )
        else:
            # 无 field 时保存 NULL。
            field_study_id = None
            field_action = "null"

        # 当前 CSV 暂时没有可靠 major mapping。
        major_id = None

        # 读取全部 Education 值。
        is_current = python_int(
            education_row[
                "is_current"
            ]
        )

        # 开始日期。
        education_start_date = clean_text_value(
            education_row[
                "education_start_date"
            ]
        )

        # 结束日期。
        education_end_date = clean_text_value(
            education_row[
                "education_end_date"
            ]
        )

        # 原始学校。
        raw_school_name = clean_text_value(
            education_row[
                "raw_school_name"
            ]
        )

        # 原始学位。
        raw_degree_name = clean_text_value(
            education_row[
                "raw_degree_name"
            ]
        )

        # 原始 field。
        raw_field_study_name = clean_text_value(
            education_row[
                "raw_field_study_name"
            ]
        )

        # 原始 major 暂时通常为 NULL。
        raw_major_name = clean_text_value(
            education_row[
                "raw_major_name"
            ]
        )

        # GPA。
        gpa = clean_text_value(
            education_row[
                "gpa"
            ]
        )

        # 按最终决定把完整原始 education entry
        # 保存到 Education.education_description。
        education_description = (
            clean_text_value(
                education_row[
                    "raw_education_text"
                ]
            )
        )

        # 先在同一个 Person 中查找完全对应的学历。
        existing_person_education_result = (
            execute_d1(
                """
                SELECT
                  pe."id" AS person_education_id,
                  e."id" AS education_id
                FROM "Person_Education" AS pe
                INNER JOIN "Education" AS e
                  ON e."id" = pe."education_id"
                WHERE pe."person_id" = ?
                  AND e."school_id" = ?
                  AND e."degree_id" IS ?
                  AND e."field_study_id" IS ?
                  AND e."major_id" IS ?
                  AND e."is_current" IS ?
                  AND e."education_start_date" IS ?
                  AND e."education_end_date" IS ?
                  AND e."raw_school_name" IS ?
                  AND e."raw_degree_name" IS ?
                  AND e."raw_field_study_name" IS ?
                  AND e."raw_major_name" IS ?
                  AND e."gpa" IS ?
                  AND e."education_description" IS ?
                ORDER BY pe."id";
                """,
                [
                    person_id,
                    school_id,
                    degree_id,
                    field_study_id,
                    major_id,
                    is_current,
                    education_start_date,
                    education_end_date,
                    raw_school_name,
                    raw_degree_name,
                    raw_field_study_name,
                    raw_major_name,
                    gpa,
                    education_description,
                ],
            )
        )

        # 提取已存在 Person_Education。
        existing_person_education_rows = (
            result_rows(
                existing_person_education_result
            )
        )

        # 多条完全相同时停止，避免猜测。
        if len(
            existing_person_education_rows
        ) > 1:
            raise ValueError(
                "同一个 Person 已存在多条完全相同的 Person_Education："
                f"person_id={person_id}, "
                f"candidate_id={candidate_id}, "
                f"entry_order={education_entry_order}"
            )

        # 已存在时复用。
        if existing_person_education_rows:
            # 读取 ids。
            person_education_id = int(
                existing_person_education_rows[0][
                    "person_education_id"
                ]
            )

            # 读取 Education id。
            education_id = int(
                existing_person_education_rows[0][
                    "education_id"
                ]
            )

            # 标记复用。
            education_action = "reused"
            person_education_action = "reused"
        else:
            # 新建 Education。
            education_insert_result = execute_d1(
                """
                INSERT INTO "Education" (
                  "degree_id",
                  "school_id",
                  "field_study_id",
                  "major_id",
                  "is_current",
                  "education_start_date",
                  "education_end_date",
                  "raw_school_name",
                  "raw_degree_name",
                  "raw_field_study_name",
                  "raw_major_name",
                  "gpa",
                  "education_description",
                  "created_at",
                  "updated_at"
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                RETURNING "id";
                """,
                [
                    degree_id,
                    school_id,
                    field_study_id,
                    major_id,
                    is_current,
                    education_start_date,
                    education_end_date,
                    raw_school_name,
                    raw_degree_name,
                    raw_field_study_name,
                    raw_major_name,
                    gpa,
                    education_description,
                    now_text,
                    now_text,
                ],
            )

            # 读取 Education id。
            education_id = int(
                result_rows(
                    education_insert_result
                )[0][
                    "id"
                ]
            )

            # 新建 Person_Education。
            person_education_insert_result = execute_d1(
                """
                INSERT INTO "Person_Education" (
                  "person_id",
                  "education_id",
                  "recorded_at",
                  "is_current",
                  "source_candidate_id",
                  "created_at",
                  "updated_at"
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
                RETURNING "id";
                """,
                [
                    person_id,
                    education_id,
                    d1_mapping[
                        "application_date"
                    ],
                    is_current,
                    candidate_id,
                    now_text,
                    now_text,
                ],
            )

            # 读取 Person_Education id。
            person_education_id = int(
                result_rows(
                    person_education_insert_result
                )[0][
                    "id"
                ]
            )

            # 标记新增。
            education_action = "inserted"
            person_education_action = "inserted"

        # 查询 Candidate_Education link。
        candidate_education_result = execute_d1(
            """
            SELECT
              "id",
              "is_highest_degree"
            FROM "Candidate_Education"
            WHERE "candidate_id" = ?
              AND "person_education_id" = ?
            ORDER BY "id";
            """,
            [
                candidate_id,
                person_education_id,
            ],
        )

        # 提取 link rows。
        candidate_education_rows = result_rows(
            candidate_education_result
        )

        # 多条 link 时停止。
        if len(
            candidate_education_rows
        ) > 1:
            raise ValueError(
                "Candidate_Education 存在重复 link："
                f"candidate_id={candidate_id}, "
                f"person_education_id={person_education_id}"
            )

        # 读取 highest flag。
        is_highest_degree = python_int(
            education_row[
                "final_is_highest_degree"
            ]
        )

        # 已存在 link 时按准入后重新计算的 flag 更新。
        if candidate_education_rows:
            # 读取 link id。
            candidate_education_id = int(
                candidate_education_rows[0][
                    "id"
                ]
            )

            # 读取旧 flag。
            existing_highest_flag = python_int(
                candidate_education_rows[0][
                    "is_highest_degree"
                ]
            )

            # flag 不同时更新。
            if (
                existing_highest_flag
                != is_highest_degree
            ):
                execute_d1(
                    """
                    UPDATE "Candidate_Education"
                    SET "is_highest_degree" = ?
                    WHERE "id" = ?;
                    """,
                    [
                        is_highest_degree,
                        candidate_education_id,
                    ],
                )

                # 标记更新。
                candidate_education_action = (
                    "updated"
                )
            else:
                # 完全一致时复用。
                candidate_education_action = (
                    "reused"
                )
        else:
            # 新建 Candidate_Education。
            candidate_education_insert_result = (
                execute_d1(
                    """
                    INSERT INTO "Candidate_Education" (
                      "candidate_id",
                      "person_education_id",
                      "is_highest_degree",
                      "created_at"
                    )
                    VALUES (?, ?, ?, ?)
                    RETURNING "id";
                    """,
                    [
                        candidate_id,
                        person_education_id,
                        is_highest_degree,
                        now_text,
                    ],
                )
            )

            # 读取 link id。
            candidate_education_id = int(
                result_rows(
                    candidate_education_insert_result
                )[0][
                    "id"
                ]
            )

            # 标记新增。
            candidate_education_action = (
                "inserted"
            )

        # 保存成功 mapping。
        mapping_records.append(
            {
                "_import_row_key": (
                    education_row[
                        "_import_row_key"
                    ]
                ),
                "rule_version": (
                    clean_text_value(
                        education_row[
                            "rule_version"
                        ]
                    )
                    if "rule_version"
                    in education_dataframe.columns
                    else None
                ),
                "application_id": (
                    application_id
                ),
                "person_id": person_id,
                "candidate_snapshot_id": (
                    candidate_id
                ),
                "education_entry_order": (
                    education_entry_order
                ),
                "school_id": school_id,
                "degree_id": degree_id,
                "field_study_id": (
                    field_study_id
                ),
                "education_id": education_id,
                "person_education_id": (
                    person_education_id
                ),
                "candidate_education_id": (
                    candidate_education_id
                ),
                "is_highest_degree": (
                    is_highest_degree
                ),
                "school_action": school_action,
                "degree_action": degree_action,
                "field_action": field_action,
                "education_action": (
                    education_action
                ),
                "person_education_action": (
                    person_education_action
                ),
                "candidate_education_action": (
                    candidate_education_action
                ),
                "database_import_status": (
                    "success"
                ),
                "database_import_error": None,
            }
        )

        # 只有当前学历成功写入或复用后，
        # 才把 Person 加入最高学历回填范围。
        affected_person_ids.add(
            person_id
        )

        # 只有当前学历成功写入或复用后，
        # 才把 Candidate 加入最高学历重新计算范围。
        affected_candidate_ids.add(
            candidate_id
        )

    except Exception as error:
        # 复制完整输入 row。
        failed_record = {
            column_name: (
                education_row[
                    column_name
                ]
            )
            for column_name
            in education_dataframe.columns
        }

        # 添加错误。
        failed_record[
            "import_error"
        ] = str(
            error
        )

        # 保存稳定 row key。
        failed_record[
            "_import_row_key"
        ] = education_row[
            "_import_row_key"
        ]

        # 保存统一失败状态。
        failed_record[
            "database_import_status"
        ] = "failed"

        # 保存统一错误字段。
        failed_record[
            "database_import_error"
        ] = str(
            error
        )

        # 保存失败。
        failed_records.append(
            failed_record
        )

        # 当前学历记录失败后继续处理下一条。
        #
        # 每条 D1 写入都使用独立 id 和唯一索引保护，
        # 因此单条失败不需要丢弃其他可以成功写入的学历。
        continue

    # 每100条显示进度，同时显示最后一条。
    processed_count = (
        dataframe_index + 1
    )

    # 显示进度。
    if (
        processed_count % 100 == 0
        or processed_count
        == len(
            education_dataframe
        )
    ):
        print(
            "Processed Candidate education records:",
            f"{processed_count}/"
            f"{len(education_dataframe)}",
        )


# ============================================================
# 重新计算本批次受影响 Candidate 的唯一最高学历
# ============================================================

# 保存实际修复过最高标记的 Candidate 数量。
candidate_highest_recalculated_count = 0

# 只处理当前增量批次中成功导入学历的 Candidate。
for candidate_index, candidate_id in enumerate(
    sorted(
        affected_candidate_ids
    ),
    start=1,
):
    # 从该 Candidate 当前已经关联的全部学历中，
    # 按学位等级、结束日期和稳定 id 选出唯一最高学历。
    candidate_highest_result = execute_d1(
        """
        SELECT
          ce."id" AS candidate_education_id
        FROM "Candidate_Education" AS ce
        INNER JOIN "Person_Education" AS pe
          ON pe."id" = ce."person_education_id"
        INNER JOIN "Education" AS e
          ON e."id" = pe."education_id"
        LEFT JOIN "Degree" AS d
          ON d."id" = e."degree_id"
        WHERE ce."candidate_id" = ?
        ORDER BY
          COALESCE(d."degree_level_rank", 0) DESC,
          COALESCE(e."education_end_date", '') DESC,
          pe."id" ASC,
          ce."id" ASC
        LIMIT 1;
        """,
        [
            candidate_id,
        ],
    )

    # 提取唯一最高学历候选记录。
    candidate_highest_rows = result_rows(
        candidate_highest_result
    )

    # 当前 Candidate 已成功写入至少一条学历，
    # 因此必须能够找到一条最高学历 link。
    if len(candidate_highest_rows) != 1:
        raise RuntimeError(
            "无法为 Candidate 取得唯一最高学历："
            f"candidate_id={candidate_id}"
        )

    # 取得应当保留 highest=1 的 Candidate_Education.id。
    highest_candidate_education_id = int(
        candidate_highest_rows[0][
            "candidate_education_id"
        ]
    )

    # 一条 SQL 同时完成：
    # 1. 最高记录设为1；
    # 2. 该 Candidate 的其他历史记录全部设为0。
    #
    # 重复运行时，结果仍然完全相同。
    execute_d1(
        """
        UPDATE "Candidate_Education"
        SET "is_highest_degree" =
          CASE
            WHEN "id" = ? THEN 1
            ELSE 0
          END
        WHERE "candidate_id" = ?;
        """,
        [
            highest_candidate_education_id,
            candidate_id,
        ],
    )

    # 增加本次重新计算数量。
    candidate_highest_recalculated_count += 1

    # 每100个显示一次进度，同时显示最后一个。
    if (
        candidate_index % 100 == 0
        or candidate_index
        == len(affected_candidate_ids)
    ):
        print(
            "Recalculated Candidate highest education:",
            f"{candidate_index}/"
            f"{len(affected_candidate_ids)}",
        )


# ============================================================
# 更新本批次受影响 Person 的 highest_education_id
# ============================================================

# 逐个 Person 计算所有历史学历中的最高记录。
for person_index, person_id in enumerate(
    sorted(
        affected_person_ids
    ),
    start=1,
):
    # 按 degree rank、结束日期和 id 稳定选择最高学历。
    highest_result = execute_d1(
        """
        SELECT
          pe."id" AS person_education_id
        FROM "Person_Education" AS pe
        INNER JOIN "Education" AS e
          ON e."id" = pe."education_id"
        LEFT JOIN "Degree" AS d
          ON d."id" = e."degree_id"
        WHERE pe."person_id" = ?
        ORDER BY
          COALESCE(d."degree_level_rank", 0) DESC,
          COALESCE(e."education_end_date", '') DESC,
          pe."id" ASC
        LIMIT 1;
        """,
        [
            person_id,
        ],
    )

    # 提取最高记录。
    highest_rows = result_rows(
        highest_result
    )

    # 当前 Person 应至少有一条。
    if len(
        highest_rows
    ) != 1:
        raise RuntimeError(
            "无法为 Person 取得唯一最高学历："
            f"{person_id}"
        )

    # 取得 Person_Education.id。
    highest_person_education_id = int(
        highest_rows[0][
            "person_education_id"
        ]
    )

    # 更新 Person。
    execute_d1(
        """
        UPDATE "Person"
        SET
          "highest_education_id" = ?,
          "updated_at" = ?
        WHERE "id" = ?
          AND "highest_education_id" IS NOT ?;
        """,
        [
            highest_person_education_id,
            now_text,
            person_id,
            highest_person_education_id,
        ],
    )

    # 每100个显示一次进度。
    if (
        person_index % 100 == 0
        or person_index
        == len(
            affected_person_ids
        )
    ):
        print(
            "Updated Person highest education:",
            f"{person_index}/"
            f"{len(affected_person_ids)}",
        )


# ============================================================
# 验证数据库结果并覆盖保存 mapping CSV
# ============================================================

# 建立 mapping DataFrame。
mapping_dataframe = pd.DataFrame(
    mapping_records
)

# 建立失败 DataFrame。
failed_dataframe = pd.DataFrame(
    failed_records,
    columns=[
        *education_dataframe.columns,
        "import_error",
    ],
)

# 成功和失败数量合计必须与 accepted 输入完全一致。
if (
    len(
        mapping_dataframe
    )
    + len(
        failed_dataframe
    )
    != len(
        education_dataframe
    )
):
    raise RuntimeError(
        "已处理记录数与 accepted 输入记录数不一致。"
    )

# 初始化记录级数据库结果 columns。
database_result_columns = [
    "database_import_status",
    "database_import_error",
    "school_id",
    "degree_id",
    "field_study_id",
    "education_id",
    "person_education_id",
    "candidate_education_id",
]

# 每次运行都重新计算，不沿用上一次 CSV 中保存的数据库状态。
for database_result_column in (
    database_result_columns
):
    all_education_dataframe[
        database_result_column
    ] = pd.NA

# 被准入规则拒绝的记录直接标记 rejected。
all_education_dataframe.loc[
    all_education_dataframe[
        "import_eligibility_status"
    ]
    == "rejected",
    "database_import_status",
] = "rejected"

# 建立内部 row key 到原始 DataFrame index 的对应关系。
output_index_by_row_key = {
    row_key: dataframe_index
    for dataframe_index, row_key
    in all_education_dataframe[
        "_import_row_key"
    ].items()
}

# 把成功数据库 ids 回填到原始 extraction rows。
for mapping_record in mapping_records:
    # 找到原始 row。
    output_dataframe_index = (
        output_index_by_row_key[
            mapping_record[
                "_import_row_key"
            ]
        ]
    )

    # 保存统一 success 状态。
    all_education_dataframe.at[
        output_dataframe_index,
        "database_import_status",
    ] = "success"

    # 保存全部数据库 ids。
    for database_id_column in [
        "school_id",
        "degree_id",
        "field_study_id",
        "education_id",
        "person_education_id",
        "candidate_education_id",
    ]:
        all_education_dataframe.at[
            output_dataframe_index,
            database_id_column,
        ] = mapping_record.get(
            database_id_column
        )

# 把失败状态和错误回填到原始 extraction rows。
for failed_record in failed_records:
    # 找到原始 row。
    output_dataframe_index = (
        output_index_by_row_key[
            failed_record[
                "_import_row_key"
            ]
        ]
    )

    # 保存失败状态。
    all_education_dataframe.at[
        output_dataframe_index,
        "database_import_status",
    ] = "failed"

    # 保存具体错误。
    all_education_dataframe.at[
        output_dataframe_index,
        "database_import_error",
    ] = failed_record[
        "database_import_error"
    ]

# 建立 Candidate 级汇总。
candidate_import_summary = {}

# 逐个 status CSV Candidate 生成结果。
for _, status_row in (
    education_status_dataframe.iterrows()
):
    # 读取 Candidate id。
    candidate_id = python_int(
        status_row[
            "candidate_snapshot_id"
        ]
    )

    # 取得当前 Candidate 的全部 extraction rows。
    candidate_rows = (
        all_education_dataframe.loc[
            pd.to_numeric(
                all_education_dataframe[
                    "candidate_snapshot_id"
                ],
                errors="coerce",
            )
            == candidate_id
        ]
    )

    # 统计各状态。
    accepted_count = int(
        (
            candidate_rows[
                "import_eligibility_status"
            ]
            == "accepted"
        ).sum()
    )

    rejected_count = int(
        (
            candidate_rows[
                "database_import_status"
            ]
            == "rejected"
        ).sum()
    )

    success_count = int(
        (
            candidate_rows[
                "database_import_status"
            ]
            == "success"
        ).sum()
    )

    failed_count = int(
        (
            candidate_rows[
                "database_import_status"
            ]
            == "failed"
        ).sum()
    )

    # Candidate 有失败记录时整体标记 failed。
    if failed_count > 0:
        candidate_database_status = (
            "failed"
        )

    # 没有任何 accepted record 时标记 rejected。
    elif accepted_count == 0:
        candidate_database_status = (
            "rejected"
        )

    # 其余 Candidate 全部成功。
    else:
        candidate_database_status = (
            "success"
        )

    # 查找成功导入的最终最高学历。
    final_highest_rows = (
        candidate_rows.loc[
            (
                candidate_rows[
                    "final_is_highest_degree"
                ]
                == 1
            )
            & (
                candidate_rows[
                    "database_import_status"
                ]
                == "success"
            )
        ]
    )

    # 正常情况下至多一条。
    if len(
        final_highest_rows
    ) > 1:
        raise RuntimeError(
            "Candidate 存在多条最终最高学历："
            f"{candidate_id}"
        )

    # 读取最终最高学历。
    final_highest_degree_category = (
        clean_text_value(
            final_highest_rows.iloc[0][
                "degree_category"
            ]
        )
        if len(
            final_highest_rows
        )
        == 1
        else None
    )

    # 读取最终最高学历 rank。
    final_highest_degree_level_rank = (
        python_int(
            final_highest_rows.iloc[0][
                "degree_level_rank"
            ]
        )
        if len(
            final_highest_rows
        )
        == 1
        else None
    )

    # 保存 Candidate 汇总。
    candidate_import_summary[
        candidate_id
    ] = {
        "accepted_education_record_count": (
            accepted_count
        ),
        "rejected_education_record_count": (
            rejected_count
        ),
        "database_imported_record_count": (
            success_count
        ),
        "database_failed_record_count": (
            failed_count
        ),
        "final_highest_degree_category": (
            final_highest_degree_category
        ),
        "final_highest_degree_level_rank": (
            final_highest_degree_level_rank
        ),
        "candidate_education_import_status": (
            candidate_database_status
        ),
    }

# 将 Candidate 汇总 columns 回填到 status CSV。
for summary_column in [
    "accepted_education_record_count",
    "rejected_education_record_count",
    "database_imported_record_count",
    "database_failed_record_count",
    "final_highest_degree_category",
    "final_highest_degree_level_rank",
    "candidate_education_import_status",
]:
    education_status_dataframe[
        summary_column
    ] = [
        candidate_import_summary[
            python_int(
                candidate_id
            )
        ][
            summary_column
        ]
        for candidate_id
        in education_status_dataframe[
            "candidate_snapshot_id"
        ]
    ]

# 删除只在本次运行内部使用的 row key。
education_output_dataframe = (
    all_education_dataframe.drop(
        columns=[
            "_import_row_key",
        ]
    )
)

# 使用临时文件后再替换原文件，减少中途写坏 CSV 的风险。
education_output_temp_path = (
    f"{EDUCATION_INPUT_CSV_PATH}.tmp"
)

# 写入记录级临时文件。
education_output_dataframe.to_csv(
    education_output_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 替换原 extraction CSV。
os.replace(
    education_output_temp_path,
    EDUCATION_INPUT_CSV_PATH,
)

# 建立 status 临时文件。
education_status_temp_path = (
    f"{EDUCATION_STATUS_CSV_PATH}.tmp"
)

# 写入 Candidate 级临时文件。
education_status_dataframe.to_csv(
    education_status_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 替换原 status CSV。
os.replace(
    education_status_temp_path,
    EDUCATION_STATUS_CSV_PATH,
)

# 存在数据库写入失败时，在 CSV 已保存失败状态后停止。
if failed_records:
    raise RuntimeError(
        "部分学历记录写入 D1 失败。"
        "具体错误已经写回 extraction CSV。"
    )

# 当前增量导入只验证本批次受影响的 Candidate，
# 不在每次增量写入时扫描全部历史 Candidate_Education。
invalid_highest_rows = []

# 每次最多验证50个 Candidate，避免单条 SQL 参数过多。
HIGHEST_VERIFICATION_CHUNK_SIZE = 50

# 分块验证当前受影响 Candidate。
affected_candidate_id_list = sorted(
    affected_candidate_ids
)

for highest_chunk_start in range(
    0,
    len(affected_candidate_id_list),
    HIGHEST_VERIFICATION_CHUNK_SIZE,
):
    # 取得当前验证分块。
    highest_chunk_ids = affected_candidate_id_list[
        highest_chunk_start:
        highest_chunk_start
        + HIGHEST_VERIFICATION_CHUNK_SIZE
    ]

    # 生成参数占位符。
    highest_placeholders = ",".join(
        ["?"] * len(highest_chunk_ids)
    )

    # 验证当前分块每个 Candidate 恰好一条 highest=1。
    highest_verification_result = execute_d1(
        f"""
        SELECT
          ce."candidate_id",
          SUM(
            CASE
              WHEN ce."is_highest_degree" = 1
              THEN 1
              ELSE 0
            END
          ) AS highest_count
        FROM "Candidate_Education" AS ce
        WHERE ce."candidate_id" IN (
          {highest_placeholders}
        )
        GROUP BY ce."candidate_id"
        HAVING highest_count != 1
        ORDER BY ce."candidate_id";
        """,
        [
            int(candidate_id)
            for candidate_id
            in highest_chunk_ids
        ],
    )

    # 合并当前分块发现的异常。
    invalid_highest_rows.extend(
        result_rows(
            highest_verification_result
        )
    )

# 检查。
if invalid_highest_rows:
    raise RuntimeError(
        "本批次 Candidate_Education 的 highest 标记数量错误："
        f"{invalid_highest_rows[:20]}"
    )

# 执行外键检查。
foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)

# 提取 violation。
foreign_key_rows = result_rows(
    foreign_key_result
)

# 有 violation 时停止。
if foreign_key_rows:
    raise RuntimeError(
        "D1 存在 foreign key violation："
        f"{foreign_key_rows[:20]}"
    )

# 显示 summary。
print(
    "Candidate education import summary:",
    {
        "input_record_count": int(
            len(
                all_education_dataframe
            )
        ),
        "accepted_record_count": int(
            len(
                education_dataframe
            )
        ),
        "rejected_record_count": int(
            len(
                rejected_education_dataframe
            )
        ),
        "database_success_record_count": int(
            len(
                mapping_dataframe
            )
        ),
        "failed_record_count": int(
            len(
                failed_dataframe
            )
        ),
        "candidate_count": int(
            education_dataframe[
                "candidate_snapshot_id"
            ].nunique()
        ),
        "person_count": int(
            education_dataframe[
                "person_id"
            ].nunique()
        ),
        "candidate_highest_recalculated_count": int(
            candidate_highest_recalculated_count
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_rows
            )
        ),
    },
)

# 显示更新后的记录级 CSV 路径。
print(
    "Updated education extraction CSV:",
    EDUCATION_INPUT_CSV_PATH,
)

# 显示更新后的 Candidate 级 CSV 路径。
print(
    "Updated education status CSV:",
    EDUCATION_STATUS_CSV_PATH,
)

# 显示记录级结果预览。
display(
    education_output_dataframe.head(
        20
    )
)

# 显示完成。
print(
    "Candidate education import completed successfully."
)


### Candidate_skill, Person_skill 数据导入 （可以重复跑，但是不建议因为运行时间比较长）

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 使用 resume_extraction_consolidated.csv 中已经匹配成功的 skills，
# 幂等写入 D1 Person_Skill 和 Candidate_Skill。
#
# 重要：
# 1. 本 Cell 不依赖其他 Colab Cell 的 variable、function 或 class。
# 2. 本 Cell 不重新从 raw resume text 提取 skills。
# 3. 本 Cell 只使用 resume_skills_matched_summary 中已有的匹配结果。
# 4. 本 Cell 使用 D1 中每一个 Skill record 的 skill_name 作为匹配关键词。
# 5. 同一个 skill_name 如果属于多个 Skill type，会保留全部对应的 skill_id。
# 6. 匹配前只执行小写化、首尾空白清理和连续空白合并。
# 7. 匹配使用完整技能名称，不使用容易误匹配的 substring 查找。
# 8. 相同输入可以重复运行，不会重复建立 Person_Skill 或 Candidate_Skill。
# 9. proficiency 和 years_experience 没有可靠来源，因此保持 SQL NULL。
# 10. 本 Cell 直接覆盖更新原 resume_extraction_consolidated.csv。
# 11. 数据库状态只使用 success、rejected 和 failed。
# ============================================================

# os 用于检查 Google Drive 输入文件和安全替换 CSV。
import os

# time 只用于 D1 临时失败后的重试等待。
import time

# datetime 用于生成统一 UTC timestamp。
from datetime import datetime, timezone

# pandas 用于读取、匹配和覆盖保存 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定 Google Drive 路径
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# Application 结果 CSV。
APPLICATION_CSV_PATH = (
    f"{DATASET_FOLDER}/dedup_results/"
    "application_dedup_v2/application.csv"
)

# 简历提取汇总 CSV。
RESUME_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "resume_extraction_consolidated.csv"
)

# 所有必要文件都必须存在。
for required_file_path in [
    APPLICATION_CSV_PATH,
    RESUME_EXTRACTION_CSV_PATH,
]:
    # 缺少文件时停止。
    if not os.path.exists(
        required_file_path
    ):
        raise FileNotFoundError(
            "找不到必要输入文件："
            f"{required_file_path}"
        )


# ============================================================
# Cloudflare D1 独立连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP session。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时失败有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # 请求和 SQL 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    d1_result,
):
    """提取第一条 D1 statement 的 results rows。"""

    # 空 result 返回空 list。
    if not d1_result:
        return []

    # 返回 results。
    return d1_result[0].get(
        "results",
        [],
    )


def clean_text_value(
    value,
):
    """把 pandas 空值和空字符串统一成 None。"""

    # pandas 空值。
    if pd.isna(
        value
    ):
        return None

    # 清理首尾空白。
    cleaned_value = str(
        value
    ).strip()

    # 空字符串和常见空值文本。
    if (
        not cleaned_value
        or cleaned_value.lower()
        in {
            "nan",
            "none",
            "null",
        }
    ):
        return None

    # 返回清理值。
    return cleaned_value


def python_int(
    value,
):
    """把 pandas/NumPy/string integer 转成标准 Python int。"""

    # 清理值。
    cleaned_value = clean_text_value(
        value
    )

    # 空值保持 None。
    if cleaned_value is None:
        return None

    # 兼容 CSV 中的 1.0。
    return int(
        float(
            cleaned_value
        )
    )


def normalize_skill_match_key(
    value,
):
    """
    生成只用于 Skill 完整名称匹配的稳定 key。

    这里只执行空值处理、首尾空白清理、连续空白合并和小写化。
    不删除 +、#、.、- 等字符，避免把 C、C++、C# 等技能混为一条。
    """

    # 先把空值和空字符串统一成 None。
    cleaned_value = clean_text_value(
        value
    )

    # 空值没有可用的匹配 key。
    if cleaned_value is None:
        return None

    # 合并连续空白，并使用 casefold 执行稳定小写化。
    return " ".join(
        cleaned_value.split()
    ).casefold()


def utc_now_text():
    """返回统一 UTC ISO timestamp。"""

    # 使用秒级 UTC。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def split_skill_summary(
    skill_summary,
):
    """按分号拆分技能，并在单份简历内按小写名称去重。"""

    # 清理 summary。
    cleaned_summary = clean_text_value(
        skill_summary
    )

    # 空 summary 返回空 list。
    if cleaned_summary is None:
        return []

    # 保存最终技能。
    skill_names = []

    # 保存已出现的小写 key。
    seen_skill_keys = set()

    # 按原 pipeline 使用的分号拆分。
    for raw_skill_name in cleaned_summary.split(
        ";"
    ):
        # 清理技能名。
        skill_name = clean_text_value(
            raw_skill_name
        )

        # 空 token 跳过。
        if skill_name is None:
            continue

        # 生成完整技能名称匹配 key。
        skill_key = normalize_skill_match_key(
            skill_name
        )

        # 同一份简历内不重复。
        if skill_key in seen_skill_keys:
            continue

        # 保存 key 和原始显示名。
        seen_skill_keys.add(
            skill_key
        )
        skill_names.append(
            skill_name
        )

    # 返回稳定原始顺序。
    return skill_names


# ============================================================
# 读取并验证两个固定 CSV
# ============================================================

# 读取 Application CSV。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 读取 resume extraction CSV。
resume_extraction_dataframe = pd.read_csv(
    RESUME_EXTRACTION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# Application 必要 columns。
required_application_columns = {
    "id",
    "raw_submission_id",
}

# 找出缺失 columns。
missing_application_columns = (
    required_application_columns
    - set(
        application_dataframe.columns
    )
)

# 缺失时停止。
if missing_application_columns:
    raise ValueError(
        "application.csv 缺少必要 columns："
        f"{sorted(missing_application_columns)}"
    )

# resume extraction 必要 columns。
required_resume_columns = {
    "raw_submission_id",
    "resume_skills_matched_summary",
}

# 找出缺失 columns。
missing_resume_columns = (
    required_resume_columns
    - set(
        resume_extraction_dataframe.columns
    )
)

# 缺失时停止。
if missing_resume_columns:
    raise ValueError(
        "resume_extraction_consolidated.csv 缺少必要 columns："
        f"{sorted(missing_resume_columns)}"
    )

# 只保留已经成功写入 D1 Application 的 CSV records。
valid_application_dataframe = (
    application_dataframe.loc[
        application_dataframe[
            "id"
        ].notna()
    ].copy()
)

# 转换 ids。
valid_application_dataframe[
    "id"
] = pd.to_numeric(
    valid_application_dataframe[
        "id"
    ],
    errors="raise",
).astype(
    "int64"
)

# 转换 raw_submission_id。
valid_application_dataframe[
    "raw_submission_id"
] = pd.to_numeric(
    valid_application_dataframe[
        "raw_submission_id"
    ],
    errors="raise",
).astype(
    "int64"
)

# Application.id 必须唯一。
if valid_application_dataframe[
    "id"
].duplicated().any():
    raise ValueError(
        "application.csv 的有效 Application.id 存在重复。"
    )

# raw_submission_id 必须唯一。
if valid_application_dataframe[
    "raw_submission_id"
].duplicated().any():
    raise ValueError(
        "application.csv 的有效 raw_submission_id 存在重复。"
    )

# 转换 resume raw id。
resume_extraction_dataframe[
    "raw_submission_id"
] = pd.to_numeric(
    resume_extraction_dataframe[
        "raw_submission_id"
    ],
    errors="raise",
).astype(
    "int64"
)

# resume raw id 必须唯一。
if resume_extraction_dataframe[
    "raw_submission_id"
].duplicated().any():
    raise ValueError(
        "resume_extraction_consolidated.csv "
        "存在重复 raw_submission_id。"
    )

# 建立 Application.id 到 raw_submission_id mapping。
raw_submission_id_by_application_id = {
    int(
        application_row[
            "id"
        ]
    ): int(
        application_row[
            "raw_submission_id"
        ]
    )
    for _, application_row
    in valid_application_dataframe.iterrows()
}

# 建立 raw_submission_id 到 resume row index mapping。
resume_index_by_raw_submission_id = {
    int(
        raw_submission_id
    ): dataframe_index
    for dataframe_index, raw_submission_id
    in resume_extraction_dataframe[
        "raw_submission_id"
    ].items()
}


# ============================================================
# 验证 D1 schema、唯一索引和 Application/Candidate mapping
# ============================================================

# 查询当前 schema。
required_table_columns = {
    "Person_Skill": {
        "id",
        "person_id",
        "skill_id",
        "proficiency_level",
        "proficiency_level_id",
        "years_experience",
        "recorded_at",
        "source_candidate_id",
        "created_at",
        "updated_at",
    },
    "Candidate_Skill": {
        "id",
        "candidate_id",
        "person_skill_id",
        "created_at",
    },
}

# 逐表验证 columns。
for table_name, expected_columns in (
    required_table_columns.items()
):
    # 查询 table info。
    table_info_result = execute_d1(
        f'PRAGMA table_info("{table_name}");'
    )

    # 保存实际 columns。
    actual_columns = {
        table_info_row[
            "name"
        ]
        for table_info_row
        in result_rows(
            table_info_result
        )
    }

    # 找出缺失 columns。
    missing_table_columns = (
        expected_columns
        - actual_columns
    )

    # 缺失时停止。
    if missing_table_columns:
        raise ValueError(
            f"{table_name} 缺少必要 columns："
            f"{sorted(missing_table_columns)}"
        )

# 保存两个必要唯一索引。
required_unique_indexes = {
    "Person_Skill": (
        "idx_Person_Skill_0_person_id_skill_id"
    ),
    "Candidate_Skill": (
        "idx_Candidate_Skill_0_candidate_id_person_skill_id"
    ),
}

# 验证唯一索引。
for table_name, index_name in (
    required_unique_indexes.items()
):
    # 查询 index list。
    index_list_result = execute_d1(
        f'PRAGMA index_list("{table_name}");'
    )

    # 找目标 index。
    matching_index_rows = [
        index_row
        for index_row
        in result_rows(
            index_list_result
        )
        if index_row.get(
            "name"
        )
        == index_name
    ]

    # 必须存在且 unique = 1。
    if (
        len(
            matching_index_rows
        )
        != 1
        or int(
            matching_index_rows[0].get(
                "unique",
                0,
            )
        )
        != 1
    ):
        raise ValueError(
            f"{table_name} 缺少必要唯一索引："
            f"{index_name}"
        )

# 查询 D1 真实 mapping。
d1_mapping_result = execute_d1(
    """
    SELECT
      app."id" AS application_id,
      app."person_id",
      cs."id" AS candidate_snapshot_id,
      cs."application_date"
    FROM "Application" AS app
    INNER JOIN "Candidate_snapshot" AS cs
      ON cs."application_id" = app."id"
    ORDER BY app."id";
    """
)

# 提取 mapping rows。
d1_mapping_rows = result_rows(
    d1_mapping_result
)

# 建立 Application mapping。
d1_mapping_by_application_id = {
    int(
        mapping_row[
            "application_id"
        ]
    ): {
        "person_id": int(
            mapping_row[
                "person_id"
            ]
        ),
        "candidate_snapshot_id": int(
            mapping_row[
                "candidate_snapshot_id"
            ]
        ),
        "application_date": clean_text_value(
            mapping_row[
                "application_date"
            ]
        ),
    }
    for mapping_row
    in d1_mapping_rows
}

# 当前 application.csv 可以只包含本次新增批次。
# 这里只要求当前 CSV 中的 Application.id 都已经存在于 D1。
current_csv_application_ids = set(
    raw_submission_id_by_application_id
)

# 查找当前 CSV 中存在、但 D1 中不存在的 Application.id。
application_ids_missing_from_d1 = sorted(
    current_csv_application_ids
    - set(
        d1_mapping_by_application_id
    )
)

# 当前批次存在无法映射的 Application 时，在任何 Skill 写入前停止。
if application_ids_missing_from_d1:
    raise ValueError(
        "当前 application.csv 中存在无法映射到 "
        "D1 Application/Candidate_snapshot 的记录："
        f"{application_ids_missing_from_d1[:20]}"
    )

# 只保留当前 application.csv 对应的 D1 mapping。
# 这样首批导入和以后任意大小的增量批次使用完全相同的逻辑，
# 也不会重新处理历史 Application 的 skills。
d1_mapping_by_application_id = {
    int(application_id): (
        d1_mapping_by_application_id[
            int(application_id)
        ]
    )
    for application_id
    in sorted(
        current_csv_application_ids
    )
}

# 显示当前批次映射验证结果。
print(
    "Candidate skill Application mapping validation:",
    {
        "current_csv_application_count": int(
            len(current_csv_application_ids)
        ),
        "current_d1_mapping_count": int(
            len(d1_mapping_by_application_id)
        ),
        "current_csv_ids_missing_from_d1_count": int(
            len(application_ids_missing_from_d1)
        ),
    },
)

# 每个 Application 的 raw submission 都必须在 extraction CSV。
missing_resume_raw_ids = sorted(
    set(
        raw_submission_id_by_application_id.values()
    )
    - set(
        resume_index_by_raw_submission_id
    )
)

# 缺失时停止。
if missing_resume_raw_ids:
    raise ValueError(
        "以下 Application raw_submission_id "
        "在 resume extraction CSV 中不存在："
        f"{missing_resume_raw_ids[:20]}"
    )


# ============================================================
# 加载 D1 Skill 与 Skill_type，并建立一对多 lookup
# ============================================================

# 查询全部 Skill 和类型名称。
skill_result = execute_d1(
    """
    SELECT
      s."id" AS skill_id,
      s."skill_name",
      s."skill_type_id",
      st."skill_type_name"
    FROM "Skill" AS s
    INNER JOIN "Skill_type" AS st
      ON st."id" = s."skill_type_id"
    ORDER BY s."id";
    """
)

# 提取 Skill rows。
skill_rows = result_rows(
    skill_result
)

# Skill 不能为空。
if not skill_rows:
    raise ValueError(
        "D1 Skill 表没有任何记录。"
    )

# 保存 skill name key 对应的所有类型候选。
skill_candidates_by_name_key = {}

# 逐条建立 lookup。
for skill_row in skill_rows:
    # 清理数据库中的 Skill 显示名称。
    database_skill_name = clean_text_value(
        skill_row[
            "skill_name"
        ]
    )

    # Skill.skill_name 为空时停止，因为无法用作匹配关键词。
    if database_skill_name is None:
        raise ValueError(
            "D1 Skill 存在空 skill_name："
            f"skill_id={skill_row['skill_id']}"
        )

    # 生成完整技能名称匹配 key。
    skill_name_key = normalize_skill_match_key(
        database_skill_name
    )

    # 保存候选。
    skill_candidates_by_name_key.setdefault(
        skill_name_key,
        [],
    ).append(
        {
            "skill_id": int(
                skill_row[
                    "skill_id"
                ]
            ),
            "skill_name": database_skill_name,
            "skill_type_id": int(
                skill_row[
                    "skill_type_id"
                ]
            ),
            "skill_type_name": clean_text_value(
                skill_row[
                    "skill_type_name"
                ]
            ),
        }
    )

def resolve_skill_candidates(
    raw_skill_name,
):
    """
    返回数据库中与一个摘要技能名称完全匹配的全部 Skill records。

    同一个技能属于多个 Skill type 时，不选择优先类型，
    而是保留每个不同类型对应的 Skill record。
    """

    # 生成 lookup key。
    skill_name_key = normalize_skill_match_key(
        raw_skill_name
    )

    # 读取全部同名候选。
    candidates = (
        skill_candidates_by_name_key.get(
            skill_name_key,
            [],
        )
    )

    # 完全找不到时返回空 list。
    if not candidates:
        return []

    # 保存每一个 skill_type_id 下的候选。
    candidates_by_type_id = {}

    # 按 skill_type_id 分组。
    for candidate in candidates:
        candidates_by_type_id.setdefault(
            candidate[
                "skill_type_id"
            ],
            [],
        ).append(
            candidate
        )

    # 同名、同一个 type 下出现多条记录时不能猜测使用哪个 Skill.id。
    duplicate_type_records = [
        {
            "skill_type_id": skill_type_id,
            "skill_type_name": (
                type_candidates[0][
                    "skill_type_name"
                ]
            ),
            "skill_ids": [
                type_candidate[
                    "skill_id"
                ]
                for type_candidate
                in type_candidates
            ],
        }
        for skill_type_id, type_candidates
        in candidates_by_type_id.items()
        if len(
            type_candidates
        ) > 1
    ]

    # 发现重复时停止。
    if duplicate_type_records:
        raise ValueError(
            "D1 Skill 存在同名同类型重复记录："
            f"skill={raw_skill_name}, "
            f"details={duplicate_type_records}"
        )

    # 按 type id 和 Skill id 排序，保证相同输入每次得到相同顺序。
    sorted_candidates = sorted(
        candidates,
        key=lambda candidate: (
            candidate[
                "skill_type_id"
            ],
            candidate[
                "skill_id"
            ],
        ),
    )

    # 返回全部不同 Skill type 对应的记录。
    return sorted_candidates


# ============================================================
# 写入前验证所有 Application 范围内的摘要技能
# ============================================================

# 保存 Application 范围内出现过的唯一摘要技能。
unique_resume_skill_names_by_key = {}

# 逐个 Application 收集技能名称。
for application_id in sorted(
    raw_submission_id_by_application_id
):
    # 取得对应 raw_submission_id。
    raw_submission_id = (
        raw_submission_id_by_application_id[
            application_id
        ]
    )

    # 找到 resume extraction CSV row。
    resume_dataframe_index = (
        resume_index_by_raw_submission_id[
            raw_submission_id
        ]
    )

    # 拆分当前简历中的技能摘要。
    current_resume_skill_names = split_skill_summary(
        resume_extraction_dataframe.at[
            resume_dataframe_index,
            "resume_skills_matched_summary",
        ]
    )

    # 保存唯一技能名称。
    for current_resume_skill_name in (
        current_resume_skill_names
    ):
        # 生成稳定匹配 key。
        current_skill_key = normalize_skill_match_key(
            current_resume_skill_name
        )

        # 第一次出现时保存原始显示名称。
        unique_resume_skill_names_by_key.setdefault(
            current_skill_key,
            current_resume_skill_name,
        )

# 保存完全无法在 D1 Skill 中找到的技能。
unmatched_resume_skill_names = []

# 保存匹配到多个 Skill type 的技能及类型。
multi_type_skill_matches = {}

# 在任何写入开始前验证所有唯一技能。
for (
    current_skill_key,
    current_resume_skill_name,
) in sorted(
    unique_resume_skill_names_by_key.items()
):
    # 取得全部同名 Skill records。
    resolved_skills = resolve_skill_candidates(
        current_resume_skill_name
    )

    # 找不到时保存到异常清单。
    if not resolved_skills:
        unmatched_resume_skill_names.append(
            current_resume_skill_name
        )
        continue

    # 同名技能匹配到多个类型时保存审计信息。
    if len(
        resolved_skills
    ) > 1:
        multi_type_skill_matches[
            current_resume_skill_name
        ] = [
            resolved_skill[
                "skill_type_name"
            ]
            for resolved_skill
            in resolved_skills
        ]

# 任一摘要技能找不到时，在写数据库之前整体停止。
if unmatched_resume_skill_names:
    raise ValueError(
        "以下 resume_skills_matched_summary 技能"
        "无法在 D1 Skill 中找到完整名称匹配："
        f"{unmatched_resume_skill_names}"
    )

# 显示写入前匹配计划。
print(
    "Skill matching plan:",
    {
        "unique_resume_skill_name_count": int(
            len(
                unique_resume_skill_names_by_key
            )
        ),
        "multi_type_skill_name_count": int(
            len(
                multi_type_skill_matches
            )
        ),
        "unmatched_skill_name_count": int(
            len(
                unmatched_resume_skill_names
            )
        ),
    },
)

# 显示部分跨类型技能例子，方便人工确认匹配逻辑。
print(
    "Multi-type skill examples:",
    dict(
        list(
            multi_type_skill_matches.items()
        )[:20]
    ),
)


# ============================================================
# 幂等写入 Person_Skill 和 Candidate_Skill
# ============================================================

# 保存统一 timestamp。
now_text = utc_now_text()

# 保存每个 raw submission 的结果。
skill_result_by_raw_submission_id = {}

# 保存统计。
total_skill_mentions = 0
successful_skill_mentions = 0
failed_skill_mentions = 0

# 按 Application.id 稳定处理。
for application_index, application_id in enumerate(
    sorted(
        d1_mapping_by_application_id
    ),
    start=1,
):
    # 读取 D1 mapping。
    d1_mapping = (
        d1_mapping_by_application_id[
            application_id
        ]
    )

    # 读取 ids。
    person_id = d1_mapping[
        "person_id"
    ]

    # 读取 Candidate id。
    candidate_snapshot_id = d1_mapping[
        "candidate_snapshot_id"
    ]

    # 读取 Application date。
    application_date = d1_mapping[
        "application_date"
    ]

    # 读取 raw submission id。
    raw_submission_id = (
        raw_submission_id_by_application_id[
            application_id
        ]
    )

    # 找到 resume CSV row。
    resume_dataframe_index = (
        resume_index_by_raw_submission_id[
            raw_submission_id
        ]
    )

    # 拆分技能。
    raw_skill_names = split_skill_summary(
        resume_extraction_dataframe.at[
            resume_dataframe_index,
            "resume_skills_matched_summary",
        ]
    )

    # 保存当前 raw submission 的结果 lists。
    resolved_skill_names = []
    resolved_skill_type_names = []
    skill_ids = []
    person_skill_ids = []
    candidate_skill_ids = []
    import_errors = []

    # 没有 skills 时标记 rejected。
    if not raw_skill_names:
        skill_result_by_raw_submission_id[
            raw_submission_id
        ] = {
            "candidate_snapshot_id": (
                candidate_snapshot_id
            ),
            "skill_import_status": (
                "rejected"
            ),
            "skill_import_error": None,
            "resolved_skill_names_summary": None,
            "resolved_skill_type_names_summary": (
                None
            ),
            "skill_ids_summary": None,
            "person_skill_ids_summary": None,
            "candidate_skill_ids_summary": None,
        }
        continue

    # 逐个摘要技能进行匹配。
    for raw_skill_name in raw_skill_names:
        # 增加原始摘要技能数量。
        total_skill_mentions += 1

        try:
            # 取得数据库中全部同名、不同 Skill type 的 records。
            resolved_skills = resolve_skill_candidates(
                raw_skill_name
            )

            # 完全找不到时，把当前摘要技能记为失败。
            if not resolved_skills:
                raise ValueError(
                    "D1 Skill 找不到完整名称匹配："
                    f"{raw_skill_name}"
                )

            # 同一个摘要技能可以展开成多个不同 type 的 Skill records。
            for resolved_skill in resolved_skills:
                # 读取当前数据库 Skill id。
                skill_id = resolved_skill[
                    "skill_id"
                ]

                # 查询这个 Person 是否已经关联当前 Skill record。
                person_skill_result = execute_d1(
                    """
                    SELECT "id"
                    FROM "Person_Skill"
                    WHERE "person_id" = ?
                      AND "skill_id" = ?
                    ORDER BY "id";
                    """,
                    [
                        person_id,
                        skill_id,
                    ],
                )

                # 提取 Person_Skill rows。
                person_skill_rows = result_rows(
                    person_skill_result
                )

                # 唯一索引存在时不应超过一条。
                if len(
                    person_skill_rows
                ) > 1:
                    raise RuntimeError(
                        "Person_Skill 存在重复记录："
                        f"person_id={person_id}, "
                        f"skill_id={skill_id}"
                    )

                # 已存在时直接复用 Person_Skill.id。
                if person_skill_rows:
                    person_skill_id = int(
                        person_skill_rows[0][
                            "id"
                        ]
                    )
                else:
                    # 不存在时新建 Person_Skill。
                    person_skill_insert_result = (
                        execute_d1(
                            """
                            INSERT INTO "Person_Skill" (
                              "person_id",
                              "skill_id",
                              "proficiency_level",
                              "proficiency_level_id",
                              "years_experience",
                              "recorded_at",
                              "source_candidate_id",
                              "created_at",
                              "updated_at"
                            )
                            VALUES (?, ?, NULL, NULL, NULL, ?, ?, ?, ?)
                            RETURNING "id";
                            """,
                            [
                                person_id,
                                skill_id,
                                application_date,
                                candidate_snapshot_id,
                                now_text,
                                now_text,
                            ],
                        )
                    )

                    # 读取新建的 Person_Skill.id。
                    person_skill_id = int(
                        result_rows(
                            person_skill_insert_result
                        )[0][
                            "id"
                        ]
                    )

                # 查询当前 Candidate 是否已经关联这个 Person_Skill。
                candidate_skill_result = execute_d1(
                    """
                    SELECT "id"
                    FROM "Candidate_Skill"
                    WHERE "candidate_id" = ?
                      AND "person_skill_id" = ?
                    ORDER BY "id";
                    """,
                    [
                        candidate_snapshot_id,
                        person_skill_id,
                    ],
                )

                # 提取 Candidate_Skill rows。
                candidate_skill_rows = result_rows(
                    candidate_skill_result
                )

                # 唯一索引存在时不应超过一条。
                if len(
                    candidate_skill_rows
                ) > 1:
                    raise RuntimeError(
                        "Candidate_Skill 存在重复记录："
                        f"candidate_id={candidate_snapshot_id}, "
                        f"person_skill_id={person_skill_id}"
                    )

                # 已存在时直接复用 Candidate_Skill.id。
                if candidate_skill_rows:
                    candidate_skill_id = int(
                        candidate_skill_rows[0][
                            "id"
                        ]
                    )
                else:
                    # 不存在时新建 Candidate_Skill。
                    candidate_skill_insert_result = (
                        execute_d1(
                            """
                            INSERT INTO "Candidate_Skill" (
                              "candidate_id",
                              "person_skill_id",
                              "created_at"
                            )
                            VALUES (?, ?, ?)
                            RETURNING "id";
                            """,
                            [
                                candidate_snapshot_id,
                                person_skill_id,
                                now_text,
                            ],
                        )
                    )

                    # 读取新建的 Candidate_Skill.id。
                    candidate_skill_id = int(
                        result_rows(
                            candidate_skill_insert_result
                        )[0][
                            "id"
                        ]
                    )

                # 保存数据库 Skill 显示名称。
                resolved_skill_names.append(
                    resolved_skill[
                        "skill_name"
                    ]
                )

                # 保存这个 Skill record 所属的类型。
                resolved_skill_type_names.append(
                    resolved_skill[
                        "skill_type_name"
                    ]
                )

                # 保存 Skill.id。
                skill_ids.append(
                    str(
                        skill_id
                    )
                )

                # 保存 Person_Skill.id。
                person_skill_ids.append(
                    str(
                        person_skill_id
                    )
                )

                # 保存 Candidate_Skill.id。
                candidate_skill_ids.append(
                    str(
                        candidate_skill_id
                    )
                )

                # 每成功建立或复用一个数据库 Skill 关联就增加一次。
                successful_skill_mentions += 1

        except Exception as error:
            # 保存当前摘要技能的具体错误。
            import_errors.append(
                f"{raw_skill_name}: {error}"
            )

            # 增加失败的摘要技能数量。
            failed_skill_mentions += 1

    # 当前 Candidate 只要有任一失败就标记 failed。
    if import_errors:
        skill_import_status = "failed"
    else:
        # 全部技能成功。
        skill_import_status = "success"

    # 保存 raw submission 级结果。
    skill_result_by_raw_submission_id[
        raw_submission_id
    ] = {
        "candidate_snapshot_id": (
            candidate_snapshot_id
        ),
        "skill_import_status": (
            skill_import_status
        ),
        "skill_import_error": (
            " | ".join(
                import_errors
            )
            if import_errors
            else None
        ),
        "resolved_skill_names_summary": (
            "; ".join(
                resolved_skill_names
            )
            if resolved_skill_names
            else None
        ),
        "resolved_skill_type_names_summary": (
            "; ".join(
                resolved_skill_type_names
            )
            if resolved_skill_type_names
            else None
        ),
        "skill_ids_summary": (
            "; ".join(
                skill_ids
            )
            if skill_ids
            else None
        ),
        "person_skill_ids_summary": (
            "; ".join(
                person_skill_ids
            )
            if person_skill_ids
            else None
        ),
        "candidate_skill_ids_summary": (
            "; ".join(
                candidate_skill_ids
            )
            if candidate_skill_ids
            else None
        ),
    }

    # 每100个显示进度，并显示最后一个。
    if (
        application_index % 100 == 0
        or application_index
        == len(
            d1_mapping_by_application_id
        )
    ):
        print(
            "Processed Candidate skills:",
            f"{application_index}/"
            f"{len(d1_mapping_by_application_id)}",
        )


# ============================================================
# 覆盖更新原 resume extraction CSV
# ============================================================

# 每次运行重新生成这些结果 columns。
result_column_names = [
    "candidate_snapshot_id",
    "skill_import_status",
    "skill_import_error",
    "resolved_skill_names_summary",
    "resolved_skill_type_names_summary",
    "skill_ids_summary",
    "person_skill_ids_summary",
    "candidate_skill_ids_summary",
]

# 初始化为空。
for result_column_name in (
    result_column_names
):
    resume_extraction_dataframe[
        result_column_name
    ] = pd.NA

# 逐个 raw submission 回填。
for raw_submission_id, skill_import_result in (
    skill_result_by_raw_submission_id.items()
):
    # 找原始 row。
    dataframe_index = (
        resume_index_by_raw_submission_id[
            raw_submission_id
        ]
    )

    # 回填所有结果字段。
    for result_column_name in (
        result_column_names
    ):
        resume_extraction_dataframe.at[
            dataframe_index,
            result_column_name,
        ] = skill_import_result.get(
            result_column_name
        )

# 使用临时文件降低写坏原 CSV 的风险。
resume_temp_path = (
    f"{RESUME_EXTRACTION_CSV_PATH}.tmp"
)

# 写入临时 CSV。
resume_extraction_dataframe.to_csv(
    resume_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 替换原 CSV。
os.replace(
    resume_temp_path,
    RESUME_EXTRACTION_CSV_PATH,
)


# ============================================================
# 验证数据库和输出结果
# ============================================================

# 外键检查。
foreign_key_result = execute_d1(
    "PRAGMA foreign_key_check;"
)

# 提取 violations。
foreign_key_rows = result_rows(
    foreign_key_result
)

# 有 violation 时停止。
if foreign_key_rows:
    raise RuntimeError(
        "D1 存在 foreign key violation："
        f"{foreign_key_rows[:20]}"
    )

# 统计状态。
application_raw_ids = set(
    raw_submission_id_by_application_id.values()
)

# 只看 Application 范围。
application_resume_dataframe = (
    resume_extraction_dataframe.loc[
        resume_extraction_dataframe[
            "raw_submission_id"
        ].isin(
            application_raw_ids
        )
    ]
)

# 显示 summary。
print(
    "Candidate skill import summary:",
    {
        "application_candidate_count": int(
            len(
                application_resume_dataframe
            )
        ),
        "candidate_with_skill_count": int(
            (
                application_resume_dataframe[
                    "skill_import_status"
                ]
                == "success"
            ).sum()
        ),
        "candidate_without_skill_count": int(
            (
                application_resume_dataframe[
                    "skill_import_status"
                ]
                == "rejected"
            ).sum()
        ),
        "candidate_failed_count": int(
            (
                application_resume_dataframe[
                    "skill_import_status"
                ]
                == "failed"
            ).sum()
        ),
        "skill_mention_count": int(
            total_skill_mentions
        ),
        "successful_database_skill_relationship_count": int(
            successful_skill_mentions
        ),
        "failed_resume_skill_name_count": int(
            failed_skill_mentions
        ),
        "multi_type_skill_name_count": int(
            len(
                multi_type_skill_matches
            )
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_rows
            )
        ),
    },
)

# 显示更新路径。
print(
    "Updated resume extraction CSV:",
    RESUME_EXTRACTION_CSV_PATH,
)

# 显示 Application 范围预览。
display(
    application_resume_dataframe[
        [
            "raw_submission_id",
            "candidate_snapshot_id",
            "resume_skills_matched_summary",
            "resolved_skill_names_summary",
            "resolved_skill_type_names_summary",
            "skill_ids_summary",
            "person_skill_ids_summary",
            "candidate_skill_ids_summary",
            "skill_import_status",
            "skill_import_error",
        ]
    ].head(
        20
    )
)

# 有失败时在保存 CSV 后停止。
if failed_skill_mentions > 0:
    raise RuntimeError(
        "部分 Candidate skill 写入失败。"
        "具体错误已经保存到 resume extraction CSV。"
    )

# 显示完成。
print(
    "Candidate skill import completed successfully."
)


### 简历工作经历数据提取生成csv + 存入 Google Drive

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 从 D1 Candidate_snapshot.raw_resume_text 中
# 使用确定性规则提取简历工作经历，并覆盖生成两个 CSV。
#
# 输出：
# 1. candidate_employment_rule_extraction.csv
#    每一行代表一段从原始简历中提取出的工作经历。
#
# 2. candidate_employment_rule_extraction_status.csv
#    每一行代表一个 Candidate 的工作经历提取状态。
#
# 重要：
# 1. 本 Cell 不依赖其他 Colab Cell 的 variable、function 或 class。
# 2. 本 Cell 不使用模型。
# 3. 本 Cell 只读取 D1，不写入任何数据库表。
# 4. 本 Cell 暂时不写 Person_Position 或 Candidate_Position。
# 5. 只处理原始简历中能够识别出明确 employment section 的内容。
# 6. 不使用 resume_extraction_consolidated.csv 的 summary fallback。
# 7. 开始日期或结束日期至少必须可靠取得一个，否则不生成正式record。
# 8. 固定覆盖同名 CSV，因此重复运行不会无限增加文件。
# 9. 相同 D1 数据和相同规则会产生相同排序和相同结果。
# ============================================================

# os 用于创建 Google Drive 输出目录。
import os

# re 用于确定性文本规则和日期匹配。
import re

# time 只用于 D1 网络请求失败后的等待。
import time

# pandas 用于生成结构化结果和保存 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定规则版本和 Google Drive 路径
# ============================================================

# 保存当前工作经历规则版本。
RULE_VERSION = (
    "candidate_employment_rule_v5"
)

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 确保输出目录存在。
os.makedirs(
    DATASET_FOLDER,
    exist_ok=True,
)

# 当前批次已经成功生成 Application records 的固定 CSV。
# 本 Cell 使用其中的 Application.id 限定本次需要解析的 Candidate。
APPLICATION_CSV_PATH = (
    f"{DATASET_FOLDER}/dedup_results/"
    "application_dedup_v2/application.csv"
)

# 当前批次 Application CSV 不存在时停止。
if not os.path.exists(
    APPLICATION_CSV_PATH
):
    raise FileNotFoundError(
        "找不到必要的 application.csv："
        f"{APPLICATION_CSV_PATH}"
    )

# 保存记录级提取结果。
EMPLOYMENT_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_employment_rule_extraction.csv"
)

# 保存 Candidate 级提取状态。
EMPLOYMENT_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_employment_rule_extraction_status.csv"
)


# ============================================================
# Cloudflare D1 独立连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 API endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 D1 request headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP connection。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行只读 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有重试机会时等待。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 重试用完后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 SQL 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 Cloudflare 或 SQL 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部失败后停止。
    raise RuntimeError(
        "D1 query 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    d1_result,
):
    """从 D1 result 中提取 results rows。"""

    # 没有 result 时返回空 list。
    if not d1_result:
        return []

    # 返回第一条 statement 的 results。
    return d1_result[0].get(
        "results",
        [],
    )


def clean_text_value(
    value,
):
    """把 pandas 空值和空字符串统一转换成 None。"""

    # pandas NA。
    if pd.isna(
        value
    ):
        return None

    # 去掉首尾空白。
    cleaned_value = str(
        value
    ).strip()

    # 空字符串和常见空值文本统一为 None。
    if (
        not cleaned_value
        or cleaned_value.lower()
        in {
            "nan",
            "none",
            "null",
        }
    ):
        return None

    # 返回清理结果。
    return cleaned_value


def normalize_lower_text(
    value,
):
    """统一空白并转换成小写文本。"""

    # 清理输入。
    cleaned_value = clean_text_value(
        value
    )

    # 空值保持 None。
    if cleaned_value is None:
        return None

    # 合并重复空白并小写化。
    return re.sub(
        r"\s+",
        " ",
        cleaned_value,
    ).strip().lower()


# ============================================================
# Employment section 与停止 section 规则
# ============================================================

# 只把明确的工作、实习或研究经历标题视为 employment section。
EMPLOYMENT_HEADER_PATTERN = re.compile(
    r"^\s*(?:"
    r"work\s+experiences?|professional\s+experiences?|"
    r"employment(?:\s+history)?|work\s+history|career\s+history|"
    r"experiences?|internship\s+experiences?|internships?|"
    r"research\s+experiences?|industry\s+experiences?|"
    r"professional\s+background|work\s*&\s*leadership\s+experience"
    r")\s*[:：]?\s*$",
    flags=re.IGNORECASE,
)

# 命中其他明确 section 时结束 employment section。
STOP_SECTION_PATTERN = re.compile(
    r"^\s*(?:"
    r"education(?:al)?(?:\s+background)?|academic\s+background|"
    r"education\s*(?:&|and)\s*certifications?|"
    r"academic\s+qualifications?|"
    r"projects?|project\s+experiences?|academic\s+projects?|"
    r"research\s+projects?|selected\s+projects?|personal\s+projects?|"
    r"research\s*(?:&|and)\s*project\s+experience|"
    r"skills?|technical\s+skills|skill\s+sets?|"
    r"skills?\s*(?:&|and|/)\s*(?:interests?|activities|hobbies)|"
    r"languages?|certifications?|courseworks?|relevant\s+courseworks?|"
    r"awards?|awards?\s*(?:&|and)\s*contests?|"
    r"honors?(?:\s*(?:&|and)\s*awards?)?|publications?|"
    r"leadership(?:\s+experience)?|"
    r"leadership\s*(?:&|and)\s*project\s+experience|"
    r"leadership\s*(?:&|and)\s*involvement|"
    r"leadership\s*(?:&|and)\s*activities|"
    r"leadership\s*(?:&|and)\s*community\s+involvement|"
    r"activities\s*(?:&|and)\s*leadership(?:\s+experience)?|"
    r"extra[- ]?curricular(?:\s+activities|\s+experience|\s+experiences)?|"
    r"activities|student\s+activities|campus\s+involvement|"
    r"community\s+involvement|volunteer(?:ing)?(?:\s+experience)?|"
    r"clubs?\s*(?:&|and)\s*organizations?|"
    r"additional\s+experience|additional\s+information|"
    r"interests?|summary|professional\s+summary|objective|references?"
    r")\s*[:：]?\s*$",
    flags=re.IGNORECASE,
)

# 常见 bullet 开头。
BULLET_PREFIX_PATTERN = re.compile(
    r"^\s*(?:[-•●▪◦‣·\uf0b7]|\d+[.)])\s*"
)

# 英文月份。
MONTH_NUMBER_BY_NAME = {
    "jan": 1,
    "january": 1,
    "feb": 2,
    "february": 2,
    "mar": 3,
    "march": 3,
    "apr": 4,
    "april": 4,
    "may": 5,
    "jun": 6,
    "june": 6,
    "jul": 7,
    "july": 7,
    "aug": 8,
    "august": 8,
    "sep": 9,
    "sept": 9,
    "september": 9,
    "oct": 10,
    "october": 10,
    "nov": 11,
    "november": 11,
    "dec": 12,
    "december": 12,
}

# 单个日期 token。
DATE_TOKEN_TEXT = (
    r"(?:"
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|"
    r"Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\.?\s+\d{4}|"
    r"\d{1,2}[./-]\d{4}|"
    r"\d{4}[./-]\d{1,2}|"
    r"\d{4}|"
    r"Present|Current|Now"
    r")"
)

# 匹配包含两个日期端点的日期范围。
DATE_RANGE_PATTERN = re.compile(
    rf"(?P<start>{DATE_TOKEN_TEXT})"
    rf"\s*(?:-|–|—|~|\bto\b)\s*"
    rf"(?P<end>{DATE_TOKEN_TEXT})",
    flags=re.IGNORECASE,
)

# 匹配日期范围，或者只有一个日期端点的情况。
#
# 日期范围放在前面，避免只截取范围中的第一个日期。
EMPLOYMENT_DATE_PATTERN = re.compile(
    rf"(?:"
    rf"(?P<range_start>{DATE_TOKEN_TEXT})"
    rf"\s*(?:-|–|—|~|\bto\b)\s*"
    rf"(?P<range_end>{DATE_TOKEN_TEXT})|"
    rf"(?P<single_date>{DATE_TOKEN_TEXT})"
    rf")",
    flags=re.IGNORECASE,
)

# 明确的职位关键词。
POSITION_KEYWORD_PATTERN = re.compile(
    r"\b(?:"
    r"intern|analyst|associate|assistant|engineer|developer|"
    r"consultant|manager|director|researcher|scientist|"
    r"coordinator|specialist|designer|accountant|advisor|"
    r"officer|president|founder|co[- ]?founder|lead|head|"
    r"professor|teacher|tutor|trainee|fellow"
    r")\b",
    flags=re.IGNORECASE,
)

# 常见组织名称关键词。
COMPANY_KEYWORD_PATTERN = re.compile(
    r"\b(?:"
    r"inc\.?|llc|ltd\.?|limited|corp\.?|corporation|company|"
    r"group|capital|bank|university|college|institute|laboratory|lab|"
    r"foundation|partners?|consulting|technologies|technology|"
    r"systems|solutions|ventures?|holdings|school|hospital|"
    r"association|organization"
    r")\b",
    flags=re.IGNORECASE,
)

# 常见工作描述动作词。
#
# 这些行可能恰好位于下一条日期之前，
# 但不能因此被当作新工作经历的header。
DESCRIPTION_ACTION_PATTERN = re.compile(
    r"^(?:"
    r"led|built|developed|created|designed|implemented|conducted|"
    r"analyzed|analysed|managed|worked|collaborated|assisted|"
    r"utilized|used|performed|generated|identified|researched|"
    r"supported|engineered|trained|deployed|achieved|improved|"
    r"prepared|provided|participated|coordinated|contributed|"
    r"converted|drafted|complied|provide|co-designed|"
    r"data\s+analysis|"
    r"reformulating|building|using|developing|designing|analyzing|"
    r"analysing|managing|supporting|performing|preparing|creating"
    r")\b",
    flags=re.IGNORECASE,
)

# 常见 location 尾部，解析公司和职位前先尽力移除。
LOCATION_SUFFIX_PATTERN = re.compile(
    r"\s*[,|]\s*"
    r"(?:[A-Z][A-Za-z .'-]+,\s*)?"
    r"(?:[A-Z]{2}|United States|USA|China|Canada|Remote)"
    r"\s*$",
    flags=re.IGNORECASE,
)

# 单独一行或一个布局区域中的地点不能作为公司或岗位。
#
# 这里包含当前数据集中已经实际出现的主要城市，
# 同时支持 City, ST 和 City, Country 格式。
LOCATION_ONLY_PATTERN = re.compile(
    r"^(?:"
    r"remote|online|hybrid|on[- ]site|"
    r"new\s+york|beijing|shanghai|shenzhen|boston|"
    r"richmond|vancouver|kunming|decatur|jersey\s+city|"
    r"atlanta|chicago|seattle|austin|houston|dallas|"
    r"san\s+francisco|los\s+angeles|washington(?:,\s*d\.?c\.?)?|"
    r"[A-Za-z .'-]+,\s*[A-Z]{2}|"
    r"[A-Za-z .'-]+,\s*(?:USA|United States|China|Canada|UK)"
    r")$",
    flags=re.IGNORECASE,
)

# 单独月份或月份缩写不能作为公司或岗位。
MONTH_ONLY_PATTERN = re.compile(
    r"^(?:"
    r"jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|"
    r"jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|"
    r"oct(?:ober)?|nov(?:ember)?|dec(?:ember)?"
    r")\.?$",
    flags=re.IGNORECASE,
)

# 单独季节不能作为公司或岗位。
SEASON_ONLY_PATTERN = re.compile(
    r"^(?:spring|summer|fall|autumn|winter)$",
    flags=re.IGNORECASE,
)

# Employment section 内再次出现的其他 section 标题，
# 即使因为PDF排版没有被前面的section扫描及时截断，
# 也不能作为公司或岗位。
NON_EMPLOYMENT_HEADER_PATTERN = re.compile(
    r"^(?:"
    r"education|projects?|skills?|activities|interests?|"
    r"awards?|contests?|leadership|involvement|"
    r"extra[- ]?curricular|additional\s+experience|"
    r"additional\s+information|courseworks?|certifications?"
    r")(?:\s|&|/|,|;|:|$)",
    flags=re.IGNORECASE,
)

# 明确不是岗位名称的描述、指导教师或数量句。
#
# 这些文本即使包含 researcher、tutor、fellow 等岗位关键词，
# 也不能仅凭关键词被保留成岗位名称。
NON_POSITION_DESCRIPTION_PATTERN = re.compile(
    r"^(?:"
    r"instructed\s+by|"
    r"(?:[a-z]\s+)?working\s+as|"
    r"cluster(?:'s|’s)\s+computing\s+resources|"
    r"\d[\d,]*\s+(?:teachers?|students?|employees?|volunteers?|members?)\b"
    r")",
    flags=re.IGNORECASE,
)

# 明确不是公司名称的工作描述句或工作成果句。
#
# 不能使用单词数量直接删除较长公司名称，因为真实机构名称也可能很长，
# 例如 Chinese Academy of Medical Sciences & Peking Union Medical College。
#
# 这里只排除当前结果中实际出现的描述句开头和常见动作表达。
NON_COMPANY_DESCRIPTION_PATTERN = re.compile(
    r"^[!•●▪◦‣·\uf0b7]*\s*(?:"
    r"made|collect(?:ed|ing)|analy[sz](?:e|ed|ing)|"
    r"support(?:ed|ing)?|wrote|write|written|"
    r"utilized|utilised|using|used|"
    r"innovated|served|launched|schemed|maintained|"
    r"evaluation|sustainability|costs?|"
    r"well\s+as|firm\s+and|game[- ]planning|"
    r"differentiation\s+and|"
    r"meet\s+weekly|"
    r"tableau\s+to"
    r")\b",
    flags=re.IGNORECASE,
)

# 明确不是公司名称的 section、课程、奖项或工具列表。
#
# 这些规则只处理当前 v4 结果中已经确认的错误类型，
# 不因为普通公司名称较长、包含 project 或包含 research 就统一删除。
NON_COMPANY_METADATA_PATTERN = re.compile(
    r"^(?:"
    r"research\s*(?:&|and)\s*project\s+experience|"
    r"technologies?\s*(?:&|and)\s*tools?\s*:|"
    r"group\s+project|university\s+joint\s+project|"
    r"(?:[A-Z]{2,5}\s*)?\d{2,4}\s*[-:]\s*.+|"
    r"(?:first|second|third)\s+prize\b|"
    r"the\s+last\s+\d+\s+honou?r\b|"
    r"\$[\d,]+(?:\.\d+)?\)?\s*,"
    r")",
    flags=re.IGNORECASE,
)

# 明确属于分析、研究、比赛或课程项目标题的公司候选。
#
# 这里只匹配标题开头的具体结构。
# 合法机构名称即使后面带有 Research Project 字样，
# 只要不符合这些标题开头，就不会被统一删除。
NON_COMPANY_PROJECT_TITLE_PATTERN = re.compile(
    r"^(?:"
    r"research\s+on\b|"
    r"analysis\s+of\b|"
    r"a\s+(?:statistical\s+)?study\s+on\b|"
    r"study\s+on\b|"
    r"regression\s+analysis\s+on\b|"
    r"medical\s+text\s+analysis\b|"
    r"aspect[- ]based\s+semantic\s+analysis\b|"
    r"capital\s+asset\s+pricing\s+model\s+applied\b|"
    r"DCF\s+valuation\b|"
    r"complete\s+SWOT\b|"
    r"assessment\s+analysis\b|"
    r"stock\s+simulation\b|"
    r"statistical\s+data\s+technologies\s+course\s+project\b|"
    r"capital\s+one\s+case\s+competition\b|"
    r"hosted\b.+\bcompetition\b"
    r")",
    flags=re.IGNORECASE,
)

# PDF bullet 损坏符号开头的文本不能作为公司名称。
BROKEN_BULLET_COMPANY_PATTERN = re.compile(
    r"^[!•●▪◦‣·\uf0b7]+\s*"
)


def looks_like_prose_description(
    value,
):
    """使用明确句子证据判断候选是否更像工作描述而不是组织名称。"""

    # 空值不是描述句。
    if value is None:
        return False

    # 只提取英文单词用于判断大小写结构。
    words = re.findall(
        r"[A-Za-z][A-Za-z'’.-]*",
        str(
            value
        ),
    )

    # 少于十个单词时不使用长句启发式规则。
    if len(
        words
    ) < 10:
        return False

    # 保存去掉首尾空白后的原始候选。
    candidate_text = str(
        value
    ).strip()

    # 以普通小写单词开头的长文本通常是上一条 bullet 的续行。
    #
    # iProspect、iResearch 和 iOS 这类品牌或技术名称虽然以小写 i
    # 开头，但第二个字符是大写，因此不会被这条规则删除。
    lowercase_sentence_start = bool(
        re.match(
            r"^[a-hj-z][a-z]+\b",
            candidate_text,
        )
    )

    # 以常见动作或陈述词开头的长文本属于描述句。
    action_sentence_start = bool(
        re.match(
            r"^(?:"
            r"arranged|assembled|being|collected|collecting|"
            r"established|explored|facilitated|made|maintained|"
            r"mobilized|searched|supported|translated|updated|won|"
            r"analyzed|analysed|created|developed|designed|implemented|"
            r"led|managed|prepared|provided|researched|served|utilized|"
            r"wrote"
            r")\b",
            candidate_text,
            flags=re.IGNORECASE,
        )
    )

    # 包含明确成果、动作关系或量化结果的长文本属于描述句。
    result_sentence_evidence = bool(
        re.search(
            r"(?:"
            r"\bresult(?:ed|ing)?\s+in\b|"
            r"\bwhich\s+(?:saved|increased|reduced|improved|resulted)\b|"
            r"\b(?:increased|reduced|improved|boosted|saved)\b.{0,40}"
            r"(?:\d+(?:\.\d+)?\s*%|\d+\s+hours?)|"
            r"\b(?:responsible\s+for|worked\s+with|assisted\s+with)\b"
            r")",
            candidate_text,
            flags=re.IGNORECASE,
        )
    )

    # 长文本以句号结束时更接近完整描述句。
    complete_sentence_ending = candidate_text.endswith(
        (
            ".",
            "。",
        )
    )

    # 必须具有明确的句子证据才删除。
    #
    # 不再单纯按照单词数量或小写单词数量删除，
    # 因此合法的长大学、研究机构和公司名称能够保留。
    return bool(
        lowercase_sentence_start
        or action_sentence_start
        or result_sentence_evidence
        or complete_sentence_ending
    )


def normalize_date_token(
    date_token,
):
    """把一个日期 token 转成 YYYY 或 YYYY-MM。"""

    # 清理 token。
    cleaned_token = clean_text_value(
        date_token
    )

    # 空值保持 None。
    if cleaned_token is None:
        return None, False

    # Present 表示当前。
    if re.fullmatch(
        r"(?:present|current|now)",
        cleaned_token,
        flags=re.IGNORECASE,
    ):
        return None, True

    # YYYY-MM / YYYY.MM。
    year_month_match = re.fullmatch(
        r"(?P<year>(?:19|20)\d{2})"
        r"[./-](?P<month>1[0-2]|0?[1-9])",
        cleaned_token,
    )

    # 匹配成功时返回 YYYY-MM。
    if year_month_match:
        return (
            f"{int(year_month_match.group('year')):04d}-"
            f"{int(year_month_match.group('month')):02d}",
            False,
        )

    # MM-YYYY。
    month_year_match = re.fullmatch(
        r"(?P<month>1[0-2]|0?[1-9])"
        r"[./-](?P<year>(?:19|20)\d{2})",
        cleaned_token,
    )

    # 匹配成功时返回 YYYY-MM。
    if month_year_match:
        return (
            f"{int(month_year_match.group('year')):04d}-"
            f"{int(month_year_match.group('month')):02d}",
            False,
        )

    # Month YYYY。
    named_month_match = re.fullmatch(
        r"(?P<month>[A-Za-z]+)\.?\s+"
        r"(?P<year>(?:19|20)\d{2})",
        cleaned_token,
    )

    # 匹配成功时转换 month。
    if named_month_match:
        month_number = MONTH_NUMBER_BY_NAME.get(
            named_month_match.group(
                "month"
            ).lower()
        )

        # 有效 month 时返回。
        if month_number is not None:
            return (
                f"{int(named_month_match.group('year')):04d}-"
                f"{month_number:02d}",
                False,
            )

    # 只有 YYYY。
    if re.fullmatch(
        r"(?:19|20)\d{2}",
        cleaned_token,
    ):
        return cleaned_token, False

    # 无法解析。
    return None, False


def extract_employment_section_lines(
    resume_text,
):
    """只提取明确 employment header 后面的 section lines。"""

    # 清理原始简历。
    cleaned_resume_text = clean_text_value(
        resume_text
    )

    # 没有简历文本。
    if cleaned_resume_text is None:
        return [], "missing_resume_text"

    # 保存 employment lines。
    employment_lines = []

    # 是否正在读取 employment section。
    in_employment_section = False

    # 是否找到明确 header。
    found_employment_header = False

    # 逐行处理。
    for raw_line in cleaned_resume_text.splitlines():
        # 保留内容但统一首尾空白。
        line = raw_line.strip()

        # 只为 section header 判断移除行尾 PDF/排版装饰线。
        #
        # 例如：
        # WORK EXPERIENCE——————————————————————
        #
        # 原始 line 仍然保持不变；这里只生成一个用于匹配标题的副本，
        # 因此不会改变原有 Employment 内容、日期或描述解析逻辑。
        section_header_line = re.sub(
            r"[\s_=\-–—─━]+$",
            "",
            line,
        ).strip()

        # employment header 开始 section。
        if EMPLOYMENT_HEADER_PATTERN.fullmatch(
            section_header_line
        ):
            in_employment_section = True
            found_employment_header = True
            continue

        # 已经进入 employment 后遇到其他 section 时停止。
        if (
            in_employment_section
            and STOP_SECTION_PATTERN.fullmatch(
                section_header_line
            )
        ):
            break

        # section 内非空行保存。
        if (
            in_employment_section
            and line
        ):
            employment_lines.append(
                line
            )

    # 没找到 header。
    if not found_employment_header:
        return [], "no_explicit_employment_section"

    # section 是空的。
    if not employment_lines:
        return [], "empty_employment_section"

    # 返回 section。
    return (
        employment_lines,
        "explicit_employment_section",
    )


def find_entry_start_indexes(
    section_lines,
):
    """寻找后面三行内至少包含一个可靠日期的非bullet entry起点。"""

    # 保存 entry 起点。
    start_indexes = []

    # 逐行检查。
    for line_index, line in enumerate(
        section_lines
    ):
        # bullet 描述不作为 entry 起点。
        if BULLET_PREFIX_PATTERN.match(
            line
        ):
            continue

        # 当前行及后两行形成小窗口。
        lookahead_text = "\n".join(
            section_lines[
                line_index:
                min(
                    line_index + 3,
                    len(
                        section_lines
                    ),
                )
            ]
        )

        # 查找窗口内第一个日期。
        date_match = EMPLOYMENT_DATE_PATTERN.search(
            lookahead_text
        )

        # 窗口中没有任何日期时跳过。
        if date_match is None:
            continue

        # 兼容 Date → Company → Position 排版。
        #
        # 原 v5 逻辑要求日期前已经出现公司或岗位证据，继续完整保留在
        # 本分支后面。这里只额外接受“当前行以日期开头，并且日期后到
        # 后两行内出现公司或岗位证据”的格式。
        date_starts_current_line = (
            date_match.start()
            == 0
        )

        # 取得日期之后的同一窗口文本。
        header_after_date = lookahead_text[
            date_match.end():
        ]

        # 日期在前，并且后续存在公司或岗位关键词时，当前日期行就是
        # 一个可靠 entry 起点。
        if (
            date_starts_current_line
            and (
                POSITION_KEYWORD_PATTERN.search(
                    header_after_date
                )
                is not None
                or COMPANY_KEYWORD_PATTERN.search(
                    header_after_date
                )
                is not None
            )
        ):
            # 与上一候选过近时仍沿用原 v5 的去重规则。
            if (
                not start_indexes
                or line_index
                > start_indexes[-1] + 2
            ):
                start_indexes.append(
                    line_index
                )

            # 当前日期起点已经完成处理，不再进入旧的日期前 header 路径。
            continue

        # 动作描述句不能作为工作经历起点。
        if DESCRIPTION_ACTION_PATTERN.match(
            line
        ):
            continue

        # 只检查日期之前的header内容。
        header_before_date = lookahead_text[
            :date_match.start()
        ]

        # header 中至少需要职位或组织关键词证据。
        #
        # 这是为了避免普通描述句只因为附近出现年份就被切成新entry。
        if (
            POSITION_KEYWORD_PATTERN.search(
                header_before_date
            )
            is None
            and COMPANY_KEYWORD_PATTERN.search(
                header_before_date
            )
            is None
        ):
            continue

        # 与上一候选过近时只保留更早起点。
        if (
            start_indexes
            and line_index
            <= start_indexes[-1] + 2
        ):
            continue

        # 保存起点。
        start_indexes.append(
            line_index
        )

    # 返回起点。
    return start_indexes


def split_employment_entries(
    section_lines,
):
    """根据稳定 entry 起点切分工作经历。"""

    # 找出起点。
    start_indexes = find_entry_start_indexes(
        section_lines
    )

    # 没有可靠起点时不生成 record。
    if not start_indexes:
        return []

    # 保存 entries。
    entries = []

    # 逐个起点切分。
    for start_order, start_index in enumerate(
        start_indexes
    ):
        # 下一起点或 section 末尾作为终点。
        end_index = (
            start_indexes[
                start_order + 1
            ]
            if start_order + 1
            < len(
                start_indexes
            )
            else len(
                section_lines
            )
        )

        # 取得 entry lines。
        entry_lines = section_lines[
            start_index:end_index
        ]

        # entry 必须至少有一个日期。
        if EMPLOYMENT_DATE_PATTERN.search(
            "\n".join(
                entry_lines
            )
        ) is None:
            continue

        # 保存 entry。
        entries.append(
            entry_lines
        )

    # 返回 entries。
    return entries


def extract_company_and_position(
    entry_lines,
):
    """从日期行之前的 header lines 保守识别公司和职位。"""

    # 拼接 entry。
    entry_text = "\n".join(
        entry_lines
    )

    # 找到日期范围或单个日期。
    date_match = EMPLOYMENT_DATE_PATTERN.search(
        entry_text
    )

    # 没有任何日期时无法继续。
    if date_match is None:
        return None, None, None, None

    # 原 v5 路径：使用日期范围之前的文本判断公司和职位。
    header_text = entry_text[
        :date_match.start()
    ].strip()

    # 新增兼容路径：Date → Company → Position。
    #
    # 当日期前没有 header 时，只取日期之后最多三行作为 header 候选。
    # 三行足以覆盖常见的 Company、Position 和一行辅助信息，同时避免
    # 把整段工作描述交给公司/岗位识别。后面的所有原 v5 清洗、排除和
    # 标准化规则继续原样执行。
    if not header_text:
        after_date_lines = [
            line.strip()
            for line in entry_text[
                date_match.end():
            ].splitlines()
            if line.strip()
        ]

        header_text = "\n".join(
            after_date_lines[:3]
        ).strip()

    # 按换行、竖线、分号拆分 header。
    header_parts = [
        part.strip(
            " ,;|:-–—"
        )
        for part in re.split(
            r"[\n|;]+",
            header_text,
        )
        if part.strip(
            " ,;|:-–—"
        )
    ]

    # 如果只有一行，进一步按两个以上空格拆分。
    if len(
        header_parts
    ) == 1:
        spaced_parts = [
            part.strip(
                " ,;|:-–—"
            )
            for part in re.split(
                r"\s{2,}",
                header_parts[0],
            )
            if part.strip(
                " ,;|:-–—"
            )
        ]

        # 拆出更多内容时使用。
        if len(
            spaced_parts
        ) > 1:
            header_parts = spaced_parts

    # 尽力移除 location suffix。
    cleaned_parts = [
        LOCATION_SUFFIX_PATTERN.sub(
            "",
            part,
        ).strip(
            " ,;|:-–—"
        )
        for part in header_parts
    ]

    # 删除空项。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if part
    ]

    # 删除单独地点。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if LOCATION_ONLY_PATTERN.fullmatch(
            part
        )
        is None
    ]

    # 删除单独月份。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if MONTH_ONLY_PATTERN.fullmatch(
            part
        )
        is None
    ]

    # 删除单独季节。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if SEASON_ONLY_PATTERN.fullmatch(
            part
        )
        is None
    ]

    # 删除其他 section 标题。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if NON_EMPLOYMENT_HEADER_PATTERN.match(
            part
        )
        is None
    ]

    # 删除明显的工作描述动作句。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if DESCRIPTION_ACTION_PATTERN.match(
            part
        )
        is None
    ]

    # 删除以连接词开头的描述续行。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if re.match(
            r"^(?:"
            r"and|or|on|which|that|with|through|by|"
            r"assignments?|topics?|key\s+investment|"
            r"data\s+of|theory\s+as"
            r")\b",
            part,
            flags=re.IGNORECASE,
        )
        is None
    ]

    # 删除只包含PDF bullet或损坏符号的内容。
    cleaned_parts = [
        part
        for part in cleaned_parts
        if re.search(
            r"[A-Za-z0-9\u4e00-\u9fff]",
            part,
        )
    ]

    # 没有 header 内容。
    if not cleaned_parts:
        return None, None, None, None

    # 找出明显职位。
    position_candidates = [
        part
        for part in cleaned_parts
        if POSITION_KEYWORD_PATTERN.search(
            part
        )
    ]

    # 找出明显公司。
    company_candidates = [
        part
        for part in cleaned_parts
        if COMPANY_KEYWORD_PATTERN.search(
            part
        )
        and part
        not in position_candidates
    ]

    # 唯一明显职位。
    raw_position_name = (
        position_candidates[0]
        if len(
            position_candidates
        )
        == 1
        else None
    )

    # 唯一明显公司。
    raw_company_name = (
        company_candidates[0]
        if len(
            company_candidates
        )
        == 1
        else None
    )

    # 两个 part 且只识别出职位时，另一个作为公司。
    if (
        raw_company_name is None
        and raw_position_name is not None
        and len(
            cleaned_parts
        )
        == 2
    ):
        raw_company_name = next(
            part
            for part in cleaned_parts
            if part
            != raw_position_name
        )

    # 两个 part 且只识别出公司时，另一个作为职位。
    if (
        raw_position_name is None
        and raw_company_name is not None
        and len(
            cleaned_parts
        )
        == 2
    ):
        raw_position_name = next(
            part
            for part in cleaned_parts
            if part
            != raw_company_name
        )

    # 恰好两个 part 且都没有关键词时，
    # 保守沿用常见 Company → Position 顺序。
    if (
        raw_company_name is None
        and raw_position_name is None
        and len(
            cleaned_parts
        )
        == 2
    ):
        raw_company_name = cleaned_parts[0]
        raw_position_name = cleaned_parts[1]

    # 最终再次防止地点被保存成公司名称。
    if (
        raw_company_name is not None
        and LOCATION_ONLY_PATTERN.fullmatch(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 最终再次防止地点被保存成岗位名称。
    if (
        raw_position_name is not None
        and LOCATION_ONLY_PATTERN.fullmatch(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 最终再次防止月份残片被保存成公司名称。
    if (
        raw_company_name is not None
        and MONTH_ONLY_PATTERN.fullmatch(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 最终再次防止月份残片被保存成岗位名称。
    if (
        raw_position_name is not None
        and MONTH_ONLY_PATTERN.fullmatch(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 最终再次防止季节被保存成公司名称。
    if (
        raw_company_name is not None
        and SEASON_ONLY_PATTERN.fullmatch(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 最终再次防止季节被保存成岗位名称。
    if (
        raw_position_name is not None
        and SEASON_ONLY_PATTERN.fullmatch(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 最终再次防止section标题被保存成公司名称。
    if (
        raw_company_name is not None
        and NON_EMPLOYMENT_HEADER_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 最终再次防止section标题被保存成岗位名称。
    if (
        raw_position_name is not None
        and NON_EMPLOYMENT_HEADER_PATTERN.match(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 明显动作描述句不能作为公司名称。
    if (
        raw_company_name is not None
        and DESCRIPTION_ACTION_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 当前数据中已经确认的工作描述句和工作成果句
    # 不能作为公司名称。
    #
    # 这里根据描述动作开头判断，不按单词数量统一删除，
    # 因此较长但合法的大学、研究机构和公司名称仍可保留。
    if (
        raw_company_name is not None
        and NON_COMPANY_DESCRIPTION_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # section、课程、奖项或工具列表不能作为公司名称。
    if (
        raw_company_name is not None
        and NON_COMPANY_METADATA_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 明确的研究、分析、比赛或课程项目标题
    # 不能作为公司名称。
    if (
        raw_company_name is not None
        and NON_COMPANY_PROJECT_TITLE_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # PDF bullet 或损坏符号开头的文本不能作为公司名称。
    if (
        raw_company_name is not None
        and BROKEN_BULLET_COMPANY_PATTERN.match(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 较长且具有明显自然语言句子大小写结构的候选，
    # 也不能作为公司名称。
    #
    # 这一步不会按“所有长名称”统一删除，
    # 合法的长机构名称如果主要由专有名词组成仍会保留。
    if (
        raw_company_name is not None
        and looks_like_prose_description(
            raw_company_name
        )
    ):
        raw_company_name = None

    # 明显动作描述句不能作为岗位名称。
    if (
        raw_position_name is not None
        and DESCRIPTION_ACTION_PATTERN.match(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 指导教师、Working as描述和数量句不能作为岗位名称。
    if (
        raw_position_name is not None
        and NON_POSITION_DESCRIPTION_PATTERN.match(
            raw_position_name
        )
    ):
        raw_position_name = None

    # 小写开头且完全没有职位关键词的短语通常是描述续行，
    # 不能作为岗位名称。
    #
    # 如果包含可靠岗位关键词则保留，因此以下合法值不会被删除：
    # data analyst intern
    # research assistant
    # iProspect ... E-commerce Analyst Intern
    # iResearch ... Investment Research Summer Analyst
    # iOS Developer Intern
    if (
        raw_position_name is not None
        and raw_position_name[
            0
        ].islower()
        and POSITION_KEYWORD_PATTERN.search(
            raw_position_name
        )
        is None
    ):
        raw_position_name = None

    # 返回原始和标准化文本。
    return (
        raw_company_name,
        normalize_lower_text(
            raw_company_name
        ),
        raw_position_name,
        normalize_lower_text(
            raw_position_name
        ),
    )


def parse_employment_entry(
    entry_lines,
    employment_entry_order,
):
    """把一段原始 employment entry 转成结构化 record。"""

    # 保存完整原始 entry。
    raw_employment_text = "\n".join(
        entry_lines
    ).strip()

    # 查找日期范围或单个日期。
    date_match = EMPLOYMENT_DATE_PATTERN.search(
        raw_employment_text
    )

    # 没有任何日期不应进入本 function。
    if date_match is None:
        return None

    # 日期范围和单个日期需要分别处理。
    if date_match.group(
        "range_start"
    ) is not None:
        # 标准化范围开始日期。
        (
            position_start_date,
            _,
        ) = normalize_date_token(
            date_match.group(
                "range_start"
            )
        )

        # 标准化范围结束日期。
        (
            position_end_date,
            is_current,
        ) = normalize_date_token(
            date_match.group(
                "range_end"
            )
        )
    else:
        # 单个明确日期暂时保存为开始日期。
        (
            position_start_date,
            single_date_is_current,
        ) = normalize_date_token(
            date_match.group(
                "single_date"
            )
        )

        # 单独的 Present/Current 不能形成可靠开始或结束日期。
        position_end_date = None
        is_current = bool(
            single_date_is_current
        )

    # 开始和结束日期都没有可靠值时拒绝。
    if (
        position_start_date is None
        and position_end_date is None
    ):
        return None

    # 提取公司和职位。
    (
        raw_company_name,
        normalized_company_name,
        raw_position_name,
        normalized_position_name,
    ) = extract_company_and_position(
        entry_lines
    )

    # 公司名称和岗位名称至少需要可靠取得一个。
    if (
        raw_company_name is None
        and raw_position_name is None
    ):
        return None

    # 日期范围之后的原始内容。
    description_text = raw_employment_text[
        date_match.end():
    ]

    # Date → Company → Position 排版中，公司和岗位位于日期之后，
    # 不能把这两行再次保存进 position_description。
    #
    # 该处理只在日期位于 entry 开头时启用；原 v5 的 Company/Position
    # → Date → Description 路径保持完全不变。
    if date_match.start() == 0:
        description_lines = [
            line.strip()
            for line in description_text.splitlines()
            if line.strip()
        ]

        # 依次移除已经识别为公司或岗位的开头行。
        extracted_header_values = {
            normalize_lower_text(
                raw_company_name
            ),
            normalize_lower_text(
                raw_position_name
            ),
        }

        # NULL 不参与文本比较。
        extracted_header_values.discard(
            None
        )

        # 只从描述开头移除已提取 header，遇到第一条非 header 后停止。
        while (
            description_lines
            and normalize_lower_text(
                description_lines[0]
            )
            in extracted_header_values
        ):
            description_lines.pop(
                0
            )

        # 重新组成纯工作描述。
        description_text = "\n".join(
            description_lines
        )

    # 清理工作描述。
    position_description = clean_text_value(
        description_text
    )

    # 计算当前记录结构状态。
    if (
        raw_company_name is not None
        and raw_position_name is not None
    ):
        extraction_status = (
            "company_and_position_extracted"
        )
    elif raw_company_name is not None:
        extraction_status = (
            "company_only"
        )
    elif raw_position_name is not None:
        extraction_status = (
            "position_only"
        )
    else:
        extraction_status = (
            "company_and_position_missing"
        )

    # 返回结构化结果。
    return {
        "employment_entry_order": int(
            employment_entry_order
        ),
        "raw_employment_text": (
            raw_employment_text
        ),
        "raw_company_name": raw_company_name,
        "normalized_company_name": (
            normalized_company_name
        ),
        "raw_position_name": raw_position_name,
        "normalized_position_name": (
            normalized_position_name
        ),
        "position_description": (
            position_description
        ),
        "position_start_date": (
            position_start_date
        ),
        "position_end_date": (
            position_end_date
        ),
        "is_current": int(
            is_current
        ),
        "entry_extraction_status": (
            extraction_status
        ),
    }


# ============================================================
# 读取当前批次 application.csv
# ============================================================

# 读取当前固定 application.csv。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# application.csv 必须包含 Application.id。
if "id" not in application_dataframe.columns:
    raise ValueError(
        "application.csv 缺少必要 column：id"
    )

# 只保留已经成功进入 D1、拥有 Application.id 的当前批次 records。
application_dataframe = (
    application_dataframe.loc[
        application_dataframe["id"].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

# 当前批次没有有效 Application.id 时停止，避免覆盖出误导性结果。
if application_dataframe.empty:
    raise ValueError(
        "当前 application.csv 没有包含有效的 Application.id。"
    )

# 把当前批次 Application.id 转成标准 Python integer。
application_dataframe["id"] = pd.to_numeric(
    application_dataframe["id"],
    errors="raise",
).astype(
    "int64"
)

# 当前批次 Application.id 必须唯一。
if application_dataframe["id"].duplicated().any():
    raise ValueError(
        "当前 application.csv 存在重复 Application.id。"
    )

# 保存当前批次 Application ids。
current_application_ids = sorted(
    int(application_id)
    for application_id
    in application_dataframe["id"].tolist()
)


# ============================================================
# 只从 D1 读取当前批次 Candidate snapshot
# ============================================================

# 每次最多查询 50 个 Application，兼容首批和任意大小增量批次。
CANDIDATE_QUERY_CHUNK_SIZE = 50

# 保存当前批次全部 Candidate rows。
candidate_rows = []

# 分块查询当前批次 Candidate、Application、Person 和原始简历文本。
for chunk_start in range(
    0,
    len(current_application_ids),
    CANDIDATE_QUERY_CHUNK_SIZE,
):
    # 取得当前分块 Application ids。
    chunk_application_ids = current_application_ids[
        chunk_start:
        chunk_start + CANDIDATE_QUERY_CHUNK_SIZE
    ]

    # 生成参数化 IN 占位符。
    placeholders = ",".join(
        ["?"] * len(chunk_application_ids)
    )

    # 只查询当前分块对应的 Candidate snapshots。
    candidate_result = execute_d1(
        f"""
        SELECT
          cs."id" AS candidate_snapshot_id,
          cs."application_id",
          app."person_id",
          cs."raw_resume_text"
        FROM "Candidate_snapshot" AS cs
        INNER JOIN "Application" AS app
          ON app."id" = cs."application_id"
        WHERE cs."application_id" IN (
          {placeholders}
        )
        ORDER BY cs."id";
        """,
        [
            int(application_id)
            for application_id
            in chunk_application_ids
        ],
    )

    # 合并当前分块结果。
    candidate_rows.extend(
        result_rows(
            candidate_result
        )
    )

# 建立 D1 实际返回的 Application id 集合。
returned_application_ids = {
    int(candidate_row["application_id"])
    for candidate_row
    in candidate_rows
}

# 当前 CSV 中的每个 Application 都必须已有 Candidate_snapshot。
application_ids_missing_candidate = sorted(
    set(current_application_ids)
    - returned_application_ids
)

# 缺少 Candidate mapping 时停止。
if application_ids_missing_candidate:
    raise ValueError(
        "当前 application.csv 中存在尚未生成 "
        "Candidate_snapshot 的 Application.id："
        f"{application_ids_missing_candidate[:20]}"
    )

# 同一个 Application 只能对应一个 Candidate snapshot。
if len(candidate_rows) != len(current_application_ids):
    raise ValueError(
        "当前 Application 与 Candidate_snapshot mapping "
        "不是一对一关系。"
    )

# 按 Candidate snapshot id 稳定排序，保持原提取逻辑的确定性。
candidate_rows = sorted(
    candidate_rows,
    key=lambda row: int(
        row["candidate_snapshot_id"]
    ),
)

# 显示当前批次 mapping 结果。
print(
    "Candidate employment extraction mapping validation:",
    {
        "current_application_count": int(
            len(current_application_ids)
        ),
        "current_candidate_count": int(
            len(candidate_rows)
        ),
        "missing_candidate_count": int(
            len(application_ids_missing_candidate)
        ),
    },
)


# ============================================================
# 对每个 Candidate 执行确定性工作经历提取
# ============================================================

# 保存记录级结果。
employment_records = []

# 保存 Candidate 级状态。
employment_status_records = []

# 按 D1 id 稳定处理。
for candidate_index, candidate_row in enumerate(
    candidate_rows,
    start=1,
):
    # 读取 ids。
    candidate_snapshot_id = int(
        candidate_row[
            "candidate_snapshot_id"
        ]
    )

    # 读取 Application id。
    application_id = int(
        candidate_row[
            "application_id"
        ]
    )

    # 读取 Person id。
    person_id = int(
        candidate_row[
            "person_id"
        ]
    )

    # 读取原始简历。
    raw_resume_text = clean_text_value(
        candidate_row.get(
            "raw_resume_text"
        )
    )

    # 提取明确 employment section。
    (
        employment_section_lines,
        employment_section_status,
    ) = extract_employment_section_lines(
        raw_resume_text
    )

    # 只有明确识别到 Employment section 时才切分工作经历。
    #
    # 没有明确 Employment section 的 Candidate：
    # 1. 不从简历其他区域猜测工作经历。
    # 2. 不生成任何工作经历明细 record。
    # 3. 仍然保留在 Candidate 级状态 CSV 中，
    #    employment_section_status 会是
    #    no_explicit_employment_section。
    if employment_section_status == (
        "explicit_employment_section"
    ):
        employment_entries = (
            split_employment_entries(
                employment_section_lines
            )
        )
    else:
        employment_entries = []

    # 保存当前 Candidate 的结构状态统计。
    entry_status_counts = {
        "company_and_position_extracted": 0,
        "company_only": 0,
        "position_only": 0,
        "company_and_position_missing": 0,
    }

    # 逐条解析 entry。
    for employment_entry_order, entry_lines in enumerate(
        employment_entries,
        start=1,
    ):
        # 解析工作经历。
        parsed_entry = parse_employment_entry(
            entry_lines,
            employment_entry_order,
        )

        # 无结果时跳过。
        if parsed_entry is None:
            continue

        # 增加当前结构状态统计。
        entry_status_counts[
            parsed_entry[
                "entry_extraction_status"
            ]
        ] += 1

        # 保存完整 record。
        employment_records.append(
            {
                "rule_version": RULE_VERSION,
                "application_id": application_id,
                "person_id": person_id,
                "candidate_snapshot_id": (
                    candidate_snapshot_id
                ),
                **parsed_entry,
            }
        )

    # 当前 Candidate 实际 record 数。
    candidate_record_count = sum(
        entry_status_counts.values()
    )

    # 生成 Candidate 总体提取状态。
    if raw_resume_text is None:
        candidate_extraction_status = (
            "missing_resume_text"
        )
    elif employment_section_status != (
        "explicit_employment_section"
    ):
        candidate_extraction_status = (
            employment_section_status
        )
    elif candidate_record_count == 0:
        candidate_extraction_status = (
            "no_reliable_dated_entry"
        )
    elif (
        entry_status_counts[
            "company_and_position_extracted"
        ]
        == candidate_record_count
    ):
        candidate_extraction_status = (
            "all_entries_structured"
        )
    else:
        candidate_extraction_status = (
            "partially_structured"
        )

    # 保存 Candidate status。
    employment_status_records.append(
        {
            "rule_version": RULE_VERSION,
            "application_id": application_id,
            "person_id": person_id,
            "candidate_snapshot_id": (
                candidate_snapshot_id
            ),
            "resume_text_present": int(
                raw_resume_text is not None
            ),
            "employment_section_status": (
                employment_section_status
            ),
            "employment_record_count": int(
                candidate_record_count
            ),
            "company_and_position_count": int(
                entry_status_counts[
                    "company_and_position_extracted"
                ]
            ),
            "company_only_count": int(
                entry_status_counts[
                    "company_only"
                ]
            ),
            "position_only_count": int(
                entry_status_counts[
                    "position_only"
                ]
            ),
            "company_and_position_missing_count": int(
                entry_status_counts[
                    "company_and_position_missing"
                ]
            ),
            "extraction_status": (
                candidate_extraction_status
            ),
        }
    )

    # 每100个显示进度，并显示最后一个。
    if (
        candidate_index % 100 == 0
        or candidate_index
        == len(
            candidate_rows
        )
    ):
        print(
            "Processed Candidate employment extraction:",
            f"{candidate_index}/"
            f"{len(candidate_rows)}",
        )


# ============================================================
# 覆盖保存两个固定 CSV
# ============================================================

# 建立记录级 DataFrame。
employment_dataframe = pd.DataFrame(
    employment_records,
    columns=[
        "rule_version",
        "application_id",
        "person_id",
        "candidate_snapshot_id",
        "employment_entry_order",
        "raw_employment_text",
        "raw_company_name",
        "normalized_company_name",
        "raw_position_name",
        "normalized_position_name",
        "position_description",
        "position_start_date",
        "position_end_date",
        "is_current",
        "entry_extraction_status",
    ],
)

# 稳定排序记录级结果。
employment_dataframe = (
    employment_dataframe.sort_values(
        [
            "application_id",
            "employment_entry_order",
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# 建立 Candidate status DataFrame。
employment_status_dataframe = pd.DataFrame(
    employment_status_records
)

# 稳定排序 Candidate status。
employment_status_dataframe = (
    employment_status_dataframe.sort_values(
        [
            "candidate_snapshot_id",
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# status 必须覆盖全部 Candidate。
if len(
    employment_status_dataframe
) != len(
    candidate_rows
):
    raise RuntimeError(
        "Candidate employment status 数量与 D1 Candidate 数量不一致。"
    )

# 使用临时文件保存记录级结果。
employment_temp_path = (
    f"{EMPLOYMENT_EXTRACTION_CSV_PATH}.tmp"
)

# 写入临时记录级 CSV。
employment_dataframe.to_csv(
    employment_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 替换固定记录级 CSV。
os.replace(
    employment_temp_path,
    EMPLOYMENT_EXTRACTION_CSV_PATH,
)

# 使用临时文件保存 Candidate status。
status_temp_path = (
    f"{EMPLOYMENT_STATUS_CSV_PATH}.tmp"
)

# 写入临时 status CSV。
employment_status_dataframe.to_csv(
    status_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 替换固定 status CSV。
os.replace(
    status_temp_path,
    EMPLOYMENT_STATUS_CSV_PATH,
)

# 显示整体统计。
print(
    "Candidate employment extraction summary:",
    {
        "candidate_count": int(
            len(
                employment_status_dataframe
            )
        ),
        "employment_record_count": int(
            len(
                employment_dataframe
            )
        ),
        "candidate_with_employment_count": int(
            (
                employment_status_dataframe[
                    "employment_record_count"
                ]
                > 0
            ).sum()
        ),
        "fully_structured_record_count": int(
            (
                employment_dataframe[
                    "entry_extraction_status"
                ]
                == "company_and_position_extracted"
            ).sum()
        ),
    },
)

# 显示记录级结果路径。
print(
    "Employment extraction CSV:",
    EMPLOYMENT_EXTRACTION_CSV_PATH,
)

# 显示 Candidate status 路径。
print(
    "Employment status CSV:",
    EMPLOYMENT_STATUS_CSV_PATH,
)

# 显示记录级预览。
display(
    employment_dataframe.head(
        20
    )
)

# 显示 status 统计。
display(
    employment_status_dataframe[
        "extraction_status"
    ].value_counts(
        dropna=False
    )
)

# 显示完成。
print(
    "Candidate employment rule extraction completed successfully."
)


### Candidate_employment, Person_employment 数据导入

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 将当前 Candidate Employment CSV 幂等导入 D1。
#
# 写入关系：
# 1. 每一段可靠工作经历写入 Person_Position。
# 2. Candidate 与该工作经历的关系写入 Candidate_Position。
#
# 重要：
# 1. 本 Cell 不依赖其他 Colab Cell 的 variable、function 或 class。
# 2. 本 Cell 只读取固定的两份 Candidate Employment CSV。
# 3. 本 Cell 不重新提取工作经历，也不检查 rule_version。
# 4. 每次运行都会重新执行质量准入和 D1 mapping 验证。
# 5. 相同输入可以重复运行，不会重复创建工作经历或 bridge。
# 6. 简历工作经历不是申请人本次申请的招聘岗位。
# 7. Person_Position.company_id 和 position_id 暂时写入 SQL NULL。
# 8. 单条失败会写回 CSV，其他记录继续处理。
# 9. 两份结果 CSV 固定覆盖，不会无限增加文件。
# ============================================================

# os 用于检查文件和原子覆盖 CSV。
import os

# time 只用于 D1 临时失败后的等待。
import time

# datetime 用于生成 UTC timestamp。
from datetime import datetime, timezone

# pandas 用于读取和更新 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定 Google Drive 路径
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 保存工作经历明细 CSV。
EMPLOYMENT_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_employment_rule_extraction.csv"
)

# 保存 Candidate 级工作经历状态 CSV。
EMPLOYMENT_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_employment_rule_extraction_status.csv"
)

# 两份输入 CSV 都必须存在。
for required_file_path in [
    EMPLOYMENT_EXTRACTION_CSV_PATH,
    EMPLOYMENT_STATUS_CSV_PATH,
]:
    # 缺少文件时停止。
    if not os.path.exists(
        required_file_path
    ):
        raise FileNotFoundError(
            "找不到必要输入文件："
            f"{required_file_path}"
        )


# ============================================================
# Cloudflare D1 独立连接配置
# ============================================================

# Cloudflare Account ID。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 Database ID。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 query endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 D1 request headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP connection。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时失败有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存本次网络错误。
            last_error = error

            # 还有重试机会时短暂等待。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 重试用完后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            # 非 JSON response 无法继续。
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 SQL 都成功时返回。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 返回的错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有重试机会时短暂等待。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部请求失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    d1_result,
):
    """提取第一条 D1 statement 的 results rows。"""

    # 空结果返回空 list。
    if not d1_result:
        return []

    # 返回第一条 statement 的 rows。
    return d1_result[0].get(
        "results",
        [],
    )


def now_utc_iso():
    """生成标准 UTC timestamp。"""

    # 使用秒级 ISO 格式。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_optional_text(
    value,
):
    """把 pandas 空值和空字符串转换成 Python None。"""

    # pandas 空值转成 None。
    if pd.isna(
        value
    ):
        return None

    # 去掉首尾空白。
    cleaned_value = str(
        value
    ).strip()

    # 空字符串转成 None。
    return (
        cleaned_value
        if cleaned_value
        else None
    )


def require_python_int(
    value,
    field_name,
):
    """把 pandas/NumPy integer 安全转换成 Python int。"""

    # 必要 id 不能为空。
    if pd.isna(
        value
    ):
        raise ValueError(
            f"{field_name} 不能为空。"
        )

    # 显式转成标准 Python int。
    return int(
        value
    )


def normalize_boolean_integer(
    value,
):
    """把 is_current 安全转换成 SQLite 0/1。"""

    # 空值按 0 处理。
    if pd.isna(
        value
    ):
        return 0

    # 数值只允许 0 或 1。
    normalized_value = int(
        value
    )

    # 非法值停止当前 record。
    if normalized_value not in {
        0,
        1,
    }:
        raise ValueError(
            "is_current 必须是 0 或 1。"
        )

    # 返回 SQLite boolean integer。
    return normalized_value


def atomic_write_csv(
    dataframe,
    output_path,
):
    """通过临时文件原子覆盖固定 CSV。"""

    # 保存临时文件路径。
    temporary_path = (
        f"{output_path}.tmp"
    )

    # 先完整写入临时文件。
    dataframe.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    # 成功写完后替换正式文件。
    os.replace(
        temporary_path,
        output_path,
    )


# ============================================================
# 读取并重新验证两份固定 CSV
# ============================================================

# 读取工作经历明细。
employment_dataframe = pd.read_csv(
    EMPLOYMENT_EXTRACTION_CSV_PATH,
    low_memory=False,
)

# 读取 Candidate 工作经历状态。
employment_status_dataframe = pd.read_csv(
    EMPLOYMENT_STATUS_CSV_PATH,
    low_memory=False,
)

# 工作经历明细不能为空。
if employment_dataframe.empty:
    raise ValueError(
        "Employment extraction CSV 没有任何 record。"
    )

# 保存明细必要字段。
required_employment_columns = [
    "application_id",
    "person_id",
    "candidate_snapshot_id",
    "raw_employment_text",
    "raw_company_name",
    "normalized_company_name",
    "raw_position_name",
    "normalized_position_name",
    "position_description",
    "position_start_date",
    "position_end_date",
    "is_current",
    "entry_extraction_status",
]

# 查找缺失字段。
missing_employment_columns = [
    column_name
    for column_name in required_employment_columns
    if column_name
    not in employment_dataframe.columns
]

# 缺少必要字段时停止。
if missing_employment_columns:
    raise ValueError(
        "Employment extraction CSV 缺少 columns："
        f"{missing_employment_columns}"
    )

# 保存状态 CSV 必要字段。
required_status_columns = [
    "candidate_snapshot_id",
    "employment_section_status",
    "employment_record_count",
    "extraction_status",
]

# 查找状态 CSV 缺失字段。
missing_status_columns = [
    column_name
    for column_name in required_status_columns
    if column_name
    not in employment_status_dataframe.columns
]

# 缺少状态字段时停止。
if missing_status_columns:
    raise ValueError(
        "Employment status CSV 缺少 columns："
        f"{missing_status_columns}"
    )

# 原始工作经历文本不能为空。
if employment_dataframe[
    "raw_employment_text"
].isna().any():
    raise ValueError(
        "Employment extraction CSV 存在空 raw_employment_text。"
    )

# 公司和岗位名称不能同时为空。
both_identity_values_missing = (
    employment_dataframe[
        "raw_company_name"
    ].isna()
    & employment_dataframe[
        "raw_position_name"
    ].isna()
)

# 存在完全无法识别公司和岗位的 record 时停止。
if both_identity_values_missing.any():
    raise ValueError(
        "Employment extraction CSV 存在公司和岗位同时为空的 record。"
    )

# 开始和结束日期不能同时为空。
both_dates_missing = (
    employment_dataframe[
        "position_start_date"
    ].isna()
    & employment_dataframe[
        "position_end_date"
    ].isna()
)

# 不满足日期准入条件时停止。
if both_dates_missing.any():
    raise ValueError(
        "Employment extraction CSV 存在开始和结束日期同时为空的 record。"
    )

# 同一个 Candidate 内不允许原始工作经历文本完全重复。
if employment_dataframe.duplicated(
    [
        "candidate_snapshot_id",
        "raw_employment_text",
    ]
).any():
    raise ValueError(
        "Employment extraction CSV 仍存在完全相同的 "
        "candidate_snapshot_id + raw_employment_text。"
    )

# 状态 CSV 的工作经历总数必须与明细一致。
if int(
    employment_status_dataframe[
        "employment_record_count"
    ].sum()
) != len(
    employment_dataframe
):
    raise ValueError(
        "Employment status 的总记录数与明细 CSV 不一致。"
    )

# 没有明确 Employment section 的 Candidate 不得出现在明细 CSV。
non_explicit_candidate_ids = {
    int(
        candidate_id
    )
    for candidate_id in employment_status_dataframe.loc[
        employment_status_dataframe[
            "employment_section_status"
        ].ne(
            "explicit_employment_section"
        ),
        "candidate_snapshot_id",
    ].dropna()
}

# 检查是否存在不允许的明细。
non_explicit_detail_mask = employment_dataframe[
    "candidate_snapshot_id"
].isin(
    non_explicit_candidate_ids
)

# 出现泄漏时停止。
if non_explicit_detail_mask.any():
    raise ValueError(
        "Employment 明细中包含没有明确 Employment section 的 Candidate。"
    )


# ============================================================
# 验证当前 D1 schema
# ============================================================

# 查询 Person_Position columns。
person_position_columns = result_rows(
    execute_d1(
        'PRAGMA table_info("Person_Position");'
    )
)

# 建立 Person_Position column set。
person_position_column_names = {
    row["name"]
    for row in person_position_columns
}

# 本 Cell 需要的 Person_Position columns。
required_person_position_columns = {
    "id",
    "person_id",
    "company_id",
    "position_id",
    "function_id",
    "seniority_id",
    "experience_level_id",
    "location_id",
    "position_start_date",
    "position_end_date",
    "experience_type",
    "is_current",
    "created_at",
    "updated_at",
    "source_candidate_id",
    "raw_company_name",
    "normalized_company_name",
    "raw_position_name",
    "normalized_position_name",
    "position_description",
}

# schema 不完整时停止。
if not required_person_position_columns.issubset(
    person_position_column_names
):
    raise RuntimeError(
        "Person_Position schema 缺少 columns："
        f"{sorted(required_person_position_columns - person_position_column_names)}"
    )

# 查询 Candidate_Position columns。
candidate_position_columns = result_rows(
    execute_d1(
        'PRAGMA table_info("Candidate_Position");'
    )
)

# 建立 Candidate_Position column set。
candidate_position_column_names = {
    row["name"]
    for row in candidate_position_columns
}

# schema 不符合预期时停止。
if not {
    "id",
    "candidate_id",
    "person_position_id",
    "created_at",
}.issubset(
    candidate_position_column_names
):
    raise RuntimeError(
        "Candidate_Position schema 不符合预期。"
    )


# ============================================================
# 验证 CSV 与 D1 Candidate → Application → Person mapping
# ============================================================

# 一次性查询正式 mapping。
candidate_mapping_rows = result_rows(
    execute_d1(
        """
        SELECT
          cs."id" AS candidate_snapshot_id,
          cs."application_id" AS application_id,
          a."person_id" AS person_id
        FROM "Candidate_snapshot" AS cs
        INNER JOIN "Application" AS a
          ON a."id" = cs."application_id";
        """
    )
)

# 建立 Candidate mapping。
candidate_mapping_by_id = {
    int(
        row["candidate_snapshot_id"]
    ): {
        "application_id": int(
            row["application_id"]
        ),
        "person_id": int(
            row["person_id"]
        ),
    }
    for row in candidate_mapping_rows
}

# 查找 D1 中不存在的 Candidate。
csv_candidate_ids = {
    int(
        candidate_id
    )
    for candidate_id in employment_dataframe[
        "candidate_snapshot_id"
    ].dropna()
}

# 计算缺失 Candidate ids。
missing_candidate_ids = sorted(
    csv_candidate_ids
    - set(
        candidate_mapping_by_id
    )
)

# 缺少 Candidate 时停止。
if missing_candidate_ids:
    raise ValueError(
        "以下 Candidate_snapshot.id 不存在于 D1："
        f"{missing_candidate_ids[:20]}"
    )

# 逐行验证 application_id 和 person_id。
for _, validation_row in employment_dataframe.iterrows():
    # 取得 Candidate id。
    candidate_snapshot_id = require_python_int(
        validation_row[
            "candidate_snapshot_id"
        ],
        "candidate_snapshot_id",
    )

    # 取得正式 mapping。
    d1_mapping = candidate_mapping_by_id[
        candidate_snapshot_id
    ]

    # 验证 Application id。
    if require_python_int(
        validation_row[
            "application_id"
        ],
        "application_id",
    ) != d1_mapping[
        "application_id"
    ]:
        raise ValueError(
            "CSV application_id 与 D1 Candidate mapping 不一致："
            f"candidate_snapshot_id={candidate_snapshot_id}"
        )

    # 验证 Person id。
    if require_python_int(
        validation_row[
            "person_id"
        ],
        "person_id",
    ) != d1_mapping[
        "person_id"
    ]:
        raise ValueError(
            "CSV person_id 与 D1 Candidate mapping 不一致："
            f"candidate_snapshot_id={candidate_snapshot_id}"
        )


# ============================================================
# 预先读取现有 D1 records，减少重复运行时的查询数量
# ============================================================

# 查询所有由 Candidate snapshot 产生的 Person_Position。
existing_person_position_rows = result_rows(
    execute_d1(
        """
        SELECT
          "id",
          "person_id",
          "source_candidate_id",
          "raw_company_name",
          "normalized_company_name",
          "raw_position_name",
          "normalized_position_name",
          "position_description",
          "position_start_date",
          "position_end_date",
          "experience_type",
          "is_current"
        FROM "Person_Position"
        WHERE "source_candidate_id" IS NOT NULL
        ORDER BY "id";
        """
    )
)


def build_person_position_key(
    person_id,
    source_candidate_id,
    raw_company_name,
    normalized_company_name,
    raw_position_name,
    normalized_position_name,
    position_description,
    position_start_date,
    position_end_date,
    experience_type,
    is_current,
):
    """建立本 Cell 的稳定 Person_Position 幂等 key。"""

    # 使用所有实际业务字段建立不可变 tuple。
    return (
        int(
            person_id
        ),
        int(
            source_candidate_id
        ),
        clean_optional_text(
            raw_company_name
        ),
        clean_optional_text(
            normalized_company_name
        ),
        clean_optional_text(
            raw_position_name
        ),
        clean_optional_text(
            normalized_position_name
        ),
        clean_optional_text(
            position_description
        ),
        clean_optional_text(
            position_start_date
        ),
        clean_optional_text(
            position_end_date
        ),
        clean_optional_text(
            experience_type
        ),
        normalize_boolean_integer(
            is_current
        ),
    )


# 建立现有 Person_Position key → id mapping。
person_position_id_by_key = {}

# 逐条读取现有记录。
for existing_row in existing_person_position_rows:
    # 建立现有 key。
    existing_key = build_person_position_key(
        existing_row[
            "person_id"
        ],
        existing_row[
            "source_candidate_id"
        ],
        existing_row.get(
            "raw_company_name"
        ),
        existing_row.get(
            "normalized_company_name"
        ),
        existing_row.get(
            "raw_position_name"
        ),
        existing_row.get(
            "normalized_position_name"
        ),
        existing_row.get(
            "position_description"
        ),
        existing_row.get(
            "position_start_date"
        ),
        existing_row.get(
            "position_end_date"
        ),
        existing_row.get(
            "experience_type"
        ),
        existing_row.get(
            "is_current"
        ),
    )

    # 数据库已经存在完全重复记录时停止，避免静默选择。
    if existing_key in person_position_id_by_key:
        raise RuntimeError(
            "D1 已存在完全重复的 Person_Position records。"
        )

    # 保存 mapping。
    person_position_id_by_key[
        existing_key
    ] = int(
        existing_row[
            "id"
        ]
    )

# 查询现有 Candidate_Position bridges。
existing_candidate_position_rows = result_rows(
    execute_d1(
        """
        SELECT
          "id",
          "candidate_id",
          "person_position_id"
        FROM "Candidate_Position"
        ORDER BY "id";
        """
    )
)

# 建立 bridge key → id mapping。
candidate_position_id_by_key = {}

# 逐条读取现有 bridge。
for existing_row in existing_candidate_position_rows:
    # 建立 bridge key。
    existing_bridge_key = (
        int(
            existing_row[
                "candidate_id"
            ]
        ),
        int(
            existing_row[
                "person_position_id"
            ]
        ),
    )

    # 出现重复 bridge 时停止。
    if existing_bridge_key in candidate_position_id_by_key:
        raise RuntimeError(
            "D1 已存在重复 Candidate_Position bridge。"
        )

    # 保存 mapping。
    candidate_position_id_by_key[
        existing_bridge_key
    ] = int(
        existing_row[
            "id"
        ]
    )


# ============================================================
# 幂等写入 Person_Position 和 Candidate_Position
# ============================================================

# 每次运行都重新计算并覆盖导入结果 columns。
for result_column_name in [
    "person_position_id",
    "candidate_position_id",
    "employment_import_status",
    "employment_import_action",
    "employment_import_error",
    "employment_imported_at",
]:
    employment_dataframe[
        result_column_name
    ] = None

# 保存本次统计。
inserted_person_position_count = 0
reused_person_position_count = 0
inserted_candidate_position_count = 0
reused_candidate_position_count = 0
failed_record_count = 0

# 逐条处理工作经历。
for dataframe_index, employment_row in (
    employment_dataframe.iterrows()
):
    try:
        # 取得标准 Python ids。
        candidate_snapshot_id = require_python_int(
            employment_row[
                "candidate_snapshot_id"
            ],
            "candidate_snapshot_id",
        )
        person_id = require_python_int(
            employment_row[
                "person_id"
            ],
            "person_id",
        )

        # 清理全部实际业务字段。
        raw_company_name = clean_optional_text(
            employment_row[
                "raw_company_name"
            ]
        )
        normalized_company_name = clean_optional_text(
            employment_row[
                "normalized_company_name"
            ]
        )
        raw_position_name = clean_optional_text(
            employment_row[
                "raw_position_name"
            ]
        )
        normalized_position_name = clean_optional_text(
            employment_row[
                "normalized_position_name"
            ]
        )
        position_description = clean_optional_text(
            employment_row[
                "position_description"
            ]
        )
        position_start_date = clean_optional_text(
            employment_row[
                "position_start_date"
            ]
        )
        position_end_date = clean_optional_text(
            employment_row[
                "position_end_date"
            ]
        )
        is_current = normalize_boolean_integer(
            employment_row[
                "is_current"
            ]
        )

        # 固定标记数据来源语义。
        experience_type = "resume_employment"

        # 再次执行单条准入检查。
        if (
            raw_company_name is None
            and raw_position_name is None
        ):
            raise ValueError(
                "公司名称和岗位名称不能同时为空。"
            )

        # 再次执行日期准入检查。
        if (
            position_start_date is None
            and position_end_date is None
        ):
            raise ValueError(
                "开始日期和结束日期不能同时为空。"
            )

        # 建立稳定 Person_Position key。
        person_position_key = build_person_position_key(
            person_id,
            candidate_snapshot_id,
            raw_company_name,
            normalized_company_name,
            raw_position_name,
            normalized_position_name,
            position_description,
            position_start_date,
            position_end_date,
            experience_type,
            is_current,
        )

        # 已存在时直接复用。
        if person_position_key in person_position_id_by_key:
            person_position_id = person_position_id_by_key[
                person_position_key
            ]
            person_position_action = "reused"
            reused_person_position_count += 1
        else:
            # 新记录的 created_at 和 updated_at 使用相同时间。
            current_timestamp = now_utc_iso()

            # 插入 Person_Position。
            inserted_rows = result_rows(
                execute_d1(
                    """
                    INSERT INTO "Person_Position" (
                      "person_id",
                      "company_id",
                      "position_id",
                      "function_id",
                      "seniority_id",
                      "experience_level_id",
                      "location_id",
                      "position_start_date",
                      "position_end_date",
                      "experience_type",
                      "is_current",
                      "created_at",
                      "updated_at",
                      "source_candidate_id",
                      "raw_company_name",
                      "normalized_company_name",
                      "raw_position_name",
                      "normalized_position_name",
                      "position_description"
                    )
                    VALUES (
                      ?, NULL, NULL, NULL, NULL, NULL, NULL,
                      ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
                    )
                    RETURNING "id";
                    """,
                    [
                        person_id,
                        position_start_date,
                        position_end_date,
                        experience_type,
                        is_current,
                        current_timestamp,
                        current_timestamp,
                        candidate_snapshot_id,
                        raw_company_name,
                        normalized_company_name,
                        raw_position_name,
                        normalized_position_name,
                        position_description,
                    ],
                )
            )

            # INSERT 必须返回唯一 id。
            if len(
                inserted_rows
            ) != 1:
                raise RuntimeError(
                    "Person_Position INSERT 没有返回唯一 id。"
                )

            # 保存新 id。
            person_position_id = int(
                inserted_rows[
                    0
                ][
                    "id"
                ]
            )

            # 立即更新本地 mapping，处理同一次运行内的复用。
            person_position_id_by_key[
                person_position_key
            ] = person_position_id
            person_position_action = "inserted"
            inserted_person_position_count += 1

        # 建立 Candidate bridge key。
        candidate_position_key = (
            candidate_snapshot_id,
            person_position_id,
        )

        # 已存在 bridge 时复用。
        if candidate_position_key in candidate_position_id_by_key:
            candidate_position_id = candidate_position_id_by_key[
                candidate_position_key
            ]
            candidate_position_action = "reused"
            reused_candidate_position_count += 1
        else:
            # 插入 Candidate_Position bridge。
            inserted_bridge_rows = result_rows(
                execute_d1(
                    """
                    INSERT INTO "Candidate_Position" (
                      "candidate_id",
                      "person_position_id",
                      "created_at"
                    )
                    VALUES (?, ?, ?)
                    RETURNING "id";
                    """,
                    [
                        candidate_snapshot_id,
                        person_position_id,
                        now_utc_iso(),
                    ],
                )
            )

            # INSERT 必须返回唯一 id。
            if len(
                inserted_bridge_rows
            ) != 1:
                raise RuntimeError(
                    "Candidate_Position INSERT 没有返回唯一 id。"
                )

            # 保存新 bridge id。
            candidate_position_id = int(
                inserted_bridge_rows[
                    0
                ][
                    "id"
                ]
            )

            # 立即更新本地 mapping。
            candidate_position_id_by_key[
                candidate_position_key
            ] = candidate_position_id
            candidate_position_action = "inserted"
            inserted_candidate_position_count += 1

        # 保存成功结果。
        employment_dataframe.at[
            dataframe_index,
            "person_position_id",
        ] = person_position_id
        employment_dataframe.at[
            dataframe_index,
            "candidate_position_id",
        ] = candidate_position_id
        employment_dataframe.at[
            dataframe_index,
            "employment_import_status",
        ] = "success"
        employment_dataframe.at[
            dataframe_index,
            "employment_import_action",
        ] = (
            f"person_position_{person_position_action};"
            f"candidate_position_{candidate_position_action}"
        )
        employment_dataframe.at[
            dataframe_index,
            "employment_import_error",
        ] = None
        employment_dataframe.at[
            dataframe_index,
            "employment_imported_at",
        ] = now_utc_iso()

    except Exception as error:
        # 单条失败时写入错误并继续下一条。
        failed_record_count += 1
        employment_dataframe.at[
            dataframe_index,
            "employment_import_status",
        ] = "failed"
        employment_dataframe.at[
            dataframe_index,
            "employment_import_action",
        ] = None
        employment_dataframe.at[
            dataframe_index,
            "employment_import_error",
        ] = str(
            error
        )
        employment_dataframe.at[
            dataframe_index,
            "employment_imported_at",
        ] = now_utc_iso()

    # 每100条和最后一条显示进度。
    processed_record_count = (
        dataframe_index + 1
    )
    if (
        processed_record_count % 100 == 0
        or processed_record_count
        == len(
            employment_dataframe
        )
    ):
        print(
            "Processed Candidate Positions:",
            f"{processed_record_count}/"
            f"{len(employment_dataframe)}",
        )


# ============================================================
# 回填 Candidate 级状态 CSV
# ============================================================

# 按 Candidate 统计成功数量。
successful_count_by_candidate = (
    employment_dataframe.loc[
        employment_dataframe[
            "employment_import_status"
        ].eq(
            "success"
        )
    ]
    .groupby(
        "candidate_snapshot_id"
    )
    .size()
    .to_dict()
)

# 按 Candidate 统计失败数量。
failed_count_by_candidate = (
    employment_dataframe.loc[
        employment_dataframe[
            "employment_import_status"
        ].eq(
            "failed"
        )
    ]
    .groupby(
        "candidate_snapshot_id"
    )
    .size()
    .to_dict()
)

# 每次运行都重新计算状态结果 columns。
for status_column_name in [
    "employment_import_status",
    "employment_import_success_count",
    "employment_import_failed_count",
]:
    employment_status_dataframe[
        status_column_name
    ] = None

# 逐个 Candidate 回填状态。
for status_index, status_row in (
    employment_status_dataframe.iterrows()
):
    # 取得 Candidate id。
    candidate_snapshot_id = require_python_int(
        status_row[
            "candidate_snapshot_id"
        ],
        "candidate_snapshot_id",
    )

    # 取得成功和失败数量。
    success_count = int(
        successful_count_by_candidate.get(
            candidate_snapshot_id,
            0,
        )
    )
    failure_count = int(
        failed_count_by_candidate.get(
            candidate_snapshot_id,
            0,
        )
    )

    # 有任意失败记录时标记 failed。
    if failure_count > 0:
        import_status = "failed"
    # 没有提取记录时不需要导入。
    elif int(
        status_row[
            "employment_record_count"
        ]
    ) == 0:
        import_status = "rejected"
    # 有记录且全部成功时标记 success。
    else:
        import_status = "success"

    # 保存 Candidate 汇总状态。
    employment_status_dataframe.at[
        status_index,
        "employment_import_status",
    ] = import_status
    employment_status_dataframe.at[
        status_index,
        "employment_import_success_count",
    ] = success_count
    employment_status_dataframe.at[
        status_index,
        "employment_import_failed_count",
    ] = failure_count


# ============================================================
# 数据库结果验证
# ============================================================

# 取得所有成功明细。
successful_employment_dataframe = (
    employment_dataframe.loc[
        employment_dataframe[
            "employment_import_status"
        ].eq(
            "success"
        )
    ]
)

# 成功记录必须同时存在两个数据库 ids。
if (
    successful_employment_dataframe[
        "person_position_id"
    ].isna().any()
    or successful_employment_dataframe[
        "candidate_position_id"
    ].isna().any()
):
    raise RuntimeError(
        "成功工作经历中存在缺失的数据库 id。"
    )

# 所有明细都应该成功；任何失败都会阻止覆盖正式 CSV。
if failed_record_count != 0:
    # 先输出失败样本，方便定位。
    display(
        employment_dataframe.loc[
            employment_dataframe[
                "employment_import_status"
            ].eq(
                "failed"
            )
        ][
            [
                "candidate_snapshot_id",
                "raw_company_name",
                "raw_position_name",
                "employment_import_error",
            ]
        ].head(
            20
        )
    )

    # 停止，不用不完整结果覆盖正式 CSV。
    raise RuntimeError(
        "存在工作经历导入失败记录："
        f"{failed_record_count}。"
        "正式 CSV 尚未覆盖。"
    )

# 查询全库外键错误。
foreign_key_violations = result_rows(
    execute_d1(
        "PRAGMA foreign_key_check;"
    )
)

# 出现外键错误时停止。
if foreign_key_violations:
    raise RuntimeError(
        "D1 存在 foreign key violations："
        f"{foreign_key_violations[:20]}"
    )


# ============================================================
# 原子覆盖两份固定 CSV
# ============================================================

# 覆盖工作经历明细 CSV。
atomic_write_csv(
    employment_dataframe,
    EMPLOYMENT_EXTRACTION_CSV_PATH,
)

# 覆盖 Candidate 工作经历状态 CSV。
atomic_write_csv(
    employment_status_dataframe,
    EMPLOYMENT_STATUS_CSV_PATH,
)


# ============================================================
# 显示最终结果
# ============================================================

# 显示导入统计。
print(
    "Candidate Employment import summary:",
    {
        "planned_employment_records": int(
            len(
                employment_dataframe
            )
        ),
        "successful_employment_records": int(
            len(
                successful_employment_dataframe
            )
        ),
        "failed_employment_records": int(
            failed_record_count
        ),
        "inserted_person_positions": int(
            inserted_person_position_count
        ),
        "reused_person_positions": int(
            reused_person_position_count
        ),
        "inserted_candidate_positions": int(
            inserted_candidate_position_count
        ),
        "reused_candidate_positions": int(
            reused_candidate_position_count
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_violations
            )
        ),
    },
)

# 显示前10条导入结果。
display(
    employment_dataframe[
        [
            "candidate_snapshot_id",
            "person_id",
            "raw_company_name",
            "raw_position_name",
            "position_start_date",
            "position_end_date",
            "person_position_id",
            "candidate_position_id",
            "employment_import_status",
            "employment_import_action",
        ]
    ].head(
        10
    )
)

# 显示完成信息。
print(
    "Candidate Employment import completed successfully."
)


### 简历 project 数据提取生成csv + 存入 Google Drive

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 从 D1 Candidate_snapshot.raw_resume_text 中
# 使用确定性规则提取简历 Project records，并覆盖生成两个 CSV。
#
# 输出：
# 1. candidate_project_rule_extraction.csv
#    每一行代表一段从明确 Project section 中提取出的项目经历。
#
# 2. candidate_project_rule_extraction_status.csv
#    每一行代表一个 Candidate 的 Project 提取状态。
#
# 重要：
# 1. 本 Cell 不依赖其他 Colab Cell 的 variable、function 或 class。
# 2. 本 Cell 不使用模型。
# 3. 本 Cell 只读取 D1，不写入 Person_Project 或 Candidate_Project。
# 4. 本 Cell 不依赖正在运行的 Skill Cell 或 Skill 导入结果 columns。
# 5. 本 Cell 只处理能够识别出明确 Project section 的简历内容。
# 6. 项目名称无法可靠确定时，不生成正式 Project record。
# 7. 项目日期、角色和 URL 缺失不会阻止 Project record 被提取。
# 8. 固定覆盖同名 CSV，因此重复运行不会无限增加文件。
# 9. 相同 D1 数据和相同规则会产生相同排序和相同结果。
# 10. 本 Cell 的结果应先经过统计和抽样检查，再写入数据库。
# 11. v2 不再按空行或 bullet 后的普通文本直接切分新项目。
# 12. v2 使用明确 Project section 加严格标题格式建立 Project record。
# 13. 日期、URL和角色都是可选字段，缺失时保存 SQL NULL。
# 14. v2 对 Leadership & Project Experience 混合 section 使用更严格规则。
# ============================================================

# os 用于创建 Google Drive 输出目录和安全替换 CSV。
import os

# re 用于 Project section、URL 和日期的确定性匹配。
import re

# time 只用于 D1 网络请求失败后的等待重试。
import time

# pandas 用于生成结构化结果和保存 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定规则版本和 Google Drive 路径
# ============================================================

# 保存当前 Project 提取规则版本。
RULE_VERSION = (
    "candidate_project_rule_v4"
)

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 确保输出目录存在。
os.makedirs(
    DATASET_FOLDER,
    exist_ok=True,
)

# 保存已经成功生成 Application records 的固定 CSV。
APPLICATION_CSV_PATH = (
    f"{DATASET_FOLDER}/dedup_results/"
    "application_dedup_v2/application.csv"
)

# Application CSV 不存在时停止。
if not os.path.exists(
    APPLICATION_CSV_PATH
):
    raise FileNotFoundError(
        "找不到必要的 application.csv："
        f"{APPLICATION_CSV_PATH}"
    )

# 保存 Project record 级提取结果。
PROJECT_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_project_rule_extraction.csv"
)

# 保存 Candidate 级 Project 提取状态。
PROJECT_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_project_rule_extraction_status.csv"
)


# ============================================================
# Cloudflare D1 独立连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 API endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 D1 request headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP connection。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行只读 D1 SQL，并对临时网络错误有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有重试机会时等待。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 重试用完后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON。
            payload = response.json()
        except ValueError as error:
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 SQL 都成功时返回 result。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 Cloudflare 或 SQL 错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部失败后停止。
    raise RuntimeError(
        "D1 query 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    d1_result,
):
    """从 D1 result 中提取第一条 statement 的 results rows。"""

    # 没有 result 时返回空 list。
    if not d1_result:
        return []

    # 返回 results rows。
    return d1_result[0].get(
        "results",
        [],
    )


def clean_text_value(
    value,
):
    """把数据库空值和空字符串统一成 None。"""

    # Python None 保持为空。
    if value is None:
        return None

    # 清理首尾空白。
    cleaned_value = str(
        value
    ).strip()

    # 空字符串保持为空。
    if not cleaned_value:
        return None

    # 返回清理后的文本。
    return cleaned_value


def normalize_resume_text(
    raw_resume_text,
):
    """统一简历换行符，并清理每一行首尾空白。"""

    # 清理输入。
    cleaned_resume_text = clean_text_value(
        raw_resume_text
    )

    # 没有简历文本时返回 None。
    if cleaned_resume_text is None:
        return None

    # 统一不同操作系统的换行符。
    normalized_newlines = (
        cleaned_resume_text
        .replace(
            "\r\n",
            "\n",
        )
        .replace(
            "\r",
            "\n",
        )
    )

    # 清理每一行首尾空白，但保留原始行顺序。
    normalized_lines = [
        line.strip()
        for line
        in normalized_newlines.split(
            "\n"
        )
    ]

    # 连续空行最多保留一行。
    compact_lines = []

    # 逐行压缩空行。
    for line in normalized_lines:
        # 连续空行不重复保存。
        if (
            not line
            and compact_lines
            and not compact_lines[-1]
        ):
            continue

        # 保存当前行。
        compact_lines.append(
            line
        )

    # 返回清理后的文本。
    return "\n".join(
        compact_lines
    ).strip()


def normalize_match_text(
    value,
):
    """生成稳定的小写文本，只用于标题和关键词规则判断。"""

    # 清理输入。
    cleaned_value = clean_text_value(
        value
    )

    # 空值返回 None。
    if cleaned_value is None:
        return None

    # 合并连续空白并小写化。
    return " ".join(
        cleaned_value.split()
    ).casefold()


def python_int(
    value,
):
    """把 pandas、NumPy 或 CSV 数字转换成标准 Python int。"""

    # 空值不能转换。
    if pd.isna(
        value
    ):
        return None

    # 兼容 CSV 中的 1 和 1.0。
    return int(
        float(
            str(
                value
            ).strip()
        )
    )


# ============================================================
# Project section 和日期规则
# ============================================================

# 明确的 Project section 标题。
PROJECT_SECTION_HEADER_PATTERN = re.compile(
    r"^(?:"
    r"projects?|"
    r"project\s+experience|"
    r"project\s+experiences|"
    r"selected\s+projects?|"
    r"academic\s+projects?|"
    r"personal\s+projects?|"
    r"technical\s+projects?|"
    r"research\s+projects?|"
    r"relevant\s+projects?|"
    r"leadership\s*(?:&|and)\s*project\s+experience"
    r")$",
    re.IGNORECASE,
)

# 能够结束 Project section 的其他简历 section 标题。
OTHER_SECTION_HEADER_PATTERN = re.compile(
    r"^(?:"
    r"education|"
    r"academic\s+background|"
    r"work\s+experience|"
    r"professional\s+experience|"
    r"employment\s+experience|"
    r"experience|"
    r"internships?|"
    r"internship\s+experience|"
    r"skills?|"
    r"technical\s+skills?|"
    r"user[- ]centered\s+design\s+skills?|"
    r"additional\s+skills?|"
    r"core\s+skills?|"
    r"computer\s+skills?|"
    r"programming\s+skills?|"
    r"skill\s+sets?|"
    r"skill\s+highlights?|"
    r"skills?\s*/\s*interests?|"
    r"skills?\s*/\s*activities|"
    r"skills?\s*(?:&|and)\s*hobbies|"
    r"skills?\s*(?:&|and)\s*courseworks?|"
    r"skills?\s*(?:&|and)\s*leadership\s+activities|"
    r"skills?\s*(?:&|and)\s*certifications?(?::.*)?|"
    r"skills?\s*(?:&|and)\s*interests?|"
    r"skills?\s*(?:&|and)\s*activities|"
    r"skills?\s*,\s*certifications?\s*(?:&|and)\s*interests?|"
    r"languages?|"
    r"languages?\s*(?:&|and)\s*skills?|"
    r"certifications?|"
    r"awards?(?:\s+and\s+honors?)?|"
    r"honors?(?:\s+and\s+awards?)?|"
    r"activities|"
    r"activities?\s*/\s*interests?|"
    r"activities?\s*(?:&|and)\s*interests?|"
    r"extra[- ]?curricular\s+activit(?:y|ies)|"
    r"extra[- ]?curricular\s+experiences?|"
    r"extra[- ]?curricular\s+activities\s+and\s+social\s+practice|"
    r"extracurricular|"
    r"extra[- ]?curricular\s*(?:&|and)\s*volunteer|"
    r"activities\s*(?:&|and)\s*leadership|"
    r"leadership\s*(?:&|and)\s*extra[- ]?curricular\s+activities|"
    r"activities\s*(?:&|and)\s*volunteer(?:ing)?|"
    r"activites\s*(?:&|and)\s*volunteer(?:ing)?|"
    r"activities?\s*/\s*volunteer(?:ing)?|"
    r"extracellular\s+activit(?:y|ies)|"
    r"leadership\s*(?:&|and)\s*activities|"
    r"leadership\s+activities|"
    r"leadership\s+experience|"
    r"leadership\s+experience\s*(?:&|and)\s*activities|"
    r"leadership\s*(?:&|and)\s*(?:volunteer|voluntory)\s+experience|"
    r"leadership\s*(?:&|and)\s*community\s+involvement|"
    r"leadership\s*(?:&|and)\s*involvement|"
    r"leadership\s*(?:&|and)\s*interests?\s*[a-z]?|"
    r"leadership\s+and\s+extracurricular|"
    r"leadership\s*/\s*extra[- ]?curricular\s+activities"
    r"(?:\s*/\s*volunteer)?"
    r"(?:\s*/\s*community\s+service)?|"
    r"awards?\s*(?:&|and)\s*activities|"
    r"honors?\s*(?:&|and)\s*activities|"
    r"leadership|"
    r"volunteer(?:ing)?|"
    r"publications?|"
    r"research\s+experience|"
    r"teaching\s+assistant|"
    r"campus\s+involvement|"
    r"additional\s+experience|"
    r"additional\s+information|"
    r"additional\s+details|"
    r"professional\s+summary|"
    r"summary|"
    r"profile|"
    r"interests?|"
    r"references?"
    r")$",
    re.IGNORECASE,
)

# 常见月份文本。
MONTH_PATTERN_TEXT = (
    r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|"
    r"may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|"
    r"oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
)

# 匹配文本中的年月、年份和 Present。
DATE_TOKEN_PATTERN = re.compile(
    rf"\b(?:"
    rf"{MONTH_PATTERN_TEXT}\.?\s+\d{{4}}|"
    rf"{MONTH_PATTERN_TEXT}\.?\s+of\s+\d{{4}}|"
    rf"\d{{1,2}}[/-]\d{{4}}|"
    rf"\d{{4}}|"
    rf"present|current|ongoing"
    rf")\b",
    re.IGNORECASE,
)

# 匹配只包含日期范围的整行。
DATE_ONLY_LINE_PATTERN = re.compile(
    rf"^(?:"
    rf"(?:spring|summer|fall|autumn|winter)\s+\d{{4}}|"
    rf"{MONTH_PATTERN_TEXT}\.?\s+\d{{4}}|"
    rf"{MONTH_PATTERN_TEXT}\.?\s+of\s+\d{{4}}|"
    rf"\d{{1,2}}[/-]\d{{4}}|"
    rf"\d{{4}}"
    rf")"
    rf"(?:\s*(?:-|–|—|to)\s*"
    rf"(?:present|current|ongoing|"
    rf"(?:spring|summer|fall|autumn|winter)\s+\d{{4}}|"
    rf"{MONTH_PATTERN_TEXT}\.?\s+\d{{4}}|"
    rf"\d{{1,2}}[/-]\d{{4}}|\d{{4}}))?$",
    re.IGNORECASE,
)

# 匹配项目 URL。
URL_PATTERN = re.compile(
    r"https?://[^\s)\]>]+|"
    r"(?:www\.)[^\s)\]>]+",
    re.IGNORECASE,
)

# 常见项目角色关键词。
PROJECT_ROLE_PATTERN = re.compile(
    r"\b(?:"
    r"team\s+lead|project\s+lead(?:er)?|technical\s+lead|"
    r"project\s+manager|product\s+manager|"
    r"researcher|research\s+assistant|"
    r"group\s+research\s+fellow\s+leader|"
    r"literature\s+reviewer|"
    r"teaching\s+assistant|"
    r"developer|software\s+engineer|data\s+scientist|"
    r"data\s+analyst|designer|consultant|"
    r"team\s+member|contributor|founder|co-founder"
    r")\b",
    re.IGNORECASE,
)

# 项目描述 bullet 的开头符号。
BULLET_PREFIX_PATTERN = re.compile(
    r"^[•●▪◦‣·\-–—*]+\s*"
)

# 可能出现在 section 内但不应作为项目名称的标签。
NON_PROJECT_TITLE_PATTERN = re.compile(
    r"^(?:"
    r"technologies?|tech\s+stack|tools?|skills?|"
    r"responsibilities|achievements?|description|"
    r"date|duration|location|github|link|website|project\s+link|"
    r"advisor|mentor|supervisor|objective|individual\s+works?"
    r")\s*:?\s*$",
    re.IGNORECASE,
)

# 明确属于技能、兴趣、培训等内容的行前缀。
NON_PROJECT_CONTENT_PREFIX_PATTERN = re.compile(
    r"^(?:"
    r"skills?|languages?|interests?|training|certifications?|"
    r"coursework|courses?|activities|volunteer(?:ing)?|honors?|awards?|"
    r"advisor|advised\s+by|mentor|supervisor|objective|"
    r"individual\s+works?|"
    r"(?:project|github|pygame)\s+link"
    r")\s*(?:[:|,-]|$)",
    re.IGNORECASE,
)

# 明确的学历行或单纯学校名称不作为Project名称。
EDUCATION_CONTENT_LINE_PATTERN = re.compile(
    r"^(?:"
    r"(?:m\.?s\.?|b\.?s\.?|b\.?a\.?|m\.?a\.?|"
    r"ph\.?d\.?|mba|master|bachelor|doctorate)\b|"
    r"(?:courses?|gpa)\s*:|"
    r".*\b(?:university|college|school)\b(?:\s*,.*)?$"
    r")",
    re.IGNORECASE,
)

# 描述句常见的开头动作词。
#
# 这些行通常是 Project bullet description，
# 不应该被当成新的项目名称。
PROJECT_DESCRIPTION_ACTION_PATTERN = re.compile(
    r"^(?:"
    r"led|built|developed|created|designed|implemented|conducted|"
    r"analyzed|analysed|managed|worked|collaborated|co-authored|"
    r"assisted|utilized|used|performed|generated|identified|"
    r"researched|oversaw|supported|engineered|trained|deployed|"
    r"achieved|improved|constructed|programmed|established|"
    r"presented|produced|provided|participated|optimized|leveraged|"
    r"wrote|made|explored|evaluated|forecasted|modelled|modeled|"
    r"proposed|launched|performed|predicted|calculated|cleaned|"
    r"collected|processed|visualized|visualised|applied|created|"
    r"responsible|coordinated|contributed|completed|demonstrated|"
    r"provide|provided|devise|devised|deliver|delivered|"
    r"fine[- ]tuned|respectively|a\s+basic|"
    r"prepare|prepared|create|design|implement|analyze|analyse|"
    r"build|manage|support|assist|conduct|utilize|use|perform"
    r")\b",
    re.IGNORECASE,
)

# 常见地点行。
LOCATION_LINE_PATTERN = re.compile(
    r"^(?:"
    r"[A-Za-z .'-]+,\s*[A-Z]{2}|"
    r"[A-Za-z .'-]+,\s*[A-Za-z .'-]+|"
    r"remote|hybrid|on[- ]site"
    r")$",
    re.IGNORECASE,
)

# 常见 Project 名称证据。
PROJECT_NAME_EVIDENCE_PATTERN = re.compile(
    r"\b(?:"
    r"project|capstone|competition|case\s+competition|hackathon|"
    r"research|study|analysis|analytics|model|prediction|forecast|"
    r"system|platform|application|app|website|dashboard|database|"
    r"game|simulation|portfolio|prototype|tool|design|robot|"
    r"visualization|visualisation|classifier|detection|recognition|"
    r"optimization|optimisation|algorithm|network|engine|pipeline"
    r")\b",
    re.IGNORECASE,
)

# 混合 Leadership & Project section 的标题。
MIXED_PROJECT_SECTION_PATTERN = re.compile(
    r"^leadership\s*(?:&|and)\s*project\s+experience$",
    re.IGNORECASE,
)

# 混合 Leadership & Project section 使用的更强项目名称证据。
MIXED_SECTION_PROJECT_NAME_EVIDENCE_PATTERN = re.compile(
    r"\b(?:"
    r"project|capstone|competition|case\s+competition|hackathon|"
    r"study|analysis|analytics|model|prediction|forecast|system|"
    r"platform|application|app|website|dashboard|database|game|"
    r"simulation|portfolio|prototype|tool|design|robot|"
    r"visualization|visualisation|classifier|detection|recognition|"
    r"optimization|optimisation|algorithm|network|engine|pipeline"
    r")\b",
    re.IGNORECASE,
)


def is_section_header(
    line,
):
    """判断一行是否是明确的 Project 或其他 section 标题。"""

    # 清理行文本。
    cleaned_line = clean_text_value(
        line
    )

    # 空行不是标题。
    if cleaned_line is None:
        return False

    # PDF文本经常把标题和地点放在同一行，并用大量空格分隔。
    #
    # 标题判断只使用第一个布局区域，避免地点导致长度和词数异常。
    cleaned_line = re.split(
        r"\s{2,}",
        cleaned_line,
        maxsplit=1,
    )[0].strip()

    # 去掉结尾冒号后判断。
    header_candidate = cleaned_line.rstrip(
        ":"
    ).strip()

    # 匹配任一 section 标题。
    return bool(
        PROJECT_SECTION_HEADER_PATTERN.fullmatch(
            header_candidate
        )
        or OTHER_SECTION_HEADER_PATTERN.fullmatch(
            header_candidate
        )
    )


def find_project_sections(
    resume_text,
):
    """从简历中截取每一个明确的 Project section。"""

    # 拆分成行。
    lines = resume_text.split(
        "\n"
    )

    # 保存 Project sections。
    project_sections = []

    # 当前扫描位置。
    line_index = 0

    # 扫描所有行。
    while line_index < len(
        lines
    ):
        # 清理当前行。
        current_line = clean_text_value(
            lines[
                line_index
            ]
        )

        # 空行继续。
        if current_line is None:
            line_index += 1
            continue

        # 去掉标题结尾冒号。
        header_candidate = current_line.rstrip(
            ":"
        ).strip()

        # 不是 Project section 时继续。
        if not PROJECT_SECTION_HEADER_PATTERN.fullmatch(
            header_candidate
        ):
            line_index += 1
            continue

        # 保存 section 标题。
        section_header = current_line

        # Project 内容从标题下一行开始。
        content_start_index = (
            line_index + 1
        )

        # 默认结束在简历最后。
        content_end_index = len(
            lines
        )

        # 向后寻找下一个明确 section 标题。
        search_index = content_start_index

        # 扫描后续行。
        while search_index < len(
            lines
        ):
            # 清理候选行。
            candidate_line = clean_text_value(
                lines[
                    search_index
                ]
            )

            # 找到其他 section 标题时结束。
            if (
                candidate_line is not None
                and is_section_header(
                    candidate_line
                )
            ):
                content_end_index = (
                    search_index
                )
                break

            # 继续扫描。
            search_index += 1

        # 提取 section 内容。
        section_lines = [
            clean_text_value(
                section_line
            )
            for section_line
            in lines[
                content_start_index:
                content_end_index
            ]
        ]

        # 保留空行作为潜在 record 边界。
        project_sections.append(
            {
                "section_header": (
                    section_header
                ),
                "section_lines": (
                    section_lines
                ),
            }
        )

        # 从 section 结尾继续扫描。
        line_index = content_end_index

    # 返回所有明确 Project sections。
    return project_sections


def is_probable_project_title(
    line,
):
    """使用 v4 保守规则判断一行是否可以作为 Project 名称锚点。"""

    # 清理输入。
    cleaned_line = clean_text_value(
        line
    )

    # 空行不能作为名称。
    if cleaned_line is None:
        return False

    # PDF文本经常把项目名称和地点放在同一行，并用大量空格分隔。
    #
    # 项目名称判断只使用第一个布局区域。
    cleaned_line = re.split(
        r"\s{2,}",
        cleaned_line,
        maxsplit=1,
    )[0].strip()

    # bullet 描述不能作为名称。
    if BULLET_PREFIX_PATTERN.match(
        cleaned_line
    ):
        return False

    # 只包含日期或season+年份的行不能作为名称。
    if DATE_ONLY_LINE_PATTERN.fullmatch(
        cleaned_line
    ):
        return False

    # 兼容 May.2023-May.2025、Jun.-Aug.2021、June 等紧凑日期写法。
    date_like_remainder = re.sub(
        rf"{MONTH_PATTERN_TEXT}|"
        r"spring|summer|fall|autumn|winter|"
        r"present|current|ongoing",
        "",
        cleaned_line,
        flags=re.IGNORECASE,
    )

    # 去掉数字和日期分隔符。
    date_like_remainder = re.sub(
        r"[\d\s./,\-–—]+",
        "",
        date_like_remainder,
    )

    # 没有其他字符时说明整行只是日期。
    if not date_like_remainder:
        return False

    # 单独的 bullet 符号不是项目名称。
    if not re.sub(
        r"[•●▪◦‣·\-–—*\s]+",
        "",
        cleaned_line,
    ):
        return False

    # 纯日期行不能作为名称。
    line_without_dates = DATE_TOKEN_PATTERN.sub(
        "",
        cleaned_line,
    )

    # 去除分隔符后没有内容，说明这一行只有日期。
    if not re.sub(
        r"[\s|,;:/\-–—]+",
        "",
        line_without_dates,
    ):
        return False

    # 常见字段标签不能作为项目名称。
    if NON_PROJECT_TITLE_PATTERN.fullmatch(
        cleaned_line
    ):
        return False

    # 技能、兴趣、培训等内容行不能作为项目名称。
    if NON_PROJECT_CONTENT_PREFIX_PATTERN.match(
        cleaned_line
    ):
        return False

    # Advisor、Mentor、Individual Works 等元数据或小标题
    # 不能作为一个独立 Project 名称。
    if re.match(
        r"^(?:"
        r"advisor|advised\s+by|mentor|supervisor|objective|"
        r"individual\s+works?"
        r")\b",
        cleaned_line,
        re.IGNORECASE,
    ):
        return False

    # 学位、课程、GPA或单纯学校名称不能作为项目名称。
    if EDUCATION_CONTENT_LINE_PATTERN.match(
        cleaned_line
    ):
        return False

    # 包含学校关键词但没有明确Project语义的行不作为项目名称。
    if (
        re.search(
            r"\b(?:university|college|school)\b",
            cleaned_line,
            re.IGNORECASE,
        )
        and PROJECT_NAME_EVIDENCE_PATTERN.search(
            cleaned_line
        )
        is None
    ):
        return False

    # 带课程代码的课程名称不作为独立项目。
    if re.search(
        r"\([A-Z]{2,}(?:-[A-Z]+)?\s*\d{2,4}",
        cleaned_line,
    ):
        return False

    # 明显的公司或机构名称不作为Project名称。
    if (
        re.search(
            r"\b(?:company|co\.?|ltd\.?|llc|inc\.?|group)\b",
            cleaned_line,
            re.IGNORECASE,
        )
        and PROJECT_NAME_EVIDENCE_PATTERN.search(
            cleaned_line
        )
        is None
    ):
        return False

    # 三个或更多逗号分隔的行通常是课程或技术栈列表。
    if (
        cleaned_line.count(
            ","
        ) >= 3
    ):
        return False

    # 明确角色行不能作为项目名称。
    if (
        PROJECT_ROLE_PATTERN.fullmatch(
            cleaned_line.strip(
                " |,;:-–—"
            )
        )
        is not None
    ):
        return False

    # 较短且只是在列举一个或多个角色的行也不能作为项目名称。
    if (
        PROJECT_ROLE_PATTERN.search(
            cleaned_line
        )
        is not None
        and len(
            cleaned_line.split()
        ) <= 7
        and PROJECT_NAME_EVIDENCE_PATTERN.search(
            cleaned_line
        )
        is None
    ):
        return False

    # 常见地点行不能作为项目名称。
    if LOCATION_LINE_PATTERN.fullmatch(
        cleaned_line
    ):
        return False

    # 地点后面附带日期的布局行仍然只是地点与时间，
    # 例如 New York, NY Jan 2024 - Present。
    if re.match(
        r"^[A-Za-z .'-]+,\s*[A-Z]{2}\b",
        cleaned_line,
    ):
        return False

    # PDF换行可能把真正标题最后一个词与地点放到下一行，
    # 例如 Modeling | New York, NY。
    #
    # 分隔符左边只有一到两个词时，把它视为上一行标题的续行，
    # 不单独创建 Project record。
    layout_location_match = re.match(
        r"^(.+?)\s*\|\s*"
        r"[A-Za-z .'-]+,\s*[A-Z]{2}\b",
        cleaned_line,
    )

    # 短续行不作为独立名称。
    if (
        layout_location_match
        and len(
            layout_location_match.group(
                1
            ).split()
        ) <= 2
    ):
        return False

    # 以动作词开头的句子默认属于项目描述。
    if PROJECT_DESCRIPTION_ACTION_PATTERN.match(
        cleaned_line
    ):
        return False

    # 邮箱或电话样式不应作为项目名称。
    if (
        "@" in cleaned_line
        or re.fullmatch(
            r"[\d\s()+.\-]+",
            cleaned_line,
        )
    ):
        return False

    # v4 继续使用严格的最大长度。
    if len(
        cleaned_line
    ) > 110:
        return False

    # v4 标题词数不能超过14个。
    if len(
        cleaned_line.split()
    ) > 14:
        return False

    # 以句号结束、并且不包含日期月份的行通常是完整描述句，
    # 而不是 Project 名称。
    #
    # 月份缩写本身可能以句号结束，例如 Jan.，因此不能误删。
    if (
        cleaned_line.rstrip(
            "\"'”’"
        ).endswith(
            "."
        )
        and re.search(
            MONTH_PATTERN_TEXT,
            cleaned_line,
            re.IGNORECASE,
        )
        is None
    ):
        return False

    # 包含分号的行通常是描述或技术栈。
    if ";" in cleaned_line:
        return False

    # 至少需要一个字母。
    if not re.search(
        r"[A-Za-z]",
        cleaned_line,
    ):
        return False

    # 提取英文单词，用于判断标题大小写特征。
    alphabetic_words = re.findall(
        r"[A-Za-z][A-Za-z0-9+#.'-]*",
        line_without_dates,
    )

    # 没有英文单词时不接受。
    if not alphabetic_words:
        return False

    # 单个普通技能词（例如 Figma、MongoDB、Survey）
    # 很容易在错误延伸的 Project section 中被误认成项目。
    #
    # 只有名称本身包含明确 Project 语义时才允许单词标题。
    if (
        len(
            alphabetic_words
        ) == 1
        and PROJECT_NAME_EVIDENCE_PATTERN.search(
            cleaned_line
        )
        is None
    ):
        return False

    # 统计大写字母开头的单词。
    title_case_word_count = sum(
        1
        for word
        in alphabetic_words
        if word[0].isupper()
    )

    # 计算标题式大小写比例。
    title_case_ratio = (
        title_case_word_count
        / len(
            alphabetic_words
        )
    )

    # 全大写行可以作为标题。
    is_uppercase_title = bool(
        re.search(
            r"[A-Z]",
            line_without_dates,
        )
        and line_without_dates.upper()
        == line_without_dates
    )

    # v4要求项目标题本身必须具备明显标题格式。
    #
    # Project语义只作为后续上下文证据，
    # 不能让普通描述句仅因为出现 model、analysis 等词就成为标题。
    return bool(
        title_case_ratio >= 0.40
        or is_uppercase_title
    )


def split_project_section_into_records(
    project_section,
):
    """
    把一个明确 Project section 切分成 Project record blocks。

    v4 不再把空行或 bullet 后的普通句子直接当成新项目。
    只有通过严格标题规则的行才能成为 Project record 起点。
    """

    # 读取 section lines。
    section_lines = [
        clean_text_value(
            section_line
        )
        for section_line
        in project_section[
            "section_lines"
        ]
    ]

    # 去掉空行和单独出现的 bullet 符号。
    lines = [
        line
        for line
        in section_lines
        if line is not None
        and re.sub(
            r"[•●▪◦‣·\-–—*\s]+",
            "",
            line,
        )
    ]

    # 没有内容时返回空 list。
    if not lines:
        return []

    # 保存严格识别出来的标题行 indexes。
    title_line_indexes = []

    # 逐行寻找Project标题锚点。
    for line_index, line in enumerate(
        lines
    ):
        # 不符合严格标题规则时继续。
        if not is_probable_project_title(
            line
        ):
            continue

        # 如果当前短标题实际上已经包含在最近一个项目名称中，
        # 它通常是PDF换行后重复出现的标题尾部。
        #
        # 例如完整标题已经包含 System (AURS)，
        # 后文描述再次换行出现 System(AURS) 时不应新建record。
        normalized_title_candidate = (
            normalize_match_text(
                line
            )
        )

        # 默认当前行不是重复的标题尾部。
        is_repeated_title_suffix = False

        # 只与最近一个已接受标题比较。
        if title_line_indexes:
            previous_title = lines[
                title_line_indexes[
                    -1
                ]
            ]

            # 标准化最近一个标题。
            normalized_previous_title = (
                normalize_match_text(
                    previous_title
                )
            )

            # 去掉空格和标点后比较，兼容
            # System(AURS) 与 System (AURS) 这种排版差异。
            compact_title_candidate = re.sub(
                r"[^a-z0-9]+",
                "",
                normalized_title_candidate,
            )

            # 同样压缩最近一个标题。
            compact_previous_title = re.sub(
                r"[^a-z0-9]+",
                "",
                normalized_previous_title,
            )

            # 短候选完整包含在最近标题中时视为重复尾部。
            is_repeated_title_suffix = bool(
                compact_title_candidate
                and compact_previous_title
                and len(
                    normalized_title_candidate.split()
                ) <= 2
                and compact_title_candidate
                in compact_previous_title
            )

        # 重复标题尾部不建立新的record。
        if is_repeated_title_suffix:
            continue

        # 通过严格标题规则后直接保存标题锚点。
        #
        # 日期和URL只是可选提取字段，
        # 不再作为Project record是否成立的必要条件。
        title_line_indexes.append(
            line_index
        )

    # 没有任何可靠标题时返回空 list。
    if not title_line_indexes:
        return []

    # 保存切分后的blocks。
    record_blocks = []

    # 按标题锚点切分，每个block延续到下一个标题之前。
    for title_position, title_line_index in enumerate(
        title_line_indexes
    ):
        # 最后一个标题延续到section结尾。
        if (
            title_position + 1
            == len(
                title_line_indexes
            )
        ):
            block_end_index = len(
                lines
            )
        else:
            # 其他标题延续到下一个标题之前。
            block_end_index = (
                title_line_indexes[
                    title_position + 1
                ]
            )

        # 保存完整block。
        record_blocks.append(
            lines[
                title_line_index:
                block_end_index
            ]
        )

    # 返回候选 blocks。
    return record_blocks


def extract_project_dates(
    raw_project_text,
):
    """尽力取得项目开始和结束日期原始 token。"""

    # 找到所有日期 token。
    date_tokens = [
        match.group(
            0
        ).strip()
        for match
        in DATE_TOKEN_PATTERN.finditer(
            raw_project_text
        )
    ]

    # 没有日期时两个字段都为空。
    if not date_tokens:
        return (
            None,
            None,
        )

    # 只有一个日期时暂时作为 start_date。
    if len(
        date_tokens
    ) == 1:
        return (
            date_tokens[0],
            None,
        )

    # 两个或更多日期时使用第一个和第二个。
    return (
        date_tokens[0],
        date_tokens[1],
    )


def extract_project_url(
    raw_project_text,
):
    """提取项目 block 中的第一个 URL。"""

    # 搜索 URL。
    url_match = URL_PATTERN.search(
        raw_project_text
    )

    # 没找到时返回 None。
    if url_match is None:
        return None

    # 清理常见结尾标点。
    return url_match.group(
        0
    ).rstrip(
        ".,;:"
    )


def extract_project_role(
    project_lines,
):
    """从项目 block 前三行中尽力提取明确角色文本。"""

    # 只检查前面三行，降低从描述中误取角色的风险。
    for line in project_lines[
        :3
    ]:
        # 搜索角色关键词。
        role_match = PROJECT_ROLE_PATTERN.search(
            line
        )

        # 找到时保留匹配到的原始文本。
        if role_match is not None:
            return role_match.group(
                0
            ).strip()

    # 没有可靠角色。
    return None


def build_project_record(
    project_lines,
    section_header,
):
    """把一个候选 block 转成结构化 Project record。"""

    # 去掉 block 中的空行。
    nonempty_lines = [
        line
        for line
        in project_lines
        if clean_text_value(
            line
        )
        is not None
    ]

    # 没有内容时拒绝。
    if not nonempty_lines:
        return (
            None,
            "empty_project_block",
        )

    # 第一行必须能够作为项目名称候选。
    first_line = nonempty_lines[
        0
    ]

    # 名称不可靠时拒绝。
    if not is_probable_project_title(
        first_line
    ):
        return (
            None,
            "no_reliable_project_name",
        )

    # 同一行存在较长PDF布局空白时，优先只保留第一个文本区域。
    #
    # 例如：
    # Oliver Wyman Impact Case Competition        New York, NY
    # 这里后面的地点不属于项目名称。
    first_line_primary_segment = re.split(
        r"\s{2,}",
        first_line,
        maxsplit=1,
    )[0]

    # 项目名称去掉同一行末尾的 URL。
    project_name_without_url = URL_PATTERN.sub(
        "",
        first_line_primary_segment,
    )

    # 如果项目名称末尾存在以左括号开始的日期片段，
    # 从日期片段开始位置整体删除。
    #
    # 这同时处理完整日期和PDF换行造成的不完整日期，例如：
    # Project Name (Aug. 2019-Dec. 2019)
    # Project Name (Aug.-Dec.
    project_name_without_parenthesized_date = re.sub(
        rf"\s*[\(\[]\s*(?:"
        rf"{MONTH_PATTERN_TEXT}|"
        r"spring|summer|fall|autumn|winter|"
        r"\d{4}"
        r").*$",
        "",
        project_name_without_url,
        flags=re.IGNORECASE,
    )

    # 项目名称去掉日期 token。
    project_name_without_date = DATE_TOKEN_PATTERN.sub(
        "",
        project_name_without_parenthesized_date,
    )

    # 删除名称末尾仍然残留的月份范围。
    #
    # 例如删除：
    # Aug.-Dec.
    # Jan. 2019-May
    project_name_without_date = re.sub(
        rf"\s*(?:[\(\[]\s*)?"
        rf"{MONTH_PATTERN_TEXT}\.?"
        rf"(?:\s+\d{{4}})?"
        rf"(?:\s*(?:-|–|—|to)\s*"
        rf"{MONTH_PATTERN_TEXT}\.?"
        rf"(?:\s+\d{{4}})?)?"
        rf"\s*[\)\]]?\s*$",
        "",
        project_name_without_date,
        flags=re.IGNORECASE,
    )

    # 清理名称末尾常见分隔符。
    raw_project_name = (
        project_name_without_date
        .strip(
            " \t|,;:./-–—"
        )
    )

    # 删除日期清理后残留的未配对右括号。
    while (
        raw_project_name.endswith(
            ")"
        )
        and raw_project_name.count(
            ")"
        )
        > raw_project_name.count(
            "("
        )
    ):
        raw_project_name = raw_project_name[
            :-1
        ].rstrip()

    # 删除日期清理后残留的未配对右方括号。
    while (
        raw_project_name.endswith(
            "]"
        )
        and raw_project_name.count(
            "]"
        )
        > raw_project_name.count(
            "["
        )
    ):
        raw_project_name = raw_project_name[
            :-1
        ].rstrip()

    # 清理后名称为空时拒绝。
    if not raw_project_name:
        return (
            None,
            "empty_project_name_after_cleaning",
        )

    # 清理后的名称如果只是Project Link等标签，也拒绝生成record。
    if NON_PROJECT_TITLE_PATTERN.fullmatch(
        raw_project_name
    ):
        return (
            None,
            "project_name_is_field_label",
        )

    # 日期清理以后如果项目名称只剩下一个角色标签，
    # 不能把角色本身当作项目名称。
    #
    # 这不会删除真实项目中的角色信息；
    # 角色仍然可以保存在 project_role column。
    if PROJECT_ROLE_PATTERN.fullmatch(
        raw_project_name
    ):
        return (
            None,
            "project_name_is_role_label",
        )

    # 清理日期和 URL 后再次排除 Advisor、Mentor、
    # Individual Works 等元数据或小标题。
    if re.match(
        r"^(?:"
        r"advisor|advised\s+by|mentor|supervisor|objective|"
        r"individual\s+works?"
        r")\b",
        raw_project_name,
        re.IGNORECASE,
    ):
        return (
            None,
            "project_name_is_metadata",
        )

    # Leadership & Project Experience 是混合section。
    #
    # 在这种section里，普通公司名或社团名不能仅凭标题格式被当成Project；
    # 项目名称本身必须包含Project、competition、analysis、system等项目证据。
    normalized_section_header = (
        clean_text_value(
            section_header
        )
        .rstrip(
            ":"
        )
        .strip()
    )

    # 混合section中缺少项目语义时拒绝。
    if (
        MIXED_PROJECT_SECTION_PATTERN.fullmatch(
            normalized_section_header
        )
        and MIXED_SECTION_PROJECT_NAME_EVIDENCE_PATTERN.search(
            raw_project_name
        )
        is None
    ):
        return (
            None,
            "mixed_section_without_project_name_evidence",
        )

    # 标准化项目名称只用于分析和未来幂等判断。
    normalized_project_name = (
        normalize_match_text(
            raw_project_name
        )
    )

    # 保存完整原始项目 block。
    raw_project_text = "\n".join(
        nonempty_lines
    )

    # 提取角色。
    project_role = extract_project_role(
        nonempty_lines
    )

    # 提取 URL。
    project_url = extract_project_url(
        raw_project_text
    )

    # 提取日期。
    (
        start_date,
        end_date,
    ) = extract_project_dates(
        raw_project_text
    )

    # 描述保留除第一行以外的完整内容。
    project_description = (
        "\n".join(
            nonempty_lines[
                1:
            ]
        )
        if len(
            nonempty_lines
        ) > 1
        else None
    )

    # 返回结构化结果。
    return (
        {
            "project_section_header": (
                section_header
            ),
            "raw_project_text": (
                raw_project_text
            ),
            "raw_project_name": (
                raw_project_name
            ),
            "normalized_project_name": (
                normalized_project_name
            ),
            "project_role": (
                project_role
            ),
            "project_description": (
                project_description
            ),
            "project_url": (
                project_url
            ),
            "start_date": (
                start_date
            ),
            "end_date": (
                end_date
            ),
        },
        None,
    )


# ============================================================
# 读取 application.csv 身份映射
# ============================================================

# 读取固定 Application CSV。
application_dataframe = pd.read_csv(
    APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    dtype="string",
)

# 保存必要 columns。
required_application_columns = {
    "id",
    "raw_submission_id",
    "submission_normalized_id",
}

# 找出缺失 columns。
missing_application_columns = (
    required_application_columns
    - set(
        application_dataframe.columns
    )
)

# 缺失时停止。
if missing_application_columns:
    raise ValueError(
        "application.csv 缺少必要 columns："
        f"{sorted(missing_application_columns)}"
    )

# 只保留已经成功写入 D1 Application 的 records。
valid_application_dataframe = (
    application_dataframe.loc[
        application_dataframe[
            "id"
        ].notna()
    ].copy()
)

# 有效 Application CSV 不能为空。
if valid_application_dataframe.empty:
    raise ValueError(
        "application.csv 没有包含 D1 Application.id 的有效 records。"
    )

# 转换 Application.id。
valid_application_dataframe[
    "id"
] = valid_application_dataframe[
    "id"
].map(
    python_int
)

# 转换 raw_submission_id。
valid_application_dataframe[
    "raw_submission_id"
] = valid_application_dataframe[
    "raw_submission_id"
].map(
    python_int
)

# 转换 submission_normalized_id。
valid_application_dataframe[
    "submission_normalized_id"
] = valid_application_dataframe[
    "submission_normalized_id"
].map(
    python_int
)

# 三个 id columns 都必须完整。
if valid_application_dataframe[
    [
        "id",
        "raw_submission_id",
        "submission_normalized_id",
    ]
].isna().any().any():
    raise ValueError(
        "application.csv 的有效 records 存在缺失的身份 mapping id。"
    )

# Application.id 必须唯一。
if valid_application_dataframe[
    "id"
].duplicated().any():
    raise ValueError(
        "application.csv 的有效 Application.id 存在重复。"
    )

# 每个 Application 应对应唯一 submission_normalized_id。
if valid_application_dataframe[
    "submission_normalized_id"
].duplicated().any():
    raise ValueError(
        "application.csv 的 submission_normalized_id 存在重复。"
    )

# 建立 Application.id 对应的上游身份 mapping。
source_mapping_by_application_id = {
    int(
        application_row[
            "id"
        ]
    ): {
        "raw_submission_id": int(
            application_row[
                "raw_submission_id"
            ]
        ),
        "submission_normalized_id": int(
            application_row[
                "submission_normalized_id"
            ]
        ),
    }
    for _, application_row
    in valid_application_dataframe.iterrows()
}


# ============================================================
# 从 D1 读取当前批次 Candidate snapshot 和 Submission_normalized
# ============================================================

# 保存当前 application.csv 中的 Application ids。
current_application_ids = sorted(
    int(application_id)
    for application_id
    in source_mapping_by_application_id
)

# 每次最多读取 50 个 Application，兼容首批和任意大小增量批次。
CANDIDATE_QUERY_CHUNK_SIZE = 50

# 保存当前批次全部 Candidate rows。
candidate_rows = []

# 分块读取当前 CSV 对应的 Candidate、Application 和 Person mapping。
for chunk_start in range(
    0,
    len(current_application_ids),
    CANDIDATE_QUERY_CHUNK_SIZE,
):
    # 取得当前分块 Application ids。
    chunk_application_ids = current_application_ids[
        chunk_start:
        chunk_start + CANDIDATE_QUERY_CHUNK_SIZE
    ]

    # 生成参数化 IN 占位符。
    placeholders = ",".join(
        ["?"] * len(chunk_application_ids)
    )

    # 只读取当前分块对应的 Candidate snapshots。
    candidate_result = execute_d1(
        f"""
        SELECT
          cs."id" AS candidate_snapshot_id,
          cs."application_id",
          app."person_id",
          cs."raw_resume_text"
        FROM "Candidate_snapshot" AS cs
        INNER JOIN "Application" AS app
          ON app."id" = cs."application_id"
        WHERE cs."application_id" IN (
          {placeholders}
        )
        ORDER BY cs."id";
        """,
        [
            int(application_id)
            for application_id
            in chunk_application_ids
        ],
    )

    # 合并当前分块查询结果。
    candidate_rows.extend(
        result_rows(
            candidate_result
        )
    )

# 当前 CSV 有 records、但 D1 没有返回 Candidate 时停止。
if not candidate_rows:
    raise ValueError(
        "当前 application.csv 对应的 D1 "
        "Candidate_snapshot 没有任何记录。"
    )

# Candidate id 必须唯一。
candidate_ids = [
    int(
        candidate_row[
            "candidate_snapshot_id"
        ]
    )
    for candidate_row
    in candidate_rows
]

# 出现重复 id 时停止。
if len(
    candidate_ids
) != len(
    set(
        candidate_ids
    )
):
    raise ValueError(
        "D1 Candidate_snapshot.id 存在重复。"
    )

# D1 Candidate 对应的 Application ids。
d1_candidate_application_ids = {
    int(
        candidate_row[
            "application_id"
        ]
    )
    for candidate_row
    in candidate_rows
}

# 当前 CSV 中每个 Application.id 都必须存在 Candidate snapshot。
application_ids_missing_candidate = sorted(
    set(current_application_ids)
    - d1_candidate_application_ids
)

# 出现无法映射的当前批次 Application 时停止。
if application_ids_missing_candidate:
    raise ValueError(
        "当前 application.csv 中存在尚未生成 "
        "Candidate_snapshot 的 Application.id："
        f"{application_ids_missing_candidate[:20]}"
    )

# 当前 Application 与 Candidate snapshot 必须保持一对一。
if len(candidate_rows) != len(current_application_ids):
    raise ValueError(
        "当前 Application 与 Candidate_snapshot mapping "
        "不是一对一关系。"
    )

# 只保留当前 CSV 对应的 rows，并按 Candidate id 稳定排序。
candidate_rows = sorted(
    candidate_rows,
    key=lambda row: int(
        row["candidate_snapshot_id"]
    ),
)

# 显示当前批次 mapping 验证结果。
print(
    "Candidate project extraction mapping validation:",
    {
        "current_application_count": int(
            len(current_application_ids)
        ),
        "current_candidate_count": int(
            len(candidate_rows)
        ),
        "missing_candidate_count": int(
            len(application_ids_missing_candidate)
        ),
    },
)

# 一次性读取 Submission_normalized 姓名和 raw submission mapping。
submission_normalized_result = execute_d1(
    """
    SELECT
      "id" AS submission_normalized_id,
      "raw_submission_id",
      "raw_person_name"
    FROM "Submission_normalized"
    ORDER BY "id";
    """
)

# 提取 Submission_normalized rows。
submission_normalized_rows = result_rows(
    submission_normalized_result
)

# 建立 Submission_normalized.id lookup。
submission_normalized_by_id = {
    int(
        normalized_row[
            "submission_normalized_id"
        ]
    ): {
        "raw_submission_id": int(
            normalized_row[
                "raw_submission_id"
            ]
        ),
        "raw_person_name": clean_text_value(
            normalized_row.get(
                "raw_person_name"
            )
        ),
    }
    for normalized_row
    in submission_normalized_rows
}

# 逐个 Application 验证 Submission_normalized mapping。
for (
    application_id,
    source_mapping,
) in source_mapping_by_application_id.items():
    # 读取对应 Submission_normalized.id。
    submission_normalized_id = (
        source_mapping[
            "submission_normalized_id"
        ]
    )

    # Submission_normalized record 必须存在。
    if submission_normalized_id not in (
        submission_normalized_by_id
    ):
        raise ValueError(
            "D1 找不到 application.csv 对应的 "
            "Submission_normalized record："
            f"application_id={application_id}, "
            f"submission_normalized_id="
            f"{submission_normalized_id}"
        )

    # 读取 D1 mapping。
    normalized_mapping = (
        submission_normalized_by_id[
            submission_normalized_id
        ]
    )

    # D1 和 CSV 的 raw_submission_id 必须一致。
    if (
        normalized_mapping[
            "raw_submission_id"
        ]
        != source_mapping[
            "raw_submission_id"
        ]
    ):
        raise ValueError(
            "application.csv 与 D1 Submission_normalized "
            "的 raw_submission_id 不一致："
            f"application_id={application_id}"
        )


# ============================================================
# 逐个 Candidate 提取 Project records
# ============================================================

# 保存 Project record 级结果。
project_record_results = []

# 保存 Candidate 级状态。
candidate_status_results = []

# 按 D1 Candidate id 稳定处理。
for candidate_index, candidate_row in enumerate(
    candidate_rows,
    start=1,
):
    # 读取标准 Python ids。
    candidate_snapshot_id = int(
        candidate_row[
            "candidate_snapshot_id"
        ]
    )

    # 读取 Application id。
    application_id = int(
        candidate_row[
            "application_id"
        ]
    )

    # 读取 application.csv 提供的上游身份 mapping。
    source_mapping = (
        source_mapping_by_application_id[
            application_id
        ]
    )

    # 读取对应 raw_submission_id。
    raw_submission_id = int(
        source_mapping[
            "raw_submission_id"
        ]
    )

    # 读取对应 submission_normalized_id。
    submission_normalized_id = int(
        source_mapping[
            "submission_normalized_id"
        ]
    )

    # 从 D1 Submission_normalized 读取原始姓名。
    raw_person_name = (
        submission_normalized_by_id[
            submission_normalized_id
        ][
            "raw_person_name"
        ]
    )

    # 读取 Person id。
    person_id = int(
        candidate_row[
            "person_id"
        ]
    )

    # 清理简历文本。
    normalized_resume_text = normalize_resume_text(
        candidate_row.get(
            "raw_resume_text"
        )
    )

    # 没有简历文本时保存状态并继续。
    if normalized_resume_text is None:
        candidate_status_results.append(
            {
                "rule_version": (
                    RULE_VERSION
                ),
                "raw_submission_id": (
                    raw_submission_id
                ),
                "submission_normalized_id": (
                    submission_normalized_id
                ),
                "application_id": (
                    application_id
                ),
                "candidate_snapshot_id": (
                    candidate_snapshot_id
                ),
                "person_id": (
                    person_id
                ),
                "raw_person_name": (
                    raw_person_name
                ),
                "project_extraction_status": (
                    "no_resume_text"
                ),
                "project_section_count": 0,
                "extracted_project_count": 0,
                "rejected_project_block_count": 0,
                "rejection_reasons": None,
            }
        )
        continue

    # 查找明确的 Project sections。
    project_sections = find_project_sections(
        normalized_resume_text
    )

    # 没找到 section 时保存状态。
    if not project_sections:
        candidate_status_results.append(
            {
                "rule_version": (
                    RULE_VERSION
                ),
                "raw_submission_id": (
                    raw_submission_id
                ),
                "submission_normalized_id": (
                    submission_normalized_id
                ),
                "application_id": (
                    application_id
                ),
                "candidate_snapshot_id": (
                    candidate_snapshot_id
                ),
                "person_id": (
                    person_id
                ),
                "raw_person_name": (
                    raw_person_name
                ),
                "project_extraction_status": (
                    "no_explicit_project_section"
                ),
                "project_section_count": 0,
                "extracted_project_count": 0,
                "rejected_project_block_count": 0,
                "rejection_reasons": None,
            }
        )
        continue

    # 保存当前 Candidate 成功提取的 records。
    current_candidate_projects = []

    # 保存被拒绝的 block 原因。
    rejected_block_reasons = []

    # 逐个 Project section 处理。
    for project_section in project_sections:
        # 切分 section。
        project_blocks = (
            split_project_section_into_records(
                project_section
            )
        )

        # 没有可用 block 时记录原因。
        if not project_blocks:
            rejected_block_reasons.append(
                "empty_project_section"
            )
            continue

        # 逐个 block 建立结构化结果。
        for project_block in project_blocks:
            # 转换成结构化 record。
            (
                project_record,
                rejection_reason,
            ) = build_project_record(
                project_block,
                project_section[
                    "section_header"
                ],
            )

            # 拒绝的 block 只记录原因。
            if project_record is None:
                rejected_block_reasons.append(
                    rejection_reason
                )
                continue

            # 保存成功提取结果。
            current_candidate_projects.append(
                project_record
            )

    # 只删除同一个 Candidate 内 raw_project_text 完全相同的记录。
    #
    # 普通同名项目不会在这里删除，因为同一个人可能在不同时间
    # 做过名称相同但内容不同的项目。
    unique_candidate_projects = []

    # 保存已经出现的完整原始项目文本。
    seen_raw_project_texts = set()

    # 按简历原始顺序检查每条项目记录。
    for project_record in (
        current_candidate_projects
    ):
        # 取得完整原始项目文本。
        raw_project_text = project_record[
            "raw_project_text"
        ]

        # 完全相同的文本已经出现时拒绝当前重复record。
        if raw_project_text in seen_raw_project_texts:
            rejected_block_reasons.append(
                "duplicate_raw_project_text"
            )
            continue

        # 记录第一次出现的完整文本。
        seen_raw_project_texts.add(
            raw_project_text
        )

        # 保留第一次出现的record。
        unique_candidate_projects.append(
            project_record
        )

    # 使用去除完全重复文本后的records继续生成结果。
    current_candidate_projects = (
        unique_candidate_projects
    )

    # 按简历出现顺序生成 sequence。
    for project_sequence, project_record in enumerate(
        current_candidate_projects,
        start=1,
    ):
        # 保存 record 级输出。
        project_record_results.append(
            {
                "rule_version": (
                    RULE_VERSION
                ),
                "raw_submission_id": (
                    raw_submission_id
                ),
                "submission_normalized_id": (
                    submission_normalized_id
                ),
                "application_id": (
                    application_id
                ),
                "candidate_snapshot_id": (
                    candidate_snapshot_id
                ),
                "person_id": (
                    person_id
                ),
                "raw_person_name": (
                    raw_person_name
                ),
                "project_sequence": (
                    project_sequence
                ),
                **project_record,
            }
        )

    # 根据是否提取到项目决定 Candidate 状态。
    if current_candidate_projects:
        project_extraction_status = (
            "extracted"
        )
    else:
        project_extraction_status = (
            "project_section_without_reliable_record"
        )

    # 保存 Candidate 状态。
    candidate_status_results.append(
        {
            "rule_version": (
                RULE_VERSION
            ),
            "raw_submission_id": (
                raw_submission_id
            ),
            "submission_normalized_id": (
                submission_normalized_id
            ),
            "application_id": (
                application_id
            ),
            "candidate_snapshot_id": (
                candidate_snapshot_id
            ),
            "person_id": (
                person_id
            ),
            "raw_person_name": (
                raw_person_name
            ),
            "project_extraction_status": (
                project_extraction_status
            ),
            "project_section_count": int(
                len(
                    project_sections
                )
            ),
            "extracted_project_count": int(
                len(
                    current_candidate_projects
                )
            ),
            "rejected_project_block_count": int(
                len(
                    rejected_block_reasons
                )
            ),
            "rejection_reasons": (
                "; ".join(
                    rejected_block_reasons
                )
                if rejected_block_reasons
                else None
            ),
        }
    )

    # 每100条显示一次进度，并显示最后一条。
    if (
        candidate_index % 100 == 0
        or candidate_index
        == len(
            candidate_rows
        )
    ):
        print(
            "Processed Candidate projects:",
            f"{candidate_index}/"
            f"{len(candidate_rows)}",
        )


# ============================================================
# 建立并固定排序两个输出 DataFrames
# ============================================================

# 固定 Project record columns。
project_record_columns = [
    "rule_version",
    "raw_submission_id",
    "submission_normalized_id",
    "application_id",
    "candidate_snapshot_id",
    "person_id",
    "raw_person_name",
    "project_sequence",
    "project_section_header",
    "raw_project_text",
    "raw_project_name",
    "normalized_project_name",
    "project_role",
    "project_description",
    "project_url",
    "start_date",
    "end_date",
]

# 建立 Project record DataFrame。
project_extraction_dataframe = pd.DataFrame(
    project_record_results,
    columns=project_record_columns,
)

# 有结果时按 Candidate 和 sequence 排序。
if not project_extraction_dataframe.empty:
    project_extraction_dataframe = (
        project_extraction_dataframe.sort_values(
            [
                "candidate_snapshot_id",
                "project_sequence",
            ],
            kind="stable",
        ).reset_index(
            drop=True
        )
    )

# 固定 Candidate status columns。
project_status_columns = [
    "rule_version",
    "raw_submission_id",
    "submission_normalized_id",
    "application_id",
    "candidate_snapshot_id",
    "person_id",
    "raw_person_name",
    "project_extraction_status",
    "project_section_count",
    "extracted_project_count",
    "rejected_project_block_count",
    "rejection_reasons",
]

# 建立 Candidate status DataFrame。
project_status_dataframe = pd.DataFrame(
    candidate_status_results,
    columns=project_status_columns,
)

# 按 Candidate id 稳定排序。
project_status_dataframe = (
    project_status_dataframe.sort_values(
        [
            "candidate_snapshot_id",
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )
)

# status 必须覆盖全部 Candidate。
if len(
    project_status_dataframe
) != len(
    candidate_rows
):
    raise RuntimeError(
        "Project status record 数量与 Candidate 数量不一致。"
    )

# status 中 Candidate id 不能重复。
if project_status_dataframe[
    "candidate_snapshot_id"
].duplicated().any():
    raise RuntimeError(
        "Project status 中 candidate_snapshot_id 存在重复。"
    )


# ============================================================
# 安全覆盖保存固定 CSV
# ============================================================

# 生成临时 Project record CSV 路径。
project_extraction_temp_path = (
    f"{PROJECT_EXTRACTION_CSV_PATH}.tmp"
)

# 写入临时 Project record CSV。
project_extraction_dataframe.to_csv(
    project_extraction_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 原子替换固定 Project record CSV。
os.replace(
    project_extraction_temp_path,
    PROJECT_EXTRACTION_CSV_PATH,
)

# 生成临时 status CSV 路径。
project_status_temp_path = (
    f"{PROJECT_STATUS_CSV_PATH}.tmp"
)

# 写入临时 status CSV。
project_status_dataframe.to_csv(
    project_status_temp_path,
    index=False,
    encoding="utf-8-sig",
)

# 原子替换固定 status CSV。
os.replace(
    project_status_temp_path,
    PROJECT_STATUS_CSV_PATH,
)


# ============================================================
# 显示提取统计和预览
# ============================================================

# 统计每个状态的 Candidate 数量。
status_counts = (
    project_status_dataframe[
        "project_extraction_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_dict()
)

# 统计存在项目 URL 的 records。
project_url_present_count = int(
    project_extraction_dataframe[
        "project_url"
    ].notna().sum()
) if not project_extraction_dataframe.empty else 0

# 统计存在日期信息的 records。
project_date_present_count = int(
    (
        project_extraction_dataframe[
            "start_date"
        ].notna()
        | project_extraction_dataframe[
            "end_date"
        ].notna()
    ).sum()
) if not project_extraction_dataframe.empty else 0

# 显示 summary。
print(
    "Candidate project rule extraction summary:",
    {
        "candidate_count": int(
            len(
                project_status_dataframe
            )
        ),
        "project_record_count": int(
            len(
                project_extraction_dataframe
            )
        ),
        "candidate_status_counts": (
            status_counts
        ),
        "project_url_present_count": (
            project_url_present_count
        ),
        "project_date_present_count": (
            project_date_present_count
        ),
    },
)

# 显示输出路径。
print(
    "Project extraction CSV:",
    PROJECT_EXTRACTION_CSV_PATH,
)

# 显示 status 输出路径。
print(
    "Project status CSV:",
    PROJECT_STATUS_CSV_PATH,
)

# 显示 Project records 预览。
display(
    project_extraction_dataframe.head(
        20
    )
)

# 显示 Candidate status 预览。
display(
    project_status_dataframe.head(
        20
    )
)

# 显示完成。
print(
    "Candidate project rule extraction completed successfully."
)


### Candidate_project, Person_project 数据导入

In [ ]:
# ============================================================
# Google Colab 完全独立 Cell：
# 将当前 Candidate Project CSV 幂等导入 D1。
#
# 重要：
# 1. 本 Cell 不依赖其他 Colab Cell 的 variable、function 或 class。
# 2. 本 Cell 只读取固定的 candidate_project_rule_extraction.csv。
# 3. 本 Cell 不重新提取或重新判断 Project。
# 4. Project 内容写入 Person_Project。
# 5. Candidate 与 Project 的关系写入 Candidate_Project。
# 6. 相同输入可以重复运行，不会重复创建 Project 或桥接记录。
# 7. project_name 使用 raw_project_name，保留正常大小写。
# 8. normalized_project_name 继续保留在 CSV，不写入当前数据库字段。
# 9. 导入结果直接回写原来的两个固定 CSV。
# 10. 单条失败只标记 failed，其他记录继续处理。
# ============================================================

# os 用于检查路径和原子替换 CSV。
import os

# time 只用于 D1 临时失败后的重试等待。
import time

# datetime 用于生成统一 UTC timestamp。
from datetime import datetime, timezone

# pandas 用于读取和更新 CSV。
import pandas as pd

# requests 用于调用 Cloudflare D1 HTTP API。
import requests

# drive 用于挂载 Google Drive。
# userdata 用于读取 CLOUDFLARE_KEY。
from google.colab import drive, userdata


# ============================================================
# 固定 Google Drive 路径
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# 保存项目根目录。
DRIVE_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline"
)

# 保存 datasets 目录。
DATASET_FOLDER = (
    f"{DRIVE_FOLDER}/datasets"
)

# 保存固定的 Project 明细 CSV 路径。
PROJECT_EXTRACTION_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_project_rule_extraction.csv"
)

# 保存固定的 Candidate Project 状态 CSV 路径。
PROJECT_STATUS_CSV_PATH = (
    f"{DATASET_FOLDER}/"
    "candidate_project_rule_extraction_status.csv"
)

# 两个输入文件都必须存在。
for required_file_path in [
    PROJECT_EXTRACTION_CSV_PATH,
    PROJECT_STATUS_CSV_PATH,
]:
    # 缺少文件时立即停止。
    if not os.path.exists(
        required_file_path
    ):
        raise FileNotFoundError(
            "找不到必要输入文件："
            f"{required_file_path}"
        )


# ============================================================
# Cloudflare D1 独立连接配置
# ============================================================

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# Cloudflare D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时停止。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。"
        "请在 Colab 左侧 Secrets 中添加并授权该密钥。"
    )

# 生成 D1 query endpoint。
D1_API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 保存 HTTP headers。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# 复用 HTTP connection。
HTTP_SESSION = requests.Session()


def execute_d1(
    sql,
    params=None,
    max_retries=3,
):
    """执行参数化 D1 SQL，并对临时失败有限重试。"""

    # 保存最后一次错误。
    last_error = None

    # 第一次请求也包含在 max_retries 中。
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            # 发送 D1 request。
            response = HTTP_SESSION.post(
                D1_API_URL,
                headers=D1_HEADERS,
                json={
                    "sql": sql,
                    "params": params or [],
                },
                timeout=60,
            )
        except requests.exceptions.RequestException as error:
            # 保存网络错误。
            last_error = error

            # 还有机会时等待后重试。
            if attempt < max_retries:
                time.sleep(
                    1.0
                )
                continue

            # 全部失败后停止。
            raise RuntimeError(
                "无法连接 Cloudflare D1："
                f"{last_error}"
            ) from error

        try:
            # 解析 Cloudflare JSON response。
            payload = response.json()
        except ValueError as error:
            # 非 JSON response 无法继续。
            raise RuntimeError(
                "Cloudflare 返回了非 JSON 响应："
                f"HTTP {response.status_code} "
                f"{response.text[:300]}"
            ) from error

        # HTTP 和 SQL 都成功时返回结果。
        if (
            response.status_code == 200
            and payload.get(
                "success",
                False,
            )
        ):
            return payload.get(
                "result",
                [],
            )

        # 保存 D1 返回的错误。
        last_error = payload.get(
            "errors",
            payload,
        )

        # 还有机会时等待后重试。
        if attempt < max_retries:
            time.sleep(
                1.0
            )

    # 全部重试失败后停止。
    raise RuntimeError(
        "D1 SQL 在重试后仍然失败："
        f"{last_error}"
    )


def result_rows(
    d1_result,
):
    """提取第一条 D1 statement 的 results rows。"""

    # 空结果返回空 list。
    if not d1_result:
        return []

    # 返回第一条 statement 的 rows。
    return d1_result[0].get(
        "results",
        [],
    )


def now_utc_iso():
    """生成标准 UTC timestamp。"""

    # 使用秒级 ISO 格式。
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def clean_optional_text(
    value,
):
    """把 pandas 空值和空字符串转换成 Python None。"""

    # pandas 空值转成 None。
    if pd.isna(
        value
    ):
        return None

    # 转成普通 Python string。
    cleaned_value = str(
        value
    ).strip()

    # 空字符串转成 None。
    return (
        cleaned_value
        if cleaned_value
        else None
    )


def require_python_int(
    value,
    field_name,
):
    """把 pandas/NumPy integer 安全转换成 Python int。"""

    # 缺少必要 id 时停止当前record。
    if pd.isna(
        value
    ):
        raise ValueError(
            f"{field_name} 不能为空。"
        )

    # 显式转换，避免 numpy.int64 无法 JSON serialization。
    return int(
        value
    )


def atomic_write_csv(
    dataframe,
    output_path,
):
    """通过临时文件原子覆盖固定 CSV。"""

    # 建立临时文件路径。
    temporary_path = (
        f"{output_path}.tmp"
    )

    # 先写入临时文件。
    dataframe.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    # 成功写完后替换正式文件。
    os.replace(
        temporary_path,
        output_path,
    )


# ============================================================
# 读取当前两个固定 Project CSV
# ============================================================

# 读取 Project 明细。
project_dataframe = pd.read_csv(
    PROJECT_EXTRACTION_CSV_PATH,
    low_memory=False,
)

# 读取 Candidate Project 状态。
project_status_dataframe = pd.read_csv(
    PROJECT_STATUS_CSV_PATH,
    low_memory=False,
)

# 当前批次允许没有任何达到质量准入条件的 Project record。
#
# 这在以下两种情况下都属于正常结果，而不是程序错误：
# 1. 第一批导入的数据中没有简历包含可靠的 Project section；
# 2. 后续某个增量批次没有提取出可靠项目。
#
# 此时仍继续执行后面的完整验证和 CSV 状态回写逻辑，但数据库 Project
# 写入循环自然为零次，从而形成一个安全、幂等的 no-op。
if project_dataframe.empty:
    print(
        "Current batch has no reliable Project records; "
        "database Project import will be a no-op."
    )

# 保存必要的 Project columns。
required_project_columns = [
    "candidate_snapshot_id",
    "person_id",
    "raw_project_text",
    "raw_project_name",
    "project_role",
    "project_description",
    "project_url",
    "start_date",
    "end_date",
]

# 检查 Project columns。
missing_project_columns = [
    column_name
    for column_name in required_project_columns
    if column_name
    not in project_dataframe.columns
]

# 缺少必要 column 时停止。
if missing_project_columns:
    raise ValueError(
        "Project extraction CSV 缺少 columns："
        f"{missing_project_columns}"
    )

# 保存必要的 status columns。
required_status_columns = [
    "candidate_snapshot_id",
    "project_extraction_status",
    "extracted_project_count",
]

# 检查 status columns。
missing_status_columns = [
    column_name
    for column_name in required_status_columns
    if column_name
    not in project_status_dataframe.columns
]

# 缺少必要 column 时停止。
if missing_status_columns:
    raise ValueError(
        "Project status CSV 缺少 columns："
        f"{missing_status_columns}"
    )

# raw_project_text 是项目质量检查和完全重复判断的必要字段。
if project_dataframe[
    "raw_project_text"
].isna().any():
    raise ValueError(
        "Project extraction CSV 存在空 raw_project_text。"
    )

# raw_project_name 要写入 NOT NULL project_name，不能为空。
if project_dataframe[
    "raw_project_name"
].isna().any():
    raise ValueError(
        "Project extraction CSV 存在空 raw_project_name。"
    )

# 同一个 Candidate 内不允许存在 raw_project_text 完全相同的记录。
#
# 普通同名项目不会被这个检查拒绝；
# 只有完整原始项目文本也完全相同时才判定为重复。
if project_dataframe.duplicated(
    [
        "candidate_snapshot_id",
        "raw_project_text",
    ]
).any():
    raise ValueError(
        "Project extraction CSV 仍存在完全相同的 "
        "candidate_snapshot_id + raw_project_text。"
    )

# status 中记录的项目总数必须与明细完全一致。
if int(
    project_status_dataframe[
        "extracted_project_count"
    ].sum()
) != len(
    project_dataframe
):
    raise ValueError(
        "Project status 的项目总数与明细 CSV 不一致。"
    )


# ============================================================
# 验证当前 D1 schema 和 Candidate → Person mapping
# ============================================================

# 查询 Person_Project columns。
person_project_columns = result_rows(
    execute_d1(
        'PRAGMA table_info("Person_Project");'
    )
)

# 建立 Person_Project column name set。
person_project_column_names = {
    row["name"]
    for row in person_project_columns
}

# 本 Cell 需要的 Person_Project columns。
required_person_project_columns = {
    "id",
    "person_id",
    "project_name",
    "project_role",
    "project_description",
    "project_url",
    "start_date",
    "end_date",
    "created_at",
    "source_candidate_id",
}

# schema 不完整时停止。
if not required_person_project_columns.issubset(
    person_project_column_names
):
    raise RuntimeError(
        "Person_Project schema 缺少 columns："
        f"{sorted(required_person_project_columns - person_project_column_names)}"
    )

# 查询 Candidate_Project columns。
candidate_project_columns = result_rows(
    execute_d1(
        'PRAGMA table_info("Candidate_Project");'
    )
)

# 建立 Candidate_Project column name set。
candidate_project_column_names = {
    row["name"]
    for row in candidate_project_columns
}

# schema 不完整时停止。
if not {
    "id",
    "candidate_id",
    "person_project_id",
    "created_at",
}.issubset(
    candidate_project_column_names
):
    raise RuntimeError(
        "Candidate_Project schema 不符合预期。"
    )

# 一次性查询所有 Candidate 与 Person 的正式映射。
candidate_mapping_rows = result_rows(
    execute_d1(
        """
        SELECT
          cs."id" AS candidate_snapshot_id,
          a."person_id" AS person_id
        FROM "Candidate_snapshot" AS cs
        INNER JOIN "Application" AS a
          ON a."id" = cs."application_id";
        """
    )
)

# 建立 Candidate → Person mapping。
person_id_by_candidate_id = {
    int(
        row["candidate_snapshot_id"]
    ): int(
        row["person_id"]
    )
    for row in candidate_mapping_rows
}

# CSV 中每一个 Candidate 都必须存在于 D1。
missing_candidate_ids = sorted(
    {
        int(
            candidate_id
        )
        for candidate_id in project_dataframe[
            "candidate_snapshot_id"
        ].dropna().tolist()
    }
    - set(
        person_id_by_candidate_id
    )
)

# 缺少 Candidate 时停止，避免产生孤立数据。
if missing_candidate_ids:
    raise ValueError(
        "以下 Candidate_snapshot.id 不存在于 D1："
        f"{missing_candidate_ids[:20]}"
    )


# ============================================================
# 幂等写入 Person_Project 和 Candidate_Project
# ============================================================

# 预先声明结果 columns。
for result_column_name in [
    "person_project_id",
    "candidate_project_id",
    "project_import_status",
    "project_import_action",
    "project_import_error",
    "project_imported_at",
]:
    # 不管旧 CSV 是否已有结果，都重新计算并覆盖。
    project_dataframe[
        result_column_name
    ] = None

# 保存本次统计。
inserted_person_project_count = 0
reused_person_project_count = 0
inserted_candidate_project_count = 0
reused_candidate_project_count = 0
failed_record_count = 0

# 逐条处理当前 Project records。
for dataframe_index, project_row in (
    project_dataframe.iterrows()
):
    try:
        # 取得标准 Python candidate id。
        candidate_snapshot_id = require_python_int(
            project_row[
                "candidate_snapshot_id"
            ],
            "candidate_snapshot_id",
        )

        # 取得 CSV person id。
        csv_person_id = require_python_int(
            project_row[
                "person_id"
            ],
            "person_id",
        )

        # 取得 D1 正式 person id。
        d1_person_id = person_id_by_candidate_id[
            candidate_snapshot_id
        ]

        # CSV 与 D1 mapping 不一致时拒绝当前record。
        if csv_person_id != d1_person_id:
            raise ValueError(
                "CSV person_id 与 D1 Candidate mapping 不一致："
                f"CSV={csv_person_id}, D1={d1_person_id}"
            )

        # 清理要写入的项目字段。
        project_name = clean_optional_text(
            project_row[
                "raw_project_name"
            ]
        )
        project_role = clean_optional_text(
            project_row[
                "project_role"
            ]
        )
        project_description = clean_optional_text(
            project_row[
                "project_description"
            ]
        )
        project_url = clean_optional_text(
            project_row[
                "project_url"
            ]
        )
        start_date = clean_optional_text(
            project_row[
                "start_date"
            ]
        )
        end_date = clean_optional_text(
            project_row[
                "end_date"
            ]
        )

        # project_name 是数据库必要字段。
        if project_name is None:
            raise ValueError(
                "raw_project_name 清理后为空。"
            )

        # 使用所有实际写入字段查找同一项目。
        existing_person_project_rows = result_rows(
            execute_d1(
                """
                SELECT "id"
                FROM "Person_Project"
                WHERE "person_id" = ?
                  AND "source_candidate_id" = ?
                  AND "project_name" = ?
                  AND "project_role" IS ?
                  AND "project_description" IS ?
                  AND "project_url" IS ?
                  AND "start_date" IS ?
                  AND "end_date" IS ?
                ORDER BY "id";
                """,
                [
                    d1_person_id,
                    candidate_snapshot_id,
                    project_name,
                    project_role,
                    project_description,
                    project_url,
                    start_date,
                    end_date,
                ],
            )
        )

        # 找到多个完全相同的数据库项目时停止当前record。
        if len(
            existing_person_project_rows
        ) > 1:
            raise RuntimeError(
                "D1 已存在多个完全相同的 Person_Project。"
            )

        # 已存在时复用。
        if existing_person_project_rows:
            person_project_id = int(
                existing_person_project_rows[
                    0
                ][
                    "id"
                ]
            )
            person_project_action = "reused"
            reused_person_project_count += 1
        else:
            # 生成创建时间。
            created_at = now_utc_iso()

            # 插入新的 Person_Project。
            inserted_person_project_rows = result_rows(
                execute_d1(
                    """
                    INSERT INTO "Person_Project" (
                      "person_id",
                      "project_name",
                      "project_role",
                      "project_description",
                      "project_url",
                      "start_date",
                      "end_date",
                      "created_at",
                      "source_candidate_id"
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                    RETURNING "id";
                    """,
                    [
                        d1_person_id,
                        project_name,
                        project_role,
                        project_description,
                        project_url,
                        start_date,
                        end_date,
                        created_at,
                        candidate_snapshot_id,
                    ],
                )
            )

            # INSERT 必须返回 id。
            if len(
                inserted_person_project_rows
            ) != 1:
                raise RuntimeError(
                    "Person_Project INSERT 没有返回唯一 id。"
                )

            # 保存新 Person_Project id。
            person_project_id = int(
                inserted_person_project_rows[
                    0
                ][
                    "id"
                ]
            )
            person_project_action = "inserted"
            inserted_person_project_count += 1

        # 查找 Candidate_Project bridge。
        existing_candidate_project_rows = result_rows(
            execute_d1(
                """
                SELECT "id"
                FROM "Candidate_Project"
                WHERE "candidate_id" = ?
                  AND "person_project_id" = ?
                ORDER BY "id";
                """,
                [
                    candidate_snapshot_id,
                    person_project_id,
                ],
            )
        )

        # 唯一索引下最多只能有一条。
        if len(
            existing_candidate_project_rows
        ) > 1:
            raise RuntimeError(
                "D1 存在重复 Candidate_Project bridge。"
            )

        # 已存在时复用 bridge。
        if existing_candidate_project_rows:
            candidate_project_id = int(
                existing_candidate_project_rows[
                    0
                ][
                    "id"
                ]
            )
            candidate_project_action = "reused"
            reused_candidate_project_count += 1
        else:
            # 插入新的 bridge。
            inserted_candidate_project_rows = result_rows(
                execute_d1(
                    """
                    INSERT INTO "Candidate_Project" (
                      "candidate_id",
                      "person_project_id",
                      "created_at"
                    )
                    VALUES (?, ?, ?)
                    RETURNING "id";
                    """,
                    [
                        candidate_snapshot_id,
                        person_project_id,
                        now_utc_iso(),
                    ],
                )
            )

            # INSERT 必须返回唯一 id。
            if len(
                inserted_candidate_project_rows
            ) != 1:
                raise RuntimeError(
                    "Candidate_Project INSERT 没有返回唯一 id。"
                )

            # 保存新 bridge id。
            candidate_project_id = int(
                inserted_candidate_project_rows[
                    0
                ][
                    "id"
                ]
            )
            candidate_project_action = "inserted"
            inserted_candidate_project_count += 1

        # 保存成功结果。
        project_dataframe.at[
            dataframe_index,
            "person_project_id",
        ] = person_project_id
        project_dataframe.at[
            dataframe_index,
            "candidate_project_id",
        ] = candidate_project_id
        project_dataframe.at[
            dataframe_index,
            "project_import_status",
        ] = "success"
        project_dataframe.at[
            dataframe_index,
            "project_import_action",
        ] = (
            f"person_project_{person_project_action};"
            f"candidate_project_{candidate_project_action}"
        )
        project_dataframe.at[
            dataframe_index,
            "project_import_error",
        ] = None
        project_dataframe.at[
            dataframe_index,
            "project_imported_at",
        ] = now_utc_iso()

    except Exception as error:
        # 单条失败时记录错误并继续。
        failed_record_count += 1
        project_dataframe.at[
            dataframe_index,
            "project_import_status",
        ] = "failed"
        project_dataframe.at[
            dataframe_index,
            "project_import_action",
        ] = None
        project_dataframe.at[
            dataframe_index,
            "project_import_error",
        ] = str(
            error
        )
        project_dataframe.at[
            dataframe_index,
            "project_imported_at",
        ] = now_utc_iso()

    # 每100条和最后一条显示进度。
    processed_record_count = (
        dataframe_index + 1
    )
    if (
        processed_record_count % 100 == 0
        or processed_record_count
        == len(
            project_dataframe
        )
    ):
        print(
            "Processed Candidate Projects:",
            f"{processed_record_count}/"
            f"{len(project_dataframe)}",
        )


# ============================================================
# 回填 Candidate status CSV
# ============================================================

# 按 Candidate 统计成功导入数量。
successful_count_by_candidate = (
    project_dataframe.loc[
        project_dataframe[
            "project_import_status"
        ].eq(
            "success"
        )
    ]
    .groupby(
        "candidate_snapshot_id"
    )
    .size()
    .to_dict()
)

# 按 Candidate 统计失败数量。
failed_count_by_candidate = (
    project_dataframe.loc[
        project_dataframe[
            "project_import_status"
        ].eq(
            "failed"
        )
    ]
    .groupby(
        "candidate_snapshot_id"
    )
    .size()
    .to_dict()
)

# 预先声明 status 导入结果 columns。
for status_column_name in [
    "project_import_status",
    "project_import_success_count",
    "project_import_failed_count",
]:
    # 每次运行都重新计算。
    project_status_dataframe[
        status_column_name
    ] = None

# 逐个 Candidate 回填汇总状态。
for status_index, status_row in (
    project_status_dataframe.iterrows()
):
    # 取得 Candidate id。
    candidate_snapshot_id = require_python_int(
        status_row[
            "candidate_snapshot_id"
        ],
        "candidate_snapshot_id",
    )

    # 取得该 Candidate 的成功数量。
    success_count = int(
        successful_count_by_candidate.get(
            candidate_snapshot_id,
            0,
        )
    )

    # 取得该 Candidate 的失败数量。
    failure_count = int(
        failed_count_by_candidate.get(
            candidate_snapshot_id,
            0,
        )
    )

    # 有失败时标记 failed。
    if failure_count > 0:
        import_status = "failed"
    # 原本没有提取项目时不需要数据库导入。
    elif int(
        status_row[
            "extracted_project_count"
        ]
    ) == 0:
        import_status = "rejected"
    # 全部项目成功时标记 success。
    else:
        import_status = "success"

    # 保存 Candidate 汇总结果。
    project_status_dataframe.at[
        status_index,
        "project_import_status",
    ] = import_status
    project_status_dataframe.at[
        status_index,
        "project_import_success_count",
    ] = success_count
    project_status_dataframe.at[
        status_index,
        "project_import_failed_count",
    ] = failure_count


# ============================================================
# 数据库结果验证
# ============================================================

# 所有 CSV success records 都必须有两个数据库 id。
successful_project_dataframe = (
    project_dataframe.loc[
        project_dataframe[
            "project_import_status"
        ].eq(
            "success"
        )
    ]
)

# success record 缺少 id 时停止。
if (
    successful_project_dataframe[
        "person_project_id"
    ].isna().any()
    or successful_project_dataframe[
        "candidate_project_id"
    ].isna().any()
):
    raise RuntimeError(
        "成功记录中存在缺失的数据库 id。"
    )

# 查询全库外键错误。
foreign_key_violations = result_rows(
    execute_d1(
        "PRAGMA foreign_key_check;"
    )
)

# 存在外键错误时停止，暂不覆盖 CSV。
if foreign_key_violations:
    raise RuntimeError(
        "D1 存在 foreign key violations："
        f"{foreign_key_violations[:20]}"
    )


# ============================================================
# 原子覆盖两个固定 CSV
# ============================================================

# 覆盖 Project 明细 CSV。
atomic_write_csv(
    project_dataframe,
    PROJECT_EXTRACTION_CSV_PATH,
)

# 覆盖 Candidate status CSV。
atomic_write_csv(
    project_status_dataframe,
    PROJECT_STATUS_CSV_PATH,
)


# ============================================================
# 显示最终结果
# ============================================================

# 显示导入统计。
print(
    "Candidate Project import summary:",
    {
        "planned_project_records": int(
            len(
                project_dataframe
            )
        ),
        "successful_project_records": int(
            len(
                successful_project_dataframe
            )
        ),
        "failed_project_records": int(
            failed_record_count
        ),
        "inserted_person_projects": int(
            inserted_person_project_count
        ),
        "reused_person_projects": int(
            reused_person_project_count
        ),
        "inserted_candidate_projects": int(
            inserted_candidate_project_count
        ),
        "reused_candidate_projects": int(
            reused_candidate_project_count
        ),
        "foreign_key_violation_count": int(
            len(
                foreign_key_violations
            )
        ),
    },
)

# 显示前10条导入结果。
display(
    project_dataframe[
        [
            "candidate_snapshot_id",
            "person_id",
            "raw_person_name",
            "raw_project_name",
            "person_project_id",
            "candidate_project_id",
            "project_import_status",
            "project_import_action",
            "project_import_error",
        ]
    ].head(
        10
    )
)

# 显示完成信息。
print(
    "Candidate Project import completed."
)


# ML 使用数据查询导出csv + 存入 Google Drive

In [ ]:
"""
Google Colab 独立 Cell：从 D1 导出一张用于 Application ML 的宽表 CSV。

本版本根据 04_d1_recruiting_pipeline.ipynb 中已经实际运行的数据导入流程编写。

设计原则：
1. 本 Cell 完全独立，不依赖其他 Cell 中的变量、函数或 Class。
2. 本 Cell 只执行 SELECT，不对 D1 执行 INSERT、UPDATE、DELETE 或 DDL。
3. 最终一行对应一个 Candidate_snapshot，也就是一条进入 Application 层的申请。
4. 只读取当前流程中已经实际写入数据、并且对 ML 有直接用途的表和字段。
5. 不读取当前为空或没有在 notebook 中完成数据导入的 Offer、Application history、
   Position requirement 等表。
6. 学历只读取 Candidate_Education.is_highest_degree = 1 的最高学历。
7. 技能、项目、工作经历等一对多数据先在 pandas 中聚合为 JSON，避免 SQL 多表
   JOIN 造成笛卡尔积和大量无意义的 D1 rows read。
8. 所有 JSON 使用 ensure_ascii=False，CSV 使用 utf-8-sig，因此中文和其他
   Unicode 字符保持原字符，不转换成 \\uXXXX。
9. raw_resume_text、education_description、position_description 和
   project_description 都会保留在最终 UTF-8 CSV 中。
10. 固定覆盖同一个输出文件，因此可重复运行，不会不断生成新文件。
"""

# 导入 json，用于把技能、项目、工作经历等多条记录保存成结构化 JSON 文本。
import json

# 导入 os，用于创建 Google Drive 输出目录和执行原子文件替换。
import os

# 导入 datetime，用于记录本次数据集导出的 UTC 时间。
from datetime import datetime, timezone

# 导入 pandas，用于接收 D1 查询结果、聚合一对多记录和生成最终 CSV。
import pandas as pd

# 导入 requests，用于通过 Cloudflare D1 HTTP API 执行只读 SQL。
import requests

# 从 Colab 导入 Google Drive 和 Secrets 工具。
from google.colab import drive, userdata


# ============================================================
# 一、独立运行所需配置
# ============================================================

# 挂载 Google Drive。
drive.mount(
    "/content/drive"
)

# Cloudflare account id。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')

# 当前 D1 database id。
DATABASE_ID = userdata.get('D1_DATABASE_ID')

# 从 Colab Secrets 读取 Cloudflare API token。
API_TOKEN = userdata.get(
    "CLOUDFLARE_KEY"
)

# 没有 token 时立即停止，避免发送无效请求。
if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。请在 Colab Secrets 中添加并授权该密钥。"
    )

# 最终输出目录。
ML_OUTPUT_FOLDER = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline/"
    "datasets/ml"
)

# 最终结果始终覆盖这个固定文件。
ML_OUTPUT_CSV_PATH = (
    f"{ML_OUTPUT_FOLDER}/"
    "application_ml_feature_dataset.csv"
)

# 当前批次 Application CSV。只用其中已经成功写入 D1 的 Application.id
# 限定本次 ML 宽表输出；当前 CSV 可以是 D1 全库 Application 的任意子集。
CURRENT_APPLICATION_CSV_PATH = (
    "/content/drive/MyDrive/"
    "recruiting-pipeline/"
    "datasets/dedup_results/application_dedup_v2/application.csv"
)

# 当前批次 CSV 必须存在。
if not os.path.isfile(CURRENT_APPLICATION_CSV_PATH):
    raise FileNotFoundError(
        f"找不到当前批次 application.csv：{CURRENT_APPLICATION_CSV_PATH}"
    )

# 读取当前批次 Application.id；其余宽表字段仍以 D1 正式数据为准。
current_application_dataframe = pd.read_csv(
    CURRENT_APPLICATION_CSV_PATH,
    encoding="utf-8-sig",
    usecols=["id"],
)

# 只保留已经获得 D1 Application.id 的记录。
current_application_dataframe = current_application_dataframe.loc[
    current_application_dataframe["id"].notna()
].copy()

# 当前批次必须至少包含一条成功写入的 Application。
if current_application_dataframe.empty:
    raise ValueError("当前 application.csv 没有有效的 Application.id。")

# ID 必须是整数且唯一。
current_application_dataframe["id"] = pd.to_numeric(
    current_application_dataframe["id"],
    errors="raise",
).astype("int64")

if current_application_dataframe["id"].duplicated().any():
    raise ValueError("当前 application.csv 存在重复 Application.id。")

current_application_ids = set(
    int(value)
    for value in current_application_dataframe["id"].tolist()
)

# 标记当前导出字段设计版本。
ML_EXPORT_VERSION = (
    "application_ml_feature_export_curated_v1"
)

# 生成 D1 query endpoint。
D1_QUERY_URL = (
    "https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/d1/database/{DATABASE_ID}/query"
)

# 生成 D1 请求头。
D1_HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


# ============================================================
# 二、D1 只读查询工具
# ============================================================

def execute_d1_select(
    sql,
    params=None,
    timeout_seconds=120,
):
    """只允许执行 SELECT 或 WITH 开头的 D1 查询。"""

    # 去掉 SQL 首尾空白并转成小写，供安全检查使用。
    normalized_sql = sql.strip().lower()

    # 本导出 Cell 不允许执行写入或 schema 修改语句。
    if not (
        normalized_sql.startswith("select")
        or normalized_sql.startswith("with")
    ):
        raise ValueError(
            "本 Cell 只允许执行 SELECT 或 WITH 查询。"
        )

    # 构建 D1 API payload。
    payload = {
        "sql": sql,
        "params": params or [],
    }

    # 向 D1 发送查询。
    response = requests.post(
        D1_QUERY_URL,
        headers=D1_HEADERS,
        json=payload,
        timeout=timeout_seconds,
    )

    # 尝试读取 Cloudflare JSON response。
    try:
        response_payload = response.json()
    except ValueError as error:
        raise RuntimeError(
            "Cloudflare 返回了非 JSON response："
            f"HTTP {response.status_code}\n"
            f"{response.text[:500]}"
        ) from error

    # HTTP 或 D1 query 失败时显示完整错误。
    if (
        not response.ok
        or not response_payload.get("success")
    ):
        raise RuntimeError(
            "D1 查询失败："
            f"HTTP {response.status_code}\n"
            f"{response_payload.get('errors') or response_payload}"
        )

    # 返回 D1 result 数组。
    return response_payload.get(
        "result",
        [],
    )


def d1_result_to_dataframe(
    result,
):
    """把 D1 result 转成 pandas DataFrame。"""

    # D1 查询结果通常存放在 result[0]["results"]。
    rows = (
        result[0].get(
            "results",
            [],
        )
        if result
        else []
    )

    # 返回 DataFrame。
    return pd.DataFrame(
        rows
    )


def query_dataframe(
    query_name,
    sql,
):
    """执行一条只读 SQL，并显示查询取得的记录数量。"""

    # 显示正在读取的数据域。
    print(
        f"Reading {query_name} from D1..."
    )

    # 执行 SQL。
    result = execute_d1_select(
        sql
    )

    # 转换成 DataFrame。
    dataframe = d1_result_to_dataframe(
        result
    )

    # 显示本次返回的行数。
    print(
        f"{query_name} rows returned:",
        len(dataframe),
    )

    # 返回查询结果。
    return dataframe


# ============================================================
# 三、JSON 聚合工具
# ============================================================

def to_python_scalar(
    value,
):
    """把 pandas/NumPy scalar 转成 JSON 可以序列化的 Python 值。"""

    # pandas 空值统一保存为 JSON null。
    if pd.isna(value):
        return None

    # NumPy scalar 通常提供 item()，可转成标准 Python scalar。
    if hasattr(
        value,
        "item",
    ):
        try:
            return value.item()
        except Exception:
            pass

    # 普通 Python 值直接返回。
    return value


def rows_to_utf8_json(
    dataframe,
    value_columns,
):
    """把一个 group 的多条记录转换成稳定、有顺序的 UTF-8 JSON。"""

    # 保存当前 group 的每一条结构化记录。
    records = []

    # 按当前 DataFrame 顺序逐行处理。
    for row in dataframe[
        value_columns
    ].to_dict(
        orient="records"
    ):
        # 把每个值转换成标准 Python scalar。
        clean_row = {
            column_name: to_python_scalar(
                column_value
            )
            for (
                column_name,
                column_value,
            )
            in row.items()
        }

        # 保存当前记录。
        records.append(
            clean_row
        )

    # ensure_ascii=False 保留原始 Unicode 字符。
    return json.dumps(
        records,
        ensure_ascii=False,
        separators=(
            ",",
            ":",
        ),
    )


def aggregate_child_rows(
    dataframe,
    key_column,
    json_column_name,
    count_column_name,
    value_columns,
    sort_columns,
):
    """将一对多子表聚合成每个父记录一行。"""

    # 子表没有记录时返回只有目标 column 的空 DataFrame。
    if dataframe.empty:
        return pd.DataFrame(
            columns=[
                key_column,
                json_column_name,
                count_column_name,
            ]
        )

    # 稳定排序确保相同数据库内容每次生成相同 JSON。
    sorted_dataframe = dataframe.sort_values(
        sort_columns,
        kind="stable",
        na_position="last",
    ).copy()

    # 逐个父 id 聚合。
    aggregated_rows = []

    # groupby 不排序，沿用前面稳定排序后的顺序。
    for (
        parent_id,
        group_dataframe,
    ) in sorted_dataframe.groupby(
        key_column,
        sort=False,
        dropna=False,
    ):
        # 保存一个父记录的 JSON 和数量。
        aggregated_rows.append(
            {
                key_column: to_python_scalar(
                    parent_id
                ),
                json_column_name: rows_to_utf8_json(
                    group_dataframe,
                    value_columns,
                ),
                count_column_name: int(
                    len(group_dataframe)
                ),
            }
        )

    # 返回一行一个父记录的结果。
    return pd.DataFrame(
        aggregated_rows
    )


# ============================================================
# 四、查询 1：Candidate/Application/Person/岗位主记录
# ============================================================

# 本查询只连接一对一或多对一表，保证一行对应一个 Candidate_snapshot。
BASE_SQL = """
SELECT
  cs."id" AS candidate_snapshot_id,
  a."id" AS application_id,
  a."person_id" AS person_id,
  cs."application_id" AS candidate_application_id,

  a."company_id" AS ml_mapped_company_id,
  c."company_name" AS ml_mapped_company_name,
  a."position_id" AS ml_mapped_position_id,
  p."position_name" AS ml_mapped_position_name,
  p."company_id" AS position_company_id,
  pc."company_name" AS position_company_name,

  a."application_cluster_id" AS application_cluster_id,
  ac."application_cycle_id" AS application_cycle_id,
  pac."position_id" AS application_cycle_position_id,
  a."hiring_pipeline_id" AS hiring_pipeline_id,
  a."current_stage_id" AS current_stage_id,

  LOWER(
    TRIM(
      a."applied_company_name"
    )
  ) AS normalized_applied_company_name,
  LOWER(
    TRIM(
      a."applied_position_name"
    )
  ) AS normalized_applied_position_name,
  a."application_cluster_display_name"
    AS application_cluster_display_name,

  a."application_status" AS application_status,
  a."applied_at" AS applied_at,
  a."current_stage_entered_at" AS current_stage_entered_at,

  cs."application_date" AS candidate_application_date,
  cs."available_start_date" AS available_start_date,
  cs."expected_end_date" AS expected_end_date,
  cs."work_duration" AS work_duration,
  cs."work_type" AS work_type,
  cs."normalized_person_name" AS normalized_person_name,
  cs."normalized_first_name" AS normalized_first_name,
  cs."normalized_middle_name" AS normalized_middle_name,
  cs."normalized_last_name" AS normalized_last_name,
  cs."normalized_email_address" AS normalized_email_address,
  cs."normalized_phone" AS normalized_phone,
  cs."normalized_linkedin_url" AS normalized_linkedin_url,
  cs."normalized_github_url" AS normalized_github_url,
  cs."raw_resume_text" AS raw_resume_text,

  person."highest_education_id" AS person_highest_education_id,
  person."latest_candidate_id" AS person_latest_candidate_id,
  person."talent_pool_status" AS talent_pool_status,

  p."position_jd" AS position_jd,
  p."function_id" AS position_function_id,
  fn."function_name" AS position_function_name,
  p."employment_type_id" AS position_employment_type_id,
  pet."employment_type_name" AS position_employment_type_name,
  p."seniority_id" AS position_seniority_id,
  sen."seniority_name" AS position_seniority_name,
  p."location_id" AS position_location_id,
  city."city_name" AS position_city_name,
  state."state_name" AS position_state_name,
  country."country_name" AS position_country_name,

  ac."cluster_code" AS application_cluster_code,
  ac."display_name" AS database_application_cluster_display_name,
  ac."status" AS application_cluster_status,
  pac."cycle_name" AS database_application_cycle_name,
  hp."hiring_pipeline_name" AS hiring_pipeline_name,
  hp."status" AS hiring_pipeline_status,
  ps."pipeline_stage_name" AS current_stage_name,
  ps."stage_type" AS current_stage_type,
  ps."sequence_no" AS current_stage_sequence_no,
  ps."is_initial" AS current_stage_is_initial,
  ps."is_terminal" AS current_stage_is_terminal

FROM "Candidate_snapshot" AS cs

INNER JOIN "Application" AS a
  ON a."id" = cs."application_id"

INNER JOIN "Person" AS person
  ON person."id" = a."person_id"

INNER JOIN "Company" AS c
  ON c."id" = a."company_id"

INNER JOIN "Position" AS p
  ON p."id" = a."position_id"

INNER JOIN "Company" AS pc
  ON pc."id" = p."company_id"

LEFT JOIN "Function" AS fn
  ON fn."id" = p."function_id"

LEFT JOIN "Position_employment_type" AS pet
  ON pet."id" = p."employment_type_id"

LEFT JOIN "Seniority" AS sen
  ON sen."id" = p."seniority_id"

LEFT JOIN "Location" AS loc
  ON loc."id" = p."location_id"

LEFT JOIN "City" AS city
  ON city."id" = loc."city_id"

LEFT JOIN "State" AS state
  ON state."id" = loc."state_id"

LEFT JOIN "Country" AS country
  ON country."id" = loc."country_id"

INNER JOIN "Application_cluster" AS ac
  ON ac."id" = a."application_cluster_id"

INNER JOIN "Position_application_cycle" AS pac
  ON pac."id" = ac."application_cycle_id"

INNER JOIN "Hiring_pipeline" AS hp
  ON hp."id" = a."hiring_pipeline_id"

LEFT JOIN "Pipeline_stage" AS ps
  ON ps."id" = a."current_stage_id"

ORDER BY cs."id";
"""


# ============================================================
# 五、查询 2：只读取每位 Candidate 的最高学历
# ============================================================

# Candidate_Education.is_highest_degree = 1 是学历导入 Cell 最终确认的最高学历。
HIGHEST_EDUCATION_SQL = """
SELECT
  ce."candidate_id" AS candidate_snapshot_id,
  ce."id" AS candidate_education_id,
  ce."person_education_id" AS person_education_id,
  pe."education_id" AS education_id,
  pe."person_id" AS education_person_id,
  pe."source_candidate_id"
    AS education_source_candidate_id,
  ce."is_highest_degree" AS is_highest_degree,

  e."degree_id" AS degree_id,
  d."degree_name" AS highest_degree_name,
  d."degree_level_rank" AS highest_degree_level_rank,

  e."school_id" AS school_id,
  s."school_name" AS highest_school_name,
  e."field_study_id" AS field_study_id,
  fs."field_study_name" AS highest_field_study_name,

  e."major_id" AS major_id,
  m."major_name" AS highest_major_name,

  e."gpa" AS highest_education_gpa,
  e."education_start_date"
    AS highest_education_start_date,
  e."education_end_date"
    AS highest_education_end_date,
  e."is_current" AS highest_education_is_current,
  e."education_description"
    AS highest_education_description

FROM "Candidate_Education" AS ce

INNER JOIN "Person_Education" AS pe
  ON pe."id" = ce."person_education_id"

INNER JOIN "Education" AS e
  ON e."id" = pe."education_id"

LEFT JOIN "Degree" AS d
  ON d."id" = e."degree_id"

LEFT JOIN "School" AS s
  ON s."id" = e."school_id"

LEFT JOIN "Field_study" AS fs
  ON fs."id" = e."field_study_id"

LEFT JOIN "Major" AS m
  ON m."id" = e."major_id"

WHERE ce."is_highest_degree" = 1

ORDER BY ce."candidate_id", ce."id";
"""


# ============================================================
# 六、查询 3：Candidate 技能
# ============================================================

SKILL_SQL = """
SELECT
  cs."candidate_id" AS candidate_snapshot_id,
  cs."id" AS candidate_skill_id,
  cs."person_skill_id" AS person_skill_id,
  ps."person_id" AS skill_person_id,
  ps."source_candidate_id" AS skill_source_candidate_id,
  ps."skill_id" AS skill_id,
  skill."skill_name" AS skill_name,
  skill."skill_type_id" AS skill_type_id,
  st."skill_type_name" AS skill_type_name,
  ps."recorded_at" AS skill_recorded_at

FROM "Candidate_Skill" AS cs

INNER JOIN "Person_Skill" AS ps
  ON ps."id" = cs."person_skill_id"

INNER JOIN "Skill" AS skill
  ON skill."id" = ps."skill_id"

INNER JOIN "Skill_type" AS st
  ON st."id" = skill."skill_type_id"

ORDER BY
  cs."candidate_id",
  st."skill_type_name",
  skill."skill_name",
  cs."id";
"""


# ============================================================
# 七、查询 4：Candidate 项目经历
# ============================================================

PROJECT_SQL = """
SELECT
  cp."candidate_id" AS candidate_snapshot_id,
  cp."id" AS candidate_project_id,
  cp."person_project_id" AS person_project_id,
  pp."person_id" AS project_person_id,
  pp."source_candidate_id" AS project_source_candidate_id,
  pp."project_name" AS project_name,
  pp."project_role" AS project_role,
  pp."project_description" AS project_description,
  pp."project_url" AS project_url,
  pp."start_date" AS project_start_date,
  pp."end_date" AS project_end_date

FROM "Candidate_Project" AS cp

INNER JOIN "Person_Project" AS pp
  ON pp."id" = cp."person_project_id"

ORDER BY
  cp."candidate_id",
  pp."start_date",
  pp."end_date",
  pp."project_name",
  cp."id";
"""


# ============================================================
# 八、查询 5：Candidate 工作经历
# ============================================================

EMPLOYMENT_SQL = """
SELECT
  cp."candidate_id" AS candidate_snapshot_id,
  cp."id" AS candidate_position_id,
  cp."person_position_id" AS person_position_id,
  pp."person_id" AS employment_person_id,
  pp."source_candidate_id"
    AS employment_source_candidate_id,

  pp."normalized_company_name"
    AS normalized_employment_company_name,
  pp."normalized_position_name"
    AS normalized_employment_position_name,
  pp."position_description" AS position_description,
  pp."position_start_date"
    AS employment_start_date,
  pp."position_end_date"
    AS employment_end_date,
  pp."experience_type" AS employment_experience_type,
  pp."is_current" AS employment_is_current

FROM "Candidate_Position" AS cp

INNER JOIN "Person_Position" AS pp
  ON pp."id" = cp."person_position_id"

ORDER BY
  cp."candidate_id",
  pp."position_start_date",
  pp."position_end_date",
  pp."normalized_company_name",
  pp."normalized_position_name",
  cp."id";
"""


# ============================================================
# 九、查询 6：ML 映射岗位的薪资
# ============================================================

POSITION_SALARY_SQL = """
SELECT
  psr."position_id" AS ml_mapped_position_id,
  psr."id" AS position_salary_range_id,
  psr."salary_min_cents" AS salary_min_cents,
  psr."salary_max_cents" AS salary_max_cents,
  psr."currency" AS salary_currency,
  psr."salary_type" AS salary_type

FROM "Position_salary_range" AS psr

WHERE psr."position_id" IN (
  SELECT DISTINCT
    a."position_id"
  FROM "Application" AS a
)

ORDER BY
  psr."position_id",
  psr."id";
"""


# ============================================================
# 十、一次性执行六个数据域查询
# ============================================================

# 读取一行一个 Candidate 的主表。
base_dataframe = query_dataframe(
    "candidate/application base records",
    BASE_SQL,
)

# 主表没有记录时停止。
if base_dataframe.empty:
    raise ValueError(
        "Candidate_snapshot/Application 没有可导出的记录。"
    )

# 读取最高学历。
highest_education_dataframe = query_dataframe(
    "highest education",
    HIGHEST_EDUCATION_SQL,
)

# 读取技能。
skill_dataframe = query_dataframe(
    "candidate skills",
    SKILL_SQL,
)

# 读取项目经历。
project_dataframe = query_dataframe(
    "candidate projects",
    PROJECT_SQL,
)

# 读取工作经历。
employment_dataframe = query_dataframe(
    "candidate employment",
    EMPLOYMENT_SQL,
)

# 只读取 Application 实际映射到的 Position 的薪资。
position_salary_dataframe = query_dataframe(
    "mapped position salary ranges",
    POSITION_SALARY_SQL,
)


# ============================================================
# 十点一、把全库只读查询结果收窄到当前 application.csv
# ============================================================

# 当前 CSV 的 Application.id 只要求是 D1 Application 的子集。
d1_application_ids = set(
    int(value)
    for value in base_dataframe["application_id"].tolist()
)

application_ids_missing_from_d1 = sorted(
    current_application_ids - d1_application_ids
)

if application_ids_missing_from_d1:
    raise ValueError(
        "当前 application.csv 中存在无法映射到 D1 Candidate/Application 的 ID："
        f"{application_ids_missing_from_d1[:20]}"
    )

# 主表只保留当前批次 Application。
base_dataframe = base_dataframe.loc[
    base_dataframe["application_id"].astype("int64").isin(
        current_application_ids
    )
].copy().reset_index(drop=True)

# 当前批次 Candidate ids 与 Position ids 用于收窄所有一对多数据域。
current_candidate_ids = set(
    int(value)
    for value in base_dataframe["candidate_snapshot_id"].tolist()
)
current_position_ids = set(
    int(value)
    for value in base_dataframe["ml_mapped_position_id"].tolist()
)

def keep_current_candidates(dataframe):
    """只保留当前批次 Candidate；空表继续保持为空。"""
    if dataframe.empty:
        return dataframe.copy()
    return dataframe.loc[
        pd.to_numeric(
            dataframe["candidate_snapshot_id"],
            errors="coerce",
        ).isin(current_candidate_ids)
    ].copy().reset_index(drop=True)

highest_education_dataframe = keep_current_candidates(
    highest_education_dataframe
)
skill_dataframe = keep_current_candidates(skill_dataframe)
project_dataframe = keep_current_candidates(project_dataframe)
employment_dataframe = keep_current_candidates(employment_dataframe)

# 薪资只保留当前批次实际映射岗位。
if not position_salary_dataframe.empty:
    position_salary_dataframe = position_salary_dataframe.loc[
        pd.to_numeric(
            position_salary_dataframe["ml_mapped_position_id"],
            errors="coerce",
        ).isin(current_position_ids)
    ].copy().reset_index(drop=True)

print(
    "Incremental ML export mapping validation:",
    {
        "current_application_count": len(current_application_ids),
        "current_candidate_count": len(current_candidate_ids),
        "missing_application_count": len(application_ids_missing_from_d1),
    },
)


# ============================================================
# 十一、验证主记录和最高学历关系
# ============================================================

# Candidate_snapshot.id 在最终结果中必须唯一。
if base_dataframe[
    "candidate_snapshot_id"
].duplicated().any():
    raise RuntimeError(
        "主查询出现重复 candidate_snapshot_id，不能生成一行一个 Candidate 的宽表。"
    )

# Application.id 在当前设计中也必须唯一。
if base_dataframe[
    "application_id"
].duplicated().any():
    raise RuntimeError(
        "主查询出现重复 application_id。"
    )

# 每个 Candidate 最多只能有一条最高学历。
if (
    not highest_education_dataframe.empty
    and highest_education_dataframe[
        "candidate_snapshot_id"
    ].duplicated().any()
):
    duplicate_candidate_ids = (
        highest_education_dataframe.loc[
            highest_education_dataframe[
                "candidate_snapshot_id"
            ].duplicated(
                keep=False
            ),
            "candidate_snapshot_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise RuntimeError(
        "发现 Candidate 存在多条 is_highest_degree = 1 的学历："
        f"{duplicate_candidate_ids[:20]}"
    )


# ============================================================
# 十二、将一对多记录聚合成 UTF-8 JSON
# ============================================================

# 技能 JSON 中保存可追溯 ID、技能名称、类型和熟练度。
skill_aggregate_dataframe = aggregate_child_rows(
    dataframe=skill_dataframe,
    key_column="candidate_snapshot_id",
    json_column_name="candidate_skills_json",
    count_column_name="candidate_skill_count",
    value_columns=[
        "candidate_skill_id",
        "person_skill_id",
        "skill_person_id",
        "skill_source_candidate_id",
        "skill_id",
        "skill_name",
        "skill_type_id",
        "skill_type_name",
        "skill_recorded_at",
    ],
    sort_columns=[
        "candidate_snapshot_id",
        "skill_type_name",
        "skill_name",
        "candidate_skill_id",
    ],
)

# 项目 JSON 中明确保留 project_description 原始长文本。
project_aggregate_dataframe = aggregate_child_rows(
    dataframe=project_dataframe,
    key_column="candidate_snapshot_id",
    json_column_name="candidate_projects_json",
    count_column_name="candidate_project_count",
    value_columns=[
        "candidate_project_id",
        "person_project_id",
        "project_person_id",
        "project_source_candidate_id",
        "project_name",
        "project_role",
        "project_description",
        "project_url",
        "project_start_date",
        "project_end_date",
    ],
    sort_columns=[
        "candidate_snapshot_id",
        "project_start_date",
        "project_end_date",
        "project_name",
        "candidate_project_id",
    ],
)

# 工作经历 JSON 中明确保留 position_description 原始长文本。
employment_aggregate_dataframe = aggregate_child_rows(
    dataframe=employment_dataframe,
    key_column="candidate_snapshot_id",
    json_column_name="candidate_employment_json",
    count_column_name="candidate_employment_count",
    value_columns=[
        "candidate_position_id",
        "person_position_id",
        "employment_person_id",
        "employment_source_candidate_id",
        "normalized_employment_company_name",
        "normalized_employment_position_name",
        "position_description",
        "employment_start_date",
        "employment_end_date",
        "employment_experience_type",
        "employment_is_current",
    ],
    sort_columns=[
        "candidate_snapshot_id",
        "employment_start_date",
        "employment_end_date",
        "normalized_employment_company_name",
        "normalized_employment_position_name",
        "candidate_position_id",
    ],
)

# 岗位薪资允许一条 Position 对应多条范围，因此也保存为 JSON。
salary_aggregate_dataframe = aggregate_child_rows(
    dataframe=position_salary_dataframe,
    key_column="ml_mapped_position_id",
    json_column_name="position_salary_ranges_json",
    count_column_name="position_salary_range_count",
    value_columns=[
        "position_salary_range_id",
        "salary_min_cents",
        "salary_max_cents",
        "salary_currency",
        "salary_type",
    ],
    sort_columns=[
        "ml_mapped_position_id",
        "position_salary_range_id",
    ],
)


# ============================================================
# 十三、合并为一张一行一个 Candidate 的 ML CSV
# ============================================================

# 从主记录开始构建最终结果。
ml_dataframe = base_dataframe.copy()

# 通过 candidate_snapshot_id 合并最高学历。
ml_dataframe = ml_dataframe.merge(
    highest_education_dataframe,
    on="candidate_snapshot_id",
    how="left",
    validate="one_to_one",
)

# 合并技能 JSON。
ml_dataframe = ml_dataframe.merge(
    skill_aggregate_dataframe,
    on="candidate_snapshot_id",
    how="left",
    validate="one_to_one",
)

# 合并项目 JSON。
ml_dataframe = ml_dataframe.merge(
    project_aggregate_dataframe,
    on="candidate_snapshot_id",
    how="left",
    validate="one_to_one",
)

# 合并工作经历 JSON。
ml_dataframe = ml_dataframe.merge(
    employment_aggregate_dataframe,
    on="candidate_snapshot_id",
    how="left",
    validate="one_to_one",
)

# 通过 ML 映射岗位 id 合并岗位薪资 JSON。
ml_dataframe = ml_dataframe.merge(
    salary_aggregate_dataframe,
    on="ml_mapped_position_id",
    how="left",
    validate="many_to_one",
)

# 没有子记录时统一保存为空 JSON array。
for json_column_name in [
    "candidate_skills_json",
    "candidate_projects_json",
    "candidate_employment_json",
    "position_salary_ranges_json",
]:
    ml_dataframe[
        json_column_name
    ] = ml_dataframe[
        json_column_name
    ].fillna(
        "[]"
    )

# 没有子记录时数量统一保存为 0。
for count_column_name in [
    "candidate_skill_count",
    "candidate_project_count",
    "candidate_employment_count",
    "position_salary_range_count",
]:
    ml_dataframe[
        count_column_name
    ] = (
        ml_dataframe[
            count_column_name
        ]
        .fillna(
            0
        )
        .astype(
            "int64"
        )
    )

# 根据最高学历 id 生成是否拥有可靠最高学历的特征。
ml_dataframe[
    "has_highest_education"
] = (
    ml_dataframe[
        "education_id"
    ]
    .notna()
    .astype(
        "int8"
    )
)

# 生成是否有技能的特征。
ml_dataframe[
    "has_candidate_skills"
] = (
    ml_dataframe[
        "candidate_skill_count"
    ]
    .gt(
        0
    )
    .astype(
        "int8"
    )
)

# 生成是否有项目经历的特征。
ml_dataframe[
    "has_candidate_projects"
] = (
    ml_dataframe[
        "candidate_project_count"
    ]
    .gt(
        0
    )
    .astype(
        "int8"
    )
)

# 生成是否有工作经历的特征。
ml_dataframe[
    "has_candidate_employment"
] = (
    ml_dataframe[
        "candidate_employment_count"
    ]
    .gt(
        0
    )
    .astype(
        "int8"
    )
)

# 原始简历为空时长度保存为 0。
ml_dataframe[
    "raw_resume_text_length_chars"
] = (
    ml_dataframe[
        "raw_resume_text"
    ]
    .fillna(
        ""
    )
    .astype(
        "string"
    )
    .str.len()
    .astype(
        "int64"
    )
)

# Position JD 为空时长度保存为 0。
ml_dataframe[
    "position_jd_length_chars"
] = (
    ml_dataframe[
        "position_jd"
    ]
    .fillna(
        ""
    )
    .astype(
        "string"
    )
    .str.len()
    .astype(
        "int64"
    )
)

# 保存导出规则版本。
ml_dataframe.insert(
    0,
    "ml_export_version",
    ML_EXPORT_VERSION,
)

# 保存本次导出的 UTC 时间。
ml_dataframe.insert(
    1,
    "ml_exported_at",
    datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    ),
)

# 按 Candidate id 稳定排序。
ml_dataframe = (
    ml_dataframe.sort_values(
        [
            "candidate_snapshot_id",
            "application_id",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 十四、最终验证
# ============================================================

# 合并后记录数必须与主查询完全一致。
if len(
    ml_dataframe
) != len(
    base_dataframe
):
    raise RuntimeError(
        "合并后记录数与 Candidate_snapshot 主查询不一致。"
    )

# 合并后 Candidate id 仍然必须唯一。
if ml_dataframe[
    "candidate_snapshot_id"
].duplicated().any():
    raise RuntimeError(
        "最终结果出现重复 candidate_snapshot_id。"
    )

# 合并后 Application id 仍然必须唯一。
if ml_dataframe[
    "application_id"
].duplicated().any():
    raise RuntimeError(
        "最终结果出现重复 application_id。"
    )

# 验证四个 JSON columns 都可以被 JSON parser 读取。
for json_column_name in [
    "candidate_skills_json",
    "candidate_projects_json",
    "candidate_employment_json",
    "position_salary_ranges_json",
]:
    for json_text in ml_dataframe[
        json_column_name
    ]:
        json.loads(
            json_text
        )


# ============================================================
# 十五、固定覆盖 UTF-8 CSV
# ============================================================

# 创建输出目录。
os.makedirs(
    ML_OUTPUT_FOLDER,
    exist_ok=True,
)

# 临时文件与最终文件位于同一 Google Drive 目录。
temporary_output_path = (
    f"{ML_OUTPUT_CSV_PATH}.tmp"
)

# 用 utf-8-sig 写出。
#
# utf-8-sig 本质上仍是 UTF-8，只是文件开头增加 BOM，
# 让 Google Drive、Excel 和其他表格工具更容易自动识别编码。
ml_dataframe.to_csv(
    temporary_output_path,
    index=False,
    encoding="utf-8-sig",
)

# 原子替换固定输出文件。
os.replace(
    temporary_output_path,
    ML_OUTPUT_CSV_PATH,
)


# ============================================================
# 十六、输出统计和预览
# ============================================================

# 显示最终统计。
print(
    "\nML export summary:"
)
print(
    {
        "candidate_record_count": int(
            len(
                ml_dataframe
            )
        ),
        "column_count": int(
            len(
                ml_dataframe.columns
            )
        ),
        "highest_education_present_count": int(
            ml_dataframe[
                "has_highest_education"
            ].sum()
        ),
        "skills_present_count": int(
            ml_dataframe[
                "has_candidate_skills"
            ].sum()
        ),
        "projects_present_count": int(
            ml_dataframe[
                "has_candidate_projects"
            ].sum()
        ),
        "employment_present_count": int(
            ml_dataframe[
                "has_candidate_employment"
            ].sum()
        ),
        "raw_resume_text_present_count": int(
            ml_dataframe[
                "raw_resume_text_length_chars"
            ].gt(
                0
            ).sum()
        ),
    }
)

# 输出所有 columns，方便核对。
print(
    "\nFinal CSV columns:"
)
print(
    list(
        ml_dataframe.columns
    )
)

# 只预览较短且关键的 columns，避免 Colab 直接渲染完整简历长文本。
display(
    ml_dataframe[
        [
            "candidate_snapshot_id",
            "application_id",
            "person_id",
            "normalized_applied_company_name",
            "normalized_applied_position_name",
            "ml_mapped_company_id",
            "ml_mapped_company_name",
            "ml_mapped_position_id",
            "ml_mapped_position_name",
            "highest_degree_name",
            "highest_school_name",
            "candidate_skill_count",
            "candidate_project_count",
            "candidate_employment_count",
            "raw_resume_text_length_chars",
            "application_status",
        ]
    ].head(
        10
    )
)

# 显示最终文件路径。
print(
    "\nApplication ML dataset saved to:",
    ML_OUTPUT_CSV_PATH,
)


# ML

In [ ]:
!pip install sentence-transformers -q

In [ ]:
"""
recruitment_analysis_pipeline.py（Google Colab 版，自包含）

四步全部封装在一个 class 里（RecruitmentAnalysisPipeline），
不在 class 外面另外定义任何函数：

  第一步（方案三）：候选人画像聚类，生成散点图展示数据分布
  第二步（方案四）：最简单的基准打分卡，存CSV
  第三步（方案二）：异常检测，排除不合适的数据点
  第四步（方案一）：只对第三步剩下的数据，做简历-JD相似度排序，存CSV

embedding 模型说明：
  用 sentence-transformers 的 all-MiniLM-L6-v2——这是本地离线跑语义
  相似度最常用的轻量模型，CPU 就能跑，不需要GPU，下载一次之后
  完全离线运行，适合 Colab 环境。下载模型这一行单独写在下面，
  运行前需要先执行：

      !pip install sentence-transformers -q

  这两行需要联网（第一次下载依赖库和模型权重），之后同一个 Colab
  session 内重复调用不会再重新下载。

本次修正：
  补上了 numpy 的 import。第四步 run_similarity_ranking() 里用到了
  np.array / np.dot / np.linalg.norm，但之前顶部没有 import numpy，
  跑到第四步会直接抛 NameError: name 'np' is not defined。
  除此之外没有改动任何逻辑。

用法（Colab cell 里直接调用，不需要任何前置变量）：
    pipeline = RecruitmentAnalysisPipeline()
    pipeline.run_all()
"""

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from google.colab import drive


# 本 Cell 完全独立，负责自行挂载 Google Drive。
drive.mount("/content/drive")


class RecruitmentAnalysisPipeline:
    DEFAULT_INPUT_CSV_PATH = (
        "/content/drive/MyDrive/recruiting-pipeline"
        "/datasets/ml/application_ml_feature_dataset.csv"
    )
    DEFAULT_OUTPUT_FOLDER = (
        "/content/drive/MyDrive/recruiting-pipeline"
        "/datasets/ml/ml_results"
    )

    REQUIRED_COLUMNS = [
        "candidate_snapshot_id", "application_id",
        "ml_mapped_position_id", "ml_mapped_position_name",
        "ml_mapped_company_id", "ml_mapped_company_name",
        "raw_resume_text", "position_jd",
        "raw_resume_text_length_chars", "position_jd_length_chars",
        "has_highest_education", "has_candidate_skills", "has_candidate_employment",
        "candidate_skill_count", "candidate_employment_count", "candidate_project_count",
        "highest_degree_level_rank", "position_seniority_name",
    ]

    NUMERIC_CLUSTER_FEATURES = [
        "highest_degree_level_rank",
        "candidate_skill_count",
        "candidate_employment_count",
        "candidate_project_count",
    ]

    MIN_CANDIDATES_FOR_CLUSTERING = 10
    DEFAULT_N_CLUSTERS = 3

    MAX_SCORE_EDUCATION_COMPLETENESS = 20
    MAX_SCORE_DEGREE_LEVEL = 20
    MAX_SCORE_SKILLS = 20
    MAX_SCORE_EMPLOYMENT = 20
    MAX_SCORE_SENIORITY_ALIGNMENT = 20
    DEGREE_LEVEL_RANK_CAP = 5
    SKILL_COUNT_CAP = 5
    EMPLOYMENT_COUNT_CAP = 3
    SENIOR_ROLE_KEYWORDS = ["senior", "lead", "director", "manager", "mid-senior", "principal"]

    MIN_USABLE_RESUME_LENGTH_CHARS = 50

    EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

    def __init__(self):
        self.raw_dataframe = None
        self.cluster_result_dataframe = None
        self.scorecard_result_dataframe = None
        self.anomaly_result_dataframe = None
        self.clean_dataframe = None
        self.ranking_result_dataframe = None
        self._embedding_model = None

    # ------------------------------------------------------------
    # 数据读取（现在读CSV；以后切换成读数据库，只改这一个方法）
    # ------------------------------------------------------------
    def load_data(self, csv_path: str = None) -> pd.DataFrame:
        import os

        csv_path = csv_path or self.DEFAULT_INPUT_CSV_PATH

        if not os.path.isfile(csv_path):
            raise FileNotFoundError(f"找不到必要输入文件：{csv_path}")

        df = pd.read_csv(csv_path, encoding="utf-8-sig")

        missing_columns = [c for c in self.REQUIRED_COLUMNS if c not in df.columns]
        if missing_columns:
            raise ValueError(f"输入数据缺少必要列: {missing_columns}")

        self.raw_dataframe = df[self.REQUIRED_COLUMNS].copy()
        return self.raw_dataframe

    # ------------------------------------------------------------
    # 第一步（方案三）：聚类 + 散点图
    # ------------------------------------------------------------
    def run_clustering_and_plot(self, output_folder: str = None) -> pd.DataFrame:
        import os

        output_folder = output_folder or self.DEFAULT_OUTPUT_FOLDER
        os.makedirs(output_folder, exist_ok=True)

        df = self.raw_dataframe.copy()
        clustered_parts = []

        for position_id, position_df in df.groupby("ml_mapped_position_id"):
            position_df = position_df.copy()

            if len(position_df) < self.MIN_CANDIDATES_FOR_CLUSTERING:
                position_df["cluster_label"] = None
                position_df["cluster_note"] = (
                    f"该职位候选人数({len(position_df)})少于{self.MIN_CANDIDATES_FOR_CLUSTERING}，不做聚类"
                )
                clustered_parts.append(position_df)
                continue

            feature_matrix = position_df[self.NUMERIC_CLUSTER_FEATURES].fillna(0).values
            scaler = StandardScaler()
            scaled_features = scaler.fit_transform(feature_matrix)

            n_clusters = min(self.DEFAULT_N_CLUSTERS, len(position_df))
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            position_df["cluster_label"] = kmeans.fit_predict(scaled_features)
            position_df["cluster_note"] = None
            clustered_parts.append(position_df)

        result_df = pd.concat(clustered_parts, ignore_index=True)
        self.cluster_result_dataframe = result_df

        # 生成散点图：把全部数值特征用PCA降到2维，方便在一张图里
        # 展示整体分布（不同职位、不同cluster的候选人整体长什么样）
        clustered_only = result_df[result_df["cluster_label"].notna()].copy()
        plt.figure(figsize=(10, 7))

        # 增量批次中每个岗位可能少于10人，此时没有可做 PCA 的聚类记录。
        # 这属于合法的首批/小批次结果，仍然覆盖生成一张说明性空图。
        if clustered_only.empty:
            plt.text(
                0.5,
                0.5,
                "No position has enough candidates for clustering in this batch",
                ha="center",
                va="center",
            )
        else:
            feature_matrix_all = clustered_only[self.NUMERIC_CLUSTER_FEATURES].fillna(0).values
            scaled_all = StandardScaler().fit_transform(feature_matrix_all)
            pca_coords = PCA(n_components=2, random_state=42).fit_transform(scaled_all)

            clustered_only["_pca_x"] = pca_coords[:, 0]
            clustered_only["_pca_y"] = pca_coords[:, 1]

            scatter = plt.scatter(
                clustered_only["_pca_x"], clustered_only["_pca_y"],
                c=clustered_only["cluster_label"], cmap="tab10", alpha=0.6, s=20,
            )
            plt.colorbar(scatter, label="cluster_label")

        plt.title("Candidate Profile Distribution (PCA 2D projection, colored by cluster)")
        plt.xlabel("PCA component 1")
        plt.ylabel("PCA component 2")
        plt.tight_layout()

        plot_path = os.path.join(output_folder, "candidate_profile_scatter.png")
        plt.savefig(plot_path, dpi=150)
        plt.close()

        output_csv_path = os.path.join(output_folder, "candidate_profile_clusters.csv")
        result_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        print(f"[Step 1] Clustering done. Positions processed: {df['ml_mapped_position_id'].nunique()}")
        print(f"[Step 1] Clustered rows: {result_df['cluster_label'].notna().sum()} / {len(result_df)}")
        print(f"[Step 1] Scatter plot saved to: {plot_path}")
        print(f"[Step 1] Cluster CSV saved to: {output_csv_path}")

        return result_df

    # ------------------------------------------------------------
    # 第二步（方案四）：最简单的基准打分卡
    # ------------------------------------------------------------
    def _score_one_row(self, row) -> dict:
        breakdown = {}

        breakdown["education_completeness"] = (
            self.MAX_SCORE_EDUCATION_COMPLETENESS if row["has_highest_education"] == 1 else 0
        )

        degree_rank = row["highest_degree_level_rank"]
        if pd.isna(degree_rank):
            breakdown["degree_level"] = 0
        else:
            capped_rank = min(float(degree_rank), self.DEGREE_LEVEL_RANK_CAP)
            breakdown["degree_level"] = round(
                capped_rank / self.DEGREE_LEVEL_RANK_CAP * self.MAX_SCORE_DEGREE_LEVEL, 1
            )

        skill_count = row["candidate_skill_count"]
        if pd.isna(skill_count):
            breakdown["skills"] = 0
        else:
            capped_skill_count = min(float(skill_count), self.SKILL_COUNT_CAP)
            breakdown["skills"] = round(
                capped_skill_count / self.SKILL_COUNT_CAP * self.MAX_SCORE_SKILLS, 1
            )

        employment_count = row["candidate_employment_count"]
        if pd.isna(employment_count):
            breakdown["employment"] = 0
        else:
            capped_employment_count = min(float(employment_count), self.EMPLOYMENT_COUNT_CAP)
            breakdown["employment"] = round(
                capped_employment_count / self.EMPLOYMENT_COUNT_CAP * self.MAX_SCORE_EMPLOYMENT, 1
            )

        seniority_name = str(row["position_seniority_name"] or "").lower()
        is_senior_role = any(kw in seniority_name for kw in self.SENIOR_ROLE_KEYWORDS)
        if not is_senior_role:
            breakdown["seniority_alignment"] = self.MAX_SCORE_SENIORITY_ALIGNMENT / 2
        else:
            has_employment = (
                row["candidate_employment_count"] > 0
                if pd.notna(row["candidate_employment_count"]) else False
            )
            breakdown["seniority_alignment"] = (
                self.MAX_SCORE_SENIORITY_ALIGNMENT if has_employment else 0
            )

        return breakdown

    def run_scorecard(self, output_folder: str = None) -> pd.DataFrame:
        import os

        output_folder = output_folder or self.DEFAULT_OUTPUT_FOLDER
        os.makedirs(output_folder, exist_ok=True)

        df = self.raw_dataframe.copy()
        breakdowns = df.apply(self._score_one_row, axis=1)
        df["scorecard_breakdown_json"] = breakdowns.apply(lambda d: json.dumps(d, ensure_ascii=False))
        df["scorecard_total_score"] = breakdowns.apply(lambda d: round(sum(d.values()), 1))
        df["scorecard_percentile_within_position"] = (
            df.groupby("ml_mapped_position_id")["scorecard_total_score"].rank(pct=True)
        )

        self.scorecard_result_dataframe = df

        output_csv_path = os.path.join(output_folder, "candidate_scorecard.csv")
        df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        print(f"[Step 2] Scorecard done. Mean score: {df['scorecard_total_score'].mean():.1f}")
        print(f"[Step 2] Scorecard CSV saved to: {output_csv_path}")

        return df

    # ------------------------------------------------------------
    # 第三步（方案二）：异常检测，排除不合适的数据点
    # ------------------------------------------------------------
    def run_anomaly_filter(self, output_folder: str = None) -> pd.DataFrame:
        import os

        output_folder = output_folder or self.DEFAULT_OUTPUT_FOLDER
        os.makedirs(output_folder, exist_ok=True)

        df = self.raw_dataframe.copy()

        anomaly_flags_per_row = []
        for _, row in df.iterrows():
            flags = []

            resume_length = row["raw_resume_text_length_chars"]
            if pd.isna(resume_length) or resume_length < self.MIN_USABLE_RESUME_LENGTH_CHARS:
                flags.append("resume_missing_or_too_short")

            has_edu = row["has_highest_education"] == 1
            has_skills = row["has_candidate_skills"] == 1
            has_employment = row["has_candidate_employment"] == 1
            if not has_edu and not has_skills and not has_employment:
                flags.append("candidate_profile_almost_empty")

            seniority_name = str(row["position_seniority_name"] or "").lower()
            is_senior_role = any(kw in seniority_name for kw in self.SENIOR_ROLE_KEYWORDS)
            if is_senior_role and not has_employment:
                flags.append("senior_role_but_no_employment_history")

            anomaly_flags_per_row.append(flags)

        df["anomaly_flags_json"] = [json.dumps(f, ensure_ascii=False) for f in anomaly_flags_per_row]
        df["has_any_anomaly"] = [1 if f else 0 for f in anomaly_flags_per_row]

        self.anomaly_result_dataframe = df

        # 排除掉有异常标记的数据点，剩下的交给下一步做相似度排序
        self.clean_dataframe = df[df["has_any_anomaly"] == 0].copy()

        output_csv_path = os.path.join(output_folder, "anomaly_detection_result.csv")
        df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        print(f"[Step 3] Anomaly detection done. Flagged: {df['has_any_anomaly'].sum()} / {len(df)}")
        print(f"[Step 3] Remaining clean rows for next step: {len(self.clean_dataframe)}")
        print(f"[Step 3] Anomaly result CSV saved to: {output_csv_path}")

        return df

    # ------------------------------------------------------------
    # 第四步（方案一）：只对第三步剩下的数据，做相似度排序
    # ------------------------------------------------------------
    def _load_embedding_model(self):
        if self._embedding_model is None:
            from sentence_transformers import SentenceTransformer
            # 这一行会触发模型下载（第一次运行需要联网，之后离线复用）
            self._embedding_model = SentenceTransformer(self.EMBEDDING_MODEL_NAME)
        return self._embedding_model

    @staticmethod
    def _clean_text_for_similarity(text) -> str:
        if pd.isna(text):
            return ""
        return str(text).strip()

    def run_similarity_ranking(self, output_folder: str = None) -> pd.DataFrame:
        import os

        if self.clean_dataframe is None:
            raise RuntimeError("请先跑 run_anomaly_filter()，才有筛选后的数据可以排序。")

        output_folder = output_folder or self.DEFAULT_OUTPUT_FOLDER
        os.makedirs(output_folder, exist_ok=True)

        df = self.clean_dataframe.copy()

        # 当前批次可能全部被 anomaly 排除。此时生成带完整表头的空排名 CSV，
        # 不调用 embedding 模型；下游 OfferSelector 会同样安全生成空结果。
        if df.empty:
            df["match_score"] = pd.Series(dtype="float64")
            self.ranking_result_dataframe = df
            output_csv_path = os.path.join(output_folder, "similarity_ranking_result.csv")
            df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
            print("[Step 4] No clean rows remain after anomaly filtering.")
            print(f"[Step 4] Empty ranking CSV saved to: {output_csv_path}")
            return df

        model = self._load_embedding_model()

        resume_texts = [self._clean_text_for_similarity(t) for t in df["raw_resume_text"]]
        jd_texts = [self._clean_text_for_similarity(t) for t in df["position_jd"]]

        resume_embeddings = model.encode(resume_texts, show_progress_bar=False)
        jd_embeddings = model.encode(jd_texts, show_progress_bar=False)

        # 逐行算余弦相似度（每个候选人只和自己申请的那个职位比较，
        # 不是拿全部职位互相比较）
        match_scores = np.array([
            np.dot(resume_embeddings[i], jd_embeddings[i])
            / (np.linalg.norm(resume_embeddings[i]) * np.linalg.norm(jd_embeddings[i]) + 1e-8)
            for i in range(len(df))
        ])

        df["match_score"] = match_scores
        df = df.sort_values("match_score", ascending=False).reset_index(drop=True)

        self.ranking_result_dataframe = df

        output_csv_path = os.path.join(output_folder, "similarity_ranking_result.csv")
        df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        print(f"[Step 4] Similarity ranking done on {len(df)} clean rows.")
        print(f"[Step 4] Score range: {df['match_score'].min():.3f} ~ {df['match_score'].max():.3f}")
        print(f"[Step 4] Ranking CSV saved to: {output_csv_path}")

        return df

    # ------------------------------------------------------------
    # 总入口：按顺序跑完四步
    # ------------------------------------------------------------
    def run_all(self, input_csv_path: str = None, output_folder: str = None):
        self.load_data(input_csv_path)
        self.run_clustering_and_plot(output_folder)
        self.run_scorecard(output_folder)
        self.run_anomaly_filter(output_folder)
        self.run_similarity_ranking(output_folder)


# ============================================================
# 运行本 cell
# ============================================================

# 需要联网下载依赖和模型，先单独跑这一行：
# !pip install sentence-transformers -q

pipeline = RecruitmentAnalysisPipeline()
pipeline.run_all()


[Step 1] Clustering done. Positions processed: 14
[Step 1] Clustered rows: 0 / 20
[Step 1] Scatter plot saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/candidate_profile_scatter.png
[Step 1] Cluster CSV saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/candidate_profile_clusters.csv
[Step 2] Scorecard done. Mean score: 52.5
[Step 2] Scorecard CSV saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/candidate_scorecard.csv
[Step 3] Anomaly detection done. Flagged: 2 / 20
[Step 3] Remaining clean rows for next step: 18
[Step 3] Anomaly result CSV saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/anomaly_detection_result.csv


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Step 4] Similarity ranking done on 18 clean rows.
[Step 4] Score range: 0.123 ~ 0.451
[Step 4] Ranking CSV saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/similarity_ranking_result.csv


### pick

In [ ]:
"""
offer_selector.py（Google Colab 版，自包含、当前批次）

读取当前批次已经算好 match_score 的 similarity_ranking_result.csv，
按「每个岗位自己设定的匹配阈值」逐条判定是否推荐进入下一轮，
并固定覆盖 offer_selection_result.csv。

判定规则：
    match_score >= 该岗位的 ml_match_threshold  ->  recommended
    否则                                        ->  not_recommended

和上一版（组内取前10%）的区别：
    上一版按比例取人，需要先看到完整的候选人池才能决定谁入选，
    不适合实时逐条处理，而且组小的时候实际比例会严重偏离10%
    （3个人的组取前10%向上取整会选中1个，实际是33%）。
    本版改成按岗位固定阈值，同样的分数在同一个岗位永远得到
    同样的结果，来一条判一条，没有先来后到的差异。

阈值来源：
    数据库 Position.ml_match_threshold 这一列。
    现有岗位统一是 0.32（931条历史数据的中位数，约对应保留前50%）。
    新岗位由招聘发布者在发布表单里选择「期望的匹配程度」档位，
    再由 MATCH_LEVEL_TO_THRESHOLD 换算成余弦阈值。
    岗位查不到或该列为空时，回退到 DEFAULT_THRESHOLD 并如实记录来源。

关于余弦分数的重要说明（不是代码问题，是这个分数本身的性质）：
    余弦相似度的数值和人直觉理解的「百分之多少像」完全不是一回事。
    931条真实数据实测：分数范围只有 -0.046 ~ 0.656。
    阈值设 0.50 实际只有 6.3% 的人能通过，设 0.70 则一个都没有。
    所以招聘发布者不应该直接填余弦值，而应该选档位由系统换算。

用法（Colab cell 里直接调用，不需要任何前置变量）：
    selector = OfferSelector()
    result_df = selector.run()

    # 职位发布流程换算阈值时：
    threshold = OfferSelector.convert_match_level_to_threshold("strict")   # -> 0.38
"""

import os

import pandas as pd
import requests
from google.colab import drive, userdata


# 本 Cell 完全独立，负责自行挂载 Google Drive。
drive.mount("/content/drive")


class OfferSelector:
    DEFAULT_ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
    DEFAULT_DATABASE_ID = userdata.get('D1_DATABASE_ID')
    DEFAULT_CLOUDFLARE_KEY_SECRET_NAME = "CLOUDFLARE_KEY"

    DEFAULT_INPUT_CSV_PATH = (
        "/content/drive/MyDrive/recruiting-pipeline"
        "/datasets/ml/ml_results/similarity_ranking_result.csv"
    )
    DEFAULT_OUTPUT_CSV_PATH = (
        "/content/drive/MyDrive/recruiting-pipeline"
        "/datasets/ml/ml_results/offer_selection_result.csv"
    )

    REQUIRED_COLUMNS = [
        "ml_mapped_company_id",
        "ml_mapped_position_id",
        "match_score",
    ]

    # 岗位没有设定 ml_match_threshold 时的兜底值。
    # 0.32 是 931 条历史数据的中位数，大致对应「保留前50%的候选人」。
    DEFAULT_THRESHOLD = 0.32

    # 招聘发布表单里的「期望匹配程度」档位到余弦阈值的换算表。
    # 每一档括号里的比例，都是用 931 条真实历史数据算出来的分位数，
    # 不是估计值。发布表单应该让用户选这里的档位，而不是直接填数值。
    MATCH_LEVEL_TO_THRESHOLD = {
        "very_loose": 0.24,        # 约保留前 70%
        "loose": 0.28,             # 约保留前 60%
        "standard": 0.32,          # 约保留前 50%（当前默认）
        "moderate": 0.35,          # 约保留前 40%
        "strict": 0.38,            # 约保留前 30%
        "very_strict": 0.42,       # 约保留前 20%
        "highly_selective": 0.47,  # 约保留前 10%
    }

    def __init__(
        self,
        account_id: str = None,
        database_id: str = None,
        api_token: str = None,
    ):
        account_id = account_id or self.DEFAULT_ACCOUNT_ID
        database_id = database_id or self.DEFAULT_DATABASE_ID

        if api_token is None:
            api_token = userdata.get(self.DEFAULT_CLOUDFLARE_KEY_SECRET_NAME)

        if not api_token:
            raise ValueError(
                "未找到 CLOUDFLARE_KEY。"
                "请先在 Colab Secrets 中保存 Cloudflare API Token。"
            )

        self.api_url = (
            "https://api.cloudflare.com/client/v4/accounts/"
            f"{account_id}/d1/database/{database_id}/query"
        )
        self.headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        }

        self.input_dataframe = None
        self.result_dataframe = None
        self._position_threshold_cache = {}

    # ------------------------------------------------------------
    # 换算表：把发布表单里选的档位转成余弦阈值
    # ------------------------------------------------------------
    @classmethod
    def convert_match_level_to_threshold(cls, match_level: str) -> float:
        """
        供职位发布流程调用：把「期望匹配程度」档位换算成余弦阈值。

        档位名不在表里时直接报错，不静默回退到默认值——
        免得表单传了个拼错的值，岗位却被静默设成了标准档，
        招聘方以为设了严格档，实际跑的是别的标准。
        """
        key = (match_level or "").strip().lower()

        if key not in cls.MATCH_LEVEL_TO_THRESHOLD:
            raise ValueError(
                f"未知的匹配程度档位: {match_level!r}，"
                f"可选值: {sorted(cls.MATCH_LEVEL_TO_THRESHOLD.keys())}"
            )

        return cls.MATCH_LEVEL_TO_THRESHOLD[key]

    # ------------------------------------------------------------
    # 底层 D1 只读查询
    # ------------------------------------------------------------
    def _execute(self, sql: str, params: list = None, timeout: int = 30):
        payload = {"sql": sql}
        if params is not None:
            payload["params"] = params

        response = requests.post(
            self.api_url,
            headers=self.headers,
            json=payload,
            timeout=timeout,
        )

        try:
            response_data = response.json()
        except ValueError:
            return (
                False,
                f"HTTP {response.status_code}, response is not valid JSON: "
                f"{response.text[:300]}",
            )

        if response.status_code != 200 or not response_data.get("success", False):
            return False, response_data.get("errors", [])

        return True, response_data.get("result", [])

    # ------------------------------------------------------------
    # 一次性把本批次用到的岗位阈值全部查出来，不逐条查库
    # ------------------------------------------------------------
    def load_position_thresholds(self, position_ids: list) -> dict:
        unique_position_ids = sorted(
            {int(position_id) for position_id in position_ids if pd.notna(position_id)}
        )

        if not unique_position_ids:
            self._position_threshold_cache = {}
            return self._position_threshold_cache

        placeholders = ",".join("?" for _ in unique_position_ids)
        ok, result = self._execute(
            f'SELECT "id", "ml_match_threshold" FROM "Position" '
            f'WHERE "id" IN ({placeholders});',
            unique_position_ids,
        )

        if not ok:
            raise RuntimeError(f"查询 Position.ml_match_threshold 失败: {result}")

        rows = result[0].get("results", []) if result else []
        self._position_threshold_cache = {
            int(row["id"]): row["ml_match_threshold"] for row in rows
        }

        print(
            "Loaded thresholds:",
            f"{len(self._position_threshold_cache)} / {len(unique_position_ids)} positions",
        )
        return self._position_threshold_cache

    def _resolve_threshold(self, position_id):
        """
        取某个岗位的阈值。

        岗位查不到、或者这一列是空的时候，回退到 DEFAULT_THRESHOLD，
        并如实记录来源，方便事后区分「这条是按岗位设定判的」
        还是「按兜底值判的」。
        """
        if pd.isna(position_id):
            return self.DEFAULT_THRESHOLD, "global_default_missing_position_id"

        threshold = self._position_threshold_cache.get(int(position_id))

        if threshold is None:
            return self.DEFAULT_THRESHOLD, "global_default_position_not_found"

        return float(threshold), "position_setting"

    # ------------------------------------------------------------
    # 读取当前批次的相似度排序结果
    # ------------------------------------------------------------
    def load_data(self, csv_path: str = None) -> pd.DataFrame:
        csv_path = csv_path or self.DEFAULT_INPUT_CSV_PATH

        if not os.path.isfile(csv_path):
            raise FileNotFoundError(f"找不到必要输入文件：{csv_path}")

        df = pd.read_csv(csv_path, encoding="utf-8-sig")

        missing_columns = [
            column_name
            for column_name in self.REQUIRED_COLUMNS
            if column_name not in df.columns
        ]
        if missing_columns:
            raise ValueError(f"输入数据缺少必要列: {missing_columns}")

        self.input_dataframe = df
        return df

    # ------------------------------------------------------------
    # 主流程
    # ------------------------------------------------------------
    def run(
        self,
        input_csv_path: str = None,
        output_csv_path: str = None,
    ) -> pd.DataFrame:
        df = self.load_data(input_csv_path)

        # 当前批次可能全部被 anomaly 排除，因而排名 CSV 只有表头。
        # 此时仍然生成带完整表头的空结果，保证下游不会因为缺列报错。
        if df.empty:
            result_df = df.copy()
            result_df["threshold_used"] = pd.Series(dtype="float64")
            result_df["threshold_source"] = pd.Series(dtype="object")
            result_df["ml_recommendation"] = pd.Series(dtype="object")
            result_df["is_selected_for_offer"] = pd.Series(dtype="int64")
        else:
            self.load_position_thresholds(df["ml_mapped_position_id"].tolist())

            threshold_values = []
            threshold_sources = []
            recommendations = []

            for _, row in df.iterrows():
                threshold, threshold_source = self._resolve_threshold(
                    row["ml_mapped_position_id"]
                )
                match_score = row["match_score"]

                # match_score 为空说明这条根本没参与打分，
                # 不能当成「分数不够」，单独一个状态标出来。
                if pd.isna(match_score):
                    recommendation = "not_evaluated_missing_score"
                elif float(match_score) >= threshold:
                    recommendation = "recommended"
                else:
                    recommendation = "not_recommended"

                threshold_values.append(threshold)
                threshold_sources.append(threshold_source)
                recommendations.append(recommendation)

            result_df = df.copy()
            result_df["threshold_used"] = threshold_values
            result_df["threshold_source"] = threshold_sources
            result_df["ml_recommendation"] = recommendations
            result_df["is_selected_for_offer"] = (
                result_df["ml_recommendation"] == "recommended"
            ).astype("int64")

            result_df = result_df.sort_values(
                [
                    "ml_mapped_company_id",
                    "ml_mapped_position_id",
                    "match_score",
                ],
                ascending=[True, True, False],
            ).reset_index(drop=True)

        self.result_dataframe = result_df

        output_csv_path = output_csv_path or self.DEFAULT_OUTPUT_CSV_PATH
        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

        # 固定覆盖，不会无限增加历史 CSV。
        result_df.to_csv(
            output_csv_path,
            index=False,
            encoding="utf-8-sig",
        )

        group_count = (
            result_df.groupby(
                ["ml_mapped_company_id", "ml_mapped_position_id"]
            ).ngroups
            if not result_df.empty
            else 0
        )
        selected_count = (
            int(result_df["is_selected_for_offer"].sum())
            if not result_df.empty
            else 0
        )

        print(f"Groups processed: {group_count}")
        print(f"Total rows: {len(result_df)}")
        print(f"Total selected: {selected_count}")
        if not result_df.empty:
            print(
                "Threshold sources:",
                result_df["threshold_source"].value_counts().to_dict(),
            )
        print(f"Saved to: {output_csv_path}")

        return result_df


# ============================================================
# 运行本 Cell
# ============================================================

selector = OfferSelector()
result_df = selector.run()


Groups processed: 13
Total rows: 18
Total selected: 13
Saved to: /content/drive/MyDrive/recruiting-pipeline/datasets/ml/ml_results/offer_selection_result.csv


# 招聘流程实现代码（Application --> Offer）

In [ ]:
"""
Google Colab 独立 Cell：创建灵活招聘流程所需的数据库结构和模板数据。

本 Cell 只负责数据库准备，不读取 ML CSV，也不处理任何 Application。
它可以无限次重复运行：已存在且结构正确的表、索引、pipeline、stage 和
transition 会被复用，不会重复插入。
"""

import time
from datetime import datetime, timezone

import pandas as pd
import requests
from google.colab import userdata


# Cloudflare D1 连接配置。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")

# 这套 pipeline 是当前项目统一使用的灵活招聘流程模板。
PIPELINE_NAME = "Flexible recruitment pipeline v1"

# Pipeline 本身的启用状态。
PIPELINE_STATUS = "active"

# 每一个 tuple 依次表示：stage name、stage type、显示顺序、是否初始、是否终点。
STAGE_DEFINITIONS = [
    ("Application received", "application_received", 1, 1, 0),
    ("Resume screening", "resume_screen", 2, 0, 0),
    ("ML recommendation", "ml_recommendation", 3, 0, 0),
    ("Written assessment 1", "written_assessment", 4, 0, 0),
    ("Written assessment 2", "written_assessment", 5, 0, 0),
    ("HR interview", "interview", 6, 0, 0),
    ("Technical interview", "interview", 7, 0, 0),
    ("Final interview", "interview", 8, 0, 0),
    ("Offer approval", "offer_approval", 9, 0, 0),
    ("Offer process", "offer", 10, 0, 0),
    ("Hired", "hired", 11, 0, 1),
    ("Rejected", "rejected", 12, 0, 1),
    ("Withdrawn", "withdrawn", 13, 0, 1),
]


if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。请在 Colab Secrets 中添加该密钥，"
        "并允许当前 notebook 访问。"
    )


API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}"
    f"/d1/database/{DATABASE_ID}/query"
)

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


def now_iso():
    """返回统一的 UTC ISO 时间。"""

    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def execute_d1(sql, params=None, max_retries=3):
    """执行一条参数化 D1 SQL；网络临时失败时最多重试三次。"""

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL,
                headers=HEADERS,
                json={"sql": sql, "params": params or []},
                timeout=60,
            )
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = str(exc)
        else:
            if response.ok and payload.get("success"):
                return payload.get("result", [])

            last_error = payload.get("errors") or payload

        if attempt < max_retries:
            time.sleep(1.0)

    raise RuntimeError(f"D1 SQL 执行失败：{last_error}\nSQL：{sql}")


def result_rows(result):
    """从 Cloudflare D1 result wrapper 中提取 records。"""

    return result[0].get("results", []) if result else []


# 新增通用 stage 执行表。
# 一条记录代表一个 Application 对某个 stage 的一次实际执行。
execute_d1(
    """
    CREATE TABLE IF NOT EXISTS "Application_stage_run" (
      "id" INTEGER PRIMARY KEY AUTOINCREMENT,
      "application_id" INTEGER NOT NULL,
      "pipeline_stage_id" INTEGER NOT NULL,
      "attempt_no" INTEGER NOT NULL,
      "run_status" TEXT NOT NULL,
      "stage_decision" TEXT,
      "scheduled_at" TEXT,
      "started_at" TEXT,
      "completed_at" TEXT,
      "score" REAL,
      "maximum_score" REAL,
      "passed_threshold" INTEGER,
      "executed_by" INTEGER,
      "result_summary" TEXT,
      "result_metadata" TEXT,
      "created_at" TEXT NOT NULL,
      "updated_at" TEXT NOT NULL,
      FOREIGN KEY ("application_id") REFERENCES "Application"("id"),
      FOREIGN KEY ("pipeline_stage_id") REFERENCES "Pipeline_stage"("id"),
      UNIQUE ("application_id", "pipeline_stage_id", "attempt_no")
    );
    """
)

# 为后续按 Application 或 stage decision 查询执行结果建立索引。
execute_d1(
    """
    CREATE INDEX IF NOT EXISTS
      "idx_application_stage_run_application_status"
    ON "Application_stage_run" (
      "application_id",
      "run_status"
    );
    """
)

execute_d1(
    """
    CREATE INDEX IF NOT EXISTS
      "idx_application_stage_run_stage_decision"
    ON "Application_stage_run" (
      "pipeline_stage_id",
      "stage_decision"
    );
    """
)

# 防止同一个 pipeline 出现重名 stage。
execute_d1(
    """
    CREATE UNIQUE INDEX IF NOT EXISTS
      "idx_pipeline_stage_pipeline_name"
    ON "Pipeline_stage" (
      "pipeline_id",
      "pipeline_stage_name"
    );
    """
)

# 防止同一条 transition 重复插入。
execute_d1(
    """
    CREATE UNIQUE INDEX IF NOT EXISTS
      "idx_pipeline_stage_transition_unique_edge"
    ON "Pipeline_stage_transition" (
      "pipeline_id",
      "from_stage_id",
      "to_stage_id"
    );
    """
)

# 保证同一个 Offer 不会出现两个相同的 version number。
execute_d1(
    """
    CREATE UNIQUE INDEX IF NOT EXISTS
      "idx_offer_version_offer_version_no"
    ON "Offer_version" (
      "offer_id",
      "version_no"
    );
    """
)

# 加速按时间查看 Application 的完整 stage history。
execute_d1(
    """
    CREATE INDEX IF NOT EXISTS
      "idx_application_stage_history_application_entered"
    ON "Application_stage_history" (
      "application_id",
      "entered_at"
    );
    """
)

# 加速按时间读取 Offer 状态历史。
execute_d1(
    """
    CREATE INDEX IF NOT EXISTS
      "idx_offer_status_history_offer_changed"
    ON "Offer_status_history" (
      "offer_id",
      "changed_at"
    );
    """
)

# 查找是否已经存在同名 pipeline。
pipeline_rows = result_rows(
    execute_d1(
        """
        SELECT "id", "status"
        FROM "Hiring_pipeline"
        WHERE "hiring_pipeline_name" = ?
        ORDER BY "id";
        """,
        [PIPELINE_NAME],
    )
)

# 多条同名 pipeline 代表数据库状态不明确，不能静默猜测。
if len(pipeline_rows) > 1:
    raise RuntimeError(
        f"发现多条同名 Hiring_pipeline：{PIPELINE_NAME}。请先检查重复数据。"
    )

# 不存在时创建；存在时直接复用。
if not pipeline_rows:
    timestamp = now_iso()
    pipeline_rows = result_rows(
        execute_d1(
            """
            INSERT INTO "Hiring_pipeline" (
              "hiring_pipeline_name",
              "status",
              "created_at",
              "updated_at"
            ) VALUES (?, ?, ?, ?)
            RETURNING "id", "status";
            """,
            [PIPELINE_NAME, PIPELINE_STATUS, timestamp, timestamp],
        )
    )

pipeline_id = int(pipeline_rows[0]["id"])

# 保存 stage name 到 D1 stage id 的稳定映射。
stage_id_by_name = {}

# 逐个创建或验证 stage。
for stage_name, stage_type, sequence_no, is_initial, is_terminal in STAGE_DEFINITIONS:
    rows = result_rows(
        execute_d1(
            """
            SELECT
              "id",
              "stage_type",
              "sequence_no",
              "is_initial",
              "is_terminal"
            FROM "Pipeline_stage"
            WHERE "pipeline_id" = ?
              AND "pipeline_stage_name" = ?;
            """,
            [pipeline_id, stage_name],
        )
    )

    if not rows:
        timestamp = now_iso()
        rows = result_rows(
            execute_d1(
                """
                INSERT INTO "Pipeline_stage" (
                  "pipeline_id",
                  "pipeline_stage_name",
                  "stage_type",
                  "sequence_no",
                  "is_initial",
                  "is_terminal",
                  "created_at",
                  "updated_at"
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                RETURNING "id";
                """,
                [
                    pipeline_id,
                    stage_name,
                    stage_type,
                    sequence_no,
                    is_initial,
                    is_terminal,
                    timestamp,
                    timestamp,
                ],
            )
        )
    else:
        # 已存在时验证定义没有被其他代码改成不兼容值。
        existing = rows[0]
        expected = (
            stage_type,
            int(sequence_no),
            int(is_initial),
            int(is_terminal),
        )
        actual = (
            existing["stage_type"],
            int(existing["sequence_no"]),
            int(existing["is_initial"]),
            int(existing["is_terminal"]),
        )

        if actual != expected:
            raise RuntimeError(
                f"Stage {stage_name!r} 已存在，但定义不一致。"
                f" expected={expected}, actual={actual}"
            )

    stage_id_by_name[stage_name] = int(rows[0]["id"])


# 用集合保存允许的有向边，天然消除重复边。
allowed_edges = set()


def allow(from_name, *to_names):
    """把一组允许的 stage transition 加入模板。"""

    for to_name in to_names:
        if from_name != to_name:
            allowed_edges.add((from_name, to_name))


# Application received 可以直接进入任意主要流程，也可以直接 Offer/Rejected。
allow(
    "Application received",
    "Resume screening",
    "ML recommendation",
    "Written assessment 1",
    "Written assessment 2",
    "HR interview",
    "Technical interview",
    "Final interview",
    "Offer approval",
    "Offer process",
    "Rejected",
    "Withdrawn",
)

# Resume screening 后可以继续、跳过中间 stage、直接 Offer 或结束。
allow(
    "Resume screening",
    "ML recommendation",
    "Written assessment 1",
    "Written assessment 2",
    "HR interview",
    "Technical interview",
    "Final interview",
    "Offer approval",
    "Offer process",
    "Rejected",
    "Withdrawn",
)

# ML recommendation 可以作为参考，也可以直接决定 Offer 或 No Offer。
allow(
    "ML recommendation",
    "Written assessment 1",
    "Written assessment 2",
    "HR interview",
    "Technical interview",
    "Final interview",
    "Offer approval",
    "Offer process",
    "Rejected",
    "Withdrawn",
)

# 两个笔试 stage 可以互相流转，也能进入面试、Offer 或结束。
for written_stage in ("Written assessment 1", "Written assessment 2"):
    allow(
        written_stage,
        "Written assessment 1",
        "Written assessment 2",
        "HR interview",
        "Technical interview",
        "Final interview",
        "Offer approval",
        "Offer process",
        "Rejected",
        "Withdrawn",
    )

# 面试后可以回到 screening/ML/笔试，进入其他面试，或直接 Offer/结束。
for interview_stage in ("HR interview", "Technical interview", "Final interview"):
    allow(
        interview_stage,
        "Resume screening",
        "ML recommendation",
        "Written assessment 1",
        "Written assessment 2",
        "HR interview",
        "Technical interview",
        "Final interview",
        "Offer approval",
        "Offer process",
        "Rejected",
        "Withdrawn",
    )

# Offer approval 可以批准、退回补充面试/测试，或者结束申请。
allow(
    "Offer approval",
    "Written assessment 1",
    "Written assessment 2",
    "HR interview",
    "Technical interview",
    "Final interview",
    "Offer process",
    "Rejected",
    "Withdrawn",
)

# Offer 发出后仍可能接受、拒绝或撤回，因此 Offer 本身不是 terminal stage。
allow("Offer process", "Hired", "Rejected", "Withdrawn")

# 逐条创建或重新启用 transition。
for from_name, to_name in sorted(allowed_edges):
    from_stage_id = stage_id_by_name[from_name]
    to_stage_id = stage_id_by_name[to_name]

    rows = result_rows(
        execute_d1(
            """
            SELECT "id", "is_allowed"
            FROM "Pipeline_stage_transition"
            WHERE "pipeline_id" = ?
              AND "from_stage_id" = ?
              AND "to_stage_id" = ?;
            """,
            [pipeline_id, from_stage_id, to_stage_id],
        )
    )

    if not rows:
        execute_d1(
            """
            INSERT INTO "Pipeline_stage_transition" (
              "pipeline_id",
              "from_stage_id",
              "to_stage_id",
              "is_allowed",
              "created_at"
            ) VALUES (?, ?, ?, 1, ?);
            """,
            [pipeline_id, from_stage_id, to_stage_id, now_iso()],
        )
    elif int(rows[0]["is_allowed"]) != 1:
        execute_d1(
            """
            UPDATE "Pipeline_stage_transition"
            SET "is_allowed" = 1
            WHERE "id" = ?;
            """,
            [int(rows[0]["id"])],
        )


# 检查迁移后是否产生外键违规。
foreign_key_violations = result_rows(execute_d1("PRAGMA foreign_key_check;"))

if foreign_key_violations:
    raise RuntimeError(
        f"迁移后发现外键违规：{foreign_key_violations[:10]}"
    )


# 生成便于人工检查的 stage preview。
stage_preview = pd.DataFrame(
    [
        {
            "pipeline_id": pipeline_id,
            "pipeline_name": PIPELINE_NAME,
            "pipeline_stage_id": stage_id_by_name[stage_name],
            "stage_name": stage_name,
            "stage_type": stage_type,
            "sequence_no": sequence_no,
            "is_initial": is_initial,
            "is_terminal": is_terminal,
        }
        for stage_name, stage_type, sequence_no, is_initial, is_terminal
        in STAGE_DEFINITIONS
    ]
)

print("Flexible recruitment pipeline setup completed successfully.")
print("Pipeline id:", pipeline_id)
print("Stage count:", len(stage_preview))
print("Allowed transition count:", len(allowed_edges))
print("Foreign key violation count:", len(foreign_key_violations))

display(stage_preview)


Flexible recruitment pipeline setup completed successfully.
Pipeline id: 2
Stage count: 13
Allowed transition count: 87
Foreign key violation count: 0


    pipeline_id                     pipeline_name  pipeline_stage_id  \
0             2  Flexible recruitment pipeline v1                  2   
1             2  Flexible recruitment pipeline v1                  3   
2             2  Flexible recruitment pipeline v1                  4   
3             2  Flexible recruitment pipeline v1                  5   
4             2  Flexible recruitment pipeline v1                  6   
5             2  Flexible recruitment pipeline v1                  7   
6             2  Flexible recruitment pipeline v1                  8   
7             2  Flexible recruitment pipeline v1                  9   
8             2  Flexible recruitment pipeline v1                 10   
9             2  Flexible recruitment pipeline v1                 11   
10            2  Flexible recruitment pipeline v1                 12   
11            2  Flexible recruitment pipeline v1                 13   
12            2  Flexible recruitment pipeline v1               

# ML结果数据导入
可以重复运行，但是每一次运行需要比较长的时间，谨慎考虑

In [ ]:
"""
Google Colab 独立 Cell：把已经完成的 ML v1 结果导入招聘流程和 Offer 层。

本 Cell 不重新运行任何 ML。它只读取：
1. anomaly_detection_result.csv（覆盖当前批次 Application）；
2. offer_selection_result.csv（覆盖当前批次中未被 anomaly 排除的记录）。

最终规则：
- anomaly excluded -> no_offer -> Rejected；
- clean + is_selected_for_offer=1 -> offer -> 创建或复用 Draft Offer；
- clean + is_selected_for_offer=0 -> no_offer -> Rejected。

本 Cell 不依赖其他 Colab Cell 中的 class、function 或 variable；它会自行连接
D1，并验证持久化在数据库中的 schema 和 Flexible pipeline。第一次使用前需要先
执行 26_flexible_recruitment_pipeline_setup.py 完成一次数据库迁移。迁移完成后，
本 Cell 可以在任意新 Colab session 中单独运行。它是可恢复且幂等的：中途断线后
可以直接重新运行；已写入的 stage run、history 和 Offer 会被验证并复用。
"""

import json
import math
import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from google.colab import drive, userdata


# 挂载 Google Drive，用于读取既有 ML CSV 和覆盖输出导入结果 CSV。
drive.mount("/content/drive")

# Cloudflare D1 连接配置。
ACCOUNT_ID = userdata.get('CLOUDFLARE_ACCOUNT_ID')
DATABASE_ID = userdata.get('D1_DATABASE_ID')
API_TOKEN = userdata.get("CLOUDFLARE_KEY")

# ML 文件所在的固定目录。
ML_RESULT_FOLDER = (
    "/content/drive/MyDrive/recruiting-pipeline"
    "/datasets/ml/ml_results"
)

# 当前批次异常检测结果；Application.id 只要求是 D1 全库的子集。
ANOMALY_CSV_PATH = os.path.join(
    ML_RESULT_FOLDER,
    "anomaly_detection_result.csv",
)

# 未被 anomaly 排除的记录及最终前 10% 选择结果。
OFFER_SELECTION_CSV_PATH = os.path.join(
    ML_RESULT_FOLDER,
    "offer_selection_result.csv",
)

# 固定覆盖写入数据库导入结果，不会无限生成新文件。
IMPORT_RESULT_CSV_PATH = os.path.join(
    ML_RESULT_FOLDER,
    "ml_recommendation_v1_database_import_result.csv",
)

# 固定覆盖写入失败明细；没有失败时仍输出只有表头的空 CSV。
IMPORT_FAILED_CSV_PATH = os.path.join(
    ML_RESULT_FOLDER,
    "ml_recommendation_v1_database_import_failed.csv",
)

# 当前冻结的 pipeline 和 ML rule version。
PIPELINE_NAME = "Flexible recruitment pipeline v1"
ML_RULE_VERSION = "ml_recommendation_v1"

# 当前导入所需的四个 stage。
REQUIRED_STAGE_NAMES = [
    "Application received",
    "ML recommendation",
    "Offer process",
    "Rejected",
]


if not API_TOKEN:
    raise ValueError(
        "未找到 CLOUDFLARE_KEY。请在 Colab Secrets 中添加该密钥，"
        "并允许当前 notebook 访问。"
    )


for required_file_path in (ANOMALY_CSV_PATH, OFFER_SELECTION_CSV_PATH):
    if not os.path.isfile(required_file_path):
        raise FileNotFoundError(f"找不到必要输入文件：{required_file_path}")


API_URL = (
    f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}"
    f"/d1/database/{DATABASE_ID}/query"
)

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}


def now_iso():
    """返回统一的 UTC ISO 时间。"""

    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def python_value(value):
    """把 pandas/NumPy scalar 转成 requests 能序列化的 Python value。"""

    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        return value.item()

    return value


def execute_d1(sql, params=None, max_retries=3):
    """执行一条参数化 D1 SQL，并对临时网络错误进行有限重试。"""

    safe_params = [python_value(value) for value in (params or [])]
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL,
                headers=HEADERS,
                json={"sql": sql, "params": safe_params},
                timeout=60,
            )
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = str(exc)
        else:
            if response.ok and payload.get("success"):
                return payload.get("result", [])

            last_error = payload.get("errors") or payload

        if attempt < max_retries:
            time.sleep(1.0)

    raise RuntimeError(f"D1 SQL 执行失败：{last_error}\nSQL：{sql}")


def result_rows(result):
    """从 Cloudflare result wrapper 中提取 records。"""

    return result[0].get("results", []) if result else []


def parse_boolean(value, column_name):
    """严格解析 CSV 中的 0/1、true/false，拒绝模糊值。"""

    if pd.isna(value):
        raise ValueError(f"{column_name} 不允许为空。")

    if isinstance(value, bool):
        return int(value)

    normalized = str(value).strip().lower()

    if normalized in {"1", "1.0", "true"}:
        return 1

    if normalized in {"0", "0.0", "false"}:
        return 0

    raise ValueError(f"{column_name} 出现无法识别的 boolean：{value!r}")


def stable_json(value):
    """以稳定 key 顺序生成 UTF-8 JSON，保证重复运行结果可比较。"""

    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def ensure_database_objects():
    """独立创建/验证本次导入必须依赖的通用表和 pipeline。"""

    execute_d1(
        """
        CREATE TABLE IF NOT EXISTS "Application_stage_run" (
          "id" INTEGER PRIMARY KEY AUTOINCREMENT,
          "application_id" INTEGER NOT NULL,
          "pipeline_stage_id" INTEGER NOT NULL,
          "attempt_no" INTEGER NOT NULL,
          "run_status" TEXT NOT NULL,
          "stage_decision" TEXT,
          "scheduled_at" TEXT,
          "started_at" TEXT,
          "completed_at" TEXT,
          "score" REAL,
          "maximum_score" REAL,
          "passed_threshold" INTEGER,
          "executed_by" INTEGER,
          "result_summary" TEXT,
          "result_metadata" TEXT,
          "created_at" TEXT NOT NULL,
          "updated_at" TEXT NOT NULL,
          FOREIGN KEY ("application_id") REFERENCES "Application"("id"),
          FOREIGN KEY ("pipeline_stage_id") REFERENCES "Pipeline_stage"("id"),
          UNIQUE ("application_id", "pipeline_stage_id", "attempt_no")
        );
        """
    )

    execute_d1(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS
          "idx_pipeline_stage_transition_unique_edge"
        ON "Pipeline_stage_transition" (
          "pipeline_id", "from_stage_id", "to_stage_id"
        );
        """
    )

    pipeline_rows = result_rows(
        execute_d1(
            """
            SELECT "id"
            FROM "Hiring_pipeline"
            WHERE "hiring_pipeline_name" = ?
            ORDER BY "id";
            """,
            [PIPELINE_NAME],
        )
    )

    if len(pipeline_rows) != 1:
        raise RuntimeError(
            f"必须存在且只能存在一条 {PIPELINE_NAME!r}。"
            "请先运行 pipeline setup，或检查重复 pipeline。"
        )

    pipeline_id = int(pipeline_rows[0]["id"])

    stage_rows = result_rows(
        execute_d1(
            """
            SELECT "id", "pipeline_stage_name", "stage_type"
            FROM "Pipeline_stage"
            WHERE "pipeline_id" = ?;
            """,
            [pipeline_id],
        )
    )

    stage_by_name = {
        row["pipeline_stage_name"]: row
        for row in stage_rows
    }

    missing_stages = [
        name for name in REQUIRED_STAGE_NAMES
        if name not in stage_by_name
    ]

    if missing_stages:
        raise RuntimeError(
            f"Flexible pipeline 缺少必要 stages：{missing_stages}"
        )

    stage_id_by_name = {
        name: int(stage_by_name[name]["id"])
        for name in REQUIRED_STAGE_NAMES
    }

    # 验证当前导入实际会使用的两条决策 transition 已配置并允许。
    for to_stage_name in ("Offer process", "Rejected"):
        transition_rows = result_rows(
            execute_d1(
                """
                SELECT "id", "is_allowed"
                FROM "Pipeline_stage_transition"
                WHERE "pipeline_id" = ?
                  AND "from_stage_id" = ?
                  AND "to_stage_id" = ?;
                """,
                [
                    pipeline_id,
                    stage_id_by_name["ML recommendation"],
                    stage_id_by_name[to_stage_name],
                ],
            )
        )

        if (
            len(transition_rows) != 1
            or int(transition_rows[0]["is_allowed"]) != 1
        ):
            raise RuntimeError(
                "缺少允许的 transition："
                f"ML recommendation -> {to_stage_name}"
            )

    return pipeline_id, stage_id_by_name


def ensure_history(
    application_id,
    from_stage_id,
    to_stage_id,
    reason_code,
    entered_at,
    note,
):
    """按 application + reason code 创建或复用一条本次 ML 专属 history。"""

    rows = result_rows(
        execute_d1(
            """
            SELECT "id", "from_stage_id", "to_stage_id"
            FROM "Application_stage_history"
            WHERE "application_id" = ?
              AND "reason_code" = ?
            ORDER BY "id";
            """,
            [application_id, reason_code],
        )
    )

    if len(rows) > 1:
        raise RuntimeError(
            f"Application {application_id} 的 history {reason_code!r} 重复。"
        )

    if rows:
        existing_from = rows[0]["from_stage_id"]
        existing_from = int(existing_from) if existing_from is not None else None
        existing_to = int(rows[0]["to_stage_id"])

        if existing_from != from_stage_id or existing_to != to_stage_id:
            raise RuntimeError(
                f"Application {application_id} 的既有 history 与预期不一致。"
            )

        return int(rows[0]["id"]), "reused"

    inserted = result_rows(
        execute_d1(
            """
            INSERT INTO "Application_stage_history" (
              "application_id",
              "from_stage_id",
              "to_stage_id",
              "changed_by",
              "entered_at",
              "exited_at",
              "reason_code",
              "note"
            ) VALUES (?, ?, ?, NULL, ?, NULL, ?, ?)
            RETURNING "id";
            """,
            [
                application_id,
                from_stage_id,
                to_stage_id,
                entered_at,
                reason_code,
                note,
            ],
        )
    )

    return int(inserted[0]["id"]), "inserted"


# -----------------------------------------------------------------------------
# 1. 读取并严格验证两个 ML CSV。
# -----------------------------------------------------------------------------
anomaly_df = pd.read_csv(ANOMALY_CSV_PATH, encoding="utf-8-sig")
selection_df = pd.read_csv(OFFER_SELECTION_CSV_PATH, encoding="utf-8-sig")

required_anomaly_columns = {
    "application_id",
    "candidate_snapshot_id",
    "ml_mapped_company_id",
    "ml_mapped_position_id",
    "anomaly_flags_json",
    "has_any_anomaly",
}

required_selection_columns = {
    "application_id",
    "candidate_snapshot_id",
    "ml_mapped_company_id",
    "ml_mapped_position_id",
    "match_score",
    "is_selected_for_offer",
    "group_size",
    "selected_count_in_group",
    "rank_within_group",
}

missing_anomaly_columns = sorted(
    required_anomaly_columns - set(anomaly_df.columns)
)

missing_selection_columns = sorted(
    required_selection_columns - set(selection_df.columns)
)

if missing_anomaly_columns:
    raise ValueError(
        f"anomaly_detection_result.csv 缺少 columns：{missing_anomaly_columns}"
    )

if missing_selection_columns:
    raise ValueError(
        f"offer_selection_result.csv 缺少 columns：{missing_selection_columns}"
    )

for dataframe_name, dataframe in (
    ("anomaly", anomaly_df),
    ("selection", selection_df),
):
    if dataframe["application_id"].isna().any():
        raise ValueError(f"{dataframe_name} CSV 存在空 application_id。")

    if dataframe["application_id"].duplicated().any():
        duplicated_ids = dataframe.loc[
            dataframe["application_id"].duplicated(keep=False),
            "application_id",
        ].tolist()
        raise ValueError(
            f"{dataframe_name} CSV 存在重复 application_id：{duplicated_ids[:20]}"
        )

    dataframe["application_id"] = dataframe["application_id"].astype("int64")


# -----------------------------------------------------------------------------
# 2. 确认 schema、pipeline 和 stages 已准备完成。
# -----------------------------------------------------------------------------
pipeline_id, stage_id_by_name = ensure_database_objects()


# -----------------------------------------------------------------------------
# 3. 一次性读取 D1 Application，避免逐条重复读取同一张表。
# -----------------------------------------------------------------------------
application_rows = result_rows(
    execute_d1(
        """
        SELECT
          "id",
          "company_id",
          "position_id",
          "hiring_pipeline_id",
          "current_stage_id",
          "application_status",
          "applied_at",
          "created_at",
          "updated_at"
        FROM "Application"
        ORDER BY "id";
        """
    )
)

application_by_id = {
    int(row["id"]): row
    for row in application_rows
}

csv_application_ids = set(anomaly_df["application_id"].astype(int))
d1_application_ids = set(application_by_id)

# 当前批次 CSV 中的每个 Application 必须存在于 D1；D1 可以同时保留
# 任意数量的历史 Application。
application_ids_missing_from_d1 = sorted(
    csv_application_ids - d1_application_ids
)

if application_ids_missing_from_d1:
    raise ValueError(
        "anomaly CSV 中存在 D1 不包含的 Application.id："
        f"{application_ids_missing_from_d1[:20]}"
    )

# 只保留当前批次 D1 mapping，后续不会循环或修改历史 Application。
application_by_id = {
    application_id: application_by_id[application_id]
    for application_id in sorted(csv_application_ids)
}

# Selection 不允许包含当前 anomaly CSV 之外的历史或其他批次 Application。
selection_application_ids = set(
    selection_df["application_id"].astype(int)
)
unexpected_selection_ids = sorted(
    selection_application_ids - csv_application_ids
)
if unexpected_selection_ids:
    raise ValueError(
        "offer selection CSV 包含当前 anomaly 批次之外的 Application.id："
        f"{unexpected_selection_ids[:20]}"
    )

print(
    "Incremental ML import mapping validation:",
    {
        "current_application_count": len(csv_application_ids),
        "current_selection_count": len(selection_application_ids),
        "missing_from_d1_count": len(application_ids_missing_from_d1),
    },
)


selection_by_application_id = {
    int(row["application_id"]): row
    for _, row in selection_df.iterrows()
}


# -----------------------------------------------------------------------------
# 4. 只为当前批次 Application 构造唯一、确定的 ML 最终决策。
# -----------------------------------------------------------------------------
decision_records = []

for _, anomaly_row in anomaly_df.sort_values("application_id").iterrows():
    application_id = int(anomaly_row["application_id"])
    application = application_by_id[application_id]
    has_any_anomaly = parse_boolean(
        anomaly_row["has_any_anomaly"],
        "has_any_anomaly",
    )

    # 验证 CSV 中的 ML company/position 与正式 Application 外键完全一致。
    if int(anomaly_row["ml_mapped_company_id"]) != int(application["company_id"]):
        raise ValueError(
            f"Application {application_id} 的 company_id 与 anomaly CSV 不一致。"
        )

    if int(anomaly_row["ml_mapped_position_id"]) != int(application["position_id"]):
        raise ValueError(
            f"Application {application_id} 的 position_id 与 anomaly CSV 不一致。"
        )

    selection_row = selection_by_application_id.get(application_id)

    if has_any_anomaly == 1:
        # 已决定：anomaly excluded 直接 no_offer，不进入人工审查。
        if selection_row is not None:
            raise ValueError(
                f"Application {application_id} 有 anomaly，"
                "但同时出现在 offer selection CSV。"
            )

        decision = "no_offer"
        decision_reason = "excluded_by_anomaly_detection_v1"
        terminal_stage_name = "Rejected"
        match_score = None
        selected = 0
        group_size = None
        selected_count = None
        rank_within_group = None
    else:
        # clean record 必须在 selection CSV 中有且只有一条结果。
        if selection_row is None:
            raise ValueError(
                f"Application {application_id} 无 anomaly，"
                "但不在 offer selection CSV。"
            )

        if int(selection_row["ml_mapped_company_id"]) != int(application["company_id"]):
            raise ValueError(
                f"Application {application_id} 的 selection company_id 不一致。"
            )

        if int(selection_row["ml_mapped_position_id"]) != int(application["position_id"]):
            raise ValueError(
                f"Application {application_id} 的 selection position_id 不一致。"
            )

        selected = parse_boolean(
            selection_row["is_selected_for_offer"],
            "is_selected_for_offer",
        )

        decision = "offer" if selected == 1 else "no_offer"
        decision_reason = (
            "selected_by_ml_recommendation_v1"
            if selected == 1
            else "not_selected_by_ml_recommendation_v1"
        )
        terminal_stage_name = "Offer process" if selected == 1 else "Rejected"
        match_score = float(selection_row["match_score"])
        group_size = int(selection_row["group_size"])
        selected_count = int(selection_row["selected_count_in_group"])
        rank_within_group = int(selection_row["rank_within_group"])

    metadata = {
        "rule_version": ML_RULE_VERSION,
        "model_name": "all-MiniLM-L6-v2",
        "decision": decision,
        "decision_reason": decision_reason,
        "has_any_anomaly": has_any_anomaly,
        "anomaly_flags_json": python_value(anomaly_row["anomaly_flags_json"]),
        "match_score": match_score,
        "is_selected_for_offer": selected,
        "group_size": group_size,
        "selected_count_in_group": selected_count,
        "rank_within_group": rank_within_group,
        "selection_ratio": 0.10,
        "round_up_selection": True,
    }

    decision_records.append(
        {
            "application_id": application_id,
            "candidate_snapshot_id": int(anomaly_row["candidate_snapshot_id"]),
            "ml_mapped_company_id": int(application["company_id"]),
            "ml_mapped_position_id": int(application["position_id"]),
            "has_any_anomaly": has_any_anomaly,
            "match_score": match_score,
            "is_selected_for_offer": selected,
            "ml_decision": decision,
            "decision_reason": decision_reason,
            "terminal_stage_name": terminal_stage_name,
            "result_metadata": stable_json(metadata),
        }
    )


decision_df = pd.DataFrame(decision_records)

# 决策总数必须覆盖当前 anomaly CSV 的全部 Application。
if len(decision_df) != len(csv_application_ids):
    raise RuntimeError("ML decision record count 与当前批次 Application count 不一致。")

print(
    "ML decision plan summary:",
    {
        "application_count": len(decision_df),
        "offer_count": int(decision_df["ml_decision"].eq("offer").sum()),
        "no_offer_count": int(decision_df["ml_decision"].eq("no_offer").sum()),
        "anomaly_no_offer_count": int(
            decision_df["decision_reason"]
            .eq("excluded_by_anomaly_detection_v1")
            .sum()
        ),
    },
)


# -----------------------------------------------------------------------------
# 5. 逐条幂等写入 stage run、history、Application terminal state 和 Offer。
# -----------------------------------------------------------------------------
result_records = []
failed_records = []

for row_number, decision_row in enumerate(decision_records, start=1):
    application_id = int(decision_row["application_id"])
    application = application_by_id[application_id]
    terminal_stage_name = decision_row["terminal_stage_name"]
    terminal_stage_id = stage_id_by_name[terminal_stage_name]
    timestamp = now_iso()

    try:
        # 如果已有非本版本创建的历史，不能静默更换 pipeline。
        foreign_history_count = int(
            result_rows(
                execute_d1(
                    """
                    SELECT COUNT(*) AS "record_count"
                    FROM "Application_stage_history"
                    WHERE "application_id" = ?
                      AND (
                        "reason_code" IS NULL
                        OR "reason_code" NOT LIKE 'ml_recommendation_v1_%'
                      );
                    """,
                    [application_id],
                )
            )[0]["record_count"]
        )

        if (
            int(application["hiring_pipeline_id"]) != pipeline_id
            and foreign_history_count > 0
        ):
            raise RuntimeError(
                "Application 已有其他 pipeline history，不能自动改用 Flexible pipeline。"
            )

        # 创建或验证唯一的 ML recommendation attempt 1。
        stage_run_rows = result_rows(
            execute_d1(
                """
                SELECT
                  "id",
                  "run_status",
                  "stage_decision",
                  "score",
                  "result_metadata"
                FROM "Application_stage_run"
                WHERE "application_id" = ?
                  AND "pipeline_stage_id" = ?
                  AND "attempt_no" = 1;
                """,
                [application_id, stage_id_by_name["ML recommendation"]],
            )
        )

        if stage_run_rows:
            existing_run = stage_run_rows[0]

            if (
                existing_run["run_status"] != "completed"
                or existing_run["stage_decision"] != decision_row["ml_decision"]
                or existing_run["result_metadata"] != decision_row["result_metadata"]
            ):
                raise RuntimeError(
                    "既有 ML stage run 与当前冻结的 v1 结果不一致。"
                )

            stage_run_id = int(existing_run["id"])
            stage_run_action = "reused"
        else:
            inserted_run = result_rows(
                execute_d1(
                    """
                    INSERT INTO "Application_stage_run" (
                      "application_id",
                      "pipeline_stage_id",
                      "attempt_no",
                      "run_status",
                      "stage_decision",
                      "scheduled_at",
                      "started_at",
                      "completed_at",
                      "score",
                      "maximum_score",
                      "passed_threshold",
                      "executed_by",
                      "result_summary",
                      "result_metadata",
                      "created_at",
                      "updated_at"
                    ) VALUES (?, ?, 1, 'completed', ?, NULL, ?, ?, ?, NULL, ?, NULL, ?, ?, ?, ?)
                    RETURNING "id";
                    """,
                    [
                        application_id,
                        stage_id_by_name["ML recommendation"],
                        decision_row["ml_decision"],
                        timestamp,
                        timestamp,
                        decision_row["match_score"],
                        decision_row["is_selected_for_offer"],
                        decision_row["decision_reason"],
                        decision_row["result_metadata"],
                        timestamp,
                        timestamp,
                    ],
                )
            )
            stage_run_id = int(inserted_run[0]["id"])
            stage_run_action = "inserted"

        # 第一次运行时补全 Application received 初始历史。
        initial_entered_at = (
            application.get("applied_at")
            or application.get("created_at")
            or timestamp
        )

        initial_history_id, initial_history_action = ensure_history(
            application_id=application_id,
            from_stage_id=None,
            to_stage_id=stage_id_by_name["Application received"],
            reason_code="ml_recommendation_v1_initialize",
            entered_at=initial_entered_at,
            note="Initialized for Flexible recruitment pipeline v1.",
        )

        # 记录从 Application received 进入 ML recommendation。
        ml_history_id, ml_history_action = ensure_history(
            application_id=application_id,
            from_stage_id=stage_id_by_name["Application received"],
            to_stage_id=stage_id_by_name["ML recommendation"],
            reason_code="ml_recommendation_v1_enter",
            entered_at=timestamp,
            note="Entered ML recommendation using frozen v1 results.",
        )

        # 记录 ML 最终流向 Offer 或 Rejected。
        terminal_reason_code = (
            "ml_recommendation_v1_offer"
            if decision_row["ml_decision"] == "offer"
            else "ml_recommendation_v1_no_offer"
        )

        terminal_history_id, terminal_history_action = ensure_history(
            application_id=application_id,
            from_stage_id=stage_id_by_name["ML recommendation"],
            to_stage_id=terminal_stage_id,
            reason_code=terminal_reason_code,
            entered_at=timestamp,
            note=decision_row["decision_reason"],
        )

        target_application_status = (
            "offer_created"
            if decision_row["ml_decision"] == "offer"
            else "rejected"
        )

        current_pipeline_id = int(application["hiring_pipeline_id"])
        current_stage_id = application.get("current_stage_id")
        current_stage_id = int(current_stage_id) if current_stage_id is not None else None

        if (
            current_pipeline_id != pipeline_id
            or current_stage_id != terminal_stage_id
            or application["application_status"] != target_application_status
        ):
            execute_d1(
                """
                UPDATE "Application"
                SET
                  "hiring_pipeline_id" = ?,
                  "current_stage_id" = ?,
                  "current_stage_entered_at" = ?,
                  "application_status" = ?,
                  "updated_at" = ?
                WHERE "id" = ?;
                """,
                [
                    pipeline_id,
                    terminal_stage_id,
                    timestamp,
                    target_application_status,
                    timestamp,
                    application_id,
                ],
            )
            application_action = "updated"
        else:
            application_action = "reused"

        offer_id = None
        offer_action = "not_applicable"

        if decision_row["ml_decision"] == "offer":
            offer_rows = result_rows(
                execute_d1(
                    """
                    SELECT "id"
                    FROM "Offer"
                    WHERE "application_id" = ?;
                    """,
                    [application_id],
                )
            )

            if offer_rows:
                offer_id = int(offer_rows[0]["id"])
                offer_action = "reused"
            else:
                inserted_offer = result_rows(
                    execute_d1(
                        """
                        INSERT INTO "Offer" (
                          "application_id",
                          "created_at",
                          "updated_at"
                        ) VALUES (?, ?, ?)
                        RETURNING "id";
                        """,
                        [application_id, timestamp, timestamp],
                    )
                )
                offer_id = int(inserted_offer[0]["id"])
                offer_action = "inserted"

            # 创建初始 draft 状态，但不伪造 Offer_version 的金额和 start date。
            draft_status_rows = result_rows(
                execute_d1(
                    """
                    SELECT "id"
                    FROM "Offer_status_history"
                    WHERE "offer_id" = ?
                      AND "status" = 'draft'
                    ORDER BY "id";
                    """,
                    [offer_id],
                )
            )

            if not draft_status_rows:
                execute_d1(
                    """
                    INSERT INTO "Offer_status_history" (
                      "offer_id",
                      "status",
                      "changed_at",
                      "changed_by",
                      "note"
                    ) VALUES (?, 'draft', ?, NULL, ?);
                    """,
                    [
                        offer_id,
                        timestamp,
                        "Created by ML recommendation v1 final decision.",
                    ],
                )
        else:
            # No-offer Application 不允许已经存在 Offer，否则业务状态自相矛盾。
            existing_offer_rows = result_rows(
                execute_d1(
                    """
                    SELECT "id"
                    FROM "Offer"
                    WHERE "application_id" = ?;
                    """,
                    [application_id],
                )
            )

            if existing_offer_rows:
                raise RuntimeError(
                    "当前 ML 决策为 no_offer，但数据库已经存在 Offer。"
                )

        result_records.append(
            {
                **decision_row,
                "pipeline_id": pipeline_id,
                "ml_stage_run_id": stage_run_id,
                "initial_history_id": initial_history_id,
                "ml_history_id": ml_history_id,
                "terminal_history_id": terminal_history_id,
                "offer_id": offer_id,
                "stage_run_action": stage_run_action,
                "initial_history_action": initial_history_action,
                "ml_history_action": ml_history_action,
                "terminal_history_action": terminal_history_action,
                "application_action": application_action,
                "offer_action": offer_action,
                "import_status": "success",
                "import_error": None,
            }
        )
    except Exception as exc:
        failed_record = {
            **decision_row,
            "import_status": "failed",
            "import_error": str(exc),
        }
        result_records.append(failed_record)
        failed_records.append(failed_record)

    if row_number % 100 == 0 or row_number == len(decision_records):
        print(
            "Processed ML recommendation records:",
            f"{row_number}/{len(decision_records)}",
        )


# -----------------------------------------------------------------------------
# 6. 固定覆盖保存导入结果和失败明细。
# -----------------------------------------------------------------------------
result_df = pd.DataFrame(result_records).sort_values("application_id")
failed_df = pd.DataFrame(failed_records)

result_df.to_csv(
    IMPORT_RESULT_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

if failed_df.empty:
    failed_df = pd.DataFrame(
        columns=list(result_df.columns)
    )

failed_df.to_csv(
    IMPORT_FAILED_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)


# -----------------------------------------------------------------------------
# 7. 只验证当前批次数据库结果，不重新计算或要求历史批次数量。
# -----------------------------------------------------------------------------

# 为参数化 IN 查询生成占位符。当前 anomaly CSV 已在前面保证非空且唯一。
current_application_id_list = sorted(csv_application_ids)
current_application_placeholders = ",".join(
    ["?"] * len(current_application_id_list)
)

verification = result_rows(
    execute_d1(
        f"""
        SELECT
          COUNT(*) AS "application_count",
          SUM(CASE WHEN "application_status" = 'offer_created' THEN 1 ELSE 0 END)
            AS "offer_application_count",
          SUM(CASE WHEN "application_status" = 'rejected' THEN 1 ELSE 0 END)
            AS "rejected_application_count"
        FROM "Application"
        WHERE "hiring_pipeline_id" = ?
          AND "id" IN ({current_application_placeholders});
        """,
        [pipeline_id, *current_application_id_list],
    )
)[0]

offer_count = int(
    result_rows(
        execute_d1(
            f"""
            SELECT COUNT(*) AS "record_count"
            FROM "Offer" AS o
            INNER JOIN "Application" AS a
              ON a."id" = o."application_id"
            WHERE a."hiring_pipeline_id" = ?
              AND a."id" IN ({current_application_placeholders});
            """,
            [pipeline_id, *current_application_id_list],
        )
    )[0]["record_count"]
)

foreign_key_violations = result_rows(execute_d1("PRAGMA foreign_key_check;"))

successful_count = int(result_df["import_status"].eq("success").sum())
failed_count = int(result_df["import_status"].eq("failed").sum())

# 当前批次动态预期数量；不再写死旧批次 101/938/1,039。
expected_offer_count = int(
    decision_df["ml_decision"].eq("offer").sum()
)
expected_rejected_count = int(
    decision_df["ml_decision"].eq("no_offer").sum()
)

print(
    "ML recommendation v1 database import summary:",
    {
        "planned_records": len(decision_df),
        "successful_records": successful_count,
        "failed_records": failed_count,
        "database_application_verification": verification,
        "database_offer_count": offer_count,
        "foreign_key_violation_count": len(foreign_key_violations),
        "result_csv": IMPORT_RESULT_CSV_PATH,
        "failed_csv": IMPORT_FAILED_CSV_PATH,
    },
)

display(result_df.head(10))

if failed_count != 0:
    raise RuntimeError(
        f"有 {failed_count} 条记录导入失败。"
        f"请查看：{IMPORT_FAILED_CSV_PATH}"
    )

if int(verification["application_count"]) != len(decision_df):
    raise RuntimeError("当前批次 Flexible pipeline Application 数量验证失败。")

if int(verification["offer_application_count"] or 0) != expected_offer_count:
    raise RuntimeError(
        "当前批次 Offer Application 数量不符合动态 ML 决策结果。"
    )

if int(verification["rejected_application_count"] or 0) != expected_rejected_count:
    raise RuntimeError(
        "当前批次 Rejected Application 数量不符合动态 ML 决策结果。"
    )

if offer_count != expected_offer_count:
    raise RuntimeError("当前批次 Offer record 数量不符合动态 ML 决策结果。")

if foreign_key_violations:
    raise RuntimeError(
        f"数据库存在外键违规：{foreign_key_violations[:10]}"
    )

print("ML recommendation v1 database import completed successfully.")
